# DS‑1 · школы и детские сады

**Kaggle → CPU → Internet → Run All.** Основной файл:
**submission_education.csv**. Прошлый ноутбук потери пересадок запускать не нужно.

Этот ноутбук сам воспроизводит лучший hourly + holiday_service (**0,88078**
со слов пользователя) и проверяет две новые группы признаков:

- учебные дни и два сценария каникул, утренние/дневные часы;
- количество школьных и дошкольных объектов рядом с остановками,
  взаимодействующее с календарём и часом.

Дополнительно проверяются отдельные почасовые профили учебных и каникулярных
дней, сохраняющие суммы суток. Это второй исследовательский опыт после
первого сравнения; его выигрыш не приписывается географии.

Школы и детсады разделены. Школьные каникулы не закрывают детсады в признаках.
Часы 06–09, 12–16, 16–19 — окна проверяемой гипотезы, а не подтверждённое
расписание всех учреждений. Неизвестно, какая школа использует какой календарь:
традиционный и модульный сценарии представлены разными признаками.

География: **© OpenStreetMap contributors, ODbL**, исторический срез 01.01.2025,
[лицензия и атрибуция](https://www.openstreetmap.org/copyright).
Радиус 500 м от центра объекта до ближайшей остановки выбран до оценки качества;
это не пеший маршрут и не число учеников. Возможны пропуски и ошибки карты.
Трассы — ретроспективная GTFS-выгрузка 2026 года; режим **competition**.

Данные POI и погода встроены. Повторно скачивать GTFS или карту не нужно.
Маршруты 7/50 и 5 защищены от новой поправки. Географический вариант должен
победить календарный контроль; эффект календаря не приписывается географии.

Минимальный выигрыш — 0,5% ошибки в каждом из сентября и октября, дополнительный
октябрь после переобучения не хуже более чем на 0,5%. Сентябрь–октябрь уже
использовались при подборе; оценка исследовательская. Если улучшения нет,
основной CSV совпадёт с лучшим 0,88078, повторно отправлять его не нужно.


## 1. Настройки
Ссылка уже указана. Режим `yandex` скачивает архив автоматически; `input` оставлен для ранее загруженных файлов.


In [ ]:
import os
import sys
import json
import csv
import hashlib
import subprocess
import tempfile
import zipfile
import importlib.metadata
from pathlib import Path

PROFILES_ONLY = True  # Базовые профили фиксированы; погодные поправки обучаются LightGBM.
RUN_TESTS = True
DATA_SOURCE = 'yandex'  # 'input' — необязательный ручной вариант через Add Input.
YANDEX_URL = 'https://disk.yandex.ru/d/DiFwlfMOauxjBg'
DOWNLOAD_CACHE = '/tmp/mthack_yandex'
INPUT_ROOT = Path(os.environ.get('MTHACK_INPUT_DIR', '/kaggle/input'))
WORK = Path(os.environ.get('MTHACK_WORK_DIR', '/kaggle/working'))
INPUT_FILES = {
    'train': '',   # например /kaggle/input/my-private-data/labels_day_train.csv
    'test': '',    # labels_day_test.csv — фактические метки сентября–октября
    'sample': '',  # исходный test_submission.csv за ноябрь–декабрь
}
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
if sys.version_info < (3, 11):
    raise RuntimeError('Нужен Python 3.11+; выберите актуальную Python-среду Kaggle.')
WORK.mkdir(parents=True, exist_ok=True)
RUNTIME = Path(tempfile.mkdtemp(prefix='mthack_ds1_code_'))
EXTRACTED = Path(tempfile.mkdtemp(prefix='mthack_ds1_inputs_'))
OUT = WORK / 'hourly_experts_results'
print('Python:', sys.version.split()[0])
if not PROFILES_ONLY:
    for package in ['numpy', 'scipy', 'scikit-learn', 'lightgbm', 'catboost']:
        try:
            print(package, importlib.metadata.version(package))
        except importlib.metadata.PackageNotFoundError:
            print(package, 'не установлен (для обучения бустинга установите пакет или PROFILES_ONLY=True)')
print('Результаты:', OUT)
# Install into the model environment using the notebook's pip.
# A target environment does not need its own pip or Kaggle startup hooks.
MODEL_READY = False
MODEL_ENV = Path('/tmp/mthack_hourly_pinned_env')
TRAIN_PYTHON = MODEL_ENV/'bin/python'
MODEL_PROCESS_ENV = {k: v for k, v in os.environ.items()
                     if k not in ('PYTHONPATH', 'PYTHONHOME', 'PYTHONSTARTUP')}
MODEL_PROCESS_ENV['PYTHONNOUSERSITE'] = '1'
subprocess.check_call([sys.executable, '-I', '-m', 'venv', '--without-pip', str(MODEL_ENV)],
                      env=MODEL_PROCESS_ENV)
PINS = ['numpy==2.0.2', 'lightgbm==4.6.0', 'scikit-learn==1.6.1', 'scipy==1.14.1']
subprocess.check_call([sys.executable, '-I', '-m', 'pip', '--python', str(TRAIN_PYTHON),
                       'install', '--quiet', *PINS], env=MODEL_PROCESS_ENV)
probe = "import numpy, lightgbm, sklearn, scipy; import importlib.metadata as m; "
probe += "expected = " + repr(dict(x.split('==') for x in PINS)) + "; "
probe += "actual = {name: m.version(name) for name in expected}; "
probe += "assert actual == expected, (actual, expected); print(actual)"
subprocess.check_call([str(TRAIN_PYTHON), '-I', '-c', probe], env=MODEL_PROCESS_ENV)
MODEL_READY = True
print('Среда обучения готова; версии и импорт библиотек проверены.')


## 2. Встроенный код модели
Ячейка разворачивает точную копию модулей DS‑1 во временной папке.
Ничего не скачивается. Исходные данные не входят в ноутбук.


In [ ]:
SOURCES = {'ml/forecast/core.py': '"""Strict readers and full-grid scoring of organizer hourly labels (MSK)."""\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport math\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nKey = tuple[str, date, int]\nLabels = dict[Key, int]\n\nRUSSIAN_HOLIDAYS_2025 = frozenset({\n    date(2025, 1, 1), date(2025, 1, 2), date(2025, 1, 3), date(2025, 1, 4),\n    date(2025, 1, 5), date(2025, 1, 6), date(2025, 1, 7), date(2025, 1, 8),\n    date(2025, 2, 23), date(2025, 3, 8),\n    date(2025, 5, 1), date(2025, 5, 2), date(2025, 5, 8), date(2025, 5, 9),\n    date(2025, 6, 12), date(2025, 6, 13),\n    date(2025, 11, 3), date(2025, 11, 4),\n    date(2025, 12, 31),\n})\n\nRUSSIAN_WORKING_WEEKENDS_2025 = frozenset({\n    date(2025, 11, 1),\n})\n\n\ndef is_workday(d: date) -> bool:\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return True\n    if d in RUSSIAN_HOLIDAYS_2025:\n        return False\n    return d.weekday() < 5\n\n\ndef effective_weekday(d: date) -> int:\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return 4\n    if d in RUSSIAN_HOLIDAYS_2025:\n        return 6\n    return d.weekday()\n\n\ndef calendar_day_type(d: date) -> int:\n    if not is_workday(d):\n        return 3 if (effective_weekday(d) == 6 or d in RUSSIAN_HOLIDAYS_2025) else 2\n    if d in RUSSIAN_WORKING_WEEKENDS_2025:\n        return 4\n    if d.weekday() == 4:\n        return 1\n    return 0\n\n\ndef days(start: date, end: date):\n    while start <= end:\n        yield start\n        start += timedelta(days=1)\n\n\ndef grid(routes, start, end):\n    return [(str(r), d, h) for r in routes for d in days(start, end) for h in range(24)]\n\n\ndef parse_key(row) -> Key:\n    route = row[\'route\']\n    if not route or not route.isascii() or not route.isdecimal() or int(route) <= 0:\n        raise ValueError(f\'Invalid route: {route!r}\')\n    day = date.fromisoformat(row[\'date\'])\n    if day.isoformat() != row[\'date\']:\n        raise ValueError(\'Date must be YYYY-MM-DD\')\n    hour = int(row[\'hour\'])\n    if str(hour) != row[\'hour\'] or not 0 <= hour <= 23:\n        raise ValueError(f\'Invalid hour: {row["hour"]!r}\')\n    return str(int(route)), day, hour\n\n\ndef load_labels(paths) -> Labels:\n    result = {}\n    for path in paths:\n        with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n            reader = csv.DictReader(f, delimiter=\';\')\n            if reader.fieldnames != [\'route\', \'date\', \'hour\', \'boardings\']:\n                raise ValueError(f\'Unexpected label schema: {reader.fieldnames}\')\n            for row in reader:\n                if None in row or any(v is None for v in row.values()):\n                    raise ValueError(\'Malformed CSV row\')\n                key = parse_key(row)\n                value = int(row[\'boardings\'])\n                if value < 0 or str(value) != row[\'boardings\']:\n                    raise ValueError(\'Boardings must be nonnegative integers\')\n                if key in result:\n                    raise ValueError(f\'Duplicate label key across input files: {key}\')\n                result[key] = value\n    if not result:\n        raise ValueError(\'Labels are empty\')\n    return result\n\n\ndef rounded(value) -> int:\n    value = float(value)\n    if not math.isfinite(value) or abs(value) > 10**12:\n        raise ValueError(\'Prediction must be finite and within 1e12\')\n    return max(0, round(value))  # Same half-to-even rule as DS-2 baseline.\n\n\ndef sha256(path):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, \'sha256\').hexdigest()\n\n\ndef score(keys, truth, predictions, peak_hours=None):\n    """Score exactly the requested grid; absent label = 0 is an explicit assumption."""\n    if len(keys) != len(set(keys)) or set(keys) != set(predictions):\n        raise ValueError(\'Prediction keys must match the unique evaluation grid exactly\')\n    if set(truth) - set(keys):\n        raise ValueError(\'Truth contains keys outside evaluation grid\')\n    groups = defaultdict(lambda: {\'rows\': 0, \'actual_sum\': 0, \'absolute_error_sum\': 0,\n                                  \'pred_minus_actual\': 0, \'missing_label_rows\': 0})\n    for key in keys:\n        route, day, hour = key\n        actual, pred = truth.get(key, 0), rounded(predictions[key])\n        if actual < 0 or not math.isfinite(actual):\n            raise ValueError(\'Invalid truth\')\n        names = [\'all\', f\'route:{route}\', f\'month:{day:%Y-%m}\',\n                 \'hours:day_06_22\' if 6 <= hour < 23 else \'hours:night_23_05\',\n                 \'labels:observed\' if key in truth else \'labels:missing_assumed_zero\']\n        if peak_hours is not None:\n            names.append(\'hours:train_peaks\' if hour in peak_hours.get(route, ())\n                         else \'hours:other\')\n        for name in names:\n            m = groups[name]\n            m[\'rows\'] += 1\n            m[\'actual_sum\'] += actual\n            m[\'absolute_error_sum\'] += abs(pred - actual)\n            m[\'pred_minus_actual\'] += pred - actual\n            m[\'missing_label_rows\'] += key not in truth\n    if not groups[\'all\'][\'actual_sum\']:\n        raise ValueError(\'Global WAPE undefined: sum(y) must be positive\')\n    for m in groups.values():\n        m[\'wape\'] = m[\'absolute_error_sum\'] / m[\'actual_sum\'] if m[\'actual_sum\'] else None\n        m[\'wape_score\'] = max(0, 1 - m[\'wape\']) if m[\'wape\'] is not None else None\n    return dict(sorted(groups.items()))\n\n\ndef peak_hours(history, routes):\n    totals = defaultdict(int)\n    for (r, _, h), y in history.items():\n        totals[r, h] += y\n    return {r: sorted(range(24), key=lambda h: (-totals[r, h], h))[:4] for r in routes}\n', 'ml/forecast/models.py': '"""Models receive ONLY history through cutoff. Predict never accepts future truth."""\nfrom __future__ import annotations\n\nimport math\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom statistics import median, mean\n\nfrom .core import grid, rounded, is_workday, effective_weekday, calendar_day_type\n\n\nclass Profile:\n    def __init__(self, statistic=\'mean\', missing=\'zero\', shrink=0.2, window=None,\n                 use_calendar=False, route50_rule=False):\n        if statistic not in (\'mean\', \'median\') or missing not in (\'zero\', \'observed\'):\n            raise ValueError(\'Invalid profile configuration\')\n        self.statistic, self.missing, self.shrink, self.window = statistic, missing, shrink, window\n        self.use_calendar = use_calendar\n        self.route50_rule = route50_rule\n\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'Training history crosses cutoff or is empty\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history})\n        if self.window:\n            start = max(start, cutoff - timedelta(days=self.window - 1))\n        filtered = {k: v for k, v in history.items() if k[1] >= start}\n        local, hour, route = defaultdict(list), defaultdict(list), defaultdict(list)\n        keys = grid(self.routes, start, cutoff) if self.missing == \'zero\' else filtered\n        for r, d, h in keys:\n            y = filtered.get((r, d, h), 0)\n            w = effective_weekday(d) if self.use_calendar else d.weekday()\n            local[r, w, h].append(y)\n            hour[r, h].append(y)\n            route[r].append(y)\n        fn = mean if self.statistic == \'mean\' else median\n        self.local = {k: fn(v) for k, v in local.items()}\n        self.hour = {k: fn(v) for k, v in hour.items()}\n        self.route = {k: fn(v) for k, v in route.items()}\n        return self\n\n    def predict(self, keys):\n        if any(k[1] <= self.cutoff for k in keys):\n            raise ValueError(\'Prediction dates must be after cutoff\')\n        result = {}\n        for key in keys:\n            r, d, h = key\n            if self.route50_rule and r == \'50\' and not is_workday(d) and date(2025, 9, 6) <= d < date(2025, 11, 15):\n                result[key] = 0\n                continue\n            w = effective_weekday(d) if self.use_calendar else d.weekday()\n            parent = self.hour.get((r, h), self.route.get(r, 0))\n            local = self.local.get((r, w, h), parent)\n            result[key] = rounded((1 - self.shrink) * local + self.shrink * parent)\n        return result\n\n\nclass CalendarProfile:\n    """Optimized calendar profile: blends recent 56-day trend with full history on effective weekdays."""\n    def __init__(self, recent_weight=0.6, window=56, route50_rule=True):\n        self.recent_weight = recent_weight\n        self.window = window\n        self.route50_rule = route50_rule\n\n    def fit(self, history, start, cutoff):\n        self.cutoff = cutoff\n        self.full_profile = Profile(\'median\', shrink=0.0, use_calendar=True,\n                                    route50_rule=self.route50_rule).fit(history, start, cutoff)\n        self.recent_profile = Profile(\'median\', shrink=0.0, window=self.window,\n                                      use_calendar=True, route50_rule=self.route50_rule).fit(history, start, cutoff)\n        return self\n\n    def predict(self, keys):\n        p_full = self.full_profile.predict(keys)\n        p_recent = self.recent_profile.predict(keys)\n        w = self.recent_weight\n        return {k: rounded(w * p_recent[k] + (1 - w) * p_full[k]) for k in keys}\n\n\nclass Boosting:\n    def __init__(self, kind=\'hgb\', recursive=False, use_calendar=True, route50_rule=False):\n        self.kind = kind\n        self.recursive = recursive\n        self.use_calendar = use_calendar\n        self.route50_rule = route50_rule\n\n    def _extract_profiles(self, history, start, cutoff):\n        full_med, full_mean = defaultdict(list), defaultdict(list)\n        rec_med, rec_mean = defaultdict(list), defaultdict(list)\n        hour_med = defaultdict(list)\n        rec_start = max(start, cutoff - timedelta(days=55))\n\n        keys = grid(self.routes, start, cutoff)\n        for r, d, h in keys:\n            y = history.get((r, d, h), 0)\n            eff_w = effective_weekday(d) if self.use_calendar else d.weekday()\n            full_med[r, eff_w, h].append(y)\n            full_mean[r, eff_w, h].append(y)\n            hour_med[r, h].append(y)\n            if d >= rec_start:\n                rec_med[r, eff_w, h].append(y)\n                rec_mean[r, eff_w, h].append(y)\n\n        self.p_full_med = {k: median(v) for k, v in full_med.items()}\n        self.p_full_mean = {k: mean(v) for k, v in full_mean.items()}\n        self.p_rec_med = {k: median(v) for k, v in rec_med.items()}\n        self.p_rec_mean = {k: mean(v) for k, v in rec_mean.items()}\n        self.p_hour_med = {k: median(v) for k, v in hour_med.items()}\n\n    def features(self, keys, context=None):\n        import numpy as np\n        result = []\n        if self.recursive:\n            for r, d, h in keys:\n                t = d.timetuple().tm_yday\n                row = [self.route_codes[r], d.weekday(), h, d.month, t,\n                       int(d.weekday() >= 5), math.sin(2 * math.pi * t / 365),\n                       math.cos(2 * math.pi * t / 365)]\n                if context is not None:\n                    for lag in (1, 7):\n                        key = r, d - timedelta(days=lag), h\n                        row.extend([context.get(key, 0), int(key in context)])\n                result.append(row)\n            return np.asarray(result, dtype=float)\n\n        for r, d, h in keys:\n            t = d.timetuple().tm_yday\n            eff_w = effective_weekday(d) if self.use_calendar else d.weekday()\n            workday = int(is_workday(d)) if self.use_calendar else int(d.weekday() < 5)\n            dtype = calendar_day_type(d) if self.use_calendar else (2 if d.weekday() >= 5 else 0)\n\n            pf_med = getattr(self, \'p_full_med\', {}).get((r, eff_w, h), getattr(self, \'p_hour_med\', {}).get((r, h), 0))\n            pf_mean = getattr(self, \'p_full_mean\', {}).get((r, eff_w, h), pf_med)\n            pr_med = getattr(self, \'p_rec_med\', {}).get((r, eff_w, h), pf_med)\n            pr_mean = getattr(self, \'p_rec_mean\', {}).get((r, eff_w, h), pf_mean)\n            ph_med = getattr(self, \'p_hour_med\', {}).get((r, h), 0)\n            trend_ratio = pr_mean / (pf_mean + 1.0) if pf_mean > 0 else 1.0\n\n            is_r50_susp = int(self.route50_rule and r == \'50\' and not is_workday(d)\n                              and date(2025, 9, 6) <= d < date(2025, 11, 15))\n\n            row = [\n                h,\n                d.weekday(),\n                eff_w,\n                dtype,\n                workday,\n                d.month,\n                d.day,\n                t,\n                math.sin(2 * math.pi * h / 24),\n                math.cos(2 * math.pi * h / 24),\n                math.sin(2 * math.pi * t / 365),\n                math.cos(2 * math.pi * t / 365),\n                pf_med,\n                pf_mean,\n                pr_med,\n                pr_mean,\n                ph_med,\n                trend_ratio,\n                is_r50_susp,\n            ]\n            result.append(row)\n        return np.asarray(result, dtype=float)\n\n    def fit(self, history, start, cutoff):\n        import numpy as np\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'Training history crosses cutoff or is empty\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history})\n        self.route_codes = {r: i for i, r in enumerate(self.routes)}\n        self._extract_profiles(history, start, cutoff)\n        self.history = dict(history)\n\n        keys = grid(self.routes, start, cutoff)\n        if self.recursive:\n            from sklearn.ensemble import HistGradientBoostingRegressor\n            keys = [k for k in keys if k[1] >= start + timedelta(days=7)]\n            self.model = HistGradientBoostingRegressor(\n                loss=\'absolute_error\', learning_rate=.08, max_iter=150,\n                max_leaf_nodes=15, min_samples_leaf=30, l2_regularization=2,\n                categorical_features=[0, 1, 2, 3, 5], early_stopping=False, random_state=2025)\n            self.model.fit(self.features(keys, history), [history.get(k, 0) for k in keys])\n            return self\n\n        # Out-of-time target aggregates: each monthly block uses ONLY prior days.\n        # Reuse the exact feature builder used at inference, with an earlier cutoff.\n        train_keys, train_features = [], []\n        block_start = start + timedelta(days=28)\n        while block_start <= cutoff:\n            next_month = (block_start.replace(day=28) + timedelta(days=4)).replace(day=1)\n            block_end = min(cutoff, next_month - timedelta(days=1))\n            prefix_end = block_start - timedelta(days=1)\n            prefix = {k:v for k,v in history.items() if k[1] <= prefix_end}\n            self._extract_profiles(prefix, start, prefix_end)\n            batch = grid(self.routes, block_start, block_end)\n            train_keys.extend(batch)\n            train_features.extend(self.features(batch))\n            block_start = block_end + timedelta(days=1)\n        if not train_keys:\n            raise ValueError(\'Direct boosting needs at least 29 days of history\')\n        # Final prediction profiles may now use all history up to the actual cutoff.\n        self._extract_profiles(history, start, cutoff)\n        route_keys, route_features = defaultdict(list), defaultdict(list)\n        for k, features in zip(train_keys, train_features):\n            route_keys[k[0]].append(k)\n            route_features[k[0]].append(features)\n\n        self.models = {}\n        for r in self.routes:\n            r_keys = route_keys[r]\n            X = np.asarray(route_features[r], dtype=float)\n            y = np.array([history.get(k, 0) for k in r_keys], dtype=float)\n\n            if self.kind == \'lgb\':\n                try:\n                    import lightgbm as lgb\n                    model = lgb.LGBMRegressor(\n                        objective=\'regression_l1\', metric=\'mae\', learning_rate=0.03,\n                        n_estimators=300, num_leaves=15, min_child_samples=10,\n                        subsample=0.85, colsample_bytree=0.85, random_state=2025,\n                        verbose=-1, n_jobs=2\n                    )\n                    model.fit(X, y)\n                    self.models[r] = model\n                    continue\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n            if self.kind == \'catboost\':\n                try:\n                    import catboost as cb\n                    model = cb.CatBoostRegressor(\n                        loss_function=\'MAE\', eval_metric=\'MAE\', iterations=300,\n                        learning_rate=0.04, depth=5, random_seed=2025, verbose=0,\n                        thread_count=2\n                    )\n                    model.fit(X, y)\n                    self.models[r] = model\n                    continue\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n            if self.kind == \'blend\':\n                submodels = []\n                try:\n                    import lightgbm as lgb\n                    m1 = lgb.LGBMRegressor(\n                        objective=\'regression_l1\', metric=\'mae\', learning_rate=0.03,\n                        n_estimators=300, num_leaves=15, min_child_samples=10,\n                        subsample=0.85, colsample_bytree=0.85, random_state=2025,\n                        verbose=-1, n_jobs=2\n                    )\n                    m1.fit(X, y)\n                    submodels.append((\'lgb\', m1, 0.5))\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n                try:\n                    import catboost as cb\n                    m2 = cb.CatBoostRegressor(\n                        loss_function=\'MAE\', eval_metric=\'MAE\', iterations=300,\n                        learning_rate=0.04, depth=5, random_seed=2025, verbose=0,\n                        thread_count=2\n                    )\n                    m2.fit(X, y)\n                    submodels.append((\'catboost\', m2, 0.5))\n                except Exception as exc:\n                    raise RuntimeError(f\'{self.kind} training failed for route {r}; no silent fallback\') from exc\n\n                if not submodels:\n                    from sklearn.ensemble import HistGradientBoostingRegressor\n                    m3 = HistGradientBoostingRegressor(\n                        loss=\'absolute_error\', learning_rate=0.04, max_iter=200,\n                        max_leaf_nodes=15, min_samples_leaf=10, l2_regularization=1,\n                        early_stopping=False, random_state=2025\n                    )\n                    m3.fit(X, y)\n                    submodels.append((\'hgb\', m3, 1.0))\n                self.models[r] = submodels\n                continue\n\n            from sklearn.ensemble import HistGradientBoostingRegressor\n            model = HistGradientBoostingRegressor(\n                loss=\'absolute_error\', learning_rate=0.04, max_iter=200,\n                max_leaf_nodes=15, min_samples_leaf=10, l2_regularization=1,\n                early_stopping=False, random_state=2025\n            )\n            model.fit(X, y)\n            self.models[r] = model\n\n        return self\n\n    def _predict_route(self, r, X):\n        m = getattr(self, \'models\', {}).get(r)\n        if m is None:\n            if hasattr(self, \'model\'):\n                return self.model.predict(X)\n            return [0] * len(X)\n        if isinstance(m, list):\n            total_w = sum(w for _, _, w in m)\n            preds = sum(w * sub.predict(X) for _, sub, w in m) / total_w\n            return preds\n        return m.predict(X)\n\n    def predict(self, keys):\n        if any(k[1] <= self.cutoff for k in keys):\n            raise ValueError(\'Prediction dates must be after cutoff\')\n        known = [k for k in keys if k[0] in self.route_codes]\n        result = {k: 0 for k in keys}\n        if not known:\n            return result\n\n        if not self.recursive:\n            route_known = defaultdict(list)\n            for k in known:\n                route_known[k[0]].append(k)\n            for r, r_keys in route_known.items():\n                X = self.features(r_keys)\n                raw_preds = self._predict_route(r, X)\n                for k, p in zip(r_keys, raw_preds):\n                    if self.route50_rule and r == \'50\' and not is_workday(k[1]) and date(2025, 9, 6) <= k[1] < date(2025, 11, 15):\n                        result[k] = 0\n                    else:\n                        result[k] = rounded(p)\n            return result\n\n        context = dict(self.history)\n        day, end = self.cutoff + timedelta(days=1), max(k[1] for k in known)\n        wanted = set(known)\n        while day <= end:\n            batch = grid(self.routes, day, day)\n            predictions = self.model.predict(self.features(batch, context))\n            for key, value in zip(batch, predictions):\n                context[key] = rounded(value)\n                if key in wanted:\n                    result[key] = context[key]\n            day += timedelta(days=1)\n        return result\n\n\ndef candidates(include_boosting=True):\n    result = {\n        \'mean_zero\': lambda: Profile(\'mean\'),\n        \'median_zero\': lambda: Profile(\'median\'),\n        \'mean_observed\': lambda: Profile(\'mean\', \'observed\'),\n        \'median_observed\': lambda: Profile(\'median\', \'observed\'),\n        \'mean_56d\': lambda: Profile(\'mean\', window=56),\n        \'profile_calendar\': lambda: CalendarProfile(recent_weight=0.6, window=56, route50_rule=True),\n    }\n    if include_boosting:\n        result.update(\n            hgb_direct=lambda: Boosting(kind=\'hgb\'),\n            lgb_direct=lambda: Boosting(kind=\'lgb\'),\n            catboost_direct=lambda: Boosting(kind=\'catboost\'),\n            blend_ensemble=lambda: Boosting(kind=\'blend\'),\n            hgb_recursive=lambda: Boosting(recursive=True),\n        )\n    return result\n\n', 'ml/forecast/route_specific.py': '"""DS-2-informed route-specific selection for a fixed 61-day forecast.\n\nNo validation labels enter fitted candidates. Selection scores are exploratory:\nSep-Oct has already been examined repeatedly by DS-2.\n"""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport importlib.util\nimport json\nimport pickle\nimport time\nfrom collections import defaultdict\nfrom datetime import date, timedelta\nfrom pathlib import Path\nimport numpy as np\nfrom .core import days, grid, load_labels, rounded, score, sha256, effective_weekday\nfrom .models import Profile, Boosting\nfrom .run import FIRST, FINAL_CUTOFF, FOLDS, dump\nfrom ml.submit.adapter import read_sample, write_submission, export_api, validate\n\nRESEARCH_COMMIT = \'b593ebb\'\nEXPECTED_HASHES = {\n    \'train\': \'20a61009b4ee00327784c6fa888109609fd3a1905ea3a8509b36964dd9baced7\',\n    \'test\': \'37479acea3e319ac613d72df7217bbe57b66af49644e538139174fd534ce2347\',\n    \'sample\': \'71c8126bb51068c936462a170da885bad860fc91a58b3d2ac731876d324d7ec3\',\n}\n\n\ndef verify_inputs(paths, synthetic=False):\n    result = {role: {\'name\': p.name, \'sha256\': sha256(p)} for role, p in paths.items()}\n    if not synthetic:\n        for role, item in result.items():\n            if item[\'sha256\'] != EXPECTED_HASHES[role]:\n                raise ValueError(f\'{role}: SHA256 does not match real DS-2 inputs. \'\n                                 \'Do not train a submission on synthetic or unverified files. \'\n                                 f"Received {item[\'sha256\']}; expected {EXPECTED_HASHES[role]}")\n    return result\n\n\nclass SeasonalProfile:\n    """Calendar regimes, robust hourly shapes, optional local history blend.\n\nSummer weekdays are a separate regime, not deleted unconditionally: if a\nforecast requests summer, use summer history when at least 3 dates exist.\nFor autumn this reproduces the DS-2 school-term exclusion candidate.\n"""\n    def __init__(self, statistic=\'median\', shrink=.2, window=None,\n                 seasonal=True, calendar=True, recent_weight=0., regime7=False):\n        self.statistic, self.shrink, self.window = statistic, shrink, window\n        self.seasonal, self.calendar = seasonal, calendar\n        self.recent_weight, self.regime7 = recent_weight, regime7\n\n    @staticmethod\n    def summer(d):\n        return d.month in (7, 8) and d.weekday() < 5\n\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'History empty or crosses cutoff\')\n        self.cutoff = cutoff\n        self.routes = sorted({k[0] for k in history}, key=int)\n        self.tables = {}\n        # All keys are explicit, including absent training labels = zero assumption.\n        for r in self.routes:\n            for season in (None, False, True):\n                ds = [d for d in days(start, cutoff)\n                      if (not self.window or (cutoff-d).days < self.window)\n                      and (season is None or self.summer(d) == season)]\n                local, parent = defaultdict(list), defaultdict(list)\n                recent = defaultdict(list)\n                regime = defaultdict(list)\n                for d in ds:\n                    w = effective_weekday(d) if self.calendar else d.weekday()\n                    for h in range(24):\n                        y = history.get((r, d, h), 0)\n                        local[w,h].append(y); parent[h].append(y)\n                        if (cutoff-d).days < 56:\n                            recent[w,h].append(y)\n                        if r == \'7\' and cutoff >= date(2025,8,16) and d >= date(2025,8,16):\n                            regime[w,h].append(y)\n                fn = np.mean if self.statistic == \'mean\' else np.median\n                self.tables[r,season] = tuple({k: float(fn(v)) for k,v in t.items() if len(v) >= 3}\n                                             for t in (local,parent,recent,regime))\n        return self\n\n    def predict(self, keys):\n        if any(d <= self.cutoff for _,d,_ in keys):\n            raise ValueError(\'Prediction dates must follow cutoff\')\n        out = {}\n        for key in keys:\n            r,d,h = key\n            if r not in self.routes:\n                out[key] = 0; continue\n            season = self.summer(d) if self.seasonal else None\n            loc,par,rec,reg = self.tables[r,season]\n            fallback,fp,_,_ = self.tables[r,None]\n            w = effective_weekday(d) if self.calendar else d.weekday()\n            parent = par.get(h,fp.get(h,0))\n            local = loc.get((w,h),fallback.get((w,h),parent))\n            p = (1-self.shrink)*local + self.shrink*parent\n            if (w,h) in rec:\n                p = (1-self.recent_weight)*p + self.recent_weight*rec[w,h]\n            if self.regime7 and r == \'7\' and d.weekday() >= 5 and (w,h) in reg:\n                # DS-2 experiment 13: 25% sparse post-Aug16 profile, no future notice.\n                p = .75*p + .25*reg[w,h]\n            out[key] = rounded(p)\n        return out\n\n\ndef candidate_factories(profiles_only=False):\n    pool = {\n        \'median_zero\': lambda: Profile(\'median\'),\n        \'mean_zero\': lambda: Profile(\'mean\'),\n        \'school_ds2\': lambda: SeasonalProfile(calendar=False),\n        \'school_calendar\': lambda: SeasonalProfile(),\n        \'school_local\': lambda: SeasonalProfile(shrink=0),\n        \'school_mean\': lambda: SeasonalProfile(statistic=\'mean\'),\n        \'school_recent25\': lambda: SeasonalProfile(recent_weight=.25),\n        \'school_recent50\': lambda: SeasonalProfile(recent_weight=.5),\n        \'school_route7\': lambda: SeasonalProfile(regime7=True),\n        \'median_56d\': lambda: SeasonalProfile(window=56, seasonal=False),\n        \'median_112d\': lambda: SeasonalProfile(window=112, seasonal=False),\n        \'mean_56d\': lambda: SeasonalProfile(statistic=\'mean\',window=56,seasonal=False),\n        \'mean_112d\': lambda: SeasonalProfile(statistic=\'mean\',window=112,seasonal=False),\n    }\n    if not profiles_only:\n        pool.update({\n            \'hgb_causal\': lambda: Boosting(kind=\'hgb\',route50_rule=False),\n            \'lgb_causal\': lambda: Boosting(kind=\'lgb\',route50_rule=False),\n            \'catboost_causal\': lambda: Boosting(kind=\'catboost\',route50_rule=False),\n        })\n    return pool\n\n\ndef choose_routes(keys, truth, predictions, routes, min_gain=.01):\n    """Conservative route choice: beat school baseline in both calendar months.\n\nA predeclared 50/50 blend is also considered. No continuous weight fitting.\nFall back to school baseline unless each month improves by >= 1% in MAE.\n"""\n    months = sorted({d.month for _,d,_ in keys})\n    choices, diagnostics = {}, {}\n    for r in routes:\n        rk = [k for k in keys if k[0] == r]\n        by_month = [[k for k in rk if k[1].month == m] for m in months]\n        def losses(name, weight):\n            return [sum(abs(rounded(weight*predictions[name][k] +\n                                (1-weight)*predictions[\'school_calendar\'][k])-truth.get(k,0))\n                        for k in batch) for batch in by_month]\n        base = losses(\'school_calendar\',1.)\n        options = [(\'school_calendar\',1.,base)]\n        diagnostics[r] = {\'baseline_month_errors\':base,\'candidates\':{}}\n        for name in predictions:\n            for weight in (0.5,1.):\n                errs = losses(name,weight)\n                diagnostics[r][\'candidates\'][f\'{name}:{weight}\'] = errs\n                if all(e <= b*(1-min_gain) for e,b in zip(errs,base)) and sum(errs)<sum(base):\n                    options.append((name,weight,errs))\n        name,weight,errs = min(options,key=lambda item:(sum(item[2]),item[0],item[1]))\n        choices[r] = {\'candidate\':name,\'weight\':weight,\'month_errors\':errs}\n    return choices,diagnostics\n\n\ndef combine(keys, predictions, choices):\n    result = {}\n    for key in keys:\n        choice = choices.get(key[0])\n        if choice is None:\n            result[key] = 0\n        else:\n            name,w = choice[\'candidate\'],choice[\'weight\']\n            result[key] = rounded(w*predictions[name][key]+(1-w)*predictions[\'school_calendar\'][key])\n    return result\n\n\nclass RouteEnsemble:\n    def __init__(self, models, choices, cutoff):\n        self.models,self.choices,self.cutoff = models,choices,cutoff\n    def predict(self,keys):\n        return combine(keys,{name:m.predict(keys) for name,m in self.models.items()},self.choices)\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    for name in (\'train\',\'test\',\'sample\',\'out\'):p.add_argument(\'--\'+name,type=Path,required=True)\n    p.add_argument(\'--profiles-only\',action=\'store_true\')\n    p.add_argument(\'--synthetic\',action=\'store_true\')\n    args=p.parse_args(argv)\n    inputs=verify_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')},args.synthetic)\n    train,test=load_labels([args.train]),load_labels([args.test])\n    if any(not FIRST<=k[1]<=date(2025,8,31) for k in train):raise ValueError(\'Invalid train period\')\n    if any(not date(2025,9,1)<=k[1]<=FINAL_CUTOFF for k in test):raise ValueError(\'Invalid test period\')\n    values=load_labels([args.train,args.test]); _,sample=read_sample(args.sample)\n    routes=sorted({k[0] for k in values},key=int)\n    if set(routes)-{k[0] for k in sample}:raise ValueError(\'Sample misses known routes\')\n    for lo,hi,part in [(FIRST,date(2025,8,31),train),(date(2025,9,1),FINAL_CUTOFF,test)]:\n        if {k[1] for k in part} != set(days(lo,hi)):raise ValueError(\'Missing complete input dates\')\n    if not args.profiles_only:\n        missing=[n for n in (\'sklearn\',\'lightgbm\',\'catboost\') if importlib.util.find_spec(n) is None]\n        if missing:raise RuntimeError(\'Missing training packages: \'+\', \'.join(missing)+\'. Install them or use --profiles-only explicitly.\')\n    factories=candidate_factories(args.profiles_only)\n    args.out.mkdir(parents=True,exist_ok=True)\n    report={\'schema_version\':2,\'research_commit\':RESEARCH_COMMIT,\'inputs\':inputs,\n            \'data_kind\':\'synthetic_not_for_submission\' if args.synthetic else \'verified_organizer_labels\',\n            \'selection_warning\':\'Research validation repeatedly examined in DS-2; not an independent test or platform score.\',\n            \'missing_label_policy\':\'zero-filled full grid for training and scoring; not proof of zero physical demand\',\n            \'external_policy\':\'calendar only; no future network notices, weather or validation labels in features\',\n            \'folds\':{},\'platform_score\':None,\'library_versions\':{}}\n    for pkg in (\'numpy\',\'scikit-learn\',\'lightgbm\',\'catboost\'):\n        try:report[\'library_versions\'][pkg]=importlib.metadata.version(pkg)\n        except importlib.metadata.PackageNotFoundError:pass\n    for label,cutoff,start,end in FOLDS:\n        history={k:v for k,v in values.items() if k[1]<=cutoff}\n        keys=grid(routes,start,end); truth={k:v for k,v in values.items() if start<=k[1]<=end}\n        forecasts={};report[\'folds\'][label]={}\n        for name,factory in factories.items():\n            t=time.perf_counter();model=factory().fit(history,FIRST,cutoff)\n            forecasts[name]=model.predict(keys)\n            m=score(keys,truth,forecasts[name])\n            report[\'folds\'][label][name]={\'metrics\':m,\'seconds\':round(time.perf_counter()-t,3)}\n            print(f\'{label} {name}: score={m["all"]["wape_score"]:.6f}\',flush=True)\n        if label==\'sep_oct\':\n            # Forward diagnostic: choose using September only, evaluate October\n            # from the SAME Aug31 forecasts (no October/September refitting).\n            sep=[k for k in keys if k[1].month==9];october=[k for k in keys if k[1].month==10]\n            sept_choices,_=choose_routes(sep,truth,forecasts,routes)\n            op=combine(october,forecasts,sept_choices)\n            report[\'october_forward_diagnostic\']={\'selection\':\'September only; forecasts fixed at Aug31\',\n                \'choices\':sept_choices,\'metrics\':score(october,{k:truth[k] for k in october if k in truth},op)}\n            choices,diagnostics=choose_routes(keys,truth,forecasts,routes)\n            ensemble=combine(keys,forecasts,choices)\n            report[\'folds\'][label][\'route_ensemble\']={\'metrics\':score(keys,truth,ensemble)}\n            report[\'selection\']={\'selected\':\'route_ensemble\',\'by_route\':choices,\'diagnostics\':diagnostics,\n                \'rule\':\'Per route: lower pooled Sep-Oct error AND >=1% reduction in each month vs school_calendar; weights 0.5 or 1.0.\'}\n        dump(args.out/\'report.json\',report)\n    selected_names={\'school_calendar\'}|{c[\'candidate\'] for c in choices.values()}\n    fitted={n:factories[n]().fit(values,FIRST,FINAL_CUTOFF) for n in sorted(selected_names)}\n    model=RouteEnsemble(fitted,choices,FINAL_CUTOFF);predictions=model.predict(sample)\n    report[\'code_sha256\']={q.name:sha256(q) for q in sorted(Path(__file__).parent.glob(\'*.py\'))}\n    identity=json.dumps({\'inputs\':inputs,\'choices\':choices,\'code\':report[\'code_sha256\'],\n                         \'libraries\':report[\'library_versions\']},sort_keys=True).encode()\n    version=\'ds1-route-\'+hashlib.sha256(identity).hexdigest()[:12]\n    if args.synthetic:version=\'synthetic-NOT-FOR-SUBMISSION-\'+version\n    report[\'model_version\']=version\n    report[\'cold_start\']={\'routes\':sorted({k[0] for k in sample}-set(routes),key=int),\n                          \'policy\':\'zero placeholder; unvalidated, not a required true target\'}\n    report[\'submission\']=write_submission(args.sample,predictions,args.out/\'submission.csv\')\n    export_api(sample,predictions,args.out/\'forecast_api.csv\',version)\n    (args.out/\'model.local.pkl\').write_bytes(pickle.dumps(model,protocol=5))\n    validate(args.sample,args.out/\'submission.csv\')\n    report[\'outputs_sha256\']={n:sha256(args.out/n) for n in (\'submission.csv\',\'forecast_api.csv\',\'model.local.pkl\')}\n    report[\'matches_previous_submission\'] = report[\'outputs_sha256\'][\'submission.csv\'] == \'371d05c44e20b63db110f5a5e27b8da8fafb785fdbc66374307b0ab7304a75ac\'\n    dump(args.out/\'report.json\',report)\n    print(\'Selected route models:\',json.dumps(choices,ensure_ascii=False),flush=True)\n    print(\'Validated:\',version, len(sample),\'rows; platform score unknown.\',flush=True)\n    return report\n\nif __name__==\'__main__\':\n    from ml.forecast.route_specific import main as run_main\n    run_main()\n', 'ml/forecast/test_route_specific.py': "import tempfile\nimport importlib.util\nimport unittest\nfrom datetime import date,timedelta\nfrom pathlib import Path\nfrom unittest.mock import patch\nimport numpy as np\nfrom ml.forecast.core import grid\nfrom ml.forecast.models import Boosting\nfrom ml.forecast.route_specific import SeasonalProfile,choose_routes,combine,verify_inputs,candidate_factories\n\n\nclass RouteTests(unittest.TestCase):\n    def test_unverified_inputs_rejected_before_training(self):\n        with tempfile.TemporaryDirectory() as td:\n            p=Path(td)/'fake.csv';p.write_text('synthetic')\n            with self.assertRaisesRegex(ValueError,'does not match real DS-2'):\n                verify_inputs({'train':p})\n            self.assertIn('train',verify_inputs({'train':p},synthetic=True))\n\n    def test_autumn_does_not_inherit_summer_weekday_slump(self):\n        start,cutoff=date(2025,1,1),date(2025,8,31)\n        hist={k: (20 if k[1].month in (7,8) and k[1].weekday()<5 else 100)\n              for k in grid(['1'],start,cutoff)}\n        model=SeasonalProfile(calendar=False).fit(hist,start,cutoff)\n        k=('1',date(2025,9,1),12)\n        self.assertEqual(model.predict([k])[k],100)\n        self.assertEqual(model.predict([('5',date(2025,9,1),12)])[('5',date(2025,9,1),12)],0)\n        hist['1',date(2025,9,1),12]=999\n        with self.assertRaises(ValueError):model.fit(hist,start,cutoff)\n\n    def test_route_choices_differ_and_single_month_overfit_rejected(self):\n        keys=[(r,date(2025,m,1),12) for r in ('1','7') for m in (9,10)]\n        truth={k:100 for k in keys}\n        preds={'school_calendar':dict.fromkeys(keys,80),\n               'a':{k:100 if k[0]=='1' else 70 for k in keys},\n               'b':{k:100 if k[0]=='7' else 70 for k in keys},\n               'overfit':{k:100 if k[1].month==9 else 10 for k in keys}}\n        choices,_=choose_routes(keys,truth,preds,['1','7'])\n        self.assertEqual(choices['1']['candidate'],'a')\n        self.assertEqual(choices['7']['candidate'],'b')\n        self.assertEqual(combine(keys,preds,choices),truth)\n\n    def test_boosting_target_aggregates_are_strictly_past(self):\n        start,cutoff=date(2025,1,1),date(2025,3,31)\n        hist={k:(k[1]-start).days*100+k[2] for k in grid(['1'],start,cutoff)}\n        observed=[]\n        class Spy(Boosting):\n            def _extract_profiles(self,history,start,cutoff):\n                self.profile_end=cutoff\n                return super()._extract_profiles(history,start,cutoff)\n            def features(self,keys,context=None):\n                observed.append((self.profile_end,min(k[1] for k in keys)))\n                assert self.profile_end<min(k[1] for k in keys)\n                return super().features(keys,context)\n        Spy(kind='hgb').fit(hist,start,cutoff)\n        self.assertGreaterEqual(len(observed),3)\n\n    @unittest.skipUnless(importlib.util.find_spec('lightgbm'), 'optional LightGBM')\n    def test_training_error_cannot_silently_change_library(self):\n        hist={k:100 for k in grid(['1'],date(2025,1,1),date(2025,2,28))}\n        with patch('lightgbm.LGBMRegressor.fit',side_effect=ValueError('broken fit')):\n            with self.assertRaisesRegex(RuntimeError,'no silent fallback'):\n                Boosting(kind='lgb').fit(hist,date(2025,1,1),date(2025,2,28))\n\n    def test_candidates_have_no_future_route50_rule(self):\n        for name,factory in candidate_factories().items():\n            model=factory()\n            self.assertFalse(getattr(model,'route50_rule',False),name)\n\nif __name__=='__main__':unittest.main()\n", 'ml/forecast/run.py': '"""One-command chronological backtest, model selection and exact submission."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nimport pickle\nimport platform\nimport time\nfrom datetime import date\nfrom pathlib import Path\n\n# A bounded CPU budget also avoids OpenMP oversubscription on desktops.\nos.environ.setdefault(\'OMP_NUM_THREADS\', \'2\')\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\', \'2\')\n\nfrom .core import grid, load_labels, peak_hours, score, sha256\nfrom .models import candidates\nfrom ml.submit.adapter import export_api, read_sample, validate, write_submission\n\nFIRST, FINAL_CUTOFF = date(2025, 1, 1), date(2025, 10, 31)\nFOLDS = [\n    (\'may_jun\', date(2025, 4, 30), date(2025, 5, 1), date(2025, 6, 30)),\n    (\'jul_aug\', date(2025, 6, 30), date(2025, 7, 1), date(2025, 8, 31)),\n    (\'sep_oct\', date(2025, 8, 31), date(2025, 9, 1), date(2025, 10, 31)),\n]\n\n\ndef dump(path, data):\n    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False) + \'\\n\',\n                          encoding=\'utf-8\')\n\n\ndef evaluate_fold(values, cutoff, start, end, factory):\n    history = {k: v for k, v in values.items() if FIRST <= k[1] <= cutoff}\n    routes = sorted({k[0] for k in history}, key=int)\n    truth = {k: v for k, v in values.items() if start <= k[1] <= end}\n    if {k[0] for k in truth} - set(routes):\n        raise ValueError(\'Validation has new routes: define cold-start evaluation explicitly\')\n    keys = grid(routes, start, end)\n    t = time.perf_counter()\n    model = factory().fit(history, FIRST, cutoff)\n    predictions = model.predict(keys)\n    metrics = score(keys, truth, predictions, peak_hours(history, routes))\n    return {\'cutoff\': str(cutoff), \'from\': str(start), \'to_inclusive\': str(end),\n            \'routes\': routes, \'metrics\': metrics,\n            \'fit_predict_score_seconds\': round(time.perf_counter() - t, 3)}\n\n\ndef select_dev(folds, names):\n    """Select on May–August only; pool absolute errors, never average route scores."""\n    losses = {name: sum(folds[fold][name][\'metrics\'][\'all\'][\'absolute_error_sum\']\n                       for fold in (\'may_jun\', \'jul_aug\')) for name in names}\n    return min(losses, key=lambda n: (losses[n], n)), losses\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--train\', required=True, type=Path)\n    parser.add_argument(\'--test\', required=True, type=Path)\n    parser.add_argument(\'--sample\', required=True, type=Path)\n    parser.add_argument(\'--out\', required=True, type=Path)\n    parser.add_argument(\'--profiles-only\', action=\'store_true\',\n                        help=\'Skip optional scikit-learn candidates explicitly\')\n    parser.add_argument(\'--synthetic\', action=\'store_true\',\n                        help=\'Mark artificial smoke data and outputs as NOT FOR SUBMISSION\')\n    args = parser.parse_args(argv)\n    from .route_specific import verify_inputs\n    verify_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')}, args.synthetic)\n    # Fail on malformed input before training or creating any outputs.\n    _, sample_keys = read_sample(args.sample)\n    train, test = load_labels([args.train]), load_labels([args.test])\n    if any(not FIRST <= k[1] <= date(2025, 8, 31) for k in train):\n        raise ValueError(\'Train must be January–August 2025 only\')\n    if any(not date(2025, 9, 1) <= k[1] <= FINAL_CUTOFF for k in test):\n        raise ValueError(\'Test labels must be September–October 2025 only\')\n    values = load_labels([args.train, args.test])\n    # Missing hours are allowed, an entirely missing date is a data availability problem.\n    for a, b, data in ((FIRST, date(2025, 8, 31), train),\n                       (date(2025, 9, 1), FINAL_CUTOFF, test)):\n        if {k[1] for k in data} != {k[1] for k in grid([\'check\'], a, b)}:\n            raise ValueError(\'Labels have missing full dates; audit before running model selection\')\n    if {k[0] for k in values} - {k[0] for k in sample_keys}:\n        raise ValueError(\'Historical routes absent from sample\')\n    models = candidates(not args.profiles_only)\n    if not args.profiles_only:\n        import sklearn  # noqa: F401; fail early instead of silently skipping boosting.\n    args.out.mkdir(parents=True, exist_ok=True)\n    inputs = {name: {\'name\': path.name, \'sha256\': sha256(path)}\n              for name, path in [(\'train\', args.train), (\'test\', args.test), (\'sample\', args.sample)]}\n    code_files = sorted(Path(__file__).parent.glob(\'*.py\')) + [\n        Path(__file__).parents[1] / \'submit\' / \'adapter.py\']\n    code = {str(p.relative_to(Path(__file__).parents[2])): sha256(p) for p in code_files}\n    installed_libs = {}\n    if not args.profiles_only:\n        for pkg in (\'numpy\', \'scipy\', \'scikit-learn\', \'lightgbm\', \'catboost\'):\n            try:\n                installed_libs[pkg] = importlib.metadata.version(pkg)\n            except importlib.metadata.PackageNotFoundError:\n                pass\n    report = {\n        \'schema_version\': 1, \'inputs\': inputs, \'code_sha256\': code,\n        \'data_kind\': \'synthetic_not_for_submission\' if args.synthetic else \'organizer_labels\',\n        \'python_version\': platform.python_version(),\n        \'library_versions\': installed_libs,\n        \'source_mode\': \'as_of_2025_10_31\', \'timezone\': \'Europe/Moscow\',\n        \'missing_label_policy\': \'zero for scoring; compare zero-filled and observed-only profiles\',\n        \'rounding\': \'clip negatives to zero; Python round half-to-even\',\n        \'peak_definition\': \'four largest hour sums per route in each training window only\',\n        \'night_definition\': \'23:00–05:59 MSK diagnostic; not an operating-hours assertion\',\n        \'external_factors\': \'Russian 2025 production calendar, route 50 weekend resumption policy\',\n        \'parameters\': {\'profile_shrink\': .2, \'short_window_days\': 56, \'recent_weight\': 0.6,\n                       \'lgb\': {\'objective\': \'regression_l1\', \'learning_rate\': .03, \'n_estimators\': 300, \'num_leaves\': 15},\n                       \'catboost\': {\'loss_function\': \'MAE\', \'iterations\': 300, \'learning_rate\': .04, \'depth\': 5},\n                       \'calendar\': \'Russian 2025 production calendar with working Saturdays and state holidays\',\n                       \'route50_policy\': \'weekend repair 2025-09-06..2025-11-14; resumed 2025-11-15\'},\n        \'folds\': {}, \'selection\': {}, \'platform_score\': None,\n    }\n    for fold, cutoff, start, end in FOLDS[:2]:\n        report[\'folds\'][fold] = {}\n        for name, factory in models.items():\n            result = evaluate_fold(values, cutoff, start, end, factory)\n            report[\'folds\'][fold][name] = result\n            print(f\'{fold} {name}: WAPE-score={result["metrics"]["all"]["wape_score"]:.6f}\', flush=True)\n        dump(args.out / \'report.json\', report)\n    proposed, losses = select_dev(report[\'folds\'], models)\n    baseline, _ = select_dev(report[\'folds\'], [\'mean_zero\', \'median_zero\'])\n    report[\'selection\'].update(development_candidate=proposed, development_baseline=baseline,\n                               development_absolute_errors=losses)\n    # Preselected candidate plus controls and calendar profile see Sep–Oct.\n    report[\'folds\'][\'sep_oct\'] = {}\n    candidates_to_eval = dict.fromkeys([baseline, \'mean_zero\', \'median_zero\', \'profile_calendar\', proposed])\n    for name in candidates_to_eval:\n        result = evaluate_fold(values, *FOLDS[2][1:], models[name])\n        report[\'folds\'][\'sep_oct\'][name] = result\n        print(f\'sep_oct {name}: WAPE-score={result["metrics"]["all"]["wape_score"]:.6f}\', flush=True)\n    latest = report[\'folds\'][\'sep_oct\']\n    err = lambda name: latest[name][\'metrics\'][\'all\'][\'absolute_error_sum\']\n    final_baseline = min([\'mean_zero\', \'median_zero\'], key=lambda n: (err(n), n))\n    best_candidate = min([c for c in [proposed, \'profile_calendar\'] if c in latest], key=lambda n: (err(n), n))\n    selected = best_candidate if err(best_candidate) < err(final_baseline) else final_baseline\n    report[\'selection\'].update(selected=selected,\n        final_baseline=final_baseline,\n        rule=\'candidate from pooled May–August error; keep it only if better than both registered baselines in Sep–Oct\',\n        caveat=\'Sep–Oct is a validation gate used in selection, not an untouched test estimate\')\n    # Refit final model on all available history through Oct 31, independently of backtest fits.\n    model = models[selected]().fit(values, FIRST, FINAL_CUTOFF)\n    predictions = model.predict(sample_keys)\n    cold = sorted({k[0] for k in sample_keys} - {k[0] for k in values}, key=int)\n    report[\'cold_start\'] = {\'routes\': cold, \'policy\': \'zero; no evidence to calibrate unseen routes\',\n                            \'evaluated\': False, \'route5_rows_preserved\': True}\n    identity = json.dumps({\'inputs\': inputs, \'code\': code, \'selected\': selected,\n                           \'parameters\': report[\'parameters\'], \'libraries\': report[\'library_versions\'],\n                           \'python\': report[\'python_version\']}, sort_keys=True).encode()\n    version = f\'ds1-{selected}-\' + hashlib.sha256(identity).hexdigest()[:12]\n    if args.synthetic:\n        version = \'synthetic-NOT-FOR-SUBMISSION-\' + version\n    report[\'model_version\'] = version\n    # Local artifact only. Never load an untrusted pickle and never publish organizer-derived artifacts.\n    artifact_path = args.out / \'model.local.pkl\'\n    artifact_path.write_bytes(pickle.dumps(model, protocol=5))\n    output_path = args.out / \'submission.csv\'\n    report[\'submission\'] = write_submission(args.sample, predictions, output_path)\n    export_api(sample_keys, predictions, args.out / \'forecast_api.csv\', version)\n    report[\'outputs_sha256\'] = {p.name: sha256(p) for p in\n                                (artifact_path, output_path, args.out / \'forecast_api.csv\')}\n    if sha256(args.sample) != inputs[\'sample\'][\'sha256\']:\n        raise RuntimeError(\'Original sample changed during run\')\n    validate(args.sample, output_path)\n    dump(args.out / \'report.json\', report)\n    print(f\'Selected {version}; validated {len(sample_keys)} rows. Platform result unknown.\', flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    main()\n', 'ml/forecast/prepare_inputs.py': '"""Extract only labels/sample/description from a local organizer ZIP, never raw events."""\nimport argparse\nimport json\nimport zipfile\nfrom pathlib import Path\n\nfrom .core import sha256\n\n\ndef extract(archive, output):\n    required = {\'labels_day_train.csv\', \'labels_day_test.csv\', \'test_submission.csv\'}\n    output = Path(output)\n    with zipfile.ZipFile(archive) as z:\n        members = {}\n        for info in z.infolist():\n            name = Path(info.filename).name\n            if name not in required and name.lower() != \'readme.md\':\n                continue\n            if name in members:\n                raise ValueError(f\'Ambiguous archive: duplicate {name}\')\n            if info.file_size > 20_000_000:\n                raise ValueError(f\'Unexpectedly large label/description file: {name}\')\n            members[name] = info\n        if not required <= members.keys():\n            raise ValueError(f\'Missing archive members: {sorted(required - members.keys())}\')\n        if any((output / name).exists() for name in members):\n            raise ValueError(\'Refuse to overwrite original inputs; use an empty directory\')\n        output.mkdir(parents=True, exist_ok=True)\n        for name, info in members.items():\n            (output / name).write_bytes(z.read(info))\n    manifest = {\'archive_name\': Path(archive).name,\n                \'source_url\': \'https://disk.yandex.ru/d/DiFwlfMOauxjBg\',\n                \'files_sha256\': {name: sha256(output / name) for name in sorted(members)}}\n    (output / \'inputs_manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n    return manifest\n\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--archive\', required=True, type=Path)\n    p.add_argument(\'--out\', required=True, type=Path)\n    a = p.parse_args()\n    print(json.dumps(extract(a.archive, a.out), indent=2))\n', 'ml/forecast/test_pipeline.py': '"""Self-contained synthetic contract/leakage tests; no organizer data required."""\nimport csv\nimport importlib.util\nimport tempfile\nimport unittest\nimport zipfile\nfrom datetime import date, timedelta\nfrom pathlib import Path\n\nfrom ml.forecast.core import grid, load_labels, rounded, score, sha256\nfrom ml.forecast.models import Boosting, Profile\nfrom ml.forecast.run import evaluate_fold, select_dev\nfrom ml.forecast.prepare_inputs import extract\nfrom ml.submit.adapter import export_api, read_sample, validate, write_submission\n\n\nclass PipelineTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.root = Path(self.tmp.name)\n\n    def tearDown(self):\n        self.tmp.cleanup()\n\n    def sample(self):\n        path = self.root / \'sample.csv\'\n        # Deliberately reverse order to detect implicit model sorting of sample rows.\n        keys = grid([str(i) for i in range(1, 11)], date(2025, 11, 1), date(2025, 12, 31))[::-1]\n        with path.open(\'w\', newline=\'\') as f:\n            writer = csv.writer(f, delimiter=\';\')\n            writer.writerow([\'route\', \'date\', \'hour\', \'prediction\'])\n            writer.writerows((r, d, h, \'\') for r, d, h in keys)\n        return path, keys\n\n    def test_exact_sample_roundtrip_api_sums_and_route5(self):\n        sample, keys = self.sample()\n        digest = sha256(sample)\n        preds = {k: (int(k[0]) * 10 + k[2] + .5) for k in keys}\n        out = self.root / \'out.csv\'\n        result = write_submission(sample, preds, out)\n        self.assertEqual(result[\'route5_rows\'], 1464)\n        self.assertEqual(read_sample(out)[1], keys)\n        self.assertEqual(sha256(sample), digest)\n        api = self.root / \'api.csv\'\n        export_api(keys, preds, api, \'synthetic-test\')\n        with api.open() as f:\n            rows = list(csv.DictReader(f))\n        self.assertEqual(sum(int(r[\'pred_validations\']) for r in rows), result[\'prediction_sum\'])\n        self.assertTrue(all(r[\'timestamp_msk\'].endswith(\'+03:00\') for r in rows))\n        daily = {}\n        monthly = {}\n        for row in rows:\n            d = row[\'timestamp_msk\'][:10]\n            daily[d] = daily.get(d, 0) + int(row[\'pred_validations\'])\n        for d, value in daily.items():\n            monthly[d[:7]] = monthly.get(d[:7], 0) + value\n        self.assertEqual(sum(monthly.values()), result[\'prediction_sum\'])\n\n    def test_reject_bad_submission_inputs(self):\n        sample, keys = self.sample()\n        predictions = dict.fromkeys(keys, 1)\n        out = self.root / \'out.csv\'\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, sample)\n        predictions.pop(keys[0])\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, out)\n        predictions[keys[0]] = float(\'nan\')\n        with self.assertRaises(ValueError):\n            write_submission(sample, predictions, out)\n        predictions[keys[0]] = 1\n        write_submission(sample, predictions, out)\n        lines = out.read_text().splitlines()\n        lines[1], lines[2] = lines[2], lines[1]\n        out.write_text(\'\\n\'.join(lines) + \'\\n\')\n        with self.assertRaises(ValueError):\n            validate(sample, out)\n\n    def test_sample_missing_duplicate_wrong_period(self):\n        path, _ = self.sample()\n        lines = path.read_text().splitlines()\n        path.write_text(\'\\n\'.join(lines[:-1]) + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n        path.write_text(\'\\n\'.join(lines[:-1] + [lines[1]]) + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n        path.write_text(\'\\n\'.join(lines).replace(\'2025-11-01\', \'2025-10-31\') + \'\\n\')\n        with self.assertRaises(ValueError):\n            read_sample(path)\n\n    def test_metric_is_pooled_and_includes_missing_hours(self):\n        keys = [(\'1\', date(2025, 9, 1), 7), (\'2\', date(2025, 9, 1), 7)]\n        m = score(keys, {keys[0]: 100}, {keys[0]: 80, keys[1]: 10})\n        self.assertAlmostEqual(m[\'all\'][\'wape_score\'], .7)\n        self.assertIsNone(m[\'route:2\'][\'wape_score\'])\n        with self.assertRaises(ValueError):\n            score(keys, {}, dict.fromkeys(keys, 0))\n        with self.assertRaises(ValueError):\n            score(keys, {keys[0]: 1}, {keys[0]: 1})\n\n    def test_reader_rejects_duplicate_files_and_negative_counts(self):\n        p = self.root / \'labels.csv\'\n        p.write_text(\'route;date;hour;boardings\\n7;2025-01-01;0;2\\n\')\n        with self.assertRaises(ValueError):\n            load_labels([p, p])\n        p.write_text(\'route;date;hour;boardings\\n7;2025-01-01;0;-2\\n\')\n        with self.assertRaises(ValueError):\n            load_labels([p])\n\n    def test_fit_rejects_future_and_predict_rejects_past(self):\n        cutoff = date(2025, 1, 31)\n        history = {(\'7\', date(2025, 1, 1), 7): 20}\n        model = Profile().fit(history, date(2025, 1, 1), cutoff)\n        with self.assertRaises(ValueError):\n            model.predict(list(history))\n        history[\'7\', cutoff + timedelta(days=1), 7] = 1000\n        with self.assertRaises(ValueError):\n            Profile().fit(history, date(2025, 1, 1), cutoff)\n\n    def test_backtest_never_passes_future_labels_to_model(self):\n        values = {(\'7\', date(2025, 1, 1), 7): 10,\n                  (\'7\', date(2025, 2, 1), 7): 2000}\n        class Spy(Profile):\n            def fit(self, history, start, cutoff):\n                assert max(k[1] for k in history) <= cutoff\n                assert max(history.values()) == 10\n                return super().fit(history, start, cutoff)\n        result = evaluate_fold(values, date(2025, 1, 31), date(2025, 2, 1), date(2025, 2, 1), Spy)\n        self.assertEqual(result[\'metrics\'][\'all\'][\'rows\'], 24)\n\n    def test_zero_cold_start_and_missing_training_sensitivity(self):\n        history = {(\'7\', date(2025, 1, 1), 7): 40}\n        keys = [(\'7\', date(2025, 2, 5), 7), (\'5\', date(2025, 2, 5), 7)]\n        zero = Profile().fit(history, date(2025, 1, 1), date(2025, 1, 31)).predict(keys)\n        obs = Profile(missing=\'observed\').fit(history, date(2025, 1, 1), date(2025, 1, 31)).predict(keys)\n        self.assertEqual(zero[keys[1]], 0)\n        self.assertLess(zero[keys[0]], obs[keys[0]])\n\n    def test_rounding_is_finite_nonnegative(self):\n        self.assertEqual([rounded(x) for x in [-2, .5, 1.5, 2.5]], [0, 0, 2, 2])\n        for x in [float(\'inf\'), float(\'nan\'), 10**15]:\n            with self.assertRaises(ValueError):\n                rounded(x)\n\n    def test_selection_does_not_use_sep_oct(self):\n        def entry(n):\n            return {\'metrics\': {\'all\': {\'absolute_error_sum\': n}}}\n        folds = {\'may_jun\': {\'a\': entry(1), \'b\': entry(10)},\n                 \'jul_aug\': {\'a\': entry(5), \'b\': entry(1)},\n                 \'sep_oct\': {\'a\': entry(10000), \'b\': entry(0)}}\n        self.assertEqual(select_dev(folds, [\'a\', \'b\'])[0], \'a\')\n\n    def test_archive_extracts_only_needed_files_and_preserves_originals(self):\n        archive = self.root / \'data.zip\'\n        with zipfile.ZipFile(archive, \'w\') as z:\n            for name in [\'labels_day_train.csv\', \'labels_day_test.csv\', \'test_submission.csv\']:\n                z.writestr(\'nested/\' + name, \'synthetic\')\n            z.writestr(\'../../raw.csv\', \'must not be extracted\')\n            z.writestr(\'README.md\', \'Synthetic description\')\n        dest = self.root / \'inputs\'\n        manifest = extract(archive, dest)\n        self.assertEqual(len(manifest[\'files_sha256\']), 4)\n        self.assertFalse((self.root / \'raw.csv\').exists())\n        self.assertFalse((dest / \'raw.csv\').exists())\n        with self.assertRaises(ValueError):\n            extract(archive, dest)\n\n    @unittest.skipUnless(importlib.util.find_spec(\'numpy\'), \'optional boosting dependency\')\n    def test_recursive_lags_use_predictions_and_fill_intervening_days(self):\n        class FakeRegressor:\n            def __init__(self):\n                self.batches = []\n            def predict(self, features):\n                self.batches.append(features)\n                return [7] * len(features)\n        model = Boosting(recursive=True)\n        model.cutoff = date(2025, 10, 31)\n        model.routes, model.route_codes = [\'7\'], {\'7\': 0}\n        model.history = {(\'7\', model.cutoff, h): 99 for h in range(24)}\n        model.model = FakeRegressor()\n        key = (\'7\', date(2025, 11, 3), 12)\n        self.assertEqual(model.predict([key]), {key: 7})\n        self.assertEqual(len(model.model.batches), 3)\n        self.assertTrue(all(row[8] == 99 for row in model.model.batches[0]))\n        self.assertTrue(all(row[8] == 7 for row in model.model.batches[1]))\n        self.assertTrue(all(row[8] == 7 for row in model.model.batches[2]))\n        self.assertEqual(model.history[\'7\', model.cutoff, 12], 99)\n\n    def test_russian_calendar_2025(self):\n        from ml.forecast.core import is_workday, effective_weekday, calendar_day_type\n        # 2025-11-01 is a working Saturday\n        self.assertTrue(is_workday(date(2025, 11, 1)))\n        self.assertEqual(effective_weekday(date(2025, 11, 1)), 4)\n        self.assertEqual(calendar_day_type(date(2025, 11, 1)), 4)\n\n        # 2025-11-02 is Sunday\n        self.assertFalse(is_workday(date(2025, 11, 2)))\n        self.assertEqual(effective_weekday(date(2025, 11, 2)), 6)\n\n        # 2025-11-03 is non-working day (transferred holiday)\n        self.assertFalse(is_workday(date(2025, 11, 3)))\n        self.assertEqual(effective_weekday(date(2025, 11, 3)), 6)\n\n        # 2025-11-04 is National Unity Day\n        self.assertFalse(is_workday(date(2025, 11, 4)))\n        self.assertEqual(effective_weekday(date(2025, 11, 4)), 6)\n\n        # 2025-11-05 is normal Wednesday\n        self.assertTrue(is_workday(date(2025, 11, 5)))\n        self.assertEqual(effective_weekday(date(2025, 11, 5)), 2)\n\n        # 2025-12-31 is non-working day\n        self.assertFalse(is_workday(date(2025, 12, 31)))\n        self.assertEqual(effective_weekday(date(2025, 12, 31)), 6)\n\n    def test_calendar_profile_and_route50_rule(self):\n        from ml.forecast.models import CalendarProfile\n        history = {}\n        for d in [date(2025, 10, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'50\', d, h] = 50 if d.weekday() < 5 else 0\n                history[\'1\', d, h] = 100 if d.weekday() < 5 else 20\n        cutoff = date(2025, 10, 31)\n        model = CalendarProfile(recent_weight=0.5, window=28, route50_rule=True).fit(history, date(2025, 10, 1), cutoff)\n\n        # 2025-11-01 is working Saturday: route 1 should have weekday traffic (~100)\n        p1_work = model.predict([(\'1\', date(2025, 11, 1), 12)])[(\'1\', date(2025, 11, 1), 12)]\n        self.assertGreater(p1_work, 80)\n\n        # 2025-11-04 is holiday: route 1 should have weekend/Sunday traffic (~20)\n        p1_hol = model.predict([(\'1\', date(2025, 11, 4), 12)])[(\'1\', date(2025, 11, 4), 12)]\n        self.assertLess(p1_hol, 30)\n\n        # Route 50 suspended on weekend before Nov 15\n        p50_susp = model.predict([(\'50\', date(2025, 11, 2), 12)])[(\'50\', date(2025, 11, 2), 12)]\n        self.assertEqual(p50_susp, 0)\n\n    @unittest.skipUnless(importlib.util.find_spec(\'lightgbm\'), \'lightgbm required\')\n    def test_lgbm_direct_fit_and_predict(self):\n        history = {}\n        for d in [date(2025, 9, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'1\', d, h] = 100 + h * 2 if d.weekday() < 5 else 20 + h\n        cutoff = date(2025, 9, 30)\n        model = Boosting(kind=\'lgb\', recursive=False).fit(history, date(2025, 9, 1), cutoff)\n        preds = model.predict([(\'1\', date(2025, 10, 1), 10), (\'1\', date(2025, 10, 4), 10)])\n        self.assertGreater(preds[\'1\', date(2025, 10, 1), 10], preds[\'1\', date(2025, 10, 4), 10])\n\n    @unittest.skipUnless(importlib.util.find_spec(\'catboost\'), \'catboost required\')\n    def test_catboost_direct_fit_and_predict(self):\n        history = {}\n        for d in [date(2025, 9, 1) + timedelta(days=i) for i in range(30)]:\n            for h in range(24):\n                history[\'1\', d, h] = 100 + h * 2 if d.weekday() < 5 else 20 + h\n        cutoff = date(2025, 9, 30)\n        model = Boosting(kind=\'catboost\', recursive=False).fit(history, date(2025, 9, 1), cutoff)\n        preds = model.predict([(\'1\', date(2025, 10, 1), 10)])\n        self.assertGreater(preds[\'1\', date(2025, 10, 1), 10], 50)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n\n', 'ml/forecast/download_yandex.py': '"""Download organizer ZIP through the public Yandex Disk API, without credentials."""\nimport hashlib\nimport json\nimport re\nimport shutil\nimport time\nimport urllib.parse\nimport urllib.request\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nPUBLIC_URL = \'https://disk.yandex.ru/d/DiFwlfMOauxjBg\'\nAPI = \'https://cloud-api.yandex.net/v1/disk/public/resources\'\n\n\ndef api_json(suffix, params, attempts=3):\n    url = API + suffix + \'?\' + urllib.parse.urlencode(params)\n    for attempt in range(attempts):\n        try:\n            with urllib.request.urlopen(url, timeout=30) as response:\n                return json.load(response)\n        except (OSError, ValueError) as exc:\n            if attempt == attempts - 1:\n                raise RuntimeError(\'API Яндекс Диска недоступен. Включите Internet в Kaggle; \'\n                                   \'если уже включён, повторите ячейку позже. \'\n                                   \'Также возможен лимит скачивания публичной ссылки.\') from exc\n            print(f\'API: повторная попытка {attempt + 2}/{attempts}\', flush=True)\n            time.sleep(2 ** (attempt + 1))\n\n\ndef digest(path, algorithm=\'sha256\'):\n    with Path(path).open(\'rb\') as f:\n        return hashlib.file_digest(f, algorithm).hexdigest()\n\n\ndef download(public_url=PUBLIC_URL, cache_dir=\'/tmp/mthack_yandex\', attempts=3):\n    params = {\'public_key\': public_url}\n    meta = api_json(\'\', params)\n    if meta.get(\'type\') == \'dir\':\n        # Organizer link is a public folder containing dataset.zip.\n        params[\'path\'] = \'/dataset.zip\'\n        meta = api_json(\'\', params)\n    if meta.get(\'type\') != \'file\' or not meta.get(\'name\', \'\').lower().endswith(\'.zip\'):\n        raise ValueError(\'Ссылка должна вести на ZIP или папку с dataset.zip\')\n    expected_size = int(meta[\'size\'])\n    if expected_size <= 0:\n        raise ValueError(\'Пустой архив в метаданных Яндекс Диска\')\n    algorithm = \'sha256\' if meta.get(\'sha256\') else \'md5\'\n    expected_hash = meta.get(algorithm)\n    identity = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:16]\n    directory = Path(cache_dir) / identity\n    directory.mkdir(parents=True, exist_ok=True)\n    target, partial = directory / \'dataset.zip\', directory / \'dataset.zip.part\'\n\n    def valid(path):\n        return (path.exists() and path.stat().st_size == expected_size\n                and (not expected_hash or digest(path, algorithm) == expected_hash)\n                and zipfile.is_zipfile(path))\n\n    if valid(target):\n        print(\'Используется ранее проверенный архив:\', target, flush=True)\n        return target\n    print(f\'Архив: {expected_size / 1024**3:.2f} ГиБ. Потоковое скачивание во временную папку.\',\n          flush=True)\n    for attempt in range(attempts):\n        try:\n            offset = partial.stat().st_size if partial.exists() else 0\n            if offset >= expected_size:\n                if valid(partial):\n                    break\n                partial.unlink()\n                offset = 0\n            if shutil.disk_usage(directory).free < expected_size - offset + 64 * 1024**2:\n                raise RuntimeError(\'Недостаточно места для архива во временной папке\')\n            href = api_json(\'/download\', params)[\'href\']\n            if urllib.parse.urlparse(href).scheme != \'https\':\n                raise ValueError(\'API вернул незащищённую ссылку скачивания\')\n            headers = {\'Range\': f\'bytes={offset}-\'} if offset else {}\n            request = urllib.request.Request(href, headers=headers)\n            with urllib.request.urlopen(request, timeout=60) as response:\n                status = response.status\n                if status == 206:\n                    match = re.fullmatch(r\'bytes (\\d+)-(\\d+)/(\\d+)\',\n                                         response.headers.get(\'Content-Range\', \'\'))\n                    if not match or int(match[1]) != offset or int(match[3]) != expected_size:\n                        raise ValueError(\'Некорректный диапазон при докачке\')\n                    mode = \'ab\'\n                elif status == 200:\n                    offset, mode = 0, \'wb\'  # Server ignored Range: safely restart.\n                else:\n                    raise ValueError(f\'Неожиданный HTTP-статус: {status}\')\n                received, logged = offset, offset\n                with partial.open(mode) as f:\n                    while chunk := response.read(4 * 1024**2):\n                        f.write(chunk)\n                        received += len(chunk)\n                        if received > expected_size:\n                            raise ValueError(\'Размер загрузки превысил метаданные\')\n                        if received - logged >= 64 * 1024**2 or received == expected_size:\n                            print(f\'Скачано {received / expected_size:.1%} \'\n                                  f\'({received / 1024**2:.0f}/{expected_size / 1024**2:.0f} МиБ)\',\n                                  flush=True)\n                            logged = received\n            if partial.stat().st_size != expected_size:\n                raise OSError(\'Соединение прервано до конца файла\')\n            if not valid(partial):\n                partial.unlink()\n                raise ValueError(\'Проверка хеша или структуры ZIP не пройдена\')\n            break\n        except (OSError, ValueError) as exc:\n            if attempt == attempts - 1:\n                raise RuntimeError(\'Архив не скачан после повторных попыток. \'\n                                   \'Проверьте Internet и повторите ячейку: частичный файл сохранён \'\n                                   \'для докачки, если сервер поддерживает Range.\') from exc\n            print(f\'Скачивание: повторная попытка {attempt + 2}/{attempts}\', flush=True)\n            time.sleep(2 ** (attempt + 1))\n    partial.replace(target)\n    manifest = {\'source_url\': public_url, \'path\': params.get(\'path\'), \'size\': expected_size,\n                \'sha256\': digest(target), \'fetched_at_utc\': datetime.now(timezone.utc).isoformat(),\n                \'server_hash_algorithm\': algorithm if expected_hash else None,\n                \'server_hash_verified\': bool(expected_hash)}\n    (directory / \'download_manifest.json\').write_text(json.dumps(manifest, indent=2) + \'\\n\')\n    print(\'Архив скачан и проверен:\', target, flush=True)\n    return target\n', 'ml/forecast/test_download_yandex.py': '"""Network-independent tests: streaming, resume, cache and integrity checks."""\nimport hashlib\nimport io\nimport tempfile\nimport unittest\nimport zipfile\nfrom pathlib import Path\nfrom unittest.mock import patch\n\nfrom ml.forecast import download_yandex as yd\n\n\nclass Response(io.BytesIO):\n    def __init__(self, data, status=200, headers=None):\n        super().__init__(data)\n        self.status = status\n        self.headers = headers or {}\n\n\nclass DownloadTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp = tempfile.TemporaryDirectory()\n        self.root = Path(self.tmp.name)\n        buffer = io.BytesIO()\n        with zipfile.ZipFile(buffer, \'w\') as z:\n            z.writestr(\'labels/synthetic.txt\', \'artificial fixture\')\n        self.blob = buffer.getvalue()\n        self.meta = {\'type\': \'file\', \'name\': \'dataset.zip\', \'size\': len(self.blob),\n                     \'sha256\': hashlib.sha256(self.blob).hexdigest()}\n\n    def tearDown(self):\n        self.tmp.cleanup()\n\n    def api(self, suffix, params):\n        if suffix == \'/download\':\n            self.assertEqual(params[\'path\'], \'/dataset.zip\')\n            return {\'href\': \'https://download.example.invalid/data.zip\'}\n        return self.meta if \'path\' in params else {\'type\': \'dir\'}\n\n    def run_download(self, opener, attempts=2):\n        with patch.object(yd, \'api_json\', side_effect=self.api), \\\n             patch.object(yd.urllib.request, \'urlopen\', side_effect=opener), \\\n             patch.object(yd.time, \'sleep\'):\n            return yd.download(cache_dir=self.root, attempts=attempts)\n\n    def test_folder_download_and_verified_cache(self):\n        target = self.run_download(lambda *a, **kw: Response(self.blob))\n        self.assertEqual(target.read_bytes(), self.blob)\n        with patch.object(yd, \'api_json\', side_effect=self.api), \\\n             patch.object(yd.urllib.request, \'urlopen\') as opened:\n            self.assertEqual(yd.download(cache_dir=self.root), target)\n            opened.assert_not_called()\n\n    def test_interruption_resumes_with_range(self):\n        blob = self.blob\n        class Interrupted(Response):\n            def read(self, size=-1):\n                if self.tell():\n                    raise OSError(\'simulated disconnect\')\n                return super().read(40)\n        calls = []\n        def open_response(request, **kwargs):\n            calls.append(request)\n            if len(calls) == 1:\n                return Interrupted(blob)\n            self.assertEqual(request.get_header(\'Range\'), \'bytes=40-\')\n            return Response(blob[40:], 206,\n                            {\'Content-Range\': f\'bytes 40-{len(blob)-1}/{len(blob)}\'})\n        self.assertEqual(self.run_download(open_response).read_bytes(), self.blob)\n        self.assertEqual(len(calls), 2)\n\n    def test_ignored_range_restarts_instead_of_appending(self):\n        class Interrupted(Response):\n            def read(self, size=-1):\n                if self.tell():\n                    raise OSError(\'disconnect\')\n                return super().read(40)\n        responses = iter([Interrupted(self.blob), Response(self.blob)])\n        self.assertEqual(self.run_download(lambda *a, **kw: next(responses)).read_bytes(), self.blob)\n\n    def test_wrong_hash_never_becomes_completed_archive(self):\n        self.meta[\'sha256\'] = \'0\' * 64\n        with self.assertRaises(RuntimeError):\n            self.run_download(lambda *a, **kw: Response(self.blob), attempts=1)\n        self.assertFalse(list(self.root.rglob(\'dataset.zip\')))\n\n    def test_wrong_range_never_becomes_completed_archive(self):\n        with self.assertRaises(RuntimeError):\n            self.run_download(lambda *a, **kw: Response(self.blob, 206,\n                              {\'Content-Range\': f\'bytes 5-{len(self.blob)-1}/{len(self.blob)}\'}),\n                              attempts=1)\n        self.assertFalse(list(self.root.rglob(\'dataset.zip\')))\n\n    def test_api_retries_and_reports_network_problem(self):\n        with patch.object(yd.urllib.request, \'urlopen\', side_effect=OSError(\'timeout\')) as opened, \\\n             patch.object(yd.time, \'sleep\'):\n            with self.assertRaisesRegex(RuntimeError, \'Internet\'):\n                yd.api_json(\'\', {\'public_key\': yd.PUBLIC_URL}, attempts=2)\n            self.assertEqual(opened.call_count, 2)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'ml/submit/adapter.py': '"""Preserve the exact organizer sample keys, ordering and non-target text."""\nfrom __future__ import annotations\n\nimport csv\nfrom datetime import date, datetime\nfrom pathlib import Path\nfrom zoneinfo import ZoneInfo\n\nfrom ml.forecast.core import grid, parse_key, rounded\n\nFIELDS = [\'route\', \'date\', \'hour\', \'prediction\']\nSTART, END = date(2025, 11, 1), date(2025, 12, 31)\n\n\ndef read_sample(path):\n    with Path(path).open(encoding=\'utf-8-sig\', newline=\'\') as f:\n        reader = csv.DictReader(f, delimiter=\';\')\n        if reader.fieldnames != FIELDS:\n            raise ValueError(f\'Unexpected sample schema: {reader.fieldnames}\')\n        rows = list(reader)\n    if any(None in r or any(v is None for v in r.values()) for r in rows):\n        raise ValueError(\'Malformed sample row\')\n    keys = [parse_key(r) for r in rows]\n    routes = sorted({k[0] for k in keys}, key=int)\n    if len(keys) != 14640 or len(set(keys)) != len(keys):\n        raise ValueError(\'Sample must contain exactly 14640 unique keys\')\n    if len(routes) != 10 or \'5\' not in routes:\n        raise ValueError(\'Sample must contain ten routes, including route 5\')\n    if set(keys) != set(grid(routes, START, END)):\n        raise ValueError(\'Sample must cover every hour of November–December 2025\')\n    return rows, keys\n\n\ndef validate(sample_path, output_path):\n    rows, keys = read_sample(sample_path)\n    other, other_keys = read_sample(output_path)\n    if keys != other_keys:\n        raise ValueError(\'Submission changed sample ordering\')\n    for expected, actual in zip(rows, other):\n        if any(expected[k] != actual[k] for k in FIELDS[:-1]):\n            raise ValueError(\'Non-target sample text changed\')\n        value = actual[\'prediction\']\n        if not value.isascii() or not value.isdecimal() or str(int(value)) != value:\n            raise ValueError(\'Submission predictions must be canonical nonnegative integers\')\n        rounded(int(value))\n    return {\'rows\': len(keys), \'routes\': len({k[0] for k in keys}),\n            \'route5_rows\': sum(k[0] == \'5\' for k in keys), \'prediction_sum\':\n            sum(int(row[\'prediction\']) for row in other)}\n\n\ndef write_submission(sample_path, predictions, output_path):\n    if Path(sample_path).resolve() == Path(output_path).resolve():\n        raise ValueError(\'Never overwrite the original sample\')\n    rows, keys = read_sample(sample_path)\n    if set(predictions) != set(keys):\n        raise ValueError(\'Prediction keys must match sample exactly; no missing/extra rows\')\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    values = [rounded(predictions[k]) for k in keys]\n    with output_path.open(\'w\', encoding=\'utf-8\', newline=\'\') as f:\n        writer = csv.DictWriter(f, fieldnames=FIELDS, delimiter=\';\', lineterminator=\'\\n\')\n        writer.writeheader()\n        for row, value in zip(rows, values):\n            writer.writerow({**row, \'prediction\': value})\n    return validate(sample_path, output_path)\n\n\ndef export_api(keys, predictions, output_path, model_version):\n    if len(set(keys)) != len(keys) or set(keys) != set(predictions):\n        raise ValueError(\'API export key mismatch\')\n    with Path(output_path).open(\'w\', encoding=\'utf-8\', newline=\'\') as f:\n        writer = csv.writer(f, lineterminator=\'\\n\')\n        writer.writerow([\'route_id\', \'timestamp_msk\', \'pred_validations\',\n                         \'model_version\', \'source_mode\'])\n        for r, d, h in keys:\n            stamp = datetime(d.year, d.month, d.day, h, tzinfo=ZoneInfo(\'Europe/Moscow\'))\n            writer.writerow([r, stamp.isoformat(), rounded(predictions[r, d, h]),\n                             model_version, \'as_of_2025_10_31\'])\n\n\nif __name__ == \'__main__\':\n    import argparse\n    import json\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--sample\', required=True)\n    parser.add_argument(\'--submission\', required=True)\n    args = parser.parse_args()\n    print(json.dumps(validate(args.sample, args.submission), ensure_ascii=False, indent=2))\n', 'ml/forecast/weather_competition.py': '"""Paired weather ablation. Competition mode uses allowed retrospective weather.\n\nNever consumes future passenger counts. The frozen reference was selected in\nprior Sep-Oct research, so those scores are exploratory, not independent tests.\n"""\nfrom __future__ import annotations\nimport argparse\nimport calendar\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport pickle\nfrom datetime import date,datetime,timedelta,timezone\nfrom pathlib import Path\nimport numpy as np\nfrom .core import grid,load_labels,rounded,score,sha256,effective_weekday,calendar_day_type\nfrom .route_specific import verify_inputs,candidate_factories,RouteEnsemble\nfrom .run import FIRST,FINAL_CUTOFF,FOLDS,dump\nfrom ml.submit.adapter import read_sample,write_submission\n\nVARIABLES=(\'temperature_2m\',\'relative_humidity_2m\',\'precipitation\',\'rain\',\n           \'snowfall\',\'wind_speed_10m\',\'cloud_cover\')\nWEIGHTS=(.25,.5,1.)\n\n\ndef weather_rows(payload):\n    if payload.get(\'timezone\')!=\'Europe/Moscow\' or payload.get(\'utc_offset_seconds\')!=10800:\n        raise ValueError(\'Weather must use Europe/Moscow (+03:00)\')\n    h=payload[\'hourly\'];times=h[\'time\']\n    if len(set(times))!=len(times):raise ValueError(\'Duplicate weather timestamp\')\n    if any(len(h[v])!=len(times) for v in VARIABLES):raise ValueError(\'Weather lengths differ\')\n    result={};previous=None\n    for i,stamp in enumerate(times):\n        t=datetime.fromisoformat(stamp)\n        if t.minute or t.second or (previous and t-previous!=timedelta(hours=1)):\n            raise ValueError(\'Weather hours are not contiguous\')\n        previous=t\n        raw=[h[v][i] for v in VARIABLES]\n        if any(v is None or not math.isfinite(float(v)) for v in raw):\n            raise ValueError(f\'Missing/nonfinite weather at {stamp}; do not silently impute\')\n        raw=list(map(float,raw))\n        # Only preceding/current weather hours in rolling summaries.\n        j=max(0,i-23)\n        raw.extend([float(np.mean(h[\'temperature_2m\'][j:i+1])),\n                    float(np.sum(h[\'precipitation\'][j:i+1])),float(raw[0]<0)])\n        result[t.date(),t.hour]=raw\n    return result\n\n\ndef fit_reference(history,cutoff,choices):\n    factories=candidate_factories(profiles_only=True)\n    names={\'school_calendar\'}|{c[\'candidate\'] for c in choices.values()}\n    models={n:factories[n]().fit(history,FIRST,cutoff) for n in names}\n    return RouteEnsemble(models,choices,cutoff)\n\n\ndef matrix(keys,base,weather,use_weather):\n    rows=[]\n    for k in keys:\n        r,d,h=k\n        row=[int(r),h,d.weekday(),effective_weekday(d),calendar_day_type(d),\n             int(d.month in (7,8)),base[k],math.sin(2*math.pi*h/24),math.cos(2*math.pi*h/24)]\n        if use_weather:\n            if (d,h) not in weather:raise ValueError(f\'Weather missing for {d} {h}\')\n            row.extend(weather[d,h])\n        rows.append(row)\n    return np.asarray(rows,dtype=float)\n\n\ndef causal_residual_training(history,cutoff,choices):\n    """Monthly out-of-time reference predictions; no same-row target encoding."""\n    routes=sorted({k[0] for k in history},key=int)\n    keys=[];base={};blocks=[]\n    lo=date(2025,2,1)\n    while lo<=cutoff:\n        hi=min(cutoff,date(lo.year,lo.month,calendar.monthrange(lo.year,lo.month)[1]))\n        training_cutoff=lo-timedelta(days=1)\n        prefix={k:y for k,y in history.items() if k[1]<=training_cutoff}\n        model=fit_reference(prefix,training_cutoff,choices)\n        batch=grid(routes,lo,hi)\n        base.update(model.predict(batch));keys.extend(batch)\n        blocks.append({\'train_through\':str(training_cutoff),\'predict_from\':str(lo),\'predict_to\':str(hi)})\n        lo=hi+timedelta(days=1)\n    if not keys:raise ValueError(\'At least two months of history required\')\n    scale=np.asarray([max(50,base[k]) for k in keys],dtype=float)\n    target=np.asarray([(history.get(k,0)-base[k])/s for k,s in zip(keys,scale)],dtype=float)\n    return keys,base,scale,target,blocks\n\n\ndef fit_pair(history,cutoff,choices,weather):\n    import lightgbm as lgb\n    keys,base,scale,y,blocks=causal_residual_training(history,cutoff,choices)\n    models={}\n    for name,use_weather in [(\'calendar\',False),(\'weather\',True)]:\n        reg=lgb.LGBMRegressor(objective=\'regression_l1\',n_estimators=220,\n            learning_rate=.025,num_leaves=12,min_child_samples=120,\n            reg_lambda=10.,max_depth=5,n_jobs=2,verbosity=-1,random_state=2025)\n        reg.fit(matrix(keys,base,weather,use_weather),y,sample_weight=scale,\n                categorical_feature=[0,1,2,3,4,5])\n        models[name]=reg\n    return models,blocks\n\n\ndef corrected(keys,base,raw,weight):\n    """Bound changes to 35% of nonzero reference; do not invent service at zero."""\n    out={}\n    for k,v in zip(keys,raw):\n        b=base[k]\n        delta=float(np.clip(float(v)*max(50,b),-.35*b,.35*b))\n        out[k]=rounded(b+weight*delta)\n    return out\n\n\ndef predictions_pair(keys,base,models,weather):\n    preds={\'reference\':base}\n    for name,use_weather in [(\'calendar\',False),(\'weather\',True)]:\n        raw=models[name].predict(matrix(keys,base,weather,use_weather))\n        for weight in WEIGHTS:preds[f\'{name}_{weight}\']=corrected(keys,base,raw,weight)\n    return preds\n\n\ndef select(keys,truth,preds,routes):\n    """Require >=0.5% gain in every validation month, with matched weather control."""\n    months=sorted({k[1].month for k in keys})\n    chosen={};control_chosen={};diagnostics={}\n    for r in routes:\n        groups=[[k for k in keys if k[0]==r and k[1].month==m] for m in months]\n        errors={name:[sum(abs(p[k]-truth.get(k,0)) for k in g) for g in groups]\n                for name,p in preds.items()}\n        wins=lambda name,ref: all(a<=.995*b for a,b in zip(errors[name],errors[ref])) and sum(errors[name])<sum(errors[ref])\n        controls=[\'reference\']+[f\'calendar_{w}\' for w in WEIGHTS if wins(f\'calendar_{w}\',\'reference\')]\n        control=min(controls,key=lambda n:(sum(errors[n]),n));control_chosen[r]=control\n        candidates=list(controls)\n        for w in WEIGHTS:\n            name=f\'weather_{w}\'\n            if wins(name,\'reference\') and wins(name,f\'calendar_{w}\') and wins(name,control):\n                candidates.append(name)\n        chosen[r]=min(candidates,key=lambda n:(sum(errors[n]),n))\n        diagnostics[r]={\'errors_by_month\':errors,\'months\':months,\'calendar_control\':control,\'selected\':chosen[r]}\n    return chosen,control_chosen,diagnostics\n\n\ndef join(keys,preds,choice):\n    return {k:preds[choice.get(k[0],\'reference\')][k] if k[0] in choice else 0 for k in keys}\n\n\nclass WeatherEnsemble:\n    def __init__(self,reference,models,weather,choices):\n        self.reference,self.models,self.weather,self.choices=reference,models,weather,choices\n    def predict(self,keys):\n        base=self.reference.predict(keys)\n        return join(keys,predictions_pair(keys,base,self.models,self.weather),self.choices)\n\n\ndef main(argv=None):\n    p=argparse.ArgumentParser(description=__doc__)\n    for n in [\'train\',\'test\',\'sample\',\'weather\',\'reference\',\'out\']:p.add_argument(\'--\'+n,type=Path,required=True)\n    args=p.parse_args(argv)\n    inputs=verify_inputs({n:getattr(args,n) for n in [\'train\',\'test\',\'sample\']})\n    train,test=load_labels([args.train]),load_labels([args.test])\n    if any(not FIRST<=k[1]<=date(2025,8,31) for k in train):raise ValueError(\'Bad train dates\')\n    if any(not date(2025,9,1)<=k[1]<=FINAL_CUTOFF for k in test):raise ValueError(\'Bad test dates\')\n    values=load_labels([args.train,args.test]);_,sample=read_sample(args.sample)\n    ref=json.loads(args.reference.read_text());choices=ref[\'choices\']\n    routes=sorted({k[0] for k in values},key=int)\n    if set(routes)!=set(choices):raise ValueError(\'Reference route set differs from real labels\')\n    weather=weather_rows(json.loads(args.weather.read_text()))\n    for _,d,h in grid([\'weather\'],FIRST,date(2025,12,31)):\n        if (d,h) not in weather:raise ValueError(\'Weather must cover the entire 2025 calendar year\')\n    args.out.mkdir(parents=True,exist_ok=True)\n    report={\'source_mode\':\'competition\',\'external_information\':\'Retrospective ERA5 weather, allowed by organizer Q&A; not an as-of Oct31 weather forecast.\',\n            \'weather_source\':\'https://open-meteo.com/en/docs/historical-weather-api\',\n            \'weather_source_metadata\':json.loads(Path(__file__).with_name(\'weather_source.json\').read_text()),\n            \'weather_sha256\':sha256(args.weather),\'reference_sha256\':sha256(args.reference),\n            \'reference\':ref,\'inputs\':inputs,\'data_kind\':\'verified_organizer_labels\',\n            \'selection_warning\':ref[\'selection_warning\'],\'folds\':{},\'platform_score\':None,\n            \'weather_features\':list(VARIABLES)+[\'temperature_24h_mean\',\'precipitation_24h_sum\',\'below_freezing\'],\n            \'excluded_variable\':\'snow_depth is all null in downloaded ERA5 response; not used\',\n            \'correction_limit\':.35,\'weights\':list(WEIGHTS),\n            \'fitted_at\':datetime.now(timezone.utc).isoformat(),\n            \'libraries\':{n:importlib.metadata.version(n) for n in [\'numpy\',\'lightgbm\',\'scikit-learn\']},\n            \'code_sha256\':{p.name:sha256(p) for p in sorted(Path(__file__).parent.glob(\'*.py\'))}}\n    for label,cutoff,lo,hi in FOLDS:\n        history={k:y for k,y in values.items() if k[1]<=cutoff}\n        keys=grid(routes,lo,hi);truth={k:y for k,y in values.items() if lo<=k[1]<=hi}\n        base=fit_reference(history,cutoff,choices).predict(keys)\n        models,blocks=fit_pair(history,cutoff,choices,weather)\n        preds=predictions_pair(keys,base,models,weather)\n        report[\'folds\'][label]={\'training_blocks\':blocks,\'variants\':{}}\n        for name,pred in preds.items():\n            m=score(keys,truth,pred);report[\'folds\'][label][\'variants\'][name]=m\n            print(f\'{label} {name}: WAPE-score={m["all"]["wape_score"]:.6f}\',flush=True)\n        if label==\'sep_oct\':\n            sep=[k for k in keys if k[1].month==9];october=[k for k in keys if k[1].month==10]\n            sept,_,_=select(sep,truth,preds,routes)\n            report[\'october_forward_diagnostic\']={\'choices_from_september\':sept,\n                \'metrics\':score(october,{k:truth[k] for k in october if k in truth},join(october,preds,sept))}\n            final_choice,calendar_choice,diag=select(keys,truth,preds,routes)\n            report[\'selection\']={\'by_route\':final_choice,\'calendar_control_by_route\':calendar_choice,\n                \'diagnostics\':diag,\'weather_accepted_routes\':[r for r,n in final_choice.items() if n.startswith(\'weather_\')]}\n            for name,ch in [(\'selected\',final_choice),(\'selected_calendar_control\',calendar_choice)]:\n                report[\'folds\'][label][\'variants\'][name]=score(keys,truth,join(keys,preds,ch))\n        dump(args.out/\'weather_report.json\',report)\n    reference=fit_reference(values,FINAL_CUTOFF,choices)\n    models,_=fit_pair(values,FINAL_CUTOFF,choices,weather)\n    model=WeatherEnsemble(reference,models,weather,final_choice)\n    base=reference.predict(sample);all_final=predictions_pair(sample,base,models,weather)\n    for name,ch in [(\'submission_competition.csv\',final_choice),(\'submission_calendar_control.csv\',calendar_choice)]:\n        report[name]=write_submission(args.sample,join(sample,all_final,ch),args.out/name)\n    # This must reproduce the protected 0.877 baseline byte-for-byte.\n    write_submission(args.sample,base,args.out/\'submission_reference.csv\')\n    if sha256(args.out/\'submission_reference.csv\')!=ref[\'submission_sha256\']:\n        raise RuntimeError(\'Frozen reference does not reproduce original CSV; stop and investigate\')\n    (args.out/\'model.local.pkl\').write_bytes(pickle.dumps(model,protocol=5))\n    report[\'outputs_sha256\']={n:sha256(args.out/n) for n in [\'submission_competition.csv\',\'submission_calendar_control.csv\',\'submission_reference.csv\',\'model.local.pkl\']}\n    final_predictions=join(sample,all_final,final_choice)\n    report[\'changed_rows_vs_reference\']=sum(final_predictions[k]!=base[k] for k in sample)\n    dump(args.out/\'weather_report.json\',report)\n    print(\'Weather accepted on routes:\',report[\'selection\'][\'weather_accepted_routes\'],flush=True)\n    print(\'Reference reproduced; 14640 rows exported. Competition score unknown.\',flush=True)\n    return report\n\nif __name__==\'__main__\':\n    from ml.forecast.weather_competition import main as run_main\n    run_main()\n', 'ml/forecast/reference_route_model.json': '{\n  "choices": {\n    "1": {\n      "candidate": "school_local",\n      "weight": 1.0,\n      "month_errors": [\n        41280,\n        55185\n      ]\n    },\n    "7": {\n      "candidate": "school_recent50",\n      "weight": 1.0,\n      "month_errors": [\n        83375,\n        78862\n      ]\n    },\n    "11": {\n      "candidate": "school_calendar",\n      "weight": 1.0,\n      "month_errors": [\n        90010,\n        92860\n      ]\n    },\n    "12": {\n      "candidate": "school_local",\n      "weight": 0.5,\n      "month_errors": [\n        87764,\n        84047\n      ]\n    },\n    "17": {\n      "candidate": "school_local",\n      "weight": 0.5,\n      "month_errors": [\n        86781,\n        120431\n      ]\n    },\n    "25": {\n      "candidate": "school_calendar",\n      "weight": 1.0,\n      "month_errors": [\n        29571,\n        49803\n      ]\n    },\n    "26": {\n      "candidate": "school_calendar",\n      "weight": 1.0,\n      "month_errors": [\n        43025,\n        59428\n      ]\n    },\n    "28": {\n      "candidate": "school_local",\n      "weight": 0.5,\n      "month_errors": [\n        42733,\n        39846\n      ]\n    },\n    "50": {\n      "candidate": "school_local",\n      "weight": 1.0,\n      "month_errors": [\n        133254,\n        136981\n      ]\n    }\n  },\n  "model_version": "ds1-route-63b89667a69c",\n  "validation_score": 0.8937443240536405,\n  "submission_sha256": "6efbbe7dc7dee64245b99573a7e92a1452a3816eb6e7cc4e2fc56e482663290f",\n  "selection_warning": "Choices were selected on Sep-Oct in prior research; all subsequent comparisons on that period remain exploratory."\n}\n', 'ml/forecast/test_weather_competition.py': '"""Small in-memory software tests; no persistent generated datasets."""\nimport unittest\nfrom datetime import date,timedelta\nfrom unittest.mock import patch\nimport numpy as np\nfrom ml.forecast.weather_competition import weather_rows,VARIABLES,corrected,select,join,causal_residual_training\n\nclass WeatherTests(unittest.TestCase):\n    def payload(self):\n        return {\'timezone\':\'Europe/Moscow\',\'utc_offset_seconds\':10800,\'hourly\':{\n            \'time\':[\'2025-01-01T00:00\',\'2025-01-01T01:00\'],\n            **{v:[1.,2.] for v in VARIABLES}}}\n\n    def test_missing_weather_is_not_silently_imputed(self):\n        p=self.payload();p[\'hourly\'][\'temperature_2m\'][1]=None\n        with self.assertRaisesRegex(ValueError,\'Missing\'):weather_rows(p)\n\n    def test_weather_time_alignment_and_rolling_values(self):\n        p=self.payload();r=weather_rows(p)\n        self.assertEqual(r[date(2025,1,1),1][-3:],[1.5,3.,0.])\n        p[\'hourly\'][\'time\'][1]=\'2025-01-01T02:00\'\n        with self.assertRaisesRegex(ValueError,\'contiguous\'):weather_rows(p)\n\n    def test_weather_must_beat_matched_control_in_both_months(self):\n        keys=[(\'1\',date(2025,m,1),12) for m in (9,10)]\n        truth=dict.fromkeys(keys,100)\n        preds={\'reference\':dict.fromkeys(keys,80)}\n        for w in (.25,.5,1.):\n            preds[f\'calendar_{w}\']=dict.fromkeys(keys,95)\n            preds[f\'weather_{w}\']={keys[0]:100,keys[1]:90}\n        chosen,control,_=select(keys,truth,preds,[\'1\'])\n        self.assertTrue(chosen[\'1\'].startswith(\'calendar\'))\n        preds[\'weather_0.5\']=dict.fromkeys(keys,99)\n        chosen,_,_=select(keys,truth,preds,[\'1\'])\n        self.assertEqual(chosen[\'1\'],\'weather_0.5\')\n\n    def test_corrections_are_bounded_and_route5_stays_zero(self):\n        keys=[(\'1\',date(2025,11,1),12),(\'5\',date(2025,11,1),12)]\n        base={keys[0]:100,keys[1]:0}\n        self.assertEqual(corrected(keys,base,[99,99],1),{keys[0]:135,keys[1]:0})\n        self.assertEqual(corrected(keys,base,[-99,-99],1),{keys[0]:65,keys[1]:0})\n\n    def test_residual_training_has_no_current_block_labels(self):\n        start=date(2025,1,1);end=date(2025,3,31)\n        history={(\'1\',start+timedelta(days=i),12):i+1 for i in range((end-start).days+1)}\n        seen=[]\n        def factory(prefix,cutoff,choices):\n            self.assertLessEqual(max(k[1] for k in prefix),cutoff)\n            class Reference:\n                def predict(_,keys):\n                    self.assertGreater(min(k[1] for k in keys),cutoff)\n                    seen.append((cutoff,min(k[1] for k in keys)))\n                    return dict.fromkeys(keys,10)\n            return Reference()\n        with patch(\'ml.forecast.weather_competition.fit_reference\',side_effect=factory):\n            keys,base,scale,target,blocks=causal_residual_training(history,end,{})\n        self.assertEqual(len(seen),2)\n        self.assertEqual(blocks[0][\'train_through\'],\'2025-01-31\')\n        self.assertEqual(keys[0][1],date(2025,2,1))\n\nif __name__==\'__main__\':unittest.main()\n', 'ml/forecast/weather_source.json': '{\n  "url": "https://archive-api.open-meteo.com/v1/archive?latitude=55.75&longitude=37.62&start_date=2025-01-01&end_date=2025-12-31&hourly=temperature_2m%2Crelative_humidity_2m%2Cprecipitation%2Crain%2Csnowfall%2Csnow_depth%2Cwind_speed_10m%2Ccloud_cover&timezone=Europe%2FMoscow&models=era5",\n  "source_mode": "competition",\n  "description": "Reanalysis, not available as a full future year at forecast cutoff",\n  "retrieved_at": "2026-09-26T19:25:02Z",\n  "attribution": "Weather data by Open-Meteo; ERA5 reanalysis by ECMWF/Copernicus."\n}\n', 'ml/forecast/network_regimes.py': '"""Route service states, sourced externally, with target-only causal profiles.\n\nCompetition mode deliberately allows retrospective transport notices. Dates in\nthese notices describe service, not a claim that recorded validations are zero.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom collections import defaultdict\nfrom datetime import date\nfrom pathlib import Path\nfrom statistics import median\n\nfrom .core import days, effective_weekday, rounded\n\nNORMAL, JULY_SHORT, WEST_AUG, WEST_SEP, BOTH_ENDS, CANCELLED = range(6)\nSTATE_NAMES = (\'normal\', \'sokolniki_july\', \'novoslobodskaya_august\',\n               \'kalanchyovskaya\', \'kalanchyovskaya_and_bogorodskoe\', \'cancelled\')\nEVENTS = json.loads(Path(__file__).with_name(\'network_sources.json\').read_text())\n\n\ndef state(route, day, *, mode=\'competition\', as_of=None):\n    """A publication cutoff applies only to as-of mode; ends have own notices.\n\n    Repair weekends are literal Saturday/Sunday, independent of the demand\n    calendar. Nov 1/3/4 are uncertain and explicitly listed in source metadata.\n    """\n    if mode not in (\'competition\', \'as_of\') or (mode == \'as_of\' and as_of is None):\n        raise ValueError(\'Choose competition or as_of with a publication cutoff\')\n    route = str(route)\n    known = lambda stamp: mode == \'competition\' or date.fromisoformat(stamp) <= as_of\n    if route not in (\'7\', \'50\'):\n        return NORMAL\n    if route == \'7\' and known(\'2025-07-02\') and day >= date(2025, 7, 10):\n        if not (known(\'2025-08-13\') and day >= date(2025, 8, 13)):\n            return JULY_SHORT\n    if day.weekday() < 5:\n        return NORMAL\n    if known(\'2025-11-15\') and day >= date(2025, 11, 15):\n        return NORMAL\n    if known(\'2025-09-05\') and day >= date(2025, 9, 6):\n        if route == \'50\':\n            return CANCELLED\n        east = known(\'2025-09-19\') and day >= date(2025, 9, 20)\n        restored = known(\'2025-10-04\') and day >= date(2025, 10, 4)\n        return BOTH_ENDS if east and not restored else WEST_SEP\n    if route == \'7\' and known(\'2025-08-15\') and day >= date(2025, 8, 16):\n        return WEST_AUG\n    return NORMAL\n\n\nclass RegimeProfile:\n    """25% same-service hourly median + 75% frozen route reference.\n\n    Distinct shortened routes never borrow a July reduction coefficient.\n    Unseen or sparse states leave the reference untouched. Only announced\n    route-50 cancellations use a zero prior, or a learned residual-service\n    median once three same-weekday or four pooled weekend days are available.\n    Positive labels survive.\n    """\n    def __init__(self, weight=.25, mode=\'competition\', as_of=None):\n        if not 0 <= weight <= .5:\n            raise ValueError(\'Regime pooling weight must be in [0, .5]\')\n        self.weight, self.mode, self.as_of = weight, mode, as_of\n\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'History is empty or crosses cutoff\')\n        self.cutoff = cutoff\n        self.routes = {k[0] for k in history}\n        groups = defaultdict(list)\n        for r in (\'7\', \'50\'):\n            if r not in self.routes:\n                continue\n            for d in days(start, cutoff):\n                s = state(r, d, mode=self.mode, as_of=self.as_of)\n                # Weekdays remain the established route model. Holiday demand\n                # is not assumed to imply a repair closure.\n                if d.weekday() < 5:\n                    continue\n                for h in range(24):\n                    groups[r, s, d.weekday(), h].append(history.get((r, d, h), 0))\n                    groups[r, s, -1, h].append(history.get((r, d, h), 0))\n        self.support = {k: len(v) for k, v in groups.items()}\n        self.profiles = {k: float(median(v)) for k, v in groups.items()\n                         if len(v) >= (4 if k[2] == -1 else 3)}\n        return self\n\n    def lookup(self, route, service, weekday, hour):\n        return self.profiles.get((route, service, weekday, hour),\n                                 self.profiles.get((route, service, -1, hour)))\n\n    def apply(self, keys, base, *, active=(\'7\', \'50\'), zero_only=False):\n        if any(d <= self.cutoff for _, d, _ in keys):\n            raise ValueError(\'Predictions must follow training cutoff\')\n        out = {}\n        for k in keys:\n            r, d, h = k\n            b = base[k]\n            if r not in self.routes:\n                out[k] = 0\n                continue\n            s = state(r, d, mode=self.mode, as_of=self.as_of)\n            local = self.lookup(r, s, d.weekday(), h)\n            if r in active and s == CANCELLED:\n                # This is a prediction, never an overwrite of known truth.\n                out[k] = 0 if zero_only or local is None else rounded(local)\n            elif r in active and d.weekday() >= 5 and effective_weekday(d) == d.weekday() and local is not None:\n                # <=50% change in the profile before 25% pooling; protects\n                # restoration from the rejected DS2 history-exclusion rule.\n                local = min(1.5 * b, max(.5 * b, local))\n                out[k] = rounded((1 - self.weight) * b + self.weight * local)\n            else:\n                out[k] = b\n        return out\n\n    def coverage(self, keys):\n        counts = defaultdict(lambda: {\'rows\': 0, \'supported_rows\': 0})\n        for r, d, h in keys:\n            if r in (\'7\', \'50\'):\n                s = state(r, d, mode=self.mode, as_of=self.as_of)\n                item = counts[f\'{r}:{STATE_NAMES[s]}\']\n                item[\'rows\'] += 1\n                item[\'supported_rows\'] += self.lookup(r, s, d.weekday(), h) is not None\n        return dict(counts)\n', 'ml/forecast/network_competition.py': '"""Fixed-horizon competition comparison: service regimes x actual weather.\n\nOrganizer labels are verified by SHA256. No generated dataset can pass the\nentry point. New scores are produced only by running on those real inputs.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport calendar\nimport importlib.metadata\nimport json\nimport pickle\nfrom collections import defaultdict\nfrom datetime import date, datetime, timedelta, timezone\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .core import grid, load_labels, score, sha256, peak_hours\nfrom .network_regimes import RegimeProfile, state, STATE_NAMES, EVENTS\nfrom .route_specific import verify_inputs\nfrom .run import FIRST, FINAL_CUTOFF, FOLDS, dump\nfrom .weather_competition import (fit_reference, matrix, weather_rows,\n                                 corrected, WEIGHTS, select as select_weather)\nfrom ml.submit.adapter import read_sample, write_submission\n\nVERIFIED_REFERENCE_CSV_SHA256 = \'6efbbe7dc7dee64245b99573a7e92a1452a3816eb6e7cc4e2fc56e482663290f\'\n\n\ndef verify_competition_inputs(paths, sample_from_reference=False):\n    if not sample_from_reference:\n        return verify_inputs(paths)\n    inputs = verify_inputs({n: paths[n] for n in (\'train\', \'test\')})\n    digest = sha256(paths[\'sample\'])\n    if digest != VERIFIED_REFERENCE_CSV_SHA256:\n        raise ValueError(\'Reference template must be the exact protected 0.877 submission\')\n    # The adapter also validates the full original key grid and schema. The\n    # previous prediction column is ignored by all training and export logic.\n    read_sample(paths[\'sample\'])\n    inputs[\'sample\'] = {\'name\': paths[\'sample\'].name, \'sha256\': digest,\n        \'role\': \'protected_previous_submission_as_key_template\',\n        \'note\': \'Original organizer sample not supplied locally; keys/order come from verified previous export, old predictions ignored.\'}\n    return inputs\n\n\ndef features(keys, base, weather, use_weather, network):\n    x = matrix(keys, base, weather, use_weather)\n    if network:\n        # Same service information is present in both matched weather arms.\n        x = np.column_stack((x, [state(r, d) for r, d, _ in keys]))\n    return x\n\n\nclass HistoryBank:\n    """Cache predictions for monthly blocks, fitted strictly before each block."""\n    def __init__(self, values, choices):\n        self.values, self.choices = values, choices\n        self.routes = sorted({k[0] for k in values}, key=int)\n        self.fits, self.blocks = {}, {}\n\n    def fitted(self, cutoff):\n        if cutoff not in self.fits:\n            history = {k: v for k, v in self.values.items() if FIRST <= k[1] <= cutoff}\n            self.fits[cutoff] = (fit_reference(history, cutoff, self.choices),\n                                RegimeProfile().fit(history, FIRST, cutoff))\n        return self.fits[cutoff]\n\n    def predict_bases(self, cutoff, keys):\n        ref, net = self.fitted(cutoff)\n        plain = ref.predict(keys)\n        return {\'plain\': plain, \'network\': net.apply(keys, plain)}\n\n    def training(self, cutoff):\n        keys, base = [], {\'plain\': {}, \'network\': {}}\n        audit = []\n        lo = date(2025, 2, 1)\n        while lo <= cutoff:\n            hi = min(cutoff, date(lo.year, lo.month, calendar.monthrange(lo.year, lo.month)[1]))\n            origin = lo - timedelta(days=1)\n            cache_key = (origin, hi)\n            if cache_key not in self.blocks:\n                batch = grid(self.routes, lo, hi)\n                self.blocks[cache_key] = (batch, self.predict_bases(origin, batch))\n            batch, predictions = self.blocks[cache_key]\n            keys.extend(batch)\n            for name in base:\n                base[name].update(predictions[name])\n            audit.append({\'train_through\': str(origin), \'predict_from\': str(lo), \'predict_to\': str(hi)})\n            lo = hi + timedelta(days=1)\n        if not keys:\n            raise ValueError(\'Need at least two months of history\')\n        # In this function the bank may contain future evaluation labels; only\n        # targets ending at cutoff are read. Each feature profile is earlier.\n        y = np.asarray([self.values.get(k, 0) for k in keys], dtype=float)\n        return keys, base, y, audit\n\n\ndef fit_models(bank, cutoff, weather):\n    import lightgbm as lgb\n    keys, bases, truth, audit = bank.training(cutoff)\n    models = {}\n    for group in (\'plain\', \'network\'):\n        base = bases[group]\n        scale = np.asarray([max(50, base[k]) for k in keys], dtype=float)\n        target = (truth - np.asarray([base[k] for k in keys])) / scale\n        models[group] = {}\n        for head, use_weather in ((\'calendar\', False), (\'weather\', True)):\n            x = features(keys, base, weather, use_weather, group == \'network\')\n            categorical = [0, 1, 2, 3, 4, 5] + ([x.shape[1]-1] if group == \'network\' else [])\n            reg = lgb.LGBMRegressor(objective=\'regression_l1\', n_estimators=220,\n                learning_rate=.025, num_leaves=12, min_child_samples=120,\n                reg_lambda=10., max_depth=5, n_jobs=2, verbosity=-1, random_state=2025)\n            reg.fit(x, target, sample_weight=scale, categorical_feature=categorical)\n            models[group][head] = reg\n    return models, audit\n\n\ndef predict_models(keys, bases, models, weather):\n    out = {}\n    for group, base in bases.items():\n        out[group] = {\'reference\': base}\n        for head, use_weather in ((\'calendar\', False), (\'weather\', True)):\n            raw = models[group][head].predict(features(keys, base, weather, use_weather, group == \'network\'))\n            for weight in WEIGHTS:\n                out[group][f\'{head}_{weight}\'] = corrected(keys, base, raw, weight)\n    return out\n\n\ndef metrics(keys, truth, predictions):\n    return score(keys, {k: truth[k] for k in keys if k in truth},\n                 {k: predictions[k] for k in keys})\n\n\ndef route_errors(keys, truth, prediction, route):\n    return sum(abs(prediction[k] - truth.get(k, 0)) for k in keys if k[0] == route)\n\n\ndef select_groups(keys, truth, preds, october_keys, october_truth, october_preds, routes):\n    """Finite, preregistered gate; no tuning of a state-specific effect size.\n\n    Route 50: >=0.5% gain in both long validation months. Route 7: the new\n    September endpoint is first learnable in the October refit, requiring\n    >=0.5% gain there and <=0.5% deterioration in each long validation month.\n    Both must also pass the October refit non-deterioration check.\n    """\n    chosen, audit = {}, {}\n    for r in routes:\n        monthly = [[k for k in keys if k[1].month == m] for m in (9, 10)]\n        errors = {g: [route_errors(x, truth, preds[g][\'reference\'], r) for x in monthly]\n                  for g in (\'plain\', \'network\')}\n        oct_errors = {g: route_errors(october_keys, october_truth, october_preds[g][\'reference\'], r)\n                      for g in (\'plain\', \'network\')}\n        wins = all(a <= .995*b for a, b in zip(errors[\'network\'], errors[\'plain\'])) and sum(errors[\'network\']) < sum(errors[\'plain\'])\n        learns_new_state = (r == \'7\' and oct_errors[\'network\'] < .995*oct_errors[\'plain\']\n                            and all(a <= 1.005*b for a, b in zip(errors[\'network\'], errors[\'plain\'])))\n        accepted = (r in (\'7\', \'50\') and (wins or learns_new_state)\n                    and oct_errors[\'network\'] <= 1.005*oct_errors[\'plain\'])\n        chosen[r] = \'network\' if accepted else \'plain\'\n        audit[r] = {\'long_month_errors\': errors, \'october_refit_errors\': oct_errors,\n                    \'passes_long_month_gate\': wins, \'passes_new_state_gate\': learns_new_state,\n                    \'selected\': chosen[r]}\n    return chosen, audit\n\n\ndef choose_heads(keys, truth, preds, groups, routes):\n    chosen, controls, audit = {}, {}, {}\n    for group in (\'plain\', \'network\'):\n        heads, no_weather, details = select_weather(keys, truth, preds[group], routes)\n        for r in routes:\n            if groups[r] == group:\n                chosen[r] = {\'group\': group, \'head\': heads[r]}\n                controls[r] = {\'group\': group, \'head\': no_weather[r]}\n                audit[r] = details[r]\n    return chosen, controls, audit\n\n\ndef assemble(keys, preds, choices):\n    return {k: preds[choices[k[0]][\'group\']][choices[k[0]][\'head\']][k]\n            if k[0] in choices else 0 for k in keys}\n\n\ndef slice_diagnostics(keys, truth, predictions, origin, history):\n    buckets = defaultdict(list)\n    peaks = peak_hours(history, sorted({k[0] for k in history}, key=int))\n    for k in keys:\n        r, d, h = k\n        lead = (d-origin).days\n        buckets[\'lead:\' + (\'01_14\' if lead <= 14 else \'15_30\' if lead <= 30 else \'31_61\')].append(k)\n        buckets[\'weekend\' if d.weekday() >= 5 else \'weekday\'].append(k)\n        if h in peaks.get(r, ()):\n            buckets[\'historical_peak_hours\'].append(k)\n        if r in (\'7\', \'50\'):\n            buckets[f\'route:{r}:state:{STATE_NAMES[state(r, d)]}\'].append(k)\n    out = {}\n    for name, subset in sorted(buckets.items()):\n        actual = sum(truth.get(k, 0) for k in subset)\n        error = sum(abs(predictions[k] - truth.get(k, 0)) for k in subset)\n        out[name] = {\'rows\': len(subset), \'actual_sum\': actual, \'absolute_error_sum\': error,\n                     \'wape_score\': max(0, 1-error/actual) if actual else None}\n    return out\n\n\nclass NetworkWeatherEnsemble:\n    def __init__(self, reference, regimes, models, weather, choices):\n        self.reference, self.regimes = reference, regimes\n        self.models, self.weather, self.choices = models, weather, choices\n\n    def predict(self, keys):\n        plain = self.reference.predict(keys)\n        bases = {\'plain\': plain, \'network\': self.regimes.apply(keys, plain)}\n        return assemble(keys, predict_models(keys, bases, self.models, self.weather), self.choices)\n\n\n\nclass HolidayServiceEnsemble:\n    """Reproduce the user-reported best holiday-service submission."""\n    def __init__(self, baseline, reference, regimes):\n        self.baseline, self.reference, self.regimes = baseline, reference, regimes\n\n    def predict(self, keys):\n        keys = list(keys)\n        result = self.baseline.predict(keys)\n        normal = [k for k in keys if k[0] in (\'7\', \'50\') and k[1] == date(2025, 11, 1)]\n        result.update(self.reference.predict(normal))\n        replacements = {k: (k[0], date(2025, 11, 2), k[2]) for k in keys\n                        if k[0] in (\'7\', \'50\') and k[1] in (date(2025, 11, 3), date(2025, 11, 4))}\n        proxies = list(dict.fromkeys(replacements.values()))\n        if proxies:\n            restricted = self.regimes.apply(proxies, self.reference.predict(proxies))\n            result.update({k: restricted[v] for k, v in replacements.items()})\n        return result\n\n\ndef main(argv=None):\n    p = argparse.ArgumentParser(description=__doc__)\n    for name in (\'train\', \'test\', \'sample\', \'weather\', \'reference\', \'out\'):\n        p.add_argument(\'--\'+name, type=Path, required=True)\n    p.add_argument(\'--hourly-experts\', action=\'store_true\',\n                   help=\'Compare separate daily-volume/hour-shape experts for 17/11/12\')\n    p.add_argument(\'--sample-from-reference\', action=\'store_true\',\n                   help=\'Use only the exact SHA-verified prior 0.877 CSV as a key/order template\')\n    args = p.parse_args(argv)\n    if args.hourly_experts:\n        from . import hourly_experts as hourly\n    inputs = verify_competition_inputs({n: getattr(args, n) for n in (\'train\', \'test\', \'sample\')},\n                                      args.sample_from_reference)\n    values = load_labels([args.train, args.test])\n    if any(not FIRST <= k[1] <= FINAL_CUTOFF for k in values):\n        raise ValueError(\'Labels cross the allowed training interval\')\n    _, sample = read_sample(args.sample)\n    ref = json.loads(args.reference.read_text())\n    routes = sorted({k[0] for k in values}, key=int)\n    if set(routes) != set(ref[\'choices\']):\n        raise ValueError(\'Reference route set differs from labels\')\n    weather = weather_rows(json.loads(args.weather.read_text()))\n    if any((d, h) not in weather for _, d, h in grid([\'weather\'], FIRST, date(2025, 12, 31))):\n        raise ValueError(\'Weather must cover all 2025 hours\')\n    args.out.mkdir(parents=True, exist_ok=True)\n    report = {\'source_mode\': \'competition\', \'data_kind\': \'verified_organizer_labels\',\n        \'inputs\': inputs, \'reference\': ref, \'network_sources\': EVENTS,\n        \'research_audit\': json.loads(Path(__file__).with_name(\'research_audit.json\').read_text()),\n        \'weather_source\': json.loads(Path(__file__).with_name(\'weather_source.json\').read_text()),\n        \'weather_sha256\': sha256(args.weather), \'platform_score\': None,\n        \'selection_warning\': \'Exploratory: frozen route choices and new selection gates have seen Sep-Oct. October refit covers 31 days, not an independent 61-day test.\',\n        \'fitted_at\': datetime.now(timezone.utc).isoformat(),\n        \'libraries\': {n: importlib.metadata.version(n) for n in (\'numpy\', \'lightgbm\', \'scikit-learn\', \'scipy\')},\n        \'code_sha256\': {f.name: sha256(f) for f in sorted(Path(__file__).parent.glob(\'*.py\'))},\n        \'folds\': {}, \'regime_weight\': .25, \'residual_correction_limit\': .35}\n    bank = HistoryBank(values, ref[\'choices\'])\n    experiments = list(FOLDS) + [(\'october_refit_31d\', date(2025, 9, 30), date(2025, 10, 1), date(2025, 10, 31))]\n    kept, fitted_hourly = {}, {}\n    for label, cutoff, lo, hi in experiments:\n        print(f\'Fitting {label}; passenger labels through {cutoff}\', flush=True)\n        keys = grid(routes, lo, hi)\n        bases = bank.predict_bases(cutoff, keys)\n        models, training_audit = fit_models(bank, cutoff, weather)\n        preds = predict_models(keys, bases, models, weather)\n        variants = {f\'{g}/{n}\': metrics(keys, values, v) for g, heads in preds.items() for n, v in heads.items()}\n        net = bank.fitted(cutoff)[1]\n        for label_part, active in ((\'only_50\', (\'50\',)), (\'only_7\', (\'7\',))):\n            variants[label_part] = metrics(keys, values, net.apply(keys, bases[\'plain\'], active=active))\n        variants[\'50_zero_prior\'] = metrics(keys, values, net.apply(keys, bases[\'plain\'], active=(\'50\',), zero_only=True))\n        history = {k: y for k, y in values.items() if k[1] <= cutoff}\n        report[\'folds\'][label] = {\'cutoff\': str(cutoff), \'from\': str(lo), \'to\': str(hi),\n            \'days\': (hi-lo).days+1, \'training_blocks\': training_audit,\n            \'state_coverage\': net.coverage(keys), \'variants\': variants,\n            \'regime_slices\': slice_diagnostics(keys, values, bases[\'network\'], cutoff, history)}\n        if args.hourly_experts:\n            fitted_hourly[label] = hourly.HourlyExperts().fit(history, FIRST, cutoff)\n            candidate_predictions = fitted_hourly[label].variants(keys, bases[\'plain\'])\n            report[\'folds\'][label][\'hourly_vs_frozen_reference\'] = {\n                name: metrics(keys, values, pred) for name, pred in candidate_predictions.items()}\n        for n in (\'plain/reference\', \'network/reference\', \'only_50\', \'only_7\'):\n            print(f\'{label} {n}: {variants[n]["all"]["wape_score"]:.6f}\', flush=True)\n        if label in (\'sep_oct\', \'october_refit_31d\'):\n            kept[label] = keys, preds\n        dump(args.out/\'network_weather_report.json\', report)\n\n    keys, preds = kept[\'sep_oct\']\n    oct_keys, oct_preds = kept[\'october_refit_31d\']\n    groups, group_audit = select_groups(keys, values, preds, oct_keys, values, oct_preds, routes)\n    choice, control, head_audit = choose_heads(keys, values, preds, groups, routes)\n    report[\'selection\'] = {\'by_route\': choice, \'without_weather_by_route\': control,\n        \'network_gate\': group_audit, \'weather_gate\': head_audit,\n        \'network_accepted_routes\': [r for r in routes if groups[r] == \'network\'],\n        \'weather_accepted_routes\': [r for r in routes if choice[r][\'head\'].startswith(\'weather_\')]}\n    for label, (ks, ps) in kept.items():\n        report[\'folds\'][label][\'variants\'][\'selected\'] = metrics(ks, values, assemble(ks, ps, choice))\n        report[\'folds\'][label][\'variants\'][\'selected_no_weather\'] = metrics(ks, values, assemble(ks, ps, control))\n    # October selection using September alone and a fixed Aug31 model remains\n    # a separate forward diagnostic, without the October refit\'s label access.\n    sep_keys = [k for k in keys if k[1].month == 9]\n    fixed_oct_keys = [k for k in keys if k[1].month == 10]\n    sep_groups = {r: (\'network\' if r in (\'7\', \'50\') and\n        route_errors(sep_keys, values, preds[\'network\'][\'reference\'], r) <\n        .995*route_errors(sep_keys, values, preds[\'plain\'][\'reference\'], r) else \'plain\') for r in routes}\n    sep_choice, _, _ = choose_heads(sep_keys, values, preds, sep_groups, routes)\n    report[\'october_forward_from_september\'] = {\'choices\': sep_choice,\n        \'metrics\': metrics(fixed_oct_keys, values, assemble(fixed_oct_keys, preds, sep_choice)),\n        \'warning\': \'Frozen reference was previously selected using Sep-Oct; only additional choices use September alone.\'}\n    if args.hourly_experts:\n        hourly_predictions = {}\n        for label, (ks, ps) in kept.items():\n            incumbent = assemble(ks, ps, choice)\n            hourly_predictions[label] = fitted_hourly[label].variants(ks, incumbent)\n        hourly_choice, hourly_audit = hourly.select(keys, values, hourly_predictions[\'sep_oct\'],\n            refit_keys=oct_keys, refit_truth=values, refit_variants=hourly_predictions[\'october_refit_31d\'])\n        report[\'hourly_experts\'] = {\n            \'routes\': list(hourly.TARGET_ROUTES), \'choices\': hourly_choice,\n            \'variants\': {k: {\'daily_weight\': v[0], \'shape_weight\': v[1]} for k, v in hourly.VARIANTS.items()},\n            \'selection\': hourly_audit, \'minimum_comparable_days\': 4,\n            \'recent_comparable_weekdays\': 8,\n            \'warning\': \'Selection is exploratory on repeatedly used Sep-Oct. A refit-October guard is 31 days, not an independent long-horizon test.\'}\n        for label, (ks, _) in kept.items():\n            hp = hourly_predictions[label]\n            selected = hourly.combine(ks, hp, hourly_choice)\n            report[\'folds\'][label][\'hourly_on_network_weather\'] = {\n                name: metrics(ks, values, pred) for name, pred in {**hp, \'selected\': selected}.items()}\n            report[\'folds\'][label][\'hourly_diagnostics\'] = {\n                \'before\': hourly.diagnostics(ks, values, hp[\'reference\']),\n                \'after\': hourly.diagnostics(ks, values, selected)}\n        # Extra forward check uses only September for both the network/weather\n        # and hourly-head choices. The original frozen reference remains exploratory.\n        forward_base = assemble(keys, preds, sep_choice)\n        forward_variants = fitted_hourly[\'sep_oct\'].variants(keys, forward_base)\n        forward_choice, _ = hourly.select(sep_keys, values, forward_variants)\n        report[\'hourly_experts\'][\'october_forward_from_september\'] = {\n            \'choices\': forward_choice, \'metrics\': metrics(fixed_oct_keys, values,\n                hourly.combine(fixed_oct_keys, forward_variants, forward_choice))}\n        dump(args.out/\'network_weather_report.json\', report)\n    # Historical restoration check corresponding to DS2 experiment 04.\n    restore_cut = date(2025, 8, 13)\n    restore_keys = grid(routes, date(2025, 8, 14), date(2025, 8, 31))\n    restore = bank.predict_bases(restore_cut, restore_keys)\n    report[\'august_restoration_check\'] = {n: metrics(restore_keys, values, v) for n, v in restore.items()}\n\n    reference, regimes = bank.fitted(FINAL_CUTOFF)\n    models, _ = fit_models(bank, FINAL_CUTOFF, weather)\n    bases = bank.predict_bases(FINAL_CUTOFF, sample)\n    predictions = predict_models(sample, bases, models, weather)\n    outputs = {\n        \'submission_network_weather.csv\': assemble(sample, predictions, choice),\n        \'submission_network_no_weather.csv\': assemble(sample, predictions, control),\n        \'submission_regimes_candidate.csv\': bases[\'network\'],\n        \'submission_reference.csv\': bases[\'plain\']}\n    primary_submission = \'submission_network_weather.csv\'\n    if args.hourly_experts:\n        final_hourly = hourly.HourlyExperts().fit(values, FIRST, FINAL_CUTOFF)\n        final_variants = final_hourly.variants(sample, outputs[\'submission_network_weather.csv\'])\n        primary_submission = \'submission_hourly_experts.csv\'\n        outputs[primary_submission] = hourly.combine(sample, final_variants, hourly_choice)\n        report[\'hourly_experts\'][\'changed_rows_vs_network_weather\'] = {\n            r: sum(outputs[primary_submission][k] != outputs[\'submission_network_weather.csv\'][k]\n                   for k in sample if k[0] == r) for r in routes}\n        if any(outputs[primary_submission][k] != outputs[\'submission_network_weather.csv\'][k]\n               for k in sample if k[0] not in hourly.TARGET_ROUTES):\n            raise RuntimeError(\'Hourly expert changed an unauthorized route\')\n    # Keep a clearly labeled alternative for the uncertain repair schedule on\n    # the working Saturday and holiday weekdays, rather than silently asserting\n    # that the production calendar is a service timetable.\n    scenario = dict(outputs[\'submission_network_weather.csv\'])\n    mapping = {date(2025, 11, 1): date(2025, 11, 7),\n               date(2025, 11, 3): date(2025, 11, 2),\n               date(2025, 11, 4): date(2025, 11, 2)}\n    replacements = {k: (k[0], mapping[k[1]], k[2]) for k in sample if k[0] in (\'7\', \'50\') and k[1] in mapping}\n    # Deliberately simple demand proxies for this separate scenario: original\n    # Nov1 reference, and Nov2 restricted profile for Nov3/4. Weather corrections\n    # on those 144 rows are not retained. This is not the default submission.\n    for k, proxy in replacements.items():\n        if k[1] == date(2025, 11, 1):\n            scenario[k] = outputs[\'submission_reference.csv\'][k]\n        else:\n            scenario[k] = outputs[\'submission_regimes_candidate.csv\'][proxy]\n    outputs[\'submission_holiday_service_scenario.csv\'] = scenario\n    if args.hourly_experts:\n        primary_submission = \'submission_hourly_holiday_service.csv\'\n        outputs[primary_submission] = hourly.combine(\n            sample, final_hourly.variants(sample, scenario), hourly_choice)\n        report[\'hourly_experts\'][\'changed_rows_vs_holiday_service\'] = {\n            r: sum(outputs[primary_submission][k] != scenario[k] for k in sample if k[0] == r)\n            for r in routes}\n        if any(outputs[primary_submission][k] != scenario[k]\n               for k in sample if k[0] not in hourly.TARGET_ROUTES):\n            raise RuntimeError(\'Hourly expert changed a protected holiday-service route\')\n        report[\'user_reported_platform_scores\'] = {\n            \'submission_holiday_service_scenario.csv\': 0.87989,\n            \'submission_network_weather.csv\': 0.87755,\n            \'source\': \'User message; new hourly submission has no platform score yet\'}\n    report[\'outputs\'] = {}\n    for name, prediction in outputs.items():\n        report[\'outputs\'][name] = write_submission(args.sample, prediction, args.out/name)\n    if sha256(args.out/\'submission_reference.csv\') != ref[\'submission_sha256\']:\n        raise RuntimeError(\'Frozen 0.877 reference did not reproduce byte-for-byte\')\n    ensemble = NetworkWeatherEnsemble(reference, regimes, models, weather, choice)\n    if args.hourly_experts:\n        ensemble = hourly.HourlyExpertEnsemble(\n            HolidayServiceEnsemble(ensemble, reference, regimes), final_hourly, hourly_choice)\n    artifact = pickle.dumps(ensemble, protocol=5)\n    if pickle.loads(artifact).predict(sample) != outputs[primary_submission]:\n        raise RuntimeError(\'Saved model prediction mismatch\')\n    (args.out/\'model.local.pkl\').write_bytes(artifact)\n    report[\'outputs_sha256\'] = {n: sha256(args.out/n) for n in (*outputs, \'model.local.pkl\')}\n    report[\'changed_rows_vs_reference\'] = {n: sum(v[k] != bases[\'plain\'][k] for k in sample) for n, v in outputs.items()}\n    report[\'final_state_coverage\'] = regimes.coverage(sample)\n    report[\'restoration_warning\'] = \'Nov15 service restoration is confirmed externally. Its hourly demand profile is pooled with earlier normal-service history; no November demand truth is available.\'\n    report[\'holiday_service_scenario\'] = \'Alternative only: Nov1 normal service, Nov3/4 Sunday restricted service for 7/50. No claim that this unpublished repair schedule is correct.\'\n    report[\'primary_submission\'] = primary_submission\n    dump(args.out/\'network_weather_report.json\', report)\n    if args.hourly_experts:\n        dump(args.out/\'hourly_experts_report.json\', report)\n        print(\'Hourly expert choices:\', hourly_choice, flush=True)\n    print(\'Accepted service regimes:\', report[\'selection\'][\'network_accepted_routes\'], flush=True)\n    print(\'Accepted weather:\', report[\'selection\'][\'weather_accepted_routes\'], flush=True)\n    print(\'Saved 14640-row submissions; route 5 retained with zeros. Platform score unknown.\', flush=True)\n    return report\n\n\nif __name__ == \'__main__\':\n    from ml.forecast.network_competition import main as run_main\n    run_main()\n', 'ml/forecast/test_network_competition.py': '"""In-memory correctness checks, not a demand-quality benchmark."""\nimport json\nimport pickle\nimport unittest\nfrom datetime import date, timedelta\nfrom unittest.mock import patch, Mock\n\nimport numpy as np\n\nfrom ml.forecast.core import grid\nfrom ml.forecast.network_regimes import (state, NORMAL, JULY_SHORT, WEST_AUG, WEST_SEP,\n                              BOTH_ENDS, CANCELLED, RegimeProfile)\nfrom ml.forecast.network_competition import HistoryBank, select_groups, assemble, features\n\n\nclass NetworkTests(unittest.TestCase):\n    def test_holiday_service_handles_partial_keys_and_preserves_other_routes(self):\n        from ml.forecast.network_competition import HolidayServiceEnsemble\n        keys = [(\'7\', date(2025, 11, 1), 12), (\'50\', date(2025, 11, 3), 9),\n                (\'7\', date(2025, 11, 4), 10), (\'11\', date(2025, 11, 3), 9),\n                (\'7\', date(2025, 11, 5), 9), (\'5\', date(2025, 11, 3), 9)]\n        baseline, reference, regimes = Mock(), Mock(), Mock()\n        baseline.predict.side_effect = lambda ks: {k: 0 if k[0] == \'5\' else 100 for k in ks}\n        reference.predict.side_effect = lambda ks: {k: k[2]*10 for k in ks}\n        regimes.apply.side_effect = lambda ks, pred: {k: pred[k]//2 for k in ks}\n        result = HolidayServiceEnsemble(baseline, reference, regimes).predict(keys)\n        self.assertEqual(list(result.values()), [120, 45, 50, 100, 100, 0])\n        proxy_keys = regimes.apply.call_args.args[0]\n        self.assertTrue(all(k[1] == date(2025, 11, 2) for k in proxy_keys))\n        self.assertEqual(set(result), set(keys))\n\n    def test_distinct_endpoints_and_restoration_boundaries(self):\n        for stamp, expected in [(\'2025-07-12\', JULY_SHORT), (\'2025-08-13\', NORMAL),\n                (\'2025-08-16\', WEST_AUG), (\'2025-09-06\', WEST_SEP),\n                (\'2025-09-20\', BOTH_ENDS), (\'2025-10-04\', WEST_SEP),\n                (\'2025-11-09\', WEST_SEP), (\'2025-11-15\', NORMAL)]:\n            self.assertEqual(state(\'7\', date.fromisoformat(stamp)), expected, stamp)\n        self.assertEqual(state(\'50\', date(2025, 9, 6)), CANCELLED)\n        self.assertEqual(state(\'50\', date(2025, 11, 15)), NORMAL)\n\n    def test_demand_holiday_does_not_imply_repair_closure(self):\n        self.assertEqual(state(\'50\', date(2025, 11, 1)), CANCELLED)  # literal Saturday\n        self.assertEqual(state(\'50\', date(2025, 11, 3)), NORMAL)  # demand holiday\n        self.assertEqual(state(\'50\', date(2025, 9, 8)), NORMAL)\n        self.assertEqual(state(\'17\', date(2025, 9, 6)), NORMAL)\n\n    def test_asof_uses_separate_publications_for_start_and_end(self):\n        d = date(2025, 11, 22)\n        self.assertEqual(state(\'50\', d, mode=\'as_of\', as_of=date(2025, 8, 31)), NORMAL)\n        self.assertEqual(state(\'50\', d, mode=\'as_of\', as_of=date(2025, 10, 31)), CANCELLED)\n        self.assertEqual(state(\'50\', d), NORMAL)\n        self.assertEqual(state(\'7\', date(2025, 10, 11), mode=\'as_of\', as_of=date(2025, 9, 30)), BOTH_ENDS)\n        with self.assertRaises(ValueError):\n            state(\'7\', d, mode=\'as_of\')\n\n    def test_unseen_state_does_not_transfer_july_factor(self):\n        keys = grid([\'7\', \'50\', \'1\'], date(2025, 1, 1), date(2025, 8, 31))\n        history = {k: 10 if state(k[0], k[1]) == JULY_SHORT else 100 for k in keys}\n        model = RegimeProfile().fit(history, date(2025, 1, 1), date(2025, 8, 31))\n        forecast = grid([\'7\', \'50\', \'1\', \'5\'], date(2025, 9, 6), date(2025, 9, 6))\n        base = dict.fromkeys(forecast, 100)\n        pred = model.apply(forecast, base)\n        self.assertEqual(pred[\'7\', date(2025, 9, 6), 12], 100)\n        self.assertEqual(pred[\'50\', date(2025, 9, 6), 12], 0)\n        self.assertEqual(pred[\'1\', date(2025, 9, 6), 12], 100)\n        self.assertEqual(pred[\'5\', date(2025, 9, 6), 12], 0)\n\n    def test_cancellation_keeps_observed_truth_and_learns_residual(self):\n        history = {k: 12 for k in grid([\'50\'], date(2025, 9, 6), date(2025, 10, 31))}\n        original = dict(history)\n        model = RegimeProfile().fit(history, date(2025, 9, 6), date(2025, 10, 31))\n        keys = [(\'50\', date(2025, 11, 2), 12)]\n        self.assertEqual(model.apply(keys, dict.fromkeys(keys, 100))[keys[0]], 12)\n        self.assertEqual(history, original)\n\n    def test_september_state_pools_four_weekend_days_for_october(self):\n        keys = grid([\'7\'], date(2025, 9, 1), date(2025, 9, 30))\n        history = {k: 40 if state(\'7\', k[1]) == WEST_SEP else 100 for k in keys}\n        model = RegimeProfile().fit(history, date(2025, 9, 1), date(2025, 9, 30))\n        future = [(\'7\', date(2025, 10, 4), 12)]\n        self.assertEqual(model.lookup(\'7\', WEST_SEP, 5, 12), 40)\n        self.assertEqual(model.apply(future, dict.fromkeys(future, 100))[future[0]], 88)\n\n    def test_restoration_removes_zero_rule_and_pooling_is_bounded(self):\n        keys = grid([\'50\'], date(2025, 1, 1), date(2025, 10, 31))\n        history = {k: 0 if state(k[0], k[1]) == CANCELLED else 1000 for k in keys}\n        model = RegimeProfile().fit(history, date(2025, 1, 1), date(2025, 10, 31))\n        future = [(\'50\', date(2025, 11, 9), 12), (\'50\', date(2025, 11, 15), 12)]\n        pred = model.apply(future, dict.fromkeys(future, 100))\n        self.assertEqual(pred[future[0]], 0)\n        self.assertEqual(pred[future[1]], 112)  # 25% of bounded 150 profile; half-even\n        self.assertEqual(pickle.loads(pickle.dumps(model)).apply(future, dict.fromkeys(future, 100)), pred)\n\n    def test_state_training_cannot_cross_cutoff(self):\n        with self.assertRaises(ValueError):\n            RegimeProfile().fit({(\'7\', date(2025, 9, 1), 12): 1}, date(2025, 1, 1), date(2025, 8, 31))\n\n    def test_training_cache_never_reads_future_labels_into_features_or_targets(self):\n        values = {(\'1\', date(2025, 1, 1)+timedelta(days=i), 12): i+1 for i in range(120)}\n        seen = []\n        def factory(prefix, cutoff, choices):\n            self.assertLessEqual(max(k[1] for k in prefix), cutoff)\n            class Model:\n                def predict(_, ks):\n                    self.assertGreater(min(k[1] for k in ks), cutoff)\n                    seen.append(cutoff)\n                    return dict.fromkeys(ks, sum(prefix.values()))\n            return Model()\n        with patch(\'ml.forecast.network_competition.fit_reference\', side_effect=factory):\n            a = HistoryBank(values, {}).training(date(2025, 3, 31))\n            perturbed = {k: (999999 if k[1] > date(2025, 3, 31) else v) for k, v in values.items()}\n            b = HistoryBank(perturbed, {}).training(date(2025, 3, 31))\n        self.assertEqual(a[0:2], b[0:2])\n        np.testing.assert_array_equal(a[2], b[2])\n        self.assertEqual(set(seen), {date(2025, 1, 31), date(2025, 2, 28)})\n\n    def test_bad_long_month_rejects_route50_even_with_good_october_refit(self):\n        ks = [(\'50\', date(2025, m, 6), 12) for m in (9, 10)]\n        truth = dict.fromkeys(ks, 100)\n        preds = {\'plain\': {\'reference\': dict.fromkeys(ks, 80)},\n                 \'network\': {\'reference\': {ks[0]: 100, ks[1]: 70}}}\n        recent = {\'plain\': {\'reference\': dict.fromkeys(ks, 80)},\n                  \'network\': {\'reference\': dict.fromkeys(ks, 100)}}\n        result, _ = select_groups(ks, truth, preds, ks, truth, recent, [\'50\'])\n        self.assertEqual(result[\'50\'], \'plain\')\n\n    def test_new_route7_state_requires_october_evidence(self):\n        ks = [(\'7\', date(2025, m, 6), 12) for m in (9, 10)]\n        truth = dict.fromkeys(ks, 100)\n        preds = {g: {\'reference\': dict.fromkeys(ks, 80)} for g in (\'plain\', \'network\')}\n        recent = {\'plain\': {\'reference\': dict.fromkeys(ks, 80)},\n                  \'network\': {\'reference\': dict.fromkeys(ks, 98)}}\n        result, _ = select_groups(ks, truth, preds, ks, truth, recent, [\'7\'])\n        self.assertEqual(result[\'7\'], \'network\')\n\n    def test_weather_pair_has_identical_service_feature(self):\n        ks = [(\'50\', date(2025, 9, 6), 12)]\n        weather = {(ks[0][1], 12): [0.]*10}\n        base = dict.fromkeys(ks, 0)\n        a = features(ks, base, weather, False, True)\n        b = features(ks, base, weather, True, True)\n        self.assertEqual(a[0, -1], CANCELLED)\n        self.assertEqual(b[0, -1], CANCELLED)\n        np.testing.assert_array_equal(a[:, :9], b[:, :9])\n\n    def test_unknown_route_preserved_as_zero_when_assembling(self):\n        ks = [(\'5\', date(2025, 11, 1), 12)]\n        self.assertEqual(assemble(ks, {}, {}), {ks[0]: 0})\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'ml/forecast/network_sources.json': '{\n  "source_mode": "competition",\n  "retrieved_at": "2026-09-27",\n  "research_branch": "ds2/research-protocol",\n  "research_commit": "b593ebbb5351bcc31881a99431d820ad97b5d140",\n  "authorization": "Organizer Q&A permits external facts published after 2025-10-31, including actual weather and transport news. Future passenger targets are never permitted.",\n  "events": [\n    {"routes": [7], "published_at": "2025-07-02", "start": "2025-07-10", "end_exclusive": "2025-08-13", "end_published_at": "2025-08-13", "state": "sokolniki_july", "days": "all", "source": "https://transport.mos.ru/mostrans/all_news/125169", "end_source": "https://transport.mos.ru/mostrans/all_news/125800"},\n    {"routes": [7], "published_at": "2025-08-15", "start": "2025-08-16", "end_exclusive": "2025-09-06", "end_published_at": "2025-09-05", "state": "novoslobodskaya_august", "days": "Saturday/Sunday", "source": "https://transport.mos.ru/mostrans/all_news/125846"},\n    {"routes": [7, 50], "published_at": "2025-09-05", "start": "2025-09-06", "end_exclusive": "2025-11-15", "end_published_at": "2025-11-15", "state_by_route": {"7": "kalanchyovskaya", "50": "cancelled"}, "days": "Saturday/Sunday", "source": "https://mosmetro.ru/news/details/7732", "readable_primary_mirror": "https://mosmetro.teletribe.ru/news/details/7732", "end_source": "https://t.me/DtOperativno/23565", "note": "DS2 registry omitted the simultaneous route-7 change in this route-50 notice; included here. Recorded positive route-50 labels are retained."},\n    {"routes": [7], "published_at": "2025-09-19", "start": "2025-09-20", "end_exclusive": "2025-10-04", "end_published_at": "2025-10-04", "state": "kalanchyovskaya_and_bogorodskoe", "days": "Saturday/Sunday", "source": "https://transport.mos.ru/mostrans/all_news/126405", "end_source": "https://mosmetro.ru/news/details/7973", "readable_end_source": "https://mosmetro.teletribe.ru/news/details/7973"}\n  ],\n  "limitations": [\n    "This is a partial daytime service registry, not a complete historical timetable.",\n    "Repair weekends are interpreted literally as Saturday/Sunday. Nov 1 is a working Saturday: whether repair closures followed the changed work calendar is unconfirmed. Nov 3/4 closures are not inferred from the public-holiday demand calendar. A separate holiday-service scenario CSV isolates this uncertainty.",\n    "Late-night changes reported Sep 12, Oct 28, Nov 5 and Dec 13 have no fully verified repeat schedule and hour-by-hour capacity. No invented zero-service mask is applied to them.",\n    "November restoration is externally confirmed, but no November labels exist locally; its passenger-demand effect is unvalidated.",\n    "July-shortening and September-east-detour magnitudes are not assumed identical. An unseen state falls back to the reference.",\n    "The 31-day October refit diagnostic can assess a state observed in September; it is not an independent 61-day test. The long Sep-Oct gate remains reported separately."\n  ]\n}\n', 'ml/forecast/research_audit.json': '{\n  "branch": "ds2/research-protocol",\n  "commit": "b593ebbb5351bcc31881a99431d820ad97b5d140",\n  "scope": "docs/research recursively, complete tracked files",\n  "review": "All Markdown reports and Python scripts inspected; all saved JSON reports parsed and metric identities checked. Experiments were not rerun without organizer labels.",\n  "file_count": 56,\n  "counts": {\n    ".md": 19,\n    ".py": 19,\n    ".json": 18\n  },\n  "metric_identity_checks": 1063,\n  "files": [\n    {\n      "path": "CASE_STUDIES_2025.md",\n      "sha256": "9a4984e5d807f4e63b16e20f03c2b5024b1a55af7ecd4588a40b9226ddb3d166",\n      "bytes": 7616\n    },\n    {\n      "path": "EXPERIMENT_02_HISTORY.md",\n      "sha256": "7740029d69ccb1ae6140aba1aad85af7a95146ab37125da37f584713c920ba49",\n      "bytes": 5964\n    },\n    {\n      "path": "EXPERIMENT_03_CALENDAR.md",\n      "sha256": "211abd3b6fb18c61d68cf396e1d51fd0d26bb61682cbe0adbddb54cdd73096c9",\n      "bytes": 4749\n    },\n    {\n      "path": "EXPERIMENT_04_ROUTE7_RESTORATION.md",\n      "sha256": "67b3b013deb111ed9091b5dcb4cfe804c2f384c105b493de466088265817908c",\n      "bytes": 5865\n    },\n    {\n      "path": "EXPERIMENT_05_ROUTE50_SHUTDOWN.md",\n      "sha256": "daf0259ba7a538a05762d53a7365f853b5179b8e8641013d1a9a8f36a8e014ae",\n      "bytes": 6491\n    },\n    {\n      "path": "EXPERIMENT_06_DS1_REAL_LABELS.md",\n      "sha256": "00ae445c9ce6a928baef74cf8484c330032d5527daf87e6394842b39607dce14",\n      "bytes": 5505\n    },\n    {\n      "path": "EXPERIMENT_07_ADAPTIVE_PROFILES.md",\n      "sha256": "309bb442b43676bbc67986ceff39f4e3422b64462e263b41defff3ac5b3cc728",\n      "bytes": 6083\n    },\n    {\n      "path": "EXPERIMENT_08_NETWORK_AND_SOURCE_AUDIT.md",\n      "sha256": "e7bc6d4096534a4bdee2a3635665f2329de618f1779d18d1941ec4a56c66c7bf",\n      "bytes": 10484\n    },\n    {\n      "path": "EXPERIMENT_09_SCHEDULE_AND_GAPS.md",\n      "sha256": "a976453ebb2b3ab9df9f592cab345faae37158d21fc7cd829facde9460283161",\n      "bytes": 9534\n    },\n    {\n      "path": "EXPERIMENT_10_SCHOOL_TERM.md",\n      "sha256": "274c2fa9ffcb55fc0f3dc1493bd38d4b1372b6d70abb44dc525bef8840470eb4",\n      "bytes": 5512\n    },\n    {\n      "path": "EXPERIMENT_11_ANNUAL_SEASONALITY.md",\n      "sha256": "72ba8ceb60b0824e79bda4248c342fb54e44fcdddb94ea5705449fbf7e16e2d7",\n      "bytes": 3243\n    },\n    {\n      "path": "EXPERIMENT_12_EXTERNAL_AND_YEAR.md",\n      "sha256": "6d23415deed545c7503a71cf5f9db0462f5ef215cd2c199260a9b23d411424a6",\n      "bytes": 8568\n    },\n    {\n      "path": "EXPERIMENT_13_AUGUST_REGIME.md",\n      "sha256": "6c0b1e0fbcfea3cb72c4475d1d4d5c8bf78b7d2630c6dfd5008db0c3756ba1c6",\n      "bytes": 5239\n    },\n    {\n      "path": "EXPERIMENT_14_NEAR_TERM.md",\n      "sha256": "f2024714ca248299d4daa3d933c722e012b77de27a570c5d41aeb367508c5c1e",\n      "bytes": 5498\n    },\n    {\n      "path": "EXPERIMENT_15_SCHOOL_POI_ASOF.md",\n      "sha256": "e54058d79116ef88f98cf5c5e769315e15dac1209876f60b86b6fe92166a954b",\n      "bytes": 6852\n    },\n    {\n      "path": "FACTOR_MATRIX.md",\n      "sha256": "888c73c456a94a51cb678dfca0031ebadd465783295b469feff22cffa0a9a1c4",\n      "bytes": 18276\n    },\n    {\n      "path": "MODEL_CRITERIA_2026-09-26.md",\n      "sha256": "737ccbdbbe4d434e99905688bc5cd282ef0ad83656f584238ba515e1603c34bf",\n      "bytes": 9542\n    },\n    {\n      "path": "README.md",\n      "sha256": "a8cd5e3b21a814ec8ad027866ff3c3218e552837a2f37e0deb427246c9fe75d4",\n      "bytes": 7066\n    },\n    {\n      "path": "RESULTS_2026-09-26.md",\n      "sha256": "f8dcd389b856420bcec8ad2ac4cdef60913245660c457ffdba718fc94debc85d",\n      "bytes": 8576\n    },\n    {\n      "path": "audit_missing_hours_raw.py",\n      "sha256": "4f8821f3d2ec9dd8ef26de8505bb81e0bfefcebf59124dec8c8e5db1635cfb45",\n      "bytes": 4478,\n      "functions": [\n        "days",\n        "main"\n      ]\n    },\n    {\n      "path": "audit_route50_sep13.py",\n      "sha256": "65e53f7b197309e06e4a62ef8e6a2e93999e77d8d5d344a43bfb118b5bb2f521",\n      "bytes": 3646,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "backtest_baseline.py",\n      "sha256": "3f28f66e3e2e095058838d0a94b3b38e7a8aea04e30c43fb99ecbb907a607f65",\n      "bytes": 5320,\n      "functions": [\n        "load",\n        "dates",\n        "fit",\n        "evaluate",\n        "main"\n      ]\n    },\n    {\n      "path": "diagnose_routes.py",\n      "sha256": "a2c24e59a3442bf3f39028926f0f313b92e38897f894794a5aaaa7f8f6f0ce80",\n      "bytes": 3848,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_14_near_term.py",\n      "sha256": "64063003ab70a52d6bd4d869f33821bd9ed7ed593a2e462a544ef2f404bc88e3",\n      "bytes": 5670,\n      "functions": [\n        "load",\n        "windows",\n        "measure",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_15_school_poi.py",\n      "sha256": "6367e8a2bd14608239f1d5f6299337a8a7efd8766f908ae39d114f79efac35b8",\n      "bytes": 5564,\n      "functions": [\n        "days",\n        "read_labels",\n        "profile_score",\n        "audited_stops",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_adaptive_profiles.py",\n      "sha256": "bea122b940756225faddab502da31d271b11c450134eed4c168fddcd91fed3f7",\n      "bytes": 6239,\n      "functions": [\n        "recent_means",\n        "predict",\n        "assess",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_annual_fourier.py",\n      "sha256": "dfc97e68ff989d510dd79cdbd42cf7d8c8119e63f94071e0baba6292f7f59d2d",\n      "bytes": 4192,\n      "functions": [\n        "vector",\n        "run_fold",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_august_regime.py",\n      "sha256": "beb45b83bc205430c431b95a269cfc700e17e7e8e8e3a6d32fff7c0fdb5e6523",\n      "bytes": 4057,\n      "functions": [\n        "run",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_calendar.py",\n      "sha256": "1a1dbb0747cbb789d3ef7f226ca73dc74073ef64147bbb6722a59f94afe543a1",\n      "bytes": 4858,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_ds1_candidates.py",\n      "sha256": "e7a199335e7e8b085037ce16510744dd1c3ebdd83d5a07b31df8fe39cc41ed4f",\n      "bytes": 4347,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_history_window.py",\n      "sha256": "fe83d4c96179fe0ecbd2605092a4e3a7ea946c685ef83651b2cfef948a35fb0b",\n      "bytes": 3105,\n      "functions": [\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_network_sept19.py",\n      "sha256": "c777658e3881c929eb0f8fca860133e135b3747c8676ed1ee8f0c9d19a952ac5",\n      "bytes": 5157,\n      "functions": [\n        "predict",\n        "route7_transfer_factor",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_route17_restoration.py",\n      "sha256": "abe7b995edbcd2f6dbbb3407d20aa93c65d0de8c9921372f8579d0dcd763cae2",\n      "bytes": 4223,\n      "functions": [\n        "revised_route17_profile",\n        "metrics",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_route50_shutdown.py",\n      "sha256": "8ac9891d653652bb48b1e24ec8a03badc36072b54e8b54be62b3aec3f1878111",\n      "bytes": 4174,\n      "functions": [\n        "compare",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_route7_restoration.py",\n      "sha256": "7edd3ece4cc68f917722d4d7316acb5fa82ab43f13f96c35beedf793c29b5c4d",\n      "bytes": 4823,\n      "functions": [\n        "fit_with_known_status",\n        "score",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_school_term_profiles.py",\n      "sha256": "1a25b05c95b5c608d0a436b5257628e1a2fa0bd872764ad70c75c37974fa5e5e",\n      "bytes": 3821,\n      "functions": [\n        "score",\n        "main"\n      ]\n    },\n    {\n      "path": "experiment_year_horizon.py",\n      "sha256": "97831b55945f01de9984d3fbd869f60a1ff09a0387df5a5e1287ce2319741338",\n      "bytes": 3013,\n      "functions": [\n        "peak_metrics",\n        "main"\n      ]\n    },\n    {\n      "path": "profile_data.py",\n      "sha256": "30db2ba842ee830a8212a31edf8db048b049f7dbe1ebe0849e6a484c96d8d0e4",\n      "bytes": 10200,\n      "functions": [\n        "open_text",\n        "parse_timestamp",\n        "parse_route",\n        "write_csv",\n        "load_labels",\n        "inspect_submission",\n        "days",\n        "audit_raw",\n        "main"\n      ]\n    },\n    {\n      "path": "results/adaptive_profiles.json",\n      "sha256": "b87dae1e03381c8556911f08fbc926c7836fe71f5f406323e411b6e7cc1b5bfa",\n      "bytes": 44543,\n      "top_level_fields": [\n        "hypothesis",\n        "calendar_source",\n        "candidate_grid",\n        "missing_label_rule",\n        "folds",\n        "selection"\n      ]\n    },\n    {\n      "path": "results/annual_fourier.json",\n      "sha256": "d6cd86d8f4e911121539c21c93592299099767e77c874b91d036d75a3c0d1bda",\n      "bytes": 3458,\n      "top_level_fields": [\n        "calendar_features",\n        "network_scale_routes",\n        "excluded_from_scale",\n        "rules",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/august_regime.json",\n      "sha256": "79c0009fcd49255c4d69626c13b9438543bc83ce4d47e6d66248579eaf15d7e7",\n      "bytes": 5314,\n      "top_level_fields": [\n        "cutoff",\n        "validation",\n        "status",\n        "source",\n        "metrics"\n      ]\n    },\n    {\n      "path": "results/backtest.json",\n      "sha256": "f711379ea7e8ebb23b94d8b259a833c4f1cd7c40e6b00178d2a47f1003994589",\n      "bytes": 7279,\n      "top_level_fields": [\n        "method",\n        "missing_label_rule",\n        "routes",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/calendar.json",\n      "sha256": "d84c45e9612a3ed23f49f52cb306f55cdd5409f2ac5940fb411a1d25f8ed509e",\n      "bytes": 4178,\n      "top_level_fields": [\n        "source_url",\n        "published_at",\n        "hypothesis",\n        "new_feature",\n        "routes",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/ds1_real_candidates.json",\n      "sha256": "8bdfc73f7a63f1b70563b6ad84bfc932116897bc7c9ceee6184875af7b4dc6d1",\n      "bytes": 90696,\n      "top_level_fields": [\n        "source_branch",\n        "source_commit",\n        "ds1_code_sha256",\n        "input_sha256",\n        "metric",\n        "policy",\n        "folds",\n        "selection"\n      ]\n    },\n    {\n      "path": "results/experiment_15_school_poi_asof.json",\n      "sha256": "0becf8b5167656b2a82f9d074e98ea2f81a1a283291f2fdb1f2ac23b7f6731fd",\n      "bytes": 1083,\n      "top_level_fields": [\n        "cutoff",\n        "holdout",\n        "hour_grid",\n        "baseline",\n        "exclude_summer_weekdays",\n        "delta_score",\n        "selection_note",\n        "stops"\n      ]\n    },\n    {\n      "path": "results/history_window.json",\n      "sha256": "595313903aabbc72610249572c09d4b88ef331a99792faf063e8821001bcdf1c",\n      "bytes": 15558,\n      "top_level_fields": [\n        "hypothesis",\n        "changed_parameter",\n        "leakage_rule",\n        "routes",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/missing_hours_raw.json",\n      "sha256": "f41b898af29f924914683b71ed20049f419ca1844fec63e9f429749d6bf87554",\n      "bytes": 13378,\n      "top_level_fields": [\n        "raw_rows_read",\n        "scope",\n        "candidate_hours_not_in_labels",\n        "candidate_hours_with_any_raw_transaction",\n        "candidate_hours_with_success",\n        "oracle_125_hour_error_removed_from_aug31_median",\n        "oracle_score_gain_if_future_missing_keys_known",\n        "raw_status_totals",\n        "raw_transaction_types",\n        "missing_hours",\n        "limitations"\n      ]\n    },\n    {\n      "path": "results/network_events_2025.json",\n      "sha256": "8659d4aee1b2b4b7e17ab91b0099f885f048ce5b5bd98d5bdf573d3b50bcbd84",\n      "bytes": 3742,\n      "top_level_fields": [\n        "scope",\n        "time_rule",\n        "routes_in_labels",\n        "routes_with_events_in_this_partial_registry",\n        "events"\n      ]\n    },\n    {\n      "path": "results/network_sept19.json",\n      "sha256": "fc70fd54fc6eed72cba400ebb571fab6e95dfe61a20cd203ca35e723b420aef2",\n      "bytes": 11711,\n      "top_level_fields": [\n        "cutoff",\n        "validation",\n        "event_dates",\n        "news",\n        "route7_scenario_transfer",\n        "training_rule",\n        "predictions",\n        "metrics"\n      ]\n    },\n    {\n      "path": "results/route17_restoration.json",\n      "sha256": "ed58b4ad0ee8c338c23b4a6ca37947274b18d78cbe09dfd07412f1cf7b2ee00c",\n      "bytes": 17182,\n      "top_level_fields": [\n        "sources",\n        "restricted_weekends_excluded_from_train",\n        "effect_isolated_to_route",\n        "folds"\n      ]\n    },\n    {\n      "path": "results/route50_sep13_raw_audit.json",\n      "sha256": "493cdf647c8dd8b50b78a55c953fe87e22a18d91acbf0112ace528a8b7c4a483",\n      "bytes": 4348,\n      "top_level_fields": [\n        "source",\n        "rows_scanned",\n        "route_50_invalid_event_time",\n        "sep13_status_counts",\n        "sep13_status_transaction_type_counts",\n        "sep13_hourly",\n        "weekend_daily"\n      ]\n    },\n    {\n      "path": "results/route50_shutdown.json",\n      "sha256": "12d4dfa2ee886b2e04fca1424442a2a0e83959b147e3d370405d32ed7e50a6fe",\n      "bytes": 9622,\n      "top_level_fields": [\n        "source",\n        "published_at",\n        "cutoff",\n        "validation",\n        "missing_label_rule",\n        "changed_rows_per_statistic",\n        "variants",\n        "affected_day_facts"\n      ]\n    },\n    {\n      "path": "results/route7_restoration.json",\n      "sha256": "d71ee09e86ed79bb323deac50b1feb7644ca3f3906be40eb32f0652bd4339e47",\n      "bytes": 8203,\n      "top_level_fields": [\n        "source_start",\n        "start_notice_published",\n        "source_restored",\n        "restoration_notice_published",\n        "cutoff",\n        "excluded_route_7_training_dates",\n        "validation",\n        "definition",\n        "variants"\n      ]\n    },\n    {\n      "path": "results/route_diagnostics.json",\n      "sha256": "a904ee4732a186530b9e1d47dad50867141d4e6b00a4083c43f26bce532bace3",\n      "bytes": 3603,\n      "top_level_fields": [\n        "method",\n        "candidate_service_hours",\n        "routes_in_labels",\n        "route_7_windows",\n        "missing_active_hours_count",\n        "missing_active_hours_by_route",\n        "missing_active_hours_by_date"\n      ]\n    },\n    {\n      "path": "results/school_term_profiles.json",\n      "sha256": "6a0d0b7d7c860b14b1f876fa7f0bee1fa0ef328d526bbee32f339b843903570d",\n      "bytes": 8347,\n      "top_level_fields": [\n        "train_cutoff",\n        "validation",\n        "rule",\n        "selection_warning",\n        "baseline_median",\n        "variants"\n      ]\n    },\n    {\n      "path": "results/year_horizon.json",\n      "sha256": "17d534587150c0b44d37c00eb85e4a7638c7e1c26a9eb7c1cf7e1bef0c1791dd",\n      "bytes": 13960,\n      "top_level_fields": [\n        "metric",\n        "model",\n        "external_sources",\n        "full_year_validated",\n        "folds"\n      ]\n    }\n  ]\n}\n', 'ml/forecast/hourly_experts.py': '"""Separable daily-volume/hour-shape candidates for routes 17, 11 and 12.\n\nThese are finite, testable alternatives to the existing hourly route model,\nnot an assertion that a finer profile necessarily improves competition WAPE.\n"""\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nfrom datetime import date\nimport numpy as np\n\nfrom .core import days, effective_weekday, rounded\nfrom .route_specific import SeasonalProfile\n\nTARGET_ROUTES = (\'17\', \'11\', \'12\')\nVARIANTS = {\'level_50\': (.5, 0.), \'shape_50\': (0., .5),\n            \'level_shape_25\': (.25, .25), \'level_shape_50\': (.5, .5)}\n\n\ndef allocate(total, proportions):\n    """Largest remainders: integer hours sum exactly to the rounded daily total."""\n    p = np.asarray(proportions, dtype=float)\n    if p.shape != (24,) or not np.isfinite(p).all() or (p < 0).any():\n        raise ValueError(\'Need 24 finite nonnegative hourly shares\')\n    total = rounded(total)\n    if not p.sum():\n        if total:\n            raise ValueError(\'Positive daily total has no supported hours\')\n        return [0]*24\n    raw = total*p/p.sum()\n    integer = np.floor(raw).astype(np.int64)\n    left = total-int(integer.sum())\n    order = sorted(range(24), key=lambda h: (-(raw[h]-integer[h]), h))\n    for h in order[:left]:\n        integer[h] += 1\n    return [int(x) for x in integer]\n\n\nclass HourlyExperts:\n    """Calendar-compatible volume and shape with shrinkage to the incumbent.\n\n    Recent means last eight comparable weekdays, not eight calendar days.\n    Summer weekdays are separate. Sparse groups fall back to longer history;\n    no future targets or predicted-period lag updates are used.\n    """\n    def fit(self, history, start, cutoff):\n        if not history or any(not start <= k[1] <= cutoff for k in history):\n            raise ValueError(\'History is empty or crosses cutoff\')\n        self.cutoff = cutoff\n        self.routes = set(TARGET_ROUTES) & {k[0] for k in history}\n        groups = defaultdict(list)\n        for r in sorted(self.routes):\n            for d in days(start, cutoff):\n                values = np.asarray([history.get((r, d, h), 0) for h in range(24)], dtype=float)\n                if (values < 0).any() or not np.isfinite(values).all():\n                    raise ValueError(\'Invalid historical demand\')\n                season = SeasonalProfile.summer(d)\n                w = effective_weekday(d)\n                for s in (None, season):\n                    groups[r, s, w].append(values)\n        self.tables = {}\n        for key, rows in groups.items():\n            if len(rows) < 4:\n                continue\n            x = np.stack(rows)\n            totals = x.sum(axis=1)\n            recent_totals = totals[-8:]\n            # Two robust training estimates, not a global autumn decay rule.\n            volume = .5*float(np.median(totals)) + .5*float(np.median(recent_totals))\n            valid = totals > 0\n            shape = None\n            if valid.sum() >= 4:\n                shares = x[valid]/totals[valid, None]\n                long = np.median(shares, axis=0)\n                recent = np.median(shares[-8:], axis=0)\n                if long.sum() and recent.sum():\n                    shape = .5*long/long.sum() + .5*recent/recent.sum()\n            self.tables[key] = {\'volume\': volume, \'shape\': shape,\n                                \'days\': len(rows), \'positive_days\': int(valid.sum()),\n                                \'recent_days\': min(8, len(rows))}\n        return self\n\n    def profile(self, route, day):\n        w = effective_weekday(day)\n        return self.tables.get((route, SeasonalProfile.summer(day), w),\n                               self.tables.get((route, None, w)))\n\n    def variants(self, keys, base):\n        if len(set(keys)) != len(keys) or set(keys) != set(base):\n            raise ValueError(\'Base predictions must match unique requested keys\')\n        if any(d <= self.cutoff for _, d, _ in keys):\n            raise ValueError(\'Prediction dates must follow cutoff\')\n        out = {\'reference\': dict(base), **{name: dict(base) for name in VARIANTS}}\n        by_day = defaultdict(set)\n        for r, d, h in keys:\n            if r in self.routes:\n                by_day[r, d].add(h)\n        for (r, d), hours in by_day.items():\n            if hours != set(range(24)):\n                raise ValueError(\'Daily/shape experts require all 24 hours for each target route-day\')\n            profile = self.profile(r, d)\n            if profile is None:\n                continue\n            b = np.asarray([base[r, d, h] for h in range(24)], dtype=float)\n            if not np.isfinite(b).all() or (b < 0).any():\n                raise ValueError(\'Invalid base prediction\')\n            total = float(b.sum())\n            if total == 0:\n                continue\n            base_shape = b/total\n            # Limit the proposed daily estimate before blending; final 50%\n            # candidate changes daily volume by no more than 15% (rounding aside).\n            volume = float(np.clip(profile[\'volume\'], .7*total, 1.3*total))\n            shape = profile[\'shape\']\n            if shape is None:\n                shape = base_shape\n            else:\n                # Preserve unsupported zero hours, then normalize the bounded\n                # shape. This avoids inventing night service from pooled days.\n                shape = np.clip(shape, .5*base_shape, 1.5*base_shape)\n                shape = shape/shape.sum()\n            for name, (volume_weight, shape_weight) in VARIANTS.items():\n                daily = (1-volume_weight)*total + volume_weight*volume\n                shares = (1-shape_weight)*base_shape + shape_weight*shape\n                prediction = allocate(daily, shares)\n                for h, value in enumerate(prediction):\n                    out[name][r, d, h] = value\n        return out\n\n\ndef select(keys, truth, variants, *, refit_keys=None, refit_truth=None, refit_variants=None):\n    """Require a 0.5% gain in each month, with an optional October refit guard."""\n    months = sorted({(k[1].year, k[1].month) for k in keys})\n    choices, audit = {}, {}\n    for r in TARGET_ROUTES:\n        groups = [[k for k in keys if k[0] == r and (k[1].year, k[1].month) == m] for m in months]\n        if not any(groups):\n            continue\n        errors = {name: [sum(abs(pred[k]-truth.get(k, 0)) for k in g) for g in groups]\n                  for name, pred in variants.items()}\n        recent_errors = {}\n        if refit_keys is not None:\n            if refit_truth is None or refit_variants is None:\n                raise ValueError(\'Refit guard requires both truth and predictions\')\n            selected = [k for k in refit_keys if k[0] == r]\n            recent_errors = {name: sum(abs(pred[k]-refit_truth.get(k, 0)) for k in selected)\n                             for name, pred in refit_variants.items()}\n        accepted = [\'reference\']\n        for name in VARIANTS:\n            monthly_win = (all(a <= .995*b for a, b in zip(errors[name], errors[\'reference\']))\n                           and sum(errors[name]) < sum(errors[\'reference\']))\n            recent_ok = not recent_errors or recent_errors[name] <= 1.005*recent_errors[\'reference\']\n            if monthly_win and recent_ok:\n                accepted.append(name)\n        choices[r] = min(accepted, key=lambda n: (sum(errors[n]), n))\n        audit[r] = {\'months\': [f\'{y}-{m:02}\' for y, m in months],\n                    \'monthly_absolute_errors\': errors, \'refit_absolute_errors\': recent_errors,\n                    \'accepted\': accepted, \'selected\': choices[r]}\n    return choices, audit\n\n\ndef combine(keys, variants, choices):\n    return {k: variants[choices.get(k[0], \'reference\')][k] for k in keys}\n\n\ndef diagnostics(keys, truth, prediction):\n    """Hour-specific errors and daily-total errors, never used as fit features."""\n    result = {}\n    for r in TARGET_ROUTES:\n        subset = [k for k in keys if k[0] == r]\n        if not subset:\n            continue\n        rows, daily = [], defaultdict(lambda: [0, 0])\n        for h in range(24):\n            ks = [k for k in subset if k[2] == h]\n            actual = sum(truth.get(k, 0) for k in ks)\n            predicted = sum(prediction[k] for k in ks)\n            error = sum(abs(prediction[k]-truth.get(k, 0)) for k in ks)\n            rows.append({\'hour\': h, \'actual_sum\': actual, \'prediction_sum\': predicted,\n                         \'absolute_error_sum\': error, \'bias\': predicted-actual})\n        for k in subset:\n            daily[k[1]][0] += truth.get(k, 0)\n            daily[k[1]][1] += prediction[k]\n        dayparts = {}\n        for name, hours in [(\'morning_06_10\', range(6, 11)), (\'midday_11_15\', range(11, 16)),\n                            (\'evening_16_20\', range(16, 21)), (\'other_hours\', (0, 1, 2, 3, 4, 5, 21, 22, 23))]:\n            dayparts[name] = {field: sum(rows[h][field] for h in hours)\n                              for field in (\'actual_sum\', \'prediction_sum\', \'absolute_error_sum\', \'bias\')}\n        result[r] = {\'by_hour\': rows, \'by_daypart\': dayparts,\n                     \'daily_total_absolute_error\': sum(abs(p-y) for y, p in daily.values()),\n                     \'hourly_absolute_error\': sum(x[\'absolute_error_sum\'] for x in rows)}\n    return result\n\n\nclass HourlyExpertEnsemble:\n    def __init__(self, baseline, experts, choices):\n        self.baseline, self.experts, self.choices = baseline, experts, choices\n\n    def predict(self, keys):\n        base = self.baseline.predict(keys)\n        return combine(keys, self.experts.variants(keys, base), self.choices)\n', 'ml/forecast/test_hourly_experts.py': '"""Correctness fixtures in memory; no generated training files or quality scores."""\nimport pickle\nimport unittest\nfrom datetime import date\n\nimport numpy as np\n\nfrom ml.forecast.core import grid\nfrom ml.forecast.hourly_experts import (HourlyExperts, allocate, combine, diagnostics,\n                                       select, VARIANTS, TARGET_ROUTES)\n\n\nclass HourlyTests(unittest.TestCase):\n    def history(self):\n        return {k: (100 if 6 <= k[2] <= 10 else 200 if 16 <= k[2] <= 20 else 20)\n                for k in grid(TARGET_ROUTES, date(2025, 1, 1), date(2025, 4, 30))}\n\n    def fitted(self):\n        return HourlyExperts().fit(self.history(), date(2025, 1, 1), date(2025, 4, 30))\n\n    def forecast(self):\n        keys = grid(TARGET_ROUTES+(\'7\', \'50\', \'5\'), date(2025, 5, 5), date(2025, 5, 5))\n        base = {k: (0 if k[0] == \'5\' or k[2] < 5 else 300 if k[2] < 12 else 50) for k in keys}\n        return keys, base\n\n    def test_shape_preserves_daily_sum_and_zero_hours(self):\n        model = self.fitted()\n        keys, base = self.forecast()\n        variants = model.variants(keys, base)\n        for r in TARGET_ROUTES:\n            ks = [k for k in keys if k[0] == r]\n            self.assertEqual(sum(base[k] for k in ks), sum(variants[\'shape_50\'][k] for k in ks))\n            self.assertTrue(any(base[k] != variants[\'shape_50\'][k] for k in ks))\n            self.assertTrue(all(variants[\'shape_50\'][k] == 0 for k in ks if base[k] == 0))\n\n    def test_other_routes_are_identical_in_every_variant(self):\n        keys, base = self.forecast()\n        for name, pred in self.fitted().variants(keys, base).items():\n            self.assertEqual(set(pred), set(keys))\n            self.assertTrue(all(pred[k] == base[k] for k in keys if k[0] not in TARGET_ROUTES), name)\n            self.assertTrue(all(isinstance(v, int) and v >= 0 for v in pred.values()))\n\n    def test_daily_level_bounds(self):\n        keys, base = self.forecast()\n        variants = self.fitted().variants(keys, base)\n        for r in TARGET_ROUTES:\n            ks = [k for k in keys if k[0] == r]\n            total = sum(base[k] for k in ks)\n            for name in VARIANTS:\n                self.assertLessEqual(abs(sum(variants[name][k] for k in ks)-total), .15*total+1)\n\n    def test_proportions_do_not_depend_on_training_volume_units(self):\n        values = self.history()\n        a = self.fitted()\n        b = HourlyExperts().fit({k: 3*v for k, v in values.items()}, date(2025, 1, 1), date(2025, 4, 30))\n        d = date(2025, 5, 5)\n        np.testing.assert_allclose(a.profile(\'17\', d)[\'shape\'], b.profile(\'17\', d)[\'shape\'])\n        self.assertAlmostEqual(b.profile(\'17\', d)[\'volume\'], 3*a.profile(\'17\', d)[\'volume\'])\n\n    def test_summer_weekday_slump_is_not_autumn_level(self):\n        keys = grid([\'17\'], date(2025, 1, 1), date(2025, 8, 31))\n        history = {k: 10 if k[1].month in (7, 8) and k[1].weekday() < 5 else 100 for k in keys}\n        model = HourlyExperts().fit(history, date(2025, 1, 1), date(2025, 8, 31))\n        self.assertEqual(model.profile(\'17\', date(2025, 9, 1))[\'volume\'], 2400)\n        self.assertEqual(model.profile(\'17\', date(2026, 7, 6))[\'volume\'], 240)\n\n    def test_history_and_prediction_cutoffs_enforced(self):\n        values = self.history()\n        values[\'17\', date(2025, 5, 1), 12] = 99999\n        with self.assertRaises(ValueError):\n            HourlyExperts().fit(values, date(2025, 1, 1), date(2025, 4, 30))\n        k = (\'17\', date(2025, 4, 30), 12)\n        with self.assertRaises(ValueError):\n            self.fitted().variants([k], {k: 1})\n\n    def test_exact_sums_and_deterministic_rounding(self):\n        result = allocate(101, np.ones(24))\n        self.assertEqual(sum(result), 101)\n        self.assertEqual(result[:5], [5]*5)\n        self.assertEqual(result[5:], [4]*19)\n        with self.assertRaises(ValueError):\n            allocate(1, np.zeros(24))\n        with self.assertRaises(ValueError):\n            allocate(1, [float(\'nan\')]*24)\n\n    def test_no_silent_partial_days_or_duplicate_keys(self):\n        model = self.fitted()\n        keys, base = self.forecast()\n        del base[keys[0]]\n        with self.assertRaises(ValueError):\n            model.variants(keys[1:], base)\n        with self.assertRaises(ValueError):\n            model.variants(keys+keys, dict.fromkeys(keys, 1))\n\n    def test_order_and_pickle_invariance(self):\n        model = self.fitted()\n        keys, base = self.forecast()\n        expected = model.variants(keys, base)\n        self.assertEqual(model.variants(list(reversed(keys)), base), expected)\n        self.assertEqual(pickle.loads(pickle.dumps(model)).variants(keys, base), expected)\n\n    def test_monthly_gate_rejects_single_month_win(self):\n        keys = [(\'17\', date(2025, m, 1), 12) for m in (9, 10)]\n        truth = dict.fromkeys(keys, 100)\n        variants = {n: dict.fromkeys(keys, 80) for n in (\'reference\', *VARIANTS)}\n        variants[\'shape_50\'] = {keys[0]: 100, keys[1]: 79}\n        choices, _ = select(keys, truth, variants)\n        self.assertEqual(choices[\'17\'], \'reference\')\n        variants[\'shape_50\'][keys[1]] = 99\n        choices, _ = select(keys, truth, variants)\n        self.assertEqual(choices[\'17\'], \'shape_50\')\n\n    def test_october_refit_can_veto_a_long_holdout_winner(self):\n        keys = [(\'17\', date(2025, m, 1), 12) for m in (9, 10)]\n        truth = dict.fromkeys(keys, 100)\n        variants = {n: dict.fromkeys(keys, 80) for n in (\'reference\', *VARIANTS)}\n        variants[\'shape_50\'] = dict.fromkeys(keys, 99)\n        refit = {n: dict(v) for n, v in variants.items()}\n        refit[\'shape_50\'] = dict.fromkeys(keys, 50)\n        choices, _ = select(keys, truth, variants, refit_keys=keys, refit_truth=truth, refit_variants=refit)\n        self.assertEqual(choices[\'17\'], \'reference\')\n\n    def test_hourly_error_detects_misshapen_day_with_perfect_total(self):\n        keys = grid([\'17\'], date(2025, 9, 1), date(2025, 9, 1))\n        truth = dict.fromkeys(keys, 0)\n        truth[\'17\', date(2025, 9, 1), 8] = 100\n        prediction = dict.fromkeys(keys, 0)\n        prediction[\'17\', date(2025, 9, 1), 18] = 100\n        d = diagnostics(keys, truth, prediction)[\'17\']\n        self.assertEqual(d[\'daily_total_absolute_error\'], 0)\n        self.assertEqual(d[\'hourly_absolute_error\'], 200)\n        self.assertEqual(d[\'by_daypart\'][\'morning_06_10\'][\'bias\'], -100)\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n', 'ml/forecast/education_competition.py': '"""Historical school/daycare POIs and calendar scenarios, with separate ablations."""\nimport argparse\nimport json\nimport pickle\nfrom datetime import date\nfrom pathlib import Path\nimport numpy as np\nfrom . import network_competition as nc\nfrom .network_regimes import state,STATE_NAMES,NORMAL\nfrom .hourly_experts import HourlyExperts,HourlyExpertEnsemble,combine\nfrom .weather_competition import corrected,weather_rows,WEIGHTS\nfrom .core import grid,load_labels,sha256\nfrom .run import FIRST,FINAL_CUTOFF\nfrom ml.submit.adapter import read_sample,write_submission\n\nfrom .education_features import REGISTRY,SOURCES,calendar_features,exposure_features\nBEST_HASH=\'b7c3dcbabc8d0915ac57ba9932fb8ee75b4f97d052fdba011d58270e1edc24a2\'\nTARGETS=(\'1\',\'11\',\'12\',\'17\',\'25\',\'26\',\'28\')\n\ndef active(key):return key[0] in TARGETS\n\ndef matrix(keys,base,weather,arm):\n    x=nc.features(keys,base,weather,True,True)\n    if arm in (\'calendar\',\'education\'):x=np.column_stack((x,np.asarray([calendar_features(k) for k in keys])))\n    if arm==\'education\':x=np.column_stack((x,np.asarray([exposure_features(k) for k in keys])))\n    return x\n\ndef fit_pair(bank,cutoff,weather):\n    import lightgbm as lgb\n    keys,bases,y,blocks=bank.training(cutoff);base=bases[\'network\']\n    scale=np.asarray([max(50,base[k]) for k in keys]);target=(y-np.asarray([base[k] for k in keys]))/scale\n    models={}\n    state_column=matrix(keys[:1],base,weather,\'control\').shape[1]-1\n    for name in (\'control\',\'calendar\',\'education\'):\n        model=lgb.LGBMRegressor(objective=\'regression_l1\',n_estimators=220,learning_rate=.025,num_leaves=12,min_child_samples=120,reg_lambda=10.,max_depth=5,n_jobs=2,verbosity=-1,random_state=2025)\n        model.fit(matrix(keys,base,weather,name),target,sample_weight=scale,categorical_feature=[0,1,2,3,4,5,state_column])\n        models[name]=model\n    return models,blocks\n\ndef variants(keys,base,incumbent,models,weather):\n    result={\'incumbent\':incumbent}\n    for name in (\'control\',\'calendar\',\'education\'):\n        raw=models[name].predict(matrix(keys,base,weather,name))\n        for w in WEIGHTS:\n            p=corrected(keys,base,raw,w)\n            result[f\'{name}_{w}\']={k:p[k] if active(k) else incumbent[k] for k in keys}\n    return result\n\ndef choose(keys,truth,preds,oct_keys,oct_preds):\n    selected={};audit={}\n    for route in TARGETS:\n        groups=[[k for k in keys if k[0]==route and k[1].month==m] for m in (9,10)]\n        errors={n:[sum(abs(p[k]-truth.get(k,0)) for k in g) for g in groups] for n,p in preds.items()}\n        oe={n:sum(abs(p[k]-truth.get(k,0)) for k in oct_keys if k[0]==route) for n,p in oct_preds.items()}\n        def beats(n,c):return all(a<=.995*b for a,b in zip(errors[n],errors[c])) and oe[n]<=1.005*oe[c]\n        calendars=[f\'calendar_{w}\' for w in WEIGHTS if beats(f\'calendar_{w}\',\'incumbent\') and beats(f\'calendar_{w}\',f\'control_{w}\')]\n        best_calendar=min([\'incumbent\']+calendars,key=lambda n:sum(errors[n]))\n        education=[f\'education_{w}\' for w in WEIGHTS if beats(f\'education_{w}\',\'incumbent\') and beats(f\'education_{w}\',f\'calendar_{w}\') and beats(f\'education_{w}\',best_calendar)]\n        selected[route]=min([\'incumbent\']+calendars+education,key=lambda n:sum(errors[n]))\n        audit[route]={\'errors_sep_oct\':errors,\'errors_october_refit\':oe,\'accepted_calendar\':calendars,\'accepted_education\':education,\'selected\':selected[route]}\n    return selected,audit\n\ndef join(keys,preds,selected):return {k:preds[selected.get(k[0],\'incumbent\')][k] for k in keys}\n\nclass EducationEnsemble:\n    def __init__(self,incumbent,reference,regimes,models,weather,selected):\n        self.incumbent,self.reference,self.regimes=incumbent,reference,regimes\n        self.models,self.weather,self.selected=models,weather,selected\n    def predict(self,keys):\n        keys=list(keys);old=self.incumbent.predict(keys)\n        base=self.regimes.apply(keys,self.reference.predict(keys))\n        replacements={k:(k[0],date(2025,11,2),k[2]) for k in keys\n                      if k[0] in (\'7\',\'50\') and k[1] in (date(2025,11,3),date(2025,11,4))}\n        proxies=list(dict.fromkeys(replacements.values()))\n        if proxies:\n            restricted=self.regimes.apply(proxies,self.reference.predict(proxies))\n            base.update({k:restricted[v] for k,v in replacements.items()})\n        return join(keys,variants(keys,base,old,self.models,self.weather),self.selected)\n\ndef main(argv=None):\n    ap=argparse.ArgumentParser()\n    for name in (\'train\',\'test\',\'sample\',\'weather\',\'baseline-dir\',\'out\'):ap.add_argument(\'--\'+name,type=Path,required=True)\n    ap.add_argument(\'--sample-from-reference\',action=\'store_true\');args=ap.parse_args(argv)\n    inputs=nc.verify_competition_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')},args.sample_from_reference)\n    values=load_labels([args.train,args.test]);assert all(FIRST<=k[1]<=FINAL_CUTOFF for k in values)\n    previous=json.loads((args.baseline_dir/\'hourly_experts_report.json\').read_text())\n    if sha256(args.baseline_dir/\'submission_hourly_holiday_service.csv\')!=BEST_HASH:raise ValueError(\'Baseline differs from verified 0.88078 submission\')\n    ref=previous[\'reference\'];choices=previous[\'selection\'][\'by_route\'];hourly=previous[\'hourly_experts\'][\'choices\']\n    weather=weather_rows(json.loads(args.weather.read_text()));bank=nc.HistoryBank(values,ref[\'choices\'])\n    args.out.mkdir(parents=True,exist_ok=True);folds={};held={}\n    for label,cutoff,start,end in [(\'sep_oct\',date(2025,8,31),date(2025,9,1),date(2025,10,31)),(\'october_refit\',date(2025,9,30),date(2025,10,1),date(2025,10,31))]:\n        print(\'Education:\',label,flush=True)\n        keys=grid(bank.routes,start,end);bases=bank.predict_bases(cutoff,keys)\n        oldmodels,_=nc.fit_models(bank,cutoff,weather)\n        incumbent=nc.assemble(keys,nc.predict_models(keys,bases,oldmodels,weather),choices)\n        hp=HourlyExperts().fit({k:v for k,v in values.items() if k[1]<=cutoff},FIRST,cutoff)\n        incumbent=combine(keys,hp.variants(keys,incumbent),hourly)\n        expected=previous[\'folds\'][label if label==\'sep_oct\' else \'october_refit_31d\'][\'hourly_on_network_weather\'][\'selected\'][\'all\']\n        if nc.metrics(keys,values,incumbent)[\'all\']!=expected:raise RuntimeError(\'Historical baseline did not reproduce\')\n        models,blocks=fit_pair(bank,cutoff,weather)\n        preds=variants(keys,bases[\'network\'],incumbent,models,weather)\n        held[label]=(keys,preds)\n        folds[label]={\'metrics\':{n:nc.metrics(keys,values,p) for n,p in preds.items()},\'training_blocks\':blocks}\n    keys,preds=held[\'sep_oct\'];oct_keys,oct_preds=held[\'october_refit\']\n    selected,audit=choose(keys,values,preds,oct_keys,oct_preds)\n    for label,(ks,ps) in held.items():folds[label][\'selected\']=nc.metrics(ks,values,join(ks,ps,selected))\n    # Fit all final model components from organizer labels, never load supplied pickle files.\n    _,sample=read_sample(args.sample);reference,regimes=bank.fitted(FINAL_CUTOFF)\n    oldmodels,_=nc.fit_models(bank,FINAL_CUTOFF,weather)\n    incumbent=nc.HolidayServiceEnsemble(nc.NetworkWeatherEnsemble(reference,regimes,oldmodels,weather,choices),reference,regimes)\n    incumbent=HourlyExpertEnsemble(incumbent,HourlyExperts().fit(values,FIRST,FINAL_CUTOFF),hourly)\n    before=incumbent.predict(sample)\n    write_submission(args.sample,before,args.out/\'submission_best_088078.csv\')\n    if sha256(args.out/\'submission_best_088078.csv\')!=BEST_HASH:raise RuntimeError(\'Final baseline hash mismatch\')\n    models,_=fit_pair(bank,FINAL_CUTOFF,weather)\n    ensemble=EducationEnsemble(incumbent,reference,regimes,models,weather,selected);after=ensemble.predict(sample)\n    assert all(after[k]==before[k] for k in sample if not active(k))\n    artifact=pickle.dumps(ensemble,protocol=5)\n    assert pickle.loads(artifact).predict(sample)==after\n    (args.out/\'model.local.pkl\').write_bytes(artifact)\n    output=write_submission(args.sample,after,args.out/\'submission_education.csv\')\n    report={\'source_mode\':\'competition_retrospective\',\'inputs\':inputs,\'registry\':REGISTRY,\'calendar_sources\':SOURCES,\'selected\':selected,\'selection\':audit,\'folds\':folds,\'gate\':\'Calendar must beat incumbent and bare control by >=0.5% per month; POI must beat incumbent, matched calendar and best admitted calendar. Oct refit deterioration <=0.5%. Routes 7/50/5 protected.\',\'selection_warning\':\'Sep-Oct reused in research. This is exploratory validation, not an independent test.\',\'best_platform_score_user_reported\':.88078,\'new_platform_score\':None,\'primary_submission\':\'submission_education.csv\',\'output\':output,\'changed_rows\':{r:sum(before[k]!=after[k] for k in sample if k[0]==r) for r in sorted({k[0] for k in sample},key=int)},\'outputs_sha256\':{n:sha256(args.out/n) for n in (\'submission_best_088078.csv\',\'submission_education.csv\')},\'weather_sha256\':sha256(args.weather),\'libraries\':previous[\'libraries\']}\n    (args.out/\'education_report.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2)+\'\\n\')\n    print(\'Selected:\',selected,\'changed rows:\',sum(report[\'changed_rows\'].values()),flush=True)\n    return report\n\nif __name__==\'__main__\':\n    from ml.forecast.education_competition import main as run_main\n    run_main()\n', 'ml/forecast/education_features.py': '"""School/daycare geographic exposure × explicitly approximate calendar scenarios."""\nimport json,math\nfrom datetime import date\nfrom pathlib import Path\nfrom .core import effective_weekday\nREGISTRY=json.loads(Path(__file__).with_name(\'education_registry.json\').read_text())\nSOURCES={\n \'quarter_2025_example\':\'https://petrovskaja-school.ru/files/education/godovoy_kalendarnyi_uchebnyi_grafic_na_2024_2025.pdf\',\n \'modular_2025_example\':\'https://st.educom.ru/eduoffices/gateways/get_file.php?id=%7BBF05FA0D-636E-CF38-639D-9C753CE71F23%7D&name=%D0%9Frotokol-16.01-03.pdf\',\n \'moscow_recommendations_2025_2026\':\'https://old-school.online/docs/op/holidays.pdf\'}\n# Separate scenario flags. No claim that any particular nearby school follows one.\nQUARTER=[(\'2025-01-01\',\'2025-01-08\'),(\'2025-03-29\',\'2025-04-06\'),(\'2025-05-27\',\'2025-08-31\'),(\'2025-10-25\',\'2025-11-02\'),(\'2025-12-31\',\'2025-12-31\')]\nMODULAR=[(\'2025-01-01\',\'2025-01-08\'),(\'2025-02-15\',\'2025-02-24\'),(\'2025-04-05\',\'2025-04-13\'),(\'2025-05-27\',\'2025-08-31\'),(\'2025-10-04\',\'2025-10-12\'),(\'2025-11-15\',\'2025-11-23\'),(\'2025-12-31\',\'2025-12-31\')]\ndef closed(d,spans):return any(date.fromisoformat(a)<=d<=date.fromisoformat(b) for a,b in spans)\ndef calendar_features(key):\n    _,d,h=key\n    if d.year!=2025:raise ValueError(\'Educational calendar defined only for 2025\')\n    work=float(effective_weekday(d)<5)\n    q=work*(not closed(d,QUARTER));m=work*(not closed(d,MODULAR))\n    # Daycare workday proxy persists through school vacations; these are hypotheses, not opening hours.\n    return [q,m,work,q*(6<=h<=9),m*(6<=h<=9),q*(12<=h<=16),m*(12<=h<=16),work*(6<=h<=9),work*(16<=h<=19)]\ndef exposure_features(key):\n    p=REGISTRY[\'routes\'].get(key[0],{\'school\':0,\'kindergarten\':0});s=math.log1p(p[\'school\']);k=math.log1p(p[\'kindergarten\']);c=calendar_features(key)\n    return [s,k]+[s*c[i] for i in (0,1,3,4,5,6)]+[k*c[i] for i in (2,7,8)]\n', 'ml/forecast/education_registry.json': '{\n  "source": "https://www.openstreetmap.org/copyright",\n  "license": "ODbL; © OpenStreetMap contributors",\n  "api": "https://overpass-api.de/api/interpreter",\n  "osm_snapshot": "2025-01-01T00:00:00Z",\n  "osm_sha256": "fd5db1a4376e94a3d7ec50e23042ffd733ae0a6fd0e83ff1cdba337cb26ae4b5",\n  "query": "[out:json][timeout:45][date:\\"2025-01-01T00:00:00Z\\"];nwr[\\"amenity\\"~\\"^(school|kindergarten)$\\"](55.584,37.44,55.898,37.834);out center tags;",\n  "gtfs_sha256": "dec3ec0ccc2efb7f1327ee6f26a0636a4a271494490e6debedcb2c1ccc579e76",\n  "radius_m": 500.0,\n  "raw_objects": 3828,\n  "retained_objects": 3717,\n  "deduplicated_ids": [\n    "relation/1284304",\n    "relation/1285682",\n    "relation/3612075",\n    "relation/7103027",\n    "relation/9897400",\n    "relation/13051361",\n    "relation/13496987",\n    "relation/17960168",\n    "way/29325329",\n    "way/29446296",\n    "way/31025605",\n    "way/37082736",\n    "way/39629328",\n    "way/40057242",\n    "way/52073302",\n    "way/96057466",\n    "way/96283968",\n    "way/96691325",\n    "way/96844626",\n    "way/98925503",\n    "way/98925579",\n    "way/105041950",\n    "way/113776912",\n    "way/117196204",\n    "way/118593455",\n    "way/129624462",\n    "way/129717682",\n    "way/130122951",\n    "way/136752940",\n    "way/154000080",\n    "way/155217711",\n    "way/162959648",\n    "way/171616601",\n    "way/179615797",\n    "way/223512281",\n    "way/227635648",\n    "way/234903720",\n    "way/282238823",\n    "way/292071887",\n    "way/304423709",\n    "way/347944650",\n    "way/377708696",\n    "way/386459645",\n    "way/398512858",\n    "way/425074014",\n    "way/459404197",\n    "way/460009813",\n    "way/460031920",\n    "way/488268384",\n    "way/568604304",\n    "way/581845124",\n    "way/591773583",\n    "way/596216286",\n    "way/596360350",\n    "way/596360354",\n    "way/596360356",\n    "way/596370524",\n    "way/632589852",\n    "way/635814043",\n    "way/641355975",\n    "way/647183330",\n    "way/650555423",\n    "way/655682580",\n    "way/656685218",\n    "way/656685220",\n    "way/690243296",\n    "way/707938467",\n    "way/881232953",\n    "way/1042681701",\n    "way/1130649958",\n    "way/1130649959",\n    "way/1144177518",\n    "way/1154845324",\n    "way/1155008743",\n    "way/1193335196",\n    "way/1194067581",\n    "way/1194073085",\n    "way/1211187706",\n    "way/1214119136",\n    "node/1094884834",\n    "node/1373636953",\n    "node/1373694625",\n    "node/1373755232",\n    "node/1373766030",\n    "node/1373778974",\n    "node/1428214514",\n    "node/1429367710",\n    "node/1456659417",\n    "node/2262304592",\n    "node/2273992710",\n    "node/2273992711",\n    "node/2942619205",\n    "node/2986435369",\n    "node/4381977804",\n    "node/4520379715",\n    "node/4752645623",\n    "node/5180531055",\n    "node/5199351212",\n    "node/5238076005",\n    "node/5732129330",\n    "node/6331788808",\n    "node/6413499948",\n    "node/6413499949",\n    "node/6794888867",\n    "node/7218108678",\n    "node/7223599185",\n    "node/7241782461",\n    "node/8154382975",\n    "node/8752581937",\n    "node/8752581938",\n    "node/9910450170"\n  ],\n  "routes": {\n    "17": {\n      "school": 27,\n      "kindergarten": 35,\n      "poi_ids": [\n        "relation/286862",\n        "relation/1270060",\n        "relation/1490894",\n        "relation/1535976",\n        "relation/1621860",\n        "relation/2108267",\n        "relation/2432227",\n        "relation/2432228",\n        "relation/2432229",\n        "relation/2589367",\n        "relation/3015734",\n        "relation/3015735",\n        "relation/3428535",\n        "relation/3428539",\n        "relation/5699016",\n        "relation/6010660",\n        "relation/9068415",\n        "relation/9068416",\n        "relation/9070447",\n        "relation/9070448",\n        "relation/9071344",\n        "relation/11550302",\n        "relation/12223764",\n        "relation/12223765",\n        "relation/17789214",\n        "way/48757559",\n        "way/59370210",\n        "way/59404034",\n        "way/59409572",\n        "way/60093986",\n        "way/69728519",\n        "way/85910107",\n        "way/102317311",\n        "way/102317313",\n        "way/102326794",\n        "way/117368268",\n        "way/226915647",\n        "way/226915648",\n        "way/227478963",\n        "way/227488072",\n        "way/241167631",\n        "way/255945625",\n        "way/270445570",\n        "way/302728535",\n        "way/325989403",\n        "way/378894408",\n        "way/378894409",\n        "way/400804215",\n        "way/445522765",\n        "way/445522767",\n        "way/469126286",\n        "way/499308094",\n        "way/532936533",\n        "way/532939500",\n        "way/618533762",\n        "way/709265694",\n        "way/709265695",\n        "way/709548475",\n        "way/867853165",\n        "way/1163425107",\n        "node/3794027238",\n        "node/4575871390"\n      ],\n      "known_opening_hours": 1\n    },\n    "1": {\n      "school": 33,\n      "kindergarten": 43,\n      "poi_ids": [\n        "relation/2074057",\n        "relation/2074058",\n        "relation/5713567",\n        "relation/5713568",\n        "relation/9310365",\n        "relation/10165007",\n        "relation/13211681",\n        "relation/13211682",\n        "relation/13372228",\n        "relation/13372229",\n        "relation/13524468",\n        "relation/13524469",\n        "relation/13606724",\n        "way/48227462",\n        "way/51812359",\n        "way/51832350",\n        "way/51832351",\n        "way/52015581",\n        "way/76593145",\n        "way/88609750",\n        "way/93181279",\n        "way/97716798",\n        "way/97730991",\n        "way/97730993",\n        "way/107728025",\n        "way/111538985",\n        "way/111538986",\n        "way/115613768",\n        "way/129475181",\n        "way/146603937",\n        "way/146603939",\n        "way/154047644",\n        "way/154047649",\n        "way/154331611",\n        "way/154331612",\n        "way/157210046",\n        "way/183948096",\n        "way/205680825",\n        "way/260177247",\n        "way/284385908",\n        "way/284401169",\n        "way/286082075",\n        "way/287304987",\n        "way/288601028",\n        "way/288609547",\n        "way/293518110",\n        "way/295308773",\n        "way/330781480",\n        "way/330783273",\n        "way/333465880",\n        "way/367879924",\n        "way/377728857",\n        "way/390483185",\n        "way/396923766",\n        "way/399595392",\n        "way/399595394",\n        "way/400514902",\n        "way/401282468",\n        "way/401282470",\n        "way/404373621",\n        "way/404373622",\n        "way/404379796",\n        "way/404896628",\n        "way/404896629",\n        "way/404896633",\n        "way/405311002",\n        "way/405413167",\n        "way/461030275",\n        "way/498121462",\n        "way/498175708",\n        "way/498314470",\n        "way/595122415",\n        "way/608127845",\n        "way/693839893",\n        "way/711304803",\n        "way/711304806"\n      ],\n      "known_opening_hours": 0\n    },\n    "12": {\n      "school": 51,\n      "kindergarten": 65,\n      "poi_ids": [\n        "relation/2018554",\n        "relation/2574596",\n        "relation/5713478",\n        "relation/6847000",\n        "relation/7329513",\n        "relation/7329517",\n        "relation/7334650",\n        "relation/7334651",\n        "relation/7814787",\n        "relation/8897320",\n        "relation/8897321",\n        "relation/8897322",\n        "relation/11186468",\n        "relation/11284691",\n        "relation/11342614",\n        "relation/12661655",\n        "relation/14237997",\n        "relation/15284663",\n        "relation/15285093",\n        "relation/18244733",\n        "way/23768961",\n        "way/27575988",\n        "way/29601128",\n        "way/43321808",\n        "way/47572148",\n        "way/48346415",\n        "way/49140332",\n        "way/50059100",\n        "way/50072669",\n        "way/52462301",\n        "way/52462352",\n        "way/52462356",\n        "way/52463398",\n        "way/64612074",\n        "way/78273780",\n        "way/78444429",\n        "way/78641368",\n        "way/79534444",\n        "way/80033791",\n        "way/93527752",\n        "way/94054910",\n        "way/105137093",\n        "way/128531064",\n        "way/128531081",\n        "way/129318472",\n        "way/129408541",\n        "way/129408542",\n        "way/129416914",\n        "way/129584044",\n        "way/129585077",\n        "way/130278483",\n        "way/130819428",\n        "way/133738328",\n        "way/145111581",\n        "way/145111582",\n        "way/145111583",\n        "way/150964760",\n        "way/168840466",\n        "way/186311028",\n        "way/190098593",\n        "way/190118996",\n        "way/196094633",\n        "way/204059721",\n        "way/297179085",\n        "way/309413338",\n        "way/362529929",\n        "way/362924386",\n        "way/372687745",\n        "way/395166265",\n        "way/395166267",\n        "way/395236550",\n        "way/395236552",\n        "way/395236553",\n        "way/399276902",\n        "way/425202129",\n        "way/425304688",\n        "way/429427494",\n        "way/444428136",\n        "way/444428137",\n        "way/450112055",\n        "way/454469587",\n        "way/461358380",\n        "way/482435459",\n        "way/524737858",\n        "way/640129189",\n        "way/824290373",\n        "way/878383228",\n        "way/878979417",\n        "way/878979512",\n        "way/1198830522",\n        "way/1285516011",\n        "node/1252450767",\n        "node/1444803462",\n        "node/1444819629",\n        "node/1456162724",\n        "node/1472892444",\n        "node/1504923799",\n        "node/1504923815",\n        "node/1588932818",\n        "node/1602410893",\n        "node/1719646868",\n        "node/1858733747",\n        "node/2146273545",\n        "node/2661667230",\n        "node/4243926389",\n        "node/4588995214",\n        "node/4792431751",\n        "node/5199351213",\n        "node/5273631003",\n        "node/5292569524",\n        "node/5305719149",\n        "node/6866820508",\n        "node/8920785773",\n        "node/9880328119",\n        "node/11085165913",\n        "node/11085236054"\n      ],\n      "known_opening_hours": 4\n    },\n    "11": {\n      "school": 41,\n      "kindergarten": 48,\n      "poi_ids": [\n        "relation/1262955",\n        "relation/1346265",\n        "relation/2108267",\n        "relation/2574596",\n        "relation/2752454",\n        "relation/7814787",\n        "relation/9068415",\n        "relation/9068416",\n        "relation/9070447",\n        "relation/9070448",\n        "relation/9071344",\n        "relation/11550302",\n        "relation/11679771",\n        "relation/11778070",\n        "relation/11805313",\n        "relation/14237997",\n        "relation/17087636",\n        "relation/17087637",\n        "relation/17789214",\n        "relation/18244733",\n        "way/23768961",\n        "way/28858683",\n        "way/50059100",\n        "way/50072669",\n        "way/51239305",\n        "way/51239411",\n        "way/51422928",\n        "way/64612074",\n        "way/78273780",\n        "way/78444429",\n        "way/78641368",\n        "way/79534444",\n        "way/94054910",\n        "way/96645733",\n        "way/105704716",\n        "way/128531064",\n        "way/128531081",\n        "way/133738328",\n        "way/148175913",\n        "way/154363995",\n        "way/154363996",\n        "way/154544725",\n        "way/155653757",\n        "way/168840466",\n        "way/186311028",\n        "way/190098593",\n        "way/190118996",\n        "way/204059721",\n        "way/302728535",\n        "way/362529929",\n        "way/362924386",\n        "way/395166265",\n        "way/395166267",\n        "way/395236550",\n        "way/395236552",\n        "way/395236553",\n        "way/425202129",\n        "way/425304688",\n        "way/450112055",\n        "way/454469587",\n        "way/461358380",\n        "way/504270193",\n        "way/520900483",\n        "way/524737858",\n        "way/555999588",\n        "way/555999622",\n        "way/709206068",\n        "way/711588628",\n        "way/867853165",\n        "way/878383228",\n        "way/878979417",\n        "way/878979512",\n        "node/1444803462",\n        "node/1444819629",\n        "node/1456162724",\n        "node/1472892444",\n        "node/1504923799",\n        "node/1504923815",\n        "node/1588932818",\n        "node/1719646868",\n        "node/4243926389",\n        "node/4393094689",\n        "node/4959152543",\n        "node/5199351213",\n        "node/5273631003",\n        "node/5305719149",\n        "node/6866820508",\n        "node/10044484701",\n        "node/11780110793"\n      ],\n      "known_opening_hours": 2\n    },\n    "25": {\n      "school": 16,\n      "kindergarten": 15,\n      "poi_ids": [\n        "relation/1262955",\n        "relation/1346265",\n        "relation/2108267",\n        "relation/7712039",\n        "relation/9068415",\n        "relation/9068416",\n        "relation/9070447",\n        "relation/9070448",\n        "relation/9071344",\n        "relation/10524778",\n        "relation/10524779",\n        "relation/10698703",\n        "relation/11126283",\n        "relation/11152601",\n        "relation/11257412",\n        "relation/11331272",\n        "relation/11331273",\n        "relation/11550302",\n        "relation/11778070",\n        "relation/11805313",\n        "relation/17789214",\n        "way/96645733",\n        "way/149908800",\n        "way/166477542",\n        "way/179615791",\n        "way/187448839",\n        "way/302728535",\n        "way/555999588",\n        "way/555999622",\n        "way/867853165",\n        "node/5465097276"\n      ],\n      "known_opening_hours": 0\n    },\n    "50": {\n      "school": 29,\n      "kindergarten": 37,\n      "poi_ids": [\n        "relation/1476090",\n        "relation/4729216",\n        "relation/6319979",\n        "relation/6476664",\n        "relation/6505043",\n        "relation/7103030",\n        "relation/7238772",\n        "relation/7268814",\n        "relation/7595822",\n        "relation/7895065",\n        "relation/9656985",\n        "relation/11186468",\n        "relation/11284691",\n        "relation/11370208",\n        "relation/11855555",\n        "relation/11997800",\n        "relation/12003323",\n        "relation/12476264",\n        "relation/13188995",\n        "relation/13470035",\n        "relation/13802230",\n        "relation/14956586",\n        "relation/14956589",\n        "relation/14977370",\n        "relation/15046320",\n        "relation/15285093",\n        "relation/15326688",\n        "relation/15451606",\n        "relation/15968755",\n        "relation/16270432",\n        "relation/16409205",\n        "relation/17057701",\n        "relation/17824771",\n        "relation/17940119",\n        "way/46740734",\n        "way/49908065",\n        "way/49908070",\n        "way/50548821",\n        "way/125849340",\n        "way/129760307",\n        "way/139617929",\n        "way/172288880",\n        "way/172839742",\n        "way/188079728",\n        "way/189136645",\n        "way/189136646",\n        "way/192393322",\n        "way/196952681",\n        "way/197907188",\n        "way/372687745",\n        "way/482435459",\n        "way/487787682",\n        "way/523390513",\n        "way/539236815",\n        "way/684216820",\n        "way/824290373",\n        "node/1429376733",\n        "node/1432138391",\n        "node/1858733747",\n        "node/1867012052",\n        "node/4802013987",\n        "node/5305719158",\n        "node/8920785773",\n        "node/9880328119",\n        "node/11085165913",\n        "node/11085236054"\n      ],\n      "known_opening_hours": 4\n    },\n    "7": {\n      "school": 43,\n      "kindergarten": 57,\n      "poi_ids": [\n        "relation/1283096",\n        "relation/1403786",\n        "relation/1407665",\n        "relation/1708757",\n        "relation/1708763",\n        "relation/2752454",\n        "relation/4719459",\n        "relation/4729216",\n        "relation/6505043",\n        "relation/7103029",\n        "relation/7238772",\n        "relation/7268814",\n        "relation/7595822",\n        "relation/7895065",\n        "relation/9656985",\n        "relation/10524778",\n        "relation/10524779",\n        "relation/10524782",\n        "relation/10698703",\n        "relation/11208335",\n        "relation/11257412",\n        "relation/11268549",\n        "relation/11268551",\n        "relation/11370208",\n        "relation/11679771",\n        "relation/11855555",\n        "relation/12106667",\n        "relation/12476264",\n        "relation/13188995",\n        "relation/13470035",\n        "relation/13936495",\n        "relation/14680298",\n        "relation/15046320",\n        "relation/15968755",\n        "relation/16270432",\n        "relation/17057701",\n        "relation/17087636",\n        "relation/17087637",\n        "relation/17940119",\n        "way/28858683",\n        "way/37927642",\n        "way/51239305",\n        "way/51239411",\n        "way/51422928",\n        "way/51480004",\n        "way/96300065",\n        "way/101264114",\n        "way/101264120",\n        "way/105704716",\n        "way/115707848",\n        "way/125849340",\n        "way/141643616",\n        "way/141643619",\n        "way/141643622",\n        "way/141643633",\n        "way/141643636",\n        "way/141643639",\n        "way/148175913",\n        "way/149908800",\n        "way/153236806",\n        "way/154000034",\n        "way/154000076",\n        "way/154000082",\n        "way/154000083",\n        "way/154343006",\n        "way/154363995",\n        "way/154363996",\n        "way/154544725",\n        "way/155551279",\n        "way/155653703",\n        "way/155653757",\n        "way/166477542",\n        "way/172288880",\n        "way/172839742",\n        "way/185426891",\n        "way/188079728",\n        "way/189136645",\n        "way/189136646",\n        "way/192393322",\n        "way/196952681",\n        "way/197907188",\n        "way/487787682",\n        "way/504270193",\n        "way/520900483",\n        "way/523390513",\n        "way/539236815",\n        "way/709206068",\n        "way/711588628",\n        "way/813559738",\n        "node/1427996071",\n        "node/1429376733",\n        "node/1432138391",\n        "node/4393094689",\n        "node/4959152543",\n        "node/5113395065",\n        "node/5334698649",\n        "node/5465097276",\n        "node/7257951974",\n        "node/10044484701",\n        "node/11780110793"\n      ],\n      "known_opening_hours": 3\n    },\n    "28": {\n      "school": 15,\n      "kindergarten": 20,\n      "poi_ids": [\n        "relation/1656086",\n        "relation/1697080",\n        "relation/1819043",\n        "relation/6376832",\n        "relation/6702161",\n        "relation/8119012",\n        "relation/8119013",\n        "relation/17232824",\n        "relation/17695569",\n        "way/27800688",\n        "way/27810681",\n        "way/27810687",\n        "way/37943640",\n        "way/52456628",\n        "way/59605079",\n        "way/66860549",\n        "way/85641529",\n        "way/103244527",\n        "way/105589602",\n        "way/106406165",\n        "way/108201816",\n        "way/108315230",\n        "way/108323089",\n        "way/108446906",\n        "way/114533462",\n        "way/114540677",\n        "way/120831644",\n        "way/146398902",\n        "way/146696317",\n        "way/244039366",\n        "way/305671071",\n        "way/643329688",\n        "way/952069499",\n        "node/8752557700",\n        "node/9687922817"\n      ],\n      "known_opening_hours": 1\n    },\n    "26": {\n      "school": 35,\n      "kindergarten": 47,\n      "poi_ids": [\n        "relation/6890548",\n        "relation/7220450",\n        "relation/7738645",\n        "relation/13176656",\n        "relation/13983628",\n        "relation/14606511",\n        "relation/14926150",\n        "relation/18134842",\n        "relation/18178325",\n        "relation/18212679",\n        "way/29127999",\n        "way/29128001",\n        "way/35977653",\n        "way/35977707",\n        "way/37030105",\n        "way/46657618",\n        "way/48399916",\n        "way/51684041",\n        "way/51684048",\n        "way/81950814",\n        "way/82137808",\n        "way/89297039",\n        "way/96447491",\n        "way/96447494",\n        "way/102781292",\n        "way/104635222",\n        "way/108715365",\n        "way/112390777",\n        "way/112390793",\n        "way/112390798",\n        "way/112390800",\n        "way/130630559",\n        "way/152795922",\n        "way/158895386",\n        "way/168744868",\n        "way/170304996",\n        "way/173206552",\n        "way/184132673",\n        "way/184132674",\n        "way/191905193",\n        "way/194309723",\n        "way/234744383",\n        "way/277070499",\n        "way/306239412",\n        "way/306239414",\n        "way/316630443",\n        "way/316835898",\n        "way/329889832",\n        "way/334658366",\n        "way/355662159",\n        "way/373614960",\n        "way/373734133",\n        "way/374271122",\n        "way/382029609",\n        "way/405103977",\n        "way/425074005",\n        "way/440610765",\n        "way/496457582",\n        "way/586808383",\n        "way/608667848",\n        "way/710644450",\n        "way/722407298",\n        "way/977876539",\n        "way/1122007526",\n        "way/1235854567",\n        "node/416876296",\n        "node/954912765",\n        "node/1277580872",\n        "node/1373669698",\n        "node/1373669718",\n        "node/1373669722",\n        "node/1373669725",\n        "node/1373669735",\n        "node/1373669736",\n        "node/1373768269",\n        "node/1373768284",\n        "node/1373768288",\n        "node/1427855785",\n        "node/1428076955",\n        "node/3741570155",\n        "node/3741570172",\n        "node/10810258626"\n      ],\n      "known_opening_hours": 2\n    }\n  },\n  "poi": [\n    {\n      "id": "relation/36980",\n      "kind": "school",\n      "name": "Школа № 1288, корпус № 6",\n      "lat": 55.7782163,\n      "lon": 37.5292095,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/62986",\n      "kind": "school",\n      "name": "Школа № 1141",\n      "lat": 55.6435543,\n      "lon": 37.7188551,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/62987",\n      "kind": "school",\n      "name": "Школа №499",\n      "lat": 55.643592,\n      "lon": 37.7205177,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/286862",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8536875,\n      "lon": 37.6593748,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/337465",\n      "kind": "school",\n      "name": "ГБОУ Школа в Капотне (Учебный корпус \\"Боровицкий\\")",\n      "lat": 55.631654,\n      "lon": 37.7968531,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/358784",\n      "kind": "school",\n      "name": "ГБОУ школа № 1363, корпус № 1",\n      "lat": 55.7077963,\n      "lon": 37.8237939,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/359070",\n      "kind": "school",\n      "name": "Школа-интернат им. Ю.В.Никулина",\n      "lat": 55.7043499,\n      "lon": 37.7902112,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1245766",\n      "kind": "school",\n      "name": "Центр образования №1470",\n      "lat": 55.8139487,\n      "lon": 37.644921,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1249599",\n      "kind": "school",\n      "name": "Центр педагогического мастерства",\n      "lat": 55.7898209,\n      "lon": 37.6212747,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1249600",\n      "kind": "kindergarten",\n      "name": "Детский сад №1189",\n      "lat": 55.7892559,\n      "lon": 37.619603,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1262955",\n      "kind": "school",\n      "name": "Школа № 1503",\n      "lat": 55.8181393,\n      "lon": 37.6628376,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1270058",\n      "kind": "school",\n      "name": "Школа №1095",\n      "lat": 55.8721435,\n      "lon": 37.6703797,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1270060",\n      "kind": "school",\n      "name": "Школа №757",\n      "lat": 55.8708491,\n      "lon": 37.6696544,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1270065",\n      "kind": "school",\n      "name": "Школа №1381",\n      "lat": 55.8729153,\n      "lon": 37.6912967,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1280704",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 12",\n      "lat": 55.8527736,\n      "lon": 37.6536023,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1280705",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 5",\n      "lat": 55.852089,\n      "lon": 37.6542093,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1283096",\n      "kind": "kindergarten",\n      "name": "СПДО № 1 школы № 1784",\n      "lat": 55.7779019,\n      "lon": 37.5762138,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1284256",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7798937,\n      "lon": 37.4917759,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1284257",\n      "kind": "school",\n      "name": "Школа №100",\n      "lat": 55.7743882,\n      "lon": 37.4855824,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1284265",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7795023,\n      "lon": 37.4850428,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1285664",\n      "kind": "school",\n      "name": "Гимназия №1522",\n      "lat": 55.7730106,\n      "lon": 37.4822545,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1285674",\n      "kind": "school",\n      "name": "ГБОУ СОШ Школа №1010 Корпус №3",\n      "lat": 55.7708868,\n      "lon": 37.4788264,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1285680",\n      "kind": "school",\n      "name": "",\n      "lat": 55.770442,\n      "lon": 37.4841783,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1285729",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7721677,\n      "lon": 37.4790474,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1317290",\n      "kind": "school",\n      "name": "Школа-интернат №11",\n      "lat": 55.7696609,\n      "lon": 37.4715224,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1338486",\n      "kind": "school",\n      "name": "Школа «Потенциал»",\n      "lat": 55.8051799,\n      "lon": 37.6405551,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1346265",\n      "kind": "school",\n      "name": "Школа № 1503",\n      "lat": 55.8186977,\n      "lon": 37.6597964,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1380084",\n      "kind": "kindergarten",\n      "name": "Детский сад №1824",\n      "lat": 55.7724937,\n      "lon": 37.4855377,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1380144",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7792391,\n      "lon": 37.4983553,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1403786",\n      "kind": "school",\n      "name": "Лицей № 1574. ОП № 1",\n      "lat": 55.7745603,\n      "lon": 37.5905282,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1407665",\n      "kind": "school",\n      "name": "Центр образования №1447",\n      "lat": 55.7797874,\n      "lon": 37.5880813,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1412042",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6761828,\n      "lon": 37.6954364,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1412043",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6773853,\n      "lon": 37.7000029,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1412051",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6809905,\n      "lon": 37.7061743,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1421038",\n      "kind": "kindergarten",\n      "name": "Школа № 2054. Дошкольный корпус № 3",\n      "lat": 55.7697175,\n      "lon": 37.5995896,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1421089",\n      "kind": "school",\n      "name": "Школа № 1574. Корпус Старопименовский",\n      "lat": 55.7692423,\n      "lon": 37.600585,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1427143",\n      "kind": "kindergarten",\n      "name": "Детский сад №513",\n      "lat": 55.7707493,\n      "lon": 37.6032283,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1450020",\n      "kind": "school",\n      "name": "ГБОУ Школа в Капотне (Учебный корпус \\"Камергерский\\")",\n      "lat": 55.6301933,\n      "lon": 37.7967704,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1459232",\n      "kind": "kindergarten",\n      "name": "Детский сад №2140",\n      "lat": 55.8146776,\n      "lon": 37.6445515,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1459233",\n      "kind": "kindergarten",\n      "name": "Детский сад №1320",\n      "lat": 55.8153467,\n      "lon": 37.6447063,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1459416",\n      "kind": "kindergarten",\n      "name": "Детский сад №1342 «Солнышко»",\n      "lat": 55.8150162,\n      "lon": 37.6505895,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1476090",\n      "kind": "school",\n      "name": "Школа «Содружество». Учебный корпус №1",\n      "lat": 55.7574312,\n      "lon": 37.7011875,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1490894",\n      "kind": "school",\n      "name": "Детская хоровая школа «Весна»",\n      "lat": 55.8706346,\n      "lon": 37.6480796,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1514803",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8776422,\n      "lon": 37.6718289,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1516929",\n      "kind": "school",\n      "name": "Школа №309",\n      "lat": 55.8632082,\n      "lon": 37.6793466,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1516932",\n      "kind": "school",\n      "name": "",\n      "lat": 55.866711,\n      "lon": 37.677224,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1535976",\n      "kind": "school",\n      "name": "Школа №281",\n      "lat": 55.8665021,\n      "lon": 37.6703141,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1587283",\n      "kind": "school",\n      "name": "Центр образования №1816",\n      "lat": 55.7367864,\n      "lon": 37.4937056,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1601011",\n      "kind": "kindergarten",\n      "name": "Школа № 1239, дошкольное отделение",\n      "lat": 55.7612391,\n      "lon": 37.5953884,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1607477",\n      "kind": "kindergarten",\n      "name": "Школа № 1950 с углубленным изучением отдельных предметов. Дошкольное отделение",\n      "lat": 55.7717618,\n      "lon": 37.5855241,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1621860",\n      "kind": "kindergarten",\n      "name": "Школа №1499 (дошкольное отделение №6)",\n      "lat": 55.8356163,\n      "lon": 37.6644287,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1640597",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8356414,\n      "lon": 37.6660974,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1642492",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6767585,\n      "lon": 37.5112879,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1650088",\n      "kind": "school",\n      "name": "Школа №426",\n      "lat": 55.6245364,\n      "lon": 37.7192144,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1655777",\n      "kind": "school",\n      "name": "Центр образования №117",\n      "lat": 55.6693832,\n      "lon": 37.534134,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1656086",\n      "kind": "school",\n      "name": "Гимназия №1517",\n      "lat": 55.7802492,\n      "lon": 37.4571477,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1664990",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.770405,\n      "lon": 37.4776106,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1697080",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8064323,\n      "lon": 37.5145754,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1708757",\n      "kind": "school",\n      "name": "Романовская школа",\n      "lat": 55.7728531,\n      "lon": 37.5820939,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1708763",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 1 «Звёздочка»",\n      "lat": 55.7733392,\n      "lon": 37.5809738,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1751600",\n      "kind": "school",\n      "name": "Центр образования №1441«Бронная слобода»",\n      "lat": 55.7620395,\n      "lon": 37.5974549,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1751601",\n      "kind": "kindergarten",\n      "name": "Детский сад №1144",\n      "lat": 55.7603816,\n      "lon": 37.5956879,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1752165",\n      "kind": "kindergarten",\n      "name": "Детский сад №3",\n      "lat": 55.7630146,\n      "lon": 37.5994673,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1757153",\n      "kind": "kindergarten",\n      "name": "Детский сад №1684",\n      "lat": 55.8721575,\n      "lon": 37.6815924,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1757154",\n      "kind": "school",\n      "name": "Школа №298",\n      "lat": 55.8724788,\n      "lon": 37.6805273,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1763003",\n      "kind": "school",\n      "name": "Школа №1929",\n      "lat": 55.6353589,\n      "lon": 37.743337,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1763004",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6345315,\n      "lon": 37.743037,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1763005",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6302235,\n      "lon": 37.6521784,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1763006",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6320931,\n      "lon": 37.6519736,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1763007",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6309712,\n      "lon": 37.6520773,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1819043",\n      "kind": "school",\n      "name": "Школа № 149 имени Героя Советского Союза Ю. Н. Зыкова",\n      "lat": 55.8057513,\n      "lon": 37.5009076,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1819916",\n      "kind": "school",\n      "name": "Школа №648",\n      "lat": 55.8554885,\n      "lon": 37.5022595,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1836516",\n      "kind": "school",\n      "name": "Школа №1043",\n      "lat": 55.6469914,\n      "lon": 37.7488536,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1840315",\n      "kind": "school",\n      "name": "Центр образования №1685",\n      "lat": 55.7475168,\n      "lon": 37.6526198,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1841866",\n      "kind": "kindergarten",\n      "name": "Детский сад №2324",\n      "lat": 55.6478415,\n      "lon": 37.7509163,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1873600",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7716519,\n      "lon": 37.4844162,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1881807",\n      "kind": "kindergarten",\n      "name": "Детский сад №2709 \\"Совёнок\\"",\n      "lat": 55.7745419,\n      "lon": 37.4902214,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930228",\n      "kind": "kindergarten",\n      "name": "Детский сад Феникс",\n      "lat": 55.5938026,\n      "lon": 37.6695414,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930229",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №8",\n      "lat": 55.5936898,\n      "lon": 37.6680902,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930230",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №11",\n      "lat": 55.591127,\n      "lon": 37.6681777,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930231",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №10",\n      "lat": 55.5910223,\n      "lon": 37.6667322,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930233",\n      "kind": "school",\n      "name": "Школа № 902 \\"Диалог\\", здание №3",\n      "lat": 55.5913274,\n      "lon": 37.6700217,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1930234",\n      "kind": "school",\n      "name": "Школа № 902 \\"Диалог\\", здание №2",\n      "lat": 55.593533,\n      "lon": 37.6660668,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/1998795",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa01368",\n      "lat": 55.6755258,\n      "lon": 37.6900136,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2018554",\n      "kind": "school",\n      "name": "Школа №1274",\n      "lat": 55.7233969,\n      "lon": 37.67137,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2074057",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Открытие»",\n      "lat": 55.6295375,\n      "lon": 37.5957873,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2074058",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус школьного образования №2",\n      "lat": 55.6300278,\n      "lon": 37.5945952,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2102794",\n      "kind": "school",\n      "name": "Школа №1314",\n      "lat": 55.7130695,\n      "lon": 37.7891315,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2102795",\n      "kind": "school",\n      "name": "Школа № 2090",\n      "lat": 55.713451,\n      "lon": 37.7879981,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2108267",\n      "kind": "school",\n      "name": "Школа №1415",\n      "lat": 55.8246563,\n      "lon": 37.6247623,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2140115",\n      "kind": "school",\n      "name": "ГБОУ школа № 507, здание № 3",\n      "lat": 55.6673747,\n      "lon": 37.6542195,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2145798",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.708118,\n      "lon": 37.4609493,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2145799",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7078435,\n      "lon": 37.4619846,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2145800",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7075471,\n      "lon": 37.4630171,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2168335",\n      "kind": "kindergarten",\n      "name": "Детский сад №1541",\n      "lat": 55.8908275,\n      "lon": 37.6768881,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2168336",\n      "kind": "kindergarten",\n      "name": "Детский сад №1495",\n      "lat": 55.8899927,\n      "lon": 37.6769625,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2183431",\n      "kind": "school",\n      "name": "Школа № 171 Школьное отделение № 2",\n      "lat": 55.7194291,\n      "lon": 37.5750929,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2183433",\n      "kind": "kindergarten",\n      "name": "Детский сад №1155",\n      "lat": 55.7176475,\n      "lon": 37.575196,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2235992",\n      "kind": "school",\n      "name": "Школа № 305",\n      "lat": 55.8755815,\n      "lon": 37.5798725,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2236017",\n      "kind": "kindergarten",\n      "name": "Детский сад №172",\n      "lat": 55.8058568,\n      "lon": 37.822233,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2236018",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №6 \\"Цветик-семицветик\\"",\n      "lat": 55.8056378,\n      "lon": 37.8206519,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2248443",\n      "kind": "school",\n      "name": "Школа-интернат для глухих детей №65",\n      "lat": 55.7162602,\n      "lon": 37.7377174,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2248444",\n      "kind": "school",\n      "name": "Школа-интернат №105",\n      "lat": 55.7165616,\n      "lon": 37.7402113,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2257315",\n      "kind": "kindergarten",\n      "name": "Школа № 2099. Дошкольное здание",\n      "lat": 55.816185,\n      "lon": 37.4961085,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2307304",\n      "kind": "kindergarten",\n      "name": "Детский сад №1633",\n      "lat": 55.7654883,\n      "lon": 37.490507,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2307305",\n      "kind": "school",\n      "name": "Школа №1114",\n      "lat": 55.7647688,\n      "lon": 37.4906759,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2378354",\n      "kind": "kindergarten",\n      "name": "ДОУ №1729",\n      "lat": 55.7668255,\n      "lon": 37.6678701,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2396447",\n      "kind": "school",\n      "name": "Школа № 996",\n      "lat": 55.6269965,\n      "lon": 37.7236231,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2425832",\n      "kind": "kindergarten",\n      "name": "Детский сад ЦРР №32",\n      "lat": 55.6193083,\n      "lon": 37.7384342,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2425833",\n      "kind": "school",\n      "name": "Спецшкола №991",\n      "lat": 55.6193772,\n      "lon": 37.7395421,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2432227",\n      "kind": "kindergarten",\n      "name": "Детский сад №2420",\n      "lat": 55.8679673,\n      "lon": 37.6582333,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2432228",\n      "kind": "school",\n      "name": "ГБОУ школа № 1558 имени Росалии де Кастро",\n      "lat": 55.8681585,\n      "lon": 37.6603169,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2432229",\n      "kind": "kindergarten",\n      "name": "Детский сад №1870",\n      "lat": 55.8686113,\n      "lon": 37.6585889,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2465066",\n      "kind": "school",\n      "name": "ГБОУ школа № 507, здание № 9",\n      "lat": 55.6721224,\n      "lon": 37.6572188,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2465075",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 507, Радуга (здание № 6)",\n      "lat": 55.6697752,\n      "lon": 37.65829,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2574596",\n      "kind": "kindergarten",\n      "name": "Детский сад №777",\n      "lat": 55.7857976,\n      "lon": 37.734087,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2578431",\n      "kind": "kindergarten",\n      "name": "Детский сад №696",\n      "lat": 55.8751193,\n      "lon": 37.6931984,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2589367",\n      "kind": "kindergarten",\n      "name": "Детский сад №1120",\n      "lat": 55.867757,\n      "lon": 37.6684159,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2624116",\n      "kind": "kindergarten",\n      "name": "Школа №158, дошкольное отделение №5",\n      "lat": 55.8671514,\n      "lon": 37.4653512,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2624119",\n      "kind": "kindergarten",\n      "name": "Школа №158, дошкольное отделение №4",\n      "lat": 55.8664185,\n      "lon": 37.4657795,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2631944",\n      "kind": "kindergarten",\n      "name": "Центр развития творчества детей и юношества \\"Планета\\"",\n      "lat": 55.7044491,\n      "lon": 37.5789552,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2631945",\n      "kind": "school",\n      "name": "Школа № 1266",\n      "lat": 55.7048374,\n      "lon": 37.578306,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2639185",\n      "kind": "kindergarten",\n      "name": "Школа № 1266. Дошкольное отделение",\n      "lat": 55.7040403,\n      "lon": 37.5775092,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2699560",\n      "kind": "school",\n      "name": "Центр образования №1421",\n      "lat": 55.7135878,\n      "lon": 37.7530688,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2699561",\n      "kind": "school",\n      "name": "Школа №1367",\n      "lat": 55.7146528,\n      "lon": 37.7524919,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2699563",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.71317,\n      "lon": 37.7496638,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2752454",\n      "kind": "school",\n      "name": "Школа №363 (начальные классы)",\n      "lat": 55.8087912,\n      "lon": 37.7119852,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2766945",\n      "kind": "school",\n      "name": "Центр образования № 1466 имени Надежды Рушевой",\n      "lat": 55.6293298,\n      "lon": 37.6749566,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2785111",\n      "kind": "school",\n      "name": "Школа №1257",\n      "lat": 55.7145546,\n      "lon": 37.6259982,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2838632",\n      "kind": "school",\n      "name": "Школа №878 отделение №2",\n      "lat": 55.6143271,\n      "lon": 37.6979107,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2880891",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7904238,\n      "lon": 37.5208434,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2880892",\n      "kind": "school",\n      "name": "Центр образования №1409",\n      "lat": 55.7840878,\n      "lon": 37.5315363,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2893419",\n      "kind": "school",\n      "name": "Школа №12",\n      "lat": 55.8846316,\n      "lon": 37.7432036,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2894192",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Звёздочка\\"",\n      "lat": 55.885443,\n      "lon": 37.7382802,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2896213",\n      "kind": "kindergarten",\n      "name": "Детский сад № 53",\n      "lat": 55.6947284,\n      "lon": 37.5746002,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2896247",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2243",\n      "lat": 55.6962045,\n      "lon": 37.561549,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/2896248",\n      "kind": "school",\n      "name": "Школа № 199 (начальные классы)",\n      "lat": 55.692326,\n      "lon": 37.5657951,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3015734",\n      "kind": "kindergarten",\n      "name": "Детский сад №1255",\n      "lat": 55.875249,\n      "lon": 37.649616,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3015735",\n      "kind": "kindergarten",\n      "name": "Дом ребенка №23",\n      "lat": 55.8761197,\n      "lon": 37.6489728,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3049322",\n      "kind": "school",\n      "name": "ГБОУ школа №463, здание № 5",\n      "lat": 55.6756477,\n      "lon": 37.6705987,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "relation/3049425",\n      "kind": "school",\n      "name": "Лицей №1523",\n      "lat": 55.6773454,\n      "lon": 37.6817597,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3049511",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1527 Корпус Дельфиненок",\n      "lat": 55.6804069,\n      "lon": 37.6860008,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3049512",\n      "kind": "school",\n      "name": "ГБОУ школа № 1527 Корпус 4",\n      "lat": 55.6808685,\n      "lon": 37.6876232,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3112266",\n      "kind": "kindergarten",\n      "name": "Детский сад №830",\n      "lat": 55.7746839,\n      "lon": 37.484466,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3113404",\n      "kind": "kindergarten",\n      "name": "Детский сад №756",\n      "lat": 55.7027047,\n      "lon": 37.8095237,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3113405",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7034962,\n      "lon": 37.8110557,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3160010",\n      "kind": "kindergarten",\n      "name": "Детский сад №2665",\n      "lat": 55.8657463,\n      "lon": 37.5296892,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3160011",\n      "kind": "kindergarten",\n      "name": "Детский сад №633",\n      "lat": 55.8663685,\n      "lon": 37.5297428,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3164563",\n      "kind": "school",\n      "name": "Школа №1213",\n      "lat": 55.8439594,\n      "lon": 37.5653435,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3175157",\n      "kind": "kindergarten",\n      "name": "ГБОУ Детский сад №9",\n      "lat": 55.7708635,\n      "lon": 37.4800582,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3201223",\n      "kind": "school",\n      "name": "Школа №1188",\n      "lat": 55.8802901,\n      "lon": 37.691191,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3305498",\n      "kind": "school",\n      "name": "Школа №396",\n      "lat": 55.7397949,\n      "lon": 37.6861473,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3325183",\n      "kind": "school",\n      "name": "Школа № 1273. Корпус школьного образования №1",\n      "lat": 55.6292478,\n      "lon": 37.5227549,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3325184",\n      "kind": "kindergarten",\n      "name": "Школа № 1273. Корпус дошкольного образования №3",\n      "lat": 55.6300119,\n      "lon": 37.5231402,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3344574",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус № 1",\n      "lat": 55.6383431,\n      "lon": 37.5280169,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3370903",\n      "kind": "school",\n      "name": "Гимназия №1570",\n      "lat": 55.7893227,\n      "lon": 37.5772352,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3425342",\n      "kind": "school",\n      "name": "Школа № 17. Школьное здание №4",\n      "lat": 55.6369476,\n      "lon": 37.5405138,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3425343",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус №3",\n      "lat": 55.6374659,\n      "lon": 37.5388157,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3428535",\n      "kind": "kindergarten",\n      "name": "Детский сад №1849",\n      "lat": 55.8845623,\n      "lon": 37.6497155,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3428539",\n      "kind": "school",\n      "name": "Детский юношеский центр",\n      "lat": 55.8838133,\n      "lon": 37.6499429,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3428541",\n      "kind": "school",\n      "name": "Школа №1380",\n      "lat": 55.8844527,\n      "lon": 37.6516384,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3498403",\n      "kind": "school",\n      "name": "Школа № 1260",\n      "lat": 55.6998705,\n      "lon": 37.5568668,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3500635",\n      "kind": "school",\n      "name": "Школа № 1273. Корпус школьного образования №2",\n      "lat": 55.6282814,\n      "lon": 37.5224043,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3509355",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус №6",\n      "lat": 55.6340578,\n      "lon": 37.5328202,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3509356",\n      "kind": "school",\n      "name": "Школа № 17. Школьное здание №2",\n      "lat": 55.6345233,\n      "lon": 37.5346001,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3792745",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 8",\n      "lat": 55.7787734,\n      "lon": 37.5213099,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3834185",\n      "kind": "school",\n      "name": "Школа №1950",\n      "lat": 55.7664959,\n      "lon": 37.5847443,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/3951778",\n      "kind": "school",\n      "name": "Школа Сотрудничества",\n      "lat": 55.7430626,\n      "lon": 37.6565492,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4090263",\n      "kind": "school",\n      "name": "СП \\"Гимназия\\"",\n      "lat": 55.7163312,\n      "lon": 37.4608347,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4197824",\n      "kind": "kindergarten",\n      "name": "Школа № 2123 имени Мигеля Эрнандеса. Дошкольное отделение \\"Журавушка\\"",\n      "lat": 55.7618487,\n      "lon": 37.5997866,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4217693",\n      "kind": "school",\n      "name": "Школа № 2072",\n      "lat": 55.743314,\n      "lon": 37.8032021,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4233096",\n      "kind": "school",\n      "name": "Школа им. Н.М. Карамзина. Корпус школьного образования №1 «Петербургский»",\n      "lat": 55.5923521,\n      "lon": 37.5406021,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4245552",\n      "kind": "kindergarten",\n      "name": "Детский сад №1611",\n      "lat": 55.8717965,\n      "lon": 37.5172604,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4245553",\n      "kind": "school",\n      "name": "Кадетская школа №1702",\n      "lat": 55.8649025,\n      "lon": 37.5153349,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4523021",\n      "kind": "kindergarten",\n      "name": "Школа № 1950 с углубленным изучением отдельных предметов. Дошкольное отделение",\n      "lat": 55.7671184,\n      "lon": 37.5867091,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4580283",\n      "kind": "school",\n      "name": "Школа №91",\n      "lat": 55.7541659,\n      "lon": 37.5970008,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4681982",\n      "kind": "kindergarten",\n      "name": "Детский сад №2635",\n      "lat": 55.6908751,\n      "lon": 37.572006,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4681983",\n      "kind": "school",\n      "name": "Школа №199",\n      "lat": 55.6901594,\n      "lon": 37.571638,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4716520",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение №2 «Рябинушка» школы №2123",\n      "lat": 55.7567566,\n      "lon": 37.5902411,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4719459",\n      "kind": "school",\n      "name": "Школа № 1525",\n      "lat": 55.7846628,\n      "lon": 37.6701265,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4729216",\n      "kind": "school",\n      "name": "Школа № 2107. Здание \\"На Мещанской\\"",\n      "lat": 55.7769562,\n      "lon": 37.6274505,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4742204",\n      "kind": "school",\n      "name": "Школа №1234",\n      "lat": 55.753599,\n      "lon": 37.5902878,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4831993",\n      "kind": "school",\n      "name": "Спецшкола №486",\n      "lat": 55.7417619,\n      "lon": 37.6585159,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4834971",\n      "kind": "school",\n      "name": "Школа №498",\n      "lat": 55.7377925,\n      "lon": 37.6521352,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4835145",\n      "kind": "kindergarten",\n      "name": "Детский сад №3",\n      "lat": 55.7365461,\n      "lon": 37.6531505,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/4835322",\n      "kind": "kindergarten",\n      "name": "Детский сад №1827",\n      "lat": 55.7355475,\n      "lon": 37.6541212,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5359641",\n      "kind": "school",\n      "name": "Школа №57 (здание №2)",\n      "lat": 55.7300903,\n      "lon": 37.5808146,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5362168",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1618",\n      "lat": 55.6933192,\n      "lon": 37.4991307,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5362169",\n      "kind": "kindergarten",\n      "name": "Центр психолого-медико-социального сопровождения \\"Раменки\\"",\n      "lat": 55.6938073,\n      "lon": 37.4999947,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5362204",\n      "kind": "kindergarten",\n      "name": "Детский сад №2164",\n      "lat": 55.7320866,\n      "lon": 37.58664,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5441013",\n      "kind": "school",\n      "name": "Школа № 7 филиал № 3",\n      "lat": 55.6794451,\n      "lon": 37.5192988,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5513696",\n      "kind": "school",\n      "name": "Школа №1253",\n      "lat": 55.7346148,\n      "lon": 37.5922405,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5569527",\n      "kind": "school",\n      "name": "Школа с углубленным изучением иностранного языка № 1231 имени В. Д. Поленова",\n      "lat": 55.7387981,\n      "lon": 37.5923222,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5573120",\n      "kind": "kindergarten",\n      "name": "Школа №1078 (Модуль №5)",\n      "lat": 55.8190706,\n      "lon": 37.822638,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5573122",\n      "kind": "school",\n      "name": "Школа №1598. Здание №2",\n      "lat": 55.8185248,\n      "lon": 37.829256,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5573123",\n      "kind": "school",\n      "name": "Школа №1078 (Модуль №2)",\n      "lat": 55.8190333,\n      "lon": 37.8208288,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5573124",\n      "kind": "kindergarten",\n      "name": "Школа №368 (Здание №3)",\n      "lat": 55.8190607,\n      "lon": 37.8148522,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5573125",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №12",\n      "lat": 55.819322,\n      "lon": 37.8292055,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5594700",\n      "kind": "kindergarten",\n      "name": "Детский сад №233",\n      "lat": 55.668933,\n      "lon": 37.5490159,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5594701",\n      "kind": "kindergarten",\n      "name": "Детский сад №293",\n      "lat": 55.668595,\n      "lon": 37.5499694,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5594702",\n      "kind": "school",\n      "name": "Школа №121",\n      "lat": 55.6697291,\n      "lon": 37.5493704,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5623081",\n      "kind": "school",\n      "name": "Школа №1259",\n      "lat": 55.7319937,\n      "lon": 37.6320087,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5686625",\n      "kind": "school",\n      "name": "Школа №633",\n      "lat": 55.7546747,\n      "lon": 37.8270964,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5699016",\n      "kind": "school",\n      "name": "Школа №1097",\n      "lat": 55.8659895,\n      "lon": 37.6574187,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713477",\n      "kind": "kindergarten",\n      "name": "Детский сад №2287",\n      "lat": 55.8756377,\n      "lon": 37.5815872,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713478",\n      "kind": "kindergarten",\n      "name": "Детский сад №36",\n      "lat": 55.7720042,\n      "lon": 37.7305262,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713479",\n      "kind": "school",\n      "name": "Лицей №1575",\n      "lat": 55.8050537,\n      "lon": 37.5320355,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713481",\n      "kind": "school",\n      "name": "Школа №302",\n      "lat": 55.8804305,\n      "lon": 37.5794377,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713482",\n      "kind": "school",\n      "name": "Школа №635",\n      "lat": 55.7571779,\n      "lon": 37.7923071,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713483",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.818824,\n      "lon": 37.571378,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713567",\n      "kind": "kindergarten",\n      "name": "Школа №1179. Корпус дошкольного образования №4",\n      "lat": 55.6351639,\n      "lon": 37.6075809,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5713568",\n      "kind": "kindergarten",\n      "name": "Школа №1179. Корпус дошкольного образования №3",\n      "lat": 55.6356447,\n      "lon": 37.6079979,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5736841",\n      "kind": "kindergarten",\n      "name": "Детский сад №2389",\n      "lat": 55.8847424,\n      "lon": 37.6844234,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5754384",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6651528,\n      "lon": 37.5575675,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5754385",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6644274,\n      "lon": 37.5580763,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5781856",\n      "kind": "kindergarten",\n      "name": "Детский сад №380",\n      "lat": 55.7377889,\n      "lon": 37.4665472,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5849302",\n      "kind": "kindergarten",\n      "name": "Детский сад №1159",\n      "lat": 55.8089016,\n      "lon": 37.6405263,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5851694",\n      "kind": "school",\n      "name": "Спортшкола № 46",\n      "lat": 55.6069056,\n      "lon": 37.6777051,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/5892824",\n      "kind": "school",\n      "name": "Гимназия №45",\n      "lat": 55.6917983,\n      "lon": 37.5858468,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6010660",\n      "kind": "school",\n      "name": "Центр образования №1499",\n      "lat": 55.8357258,\n      "lon": 37.6605556,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6022234",\n      "kind": "kindergarten",\n      "name": "Детский сад № 11 «Незабудка»",\n      "lat": 55.8909247,\n      "lon": 37.7221782,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6116666",\n      "kind": "school",\n      "name": "Школа «Золотое сечение»",\n      "lat": 55.7385903,\n      "lon": 37.577139,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6228956",\n      "kind": "kindergarten",\n      "name": "Детский сад №869",\n      "lat": 55.8592521,\n      "lon": 37.4700925,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6282939",\n      "kind": "school",\n      "name": "Школа №692",\n      "lat": 55.8563497,\n      "lon": 37.4730511,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6319979",\n      "kind": "school",\n      "name": "Школа «Содружество». Учебный корпус №2",\n      "lat": 55.7607457,\n      "lon": 37.6915744,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6322609",\n      "kind": "kindergarten",\n      "name": "Детский сад №1262",\n      "lat": 55.8670411,\n      "lon": 37.5687592,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6322610",\n      "kind": "school",\n      "name": "Начальная школа №1826",\n      "lat": 55.8667093,\n      "lon": 37.5675411,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6334556",\n      "kind": "school",\n      "name": "Детский сад №1548",\n      "lat": 55.8611376,\n      "lon": 37.5604878,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6339395",\n      "kind": "kindergarten",\n      "name": "Школа № 1383. Корпус 3",\n      "lat": 55.8593028,\n      "lon": 37.5661993,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6339396",\n      "kind": "kindergarten",\n      "name": "Школа № 1383. Корпус 4",\n      "lat": 55.8586097,\n      "lon": 37.5662128,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6339397",\n      "kind": "school",\n      "name": "Школа №1383",\n      "lat": 55.8579072,\n      "lon": 37.5663031,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6344000",\n      "kind": "kindergarten",\n      "name": "Детский сад №1472",\n      "lat": 55.7280741,\n      "lon": 37.5872774,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6362542",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Юниор-парк\\"",\n      "lat": 55.8657052,\n      "lon": 37.5543541,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6362543",\n      "kind": "kindergarten",\n      "name": "Школа № 656 имени А. С. Макаренко, дошкольное отделение",\n      "lat": 55.8650348,\n      "lon": 37.5548924,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6376832",\n      "kind": "school",\n      "name": "Школа №\xa01252 имени Сервантеса",\n      "lat": 55.8102107,\n      "lon": 37.4963393,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6476664",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1951 комбинированного вида",\n      "lat": 55.7763858,\n      "lon": 37.6003314,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6498938",\n      "kind": "school",\n      "name": "Школа №1285",\n      "lat": 55.8338476,\n      "lon": 37.4485177,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6505043",\n      "kind": "kindergarten",\n      "name": "Детский сад №49",\n      "lat": 55.7844535,\n      "lon": 37.6313068,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6505044",\n      "kind": "school",\n      "name": "Школа № 2107. Школа им. Ромена Роллана",\n      "lat": 55.7854975,\n      "lon": 37.6320923,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6517532",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1021",\n      "lat": 55.7897922,\n      "lon": 37.6257217,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6519019",\n      "kind": "kindergarten",\n      "name": "Детский сад №1517",\n      "lat": 55.7763258,\n      "lon": 37.4605473,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6519020",\n      "kind": "kindergarten",\n      "name": "Гимназия №1517",\n      "lat": 55.7754771,\n      "lon": 37.4658991,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6527144",\n      "kind": "school",\n      "name": "Школа №1231 имени В. Д. Поленова",\n      "lat": 55.7403285,\n      "lon": 37.5774833,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6528274",\n      "kind": "kindergarten",\n      "name": "Гимназия №1517",\n      "lat": 55.775982,\n      "lon": 37.4633056,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6533359",\n      "kind": "school",\n      "name": "Гимназия №1517",\n      "lat": 55.7751017,\n      "lon": 37.4629548,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6598494",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Сказка\\"",\n      "lat": 55.7275453,\n      "lon": 37.7611404,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6609198",\n      "kind": "school",\n      "name": "Психоневрологический интернат №4",\n      "lat": 55.7362311,\n      "lon": 37.4590944,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6618012",\n      "kind": "kindergarten",\n      "name": "Детский сад №2444",\n      "lat": 55.8712089,\n      "lon": 37.5673144,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6618014",\n      "kind": "school",\n      "name": "Центр образования №1487",\n      "lat": 55.8696222,\n      "lon": 37.5658385,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6618015",\n      "kind": "school",\n      "name": "Школа №1029",\n      "lat": 55.8700469,\n      "lon": 37.567721,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6696799",\n      "kind": "school",\n      "name": "Школа № 1323",\n      "lat": 55.7430375,\n      "lon": 37.625783,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6698208",\n      "kind": "kindergarten",\n      "name": "ДОУ № 729",\n      "lat": 55.7562398,\n      "lon": 37.586336,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6698209",\n      "kind": "school",\n      "name": "Школа №1234",\n      "lat": 55.7562998,\n      "lon": 37.5852289,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6702161",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8135142,\n      "lon": 37.4932529,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6712910",\n      "kind": "kindergarten",\n      "name": "Детский сад №1389",\n      "lat": 55.8714024,\n      "lon": 37.5591549,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6712911",\n      "kind": "kindergarten",\n      "name": "Детский сад №642",\n      "lat": 55.8712824,\n      "lon": 37.5574807,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6718174",\n      "kind": "school",\n      "name": "Школа №1223",\n      "lat": 55.8448107,\n      "lon": 37.5534485,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6728142",\n      "kind": "school",\n      "name": "Лицей № 1535",\n      "lat": 55.7260387,\n      "lon": 37.5650015,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6728144",\n      "kind": "kindergarten",\n      "name": "Филипповская школа",\n      "lat": 55.7256014,\n      "lon": 37.5641975,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6728834",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8037758,\n      "lon": 37.7772296,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6728835",\n      "kind": "school",\n      "name": "Школа №343",\n      "lat": 55.8034367,\n      "lon": 37.7770052,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6741550",\n      "kind": "school",\n      "name": "Школьное отделение № 22 Лицея № 1501",\n      "lat": 55.7699198,\n      "lon": 37.6066168,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6751348",\n      "kind": "kindergarten",\n      "name": "Детский сад №1486",\n      "lat": 55.8758657,\n      "lon": 37.5444077,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6751349",\n      "kind": "kindergarten",\n      "name": "Детский сад №2298",\n      "lat": 55.8755071,\n      "lon": 37.5435031,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6751350",\n      "kind": "school",\n      "name": "Школа №849",\n      "lat": 55.8752844,\n      "lon": 37.5396717,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6751351",\n      "kind": "school",\n      "name": "Школа №1121",\n      "lat": 55.8754473,\n      "lon": 37.5416211,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6751356",\n      "kind": "kindergarten",\n      "name": "Детский сад №1578",\n      "lat": 55.8770485,\n      "lon": 37.5507131,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6758020",\n      "kind": "kindergarten",\n      "name": "Детский сад №669",\n      "lat": 55.7414618,\n      "lon": 37.5747631,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6758021",\n      "kind": "school",\n      "name": "Спецшкола №30",\n      "lat": 55.7370603,\n      "lon": 37.5714404,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6758026",\n      "kind": "school",\n      "name": "Школа №57 (здание №3)",\n      "lat": 55.7205716,\n      "lon": 37.569045,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6758127",\n      "kind": "school",\n      "name": "Школа №45",\n      "lat": 55.7240585,\n      "lon": 37.5602548,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6758920",\n      "kind": "kindergarten",\n      "name": "Детский сад №2044",\n      "lat": 55.842186,\n      "lon": 37.504784,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6796156",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8441523,\n      "lon": 37.4870092,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6847000",\n      "kind": "school",\n      "name": "Школа № 647",\n      "lat": 55.763312,\n      "lon": 37.7377824,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6886515",\n      "kind": "school",\n      "name": "Школа № 1329",\n      "lat": 55.6689647,\n      "lon": 37.4649297,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6890548",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Хавской\\"",\n      "lat": 55.7133694,\n      "lon": 37.6146213,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6909136",\n      "kind": "school",\n      "name": "Школа №1409",\n      "lat": 55.7885255,\n      "lon": 37.5444968,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6913392",\n      "kind": "school",\n      "name": "Московский академический художественный лицей",\n      "lat": 55.7323416,\n      "lon": 37.6079558,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6913847",\n      "kind": "school",\n      "name": "Начальная школа - детский сад №1761",\n      "lat": 55.7534586,\n      "lon": 37.6784135,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6929207",\n      "kind": "kindergarten",\n      "name": "Детский сад №1080",\n      "lat": 55.7107297,\n      "lon": 37.7403293,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6947137",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 1268",\n      "lat": 55.8913369,\n      "lon": 37.5350199,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6957072",\n      "kind": "kindergarten",\n      "name": "Школа №2054, Дошкольное отделение №1",\n      "lat": 55.7672766,\n      "lon": 37.6164575,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6968315",\n      "kind": "school",\n      "name": "Школа № 812",\n      "lat": 55.6698785,\n      "lon": 37.4602345,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/6981012",\n      "kind": "school",\n      "name": "Школа №1236 имени С.В.Милашенкова. Отделение №250",\n      "lat": 55.817851,\n      "lon": 37.5867705,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7004980",\n      "kind": "kindergarten",\n      "name": "Детский сад №632",\n      "lat": 55.8709959,\n      "lon": 37.5496012,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7004981",\n      "kind": "school",\n      "name": "Школа № 656 им. А. С. Макаренко",\n      "lat": 55.8704118,\n      "lon": 37.5526854,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7013096",\n      "kind": "school",\n      "name": "Школа №767",\n      "lat": 55.8209221,\n      "lon": 37.7948285,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7055165",\n      "kind": "kindergarten",\n      "name": "Детский сад №822",\n      "lat": 55.8751489,\n      "lon": 37.5522452,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7092147",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\". Здание на Б. Казенном",\n      "lat": 55.7605664,\n      "lon": 37.6555793,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103021",\n      "kind": "school",\n      "name": "Иракская школа",\n      "lat": 55.7563414,\n      "lon": 37.5948429,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103024",\n      "kind": "school",\n      "name": "Пушкинская гимназия на Пресне",\n      "lat": 55.7586151,\n      "lon": 37.5706237,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103026",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7605417,\n      "lon": 37.5657947,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103028",\n      "kind": "school",\n      "name": "Романовская школа",\n      "lat": 55.7672908,\n      "lon": 37.5695366,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103029",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1570. Дошкольное отделение",\n      "lat": 55.7813388,\n      "lon": 37.5845225,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103030",\n      "kind": "kindergarten",\n      "name": "Школа «Содружество». Учебный корпус №4",\n      "lat": 55.7568008,\n      "lon": 37.6964894,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103031",\n      "kind": "kindergarten",\n      "name": "Школа № 1228 \\"Лефортово\\". Корпус \\"Цветик-семицветик\\"",\n      "lat": 55.7638631,\n      "lon": 37.7148521,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103032",\n      "kind": "kindergarten",\n      "name": "Школа № 1228 \\"Лефортово\\". Корпус \\"Аленький цветочек\\"",\n      "lat": 55.7635683,\n      "lon": 37.7156527,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103198",\n      "kind": "school",\n      "name": "Школа № 962",\n      "lat": 55.8553388,\n      "lon": 37.5877126,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103199",\n      "kind": "kindergarten",\n      "name": "Школа № 962. Дошкольное отделение № 5",\n      "lat": 55.8545481,\n      "lon": 37.5869014,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103200",\n      "kind": "kindergarten",\n      "name": "Лицей № 1575. Дошкольное отделение",\n      "lat": 55.7969242,\n      "lon": 37.5573494,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103202",\n      "kind": "school",\n      "name": "Инженерно-техническая школа им. дважды Героя Советского Союза П.Р. Поповича",\n      "lat": 55.8057821,\n      "lon": 37.5480073,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103203",\n      "kind": "kindergarten",\n      "name": "Инженерно-техническая школа имени дважды Героя Советского Союза П. Р. Поповича. Дошкольное отделение Коккинаки № 7",\n      "lat": 55.8054757,\n      "lon": 37.5470409,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103204",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1266",\n      "lat": 55.8664607,\n      "lon": 37.6181239,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103205",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1554. Дошкольное отделение № 1",\n      "lat": 55.8673444,\n      "lon": 37.6176933,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103206",\n      "kind": "kindergarten",\n      "name": "Школа № 152. Детский сад № 286",\n      "lat": 55.8031036,\n      "lon": 37.5309517,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103207",\n      "kind": "kindergarten",\n      "name": "Школа-сад № 1847",\n      "lat": 55.8064758,\n      "lon": 37.5332772,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103210",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1570. Дошкольное отделение",\n      "lat": 55.7825166,\n      "lon": 37.5776695,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103211",\n      "kind": "school",\n      "name": "Специальная (коррекционная) общеобразовательная школа-интернат № 2",\n      "lat": 55.8098255,\n      "lon": 37.5347202,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7103213",\n      "kind": "school",\n      "name": "Кадетская школа № 1784",\n      "lat": 55.7803093,\n      "lon": 37.5684252,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7119651",\n      "kind": "school",\n      "name": "Школа № 2054. Школьное отделение №3",\n      "lat": 55.7650282,\n      "lon": 37.6142371,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7141808",\n      "kind": "school",\n      "name": "Школа № 2030",\n      "lat": 55.7598673,\n      "lon": 37.5557759,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7153404",\n      "kind": "school",\n      "name": "Планета успеха",\n      "lat": 55.8366676,\n      "lon": 37.5280176,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7156736",\n      "kind": "school",\n      "name": "Школа с углубленным изучением отдельных предметов № 1234",\n      "lat": 55.7651748,\n      "lon": 37.6013492,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7173366",\n      "kind": "kindergarten",\n      "name": "Школа № 2099. Здание 8",\n      "lat": 55.819026,\n      "lon": 37.4954988,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7184365",\n      "kind": "kindergarten",\n      "name": "Детский сад №1592",\n      "lat": 55.6735542,\n      "lon": 37.7427032,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7190748",\n      "kind": "school",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 1631",\n      "lat": 55.8921625,\n      "lon": 37.5395776,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7220450",\n      "kind": "kindergarten",\n      "name": "Детский сад №2204",\n      "lat": 55.7234383,\n      "lon": 37.6124879,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7238772",\n      "kind": "school",\n      "name": "Гимназия Марьина Роща имени В. Ф. Орлова. Учебный корпус 35",\n      "lat": 55.7865131,\n      "lon": 37.6046806,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7268814",\n      "kind": "school",\n      "name": "Школа №1501",\n      "lat": 55.7796397,\n      "lon": 37.6141136,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7295266",\n      "kind": "school",\n      "name": "Интeрнат \\"Третий Московский кадетский корпус имени Александра Невского\\"",\n      "lat": 55.7328784,\n      "lon": 37.6522596,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7299944",\n      "kind": "kindergarten",\n      "name": "Детский сад №1497",\n      "lat": 55.7740821,\n      "lon": 37.6018766,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7307938",\n      "kind": "kindergarten",\n      "name": "ДОУ № 1465",\n      "lat": 55.7633218,\n      "lon": 37.5392107,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7307939",\n      "kind": "kindergarten",\n      "name": "ДОУ №1930",\n      "lat": 55.7634294,\n      "lon": 37.5382958,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7309883",\n      "kind": "kindergarten",\n      "name": "ДОУ № 749",\n      "lat": 55.7604989,\n      "lon": 37.5381105,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7309884",\n      "kind": "kindergarten",\n      "name": "ДОУ № 1927",\n      "lat": 55.7606072,\n      "lon": 37.5372349,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7325096",\n      "kind": "school",\n      "name": "Школа №\xa0340",\n      "lat": 55.7588422,\n      "lon": 37.5366524,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7326339",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 244",\n      "lat": 55.8964256,\n      "lon": 37.5388947,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7326343",\n      "kind": "school",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 184",\n      "lat": 55.8955124,\n      "lon": 37.5378709,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7329513",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7171504,\n      "lon": 37.6726984,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7329517",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7178924,\n      "lon": 37.6693535,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7330425",\n      "kind": "kindergarten",\n      "name": "Детский сад №390",\n      "lat": 55.8816791,\n      "lon": 37.5723914,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7334650",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7415593,\n      "lon": 37.6783755,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7334651",\n      "kind": "school",\n      "name": "Школа №1221",\n      "lat": 55.7407482,\n      "lon": 37.6789543,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7357058",\n      "kind": "school",\n      "name": "Школа № 56",\n      "lat": 55.7467017,\n      "lon": 37.5492835,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7359503",\n      "kind": "kindergarten",\n      "name": "Детский сад №703",\n      "lat": 55.8818296,\n      "lon": 37.5347146,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7359504",\n      "kind": "school",\n      "name": "Школа №771",\n      "lat": 55.8827959,\n      "lon": 37.5346768,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7402655",\n      "kind": "kindergarten",\n      "name": "Школа № 1541. Корпус дошкольного образования №4",\n      "lat": 55.6688623,\n      "lon": 37.5057022,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7405009",\n      "kind": "school",\n      "name": "Школа №771",\n      "lat": 55.8865095,\n      "lon": 37.5373528,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7405010",\n      "kind": "school",\n      "name": "Школа №1291 с углублённым изучением английского языка",\n      "lat": 55.8872557,\n      "lon": 37.5373531,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7435037",\n      "kind": "school",\n      "name": "Инженерная школа № 1581. Корпус 2",\n      "lat": 55.768521,\n      "lon": 37.65502,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7484311",\n      "kind": "school",\n      "name": "Многопрофильный лицей № 1799. СП \\"Лицей на Полянке\\"",\n      "lat": 55.7369958,\n      "lon": 37.6180798,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7500830",\n      "kind": "school",\n      "name": "Многопрофильный лицей № 1799. СП «Школа № 19»",\n      "lat": 55.7441771,\n      "lon": 37.6217091,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7540984",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1143",\n      "lat": 55.8535421,\n      "lon": 37.5789539,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7568257",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7047134,\n      "lon": 37.4936867,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7595822",\n      "kind": "kindergarten",\n      "name": "Детский сад № 516",\n      "lat": 55.7801277,\n      "lon": 37.5958008,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7612096",\n      "kind": "school",\n      "name": "Лицей НИУ ВШЭ",\n      "lat": 55.7696296,\n      "lon": 37.6188347,\n      "opening_hours": "Mo-Sa 09:00-18:00"\n    },\n    {\n      "id": "relation/7612099",\n      "kind": "school",\n      "name": "Центр образования №1479",\n      "lat": 55.7708346,\n      "lon": 37.619002,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7612111",\n      "kind": "kindergarten",\n      "name": "Колледж малого бизнеса № 4. Дошкольный корпус 2",\n      "lat": 55.772255,\n      "lon": 37.6173229,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7613478",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8625097,\n      "lon": 37.6930833,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7621468",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\". Здание на Покровке",\n      "lat": 55.7585886,\n      "lon": 37.6444861,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7629953",\n      "kind": "kindergarten",\n      "name": "Детский сад № 395 Медицинского центра Управления делами Мэра и Правительства Москвы",\n      "lat": 55.7549191,\n      "lon": 37.6417214,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7637088",\n      "kind": "school",\n      "name": "Курчатовская школа корпус \\"Маршал\\"",\n      "lat": 55.7917885,\n      "lon": 37.4985782,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7639829",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1428",\n      "lat": 55.6678026,\n      "lon": 37.4682709,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7641051",\n      "kind": "school",\n      "name": "Школа № 627",\n      "lat": 55.7330417,\n      "lon": 37.6360641,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7658483",\n      "kind": "kindergarten",\n      "name": "ДОУ №1982",\n      "lat": 55.7641954,\n      "lon": 37.6661706,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7658484",\n      "kind": "school",\n      "name": "Центр образования №1480",\n      "lat": 55.7645917,\n      "lon": 37.6659139,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7666664",\n      "kind": "kindergarten",\n      "name": "Детский сад №8",\n      "lat": 55.7389176,\n      "lon": 37.4996479,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7666666",\n      "kind": "school",\n      "name": "Школа №700",\n      "lat": 55.7389273,\n      "lon": 37.5020715,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7672222",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"В гостях у сказки\\"",\n      "lat": 55.734339,\n      "lon": 37.6309811,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7672224",\n      "kind": "kindergarten",\n      "name": "Школа № 1259. Дошкольное отделение",\n      "lat": 55.7342307,\n      "lon": 37.6294031,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7712039",\n      "kind": "school",\n      "name": "",\n      "lat": 55.806347,\n      "lon": 37.6895545,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7715842",\n      "kind": "school",\n      "name": "Гимназия №1257",\n      "lat": 55.7144854,\n      "lon": 37.6256635,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7716814",\n      "kind": "kindergarten",\n      "name": "Детский сад №2032",\n      "lat": 55.7599186,\n      "lon": 37.6356376,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7717004",\n      "kind": "school",\n      "name": "Лицей № 1535",\n      "lat": 55.7309714,\n      "lon": 37.5619802,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7719254",\n      "kind": "school",\n      "name": "Школа № 2124 \\"Центр развития и коррекции\\" Здание №3",\n      "lat": 55.7610715,\n      "lon": 37.6365513,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7722663",\n      "kind": "school",\n      "name": "Школа № 554",\n      "lat": 55.6609508,\n      "lon": 37.5815156,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7722664",\n      "kind": "kindergarten",\n      "name": "Школа № 554",\n      "lat": 55.6600307,\n      "lon": 37.5823025,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7722665",\n      "kind": "kindergarten",\n      "name": "Школа № 554",\n      "lat": 55.6596599,\n      "lon": 37.5809665,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7722666",\n      "kind": "kindergarten",\n      "name": "Школа № 554",\n      "lat": 55.6578611,\n      "lon": 37.5804658,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7722668",\n      "kind": "kindergarten",\n      "name": "Школа № 554",\n      "lat": 55.658422,\n      "lon": 37.5818665,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7725968",\n      "kind": "school",\n      "name": "Школа № 1231 им. В. Д. Поленова",\n      "lat": 55.7485171,\n      "lon": 37.5966758,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7738645",\n      "kind": "kindergarten",\n      "name": "Детский сад №1170",\n      "lat": 55.7128498,\n      "lon": 37.614267,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7771762",\n      "kind": "kindergarten",\n      "name": "Школа №830 дошкольное отделение Незабудка-1",\n      "lat": 55.8239873,\n      "lon": 37.4447691,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7771763",\n      "kind": "kindergarten",\n      "name": "Школа №830 дошкольное отделение Незабудка-2",\n      "lat": 55.8235078,\n      "lon": 37.4442511,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7795518",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8041486,\n      "lon": 37.5489507,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7798211",\n      "kind": "kindergarten",\n      "name": "Школа № 345 имени А.С. Пушкина. Дошкольное отделение",\n      "lat": 55.7666126,\n      "lon": 37.6563882,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7799935",\n      "kind": "kindergarten",\n      "name": "Детский сад №2529",\n      "lat": 55.8070525,\n      "lon": 37.5199063,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7814787",\n      "kind": "school",\n      "name": "Школа №1301 им. Е.Т. Гайдара",\n      "lat": 55.7800317,\n      "lon": 37.7385422,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7839822",\n      "kind": "school",\n      "name": "Школа №1262",\n      "lat": 55.7305746,\n      "lon": 37.6205231,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7840004",\n      "kind": "kindergarten",\n      "name": "Школа № 1231 имени В. Д. Поленова. Дошкольный корпус",\n      "lat": 55.7298123,\n      "lon": 37.5617337,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7877996",\n      "kind": "school",\n      "name": "Школа №1231",\n      "lat": 55.7498071,\n      "lon": 37.5890358,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7891536",\n      "kind": "school",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 2",\n      "lat": 55.7451167,\n      "lon": 37.5925565,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7891541",\n      "kind": "school",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 4",\n      "lat": 55.7469797,\n      "lon": 37.595742,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7895065",\n      "kind": "school",\n      "name": "Школа № 1277",\n      "lat": 55.7838582,\n      "lon": 37.6020225,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7903980",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7851838,\n      "lon": 37.5361895,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7903981",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7857234,\n      "lon": 37.5370061,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7916369",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 3",\n      "lat": 55.8507959,\n      "lon": 37.635942,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7927700",\n      "kind": "school",\n      "name": "Школа № 1288, корпус № 10",\n      "lat": 55.7761138,\n      "lon": 37.551338,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7966560",\n      "kind": "school",\n      "name": "Школа № 1270 «Вектор»",\n      "lat": 55.7439793,\n      "lon": 37.6651545,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/7997756",\n      "kind": "school",\n      "name": "Лицей НИУ ВШЭ",\n      "lat": 55.7636235,\n      "lon": 37.6440206,\n      "opening_hours": "Mo-Sa 09:00-18:00"\n    },\n    {\n      "id": "relation/8010397",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №6",\n      "lat": 55.8522782,\n      "lon": 37.6950801,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8016230",\n      "kind": "school",\n      "name": "Частная школа \\"Бнот Менахем\\"",\n      "lat": 55.7664398,\n      "lon": 37.6429479,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8025165",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8947465,\n      "lon": 37.7315412,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8062611",\n      "kind": "kindergarten",\n      "name": "Школа \\"Покровский квартал\\". Детский сад № 1",\n      "lat": 55.7551529,\n      "lon": 37.6514914,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8119012",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Ромашка\\"",\n      "lat": 55.8090257,\n      "lon": 37.4742969,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8119013",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Колокольчик\\"",\n      "lat": 55.8097715,\n      "lon": 37.4741278,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8157274",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8016983,\n      "lon": 37.4650659,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8178929",\n      "kind": "school",\n      "name": "Школа №123",\n      "lat": 55.7579814,\n      "lon": 37.6022541,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8197027",\n      "kind": "kindergarten",\n      "name": "Детский сад №298",\n      "lat": 55.8353875,\n      "lon": 37.533436,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8197028",\n      "kind": "kindergarten",\n      "name": "Детский сад №309",\n      "lat": 55.836043,\n      "lon": 37.5339836,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8197029",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8361141,\n      "lon": 37.5325804,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8209718",\n      "kind": "kindergarten",\n      "name": "Школа № 717. Дошкольное отделение",\n      "lat": 55.8202265,\n      "lon": 37.515923,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8224251",\n      "kind": "school",\n      "name": "Школа №1232",\n      "lat": 55.7501767,\n      "lon": 37.5612288,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8298825",\n      "kind": "school",\n      "name": "Британская Международная Школа № 2",\n      "lat": 55.8186452,\n      "lon": 37.5275612,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8298826",\n      "kind": "kindergarten",\n      "name": "Детский сад №2394",\n      "lat": 55.8181515,\n      "lon": 37.5289511,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8304326",\n      "kind": "school",\n      "name": "Школа №\xa067",\n      "lat": 55.7413846,\n      "lon": 37.5232664,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8318510",\n      "kind": "school",\n      "name": "Школа №1125",\n      "lat": 55.8829353,\n      "lon": 37.4921408,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8331322",\n      "kind": "school",\n      "name": "ГБОУ Школа № 1249",\n      "lat": 55.8012999,\n      "lon": 37.5185448,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8342319",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1576\\" Дошкольное отделение корпус № 11 (\\"Радуга\\")",\n      "lat": 55.8270129,\n      "lon": 37.531832,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8342560",\n      "kind": "school",\n      "name": "Британская Международная Школа № 1",\n      "lat": 55.8261245,\n      "lon": 37.5319324,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8361003",\n      "kind": "kindergarten",\n      "name": "Школа \\"Бескудниково\\". Дошкольный корпус №3",\n      "lat": 55.8700128,\n      "lon": 37.541388,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8361042",\n      "kind": "school",\n      "name": "Прогимназия №1661",\n      "lat": 55.8740524,\n      "lon": 37.5307217,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8361054",\n      "kind": "school",\n      "name": "Школа \\"Бескудниково\\". Корпус 2",\n      "lat": 55.8718693,\n      "lon": 37.5422574,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8367574",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7585504,\n      "lon": 37.5720282,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8439571",\n      "kind": "kindergarten",\n      "name": "Детский сад №34 «Колосок»",\n      "lat": 55.8967424,\n      "lon": 37.4535117,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8439572",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa02 «Журавушка»",\n      "lat": 55.8968427,\n      "lon": 37.4524757,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8444542",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.741603,\n      "lon": 37.6358854,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8446521",\n      "kind": "kindergarten",\n      "name": "Школа № 1799. Дошкольное отделение Пятницкое",\n      "lat": 55.7340469,\n      "lon": 37.6266797,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8606729",\n      "kind": "kindergarten",\n      "name": "Школа № 1583 имени К. А. Керимова (Дошкольный корпус)",\n      "lat": 55.8517902,\n      "lon": 37.5071134,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8606732",\n      "kind": "school",\n      "name": "Школьная академия",\n      "lat": 55.8506129,\n      "lon": 37.5146245,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8650883",\n      "kind": "school",\n      "name": "Образовательный комплекс градостроительства \\"Столица\\". Отделение Саратовское",\n      "lat": 55.7116724,\n      "lon": 37.7364235,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8675635",\n      "kind": "school",\n      "name": "Школа № 179",\n      "lat": 55.7596561,\n      "lon": 37.614753,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8709255",\n      "kind": "school",\n      "name": "Школа № 1601 имени Героя Советского Союза Е. К. Лютикова. Основная и средняя школа (4)",\n      "lat": 55.8060143,\n      "lon": 37.5761265,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8710076",\n      "kind": "kindergarten",\n      "name": "slamiKIDS",\n      "lat": 55.8061541,\n      "lon": 37.5300521,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8712975",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\". Здание на Лялином",\n      "lat": 55.7589931,\n      "lon": 37.6516152,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8714875",\n      "kind": "school",\n      "name": "Школа № 1499 им. И. А. Докукина",\n      "lat": 55.8344476,\n      "lon": 37.6443723,\n      "opening_hours": "Mo-Fr 09:00-17:30"\n    },\n    {\n      "id": "relation/8748581",\n      "kind": "school",\n      "name": "ГБОУ Школа №1373, корпус №2",\n      "lat": 55.7782872,\n      "lon": 37.823674,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8748583",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1373, корпус №6",\n      "lat": 55.7772644,\n      "lon": 37.8222188,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8751770",\n      "kind": "kindergarten",\n      "name": "Школа № 1576. Корпус № 12",\n      "lat": 55.8434623,\n      "lon": 37.5493667,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8751771",\n      "kind": "kindergarten",\n      "name": "Школа № 1576. Корпус № 16",\n      "lat": 55.8430116,\n      "lon": 37.5498226,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8769465",\n      "kind": "school",\n      "name": "Школа №1361",\n      "lat": 55.6749953,\n      "lon": 37.5367835,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8775881",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\". Здание на Земляном валу",\n      "lat": 55.7539838,\n      "lon": 37.6551128,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8789251",\n      "kind": "kindergarten",\n      "name": "Школа №1231 им. В. Д. Поленова. Дошкольный корпус",\n      "lat": 55.7472311,\n      "lon": 37.5877923,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8791878",\n      "kind": "school",\n      "name": "Школа № 2123 имени Мигеля Эрнандеса. Корпус \\"Стодесятая\\"",\n      "lat": 55.7568883,\n      "lon": 37.5949346,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8796311",\n      "kind": "school",\n      "name": "Лицей НИУ ВШЭ",\n      "lat": 55.7608039,\n      "lon": 37.6520715,\n      "opening_hours": "Mo-Sa 09:00-18:00"\n    },\n    {\n      "id": "relation/8801932",\n      "kind": "kindergarten",\n      "name": "Детское поместье",\n      "lat": 55.7601523,\n      "lon": 37.5887812,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8801933",\n      "kind": "school",\n      "name": "Школа № 1239",\n      "lat": 55.7606239,\n      "lon": 37.588361,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8849889",\n      "kind": "school",\n      "name": "Школа №1113",\n      "lat": 55.7686461,\n      "lon": 37.6021606,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8897320",\n      "kind": "school",\n      "name": "Школа № 498. Школьный корпус Марксистская улица, дом 9, корпус 2",\n      "lat": 55.7364752,\n      "lon": 37.6679886,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8897321",\n      "kind": "kindergarten",\n      "name": "Школа № 498. Дошкольный корпус Добровольческая улица, дом 28, строение 1",\n      "lat": 55.7418933,\n      "lon": 37.66924,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8897322",\n      "kind": "school",\n      "name": "Школа № 498. Школьный корпус Большой Факельный переулок, дом 23",\n      "lat": 55.7413421,\n      "lon": 37.6680822,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8898272",\n      "kind": "kindergarten",\n      "name": "Школа № 1259. Дошкольный корпус",\n      "lat": 55.7348096,\n      "lon": 37.6343849,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8904131",\n      "kind": "kindergarten",\n      "name": "Школа № 138. Дошкольный корпус № 7",\n      "lat": 55.7877506,\n      "lon": 37.4666234,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8904132",\n      "kind": "kindergarten",\n      "name": "Школа № 138. Дошкольный корпус № 8",\n      "lat": 55.7876334,\n      "lon": 37.4686236,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8907712",\n      "kind": "school",\n      "name": "Школа № 1694 «Ясенево». Здание 3",\n      "lat": 55.6151733,\n      "lon": 37.5478396,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/8907713",\n      "kind": "school",\n      "name": "Школа № 1694 «Ясенево». Здание 2",\n      "lat": 55.6143672,\n      "lon": 37.5478722,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9068415",\n      "kind": "school",\n      "name": "Школа №1220",\n      "lat": 55.8174236,\n      "lon": 37.621538,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9068416",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8188251,\n      "lon": 37.6234868,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9070447",\n      "kind": "school",\n      "name": "Школа №304",\n      "lat": 55.8186767,\n      "lon": 37.6338394,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9070448",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8194776,\n      "lon": 37.627331,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9071344",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8233012,\n      "lon": 37.6279675,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9072849",\n      "kind": "school",\n      "name": "Пироговская школа",\n      "lat": 55.7353638,\n      "lon": 37.6112445,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9080263",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №299",\n      "lat": 55.7089057,\n      "lon": 37.7586704,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9081239",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8283747,\n      "lon": 37.5971789,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9081441",\n      "kind": "school",\n      "name": "Школа № 1494",\n      "lat": 55.8260892,\n      "lon": 37.6002734,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9081470",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8232807,\n      "lon": 37.5967986,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9081474",\n      "kind": "kindergarten",\n      "name": "Детский сад №392",\n      "lat": 55.8234987,\n      "lon": 37.6002273,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9081476",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8243167,\n      "lon": 37.6003037,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9083102",\n      "kind": "school",\n      "name": "Школа № 1950",\n      "lat": 55.7683739,\n      "lon": 37.5880679,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9093873",\n      "kind": "school",\n      "name": "Школа № 1494",\n      "lat": 55.8279267,\n      "lon": 37.5921387,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9097255",\n      "kind": "school",\n      "name": "Лицей №1571 корпус №1",\n      "lat": 55.857786,\n      "lon": 37.4472314,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9143532",\n      "kind": "school",\n      "name": "Школа им. Н.М. Карамзина. Корпус школьного образования №2 «Преображенский»",\n      "lat": 55.5927681,\n      "lon": 37.5383377,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9143533",\n      "kind": "school",\n      "name": "Школа им. Н.М. Карамзина. Корпус школьного образования №3 «Московский»",\n      "lat": 55.5942859,\n      "lon": 37.5407904,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9154405",\n      "kind": "kindergarten",\n      "name": "P\'titCREF \\"1905 года\\"",\n      "lat": 55.7585429,\n      "lon": 37.5649543,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9170797",\n      "kind": "kindergarten",\n      "name": "Школа №1516. Дошкольный корпус",\n      "lat": 55.8191574,\n      "lon": 37.832088,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9170798",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №11",\n      "lat": 55.8184236,\n      "lon": 37.8321488,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9170799",\n      "kind": "school",\n      "name": "Школа №1598. Здание №3",\n      "lat": 55.8247403,\n      "lon": 37.8278634,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9174029",\n      "kind": "school",\n      "name": "Школа с углубленным изучением иностранного языка № 1241. Корпус № 2",\n      "lat": 55.7565362,\n      "lon": 37.5681522,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9223416",\n      "kind": "school",\n      "name": "Православная Свято-Петровская школа",\n      "lat": 55.7510844,\n      "lon": 37.6507548,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9310365",\n      "kind": "school",\n      "name": "Школа 1158",\n      "lat": 55.6343175,\n      "lon": 37.596299,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9332123",\n      "kind": "school",\n      "name": "Школа №110",\n      "lat": 55.7592727,\n      "lon": 37.5967673,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9355953",\n      "kind": "kindergarten",\n      "name": "Школа № 518. Дошкольные группы",\n      "lat": 55.7463645,\n      "lon": 37.6303216,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9384689",\n      "kind": "kindergarten",\n      "name": "Детский сад №33",\n      "lat": 55.8907739,\n      "lon": 37.439701,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9418965",\n      "kind": "kindergarten",\n      "name": "Детский сад № 82",\n      "lat": 55.7127797,\n      "lon": 37.5177828,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9453295",\n      "kind": "school",\n      "name": "Школа №590",\n      "lat": 55.7404377,\n      "lon": 37.4956145,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9464683",\n      "kind": "school",\n      "name": "Школа с углублённым изучением иностранного языка № 1231 имени В. Д. Поленова",\n      "lat": 55.7441076,\n      "lon": 37.5976314,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9482182",\n      "kind": "school",\n      "name": "Школа № 627 имени генерала Д. Д. Лелюшенко. Учебный корпус № 6",\n      "lat": 55.7354889,\n      "lon": 37.6216001,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9489535",\n      "kind": "kindergarten",\n      "name": "Школа № 1799. Дошкольное отделение \\"Полянка\\"",\n      "lat": 55.7337689,\n      "lon": 37.6175331,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9569040",\n      "kind": "school",\n      "name": "Лицей № 1511",\n      "lat": 55.6484277,\n      "lon": 37.6522543,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9597254",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7677359,\n      "lon": 37.5563097,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9611653",\n      "kind": "school",\n      "name": "Школа №1284",\n      "lat": 55.7707099,\n      "lon": 37.6401554,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9628854",\n      "kind": "school",\n      "name": "Школа №\xa067. Отделение общего образования",\n      "lat": 55.7344278,\n      "lon": 37.5214083,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9628855",\n      "kind": "kindergarten",\n      "name": "Школа №\xa067. Отделение дошкольного образования",\n      "lat": 55.7348801,\n      "lon": 37.5209666,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9633273",\n      "kind": "school",\n      "name": "Школа №2054, Школьный корпус №1",\n      "lat": 55.7724802,\n      "lon": 37.6262314,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9652869",\n      "kind": "school",\n      "name": "Школа № 1259",\n      "lat": 55.7397462,\n      "lon": 37.6398702,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9656985",\n      "kind": "kindergarten",\n      "name": "Школа № 345 им. А.С. Пушкина. Дошкольное отделение",\n      "lat": 55.7796764,\n      "lon": 37.6683067,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9661837",\n      "kind": "school",\n      "name": "Колледж малого бизнеса № 4. Школьный корпус 2",\n      "lat": 55.7396251,\n      "lon": 37.6419205,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9694333",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8052642,\n      "lon": 37.5905905,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9812322",\n      "kind": "school",\n      "name": "Brookes Moscow",\n      "lat": 55.8482212,\n      "lon": 37.6331828,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9830040",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\"",\n      "lat": 55.7633591,\n      "lon": 37.648703,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9830043",\n      "kind": "kindergarten",\n      "name": "Школа \\"Покровский квартал\\". Детский сад № 2",\n      "lat": 55.7634365,\n      "lon": 37.6497945,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865157",\n      "kind": "kindergarten",\n      "name": "Детский сад №1632",\n      "lat": 55.8527453,\n      "lon": 37.6119168,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865158",\n      "kind": "school",\n      "name": "Школа № 950",\n      "lat": 55.8512506,\n      "lon": 37.6148638,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865159",\n      "kind": "school",\n      "name": "Центр образования № 264",\n      "lat": 55.8519235,\n      "lon": 37.6158783,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865160",\n      "kind": "kindergarten",\n      "name": "Детский сад №402",\n      "lat": 55.8583931,\n      "lon": 37.6160749,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865161",\n      "kind": "kindergarten",\n      "name": "Детский сад №1123",\n      "lat": 55.8589036,\n      "lon": 37.6144071,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865162",\n      "kind": "school",\n      "name": "Школа №950",\n      "lat": 55.8575581,\n      "lon": 37.6130821,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865163",\n      "kind": "school",\n      "name": "Школа №950",\n      "lat": 55.857052,\n      "lon": 37.6146396,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865164",\n      "kind": "kindergarten",\n      "name": "Детский сад №1125",\n      "lat": 55.7220863,\n      "lon": 37.8208855,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865165",\n      "kind": "school",\n      "name": "Школа №36",\n      "lat": 55.7212744,\n      "lon": 37.8199101,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865170",\n      "kind": "kindergarten",\n      "name": "Детский сад №761",\n      "lat": 55.7263712,\n      "lon": 37.8316232,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9865171",\n      "kind": "kindergarten",\n      "name": "Школа №402 имени Алии Молдагуловой (дошкольный корпус \\"Семицветик\\")",\n      "lat": 55.726888,\n      "lon": 37.8319292,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9866094",\n      "kind": "school",\n      "name": "Школа №969",\n      "lat": 55.8654822,\n      "lon": 37.61946,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9866095",\n      "kind": "school",\n      "name": "Школа №240",\n      "lat": 55.8656146,\n      "lon": 37.6218607,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/9897399",\n      "kind": "school",\n      "name": "Гбоу Гимназия № 1520 имени Капцовых",\n      "lat": 55.7564684,\n      "lon": 37.6029762,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10133035",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7246115,\n      "lon": 37.7419583,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10165007",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6508734,\n      "lon": 37.6022408,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10377236",\n      "kind": "school",\n      "name": "Московская экономическая школа",\n      "lat": 55.760909,\n      "lon": 37.5662638,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10391671",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 3 «Мозаика»",\n      "lat": 55.7644286,\n      "lon": 37.5725901,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10523482",\n      "kind": "kindergarten",\n      "name": "Феникс",\n      "lat": 55.5841196,\n      "lon": 37.6031506,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10524778",\n      "kind": "kindergarten",\n      "name": "Школа № 1530 \\"Школа Ломоносова\\". Дошкольные группы на Егерской",\n      "lat": 55.7927301,\n      "lon": 37.683713,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10524779",\n      "kind": "school",\n      "name": "Школа № 1530 \\"Школа Ломоносова\\". Школа на Егерской",\n      "lat": 55.7933156,\n      "lon": 37.6849364,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10524782",\n      "kind": "kindergarten",\n      "name": "Пушкинская школа № 1500. Детский сад",\n      "lat": 55.7838364,\n      "lon": 37.6716538,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10545944",\n      "kind": "school",\n      "name": "Школа № 1601 имени Героя Советского Союза Е. К. Лютикова",\n      "lat": 55.8007293,\n      "lon": 37.5573277,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10559479",\n      "kind": "kindergarten",\n      "name": "Leaders International School",\n      "lat": 55.778889,\n      "lon": 37.5749262,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10563386",\n      "kind": "kindergarten",\n      "name": "Школа № 149 имени Героя Советского Союза Ю. Н. Зыкова. Корпус Левитана 3А",\n      "lat": 55.8008619,\n      "lon": 37.500553,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10563387",\n      "kind": "kindergarten",\n      "name": "Школа № 149 имени Героя Советского Союза Ю. Н. Зыкова. Корпус Левитана 5А",\n      "lat": 55.801186,\n      "lon": 37.4992869,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10617155",\n      "kind": "school",\n      "name": "Школа № 1621 Древо Жизни",\n      "lat": 55.7648533,\n      "lon": 37.6493763,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10698703",\n      "kind": "school",\n      "name": "Школа №1282 \\"Сокольники\\"",\n      "lat": 55.7895468,\n      "lon": 37.6846896,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10723137",\n      "kind": "school",\n      "name": "Спортивный интернат ЦСКА",\n      "lat": 55.7974366,\n      "lon": 37.5562853,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10723138",\n      "kind": "school",\n      "name": "Лицей № 1575. Структурное подразделение дошкольного отделения",\n      "lat": 55.7978021,\n      "lon": 37.5549136,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10846794",\n      "kind": "kindergarten",\n      "name": "Детский сад №631",\n      "lat": 55.8397845,\n      "lon": 37.531076,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10906781",\n      "kind": "kindergarten",\n      "name": "ГБОУ СОШ № 2105 Дошкольный корпус №8,9",\n      "lat": 55.7773505,\n      "lon": 37.7059496,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10918094",\n      "kind": "school",\n      "name": "Инженерно-техническая школа имени дважды Героя Советского Союза П. Р. Поповича",\n      "lat": 55.8038797,\n      "lon": 37.5415481,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10981163",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Совёнок\\"",\n      "lat": 55.7899501,\n      "lon": 37.4899315,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/10981165",\n      "kind": "school",\n      "name": "Курчатовская школа корпус \\"5-й элемент\\"",\n      "lat": 55.8055921,\n      "lon": 37.4824775,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086454",\n      "kind": "school",\n      "name": "Школа №939",\n      "lat": 55.6160556,\n      "lon": 37.7258166,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086455",\n      "kind": "school",\n      "name": "Школа №438",\n      "lat": 55.6162396,\n      "lon": 37.7239994,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086459",\n      "kind": "kindergarten",\n      "name": "Детский сад №1077",\n      "lat": 55.6078815,\n      "lon": 37.7347008,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086461",\n      "kind": "school",\n      "name": "Центр образования № 548",\n      "lat": 55.6027936,\n      "lon": 37.7378758,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086463",\n      "kind": "kindergarten",\n      "name": "Детский сад №1114",\n      "lat": 55.6037931,\n      "lon": 37.7418284,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11086464",\n      "kind": "kindergarten",\n      "name": "Детский сад №1383",\n      "lat": 55.6040382,\n      "lon": 37.7431162,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11126283",\n      "kind": "school",\n      "name": "Центр образования №1404 \\"Гамма\\"",\n      "lat": 55.8039145,\n      "lon": 37.6881613,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11152601",\n      "kind": "school",\n      "name": "Центр образования №1404 \\"Гамма\\"",\n      "lat": 55.8079392,\n      "lon": 37.6869219,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11163001",\n      "kind": "school",\n      "name": "Школа № 1164",\n      "lat": 55.7976766,\n      "lon": 37.5742211,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11164803",\n      "kind": "school",\n      "name": "Центр образования №1601 (начальная школа)",\n      "lat": 55.7978457,\n      "lon": 37.5808804,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11181083",\n      "kind": "school",\n      "name": "Курчатовская школа корпус \\"Вектор\\"",\n      "lat": 55.7948237,\n      "lon": 37.48661,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11186468",\n      "kind": "kindergarten",\n      "name": "Bright Children Academy",\n      "lat": 55.7523385,\n      "lon": 37.7090175,\n      "opening_hours": "Mo-Fr 08:00-20:00; Sa 10:00-17:00"\n    },\n    {\n      "id": "relation/11187072",\n      "kind": "kindergarten",\n      "name": "Школа №1208 им. Героя Советского Союза М.С. Шумилова",\n      "lat": 55.709847,\n      "lon": 37.7769369,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11187073",\n      "kind": "school",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa03",\n      "lat": 55.7084957,\n      "lon": 37.7761886,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11207186",\n      "kind": "school",\n      "name": "Школа №439 \\"Интеллект\\" (корпус начального образования №2)",\n      "lat": 55.7389672,\n      "lon": 37.7408622,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11208335",\n      "kind": "school",\n      "name": "Школа №1530",\n      "lat": 55.7860481,\n      "lon": 37.685283,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11257412",\n      "kind": "school",\n      "name": "Гимназия №1530",\n      "lat": 55.7880045,\n      "lon": 37.6836278,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11268549",\n      "kind": "kindergarten",\n      "name": "Детский сад № 74",\n      "lat": 55.7863591,\n      "lon": 37.5926044,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11268551",\n      "kind": "school",\n      "name": "Гимназия № 1540",\n      "lat": 55.786841,\n      "lon": 37.5921546,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11279390",\n      "kind": "kindergarten",\n      "name": "Школа № 1540. Дошкольные группы",\n      "lat": 55.7872697,\n      "lon": 37.590681,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11284691",\n      "kind": "kindergarten",\n      "name": "Школа № 1321 \\"Ковчег\\". Дошкольное отделение здание №2",\n      "lat": 55.7472332,\n      "lon": 37.7183324,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11284692",\n      "kind": "kindergarten",\n      "name": "Школа № 1321 \\"Ковчег\\". Дошкольное отделение здание №1",\n      "lat": 55.7445728,\n      "lon": 37.7201853,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11292978",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7894355,\n      "lon": 37.7370935,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11295523",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7895842,\n      "lon": 37.740588,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11297320",\n      "kind": "school",\n      "name": "Школа №1228. Корпус 1",\n      "lat": 55.7408011,\n      "lon": 37.7199991,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11331272",\n      "kind": "school",\n      "name": "Школа-интернат №30 им. К. А. Микаэльяна",\n      "lat": 55.7995763,\n      "lon": 37.6925662,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11331273",\n      "kind": "kindergarten",\n      "name": "Детский сад №1860",\n      "lat": 55.7986335,\n      "lon": 37.6922682,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11342614",\n      "kind": "school",\n      "name": "Школа №457",\n      "lat": 55.7451548,\n      "lon": 37.6769573,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11355615",\n      "kind": "school",\n      "name": "Школа №354 имени Д.М. Карбышева ШО-1/1",\n      "lat": 55.7691447,\n      "lon": 37.6680972,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11356961",\n      "kind": "school",\n      "name": "Школа №760 им. А. П. Маресьева. Школьное отделение",\n      "lat": 55.8810918,\n      "lon": 37.7203272,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11356962",\n      "kind": "kindergarten",\n      "name": "Школа №760 им. А. П. Маресьева. Дошкольное отделение",\n      "lat": 55.8816904,\n      "lon": 37.7189245,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11370208",\n      "kind": "kindergarten",\n      "name": "Детский сад №115",\n      "lat": 55.7825002,\n      "lon": 37.6621864,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11478852",\n      "kind": "kindergarten",\n      "name": "ЧДОУ Детский сад №36 ОАО РЖД",\n      "lat": 55.7991085,\n      "lon": 37.6519322,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11490679",\n      "kind": "kindergarten",\n      "name": "Школа № 118. Дошкольное отделение",\n      "lat": 55.6872676,\n      "lon": 37.5311302,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11545654",\n      "kind": "kindergarten",\n      "name": "Brookes Life",\n      "lat": 55.8484077,\n      "lon": 37.6300392,\n      "opening_hours": "Mo-Fr 07:30-18:00"\n    },\n    {\n      "id": "relation/11550302",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8272414,\n      "lon": 37.6524778,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11561771",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.6976818,\n      "lon": 37.4918342,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11565410",\n      "kind": "school",\n      "name": "Школа №2094. Школьная образовательная площадка №1",\n      "lat": 55.6207443,\n      "lon": 37.4765156,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11601689",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa0624",\n      "lat": 55.8111301,\n      "lon": 37.574118,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11601690",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8108034,\n      "lon": 37.5747916,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11601691",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.810404,\n      "lon": 37.5754206,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11648361",\n      "kind": "kindergarten",\n      "name": "ГБУ Школа №1540 дошкольное отделение",\n      "lat": 55.7906654,\n      "lon": 37.5928186,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11654990",\n      "kind": "school",\n      "name": "Школа №1250",\n      "lat": 55.8293519,\n      "lon": 37.4846268,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11655326",\n      "kind": "school",\n      "name": "Школа № 46",\n      "lat": 55.6578013,\n      "lon": 37.5110612,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11679771",\n      "kind": "school",\n      "name": "Лицей №1502. Корпус \\"Гамма\\"",\n      "lat": 55.7980676,\n      "lon": 37.707357,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11698101",\n      "kind": "school",\n      "name": "Школа с углубленным изучением иностранного языка № 1236 имени С.В. Милашенкова. Школьное Отделение №230",\n      "lat": 55.8235514,\n      "lon": 37.5825689,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11730496",\n      "kind": "kindergarten",\n      "name": "Школа № 627 имени генерала Д.Д. Лелюшенко. Учебный корпус № 9",\n      "lat": 55.7327132,\n      "lon": 37.616815,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11749423",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7987357,\n      "lon": 37.5407179,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11749425",\n      "kind": "school",\n      "name": "i-Школа",\n      "lat": 55.8001555,\n      "lon": 37.5419484,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11750181",\n      "kind": "school",\n      "name": "Петровская школа",\n      "lat": 55.7973568,\n      "lon": 37.5635229,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11753300",\n      "kind": "kindergarten",\n      "name": "Детский сад 1920",\n      "lat": 55.7931333,\n      "lon": 37.5834959,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11778070",\n      "kind": "school",\n      "name": "ГБОУ Школа № 293 им. А. Т. Твардовского",\n      "lat": 55.8220118,\n      "lon": 37.6530343,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11805313",\n      "kind": "school",\n      "name": "Частная школа «Академическая гимназия»",\n      "lat": 55.8127078,\n      "lon": 37.6828974,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11807076",\n      "kind": "school",\n      "name": "Школа Марьина Роща имени В. Ф. Орлова. Отделение Берендеево царство",\n      "lat": 55.7938728,\n      "lon": 37.6216194,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11855555",\n      "kind": "school",\n      "name": "Пушкинская школа № 1500",\n      "lat": 55.7809283,\n      "lon": 37.6635559,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11880178",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1797 \\"Богородская\\" Дошкольное отделение",\n      "lat": 55.811311,\n      "lon": 37.7139661,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11909217",\n      "kind": "school",\n      "name": "Школа № 2107. Здание школы \\"Школа им. В. М. Комарова\\"",\n      "lat": 55.7875347,\n      "lon": 37.6303219,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11913160",\n      "kind": "school",\n      "name": "Школа №2200 корпус 2",\n      "lat": 55.8088228,\n      "lon": 37.8011446,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11951886",\n      "kind": "school",\n      "name": "Школа № 1297 основного и среднего общего образования",\n      "lat": 55.7875579,\n      "lon": 37.6371635,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11956021",\n      "kind": "school",\n      "name": "Школа №2200 корпус 9",\n      "lat": 55.8083187,\n      "lon": 37.8147277,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11956022",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 12 (дошкольное отделение)",\n      "lat": 55.8093652,\n      "lon": 37.8146346,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11956023",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 13 (дошкольное отделение)",\n      "lat": 55.8094517,\n      "lon": 37.8159521,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11970107",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 4 Семицветик",\n      "lat": 55.7637842,\n      "lon": 37.5646012,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11977541",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8638641,\n      "lon": 37.5355138,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11996436",\n      "kind": "kindergarten",\n      "name": "ЦРР-д/с №780",\n      "lat": 55.8301113,\n      "lon": 37.7632098,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/11997800",\n      "kind": "kindergarten",\n      "name": "Центр образования №345 (детский сад)",\n      "lat": 55.7743628,\n      "lon": 37.6757305,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12003323",\n      "kind": "kindergarten",\n      "name": "Школа №1429 имени Героя Советского Союза Н.А. Боброва (Дошкольное отделение)",\n      "lat": 55.7777536,\n      "lon": 37.682845,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12102755",\n      "kind": "school",\n      "name": "Школа \\"Венда\\"",\n      "lat": 55.8154275,\n      "lon": 37.6156453,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12106667",\n      "kind": "school",\n      "name": "Школа №315",\n      "lat": 55.7829817,\n      "lon": 37.6744233,\n      "opening_hours": "Mo-Su 07:30-21:30"\n    },\n    {\n      "id": "relation/12110737",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6009517,\n      "lon": 37.7308125,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12117271",\n      "kind": "kindergarten",\n      "name": "Центр образования №1601 (детский сад)",\n      "lat": 55.7963101,\n      "lon": 37.5613984,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12188454",\n      "kind": "kindergarten",\n      "name": "Детский сад AcademKids",\n      "lat": 55.8745656,\n      "lon": 37.5975229,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "relation/12217953",\n      "kind": "kindergarten",\n      "name": "Центр развития ребенка №430",\n      "lat": 55.7963431,\n      "lon": 37.5740919,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12223764",\n      "kind": "kindergarten",\n      "name": "Школа №1499 Дошкольное отделение",\n      "lat": 55.8380287,\n      "lon": 37.6517619,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12223765",\n      "kind": "school",\n      "name": "Школа №1499",\n      "lat": 55.8373491,\n      "lon": 37.6530058,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12231186",\n      "kind": "kindergarten",\n      "name": "Лучик",\n      "lat": 55.807892,\n      "lon": 37.5575178,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12250223",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.7017285,\n      "lon": 37.5027379,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12278837",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6936683,\n      "lon": 37.5727668,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12285338",\n      "kind": "kindergarten",\n      "name": "Детский сад №525",\n      "lat": 55.7966548,\n      "lon": 37.5648636,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12294630",\n      "kind": "kindergarten",\n      "name": "ДОП \\"Красностуденческая 2\\"",\n      "lat": 55.8268461,\n      "lon": 37.5675593,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12294632",\n      "kind": "school",\n      "name": "Кадетская школа-интернат № 1 \\"Первый Московский кадетский корпус",\n      "lat": 55.8151469,\n      "lon": 37.5590016,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12314255",\n      "kind": "school",\n      "name": "Школа №285 имени В.А. Молодцова",\n      "lat": 55.8771321,\n      "lon": 37.6182724,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12314803",\n      "kind": "kindergarten",\n      "name": "Остров сокровищ",\n      "lat": 55.8213995,\n      "lon": 37.5694061,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12344755",\n      "kind": "kindergarten",\n      "name": "Школа № 1259. Дошкольное отделение",\n      "lat": 55.7317743,\n      "lon": 37.6341318,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12347999",\n      "kind": "school",\n      "name": "Средняя общеобразовательная лингвистическая школа",\n      "lat": 55.7832479,\n      "lon": 37.6608182,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12360702",\n      "kind": "school",\n      "name": "Школа № 1228. Корпус № 4",\n      "lat": 55.7636365,\n      "lon": 37.7079688,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12367974",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа №1531\\", корпус 2",\n      "lat": 55.8085781,\n      "lon": 37.6312862,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12380317",\n      "kind": "school",\n      "name": "ГБОУ Инженерно-техническая школа имени дважды героя Советского Союза Павла Романовича Поповича",\n      "lat": 55.8097084,\n      "lon": 37.531497,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12380320",\n      "kind": "kindergarten",\n      "name": "Школа № 1601 имени Героя Советского Союза Е. К. Лютикова. Дошкольное отделение",\n      "lat": 55.8021578,\n      "lon": 37.556378,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12380321",\n      "kind": "school",\n      "name": "Гимназия Свет",\n      "lat": 55.8019017,\n      "lon": 37.5557474,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12417771",\n      "kind": "school",\n      "name": "Гимназия №1579",\n      "lat": 55.6468364,\n      "lon": 37.6595344,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12476264",\n      "kind": "kindergarten",\n      "name": "Детский сад №694",\n      "lat": 55.7746236,\n      "lon": 37.6461949,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12533163",\n      "kind": "kindergarten",\n      "name": "Детский сад №1727",\n      "lat": 55.7773447,\n      "lon": 37.5537523,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12580079",\n      "kind": "kindergarten",\n      "name": "Школа №1282 \\"Сокольники\\" (дошкольное отделение 3)",\n      "lat": 55.7950346,\n      "lon": 37.6561801,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12580080",\n      "kind": "kindergarten",\n      "name": "Школа №1282 \\"Сокольники\\" (дошкольное отделение 4)",\n      "lat": 55.7963166,\n      "lon": 37.655646,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12644180",\n      "kind": "kindergarten",\n      "name": "Школа № 345 имени А.С. Пушкина. Дошкольное отделение",\n      "lat": 55.7654735,\n      "lon": 37.6558909,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12661654",\n      "kind": "kindergarten",\n      "name": "Детский сад №306",\n      "lat": 55.7733982,\n      "lon": 37.7349016,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12661655",\n      "kind": "school",\n      "name": "Школа №1301 имени Е.Т. Гайдара",\n      "lat": 55.7731515,\n      "lon": 37.7338592,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12688429",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1290. Дошкольное отделение № 4/1",\n      "lat": 55.7982156,\n      "lon": 37.7635035,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12689325",\n      "kind": "school",\n      "name": "Школа № 547",\n      "lat": 55.5911562,\n      "lon": 37.4594967,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12704262",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е.В. Михайлова. Школьный корпус Рязанский",\n      "lat": 55.7187639,\n      "lon": 37.7933316,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12717206",\n      "kind": "kindergarten",\n      "name": "ДОУ №1733",\n      "lat": 55.7678399,\n      "lon": 37.6708091,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12730222",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8098502,\n      "lon": 37.7614967,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12739902",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Интеграл\\"",\n      "lat": 55.7083936,\n      "lon": 37.6192842,\n      "opening_hours": "Mo-Fr 07:30-19:30"\n    },\n    {\n      "id": "relation/12745351",\n      "kind": "school",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №1",\n      "lat": 55.8524168,\n      "lon": 37.6906082,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12745352",\n      "kind": "school",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №2",\n      "lat": 55.8515144,\n      "lon": 37.6883851,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12773236",\n      "kind": "kindergarten",\n      "name": "Детский сад №863",\n      "lat": 55.7093416,\n      "lon": 37.469002,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12921870",\n      "kind": "school",\n      "name": "ГБОУ школа на Юго-Востоке имени Маршала В.И. Чуйкова",\n      "lat": 55.6970664,\n      "lon": 37.7725554,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12966936",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №20",\n      "lat": 55.8253601,\n      "lon": 37.8105283,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12966937",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №18",\n      "lat": 55.8260773,\n      "lon": 37.8104018,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12966938",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №16",\n      "lat": 55.8249313,\n      "lon": 37.818907,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12966939",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №13",\n      "lat": 55.8257435,\n      "lon": 37.8188648,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12969784",\n      "kind": "school",\n      "name": "",\n      "lat": 55.65662,\n      "lon": 37.7559216,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12969785",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №10",\n      "lat": 55.6561894,\n      "lon": 37.7584447,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12969787",\n      "kind": "school",\n      "name": "Школа № 1357 «На Братиславской». Здание №15",\n      "lat": 55.666917,\n      "lon": 37.7775141,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980861",\n      "kind": "school",\n      "name": "Шуваловская школа № 1448. Корпус «Мосфильмовский»",\n      "lat": 55.7208063,\n      "lon": 37.5116901,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980862",\n      "kind": "kindergarten",\n      "name": "Шуваловская школа № 1448. Корпус «Мосфильмовский»",\n      "lat": 55.7213245,\n      "lon": 37.5126873,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980864",\n      "kind": "school",\n      "name": "Школа №37. Корпус школьного образования",\n      "lat": 55.6964498,\n      "lon": 37.4909324,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980875",\n      "kind": "kindergarten",\n      "name": "Шуваловская школа № 1448. Корпус «Шуваловский»",\n      "lat": 55.7030495,\n      "lon": 37.5130607,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980879",\n      "kind": "school",\n      "name": "Шуваловская школа № 1448. Основной корпус «Шуваловский»",\n      "lat": 55.7025401,\n      "lon": 37.5126036,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12980882",\n      "kind": "kindergarten",\n      "name": "Шуваловская школа № 1448. Корпус «Академия детства»",\n      "lat": 55.7017511,\n      "lon": 37.5130444,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994086",\n      "kind": "school",\n      "name": "Школа № 1248. Центр «Давыдково»",\n      "lat": 55.7238334,\n      "lon": 37.4664046,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994088",\n      "kind": "kindergarten",\n      "name": "Детский сад №553",\n      "lat": 55.7231641,\n      "lon": 37.4636622,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994089",\n      "kind": "kindergarten",\n      "name": "Школа № 1248. Здание дошкольных групп 7",\n      "lat": 55.7232068,\n      "lon": 37.4646303,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994090",\n      "kind": "kindergarten",\n      "name": "Школа № 1248. Здание дошкольных групп 5",\n      "lat": 55.7233884,\n      "lon": 37.4746995,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994098",\n      "kind": "kindergarten",\n      "name": "Школа № 1248. Здание дошкольных групп 4",\n      "lat": 55.7252384,\n      "lon": 37.4766505,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12994102",\n      "kind": "school",\n      "name": "Школа №1248. Школьное здание 1",\n      "lat": 55.7256354,\n      "lon": 37.4736251,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12998410",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение 3/5",\n      "lat": 55.6297755,\n      "lon": 37.4880568,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12998411",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение 3/4",\n      "lat": 55.6301752,\n      "lon": 37.4894585,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/12998414",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение 3/8",\n      "lat": 55.6344312,\n      "lon": 37.4864553,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004333",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №4",\n      "lat": 55.6122302,\n      "lon": 37.548642,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004334",\n      "kind": "kindergarten",\n      "name": "Школа № 1694 «Ясенево». Дошкольное отделение 6",\n      "lat": 55.6119901,\n      "lon": 37.5478155,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004335",\n      "kind": "school",\n      "name": "Школа № 1206. Корпус школьного образования №10 «Наш дом»",\n      "lat": 55.6134269,\n      "lon": 37.5517052,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004336",\n      "kind": "kindergarten",\n      "name": "Школа № 1694 «Ясенево». Дошкольное отделение 5",\n      "lat": 55.613174,\n      "lon": 37.5509399,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004340",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6142246,\n      "lon": 37.542505,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13004341",\n      "kind": "kindergarten",\n      "name": "Школа № 1694 «Ясенево». Дошкольное отделение 4",\n      "lat": 55.6147396,\n      "lon": 37.5428393,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13008017",\n      "kind": "school",\n      "name": "Школа № 17. Школьное здание №3",\n      "lat": 55.6353393,\n      "lon": 37.5283704,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13008020",\n      "kind": "school",\n      "name": "",\n      "lat": 55.636754,\n      "lon": 37.5315408,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13010684",\n      "kind": "school",\n      "name": "Школа №49. Школьный корпус",\n      "lat": 55.645723,\n      "lon": 37.5360213,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13011079",\n      "kind": "school",\n      "name": "Школа № 1273. Корпус школьного образования №8",\n      "lat": 55.6307482,\n      "lon": 37.5239063,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13024797",\n      "kind": "school",\n      "name": "Школа № 113. Корпус №2",\n      "lat": 55.6425203,\n      "lon": 37.5143281,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13024800",\n      "kind": "kindergarten",\n      "name": "Школа № 113. Корпус дошкольного образования №5",\n      "lat": 55.6439453,\n      "lon": 37.5176007,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13025525",\n      "kind": "school",\n      "name": "Школа № 1507. Корпус школьного образования",\n      "lat": 55.6270586,\n      "lon": 37.5042155,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13025527",\n      "kind": "school",\n      "name": "Школа № 1507. Корпус школьного образования",\n      "lat": 55.6303727,\n      "lon": 37.5065736,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13037120",\n      "kind": "kindergarten",\n      "name": "Детский сад «Белый кролик»",\n      "lat": 55.5914152,\n      "lon": 37.4622392,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13051360",\n      "kind": "kindergarten",\n      "name": "Школа №1995. Дошкольный уровень",\n      "lat": 55.6467494,\n      "lon": 37.4961936,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13051973",\n      "kind": "school",\n      "name": "Школа №109. Основная школа",\n      "lat": 55.6407698,\n      "lon": 37.4753687,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13057844",\n      "kind": "school",\n      "name": "Школа на Яузе корпус 7",\n      "lat": 55.7735435,\n      "lon": 37.7074005,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13075779",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6233417,\n      "lon": 37.4867878,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13075780",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/7",\n      "lat": 55.6236575,\n      "lon": 37.4878849,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13101201",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.7029865,\n      "lon": 37.5005314,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151519",\n      "kind": "kindergarten",\n      "name": "Школа № 1103. Корпус дошкольного образования №1",\n      "lat": 55.6048443,\n      "lon": 37.5476809,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151520",\n      "kind": "school",\n      "name": "Британская Международная школа",\n      "lat": 55.6046949,\n      "lon": 37.5487683,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151521",\n      "kind": "kindergarten",\n      "name": "Школа № 1103. Корпус дошкольного образования №4",\n      "lat": 55.6040143,\n      "lon": 37.5517382,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151522",\n      "kind": "kindergarten",\n      "name": "детский сад №1461",\n      "lat": 55.6038862,\n      "lon": 37.5528075,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151527",\n      "kind": "school",\n      "name": "Школа № 1103. Корпус школьного образования №7",\n      "lat": 55.6060392,\n      "lon": 37.5522041,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151528",\n      "kind": "school",\n      "name": "Школа № 1103. Корпус школьного образования №8",\n      "lat": 55.6058516,\n      "lon": 37.5539773,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151529",\n      "kind": "school",\n      "name": "Школа № 1103. Корпус школьного образования №9",\n      "lat": 55.6081561,\n      "lon": 37.5423683,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13151530",\n      "kind": "school",\n      "name": "Школа № 1103. Корпус школьного образования №10",\n      "lat": 55.607317,\n      "lon": 37.5420445,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13176656",\n      "kind": "kindergarten",\n      "name": "Школа № 627 имени генерала Д. Д. Лелюшенко. Учебный корпус № 8",\n      "lat": 55.7261421,\n      "lon": 37.620965,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179273",\n      "kind": "school",\n      "name": "Школа № 1206. Корпус школьного образования №2",\n      "lat": 55.6101516,\n      "lon": 37.5290254,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179274",\n      "kind": "school",\n      "name": "Школа № 1206. Корпус школьного образования №1",\n      "lat": 55.610966,\n      "lon": 37.5295453,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179275",\n      "kind": "school",\n      "name": "Школа № 1206. Корпус начального образования №3 «Ресурсный центр»",\n      "lat": 55.6164806,\n      "lon": 37.5372548,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179276",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №6",\n      "lat": 55.6165483,\n      "lon": 37.5363215,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179803",\n      "kind": "kindergarten",\n      "name": "Школа Тропарево. Корпус дошкольного образования",\n      "lat": 55.6524151,\n      "lon": 37.4685564,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13179804",\n      "kind": "school",\n      "name": "Школа Тропарево. Корпус школьного образования",\n      "lat": 55.6534828,\n      "lon": 37.4692649,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13180366",\n      "kind": "school",\n      "name": "Гимназия №201",\n      "lat": 55.8213088,\n      "lon": 37.5049687,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13188995",\n      "kind": "school",\n      "name": "Школа № 2107. Физико-математическая школа",\n      "lat": 55.7821585,\n      "lon": 37.6350583,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13211681",\n      "kind": "kindergarten",\n      "name": "Школа №556. Корпус дошкольного образования",\n      "lat": 55.6215044,\n      "lon": 37.5898665,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13211682",\n      "kind": "kindergarten",\n      "name": "Школа №556. Корпус дошкольного образования",\n      "lat": 55.6212446,\n      "lon": 37.5915229,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13234192",\n      "kind": "school",\n      "name": "ГБОУ Школа № 2129 ШО №3",\n      "lat": 55.7078821,\n      "lon": 37.675786,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13248115",\n      "kind": "school",\n      "name": "Школа №875. Корпус школьного образования",\n      "lat": 55.6638663,\n      "lon": 37.4911326,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13262831",\n      "kind": "kindergarten",\n      "name": "Детское посольство",\n      "lat": 55.8311798,\n      "lon": 37.6162923,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13347424",\n      "kind": "school",\n      "name": "Школа № 324 «Жар-птица». Корпус школьного образования «Журавушка»",\n      "lat": 55.6745677,\n      "lon": 37.4930911,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13347425",\n      "kind": "kindergarten",\n      "name": "Школа № 324 «Жар-птица». Корпус дошкольного образования «Соловушка»",\n      "lat": 55.6738448,\n      "lon": 37.4930525,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13372228",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №6",\n      "lat": 55.6103211,\n      "lon": 37.594732,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13372229",\n      "kind": "school",\n      "name": "Школа № 1582. Корпус школьного образования",\n      "lat": 55.6093183,\n      "lon": 37.5969963,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13401623",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №9",\n      "lat": 55.59885,\n      "lon": 37.6115738,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13401624",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.5986382,\n      "lon": 37.6129509,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13422176",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6054818,\n      "lon": 37.6181761,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13443141",\n      "kind": "kindergarten",\n      "name": "Гимназия № 2027. Дошкольное отделение № 3",\n      "lat": 55.7476059,\n      "lon": 37.7956939,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13470035",\n      "kind": "school",\n      "name": "Школа № 2107. Физико-математическая школа",\n      "lat": 55.7809762,\n      "lon": 37.6427738,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13472246",\n      "kind": "school",\n      "name": "Школа №1179. Корпус школьного образования №3",\n      "lat": 55.6378887,\n      "lon": 37.6132711,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13472247",\n      "kind": "kindergarten",\n      "name": "Детский сад № 853",\n      "lat": 55.6377669,\n      "lon": 37.6145602,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13492626",\n      "kind": "school",\n      "name": "Школа на Юго-Востоке имени Маршала В.И. Чуйкова",\n      "lat": 55.7118079,\n      "lon": 37.7669083,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13492627",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа имени Маршала Чуйкова, здание № 3",\n      "lat": 55.7119586,\n      "lon": 37.7644685,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13495476",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6499319,\n      "lon": 37.5740018,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13495477",\n      "kind": "school",\n      "name": "Школа № 538 имени С.В. Гришина. Корпус школьного образования",\n      "lat": 55.6500286,\n      "lon": 37.572277,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13496928",\n      "kind": "school",\n      "name": "Школа № 538 имени С.В. Гришина. Корпус школьного образования",\n      "lat": 55.6519599,\n      "lon": 37.5763527,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13496929",\n      "kind": "school",\n      "name": "Школа № 538 имени С.В. Гришина. Главный корпус школьного образования",\n      "lat": 55.6524845,\n      "lon": 37.577197,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13496986",\n      "kind": "kindergarten",\n      "name": "Школа № 538 имени С.В. Гришина. Корпус дошкольного образования",\n      "lat": 55.6531645,\n      "lon": 37.5731206,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13512774",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.841191,\n      "lon": 37.4974225,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13524468",\n      "kind": "school",\n      "name": "Школа № 1245. Корпус школьного образования №1",\n      "lat": 55.5997889,\n      "lon": 37.593382,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13524469",\n      "kind": "school",\n      "name": "Центр образования №504",\n      "lat": 55.6000823,\n      "lon": 37.5916094,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13549475",\n      "kind": "school",\n      "name": "Школа № 667. Корпус школьного образования №4",\n      "lat": 55.5837166,\n      "lon": 37.6427537,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13549476",\n      "kind": "kindergarten",\n      "name": "Школа № 667. Корпус дошкольного образования №6",\n      "lat": 55.5850103,\n      "lon": 37.6428602,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13549477",\n      "kind": "school",\n      "name": "Школа № 667. Корпус школьного образования №5",\n      "lat": 55.5848141,\n      "lon": 37.6416728,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13575969",\n      "kind": "school",\n      "name": "Школа № 1862. Корпус школьного образования №3",\n      "lat": 55.6464016,\n      "lon": 37.6144554,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13575970",\n      "kind": "school",\n      "name": "Школа № 1862. Корпус школьного образования №2",\n      "lat": 55.6478098,\n      "lon": 37.6144253,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13606724",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус школьного образования №1",\n      "lat": 55.6315816,\n      "lon": 37.5945193,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13653023",\n      "kind": "school",\n      "name": "Центр образования и спорта \\"Москва-98\\". Школа № 1",\n      "lat": 55.6846361,\n      "lon": 37.4943333,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13688258",\n      "kind": "kindergarten",\n      "name": "Школа № 1536. Корпус дошкольного образования",\n      "lat": 55.6810882,\n      "lon": 37.5435718,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13688266",\n      "kind": "kindergarten",\n      "name": "Школа № 1536. Корпус дошкольного образования",\n      "lat": 55.6815899,\n      "lon": 37.5421721,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13688421",\n      "kind": "school",\n      "name": "Школа № 7. Подразделение ул. Панфёрова, дом 8 к1",\n      "lat": 55.6811038,\n      "lon": 37.540903,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13719712",\n      "kind": "school",\n      "name": "Школа № 324 «Жар-птица». Корпус школьного образования «Жар-птица»",\n      "lat": 55.6730663,\n      "lon": 37.4928015,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13719919",\n      "kind": "kindergarten",\n      "name": "Школа № 1541. Корпус дошкольного образования №1",\n      "lat": 55.6757263,\n      "lon": 37.5133521,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13719951",\n      "kind": "kindergarten",\n      "name": "Школа № 1541. Корпус дошкольного образования №2",\n      "lat": 55.6751362,\n      "lon": 37.5143975,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13768991",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6697985,\n      "lon": 37.7222908,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13778835",\n      "kind": "kindergarten",\n      "name": "Детский сад «Горница-узорница»",\n      "lat": 55.7613907,\n      "lon": 37.5491346,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13802230",\n      "kind": "school",\n      "name": "ЦО №354",\n      "lat": 55.7694638,\n      "lon": 37.6779451,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13851281",\n      "kind": "school",\n      "name": "Школа №436",\n      "lat": 55.7121531,\n      "lon": 37.8057751,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13851369",\n      "kind": "kindergarten",\n      "name": "Центр образования №1998",\n      "lat": 55.6345523,\n      "lon": 37.7394872,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13874935",\n      "kind": "school",\n      "name": "Школа при посольстве Индии",\n      "lat": 55.7467413,\n      "lon": 37.5632821,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13936495",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8117601,\n      "lon": 37.739361,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13947476",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8036435,\n      "lon": 37.5964321,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13950665",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8012808,\n      "lon": 37.5994677,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/13983628",\n      "kind": "school",\n      "name": "Школа №522",\n      "lat": 55.6823227,\n      "lon": 37.5967786,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14060443",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение № 6",\n      "lat": 55.8621308,\n      "lon": 37.6981965,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14062188",\n      "kind": "kindergarten",\n      "name": "Школа №1503 Дошкольное отделение",\n      "lat": 55.8093945,\n      "lon": 37.6522298,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14075840",\n      "kind": "school",\n      "name": "Школа № 1503",\n      "lat": 55.8131357,\n      "lon": 37.6548427,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14075841",\n      "kind": "kindergarten",\n      "name": "Школа № 1503 Дошкольное отделение",\n      "lat": 55.8137677,\n      "lon": 37.6546553,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14080267",\n      "kind": "school",\n      "name": "Гимназия №1551 корпус №8 (начальная школа и основное общее)",\n      "lat": 55.8395614,\n      "lon": 37.4566219,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14097751",\n      "kind": "school",\n      "name": "Лицей № 1581 имени Ивана Бунина",\n      "lat": 55.7789394,\n      "lon": 37.6955034,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14097752",\n      "kind": "school",\n      "name": "Школа №1204",\n      "lat": 55.7728316,\n      "lon": 37.6914015,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14108489",\n      "kind": "kindergarten",\n      "name": "Планета детства",\n      "lat": 55.7326403,\n      "lon": 37.6316154,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14124596",\n      "kind": "school",\n      "name": "Школа № 518",\n      "lat": 55.7438014,\n      "lon": 37.6371272,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14134782",\n      "kind": "kindergarten",\n      "name": "Детский сад №30",\n      "lat": 55.776776,\n      "lon": 37.6857492,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14134784",\n      "kind": "kindergarten",\n      "name": "Детский сад №1308",\n      "lat": 55.772204,\n      "lon": 37.7104349,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14179520",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7504759,\n      "lon": 37.5166289,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14227468",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6580704,\n      "lon": 37.5583023,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14237997",\n      "kind": "school",\n      "name": "Школа № 1362",\n      "lat": 55.7819004,\n      "lon": 37.7323858,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14275411",\n      "kind": "school",\n      "name": "КиберШкола",\n      "lat": 55.8070538,\n      "lon": 37.7865574,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14275412",\n      "kind": "kindergarten",\n      "name": "Центр \\"Исток\\"",\n      "lat": 55.8070472,\n      "lon": 37.7852393,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14328396",\n      "kind": "school",\n      "name": "Школа № 1236 имени С.В. Милашенкова. Учебный корпус № 963",\n      "lat": 55.8248631,\n      "lon": 37.5758929,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14328397",\n      "kind": "kindergarten",\n      "name": "Школа № 1236 имени С.В. Милашенкова. Учебный корпус № 567",\n      "lat": 55.8252687,\n      "lon": 37.5763221,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14331020",\n      "kind": "kindergarten",\n      "name": "Школа № 158. Дошкольное отделение",\n      "lat": 55.8686474,\n      "lon": 37.4703356,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14331021",\n      "kind": "school",\n      "name": "Школа № 158. Школьное отделение № 4",\n      "lat": 55.867572,\n      "lon": 37.4707942,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14331022",\n      "kind": "school",\n      "name": "Школа № 158. Школьное отделение № 1",\n      "lat": 55.8681307,\n      "lon": 37.4684484,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14331375",\n      "kind": "school",\n      "name": "Школа № 1474. Подразделение 4",\n      "lat": 55.8721899,\n      "lon": 37.4900897,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14333603",\n      "kind": "school",\n      "name": "Школа №2098 корпус 5",\n      "lat": 55.8788068,\n      "lon": 37.5104288,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14336312",\n      "kind": "school",\n      "name": "Школа №\xa067. Подразделение №\xa02",\n      "lat": 55.7367153,\n      "lon": 37.528245,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14352898",\n      "kind": "kindergarten",\n      "name": "Школа № 2104 на Таганке. Корпус 1",\n      "lat": 55.7445795,\n      "lon": 37.6458444,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14358472",\n      "kind": "school",\n      "name": "Школа №1474",\n      "lat": 55.8707703,\n      "lon": 37.4913039,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14374244",\n      "kind": "school",\n      "name": "Пушкинская школа № 1500. Отделение \\"Сретенка\\"",\n      "lat": 55.769069,\n      "lon": 37.6328317,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14374917",\n      "kind": "school",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 3",\n      "lat": 55.7355099,\n      "lon": 37.5979576,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14389122",\n      "kind": "kindergarten",\n      "name": "Детский сад «Горница-Узорница»",\n      "lat": 55.7906043,\n      "lon": 37.4760979,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14397087",\n      "kind": "kindergarten",\n      "name": "Маленькая страна",\n      "lat": 55.8789834,\n      "lon": 37.539014,\n      "opening_hours": "Mo-Fr 07:00-20:00"\n    },\n    {\n      "id": "relation/14397259",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Чудо городок\\"",\n      "lat": 55.8779173,\n      "lon": 37.5386431,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14418711",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8160949,\n      "lon": 37.7524524,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14573448",\n      "kind": "school",\n      "name": "Школа №57 (здание №1)",\n      "lat": 55.7487141,\n      "lon": 37.6050056,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14585413",\n      "kind": "school",\n      "name": "W.I.D.E. School",\n      "lat": 55.8755887,\n      "lon": 37.6008585,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14585594",\n      "kind": "kindergarten",\n      "name": "Школа № 1465 имени адмирала Н.Г. Кузнецова. Дошкольное отделение № 4",\n      "lat": 55.7415939,\n      "lon": 37.5418569,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14598740",\n      "kind": "kindergarten",\n      "name": "W.I.D.E.",\n      "lat": 55.8516147,\n      "lon": 37.6191599,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14599775",\n      "kind": "kindergarten",\n      "name": "Школа № 1499 имени Героя Советского Союза Ивана Архиповича Докукина. Дошкольное отделение № 8",\n      "lat": 55.8354344,\n      "lon": 37.6414028,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14606511",\n      "kind": "school",\n      "name": "Школа № 120",\n      "lat": 55.6881157,\n      "lon": 37.5510905,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14635035",\n      "kind": "kindergarten",\n      "name": "Школа им. Ф. М. Достоевского. Учебный корпус № 9",\n      "lat": 55.6616762,\n      "lon": 37.7278496,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14663499",\n      "kind": "kindergarten",\n      "name": "Школа № 1465 имени адмирала Н.Г. Кузнецова. Здание на Студенческой, 29",\n      "lat": 55.7395482,\n      "lon": 37.5467823,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14663500",\n      "kind": "school",\n      "name": "Школа № 1465 имени адмирала Н.Г. Кузнецова. Здание на Студенческой, 29",\n      "lat": 55.7400871,\n      "lon": 37.5456408,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14680298",\n      "kind": "kindergarten",\n      "name": "Школа № 1530 Дошкольная группа",\n      "lat": 55.7849156,\n      "lon": 37.6764424,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14926150",\n      "kind": "kindergarten",\n      "name": "Детский сад «Sun School»",\n      "lat": 55.6828952,\n      "lon": 37.5824489,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14956586",\n      "kind": "school",\n      "name": "Центр образования №345",\n      "lat": 55.7731486,\n      "lon": 37.6724634,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14956589",\n      "kind": "school",\n      "name": "Школа №\xa0345 им. А.\xa0С.\xa0Пушкина",\n      "lat": 55.7707133,\n      "lon": 37.6775854,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14962273",\n      "kind": "school",\n      "name": "Школа № 158. Школьное отделение № 3",\n      "lat": 55.8735021,\n      "lon": 37.4674782,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14962274",\n      "kind": "school",\n      "name": "Школа № 158. Школьное отделение № 2",\n      "lat": 55.8735256,\n      "lon": 37.4652311,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14962275",\n      "kind": "kindergarten",\n      "name": "Школа № 158. Дошкольное отделение № 2",\n      "lat": 55.8729118,\n      "lon": 37.4659243,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14962523",\n      "kind": "kindergarten",\n      "name": "Детский сад «Академическая гимназия»",\n      "lat": 55.6010535,\n      "lon": 37.7261168,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14977370",\n      "kind": "school",\n      "name": "Школа № 1501. Многопрофильные классы на Тихвинской",\n      "lat": 55.7916642,\n      "lon": 37.5982605,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14978852",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6014696,\n      "lon": 37.7335755,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14983573",\n      "kind": "kindergarten",\n      "name": "Частный детский сад «Горница-Узорница»",\n      "lat": 55.6691946,\n      "lon": 37.7717367,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/14989885",\n      "kind": "school",\n      "name": "Лицей №1550, подр №2",\n      "lat": 55.7853699,\n      "lon": 37.5586262,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15001788",\n      "kind": "school",\n      "name": "Школа №152",\n      "lat": 55.8008186,\n      "lon": 37.5352032,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15013450",\n      "kind": "school",\n      "name": "Школа №1690",\n      "lat": 55.7926731,\n      "lon": 37.7251975,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15040197",\n      "kind": "kindergarten",\n      "name": "ONE! International School",\n      "lat": 55.7870994,\n      "lon": 37.4961103,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "relation/15045970",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7870087,\n      "lon": 37.619206,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15046320",\n      "kind": "kindergarten",\n      "name": "Детский сад №2718",\n      "lat": 55.7871198,\n      "lon": 37.599392,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15049348",\n      "kind": "kindergarten",\n      "name": "ONE! International School",\n      "lat": 55.786609,\n      "lon": 37.4937838,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "relation/15049712",\n      "kind": "school",\n      "name": "Школа №1287 (2)",\n      "lat": 55.7988898,\n      "lon": 37.5254497,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15049713",\n      "kind": "school",\n      "name": "Школа №1384 имени А. А. Леманского",\n      "lat": 55.7982404,\n      "lon": 37.5234806,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15087742",\n      "kind": "kindergarten",\n      "name": "Детский сад №864",\n      "lat": 55.8328953,\n      "lon": 37.5852464,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15087743",\n      "kind": "kindergarten",\n      "name": "Детский сад №2154",\n      "lat": 55.8328628,\n      "lon": 37.5871574,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15284663",\n      "kind": "school",\n      "name": "Школа №1228 \\"Лефортово\\"",\n      "lat": 55.7453136,\n      "lon": 37.7039961,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15285093",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7504735,\n      "lon": 37.7040729,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15326688",\n      "kind": "kindergarten",\n      "name": "Детский сад №2663",\n      "lat": 55.7747433,\n      "lon": 37.6800078,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15348984",\n      "kind": "kindergarten",\n      "name": "Детский сад",\n      "lat": 55.8610406,\n      "lon": 37.5398508,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15349064",\n      "kind": "school",\n      "name": "Школа №1236",\n      "lat": 55.8174682,\n      "lon": 37.582646,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15450200",\n      "kind": "school",\n      "name": "Центр образования №1480",\n      "lat": 55.7672761,\n      "lon": 37.6709812,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15451606",\n      "kind": "kindergarten",\n      "name": "ДОУ №\xa02555",\n      "lat": 55.7672952,\n      "lon": 37.6760037,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15518043",\n      "kind": "kindergarten",\n      "name": "Детский сад №1253",\n      "lat": 55.8185327,\n      "lon": 37.5635915,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15543373",\n      "kind": "school",\n      "name": "Бауманская инженерная школа № 1580. Корпус 3",\n      "lat": 55.7137027,\n      "lon": 37.5953765,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15671989",\n      "kind": "school",\n      "name": "Школа «Алгоритм»",\n      "lat": 55.7647665,\n      "lon": 37.6332242,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15671999",\n      "kind": "school",\n      "name": "Французский лицей им. А. Дюма при посольстве Франции",\n      "lat": 55.7630565,\n      "lon": 37.6316021,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15691315",\n      "kind": "school",\n      "name": "Школа №285 имени В.А. Молодцова, дополнительное образование",\n      "lat": 55.8780127,\n      "lon": 37.6245895,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15697831",\n      "kind": "school",\n      "name": "Школа № 2104 на Таганке. Корпус 2",\n      "lat": 55.7436285,\n      "lon": 37.6462823,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15700997",\n      "kind": "school",\n      "name": "Школа № 1271",\n      "lat": 55.7421669,\n      "lon": 37.6488379,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15710120",\n      "kind": "school",\n      "name": "Кадетский корпус Следственного комитета РФ им. Александра Невского",\n      "lat": 55.6977469,\n      "lon": 37.7582214,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15710121",\n      "kind": "school",\n      "name": "Школа №466",\n      "lat": 55.6987181,\n      "lon": 37.757896,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15737346",\n      "kind": "kindergarten",\n      "name": "Частное общеобразовательное учреждение «Венда»",\n      "lat": 55.8491512,\n      "lon": 37.5781226,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15767588",\n      "kind": "kindergarten",\n      "name": "Школа № 56 имени академика В.А. Легасова. Дошкольный корпус",\n      "lat": 55.7435864,\n      "lon": 37.5406149,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15898872",\n      "kind": "kindergarten",\n      "name": "Детский сад №2144",\n      "lat": 55.799081,\n      "lon": 37.6172375,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15913197",\n      "kind": "kindergarten",\n      "name": "Детский сад №133",\n      "lat": 55.7192268,\n      "lon": 37.7959831,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15922271",\n      "kind": "kindergarten",\n      "name": "Школа № 1317. Дошкольное отделение",\n      "lat": 55.6707721,\n      "lon": 37.4696285,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15922272",\n      "kind": "school",\n      "name": "Школа № 1317. Начальное отделение",\n      "lat": 55.6713862,\n      "lon": 37.4705109,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/15968755",\n      "kind": "school",\n      "name": "Школа № 1501. Инженерные и ИТ-классы",\n      "lat": 55.7877555,\n      "lon": 37.5957341,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16094457",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение В4",\n      "lat": 55.735661,\n      "lon": 37.8184396,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16125998",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Д.А.Р.»",\n      "lat": 55.6320413,\n      "lon": 37.5875136,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16145301",\n      "kind": "school",\n      "name": "Школа № 2099. Здание 2",\n      "lat": 55.8199537,\n      "lon": 37.4910476,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16257777",\n      "kind": "school",\n      "name": "Школа №\xa01203",\n      "lat": 55.7688158,\n      "lon": 37.6880488,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16270432",\n      "kind": "school",\n      "name": "Лицей №1574",\n      "lat": 55.7797743,\n      "lon": 37.5968479,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16330701",\n      "kind": "kindergarten",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 1",\n      "lat": 55.7431445,\n      "lon": 37.5947904,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16409165",\n      "kind": "kindergarten",\n      "name": "Школа \\"Покровский квартал\\", Детский сад №2",\n      "lat": 55.7638996,\n      "lon": 37.653529,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16409205",\n      "kind": "school",\n      "name": "Школа №\xa01388",\n      "lat": 55.776072,\n      "lon": 37.6047918,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16508347",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.681964,\n      "lon": 37.7198444,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16542182",\n      "kind": "school",\n      "name": "Татьянинская школа",\n      "lat": 55.7339582,\n      "lon": 37.7481277,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16617553",\n      "kind": "school",\n      "name": "Французский лицей им. Александра Дюма",\n      "lat": 55.7336784,\n      "lon": 37.6157276,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16769249",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7040582,\n      "lon": 37.6707193,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16844710",\n      "kind": "kindergarten",\n      "name": "Детский сад №23",\n      "lat": 55.6162384,\n      "lon": 37.6987172,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16867327",\n      "kind": "kindergarten",\n      "name": "Детский сад №2724",\n      "lat": 55.7106797,\n      "lon": 37.7393524,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/16983105",\n      "kind": "kindergarten",\n      "name": "Школа № 1799. Дошкольный корпус",\n      "lat": 55.7326525,\n      "lon": 37.6099083,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17034920",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7244379,\n      "lon": 37.748801,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17057701",\n      "kind": "school",\n      "name": "Греко-латинский кабинет Ю. А. Шичалина",\n      "lat": 55.7839377,\n      "lon": 37.6402746,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17087636",\n      "kind": "kindergarten",\n      "name": "Детский сад №1392",\n      "lat": 55.8122522,\n      "lon": 37.7037854,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17087637",\n      "kind": "kindergarten",\n      "name": "Детский сад №1299",\n      "lat": 55.8117206,\n      "lon": 37.7048854,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17138480",\n      "kind": "school",\n      "name": "Школа № 1297 для детей с особыми образовательными потребностями",\n      "lat": 55.789311,\n      "lon": 37.6377293,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17138481",\n      "kind": "kindergarten",\n      "name": "Школа № 1297 дошкольные группы",\n      "lat": 55.7892993,\n      "lon": 37.638729,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17232824",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01252 имени Сервантеса. Начальная школа",\n      "lat": 55.8103018,\n      "lon": 37.4948845,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17237320",\n      "kind": "kindergarten",\n      "name": "Протон - дошкольные группы",\n      "lat": 55.7480479,\n      "lon": 37.5023391,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17237321",\n      "kind": "kindergarten",\n      "name": "Детский сад №514",\n      "lat": 55.7471283,\n      "lon": 37.5032456,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17287634",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Brosco Junior\\"",\n      "lat": 55.7687016,\n      "lon": 37.6253893,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17386620",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7525562,\n      "lon": 37.5101156,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17473831",\n      "kind": "school",\n      "name": "Школа № 627 имени генерала Д. Д. Лелюшенко. Учебный корпус № 4",\n      "lat": 55.7260108,\n      "lon": 37.6275672,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17499698",\n      "kind": "school",\n      "name": "Школа № 1576",\n      "lat": 55.841583,\n      "lon": 37.5419044,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17499699",\n      "kind": "kindergarten",\n      "name": "Школа №1576, дошкольное отделение",\n      "lat": 55.8408704,\n      "lon": 37.5411516,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17501327",\n      "kind": "school",\n      "name": "Школа № 1241 «На Красной Пресне». Корпус 1",\n      "lat": 55.7601254,\n      "lon": 37.5636358,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17561174",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa04",\n      "lat": 55.7084812,\n      "lon": 37.7702523,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17561175",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa06",\n      "lat": 55.7063442,\n      "lon": 37.7749862,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17561176",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa07",\n      "lat": 55.7070212,\n      "lon": 37.7738603,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17561177",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7072764,\n      "lon": 37.7717679,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17561178",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa05",\n      "lat": 55.7062367,\n      "lon": 37.7737886,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17577569",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2",\n      "lat": 55.8815516,\n      "lon": 37.4949445,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17620396",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7000616,\n      "lon": 37.555451,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17626072",\n      "kind": "kindergarten",\n      "name": "Школа №1550 Дошкольное отделение",\n      "lat": 55.7929019,\n      "lon": 37.7278789,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17626073",\n      "kind": "kindergarten",\n      "name": "Детский сад №2518",\n      "lat": 55.7935209,\n      "lon": 37.7273127,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17671292",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01231 имени В.\xa0Д.\xa0Поленова. Дошкольный корпус",\n      "lat": 55.7438277,\n      "lon": 37.5801976,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17682458",\n      "kind": "school",\n      "name": "Школа №\xa01231 имени В. Д. Поленова. Школьный корпус",\n      "lat": 55.7418873,\n      "lon": 37.5807267,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17695569",\n      "kind": "kindergarten",\n      "name": "Детский сад №890",\n      "lat": 55.8074198,\n      "lon": 37.4554918,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17734461",\n      "kind": "kindergarten",\n      "name": "Детский сад №271",\n      "lat": 55.8177234,\n      "lon": 37.6423024,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17734508",\n      "kind": "school",\n      "name": "Школа №293 имени Твардовского",\n      "lat": 55.8178531,\n      "lon": 37.6404351,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17789214",\n      "kind": "kindergarten",\n      "name": "Детский сад №1132",\n      "lat": 55.8176934,\n      "lon": 37.6243185,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17807277",\n      "kind": "kindergarten",\n      "name": "Детский сад № 116",\n      "lat": 55.8468676,\n      "lon": 37.5659983,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17824771",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.756781,\n      "lon": 37.7068846,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17852619",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7195742,\n      "lon": 37.6899939,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17855597",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7185499,\n      "lon": 37.6564335,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17865216",\n      "kind": "kindergarten",\n      "name": "Детское посольство",\n      "lat": 55.8321029,\n      "lon": 37.6174232,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17865577",\n      "kind": "school",\n      "name": "Школа №1699",\n      "lat": 55.727948,\n      "lon": 37.484443,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17866091",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7184766,\n      "lon": 37.637083,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17940119",\n      "kind": "school",\n      "name": "Школа № 2107. Школа на Гиляровского",\n      "lat": 55.7776722,\n      "lon": 37.6298123,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17958851",\n      "kind": "school",\n      "name": "Школа № 1465 имени адмирала Н.Г. Кузнецова. Корпус 3",\n      "lat": 55.7428186,\n      "lon": 37.5470449,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/17987677",\n      "kind": "school",\n      "name": "Лингвистический лицей № 1555 при МГЛУ",\n      "lat": 55.7374677,\n      "lon": 37.598308,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18096256",\n      "kind": "school",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 5",\n      "lat": 55.7414824,\n      "lon": 37.6033899,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18122613",\n      "kind": "school",\n      "name": "Школа №319 (здание №1)",\n      "lat": 55.8067124,\n      "lon": 37.7591769,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18133893",\n      "kind": "school",\n      "name": "Школа № 215",\n      "lat": 55.8361889,\n      "lon": 37.527125,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18134842",\n      "kind": "school",\n      "name": "Школа № 1553 имени В.И. Вернадского",\n      "lat": 55.720123,\n      "lon": 37.6138718,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18163741",\n      "kind": "kindergarten",\n      "name": "Школа № 1223. Отделение дошкольного образования №1",\n      "lat": 55.8378584,\n      "lon": 37.5291441,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18178325",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7191932,\n      "lon": 37.606956,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18212679",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Варшавке\\"",\n      "lat": 55.6969219,\n      "lon": 37.6193182,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18244733",\n      "kind": "school",\n      "name": "Школа № 444. Начальная школа",\n      "lat": 55.7900437,\n      "lon": 37.7926185,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18282668",\n      "kind": "school",\n      "name": "Школа им. Ф. М. Достоевского. Учебный корпус № 10",\n      "lat": 55.6599807,\n      "lon": 37.7265512,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18302142",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6994816,\n      "lon": 37.6329135,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18396247",\n      "kind": "school",\n      "name": "Школа №745",\n      "lat": 55.865519,\n      "lon": 37.587801,\n      "opening_hours": null\n    },\n    {\n      "id": "relation/18451116",\n      "kind": "school",\n      "name": "Школа №\xa01282 \\"Сокольники\\" (главное здание)",\n      "lat": 55.789805,\n      "lon": 37.6697848,\n      "opening_hours": null\n    },\n    {\n      "id": "way/23426808",\n      "kind": "kindergarten",\n      "name": "Д/C № 597",\n      "lat": 55.6352633,\n      "lon": 37.664568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/23768961",\n      "kind": "school",\n      "name": "Детский сад №970",\n      "lat": 55.7905444,\n      "lon": 37.8040977,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27256911",\n      "kind": "kindergarten",\n      "name": "Детский сад №1267",\n      "lat": 55.6178306,\n      "lon": 37.7315572,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27256934",\n      "kind": "kindergarten",\n      "name": "Детский сад №1966",\n      "lat": 55.6176794,\n      "lon": 37.7328893,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27474375",\n      "kind": "school",\n      "name": "Школа-интернат \\"Интеллектуал\\"",\n      "lat": 55.7181295,\n      "lon": 37.4647244,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27546250",\n      "kind": "school",\n      "name": "школа №138",\n      "lat": 55.7812025,\n      "lon": 37.4644698,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27548429",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.720522,\n      "lon": 37.4580375,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27548438",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7199881,\n      "lon": 37.4583296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27548539",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7198279,\n      "lon": 37.4536159,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27549008",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7229605,\n      "lon": 37.4550246,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27549026",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7232209,\n      "lon": 37.4560019,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27560499",\n      "kind": "school",\n      "name": "Школа №467",\n      "lat": 55.7289891,\n      "lon": 37.6713802,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27569168",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7286567,\n      "lon": 37.4832217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27575988",\n      "kind": "kindergarten",\n      "name": "Детский сад №2210",\n      "lat": 55.7301274,\n      "lon": 37.6714715,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27651219",\n      "kind": "school",\n      "name": "Школа № 1288, корпус № 5",\n      "lat": 55.7776147,\n      "lon": 37.5291545,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27728924",\n      "kind": "kindergarten",\n      "name": "Детский сад №1888",\n      "lat": 55.799537,\n      "lon": 37.4900072,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27728931",\n      "kind": "school",\n      "name": "Школа №1212",\n      "lat": 55.7991425,\n      "lon": 37.487259,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27760624",\n      "kind": "kindergarten",\n      "name": "ДОУ 1506",\n      "lat": 55.6610066,\n      "lon": 37.5261642,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27787667",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7073227,\n      "lon": 37.4680125,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27787837",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7056023,\n      "lon": 37.4740525,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27787841",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7062635,\n      "lon": 37.4728918,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27793471",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7118038,\n      "lon": 37.4828666,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27793477",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7113993,\n      "lon": 37.4841003,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27800688",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8074512,\n      "lon": 37.4796295,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27804313",\n      "kind": "school",\n      "name": "Школа 1910",\n      "lat": 55.650675,\n      "lon": 37.7620407,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27810681",\n      "kind": "school",\n      "name": "Дом-интернат №21 для умственно отсталых детей",\n      "lat": 55.8074948,\n      "lon": 37.4808947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27810687",\n      "kind": "school",\n      "name": "Школа Ювенес Начальные классы",\n      "lat": 55.8067105,\n      "lon": 37.4828153,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27811752",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6550565,\n      "lon": 37.750546,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27811792",\n      "kind": "school",\n      "name": "Школа №1967",\n      "lat": 55.6629901,\n      "lon": 37.7507997,\n      "opening_hours": null\n    },\n    {\n      "id": "way/27811803",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6634465,\n      "lon": 37.7476784,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28115021",\n      "kind": "school",\n      "name": "Центр ЮССТ",\n      "lat": 55.7157512,\n      "lon": 37.4403522,\n      "opening_hours": "Mo-Fr 07:00-18:00; Sa-Su 00:00-24:00"\n    },\n    {\n      "id": "way/28150399",\n      "kind": "school",\n      "name": "Школа №90 им. Героя Советского Союза Е.Г. Ларикова",\n      "lat": 55.8341134,\n      "lon": 37.5256301,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28150828",\n      "kind": "kindergarten",\n      "name": "Детский сад №831",\n      "lat": 55.8390877,\n      "lon": 37.5313954,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28151565",\n      "kind": "kindergarten",\n      "name": "Центр образования №1679 Школа здоровья (детский сад)",\n      "lat": 55.8283668,\n      "lon": 37.5213005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28187032",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8699297,\n      "lon": 37.7057741,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28187033",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8706284,\n      "lon": 37.7044591,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28194282",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8660446,\n      "lon": 37.5635627,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28276380",\n      "kind": "school",\n      "name": "ГБОУ г. Москвы Школа № 1159",\n      "lat": 55.8592247,\n      "lon": 37.5231898,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28276418",\n      "kind": "kindergarten",\n      "name": "ГБОУ г. Москвы Школа № 1159 (ДО)",\n      "lat": 55.8602149,\n      "lon": 37.5229593,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28287586",\n      "kind": "kindergarten",\n      "name": "Детский сад №1938",\n      "lat": 55.7339579,\n      "lon": 37.486005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28287588",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7351321,\n      "lon": 37.4836517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28305744",\n      "kind": "school",\n      "name": "школа №887",\n      "lat": 55.7321563,\n      "lon": 37.4396771,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28317427",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1628",\n      "lat": 55.6508731,\n      "lon": 37.7312477,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28377590",\n      "kind": "school",\n      "name": "Школа № 1321. Филиал № 3",\n      "lat": 55.6721357,\n      "lon": 37.7500955,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28379114",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7246288,\n      "lon": 37.442373,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28379124",\n      "kind": "school",\n      "name": "Школа №888",\n      "lat": 55.7248085,\n      "lon": 37.443918,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28395095",\n      "kind": "kindergarten",\n      "name": "Детский сад №2",\n      "lat": 55.67084,\n      "lon": 37.7498762,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28587796",\n      "kind": "school",\n      "name": "Гимназия №1583",\n      "lat": 55.8499085,\n      "lon": 37.4945518,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28683132",\n      "kind": "school",\n      "name": "Школа №830 корпус №4",\n      "lat": 55.8141499,\n      "lon": 37.4593779,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28858683",\n      "kind": "school",\n      "name": "Школа №1360",\n      "lat": 55.8126546,\n      "lon": 37.702082,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28859009",\n      "kind": "school",\n      "name": "Школа №386",\n      "lat": 55.8095089,\n      "lon": 37.7203059,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28879187",\n      "kind": "school",\n      "name": "Школа №1315",\n      "lat": 55.8460062,\n      "lon": 37.4811199,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28896147",\n      "kind": "school",\n      "name": "Лицей №1571 корпус №6",\n      "lat": 55.8567494,\n      "lon": 37.4440687,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28938786",\n      "kind": "school",\n      "name": "Школа № 830. Корпус № 3",\n      "lat": 55.8251923,\n      "lon": 37.4493216,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28944074",\n      "kind": "kindergarten",\n      "name": "Детский сад №335 Института естественных наук и экологии (ИНЕСНЭК)",\n      "lat": 55.8022883,\n      "lon": 37.4784098,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28950132",\n      "kind": "school",\n      "name": "Школа №224",\n      "lat": 55.8291421,\n      "lon": 37.4910137,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28950143",\n      "kind": "school",\n      "name": "Школа №224",\n      "lat": 55.8378457,\n      "lon": 37.4843581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/28950763",\n      "kind": "school",\n      "name": "Центр образования №1679 Школа здоровья",\n      "lat": 55.8277391,\n      "lon": 37.5192129,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29008845",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6955402,\n      "lon": 37.4573312,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29019244",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6735931,\n      "lon": 37.4684407,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29019245",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6731947,\n      "lon": 37.4675836,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29019247",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6702971,\n      "lon": 37.4624472,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29019273",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6756642,\n      "lon": 37.4548492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29034683",\n      "kind": "school",\n      "name": "Воскресная школа",\n      "lat": 55.7572129,\n      "lon": 37.8139517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29092187",\n      "kind": "school",\n      "name": "Интернат №108",\n      "lat": 55.681983,\n      "lon": 37.5265689,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29114815",\n      "kind": "school",\n      "name": "Школа № 1514",\n      "lat": 55.686297,\n      "lon": 37.5283695,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29127999",\n      "kind": "kindergarten",\n      "name": "Школа № 199. Дошкольное отделение № 7",\n      "lat": 55.6825082,\n      "lon": 37.5578716,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29128001",\n      "kind": "kindergarten",\n      "name": "Школа № 199. Дошкольное отделение № 7",\n      "lat": 55.6830478,\n      "lon": 37.5584461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29172983",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8682239,\n      "lon": 37.5323734,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29243290",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6937689,\n      "lon": 37.4941239,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29265266",\n      "kind": "school",\n      "name": "Школа №115",\n      "lat": 55.6532486,\n      "lon": 37.5270247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29277434",\n      "kind": "school",\n      "name": "Специальная школа №3",\n      "lat": 55.6699349,\n      "lon": 37.5760101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29277435",\n      "kind": "school",\n      "name": "Школа-интернат №61",\n      "lat": 55.6696845,\n      "lon": 37.5767271,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29277495",\n      "kind": "school",\n      "name": "Интернат № 56",\n      "lat": 55.6708326,\n      "lon": 37.5962506,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29325314",\n      "kind": "school",\n      "name": "Центр внешкольной работы \\"Зюзино\\" (УПК №2)",\n      "lat": 55.6479616,\n      "lon": 37.5905415,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386096",\n      "kind": "school",\n      "name": "Школа №398",\n      "lat": 55.7175457,\n      "lon": 37.8048564,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386727",\n      "kind": "kindergarten",\n      "name": "Школа №1420. Корпус №1",\n      "lat": 55.7021253,\n      "lon": 37.8189579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386728",\n      "kind": "kindergarten",\n      "name": "Школа №1420. Корпус №5",\n      "lat": 55.7019684,\n      "lon": 37.8175424,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386752",\n      "kind": "school",\n      "name": "Спецшкола №895",\n      "lat": 55.7003552,\n      "lon": 37.8137004,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386753",\n      "kind": "school",\n      "name": "Центр образования №1420",\n      "lat": 55.7009762,\n      "lon": 37.8148969,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386777",\n      "kind": "kindergarten",\n      "name": "Детский сад №1734",\n      "lat": 55.6986182,\n      "lon": 37.8107168,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29386778",\n      "kind": "kindergarten",\n      "name": "Детский сад №2118",\n      "lat": 55.6984591,\n      "lon": 37.8124211,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29388396",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1393 ДО",\n      "lat": 55.6960676,\n      "lon": 37.8150581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29388405",\n      "kind": "kindergarten",\n      "name": "Детский сад №326",\n      "lat": 55.6965439,\n      "lon": 37.8237105,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29396623",\n      "kind": "school",\n      "name": "Школа имени Ф. М. Достоевского",\n      "lat": 55.6865233,\n      "lon": 37.7510571,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29396677",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6824659,\n      "lon": 37.7529292,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29400194",\n      "kind": "kindergarten",\n      "name": "Детский сад №1333",\n      "lat": 55.7053222,\n      "lon": 37.7900766,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29400195",\n      "kind": "kindergarten",\n      "name": "Детский сад №1677",\n      "lat": 55.7058003,\n      "lon": 37.7890398,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29400198",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7065067,\n      "lon": 37.7833545,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29400566",\n      "kind": "school",\n      "name": "Школа №623",\n      "lat": 55.7055473,\n      "lon": 37.7871892,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29430157",\n      "kind": "school",\n      "name": "Школа №2090",\n      "lat": 55.7144209,\n      "lon": 37.7921939,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29430159",\n      "kind": "school",\n      "name": "Школа №776",\n      "lat": 55.7144452,\n      "lon": 37.7927937,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29430209",\n      "kind": "school",\n      "name": "Начальная школа - детский сад №1883",\n      "lat": 55.7184371,\n      "lon": 37.8030253,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29430212",\n      "kind": "kindergarten",\n      "name": "Детский сад №581",\n      "lat": 55.719514,\n      "lon": 37.7937924,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29431666",\n      "kind": "school",\n      "name": "Школа №797",\n      "lat": 55.7548712,\n      "lon": 37.8224523,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446097",\n      "kind": "kindergarten",\n      "name": "Детский сад №724",\n      "lat": 55.7568191,\n      "lon": 37.7858978,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446098",\n      "kind": "school",\n      "name": "Центр образования №1637",\n      "lat": 55.7560708,\n      "lon": 37.7864197,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446190",\n      "kind": "kindergarten",\n      "name": "Детский сад №1900",\n      "lat": 55.7580628,\n      "lon": 37.7998375,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446272",\n      "kind": "kindergarten",\n      "name": "Центр развития ребенка №1908",\n      "lat": 55.7550447,\n      "lon": 37.7807216,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446275",\n      "kind": "school",\n      "name": "Центр образования №1852",\n      "lat": 55.7560122,\n      "lon": 37.7792385,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446294",\n      "kind": "kindergarten",\n      "name": "Детский сад №1216",\n      "lat": 55.7585526,\n      "lon": 37.7773452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446299",\n      "kind": "school",\n      "name": "Школа №681",\n      "lat": 55.7589275,\n      "lon": 37.7757171,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446339",\n      "kind": "kindergarten",\n      "name": "Московская международная школа. Подразделение \\"Планета Земля\\"",\n      "lat": 55.7597229,\n      "lon": 37.7857198,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446344",\n      "kind": "school",\n      "name": "Московская международная школа. Подразделение \\"Планета Уран\\"",\n      "lat": 55.7633918,\n      "lon": 37.7813803,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29446378",\n      "kind": "school",\n      "name": "Московская международная школа. Подразделение \\"Планета Солнце\\"",\n      "lat": 55.7623385,\n      "lon": 37.7867543,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29449572",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.650583,\n      "lon": 37.7238597,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29449574",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.650188,\n      "lon": 37.7209587,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29449575",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6508963,\n      "lon": 37.7216145,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29449576",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6487741,\n      "lon": 37.7182695,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29449577",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6479391,\n      "lon": 37.717653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29450186",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6435458,\n      "lon": 37.7161641,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29450778",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6433703,\n      "lon": 37.7126553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29450878",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6537227,\n      "lon": 37.7145432,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29450880",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6541151,\n      "lon": 37.7134389,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29451029",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6508977,\n      "lon": 37.7129595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29601128",\n      "kind": "school",\n      "name": "Интернат №53",\n      "lat": 55.7767955,\n      "lon": 37.728609,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29689770",\n      "kind": "school",\n      "name": "Школа №1324",\n      "lat": 55.755097,\n      "lon": 37.8201818,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29696444",\n      "kind": "school",\n      "name": "ЦРР №837",\n      "lat": 55.7998586,\n      "lon": 37.785213,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29728292",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8090605,\n      "lon": 37.8210452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/29728293",\n      "kind": "school",\n      "name": "",\n      "lat": 55.807993,\n      "lon": 37.8222138,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30049930",\n      "kind": "school",\n      "name": "Школа \\"Вита\\"",\n      "lat": 55.7343311,\n      "lon": 37.4798654,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30132609",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7472919,\n      "lon": 37.7716329,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30414740",\n      "kind": "school",\n      "name": "Школа №22",\n      "lat": 55.6986533,\n      "lon": 37.5553316,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30414751",\n      "kind": "school",\n      "name": "Школа №1265",\n      "lat": 55.6991038,\n      "lon": 37.5582731,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30414756",\n      "kind": "school",\n      "name": "Лицей \\"Вторая Школа\\"",\n      "lat": 55.6978181,\n      "lon": 37.5563936,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30433167",\n      "kind": "school",\n      "name": "Школа №171",\n      "lat": 55.7222711,\n      "lon": 37.5795407,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30853347",\n      "kind": "school",\n      "name": "Центр образования №775",\n      "lat": 55.6752927,\n      "lon": 37.7535069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30853467",\n      "kind": "school",\n      "name": "НОУ Школа \\"Эрудит-2\\"",\n      "lat": 55.6757566,\n      "lon": 37.7499035,\n      "opening_hours": null\n    },\n    {\n      "id": "way/30990497",\n      "kind": "kindergarten",\n      "name": "Детский сад № 621",\n      "lat": 55.6268177,\n      "lon": 37.6502032,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31024968",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2680",\n      "lat": 55.6323548,\n      "lon": 37.6743605,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31025606",\n      "kind": "school",\n      "name": "ЦДТ \\"Царицыно\\"",\n      "lat": 55.6308695,\n      "lon": 37.6522876,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31025623",\n      "kind": "school",\n      "name": "Школа №1640",\n      "lat": 55.6323645,\n      "lon": 37.6519217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31172614",\n      "kind": "kindergarten",\n      "name": "ДОУ №1873",\n      "lat": 55.6808136,\n      "lon": 37.6413283,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31650953",\n      "kind": "kindergarten",\n      "name": "Детский сад №674",\n      "lat": 55.6871685,\n      "lon": 37.7285679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31653611",\n      "kind": "school",\n      "name": "Лицей №504",\n      "lat": 55.6678229,\n      "lon": 37.6270009,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31663291",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6781056,\n      "lon": 37.6744201,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31663292",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6781643,\n      "lon": 37.672463,\n      "opening_hours": null\n    },\n    {\n      "id": "way/31663877",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 507, Светлячок  (здание № 4)",\n      "lat": 55.6691759,\n      "lon": 37.6516045,\n      "opening_hours": null\n    },\n    {\n      "id": "way/32605841",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Ленинском\\"",\n      "lat": 55.7180444,\n      "lon": 37.593162,\n      "opening_hours": null\n    },\n    {\n      "id": "way/32611554",\n      "kind": "school",\n      "name": "Школа №730",\n      "lat": 55.6599999,\n      "lon": 37.6266505,\n      "opening_hours": null\n    },\n    {\n      "id": "way/33060422",\n      "kind": "school",\n      "name": "Школа №183",\n      "lat": 55.8740976,\n      "lon": 37.5560007,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34321752",\n      "kind": "school",\n      "name": "Школа 1040",\n      "lat": 55.6438454,\n      "lon": 37.7374497,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34321753",\n      "kind": "school",\n      "name": "Школа 1147",\n      "lat": 55.6441597,\n      "lon": 37.7361151,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34329690",\n      "kind": "school",\n      "name": "Центр образования №2010",\n      "lat": 55.6673041,\n      "lon": 37.748292,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34585487",\n      "kind": "school",\n      "name": "Школа № 296",\n      "lat": 55.8312193,\n      "lon": 37.5914633,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34873806",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 1",\n      "lat": 55.7730658,\n      "lon": 37.5415792,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34908291",\n      "kind": "school",\n      "name": "Школа №1770",\n      "lat": 55.6897412,\n      "lon": 37.6886911,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34908295",\n      "kind": "school",\n      "name": "Кадетская школа №1770",\n      "lat": 55.6861271,\n      "lon": 37.6886855,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34911740",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6784852,\n      "lon": 37.6998653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34911744",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6807628,\n      "lon": 37.7034581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34911748",\n      "kind": "school",\n      "name": "Детский оздоровительно-образовательный центр \\"Южный\\"",\n      "lat": 55.6828058,\n      "lon": 37.7049521,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34911749",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6824531,\n      "lon": 37.7064009,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34923064",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1527 Корпус Медвежонок",\n      "lat": 55.6793522,\n      "lon": 37.6806842,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34923065",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6803916,\n      "lon": 37.6811359,\n      "opening_hours": null\n    },\n    {\n      "id": "way/34990988",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1527 Корпус Сказка",\n      "lat": 55.6783257,\n      "lon": 37.6837186,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35013513",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8012961,\n      "lon": 37.5181267,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069953",\n      "kind": "kindergarten",\n      "name": "Детский сад №1047",\n      "lat": 55.8913307,\n      "lon": 37.6697913,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069955",\n      "kind": "kindergarten",\n      "name": "Детский сад №112",\n      "lat": 55.8913889,\n      "lon": 37.667136,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069957",\n      "kind": "kindergarten",\n      "name": "Детский сад №1856",\n      "lat": 55.8896475,\n      "lon": 37.6659392,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069958",\n      "kind": "kindergarten",\n      "name": "Детский сад №2576",\n      "lat": 55.8896526,\n      "lon": 37.6685231,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069959",\n      "kind": "school",\n      "name": "Школа № 283. Корпус № 2",\n      "lat": 55.8914151,\n      "lon": 37.6643298,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35069960",\n      "kind": "school",\n      "name": "Школа №283",\n      "lat": 55.8895329,\n      "lon": 37.671121,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35134626",\n      "kind": "school",\n      "name": "Школа №950",\n      "lat": 55.8602701,\n      "lon": 37.6046609,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35158327",\n      "kind": "school",\n      "name": "Социально-ребилитационный центр Отрадное",\n      "lat": 55.8613273,\n      "lon": 37.6059881,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35172004",\n      "kind": "school",\n      "name": "Школа № 536",\n      "lat": 55.6436935,\n      "lon": 37.5956243,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35379957",\n      "kind": "school",\n      "name": "Центр образования №548 Царицыно",\n      "lat": 55.6020586,\n      "lon": 37.7169951,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35403477",\n      "kind": "school",\n      "name": "Школа №544, корпус 3",\n      "lat": 55.602306,\n      "lon": 37.7190308,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35403511",\n      "kind": "kindergarten",\n      "name": "Детский сад №1166",\n      "lat": 55.6031909,\n      "lon": 37.7163099,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35403550",\n      "kind": "school",\n      "name": "Школа №544",\n      "lat": 55.6068626,\n      "lon": 37.7146404,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35680206",\n      "kind": "school",\n      "name": "Школа № 370",\n      "lat": 55.6902904,\n      "lon": 37.5845344,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35680207",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1216",\n      "lat": 55.6904938,\n      "lon": 37.5839383,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35763140",\n      "kind": "school",\n      "name": "Школа №1205 \\"Единство\\"",\n      "lat": 55.6770512,\n      "lon": 37.5434015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35960354",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6647818,\n      "lon": 37.4944948,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35960355",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6644009,\n      "lon": 37.4954367,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35977653",\n      "kind": "school",\n      "name": "Лицей им. Софьи Нюберг",\n      "lat": 55.6763331,\n      "lon": 37.5817519,\n      "opening_hours": null\n    },\n    {\n      "id": "way/35977707",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2015",\n      "lat": 55.6759253,\n      "lon": 37.5782656,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36243218",\n      "kind": "school",\n      "name": "Школа №585",\n      "lat": 55.5985768,\n      "lon": 37.7191269,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36306984",\n      "kind": "kindergarten",\n      "name": "Детский сад №1018",\n      "lat": 55.8641133,\n      "lon": 37.4904729,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36358903",\n      "kind": "school",\n      "name": "Школа-вуз «Современное образование»",\n      "lat": 55.7687625,\n      "lon": 37.6355598,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36476905",\n      "kind": "school",\n      "name": "Высшая коммерческая школа ВАВТ Минэкономразвития России",\n      "lat": 55.8789001,\n      "lon": 37.4680313,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36515109",\n      "kind": "school",\n      "name": "Центр образования №117",\n      "lat": 55.6736121,\n      "lon": 37.531363,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36515110",\n      "kind": "kindergarten",\n      "name": "Детский сад №1505",\n      "lat": 55.6719177,\n      "lon": 37.5356426,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36515111",\n      "kind": "kindergarten",\n      "name": "Детский сад №1609",\n      "lat": 55.6719208,\n      "lon": 37.5387011,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36515112",\n      "kind": "school",\n      "name": "Школа №117",\n      "lat": 55.6720714,\n      "lon": 37.5340263,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36515113",\n      "kind": "kindergarten",\n      "name": "Детский сад №515",\n      "lat": 55.6725719,\n      "lon": 37.5374105,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36746759",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8640461,\n      "lon": 37.4877039,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36868611",\n      "kind": "school",\n      "name": "Школа №2000 корпус №3",\n      "lat": 55.6411232,\n      "lon": 37.6598579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/36953234",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №9 (дошкольное отделение)",\n      "lat": 55.8405956,\n      "lon": 37.4618454,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37030105",\n      "kind": "school",\n      "name": "ЦПМСС \\"Взаимодействие\\"",\n      "lat": 55.6910606,\n      "lon": 37.6118144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37082803",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7141953,\n      "lon": 37.5145849,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37330394",\n      "kind": "school",\n      "name": "Школа №49. Школьный корпус",\n      "lat": 55.6420096,\n      "lon": 37.546718,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37330635",\n      "kind": "kindergarten",\n      "name": "Школа № 49. Дошкольный корпус",\n      "lat": 55.64255,\n      "lon": 37.5450393,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37870407",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6821161,\n      "lon": 37.6877681,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37927642",\n      "kind": "kindergarten",\n      "name": "Детский сад №66",\n      "lat": 55.8157405,\n      "lon": 37.7123382,\n      "opening_hours": null\n    },\n    {\n      "id": "way/37943640",\n      "kind": "kindergarten",\n      "name": "Детский сад №526",\n      "lat": 55.7863181,\n      "lon": 37.4520041,\n      "opening_hours": null\n    },\n    {\n      "id": "way/38298522",\n      "kind": "school",\n      "name": "Цетр образования №1601",\n      "lat": 55.792534,\n      "lon": 37.573236,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39254578",\n      "kind": "kindergarten",\n      "name": "Школа № 113. Корпус дошкольного образования №3",\n      "lat": 55.6404443,\n      "lon": 37.5195128,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39393807",\n      "kind": "school",\n      "name": "Школа № 489",\n      "lat": 55.7176541,\n      "lon": 37.7368662,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39393808",\n      "kind": "school",\n      "name": "Школа № 841",\n      "lat": 55.7173734,\n      "lon": 37.735468,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39505342",\n      "kind": "school",\n      "name": "Школа №597",\n      "lat": 55.8658618,\n      "lon": 37.4895788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39524490",\n      "kind": "kindergarten",\n      "name": "Детский сад №51",\n      "lat": 55.7112872,\n      "lon": 37.7329596,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39577157",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №264",\n      "lat": 55.671468,\n      "lon": 37.7218289,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39577161",\n      "kind": "school",\n      "name": "Школа №918",\n      "lat": 55.6702373,\n      "lon": 37.7224202,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39627110",\n      "kind": "school",\n      "name": "Центр образования №773",\n      "lat": 55.6894716,\n      "lon": 37.7288475,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39627114",\n      "kind": "school",\n      "name": "Школа №773",\n      "lat": 55.6881679,\n      "lon": 37.7293032,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39627119",\n      "kind": "kindergarten",\n      "name": "Детский сад №671",\n      "lat": 55.6879414,\n      "lon": 37.7275966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39629314",\n      "kind": "kindergarten",\n      "name": "Детский сад №2224",\n      "lat": 55.675903,\n      "lon": 37.7188544,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39629327",\n      "kind": "kindergarten",\n      "name": "Детский сад № 829",\n      "lat": 55.682038,\n      "lon": 37.7191801,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39651080",\n      "kind": "school",\n      "name": "Школа №1256 имени Героя Советского Союза И. С. Полбина (корпус 1)",\n      "lat": 55.6803772,\n      "lon": 37.7270089,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39651117",\n      "kind": "kindergarten",\n      "name": "Школа №1256 имени Героя Советского Союза И. С. Полбина (дошкольное отделение, корпус 2)",\n      "lat": 55.6810646,\n      "lon": 37.7248692,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39651120",\n      "kind": "school",\n      "name": "Школа №1256 имени Героя Советского Союза И. С. Полбина (корпус 2)",\n      "lat": 55.6818471,\n      "lon": 37.7275647,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39651153",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1550",\n      "lat": 55.6859908,\n      "lon": 37.7285406,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39665778",\n      "kind": "kindergarten",\n      "name": "Школа №1256 имени Героя Советского Союза И. С. Полбина (дошкольное отделение, корпус 1)",\n      "lat": 55.6788088,\n      "lon": 37.725612,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39894769",\n      "kind": "school",\n      "name": "Центр образования №1465 имени адмирала Н.Г. Кузнецова",\n      "lat": 55.7451143,\n      "lon": 37.5598952,\n      "opening_hours": null\n    },\n    {\n      "id": "way/39958817",\n      "kind": "school",\n      "name": "Школа № 2101",\n      "lat": 55.7415895,\n      "lon": 37.4777674,\n      "opening_hours": "Mo-Sa 08:00-19:00"\n    },\n    {\n      "id": "way/40057230",\n      "kind": "kindergarten",\n      "name": "Школа №1100 (дошкольное отделение)",\n      "lat": 55.6623467,\n      "lon": 37.5288537,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40057245",\n      "kind": "school",\n      "name": "Школа №1100",\n      "lat": 55.6628789,\n      "lon": 37.5317421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40445965",\n      "kind": "kindergarten",\n      "name": "Школа \\"Покровский квартал\\", Детский сад 1",\n      "lat": 55.7555356,\n      "lon": 37.6524788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40607358",\n      "kind": "kindergarten",\n      "name": "Детский сад № 63",\n      "lat": 55.6885878,\n      "lon": 37.564302,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40792993",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.743724,\n      "lon": 37.7561889,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40793349",\n      "kind": "school",\n      "name": "Школа № 2126 \\"Перово\\"",\n      "lat": 55.7428651,\n      "lon": 37.7590491,\n      "opening_hours": null\n    },\n    {\n      "id": "way/40983857",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8234461,\n      "lon": 37.599908,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41088548",\n      "kind": "kindergarten",\n      "name": "Детский сад №1855",\n      "lat": 55.7369908,\n      "lon": 37.4639043,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41088549",\n      "kind": "school",\n      "name": "Школа №2101",\n      "lat": 55.7360516,\n      "lon": 37.4624765,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41347146",\n      "kind": "kindergarten",\n      "name": "Детский сад №103",\n      "lat": 55.6991646,\n      "lon": 37.7821279,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41347147",\n      "kind": "kindergarten",\n      "name": "Детский сад №139",\n      "lat": 55.6989681,\n      "lon": 37.7835401,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41657209",\n      "kind": "school",\n      "name": "Гимназия №1592. Отделение №2043",\n      "lat": 55.8749129,\n      "lon": 37.5544011,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41667074",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6781004,\n      "lon": 37.4876995,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41667088",\n      "kind": "school",\n      "name": "Центр образования и спорта \\"Москва-98\\". Школа № 2",\n      "lat": 55.6802679,\n      "lon": 37.4849926,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41826105",\n      "kind": "school",\n      "name": "Школа №964",\n      "lat": 55.8971022,\n      "lon": 37.6161054,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41826342",\n      "kind": "school",\n      "name": "Школа №1413",\n      "lat": 55.8966871,\n      "lon": 37.6174649,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41870646",\n      "kind": "school",\n      "name": "Центр образования №1852",\n      "lat": 55.7570633,\n      "lon": 37.8035144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41981684",\n      "kind": "school",\n      "name": "Центр образования №1497",\n      "lat": 55.7446262,\n      "lon": 37.4927263,\n      "opening_hours": null\n    },\n    {\n      "id": "way/41981695",\n      "kind": "kindergarten",\n      "name": "Детский сад №1328",\n      "lat": 55.7435007,\n      "lon": 37.4919555,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42031082",\n      "kind": "school",\n      "name": "Мытищинская детская дом-школа музыкального воспитания",\n      "lat": 55.8893147,\n      "lon": 37.7536061,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42032375",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа Глория\\"",\n      "lat": 55.8958252,\n      "lon": 37.6157141,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42102028",\n      "kind": "school",\n      "name": "Школа Глория, корпус № 10",\n      "lat": 55.8951174,\n      "lon": 37.6250771,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42170724",\n      "kind": "school",\n      "name": "Школа №2101",\n      "lat": 55.7347513,\n      "lon": 37.4804259,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42170744",\n      "kind": "kindergarten",\n      "name": "Детский сад №545",\n      "lat": 55.7330745,\n      "lon": 37.4731726,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42213016",\n      "kind": "kindergarten",\n      "name": "ГУ МВД РФ по МО",\n      "lat": 55.7123225,\n      "lon": 37.6602774,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42237425",\n      "kind": "school",\n      "name": "Школа №73",\n      "lat": 55.7328756,\n      "lon": 37.4672408,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42237426",\n      "kind": "school",\n      "name": "Школа №262",\n      "lat": 55.732406,\n      "lon": 37.4647222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42237493",\n      "kind": "kindergarten",\n      "name": "Детский сад №1394",\n      "lat": 55.7322756,\n      "lon": 37.4582185,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42237542",\n      "kind": "kindergarten",\n      "name": "Детский сад №77",\n      "lat": 55.7312015,\n      "lon": 37.4668464,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42237543",\n      "kind": "kindergarten",\n      "name": "Детский сад №659",\n      "lat": 55.7315904,\n      "lon": 37.4692174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42239788",\n      "kind": "kindergarten",\n      "name": "Детский сад №1240",\n      "lat": 55.8654668,\n      "lon": 37.6840473,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42244977",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Солнышко\\"",\n      "lat": 55.7328614,\n      "lon": 37.4512911,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42368435",\n      "kind": "school",\n      "name": "Центр образования №1854 (8-11 классы)",\n      "lat": 55.8025995,\n      "lon": 37.4717526,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42394177",\n      "kind": "school",\n      "name": "Навигацкая школа",\n      "lat": 55.7310613,\n      "lon": 37.4555855,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42484743",\n      "kind": "kindergarten",\n      "name": "Ясли - сад №262",\n      "lat": 55.7093657,\n      "lon": 37.7551514,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42484744",\n      "kind": "school",\n      "name": "Школа №336",\n      "lat": 55.7095905,\n      "lon": 37.7571555,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42485986",\n      "kind": "kindergarten",\n      "name": "Детский сад №1130",\n      "lat": 55.6344211,\n      "lon": 37.7939016,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42563777",\n      "kind": "kindergarten",\n      "name": "Детский сад №2457",\n      "lat": 55.6324949,\n      "lon": 37.7974361,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42708942",\n      "kind": "kindergarten",\n      "name": "Детский сад №2556",\n      "lat": 55.7414863,\n      "lon": 37.6662469,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42774926",\n      "kind": "school",\n      "name": "Московская богословская семинария евангельских христиан-баптистов",\n      "lat": 55.7461854,\n      "lon": 37.7561497,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42774928",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7461639,\n      "lon": 37.7575604,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42855891",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6522714,\n      "lon": 37.7127087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/42946013",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1135",\n      "lat": 55.6577756,\n      "lon": 37.7045004,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43175483",\n      "kind": "kindergarten",\n      "name": "Детский сад №911",\n      "lat": 55.652137,\n      "lon": 37.7011088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43175547",\n      "kind": "kindergarten",\n      "name": "Детский сад №774",\n      "lat": 55.6546546,\n      "lon": 37.7038672,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43198474",\n      "kind": "school",\n      "name": "Школа №493",\n      "lat": 55.6573417,\n      "lon": 37.6995277,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43198475",\n      "kind": "school",\n      "name": "Центр Непрерывного Ускоренного Обучения",\n      "lat": 55.6575068,\n      "lon": 37.7021117,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43321808",\n      "kind": "school",\n      "name": "Школа №1229",\n      "lat": 55.7504613,\n      "lon": 37.6793611,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43566667",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7175522,\n      "lon": 37.7361626,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43790668",\n      "kind": "kindergarten",\n      "name": "Детский сад №1541",\n      "lat": 55.7002506,\n      "lon": 37.7350172,\n      "opening_hours": null\n    },\n    {\n      "id": "way/43790866",\n      "kind": "kindergarten",\n      "name": "Детский сад №1275",\n      "lat": 55.7034525,\n      "lon": 37.7349309,\n      "opening_hours": null\n    },\n    {\n      "id": "way/44146666",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8687457,\n      "lon": 37.7120246,\n      "opening_hours": null\n    },\n    {\n      "id": "way/44384266",\n      "kind": "kindergarten",\n      "name": "Детский сад №1838",\n      "lat": 55.7990431,\n      "lon": 37.7729884,\n      "opening_hours": null\n    },\n    {\n      "id": "way/44502324",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8903859,\n      "lon": 37.4758059,\n      "opening_hours": null\n    },\n    {\n      "id": "way/44503057",\n      "kind": "school",\n      "name": "Школа №707",\n      "lat": 55.7971548,\n      "lon": 37.7735426,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45138130",\n      "kind": "kindergarten",\n      "name": "Школа №1056 (дошкольное отделение №1)",\n      "lat": 55.8465531,\n      "lon": 37.4429213,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45249027",\n      "kind": "kindergarten",\n      "name": "Центр образования № 1858 - д/с \\"Сказка\\"",\n      "lat": 55.6394707,\n      "lon": 37.7944318,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45249666",\n      "kind": "kindergarten",\n      "name": "Центр образования № 1858 - д/с \\"Львёнок\\"",\n      "lat": 55.6390333,\n      "lon": 37.7944197,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45249984",\n      "kind": "kindergarten",\n      "name": "Детский сад №692",\n      "lat": 55.6351427,\n      "lon": 37.7999839,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45250579",\n      "kind": "kindergarten",\n      "name": "Детский сад №1504",\n      "lat": 55.6325715,\n      "lon": 37.8001879,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45256626",\n      "kind": "school",\n      "name": "ГБОУ Школа в Капотне (Учебный корпус \\"Кузнецкий\\")",\n      "lat": 55.6359101,\n      "lon": 37.7976899,\n      "opening_hours": "Mo-Sa 08:00-19:00"\n    },\n    {\n      "id": "way/45398742",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6838216,\n      "lon": 37.4570798,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45399802",\n      "kind": "kindergarten",\n      "name": "Детский сад №874",\n      "lat": 55.8710096,\n      "lon": 37.673611,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45463106",\n      "kind": "school",\n      "name": "Воскресная школа при храме Св. Николая",\n      "lat": 55.7320652,\n      "lon": 37.5904566,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45487360",\n      "kind": "kindergarten",\n      "name": "Школа № 1363, ДО Радуга",\n      "lat": 55.7057657,\n      "lon": 37.8252415,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45739177",\n      "kind": "school",\n      "name": "НОУ \\"Развитие\\" (школа компенсирующего обучения)",\n      "lat": 55.6715593,\n      "lon": 37.5647551,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45778892",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\". Здание на Б. Трехсвятительском",\n      "lat": 55.7549631,\n      "lon": 37.6460396,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45820293",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1084",\n      "lat": 55.7156782,\n      "lon": 37.6188496,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45823256",\n      "kind": "kindergarten",\n      "name": "Детский сад №469",\n      "lat": 55.672203,\n      "lon": 37.59147,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45826027",\n      "kind": "kindergarten",\n      "name": "Школа Эрудит",\n      "lat": 55.7309182,\n      "lon": 37.7147858,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45829275",\n      "kind": "kindergarten",\n      "name": "Школа № 1222 имени Маршала Советского Союза И. Х. Баграмяна. Дошкольное  отделение № 1",\n      "lat": 55.7321958,\n      "lon": 37.7100014,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45831433",\n      "kind": "school",\n      "name": "Школа №1344",\n      "lat": 55.6951689,\n      "lon": 37.7999825,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45850376",\n      "kind": "school",\n      "name": "ГБОУ школа № 1363, корпус 2",\n      "lat": 55.7066325,\n      "lon": 37.8233965,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45850682",\n      "kind": "school",\n      "name": "Школа №1937",\n      "lat": 55.6997432,\n      "lon": 37.8146225,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45875704",\n      "kind": "school",\n      "name": "Школа №394",\n      "lat": 55.70516,\n      "lon": 37.7859633,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45876203",\n      "kind": "kindergarten",\n      "name": "Детский сад №779",\n      "lat": 55.7096469,\n      "lon": 37.7880116,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45876992",\n      "kind": "school",\n      "name": "Школа № 468",\n      "lat": 55.7129864,\n      "lon": 37.7710257,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45897345",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7317013,\n      "lon": 37.7410536,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45934474",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1315",\n      "lat": 55.6333705,\n      "lon": 37.4788494,\n      "opening_hours": null\n    },\n    {\n      "id": "way/45936847",\n      "kind": "school",\n      "name": "ЦО \\"Самбо-70\\"",\n      "lat": 55.6308846,\n      "lon": 37.4872392,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46166572",\n      "kind": "kindergarten",\n      "name": "Детский сад № 313",\n      "lat": 55.6496975,\n      "lon": 37.5897187,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46166579",\n      "kind": "kindergarten",\n      "name": "Детский сад № 314",\n      "lat": 55.6488776,\n      "lon": 37.5900296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46578351",\n      "kind": "school",\n      "name": "Лицей №1571 корпус №2",\n      "lat": 55.8618588,\n      "lon": 37.4455613,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46581741",\n      "kind": "school",\n      "name": "Лицей №\xa01571 корпус 3",\n      "lat": 55.8626615,\n      "lon": 37.4417625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46604063",\n      "kind": "kindergarten",\n      "name": "Детский сад №1265",\n      "lat": 55.6994677,\n      "lon": 37.5596529,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46657069",\n      "kind": "kindergarten",\n      "name": "Детский сад «Исток 2»",\n      "lat": 55.6783991,\n      "lon": 37.5550965,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46657618",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1335",\n      "lat": 55.6877506,\n      "lon": 37.5489992,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46657631",\n      "kind": "school",\n      "name": "Школа № 26",\n      "lat": 55.6944192,\n      "lon": 37.5504454,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46657632",\n      "kind": "school",\n      "name": "Школа № 25",\n      "lat": 55.6952633,\n      "lon": 37.5489463,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46671693",\n      "kind": "kindergarten",\n      "name": "Детский сад №549",\n      "lat": 55.6725462,\n      "lon": 37.5330648,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46671697",\n      "kind": "kindergarten",\n      "name": "детский сад 942",\n      "lat": 55.6758118,\n      "lon": 37.534604,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46671699",\n      "kind": "kindergarten",\n      "name": "Детский сад №1511",\n      "lat": 55.6753419,\n      "lon": 37.5299693,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46740734",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7873449,\n      "lon": 37.6151356,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46830401",\n      "kind": "school",\n      "name": "Школа №1532. Корпус школьного образования №01",\n      "lat": 55.6219218,\n      "lon": 37.4914937,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46831048",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/1",\n      "lat": 55.6224351,\n      "lon": 37.4933496,\n      "opening_hours": null\n    },\n    {\n      "id": "way/46922890",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 8",\n      "lat": 55.8483814,\n      "lon": 37.6430853,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47067455",\n      "kind": "school",\n      "name": "ГБОУ школа на Яузе",\n      "lat": 55.7736771,\n      "lon": 37.707104,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47067457",\n      "kind": "kindergarten",\n      "name": "Детский сад №1979",\n      "lat": 55.7733274,\n      "lon": 37.7078097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47346175",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6514957,\n      "lon": 37.73924,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47514087",\n      "kind": "school",\n      "name": "Школа №951",\n      "lat": 55.8920145,\n      "lon": 37.6769757,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47572148",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7330721,\n      "lon": 37.6610855,\n      "opening_hours": null\n    },\n    {\n      "id": "way/47584412",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6440596,\n      "lon": 37.7367633,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48036691",\n      "kind": "kindergarten",\n      "name": "Школа №967, дошкольное отделение №4",\n      "lat": 55.880319,\n      "lon": 37.6542186,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48038280",\n      "kind": "kindergarten",\n      "name": "Детский сад №2101",\n      "lat": 55.6446565,\n      "lon": 37.7402371,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48038333",\n      "kind": "kindergarten",\n      "name": "Детский сад №2162",\n      "lat": 55.6457544,\n      "lon": 37.7402389,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48038725",\n      "kind": "kindergarten",\n      "name": "Детский сад №3 Сказка",\n      "lat": 55.8910272,\n      "lon": 37.4454591,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48042142",\n      "kind": "kindergarten",\n      "name": "Детский сад №1797",\n      "lat": 55.6455114,\n      "lon": 37.7328499,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48042227",\n      "kind": "kindergarten",\n      "name": "Детский сад №1799",\n      "lat": 55.6444231,\n      "lon": 37.733024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48057819",\n      "kind": "kindergarten",\n      "name": "Детский сад № 735",\n      "lat": 55.7240301,\n      "lon": 37.6283509,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48125946",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа № 1251 имени генерала Шарля де Голля\\"",\n      "lat": 55.7999928,\n      "lon": 37.5081815,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48126019",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1251 имени генерала Шарля де Голля\\"",\n      "lat": 55.8006034,\n      "lon": 37.5085065,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48167354",\n      "kind": "kindergarten",\n      "name": "детский сад №1070",\n      "lat": 55.601711,\n      "lon": 37.5450614,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48174162",\n      "kind": "kindergarten",\n      "name": "ОСЗН района Ясенево",\n      "lat": 55.5964108,\n      "lon": 37.5417582,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48178990",\n      "kind": "kindergarten",\n      "name": "Школа №1429 имени Героя Советского Союза Н.А. Боброва (Дошкольное отделение)",\n      "lat": 55.7783694,\n      "lon": 37.6876405,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48179027",\n      "kind": "kindergarten",\n      "name": "Детский сад №1061",\n      "lat": 55.6076334,\n      "lon": 37.7273522,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48222931",\n      "kind": "school",\n      "name": "НОУ \\"Школа св. Георгия\\"",\n      "lat": 55.6868672,\n      "lon": 37.7454194,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48227462",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1512",\n      "lat": 55.657015,\n      "lon": 37.6155436,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48239580",\n      "kind": "school",\n      "name": "Школа №1429 имени Героя Советского Союза Н.А. Боброва",\n      "lat": 55.77892,\n      "lon": 37.6867318,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48298114",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1490",\n      "lat": 55.7170934,\n      "lon": 37.730874,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48298115",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1941",\n      "lat": 55.7174803,\n      "lon": 37.732506,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48298364",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1304",\n      "lat": 55.7151618,\n      "lon": 37.7327344,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48303492",\n      "kind": "kindergarten",\n      "name": "Детский сад № 295",\n      "lat": 55.7362894,\n      "lon": 37.6437021,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48337556",\n      "kind": "school",\n      "name": "Школа №478",\n      "lat": 55.71377,\n      "lon": 37.741121,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48337573",\n      "kind": "school",\n      "name": "ЦТРиГО \\"Жемчужина\\"",\n      "lat": 55.7134945,\n      "lon": 37.7377764,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48346415",\n      "kind": "kindergarten",\n      "name": "Детский сад № 492",\n      "lat": 55.7327459,\n      "lon": 37.6564484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48399916",\n      "kind": "school",\n      "name": "Центр образования №1496",\n      "lat": 55.7234639,\n      "lon": 37.6057452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48524187",\n      "kind": "school",\n      "name": "Школа №954",\n      "lat": 55.8952138,\n      "lon": 37.6353503,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48587459",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6435928,\n      "lon": 37.7196933,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48599543",\n      "kind": "kindergarten",\n      "name": "Лицей №\xa01571, дошкольное отделение «Яблонька»",\n      "lat": 55.8652504,\n      "lon": 37.4419269,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48634158",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1791",\n      "lat": 55.7035769,\n      "lon": 37.7501424,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48637694",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6969813,\n      "lon": 37.7574855,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48637695",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6967204,\n      "lon": 37.7588024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48757559",\n      "kind": "kindergarten",\n      "name": "Детский сад «Горница-Узорница»",\n      "lat": 55.8677108,\n      "lon": 37.6657108,\n      "opening_hours": null\n    },\n    {\n      "id": "way/48892974",\n      "kind": "kindergarten",\n      "name": "Детский сад №2617",\n      "lat": 55.7718249,\n      "lon": 37.704149,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49059813",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа № 1251 имени генерала Шарля де Голля\\"",\n      "lat": 55.7960365,\n      "lon": 37.5056424,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49140332",\n      "kind": "school",\n      "name": "Школа XXI век",\n      "lat": 55.7459077,\n      "lon": 37.701383,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49251892",\n      "kind": "kindergarten",\n      "name": "Детский сад №84",\n      "lat": 55.7376797,\n      "lon": 37.7072893,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49251909",\n      "kind": "school",\n      "name": "Школа №459",\n      "lat": 55.7388464,\n      "lon": 37.7060765,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49260760",\n      "kind": "school",\n      "name": "Рогожская Древлеправославная общеобразовательная школа",\n      "lat": 55.7387866,\n      "lon": 37.7012173,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49432199",\n      "kind": "school",\n      "name": "ГБПОУ ДЗМ Медицинский колледж № 1 СП № 1",\n      "lat": 55.6662877,\n      "lon": 37.6315817,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49869363",\n      "kind": "kindergarten",\n      "name": "Детский сад №502",\n      "lat": 55.8488484,\n      "lon": 37.4791162,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49869365",\n      "kind": "kindergarten",\n      "name": "Детский сад №787",\n      "lat": 55.8474061,\n      "lon": 37.4801653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49906444",\n      "kind": "school",\n      "name": "Лицей №1303",\n      "lat": 55.7518616,\n      "lon": 37.6791705,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49908065",\n      "kind": "school",\n      "name": "Школа №415",\n      "lat": 55.7574042,\n      "lon": 37.6897461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/49908070",\n      "kind": "school",\n      "name": "Школа № 1321 \\"Ковчег\\"",\n      "lat": 55.755963,\n      "lon": 37.689715,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50055625",\n      "kind": "school",\n      "name": "Детский сад №617",\n      "lat": 55.7987844,\n      "lon": 37.7951505,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50059100",\n      "kind": "kindergarten",\n      "name": "Детский сад №687",\n      "lat": 55.7955806,\n      "lon": 37.7954579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50072669",\n      "kind": "school",\n      "name": "Школа №444",\n      "lat": 55.7914234,\n      "lon": 37.7963295,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123050",\n      "kind": "school",\n      "name": "Школа 1379",\n      "lat": 55.673814,\n      "lon": 37.5510975,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123057",\n      "kind": "kindergarten",\n      "name": "Детский сад 78 РАН",\n      "lat": 55.6743555,\n      "lon": 37.5503543,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123061",\n      "kind": "kindergarten",\n      "name": "Детский сад 2242 Института Физической Химии и Электрохимии им. Фрумкина А.Н.",\n      "lat": 55.6733306,\n      "lon": 37.5534475,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123062",\n      "kind": "kindergarten",\n      "name": "Детский сад 1780 РАН",\n      "lat": 55.6730881,\n      "lon": 37.5517606,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123064",\n      "kind": "school",\n      "name": "Школа 1205",\n      "lat": 55.676182,\n      "lon": 37.5566572,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50123075",\n      "kind": "school",\n      "name": "Школа 1205",\n      "lat": 55.6769536,\n      "lon": 37.5544505,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50319730",\n      "kind": "kindergarten",\n      "name": "Семейный центр раннего развития и творчества \\"Умняша\\"",\n      "lat": 55.6365119,\n      "lon": 37.5254485,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50343593",\n      "kind": "school",\n      "name": "",\n      "lat": 55.668754,\n      "lon": 37.6537782,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50548821",\n      "kind": "kindergarten",\n      "name": "Детский сад №1280 Общевойсковой академии ВC РФ",\n      "lat": 55.7583177,\n      "lon": 37.7029153,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50584301",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7430308,\n      "lon": 37.7714362,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50584411",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7460384,\n      "lon": 37.7702369,\n      "opening_hours": null\n    },\n    {\n      "id": "way/50672832",\n      "kind": "school",\n      "name": "Школа №751 (начальные классы)",\n      "lat": 55.8843317,\n      "lon": 37.6969641,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51036156",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус №5",\n      "lat": 55.6328197,\n      "lon": 37.5345137,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51036163",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус №4",\n      "lat": 55.6340595,\n      "lon": 37.53756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51239305",\n      "kind": "kindergarten",\n      "name": "Гимназия №1799, дошкольное отделение 4",\n      "lat": 55.801374,\n      "lon": 37.7183797,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51239312",\n      "kind": "school",\n      "name": "Гимназия № 1799, Школьное отделение 1",\n      "lat": 55.8016238,\n      "lon": 37.7211325,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51239411",\n      "kind": "kindergarten",\n      "name": "Гимназия №1080, дошкольное отделение 1",\n      "lat": 55.7992081,\n      "lon": 37.7206594,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51244872",\n      "kind": "kindergarten",\n      "name": "Гимназия №1799, дошкольное отделение 2",\n      "lat": 55.8003305,\n      "lon": 37.7291861,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51244893",\n      "kind": "kindergarten",\n      "name": "Гимназия №1799, дошкольное отделение 3",\n      "lat": 55.8002886,\n      "lon": 37.7257222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51273173",\n      "kind": "school",\n      "name": "Школа №1310",\n      "lat": 55.7426753,\n      "lon": 37.7825148,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51273194",\n      "kind": "kindergarten",\n      "name": "Детский сад №2001",\n      "lat": 55.7581305,\n      "lon": 37.7922867,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51304603",\n      "kind": "school",\n      "name": "Учебно-производственный центр \\"МКМ\\"",\n      "lat": 55.7449401,\n      "lon": 37.713418,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51357392",\n      "kind": "school",\n      "name": "Школа № 1273. Корпус школьного образования №6",\n      "lat": 55.627384,\n      "lon": 37.5241024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51357394",\n      "kind": "kindergarten",\n      "name": "Школа № 1273. Корпус дошкольного образования №5",\n      "lat": 55.6265796,\n      "lon": 37.5234083,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51357399",\n      "kind": "kindergarten",\n      "name": "Школа № 1273. Корпус дошкольного образования №4",\n      "lat": 55.6261186,\n      "lon": 37.5227277,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51422928",\n      "kind": "school",\n      "name": "Гимназия №1799 Школьное отделение 3",\n      "lat": 55.8018196,\n      "lon": 37.7168169,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51422970",\n      "kind": "school",\n      "name": "Школа 1080",\n      "lat": 55.8041024,\n      "lon": 37.72746,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51480004",\n      "kind": "kindergarten",\n      "name": "Детский сад №75",\n      "lat": 55.8102594,\n      "lon": 37.7323787,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51550808",\n      "kind": "school",\n      "name": "",\n      "lat": 55.73437,\n      "lon": 37.5667191,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51675683",\n      "kind": "school",\n      "name": "Школа № 1987. Начальная школа",\n      "lat": 55.6597354,\n      "lon": 37.7758,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51684041",\n      "kind": "school",\n      "name": "Лицей № 1533",\n      "lat": 55.6807249,\n      "lon": 37.5617659,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51684048",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6814392,\n      "lon": 37.5604461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51690266",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6510804,\n      "lon": 37.713926,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51690287",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6519557,\n      "lon": 37.7134848,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51812359",\n      "kind": "school",\n      "name": "Школа № 1862. Корпус школьного образования №4",\n      "lat": 55.650346,\n      "lon": 37.6082315,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51832350",\n      "kind": "school",\n      "name": "Школа № 2016. Корпус школьного образования №1",\n      "lat": 55.651927,\n      "lon": 37.6080146,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51832351",\n      "kind": "kindergarten",\n      "name": "Школа № 1862. Корпус дошкольного образования №7",\n      "lat": 55.6519774,\n      "lon": 37.6097511,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51858795",\n      "kind": "kindergarten",\n      "name": "Детский сад № 62 «Жемчужина» (второе здание)",\n      "lat": 55.8945296,\n      "lon": 37.7236455,\n      "opening_hours": null\n    },\n    {\n      "id": "way/51858798",\n      "kind": "school",\n      "name": "МБОУ СОШ №5",\n      "lat": 55.8912361,\n      "lon": 37.7183033,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52015581",\n      "kind": "kindergarten",\n      "name": "Школа № 1862. Корпус дошкольного образования №6",\n      "lat": 55.6471299,\n      "lon": 37.6081545,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52018812",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8465755,\n      "lon": 37.487374,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52025650",\n      "kind": "school",\n      "name": "Школа №\xa01315",\n      "lat": 55.8478303,\n      "lon": 37.4861106,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52047452",\n      "kind": "kindergarten",\n      "name": "Детский сад № 62 «Жемчужина» (первое здание)",\n      "lat": 55.8966812,\n      "lon": 37.7109465,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52454805",\n      "kind": "kindergarten",\n      "name": "Детский сад №1942",\n      "lat": 55.7313906,\n      "lon": 37.6803359,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52456628",\n      "kind": "school",\n      "name": "Школа раннего развития «Приоритет»",\n      "lat": 55.8009232,\n      "lon": 37.4526391,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52462301",\n      "kind": "school",\n      "name": "МЦО \\"Атлантик\\"",\n      "lat": 55.7174892,\n      "lon": 37.6722807,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52462352",\n      "kind": "kindergarten",\n      "name": "Детский сад №476",\n      "lat": 55.7168388,\n      "lon": 37.6731276,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52462356",\n      "kind": "school",\n      "name": "Школа № 500",\n      "lat": 55.7133713,\n      "lon": 37.6727839,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52463398",\n      "kind": "kindergarten",\n      "name": "Детский сад №77",\n      "lat": 55.723668,\n      "lon": 37.6652131,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52476785",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7276914,\n      "lon": 37.6751008,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52476786",\n      "kind": "school",\n      "name": "Интернат №4",\n      "lat": 55.7271974,\n      "lon": 37.6752442,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52564709",\n      "kind": "school",\n      "name": "Кадетская школа-интернат №69",\n      "lat": 55.6704931,\n      "lon": 37.523219,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52564710",\n      "kind": "school",\n      "name": "Школа-интернат №62",\n      "lat": 55.6706078,\n      "lon": 37.5254816,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52564712",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6709763,\n      "lon": 37.5248203,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52564713",\n      "kind": "school",\n      "name": "Центр образования №1311 Тхия",\n      "lat": 55.6716899,\n      "lon": 37.5236035,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52564722",\n      "kind": "school",\n      "name": "Школа-интернат №62",\n      "lat": 55.6713823,\n      "lon": 37.5251384,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52577428",\n      "kind": "school",\n      "name": "СУНЦ МГУ",\n      "lat": 55.7194408,\n      "lon": 37.464605,\n      "opening_hours": null\n    },\n    {\n      "id": "way/52582033",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7680701,\n      "lon": 37.5918264,\n      "opening_hours": null\n    },\n    {\n      "id": "way/53213796",\n      "kind": "school",\n      "name": "Школа № 1206. Корпус школьного образования №9",\n      "lat": 55.615628,\n      "lon": 37.5333957,\n      "opening_hours": null\n    },\n    {\n      "id": "way/53445879",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7435004,\n      "lon": 37.6639174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/53587339",\n      "kind": "school",\n      "name": "Школа №967, школьное отделение №1",\n      "lat": 55.8791233,\n      "lon": 37.6522682,\n      "opening_hours": null\n    },\n    {\n      "id": "way/53722254",\n      "kind": "school",\n      "name": "ГБОУ ШМК Бибирево",\n      "lat": 55.8882138,\n      "lon": 37.6191407,\n      "opening_hours": null\n    },\n    {\n      "id": "way/53816214",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7340621,\n      "lon": 37.6279421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54255822",\n      "kind": "school",\n      "name": "Школа № 2126 \\"Перово\\"",\n      "lat": 55.7425888,\n      "lon": 37.7623087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54365845",\n      "kind": "school",\n      "name": "Лицей №1571 корпус №5",\n      "lat": 55.8560964,\n      "lon": 37.4442916,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54571601",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.875911,\n      "lon": 37.6762791,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54571602",\n      "kind": "school",\n      "name": "Центр дополнительного образования 1095",\n      "lat": 55.8768013,\n      "lon": 37.6775765,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54625418",\n      "kind": "school",\n      "name": "",\n      "lat": 55.65413,\n      "lon": 37.5573682,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54675243",\n      "kind": "school",\n      "name": "Специальная (коррекционная) школа-интернат № 102",\n      "lat": 55.8534073,\n      "lon": 37.6517343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54830058",\n      "kind": "school",\n      "name": "Центр образования № 1828 \\"Сабурово\\"",\n      "lat": 55.639843,\n      "lon": 37.6971665,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54834851",\n      "kind": "school",\n      "name": "Спорткомплекс «Борисово»",\n      "lat": 55.636303,\n      "lon": 37.726978,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54864830",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №433",\n      "lat": 55.7227709,\n      "lon": 37.7370463,\n      "opening_hours": null\n    },\n    {\n      "id": "way/54905924",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение Ж5",\n      "lat": 55.7328546,\n      "lon": 37.8244261,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55340453",\n      "kind": "school",\n      "name": "Школа №\xa0519",\n      "lat": 55.6839751,\n      "lon": 37.7201125,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55503336",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 8 «Ласточка»",\n      "lat": 55.7562818,\n      "lon": 37.5209215,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55884817",\n      "kind": "kindergarten",\n      "name": "Детский сад №1932",\n      "lat": 55.6498974,\n      "lon": 37.7366286,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55914705",\n      "kind": "school",\n      "name": "Школа №1223",\n      "lat": 55.8386699,\n      "lon": 37.5272356,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55914757",\n      "kind": "school",\n      "name": "Школа №90 \\"Многопрофильный образовательный комплекс\\" имени Героя Советского Союза Е.Г.Ларикова",\n      "lat": 55.8390658,\n      "lon": 37.5259885,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55970598",\n      "kind": "kindergarten",\n      "name": "Начальная школа - детский сад №1645",\n      "lat": 55.7145319,\n      "lon": 37.7737869,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55970606",\n      "kind": "kindergarten",\n      "name": "Детский сад №934",\n      "lat": 55.7132453,\n      "lon": 37.768591,\n      "opening_hours": null\n    },\n    {\n      "id": "way/55974513",\n      "kind": "school",\n      "name": "Знайка",\n      "lat": 55.8421141,\n      "lon": 37.5314573,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56077411",\n      "kind": "school",\n      "name": "Школа №891",\n      "lat": 55.7211327,\n      "lon": 37.8292776,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56080280",\n      "kind": "kindergarten",\n      "name": "Детский сад №856",\n      "lat": 55.7220138,\n      "lon": 37.8301334,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56460825",\n      "kind": "school",\n      "name": "Лицей №1553",\n      "lat": 55.7210683,\n      "lon": 37.6491053,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56460850",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7266649,\n      "lon": 37.650446,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56461516",\n      "kind": "kindergarten",\n      "name": "Детский сад №1091",\n      "lat": 55.6042732,\n      "lon": 37.7284765,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56677805",\n      "kind": "kindergarten",\n      "name": "Детский сад №2542",\n      "lat": 55.7141247,\n      "lon": 37.7560949,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56677826",\n      "kind": "kindergarten",\n      "name": "Детский сад №2612",\n      "lat": 55.7153842,\n      "lon": 37.7490634,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56677833",\n      "kind": "kindergarten",\n      "name": "Детский сад",\n      "lat": 55.7180951,\n      "lon": 37.7459423,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56677839",\n      "kind": "school",\n      "name": "Школа №1386",\n      "lat": 55.7164928,\n      "lon": 37.7488785,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56725006",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1713",\n      "lat": 55.7126456,\n      "lon": 37.7469512,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56757597",\n      "kind": "school",\n      "name": "Школа №488",\n      "lat": 55.7121068,\n      "lon": 37.742159,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56759631",\n      "kind": "school",\n      "name": "Центр образования №1404 Гамма",\n      "lat": 55.7161704,\n      "lon": 37.8271856,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56759689",\n      "kind": "school",\n      "name": "Гимназия №1512",\n      "lat": 55.7184103,\n      "lon": 37.8302964,\n      "opening_hours": null\n    },\n    {\n      "id": "way/56871236",\n      "kind": "kindergarten",\n      "name": "Детский сад №755",\n      "lat": 55.7127134,\n      "lon": 37.7856828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/57393909",\n      "kind": "kindergarten",\n      "name": "Начальная школа - Детский сад №1767",\n      "lat": 55.8853988,\n      "lon": 37.6586488,\n      "opening_hours": null\n    },\n    {\n      "id": "way/58742202",\n      "kind": "school",\n      "name": "Школа №751",\n      "lat": 55.8841648,\n      "lon": 37.698794,\n      "opening_hours": null\n    },\n    {\n      "id": "way/58780525",\n      "kind": "school",\n      "name": "Школа №947",\n      "lat": 55.5847305,\n      "lon": 37.6739715,\n      "opening_hours": null\n    },\n    {\n      "id": "way/58987047",\n      "kind": "kindergarten",\n      "name": "Детский сад №92",\n      "lat": 55.8644922,\n      "lon": 37.6714101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59370210",\n      "kind": "school",\n      "name": "Школа №233",\n      "lat": 55.8746037,\n      "lon": 37.6504494,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59374931",\n      "kind": "school",\n      "name": "Школа № 1375",\n      "lat": 55.6813942,\n      "lon": 37.6566679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59404034",\n      "kind": "school",\n      "name": "Школа №311",\n      "lat": 55.8742486,\n      "lon": 37.6624178,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59409572",\n      "kind": "kindergarten",\n      "name": "Детский сад №598",\n      "lat": 55.8667985,\n      "lon": 37.6605097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59603291",\n      "kind": "kindergarten",\n      "name": "Детский сад №2082",\n      "lat": 55.7971147,\n      "lon": 37.5112565,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59605079",\n      "kind": "kindergarten",\n      "name": "Детский сад № 325",\n      "lat": 55.8034453,\n      "lon": 37.5154065,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59684131",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №14",\n      "lat": 55.8266437,\n      "lon": 37.8188488,\n      "opening_hours": null\n    },\n    {\n      "id": "way/59966660",\n      "kind": "school",\n      "name": "",\n      "lat": 55.793969,\n      "lon": 37.5071484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/60093986",\n      "kind": "kindergarten",\n      "name": "Детский сад №1591",\n      "lat": 55.8669753,\n      "lon": 37.6525592,\n      "opening_hours": null\n    },\n    {\n      "id": "way/60335496",\n      "kind": "school",\n      "name": "Гимназия №1505",\n      "lat": 55.7955071,\n      "lon": 37.7283299,\n      "opening_hours": null\n    },\n    {\n      "id": "way/60417707",\n      "kind": "kindergarten",\n      "name": "Детский сад 2688",\n      "lat": 55.649298,\n      "lon": 37.5802359,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61240218",\n      "kind": "kindergarten",\n      "name": "Детский сад №1744",\n      "lat": 55.8733695,\n      "lon": 37.6707086,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61299004",\n      "kind": "school",\n      "name": "Школа №368 (Здание №1)",\n      "lat": 55.8221012,\n      "lon": 37.8180648,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61724032",\n      "kind": "school",\n      "name": "Школа №953, ШО 4",\n      "lat": 55.8966981,\n      "lon": 37.5960061,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61732784",\n      "kind": "school",\n      "name": "Школа №249",\n      "lat": 55.8944131,\n      "lon": 37.5941337,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61733498",\n      "kind": "school",\n      "name": "Школа №953, ШО 1",\n      "lat": 55.8966844,\n      "lon": 37.5896343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61891745",\n      "kind": "kindergarten",\n      "name": "Детский сад №1765",\n      "lat": 55.8954586,\n      "lon": 37.5900087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61891748",\n      "kind": "kindergarten",\n      "name": "Детский сад №938",\n      "lat": 55.896791,\n      "lon": 37.5924669,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61893179",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 13",\n      "lat": 55.8607097,\n      "lon": 37.6442328,\n      "opening_hours": null\n    },\n    {\n      "id": "way/61915162",\n      "kind": "school",\n      "name": "Школа №139",\n      "lat": 55.8914523,\n      "lon": 37.5930648,\n      "opening_hours": null\n    },\n    {\n      "id": "way/62112479",\n      "kind": "school",\n      "name": "Общеобразовательная школа «Локомотив»",\n      "lat": 55.8040485,\n      "lon": 37.74375,\n      "opening_hours": null\n    },\n    {\n      "id": "way/62224282",\n      "kind": "school",\n      "name": "Лицей №1571",\n      "lat": 55.8565268,\n      "lon": 37.4440965,\n      "opening_hours": null\n    },\n    {\n      "id": "way/63089977",\n      "kind": "school",\n      "name": "Школа совхоза им. Ленина",\n      "lat": 55.5848122,\n      "lon": 37.7281802,\n      "opening_hours": null\n    },\n    {\n      "id": "way/63090016",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.5858246,\n      "lon": 37.7306579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/63090060",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.5863722,\n      "lon": 37.7306297,\n      "opening_hours": null\n    },\n    {\n      "id": "way/64612074",\n      "kind": "school",\n      "name": "Центр образования №1483 (встр)",\n      "lat": 55.790625,\n      "lon": 37.7936681,\n      "opening_hours": null\n    },\n    {\n      "id": "way/66860549",\n      "kind": "kindergarten",\n      "name": "Детский сад №76",\n      "lat": 55.8035138,\n      "lon": 37.4514418,\n      "opening_hours": null\n    },\n    {\n      "id": "way/67071359",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1129",\n      "lat": 55.6816,\n      "lon": 37.6593821,\n      "opening_hours": null\n    },\n    {\n      "id": "way/68001687",\n      "kind": "school",\n      "name": "Школа №878, отделение №1",\n      "lat": 55.6097946,\n      "lon": 37.7011671,\n      "opening_hours": null\n    },\n    {\n      "id": "way/68260689",\n      "kind": "school",\n      "name": "ЦО №627",\n      "lat": 55.7242809,\n      "lon": 37.6355556,\n      "opening_hours": null\n    },\n    {\n      "id": "way/68260701",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8930185,\n      "lon": 37.621498,\n      "opening_hours": null\n    },\n    {\n      "id": "way/68260702",\n      "kind": "kindergarten",\n      "name": "Детский сад №2152",\n      "lat": 55.893079,\n      "lon": 37.6239456,\n      "opening_hours": null\n    },\n    {\n      "id": "way/69323123",\n      "kind": "school",\n      "name": "Школа №\xa01527",\n      "lat": 55.6822055,\n      "lon": 37.6689442,\n      "opening_hours": null\n    },\n    {\n      "id": "way/69728519",\n      "kind": "school",\n      "name": "Школа №956",\n      "lat": 55.8678901,\n      "lon": 37.6426931,\n      "opening_hours": null\n    },\n    {\n      "id": "way/73854348",\n      "kind": "school",\n      "name": "ГБОУ СОШ № 978 здание № 2",\n      "lat": 55.6816762,\n      "lon": 37.6427718,\n      "opening_hours": null\n    },\n    {\n      "id": "way/73855179",\n      "kind": "school",\n      "name": "ГБОУ школа № 463 корпус №1",\n      "lat": 55.6793275,\n      "lon": 37.6726477,\n      "opening_hours": "Mo-Sa 08:00-20:00"\n    },\n    {\n      "id": "way/75392188",\n      "kind": "kindergarten",\n      "name": "Школа № 1503 Дошкольное отделение",\n      "lat": 55.8158418,\n      "lon": 37.6601449,\n      "opening_hours": null\n    },\n    {\n      "id": "way/75649333",\n      "kind": "kindergarten",\n      "name": "Детский сад №1360",\n      "lat": 55.6068614,\n      "lon": 37.7075582,\n      "opening_hours": null\n    },\n    {\n      "id": "way/76212064",\n      "kind": "school",\n      "name": "Школа №218",\n      "lat": 55.8085809,\n      "lon": 37.5608891,\n      "opening_hours": null\n    },\n    {\n      "id": "way/76345960",\n      "kind": "school",\n      "name": "Центр образования №\xa0218",\n      "lat": 55.8121972,\n      "lon": 37.5747946,\n      "opening_hours": null\n    },\n    {\n      "id": "way/76593145",\n      "kind": "kindergarten",\n      "name": "Детский сад № 706",\n      "lat": 55.6115097,\n      "lon": 37.5873778,\n      "opening_hours": null\n    },\n    {\n      "id": "way/77352043",\n      "kind": "school",\n      "name": "Школа №88",\n      "lat": 55.7368012,\n      "lon": 37.6812459,\n      "opening_hours": null\n    },\n    {\n      "id": "way/77352047",\n      "kind": "kindergarten",\n      "name": "Детский сад №2639",\n      "lat": 55.7366715,\n      "lon": 37.6827566,\n      "opening_hours": null\n    },\n    {\n      "id": "way/77611759",\n      "kind": "school",\n      "name": "Школа №646",\n      "lat": 55.7976044,\n      "lon": 37.7853647,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78273780",\n      "kind": "kindergarten",\n      "name": "Детский сад №1517",\n      "lat": 55.7810407,\n      "lon": 37.7265749,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78287618",\n      "kind": "school",\n      "name": "Школа №1955",\n      "lat": 55.8796704,\n      "lon": 37.68846,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78287658",\n      "kind": "kindergarten",\n      "name": "Детский сад №769",\n      "lat": 55.882664,\n      "lon": 37.6880101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78444429",\n      "kind": "kindergarten",\n      "name": "Детский сад №1755",\n      "lat": 55.7957967,\n      "lon": 37.7923881,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78446879",\n      "kind": "kindergarten",\n      "name": "Детский сад №398",\n      "lat": 55.7349748,\n      "lon": 37.4925174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78628390",\n      "kind": "school",\n      "name": "Школа № 1524 к.2",\n      "lat": 55.6927455,\n      "lon": 37.7200943,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78641368",\n      "kind": "kindergarten",\n      "name": "Детский сад №660",\n      "lat": 55.7945997,\n      "lon": 37.7971727,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78712467",\n      "kind": "school",\n      "name": "Гармония",\n      "lat": 55.5968405,\n      "lon": 37.7467676,\n      "opening_hours": null\n    },\n    {\n      "id": "way/78719017",\n      "kind": "school",\n      "name": "Образовательный центр \\"Газпром\\"",\n      "lat": 55.6628519,\n      "lon": 37.5586915,\n      "opening_hours": "Mo-Fr 08:00-18:30"\n    },\n    {\n      "id": "way/78730245",\n      "kind": "school",\n      "name": "Развилковская средняя школа",\n      "lat": 55.5876329,\n      "lon": 37.7606004,\n      "opening_hours": null\n    },\n    {\n      "id": "way/79054130",\n      "kind": "school",\n      "name": "Школа №419",\n      "lat": 55.799679,\n      "lon": 37.7963295,\n      "opening_hours": null\n    },\n    {\n      "id": "way/79181708",\n      "kind": "kindergarten",\n      "name": "Детский сад №184",\n      "lat": 55.7370857,\n      "lon": 37.4790318,\n      "opening_hours": null\n    },\n    {\n      "id": "way/79394593",\n      "kind": "kindergarten",\n      "name": "Лицей №\xa01571, дошкольное отделение «Ладушки»",\n      "lat": 55.8644779,\n      "lon": 37.4443528,\n      "opening_hours": null\n    },\n    {\n      "id": "way/79529590",\n      "kind": "kindergarten",\n      "name": "Детский сад №1435",\n      "lat": 55.8012194,\n      "lon": 37.7868468,\n      "opening_hours": null\n    },\n    {\n      "id": "way/79534444",\n      "kind": "school",\n      "name": "Школа №1301 имени Е. Т. Гайдара",\n      "lat": 55.7805738,\n      "lon": 37.7237752,\n      "opening_hours": null\n    },\n    {\n      "id": "way/80033791",\n      "kind": "school",\n      "name": "Центр образования №429",\n      "lat": 55.7754703,\n      "lon": 37.7248509,\n      "opening_hours": null\n    },\n    {\n      "id": "way/80632666",\n      "kind": "kindergarten",\n      "name": "Школа - интернат №1",\n      "lat": 55.8173565,\n      "lon": 37.6540721,\n      "opening_hours": null\n    },\n    {\n      "id": "way/81020739",\n      "kind": "kindergarten",\n      "name": "Школа №760 им. А. П. Маресьева. Дошкольное отделение",\n      "lat": 55.8752972,\n      "lon": 37.713858,\n      "opening_hours": null\n    },\n    {\n      "id": "way/81240143",\n      "kind": "kindergarten",\n      "name": "Детский сад №766 присмотра и оздоровления",\n      "lat": 55.8105527,\n      "lon": 37.561927,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/81950814",\n      "kind": "school",\n      "name": "Школа №11",\n      "lat": 55.6887606,\n      "lon": 37.538679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/82083171",\n      "kind": "kindergarten",\n      "name": "Школа № 49. Дошкольный корпус",\n      "lat": 55.6400957,\n      "lon": 37.544641,\n      "opening_hours": null\n    },\n    {\n      "id": "way/82083420",\n      "kind": "school",\n      "name": "Начальная школа №2553",\n      "lat": 55.7522833,\n      "lon": 37.8284181,\n      "opening_hours": null\n    },\n    {\n      "id": "way/82083424",\n      "kind": "kindergarten",\n      "name": "Детский сад №1559",\n      "lat": 55.753749,\n      "lon": 37.8290121,\n      "opening_hours": null\n    },\n    {\n      "id": "way/82137808",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.687607,\n      "lon": 37.5373639,\n      "opening_hours": null\n    },\n    {\n      "id": "way/82210430",\n      "kind": "kindergarten",\n      "name": "Детский сад №1773",\n      "lat": 55.7396047,\n      "lon": 37.4919244,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83015042",\n      "kind": "kindergarten",\n      "name": "Лицей №\xa01571, дошкольное отделение «Радуга»",\n      "lat": 55.8639723,\n      "lon": 37.4397699,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83059652",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7714574,\n      "lon": 37.4889188,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83103906",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение В6",\n      "lat": 55.7359062,\n      "lon": 37.8232353,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83138645",\n      "kind": "school",\n      "name": "Школа №1139",\n      "lat": 55.8609718,\n      "lon": 37.6771567,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83138648",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8595676,\n      "lon": 37.6761092,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83147492",\n      "kind": "kindergarten",\n      "name": "Детский сад № 317",\n      "lat": 55.8139278,\n      "lon": 37.5616617,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83147494",\n      "kind": "school",\n      "name": "Школа № 1454 «Тимирязевская»",\n      "lat": 55.8125141,\n      "lon": 37.5605762,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83168208",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1738",\n      "lat": 55.7705664,\n      "lon": 37.4911491,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83168243",\n      "kind": "school",\n      "name": "Школа № 1010",\n      "lat": 55.770877,\n      "lon": 37.492178,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83238455",\n      "kind": "school",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ШП-3)",\n      "lat": 55.6743875,\n      "lon": 37.8097393,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83257895",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7779766,\n      "lon": 37.4831405,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83257957",\n      "kind": "school",\n      "name": "Школа 1605",\n      "lat": 55.7789532,\n      "lon": 37.4819083,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83258043",\n      "kind": "school",\n      "name": "Лицей №1560 Многопрофильный",\n      "lat": 55.7803493,\n      "lon": 37.4840806,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83290498",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7801254,\n      "lon": 37.4958345,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83389681",\n      "kind": "kindergarten",\n      "name": "Д/C № 367",\n      "lat": 55.6205707,\n      "lon": 37.6510102,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83389694",\n      "kind": "school",\n      "name": "Школа № 868",\n      "lat": 55.6212498,\n      "lon": 37.6533046,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83461703",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.8717013,\n      "lon": 37.6291789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83686950",\n      "kind": "school",\n      "name": "Школа №606",\n      "lat": 55.8718432,\n      "lon": 37.6135458,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83686964",\n      "kind": "kindergarten",\n      "name": "Школа № 1411. Структурное подразделение Детский сад № 370",\n      "lat": 55.8708899,\n      "lon": 37.6022976,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83686968",\n      "kind": "kindergarten",\n      "name": "Школа № 1411. Структурное подразделение Детский сад № 939",\n      "lat": 55.8699699,\n      "lon": 37.5981217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83686971",\n      "kind": "school",\n      "name": "Школа №1411 (СП-761)",\n      "lat": 55.8687417,\n      "lon": 37.5961069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/83687271",\n      "kind": "school",\n      "name": "Школа №970",\n      "lat": 55.8726849,\n      "lon": 37.588898,\n      "opening_hours": null\n    },\n    {\n      "id": "way/84435931",\n      "kind": "kindergarten",\n      "name": "Лицей №\xa01571, дошкольное отделение «Почемучки»",\n      "lat": 55.8584314,\n      "lon": 37.4396335,\n      "opening_hours": null\n    },\n    {\n      "id": "way/84973400",\n      "kind": "school",\n      "name": "Специальная (коррекционная) общеобразовательная школа-интернат № 52",\n      "lat": 55.8135341,\n      "lon": 37.5655981,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85122909",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8114369,\n      "lon": 37.5636635,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85152965",\n      "kind": "kindergarten",\n      "name": "СПДО № 2 школы № 1784",\n      "lat": 55.7826534,\n      "lon": 37.5640949,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85282869",\n      "kind": "school",\n      "name": "Школа № 2115. Корпус школьного образования № 3",\n      "lat": 55.6693903,\n      "lon": 37.5638749,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85641529",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1384 имени А. А. Леманского\\"",\n      "lat": 55.8036484,\n      "lon": 37.5112167,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85642991",\n      "kind": "school",\n      "name": "Школа № 870",\n      "lat": 55.6238793,\n      "lon": 37.6639606,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85666971",\n      "kind": "school",\n      "name": "Школа №1363",\n      "lat": 55.6978026,\n      "lon": 37.8194295,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85711998",\n      "kind": "kindergarten",\n      "name": "Школа №1310 (дошкольный корпус №1)",\n      "lat": 55.7446156,\n      "lon": 37.7866135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85851078",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение школы № 547",\n      "lat": 55.7099323,\n      "lon": 37.6408752,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85910107",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 15",\n      "lat": 55.8528525,\n      "lon": 37.6599388,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85982043",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7979461,\n      "lon": 37.5042209,\n      "opening_hours": null\n    },\n    {\n      "id": "way/85982044",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7972561,\n      "lon": 37.504341,\n      "opening_hours": null\n    },\n    {\n      "id": "way/86163498",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №3",\n      "lat": 55.6594257,\n      "lon": 37.7625936,\n      "opening_hours": null\n    },\n    {\n      "id": "way/86251431",\n      "kind": "school",\n      "name": "Лицей №\xa01571, корпус 7 «Созвездие»",\n      "lat": 55.8588754,\n      "lon": 37.4441623,\n      "opening_hours": null\n    },\n    {\n      "id": "way/86251581",\n      "kind": "kindergarten",\n      "name": "Лицей №\xa01571, дошкольное отделение «Ручеёк»",\n      "lat": 55.8587131,\n      "lon": 37.442513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/86552971",\n      "kind": "school",\n      "name": "Школа № 950",\n      "lat": 55.8589225,\n      "lon": 37.6185548,\n      "opening_hours": null\n    },\n    {\n      "id": "way/87035878",\n      "kind": "kindergarten",\n      "name": "детский сад 1948 РАН",\n      "lat": 55.6738059,\n      "lon": 37.5497594,\n      "opening_hours": null\n    },\n    {\n      "id": "way/87296431",\n      "kind": "school",\n      "name": "Школа №2000 корпус №1",\n      "lat": 55.6421954,\n      "lon": 37.6600432,\n      "opening_hours": null\n    },\n    {\n      "id": "way/87297154",\n      "kind": "kindergarten",\n      "name": "Детский сад №800",\n      "lat": 55.641486,\n      "lon": 37.6587902,\n      "opening_hours": null\n    },\n    {\n      "id": "way/87681246",\n      "kind": "school",\n      "name": "ГОУ ЦО «Технологии обучения»",\n      "lat": 55.7411982,\n      "lon": 37.5501732,\n      "opening_hours": null\n    },\n    {\n      "id": "way/88175214",\n      "kind": "school",\n      "name": "Школа надомного обучения №388",\n      "lat": 55.8938225,\n      "lon": 37.5896932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/88301012",\n      "kind": "school",\n      "name": "Школа № 547",\n      "lat": 55.7092631,\n      "lon": 37.6404573,\n      "opening_hours": null\n    },\n    {\n      "id": "way/88306971",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №4",\n      "lat": 55.61007,\n      "lon": 37.6005377,\n      "opening_hours": null\n    },\n    {\n      "id": "way/88397811",\n      "kind": "school",\n      "name": "Центр образования № 1408",\n      "lat": 55.7006786,\n      "lon": 37.7750219,\n      "opening_hours": null\n    },\n    {\n      "id": "way/88609750",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №5",\n      "lat": 55.6108738,\n      "lon": 37.5966035,\n      "opening_hours": null\n    },\n    {\n      "id": "way/89153919",\n      "kind": "school",\n      "name": "Интернат №73",\n      "lat": 55.6647519,\n      "lon": 37.6081418,\n      "opening_hours": null\n    },\n    {\n      "id": "way/89177699",\n      "kind": "kindergarten",\n      "name": "Детский сад №1046",\n      "lat": 55.8925978,\n      "lon": 37.5926686,\n      "opening_hours": null\n    },\n    {\n      "id": "way/89177701",\n      "kind": "kindergarten",\n      "name": "Детский сад №908",\n      "lat": 55.8944979,\n      "lon": 37.5924914,\n      "opening_hours": null\n    },\n    {\n      "id": "way/89177709",\n      "kind": "kindergarten",\n      "name": "Детский сад №1751",\n      "lat": 55.8959559,\n      "lon": 37.6008785,\n      "opening_hours": null\n    },\n    {\n      "id": "way/89297039",\n      "kind": "school",\n      "name": "Воскресная школа при храме",\n      "lat": 55.6912113,\n      "lon": 37.5938082,\n      "opening_hours": null\n    },\n    {\n      "id": "way/90834744",\n      "kind": "school",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 3.",\n      "lat": 55.651123,\n      "lon": 37.5190565,\n      "opening_hours": null\n    },\n    {\n      "id": "way/91211981",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8129805,\n      "lon": 37.639327,\n      "opening_hours": null\n    },\n    {\n      "id": "way/91233158",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8063841,\n      "lon": 37.6381811,\n      "opening_hours": null\n    },\n    {\n      "id": "way/92139340",\n      "kind": "school",\n      "name": "Гимназия №1539",\n      "lat": 55.8105066,\n      "lon": 37.6418595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/93181279",\n      "kind": "school",\n      "name": "Школа №880",\n      "lat": 55.609456,\n      "lon": 37.5888457,\n      "opening_hours": null\n    },\n    {\n      "id": "way/93476938",\n      "kind": "school",\n      "name": "Школа №\xa01474, подразделение 12",\n      "lat": 55.8707138,\n      "lon": 37.4931824,\n      "opening_hours": null\n    },\n    {\n      "id": "way/93527752",\n      "kind": "school",\n      "name": "Школа № 688",\n      "lat": 55.7685831,\n      "lon": 37.7331741,\n      "opening_hours": null\n    },\n    {\n      "id": "way/93911126",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №4",\n      "lat": 55.6612938,\n      "lon": 37.7621642,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94054910",\n      "kind": "school",\n      "name": "Гимназия №1508",\n      "lat": 55.7920862,\n      "lon": 37.8026258,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94448275",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Росток\\"",\n      "lat": 55.7937162,\n      "lon": 37.488517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94527801",\n      "kind": "kindergarten",\n      "name": "Детский сад №2348",\n      "lat": 55.6532855,\n      "lon": 37.7615797,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94527802",\n      "kind": "school",\n      "name": "Школа №1319",\n      "lat": 55.6525027,\n      "lon": 37.7603479,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94528469",\n      "kind": "school",\n      "name": "Курчатовская школа корпус \\"Факультет\\"",\n      "lat": 55.7973449,\n      "lon": 37.4884448,\n      "opening_hours": null\n    },\n    {\n      "id": "way/94837685",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Винни Пух\\"",\n      "lat": 55.7907943,\n      "lon": 37.4946621,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95172319",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8842057,\n      "lon": 37.6980489,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95172322",\n      "kind": "school",\n      "name": "Школа №763",\n      "lat": 55.8837346,\n      "lon": 37.694233,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95172323",\n      "kind": "kindergarten",\n      "name": "Детский сад №1313",\n      "lat": 55.8852125,\n      "lon": 37.6947536,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95406391",\n      "kind": "school",\n      "name": "Школа №51. Корпус №4",\n      "lat": 55.6442337,\n      "lon": 37.482071,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95692120",\n      "kind": "kindergarten",\n      "name": "Школа 1948 \\"Лингвист-М\\" (д/с 2481)",\n      "lat": 55.6644719,\n      "lon": 37.5749833,\n      "opening_hours": null\n    },\n    {\n      "id": "way/95951840",\n      "kind": "school",\n      "name": "Школа №1955",\n      "lat": 55.8849929,\n      "lon": 37.6831031,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96018896",\n      "kind": "school",\n      "name": "Школа №1207",\n      "lat": 55.6058069,\n      "lon": 37.7097687,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96059113",\n      "kind": "kindergarten",\n      "name": "Детский сад №1241",\n      "lat": 55.607824,\n      "lon": 37.7118112,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96107115",\n      "kind": "kindergarten",\n      "name": "Детский сад №903",\n      "lat": 55.6183098,\n      "lon": 37.721841,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96109435",\n      "kind": "kindergarten",\n      "name": "Д/C № 563",\n      "lat": 55.6259636,\n      "lon": 37.6496628,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96112681",\n      "kind": "kindergarten",\n      "name": "Д/C № 591",\n      "lat": 55.6211785,\n      "lon": 37.6511437,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96114968",\n      "kind": "school",\n      "name": "Школа № 869",\n      "lat": 55.6155795,\n      "lon": 37.6468005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96124576",\n      "kind": "school",\n      "name": "Школа №575",\n      "lat": 55.6012632,\n      "lon": 37.7131464,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96124582",\n      "kind": "kindergarten",\n      "name": "Детский сад №1021",\n      "lat": 55.6027384,\n      "lon": 37.7100135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96124599",\n      "kind": "kindergarten",\n      "name": "Детский сад №857",\n      "lat": 55.6020808,\n      "lon": 37.7112422,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96202738",\n      "kind": "school",\n      "name": "Школа №916",\n      "lat": 55.6118493,\n      "lon": 37.7128652,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96208205",\n      "kind": "kindergarten",\n      "name": "Детский сад № 895",\n      "lat": 55.611131,\n      "lon": 37.7014884,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96264956",\n      "kind": "kindergarten",\n      "name": "Детский сад №986",\n      "lat": 55.6173257,\n      "lon": 37.7227532,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96283955",\n      "kind": "kindergarten",\n      "name": "Д/C № 163",\n      "lat": 55.6317878,\n      "lon": 37.6765845,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96283958",\n      "kind": "kindergarten",\n      "name": "Школа № 982 им. маршала бронетанковых войск П.П. Полубоярова. Здание № 4",\n      "lat": 55.6341199,\n      "lon": 37.6766468,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96283965",\n      "kind": "school",\n      "name": "ЦЛПиДО",\n      "lat": 55.6299408,\n      "lon": 37.6753011,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96300065",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8121897,\n      "lon": 37.7301897,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96324790",\n      "kind": "kindergarten",\n      "name": "Д/C № 600",\n      "lat": 55.6318039,\n      "lon": 37.6649141,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96371786",\n      "kind": "kindergarten",\n      "name": "Детский сад № 24",\n      "lat": 55.8087556,\n      "lon": 37.7242701,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96447491",\n      "kind": "school",\n      "name": "Гимназия №1534 Школьное отделение №2",\n      "lat": 55.6820925,\n      "lon": 37.5707995,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96447494",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6752217,\n      "lon": 37.5785207,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96455034",\n      "kind": "school",\n      "name": "ГБОУ Гимназия № 1534",\n      "lat": 55.6851349,\n      "lon": 37.5741934,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96597526",\n      "kind": "kindergarten",\n      "name": "Детский сад №38",\n      "lat": 55.5985097,\n      "lon": 37.7164217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96597533",\n      "kind": "kindergarten",\n      "name": "Детский сад №2375",\n      "lat": 55.5977136,\n      "lon": 37.7176618,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96597535",\n      "kind": "school",\n      "name": "Школа №544",\n      "lat": 55.5961737,\n      "lon": 37.7195198,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96638899",\n      "kind": "kindergarten",\n      "name": "Детский сад №2269",\n      "lat": 55.8852757,\n      "lon": 37.6928876,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96638900",\n      "kind": "kindergarten",\n      "name": "Детский сад №925",\n      "lat": 55.884937,\n      "lon": 37.6909481,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96644706",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7925475,\n      "lon": 37.4846736,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96645733",\n      "kind": "kindergarten",\n      "name": "Школа № 1503 Дошкольное отделение",\n      "lat": 55.8199575,\n      "lon": 37.6545599,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96675547",\n      "kind": "school",\n      "name": "Школа № 770",\n      "lat": 55.6348918,\n      "lon": 37.6821351,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96709458",\n      "kind": "school",\n      "name": "Школа №7",\n      "lat": 55.8922821,\n      "lon": 37.7346039,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96759949",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1874",\n      "lat": 55.7330247,\n      "lon": 37.6553346,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96844581",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1777",\n      "lat": 55.6270008,\n      "lon": 37.6720864,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96844598",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1974",\n      "lat": 55.6264337,\n      "lon": 37.6705947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96927392",\n      "kind": "school",\n      "name": "ЦДТ \\"Логос\\"",\n      "lat": 55.6304467,\n      "lon": 37.6799684,\n      "opening_hours": null\n    },\n    {\n      "id": "way/96960537",\n      "kind": "kindergarten",\n      "name": "Детский сад №1344",\n      "lat": 55.8777739,\n      "lon": 37.7250149,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97145781",\n      "kind": "kindergarten",\n      "name": "Детский сад №1570",\n      "lat": 55.6080444,\n      "lon": 37.7146544,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97226291",\n      "kind": "kindergarten",\n      "name": "Детский сад № 529",\n      "lat": 55.6357515,\n      "lon": 37.6600789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97337335",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6319089,\n      "lon": 37.664614,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97631336",\n      "kind": "school",\n      "name": "ЦДТ \\"Царицыно\\"",\n      "lat": 55.6341756,\n      "lon": 37.6671377,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97716798",\n      "kind": "school",\n      "name": "Школа №1179. Корпус школьного образования №2",\n      "lat": 55.6365862,\n      "lon": 37.6090692,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97716807",\n      "kind": "kindergarten",\n      "name": "Школа №1179. Корпус дошкольного образования №5",\n      "lat": 55.6360687,\n      "lon": 37.6158065,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97730991",\n      "kind": "kindergarten",\n      "name": "Школа №1179. Корпус дошкольного образования №1",\n      "lat": 55.639098,\n      "lon": 37.6113294,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97730993",\n      "kind": "school",\n      "name": "Школа №1179. Корпус школьного образования №1 (Главное здание)",\n      "lat": 55.6382055,\n      "lon": 37.6092185,\n      "opening_hours": null\n    },\n    {\n      "id": "way/97911511",\n      "kind": "kindergarten",\n      "name": "Детский сад №1787",\n      "lat": 55.6397391,\n      "lon": 37.6614493,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98185508",\n      "kind": "kindergarten",\n      "name": "Детский сад №1153",\n      "lat": 55.6105722,\n      "lon": 37.7036912,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98212210",\n      "kind": "kindergarten",\n      "name": "Детский сад №1092",\n      "lat": 55.6107658,\n      "lon": 37.7049468,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98229514",\n      "kind": "kindergarten",\n      "name": "Детский сад №935",\n      "lat": 55.614347,\n      "lon": 37.7003563,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98229537",\n      "kind": "kindergarten",\n      "name": "Д/C №1969",\n      "lat": 55.6148465,\n      "lon": 37.7024205,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98229550",\n      "kind": "kindergarten",\n      "name": "Детский сад №842",\n      "lat": 55.6154347,\n      "lon": 37.7005803,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98247341",\n      "kind": "school",\n      "name": "Школа №979",\n      "lat": 55.6033917,\n      "lon": 37.7459276,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98330543",\n      "kind": "school",\n      "name": "Школа №832",\n      "lat": 55.605545,\n      "lon": 37.7043986,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98343467",\n      "kind": "school",\n      "name": "Школа №1771 (начальная)",\n      "lat": 55.6062206,\n      "lon": 37.7271051,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98354226",\n      "kind": "kindergarten",\n      "name": "Дом ребенка №25 (неврологический)",\n      "lat": 55.6081123,\n      "lon": 37.730773,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98354245",\n      "kind": "kindergarten",\n      "name": "Детский сад №1275",\n      "lat": 55.605061,\n      "lon": 37.7320255,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98599498",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1527 Корпус Кораблик",\n      "lat": 55.6775586,\n      "lon": 37.6836862,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98607374",\n      "kind": "kindergarten",\n      "name": "Детский сад №1234",\n      "lat": 55.7175321,\n      "lon": 37.6290721,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98668679",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2679",\n      "lat": 55.6336053,\n      "lon": 37.6511293,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98668689",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1970",\n      "lat": 55.6334128,\n      "lon": 37.6523947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98925501",\n      "kind": "kindergarten",\n      "name": "Д/C № 1516",\n      "lat": 55.6252525,\n      "lon": 37.6535702,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98925570",\n      "kind": "school",\n      "name": "Школа №840",\n      "lat": 55.6260928,\n      "lon": 37.6527135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/98943164",\n      "kind": "school",\n      "name": "Школа № 904",\n      "lat": 55.633515,\n      "lon": 37.6658515,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99166330",\n      "kind": "school",\n      "name": "Курчатовская школа \\"ФОК\\"",\n      "lat": 55.8050206,\n      "lon": 37.4699896,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99373841",\n      "kind": "kindergarten",\n      "name": "Детский сад школы №1370 корпус 5",\n      "lat": 55.8855669,\n      "lon": 37.5933237,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99373886",\n      "kind": "kindergarten",\n      "name": "Детский сад школы №1370 корпус 8",\n      "lat": 55.8848474,\n      "lon": 37.5889489,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99510199",\n      "kind": "school",\n      "name": "Школа № 733",\n      "lat": 55.7694685,\n      "lon": 37.7127513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99598008",\n      "kind": "school",\n      "name": "Школа №871",\n      "lat": 55.6174085,\n      "lon": 37.7357033,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99598020",\n      "kind": "school",\n      "name": "Центр образования №1433",\n      "lat": 55.61935,\n      "lon": 37.7356804,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99622644",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8097933,\n      "lon": 37.7211013,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99752573",\n      "kind": "school",\n      "name": "Школа №90 (СП-1)",\n      "lat": 55.7942603,\n      "lon": 37.5162208,\n      "opening_hours": null\n    },\n    {\n      "id": "way/99780170",\n      "kind": "school",\n      "name": "Школа №1287 (1)",\n      "lat": 55.7962258,\n      "lon": 37.5205717,\n      "opening_hours": "Mo-Fr 09:00-18:00; Sa 09:00-15:00"\n    },\n    {\n      "id": "way/100169483",\n      "kind": "kindergarten",\n      "name": "Детский сад №1574",\n      "lat": 55.6109423,\n      "lon": 37.7067648,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101161295",\n      "kind": "kindergarten",\n      "name": "Детский сад №1551",\n      "lat": 55.7513543,\n      "lon": 37.807635,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101264114",\n      "kind": "kindergarten",\n      "name": "Детский сад №1041",\n      "lat": 55.8186345,\n      "lon": 37.7172154,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101264120",\n      "kind": "school",\n      "name": "Школа №390",\n      "lat": 55.8176903,\n      "lon": 37.7158655,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101311726",\n      "kind": "kindergarten",\n      "name": "Детский сад №1823",\n      "lat": 55.6395708,\n      "lon": 37.6661035,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101321916",\n      "kind": "school",\n      "name": "Школа №2000 корпус №2",\n      "lat": 55.641803,\n      "lon": 37.6608917,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101321942",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6415465,\n      "lon": 37.6607226,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101321948",\n      "kind": "kindergarten",\n      "name": "Детский сад №577",\n      "lat": 55.6414308,\n      "lon": 37.6625542,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101502907",\n      "kind": "kindergarten",\n      "name": "Детский сад №753",\n      "lat": 55.7494983,\n      "lon": 37.8103495,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101502964",\n      "kind": "school",\n      "name": "Школа № 2093",\n      "lat": 55.7504036,\n      "lon": 37.8037409,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101540604",\n      "kind": "kindergarten",\n      "name": "Школа №2094. Дошкольная образовательная площадка №1",\n      "lat": 55.6231653,\n      "lon": 37.4741144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101616652",\n      "kind": "kindergarten",\n      "name": "Д/C № 2371",\n      "lat": 55.6251211,\n      "lon": 37.6595298,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101696224",\n      "kind": "kindergarten",\n      "name": "Детский сад №1909",\n      "lat": 55.7515251,\n      "lon": 37.8043551,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101696225",\n      "kind": "kindergarten",\n      "name": "Гимназия №1798 - дошкольное образование «Фениксёнок»",\n      "lat": 55.7512753,\n      "lon": 37.8132308,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101696226",\n      "kind": "school",\n      "name": "Школа №787",\n      "lat": 55.746301,\n      "lon": 37.8108187,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101830726",\n      "kind": "school",\n      "name": "Школа № 501",\n      "lat": 55.6210528,\n      "lon": 37.663093,\n      "opening_hours": null\n    },\n    {\n      "id": "way/101830737",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2071",\n      "lat": 55.6229417,\n      "lon": 37.6607776,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102000128",\n      "kind": "school",\n      "name": "Школа №1370 корпус 3",\n      "lat": 55.8829323,\n      "lon": 37.5911914,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102000135",\n      "kind": "kindergarten",\n      "name": "Детский сад школы №1370 корпус 6",\n      "lat": 55.8830588,\n      "lon": 37.59274,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102199974",\n      "kind": "kindergarten",\n      "name": "Детский сад №548",\n      "lat": 55.6198018,\n      "lon": 37.7025383,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102199982",\n      "kind": "kindergarten",\n      "name": "Детский сад №1635",\n      "lat": 55.6196054,\n      "lon": 37.7035951,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102317311",\n      "kind": "kindergarten",\n      "name": "Школа №1568, дошкольное отделение",\n      "lat": 55.8774177,\n      "lon": 37.6301756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102317312",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.877194,\n      "lon": 37.621689,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102317313",\n      "kind": "school",\n      "name": "Школа №285 имени В.А. Молодцова",\n      "lat": 55.8762763,\n      "lon": 37.6302045,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102326794",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.8732048,\n      "lon": 37.6317888,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102334393",\n      "kind": "kindergarten",\n      "name": "Детский сад №1082",\n      "lat": 55.6177507,\n      "lon": 37.725568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102334394",\n      "kind": "kindergarten",\n      "name": "Детский сад №995",\n      "lat": 55.6188684,\n      "lon": 37.7241093,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102714124",\n      "kind": "kindergarten",\n      "name": "Детский сад №1754",\n      "lat": 55.6389506,\n      "lon": 37.6756264,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102714130",\n      "kind": "school",\n      "name": "Школа №982",\n      "lat": 55.6407751,\n      "lon": 37.6745832,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102714133",\n      "kind": "kindergarten",\n      "name": "Детский сад №922",\n      "lat": 55.6385253,\n      "lon": 37.6784813,\n      "opening_hours": null\n    },\n    {\n      "id": "way/102781292",\n      "kind": "kindergarten",\n      "name": "Детский сад №2620",\n      "lat": 55.6809863,\n      "lon": 37.5760262,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103134505",\n      "kind": "school",\n      "name": "Школа №975",\n      "lat": 55.640089,\n      "lon": 37.7741036,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103134518",\n      "kind": "kindergarten",\n      "name": "Детский сад №1824",\n      "lat": 55.6410917,\n      "lon": 37.7728452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103244527",\n      "kind": "school",\n      "name": "Гимназия №1517",\n      "lat": 55.7858566,\n      "lon": 37.4580566,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103267450",\n      "kind": "school",\n      "name": "Школа №2200 корпус 4",\n      "lat": 55.8047933,\n      "lon": 37.7906194,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103467375",\n      "kind": "school",\n      "name": "Школа №2200 корпус 1",\n      "lat": 55.8073783,\n      "lon": 37.8063305,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103562938",\n      "kind": "school",\n      "name": "Школа № 1384 имени А. А. Леманского",\n      "lat": 55.7982326,\n      "lon": 37.515783,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103566503",\n      "kind": "kindergarten",\n      "name": "Детский сад №1579",\n      "lat": 55.6390562,\n      "lon": 37.76572,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103566512",\n      "kind": "kindergarten",\n      "name": "Детский сад №1663",\n      "lat": 55.6368448,\n      "lon": 37.7677669,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103566513",\n      "kind": "kindergarten",\n      "name": "Детский сад №858",\n      "lat": 55.6375917,\n      "lon": 37.7670916,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103568126",\n      "kind": "kindergarten",\n      "name": "Детский сад №1803",\n      "lat": 55.6432358,\n      "lon": 37.7736275,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103858727",\n      "kind": "kindergarten",\n      "name": "Детский сад №985",\n      "lat": 55.6154952,\n      "lon": 37.7390374,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103858738",\n      "kind": "school",\n      "name": "Школа №945",\n      "lat": 55.6153101,\n      "lon": 37.7366679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103858760",\n      "kind": "kindergarten",\n      "name": "Детский сад-начальная школа Премьер",\n      "lat": 55.6157908,\n      "lon": 37.7400707,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103884054",\n      "kind": "school",\n      "name": "Школа № 1370. Корпус 4",\n      "lat": 55.8815106,\n      "lon": 37.5942509,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103884059",\n      "kind": "kindergarten",\n      "name": "Детский сад №1233 Теремок",\n      "lat": 55.8823738,\n      "lon": 37.5968444,\n      "opening_hours": null\n    },\n    {\n      "id": "way/103884064",\n      "kind": "kindergarten",\n      "name": "Детский сад школы №1370 корпус 7",\n      "lat": 55.8811043,\n      "lon": 37.5902274,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104128400",\n      "kind": "kindergarten",\n      "name": "Центр развития ребенка \\"Дошколёнок\\"",\n      "lat": 55.8149763,\n      "lon": 37.4624898,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104222879",\n      "kind": "school",\n      "name": "Школа №1352 (корпус на Щёлковском шс., 77А)",\n      "lat": 55.813087,\n      "lon": 37.8042638,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104337973",\n      "kind": "kindergarten",\n      "name": "ДОУ №2661",\n      "lat": 55.723152,\n      "lon": 37.5570874,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104531903",\n      "kind": "school",\n      "name": "Школа №1370 корпус 1",\n      "lat": 55.8837855,\n      "lon": 37.5940064,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104531904",\n      "kind": "school",\n      "name": "Школа №1370 корпус 2",\n      "lat": 55.8841771,\n      "lon": 37.5958879,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104539332",\n      "kind": "kindergarten",\n      "name": "Детский сад №914",\n      "lat": 55.6072533,\n      "lon": 37.7041397,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104635222",\n      "kind": "school",\n      "name": "Школа № 1536. Корпус школьного образования № 3",\n      "lat": 55.6839078,\n      "lon": 37.5458227,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104662200",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6450672,\n      "lon": 37.7165202,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104720652",\n      "kind": "kindergarten",\n      "name": "Детский сад №1515",\n      "lat": 55.720801,\n      "lon": 37.567976,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104978925",\n      "kind": "school",\n      "name": "Детская школа искусств «Родник»",\n      "lat": 55.6067823,\n      "lon": 37.7369376,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104978930",\n      "kind": "school",\n      "name": "Школа №1636, здание 10",\n      "lat": 55.6077717,\n      "lon": 37.7384529,\n      "opening_hours": null\n    },\n    {\n      "id": "way/104978931",\n      "kind": "school",\n      "name": "Школа №1636, здание 9",\n      "lat": 55.6085976,\n      "lon": 37.7372412,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105078029",\n      "kind": "kindergarten",\n      "name": "Школа № 1636 \\"НИКА\\", дошкольное отделение",\n      "lat": 55.6056211,\n      "lon": 37.735715,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105137093",\n      "kind": "school",\n      "name": "Школа № 433 им. И.И.Якушкина",\n      "lat": 55.7627029,\n      "lon": 37.7319887,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105197703",\n      "kind": "kindergarten",\n      "name": "Детский сад №1843",\n      "lat": 55.6097727,\n      "lon": 37.7381162,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105199600",\n      "kind": "school",\n      "name": "Православная гимназия Сабурово",\n      "lat": 55.6063226,\n      "lon": 37.7388712,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105200501",\n      "kind": "kindergarten",\n      "name": "Детский сад №69",\n      "lat": 55.6075388,\n      "lon": 37.7415555,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105208194",\n      "kind": "kindergarten",\n      "name": "Детский сад №1277",\n      "lat": 55.6024077,\n      "lon": 37.7428534,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105208196",\n      "kind": "kindergarten",\n      "name": "Детский сад №1274",\n      "lat": 55.6093594,\n      "lon": 37.7425,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105208197",\n      "kind": "kindergarten",\n      "name": "Детский сад №1719",\n      "lat": 55.6010046,\n      "lon": 37.7431306,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105219753",\n      "kind": "kindergarten",\n      "name": "Детский сад № 840",\n      "lat": 55.6160749,\n      "lon": 37.7074916,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105589602",\n      "kind": "school",\n      "name": "Школа №1005",\n      "lat": 55.8072707,\n      "lon": 37.4522987,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105704716",\n      "kind": "school",\n      "name": "Центр образования №1452 Богородский",\n      "lat": 55.8071051,\n      "lon": 37.708861,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105845924",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №2",\n      "lat": 55.5919539,\n      "lon": 37.6115347,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105897910",\n      "kind": "school",\n      "name": "Гимназия \\"Эллада\\"",\n      "lat": 55.643315,\n      "lon": 37.6714518,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105906586",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1768",\n      "lat": 55.6385246,\n      "lon": 37.7001222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105906603",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1762",\n      "lat": 55.6405388,\n      "lon": 37.6942483,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105906667",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1767",\n      "lat": 55.6390277,\n      "lon": 37.6988902,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105906681",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6399842,\n      "lon": 37.6969484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105917632",\n      "kind": "school",\n      "name": "Школа № 2017",\n      "lat": 55.6361605,\n      "lon": 37.7178026,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105917633",\n      "kind": "kindergarten",\n      "name": "Детский сад №2502",\n      "lat": 55.6370202,\n      "lon": 37.7174325,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105917639",\n      "kind": "school",\n      "name": "Наследник",\n      "lat": 55.6360298,\n      "lon": 37.7221514,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105920324",\n      "kind": "kindergarten",\n      "name": "Детский сад №2470",\n      "lat": 55.6369044,\n      "lon": 37.7220333,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105933133",\n      "kind": "kindergarten",\n      "name": "Детский сад №2378",\n      "lat": 55.6327237,\n      "lon": 37.7360331,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105937693",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6336124,\n      "lon": 37.7464629,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105938066",\n      "kind": "kindergarten",\n      "name": "Детский сад №1612",\n      "lat": 55.6336305,\n      "lon": 37.7566248,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105938070",\n      "kind": "kindergarten",\n      "name": "Детский сад №1552",\n      "lat": 55.6350374,\n      "lon": 37.7553313,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105938072",\n      "kind": "kindergarten",\n      "name": "Детский сад №1554",\n      "lat": 55.635758,\n      "lon": 37.7547774,\n      "opening_hours": null\n    },\n    {\n      "id": "way/105938073",\n      "kind": "kindergarten",\n      "name": "Детский сад №1577",\n      "lat": 55.634333,\n      "lon": 37.755985,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106182903",\n      "kind": "school",\n      "name": "Школа №1034",\n      "lat": 55.6329595,\n      "lon": 37.7581561,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106406165",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8072453,\n      "lon": 37.4541005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106445340",\n      "kind": "kindergarten",\n      "name": "Детский сад №928",\n      "lat": 55.8777319,\n      "lon": 37.5736031,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106445366",\n      "kind": "kindergarten",\n      "name": "Детский сад №708",\n      "lat": 55.8755833,\n      "lon": 37.5755957,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106535939",\n      "kind": "school",\n      "name": "Школа № 1034",\n      "lat": 55.6316908,\n      "lon": 37.7584732,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106535942",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2356",\n      "lat": 55.6317389,\n      "lon": 37.7608247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106549295",\n      "kind": "kindergarten",\n      "name": "Детский сад №2365",\n      "lat": 55.6335593,\n      "lon": 37.7683241,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106549299",\n      "kind": "school",\n      "name": "Гимназия №1587",\n      "lat": 55.6353301,\n      "lon": 37.7674789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106573641",\n      "kind": "school",\n      "name": "Начальная школа-детский сад № 1708",\n      "lat": 55.6758483,\n      "lon": 37.5238924,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106573647",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад № 1708",\n      "lat": 55.6754142,\n      "lon": 37.5232704,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106576502",\n      "kind": "kindergarten",\n      "name": "Детский сад №1613",\n      "lat": 55.6395444,\n      "lon": 37.7688964,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106576512",\n      "kind": "kindergarten",\n      "name": "Детский сад №1576",\n      "lat": 55.6405957,\n      "lon": 37.7679666,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106577838",\n      "kind": "school",\n      "name": "Школа №998",\n      "lat": 55.640508,\n      "lon": 37.7663596,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106906402",\n      "kind": "kindergarten",\n      "name": "Детский сад №1664",\n      "lat": 55.8903418,\n      "lon": 37.5255754,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106918211",\n      "kind": "school",\n      "name": "Школа-интернат №5 \\"Преображенский кадетский корпус\\"",\n      "lat": 55.8253329,\n      "lon": 37.710076,\n      "opening_hours": null\n    },\n    {\n      "id": "way/106995473",\n      "kind": "kindergarten",\n      "name": "Центр образования №1858 (Детский сад Стрелец)",\n      "lat": 55.6294937,\n      "lon": 37.7957492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/107101977",\n      "kind": "school",\n      "name": "Школа №999",\n      "lat": 55.639643,\n      "lon": 37.7640597,\n      "opening_hours": null\n    },\n    {\n      "id": "way/107101982",\n      "kind": "school",\n      "name": "Начальная школа №992",\n      "lat": 55.6389934,\n      "lon": 37.7628296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/107715061",\n      "kind": "kindergarten",\n      "name": "Детский сад №1498",\n      "lat": 55.5839058,\n      "lon": 37.6845133,\n      "opening_hours": null\n    },\n    {\n      "id": "way/107715062",\n      "kind": "kindergarten",\n      "name": "Детский дом №18",\n      "lat": 55.5840872,\n      "lon": 37.6874257,\n      "opening_hours": null\n    },\n    {\n      "id": "way/107728025",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Притяжение»",\n      "lat": 55.6393722,\n      "lon": 37.5985664,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108093619",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №6)",\n      "lat": 55.8077904,\n      "lon": 37.7632804,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108201816",\n      "kind": "school",\n      "name": "Школа №1210",\n      "lat": 55.8003725,\n      "lon": 37.4567543,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108202805",\n      "kind": "school",\n      "name": "Коррекционная спецшкола №442",\n      "lat": 55.8015355,\n      "lon": 37.8021316,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108295810",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа № 1454 \\"Тимирязевская\\"",\n      "lat": 55.834045,\n      "lon": 37.5609513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108315230",\n      "kind": "school",\n      "name": "Школа № 1210",\n      "lat": 55.7929162,\n      "lon": 37.4556715,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108322388",\n      "kind": "school",\n      "name": "Специальная общеобразовательная школа №7",\n      "lat": 55.8012596,\n      "lon": 37.490871,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108323089",\n      "kind": "kindergarten",\n      "name": "Детский сад №265",\n      "lat": 55.7947611,\n      "lon": 37.4578265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108446906",\n      "kind": "kindergarten",\n      "name": "Центр развития ребёнка — Детский сад №2452",\n      "lat": 55.8084872,\n      "lon": 37.4526996,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108447327",\n      "kind": "school",\n      "name": "Центр Образования №1874 (Старшая школа)",\n      "lat": 55.8033777,\n      "lon": 37.4635543,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108581514",\n      "kind": "kindergarten",\n      "name": "Детский сад №412",\n      "lat": 55.7960671,\n      "lon": 37.7723484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108680565",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1672",\n      "lat": 55.7041605,\n      "lon": 37.662774,\n      "opening_hours": null\n    },\n    {\n      "id": "way/108715365",\n      "kind": "school",\n      "name": "Школа № 104",\n      "lat": 55.6841211,\n      "lon": 37.5646599,\n      "opening_hours": null\n    },\n    {\n      "id": "way/109308297",\n      "kind": "school",\n      "name": "Школа № 1232",\n      "lat": 55.743093,\n      "lon": 37.5387829,\n      "opening_hours": null\n    },\n    {\n      "id": "way/109562529",\n      "kind": "school",\n      "name": "Школа №1269",\n      "lat": 55.7550073,\n      "lon": 37.7868144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/109562535",\n      "kind": "kindergarten",\n      "name": "Детский сад №1910",\n      "lat": 55.7544601,\n      "lon": 37.7853713,\n      "opening_hours": null\n    },\n    {\n      "id": "way/109562536",\n      "kind": "school",\n      "name": "Школа №450",\n      "lat": 55.7539862,\n      "lon": 37.7863316,\n      "opening_hours": null\n    },\n    {\n      "id": "way/110298846",\n      "kind": "school",\n      "name": "Школа №74",\n      "lat": 55.727616,\n      "lon": 37.523302,\n      "opening_hours": null\n    },\n    {\n      "id": "way/110298855",\n      "kind": "kindergarten",\n      "name": "Детский сад № 805",\n      "lat": 55.7283238,\n      "lon": 37.5228253,\n      "opening_hours": null\n    },\n    {\n      "id": "way/110362226",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №1",\n      "lat": 55.5930469,\n      "lon": 37.608581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/110780000",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6803577,\n      "lon": 37.6008963,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111088772",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6802407,\n      "lon": 37.605767,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111088775",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6802494,\n      "lon": 37.6046395,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111090419",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6810181,\n      "lon": 37.6071642,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111136360",\n      "kind": "school",\n      "name": "Елизаветинская гимназия",\n      "lat": 55.7373544,\n      "lon": 37.6239743,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111538985",\n      "kind": "school",\n      "name": "Средняя школа № 536, отделение № 2",\n      "lat": 55.6495675,\n      "lon": 37.5998812,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111538986",\n      "kind": "school",\n      "name": "ГБПОУ Технический пожарно-спасательный колледж им. В.М. Максимчука",\n      "lat": 55.649517,\n      "lon": 37.6029872,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111543362",\n      "kind": "school",\n      "name": "Школа №1623",\n      "lat": 55.6275861,\n      "lon": 37.6137114,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111543367",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1711",\n      "lat": 55.6282777,\n      "lon": 37.6138966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111609675",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №7",\n      "lat": 55.6152411,\n      "lon": 37.5314247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111859126",\n      "kind": "kindergarten",\n      "name": "Детский сад №2580",\n      "lat": 55.7381186,\n      "lon": 37.6916792,\n      "opening_hours": null\n    },\n    {\n      "id": "way/111988381",\n      "kind": "school",\n      "name": "Школа № 56 им. академика В.А. Легасова. Корпус 2",\n      "lat": 55.7450747,\n      "lon": 37.544995,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112152372",\n      "kind": "kindergarten",\n      "name": "Школа №283, дошкольное отделение 9",\n      "lat": 55.88623,\n      "lon": 37.6706752,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112165344",\n      "kind": "school",\n      "name": "ГБОУ \\"Школа № 67\\", отделение общего образования",\n      "lat": 55.7385691,\n      "lon": 37.5183621,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112390777",\n      "kind": "kindergarten",\n      "name": "Детский сад №6",\n      "lat": 55.6915852,\n      "lon": 37.5476242,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112390793",\n      "kind": "school",\n      "name": "Начальная школа №2086",\n      "lat": 55.6917126,\n      "lon": 37.5448426,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112390798",\n      "kind": "kindergarten",\n      "name": "Детский сад №1260",\n      "lat": 55.6924925,\n      "lon": 37.5454649,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112390800",\n      "kind": "school",\n      "name": "Лицей №1533",\n      "lat": 55.6922876,\n      "lon": 37.5438838,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112398203",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6949539,\n      "lon": 37.5498379,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112398251",\n      "kind": "kindergarten",\n      "name": "Детский сад №2042",\n      "lat": 55.6944455,\n      "lon": 37.5489711,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112427703",\n      "kind": "school",\n      "name": "Школа №1099",\n      "lat": 55.8743686,\n      "lon": 37.724078,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112433183",\n      "kind": "school",\n      "name": "Лицей №1537",\n      "lat": 55.8742268,\n      "lon": 37.719222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112586642",\n      "kind": "kindergarten",\n      "name": "Детский сад №1473",\n      "lat": 55.6292784,\n      "lon": 37.7376793,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112586645",\n      "kind": "school",\n      "name": "Школа № 534 Корпус 1",\n      "lat": 55.6281801,\n      "lon": 37.7382903,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112587135",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6279543,\n      "lon": 37.7356076,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112594626",\n      "kind": "kindergarten",\n      "name": "Детский сад №1745",\n      "lat": 55.6275318,\n      "lon": 37.7477033,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112594627",\n      "kind": "school",\n      "name": "Школа № 534 Корпус 3",\n      "lat": 55.6285027,\n      "lon": 37.7474005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112594628",\n      "kind": "kindergarten",\n      "name": "Детский сад №1752",\n      "lat": 55.6274827,\n      "lon": 37.7449991,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610213",\n      "kind": "school",\n      "name": "Специальная (коррекционная) школа №991",\n      "lat": 55.6248291,\n      "lon": 37.7498145,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610214",\n      "kind": "school",\n      "name": "Гимназия № 1569 \\"Созвездие\\"",\n      "lat": 55.6237287,\n      "lon": 37.7478421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610216",\n      "kind": "kindergarten",\n      "name": "Детский сад ЦРР №1271",\n      "lat": 55.6232964,\n      "lon": 37.7462482,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610217",\n      "kind": "school",\n      "name": "Школа № 1552, здание №2",\n      "lat": 55.6250583,\n      "lon": 37.7448679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610218",\n      "kind": "kindergarten",\n      "name": "Детский дом №9",\n      "lat": 55.6255205,\n      "lon": 37.7456952,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112610219",\n      "kind": "kindergarten",\n      "name": "Школа № 1552, здание №7",\n      "lat": 55.6248228,\n      "lon": 37.7463153,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112628689",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6255601,\n      "lon": 37.7394427,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112628690",\n      "kind": "kindergarten",\n      "name": "Детский сад ЦРР №1271",\n      "lat": 55.6245258,\n      "lon": 37.7382005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/112688156",\n      "kind": "school",\n      "name": "Вешняковская школа. Школьное отделение Ж7",\n      "lat": 55.7332515,\n      "lon": 37.8269055,\n      "opening_hours": null\n    },\n    {\n      "id": "way/113558368",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Маграритка\\"",\n      "lat": 55.7888997,\n      "lon": 37.4930419,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114421932",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2176",\n      "lat": 55.7967979,\n      "lon": 37.484242,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114421935",\n      "kind": "kindergarten",\n      "name": "Детский сад №1106",\n      "lat": 55.8021799,\n      "lon": 37.4890696,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114421948",\n      "kind": "school",\n      "name": "Курчатовская школа корпус \\"Интеграция\\"",\n      "lat": 55.7900674,\n      "lon": 37.477882,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114533462",\n      "kind": "school",\n      "name": "Школа-лицей Возрождение",\n      "lat": 55.7947491,\n      "lon": 37.4596483,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114533471",\n      "kind": "kindergarten",\n      "name": "Детский сад №1679",\n      "lat": 55.8032203,\n      "lon": 37.4683426,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114540677",\n      "kind": "kindergarten",\n      "name": "Центр образования №1874 (детский сад)",\n      "lat": 55.8068079,\n      "lon": 37.4610261,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114553687",\n      "kind": "school",\n      "name": "Школа № 1454 «Тимирязевская»",\n      "lat": 55.8240813,\n      "lon": 37.5676768,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114600871",\n      "kind": "kindergarten",\n      "name": "ДОП \\"Дмитровская 29\\"",\n      "lat": 55.8278722,\n      "lon": 37.5702414,\n      "opening_hours": null\n    },\n    {\n      "id": "way/114671844",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №1397",\n      "lat": 55.7980618,\n      "lon": 37.4618443,\n      "opening_hours": null\n    },\n    {\n      "id": "way/115613768",\n      "kind": "kindergarten",\n      "name": "Детский сад №768",\n      "lat": 55.6508747,\n      "lon": 37.5996555,\n      "opening_hours": null\n    },\n    {\n      "id": "way/115646586",\n      "kind": "school",\n      "name": "Кадетская школа № 1784 имени генерала армии В.А. Матросова",\n      "lat": 55.7848327,\n      "lon": 37.5744311,\n      "opening_hours": null\n    },\n    {\n      "id": "way/115707848",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7733715,\n      "lon": 37.5843441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/115976870",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7910999,\n      "lon": 37.5000854,\n      "opening_hours": null\n    },\n    {\n      "id": "way/116299193",\n      "kind": "kindergarten",\n      "name": "Детский сад №1462",\n      "lat": 55.6665749,\n      "lon": 37.5893849,\n      "opening_hours": null\n    },\n    {\n      "id": "way/116299212",\n      "kind": "school",\n      "name": "Экспериментальная школа (центр образования МЭШ)",\n      "lat": 55.6545965,\n      "lon": 37.5642906,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117196191",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8724675,\n      "lon": 37.7162291,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117348087",\n      "kind": "school",\n      "name": "Школа №1598. Здание №4",\n      "lat": 55.8281604,\n      "lon": 37.8214628,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117348099",\n      "kind": "school",\n      "name": "Школа №1598. Здание №5",\n      "lat": 55.8280996,\n      "lon": 37.8180492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117348110",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №15",\n      "lat": 55.8291357,\n      "lon": 37.8219177,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117368268",\n      "kind": "kindergarten",\n      "name": "Школа №1499 (дошкольное отделение)",\n      "lat": 55.8360441,\n      "lon": 37.6625174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117763674",\n      "kind": "school",\n      "name": "Школа №285 имени В. А. Молодцова",\n      "lat": 55.8769452,\n      "lon": 37.6244087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/117904490",\n      "kind": "school",\n      "name": "Школа № 656 им. А. С. Макаренко",\n      "lat": 55.8676155,\n      "lon": 37.5550262,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118013507",\n      "kind": "school",\n      "name": "Школа № 814. Учебный корпус № 1",\n      "lat": 55.7068301,\n      "lon": 37.4742622,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118034079",\n      "kind": "kindergarten",\n      "name": "Детский сад",\n      "lat": 55.8680186,\n      "lon": 37.48728,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118037513",\n      "kind": "school",\n      "name": "Школа №815",\n      "lat": 55.708096,\n      "lon": 37.4755349,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118037514",\n      "kind": "school",\n      "name": "ГБОУ школа № 814 Учебный корпус 2",\n      "lat": 55.7079276,\n      "lon": 37.4757859,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118055399",\n      "kind": "kindergarten",\n      "name": "Школа № 158. Дошкольное отделение № 3",\n      "lat": 55.8712906,\n      "lon": 37.4665694,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118193796",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №177",\n      "lat": 55.6729938,\n      "lon": 37.7233517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118271893",\n      "kind": "kindergarten",\n      "name": "Центр развития ребенка. Детский сад №2558",\n      "lat": 55.6835407,\n      "lon": 37.5360968,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118555415",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7184905,\n      "lon": 37.5799125,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118555424",\n      "kind": "kindergarten",\n      "name": "Детский сад №2107",\n      "lat": 55.7187486,\n      "lon": 37.5737354,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118570499",\n      "kind": "school",\n      "name": "Центр духовного развития молодежи",\n      "lat": 55.7119819,\n      "lon": 37.6318634,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118641707",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7218279,\n      "lon": 37.578717,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118648732",\n      "kind": "kindergarten",\n      "name": "ДОУ №1975",\n      "lat": 55.7764558,\n      "lon": 37.7102975,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118738013",\n      "kind": "school",\n      "name": "Школа №1532. Корпус школьного образования №05/3ш",\n      "lat": 55.6205856,\n      "lon": 37.4941386,\n      "opening_hours": null\n    },\n    {\n      "id": "way/118792575",\n      "kind": "school",\n      "name": "Школа №1532. Корпус школьного образования №04",\n      "lat": 55.6202798,\n      "lon": 37.4924015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/119072133",\n      "kind": "school",\n      "name": "Школа №352",\n      "lat": 55.8319484,\n      "lon": 37.6639127,\n      "opening_hours": null\n    },\n    {\n      "id": "way/119072246",\n      "kind": "kindergarten",\n      "name": "Детский сад №1247",\n      "lat": 55.8303266,\n      "lon": 37.661106,\n      "opening_hours": null\n    },\n    {\n      "id": "way/119335328",\n      "kind": "school",\n      "name": "Школа №603",\n      "lat": 55.8250739,\n      "lon": 37.5238508,\n      "opening_hours": null\n    },\n    {\n      "id": "way/119917726",\n      "kind": "school",\n      "name": "Школа №1515",\n      "lat": 55.7787929,\n      "lon": 37.4742991,\n      "opening_hours": null\n    },\n    {\n      "id": "way/119917732",\n      "kind": "kindergarten",\n      "name": "Детский сад №1021",\n      "lat": 55.777652,\n      "lon": 37.4737507,\n      "opening_hours": null\n    },\n    {\n      "id": "way/120831644",\n      "kind": "kindergarten",\n      "name": "Детский сад №564",\n      "lat": 55.7831392,\n      "lon": 37.4596205,\n      "opening_hours": null\n    },\n    {\n      "id": "way/122231613",\n      "kind": "school",\n      "name": "Школа №947, Кадетский корпус государственной дипломатической службы им. И.А. Рогачева",\n      "lat": 55.5841777,\n      "lon": 37.6651589,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123099143",\n      "kind": "kindergarten",\n      "name": "Детский сад 1783",\n      "lat": 55.6748188,\n      "lon": 37.5951549,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123099144",\n      "kind": "kindergarten",\n      "name": "Детский сад 1835",\n      "lat": 55.6768991,\n      "lon": 37.5954003,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123099148",\n      "kind": "kindergarten",\n      "name": "Детский сад 577",\n      "lat": 55.6761518,\n      "lon": 37.5967154,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123186332",\n      "kind": "kindergarten",\n      "name": "Детский сад №1",\n      "lat": 55.8844592,\n      "lon": 37.4893802,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123186335",\n      "kind": "kindergarten",\n      "name": "Детский сад № 3",\n      "lat": 55.8834557,\n      "lon": 37.4936137,\n      "opening_hours": null\n    },\n    {\n      "id": "way/123226397",\n      "kind": "school",\n      "name": "Интернат №24",\n      "lat": 55.6528795,\n      "lon": 37.5907662,\n      "opening_hours": null\n    },\n    {\n      "id": "way/124188649",\n      "kind": "kindergarten",\n      "name": "Детский сад №1236 Солнышко",\n      "lat": 55.8082842,\n      "lon": 37.5639872,\n      "opening_hours": null\n    },\n    {\n      "id": "way/124792732",\n      "kind": "school",\n      "name": "Частная школа детский сад «Лотос»",\n      "lat": 55.8929323,\n      "lon": 37.5899646,\n      "opening_hours": null\n    },\n    {\n      "id": "way/124797767",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад №1817",\n      "lat": 55.890783,\n      "lon": 37.5834239,\n      "opening_hours": null\n    },\n    {\n      "id": "way/124797790",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 1416 корпус № 6",\n      "lat": 55.8925464,\n      "lon": 37.5788197,\n      "opening_hours": null\n    },\n    {\n      "id": "way/124797791",\n      "kind": "school",\n      "name": "ГБОУ Школа № 1416 корпус № 2",\n      "lat": 55.8925641,\n      "lon": 37.5810127,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125070311",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Счастливое детство\\"",\n      "lat": 55.8045714,\n      "lon": 37.5720934,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "way/125105442",\n      "kind": "school",\n      "name": "ГБОУ Школа №1797 \\"Богородская\\"",\n      "lat": 55.8119017,\n      "lon": 37.7152192,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125105473",\n      "kind": "kindergarten",\n      "name": "Детский сад №1809",\n      "lat": 55.8097851,\n      "lon": 37.7190151,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125105474",\n      "kind": "kindergarten",\n      "name": "Детский сад №2065",\n      "lat": 55.808692,\n      "lon": 37.7189908,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200345",\n      "kind": "kindergarten",\n      "name": "Детский сад Светлячок",\n      "lat": 55.8969798,\n      "lon": 37.5698469,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200349",\n      "kind": "kindergarten",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №3)",\n      "lat": 55.896814,\n      "lon": 37.5687375,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200362",\n      "kind": "school",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №5)",\n      "lat": 55.8925968,\n      "lon": 37.5711057,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200363",\n      "kind": "kindergarten",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №4)",\n      "lat": 55.897039,\n      "lon": 37.5664383,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200365",\n      "kind": "kindergarten",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №6)",\n      "lat": 55.8939503,\n      "lon": 37.5703869,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200366",\n      "kind": "school",\n      "name": "Знак",\n      "lat": 55.8940178,\n      "lon": 37.5757154,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200368",\n      "kind": "school",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №1)",\n      "lat": 55.8950656,\n      "lon": 37.570219,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125200370",\n      "kind": "school",\n      "name": "Средняя общеобразовательная школа №1449 с дошкольным отделением (Учебный корпус №2)",\n      "lat": 55.8971171,\n      "lon": 37.5715637,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204825",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 1416 корпус № 8",\n      "lat": 55.8925835,\n      "lon": 37.5843378,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204840",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 1416 корпус № 5",\n      "lat": 55.8933469,\n      "lon": 37.5843465,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204842",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 1416 корпус № 4",\n      "lat": 55.8962966,\n      "lon": 37.5840013,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204843",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 1416 корпус № 7",\n      "lat": 55.8953805,\n      "lon": 37.5782693,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204844",\n      "kind": "school",\n      "name": "ГБОУ Школа № 1416 корпус № 1",\n      "lat": 55.8953733,\n      "lon": 37.5814333,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125204845",\n      "kind": "school",\n      "name": "Школа №1416",\n      "lat": 55.8934769,\n      "lon": 37.5813015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/125849340",\n      "kind": "school",\n      "name": "Гимназия №1540",\n      "lat": 55.7853154,\n      "lon": 37.5969853,\n      "opening_hours": null\n    },\n    {\n      "id": "way/126326179",\n      "kind": "school",\n      "name": "Школа № 108",\n      "lat": 55.6761274,\n      "lon": 37.5510441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/126326596",\n      "kind": "school",\n      "name": "Школа № 58",\n      "lat": 55.6783805,\n      "lon": 37.5393496,\n      "opening_hours": null\n    },\n    {\n      "id": "way/127417192",\n      "kind": "kindergarten",\n      "name": "Школа № 962. Дошкольное отделение",\n      "lat": 55.8645895,\n      "lon": 37.5961427,\n      "opening_hours": null\n    },\n    {\n      "id": "way/127868208",\n      "kind": "school",\n      "name": "Школа \\"Ника\\"",\n      "lat": 55.6543635,\n      "lon": 37.595068,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128056567",\n      "kind": "kindergarten",\n      "name": "Детский сад №937",\n      "lat": 55.8896508,\n      "lon": 37.6130377,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128056568",\n      "kind": "school",\n      "name": "Школа №180",\n      "lat": 55.8896209,\n      "lon": 37.6106233,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128111685",\n      "kind": "kindergarten",\n      "name": "Детский сад № 18 \\"Кораблик\\"",\n      "lat": 55.5922283,\n      "lon": 37.7512445,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128116005",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.5891372,\n      "lon": 37.7629338,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128116767",\n      "kind": "kindergarten",\n      "name": "Детский сад № 45 \\"Сказка\\"",\n      "lat": 55.5898354,\n      "lon": 37.7565267,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128263509",\n      "kind": "school",\n      "name": "Школа №225",\n      "lat": 55.824626,\n      "lon": 37.5153224,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128531064",\n      "kind": "school",\n      "name": "Центр образования № 1811 «Измайлово» – начальная школа",\n      "lat": 55.7960544,\n      "lon": 37.8088873,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128531081",\n      "kind": "kindergarten",\n      "name": "Центр образования №1811 Измайлово (детский сад)",\n      "lat": 55.7956776,\n      "lon": 37.8068841,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128594640",\n      "kind": "school",\n      "name": "Школа № 1383. Корпус 2",\n      "lat": 55.8623681,\n      "lon": 37.5657504,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128754875",\n      "kind": "school",\n      "name": "Школа №2200 корпус 3",\n      "lat": 55.8060903,\n      "lon": 37.7967024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128754879",\n      "kind": "school",\n      "name": "Школа №619",\n      "lat": 55.8077222,\n      "lon": 37.7885859,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128754887",\n      "kind": "kindergarten",\n      "name": "Детский сад №211",\n      "lat": 55.8052689,\n      "lon": 37.7936768,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128754894",\n      "kind": "kindergarten",\n      "name": "Детский сад №1457",\n      "lat": 55.8045424,\n      "lon": 37.7936956,\n      "opening_hours": null\n    },\n    {\n      "id": "way/128754895",\n      "kind": "school",\n      "name": "Центр развития творчества детей и юношества им. А.В. Косарева",\n      "lat": 55.8059688,\n      "lon": 37.7892282,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129239209",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 6 (дошкольное отделение)",\n      "lat": 55.8083601,\n      "lon": 37.7963659,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129239218",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 5 (дошкольное отделение)",\n      "lat": 55.8082133,\n      "lon": 37.7913644,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129307720",\n      "kind": "kindergarten",\n      "name": "Детский сад № 841",\n      "lat": 55.7251804,\n      "lon": 37.6321433,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129318472",\n      "kind": "kindergarten",\n      "name": "Детский сад №1828",\n      "lat": 55.7335274,\n      "lon": 37.6749952,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129357954",\n      "kind": "school",\n      "name": "Гимназия №1562 им. Артема Боровика",\n      "lat": 55.6654393,\n      "lon": 37.7507745,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129405498",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6682715,\n      "lon": 37.6510233,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129408334",\n      "kind": "kindergarten",\n      "name": "ДОУ №144",\n      "lat": 55.6773572,\n      "lon": 37.6280809,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129408541",\n      "kind": "kindergarten",\n      "name": "Детский сад №2161",\n      "lat": 55.7239539,\n      "lon": 37.6609878,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129408542",\n      "kind": "kindergarten",\n      "name": "Школа-сад № 494 им. Героя Российской Федерации А. Н. Рожкова",\n      "lat": 55.7255936,\n      "lon": 37.6607453,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129413586",\n      "kind": "kindergarten",\n      "name": "Детский сад №1186",\n      "lat": 55.7313266,\n      "lon": 37.6758185,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129416914",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7396872,\n      "lon": 37.6780399,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129474366",\n      "kind": "school",\n      "name": "Интернат №72",\n      "lat": 55.6656442,\n      "lon": 37.6127148,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129475181",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6497603,\n      "lon": 37.6040084,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129475373",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6639979,\n      "lon": 37.621583,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129529144",\n      "kind": "school",\n      "name": "Школа №473",\n      "lat": 55.6389719,\n      "lon": 37.7976845,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129531218",\n      "kind": "kindergarten",\n      "name": "Центр образования № 1858 - центр дошкольного дополнительного образования",\n      "lat": 55.6391644,\n      "lon": 37.7938312,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129584044",\n      "kind": "school",\n      "name": "Школа №462",\n      "lat": 55.715703,\n      "lon": 37.673287,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129585077",\n      "kind": "school",\n      "name": "Вечерняя школа № 66 ЮВАО",\n      "lat": 55.7289623,\n      "lon": 37.6627126,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129624464",\n      "kind": "kindergarten",\n      "name": "Детский сад №962",\n      "lat": 55.867025,\n      "lon": 37.7012276,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129691146",\n      "kind": "school",\n      "name": "Школа-интернат №9",\n      "lat": 55.695253,\n      "lon": 37.7522334,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129704196",\n      "kind": "kindergarten",\n      "name": "Школа № 1331 Молодая Иверия. Дошкольное отделение",\n      "lat": 55.7481725,\n      "lon": 37.5944135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129704229",\n      "kind": "kindergarten",\n      "name": "ЦВР \\"На Сумском\\" (филиал)",\n      "lat": 55.677459,\n      "lon": 37.6295932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129747822",\n      "kind": "school",\n      "name": "Школа №1321 \\"Ковчег\\"",\n      "lat": 55.7445935,\n      "lon": 37.7192621,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129749845",\n      "kind": "kindergarten",\n      "name": "Детский сад №1946",\n      "lat": 55.76828,\n      "lon": 37.7127492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129760307",\n      "kind": "school",\n      "name": "Учебный центр \\"МИР\\"",\n      "lat": 55.7591469,\n      "lon": 37.7002674,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129760407",\n      "kind": "kindergarten",\n      "name": "Ясли-Сад №1",\n      "lat": 55.7963767,\n      "lon": 37.4621816,\n      "opening_hours": null\n    },\n    {\n      "id": "way/129831577",\n      "kind": "kindergarten",\n      "name": "Детский сад №2651",\n      "lat": 55.7488992,\n      "lon": 37.8127391,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130011189",\n      "kind": "kindergarten",\n      "name": "Школа № 2030. Дошкольное отделение",\n      "lat": 55.7649536,\n      "lon": 37.5590889,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130065838",\n      "kind": "school",\n      "name": "Школа №2048",\n      "lat": 55.7593184,\n      "lon": 37.5177825,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130105062",\n      "kind": "kindergarten",\n      "name": "Детский сад №354",\n      "lat": 55.7565427,\n      "lon": 37.7740169,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130109360",\n      "kind": "school",\n      "name": "Школа № 2115. Корпус школьного образования № 1",\n      "lat": 55.6724679,\n      "lon": 37.5667449,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130278483",\n      "kind": "school",\n      "name": "Школа №1216",\n      "lat": 55.7350969,\n      "lon": 37.6725978,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130512523",\n      "kind": "kindergarten",\n      "name": "Школа №368 (Здание №5)",\n      "lat": 55.8213288,\n      "lon": 37.818129,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130537875",\n      "kind": "kindergarten",\n      "name": "ДОУ №53",\n      "lat": 55.7729035,\n      "lon": 37.7036471,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130630559",\n      "kind": "school",\n      "name": "Гимназия № 45 имени Л. И. Мильграма Общеобразовательное отделение Новочеремушкинская 11а",\n      "lat": 55.6861165,\n      "lon": 37.5866625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130643279",\n      "kind": "kindergarten",\n      "name": "Детский сад №793",\n      "lat": 55.7979832,\n      "lon": 37.8035868,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130688441",\n      "kind": "school",\n      "name": "Школа №1515",\n      "lat": 55.7803156,\n      "lon": 37.4769756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130688456",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7821353,\n      "lon": 37.4746938,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130753111",\n      "kind": "school",\n      "name": "Школа №1381 ШО 3",\n      "lat": 55.8762716,\n      "lon": 37.6971147,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130753112",\n      "kind": "kindergarten",\n      "name": "Детский сад №2379",\n      "lat": 55.8778973,\n      "lon": 37.697789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130819428",\n      "kind": "school",\n      "name": "Детский сад №210",\n      "lat": 55.7768112,\n      "lon": 37.7261106,\n      "opening_hours": null\n    },\n    {\n      "id": "way/130998381",\n      "kind": "school",\n      "name": "Школа №313",\n      "lat": 55.8792135,\n      "lon": 37.7038627,\n      "opening_hours": null\n    },\n    {\n      "id": "way/131000314",\n      "kind": "school",\n      "name": "Московский колледж градостроительства и предпринимательства",\n      "lat": 55.8841624,\n      "lon": 37.7136584,\n      "opening_hours": null\n    },\n    {\n      "id": "way/131483181",\n      "kind": "kindergarten",\n      "name": "Школа №1284",\n      "lat": 55.769578,\n      "lon": 37.6375994,\n      "opening_hours": null\n    },\n    {\n      "id": "way/131663201",\n      "kind": "kindergarten",\n      "name": "Детский сад №1958",\n      "lat": 55.8760183,\n      "lon": 37.6994323,\n      "opening_hours": null\n    },\n    {\n      "id": "way/131751466",\n      "kind": "school",\n      "name": "Свято-Владимирская школа",\n      "lat": 55.7536688,\n      "lon": 37.6406339,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132083765",\n      "kind": "kindergarten",\n      "name": "Детский сад №1518",\n      "lat": 55.7463821,\n      "lon": 37.7852695,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132084593",\n      "kind": "kindergarten",\n      "name": "Детский сад №1617",\n      "lat": 55.7475809,\n      "lon": 37.7882575,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132084698",\n      "kind": "kindergarten",\n      "name": "Детский сад №527",\n      "lat": 55.7487245,\n      "lon": 37.7944755,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132117555",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №4 \\"Белоснежка\\"",\n      "lat": 55.8055241,\n      "lon": 37.8247901,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132725687",\n      "kind": "school",\n      "name": "Школа №245",\n      "lat": 55.8911685,\n      "lon": 37.6130868,\n      "opening_hours": null\n    },\n    {\n      "id": "way/132832339",\n      "kind": "kindergarten",\n      "name": "Детский сад №1108",\n      "lat": 55.8929269,\n      "lon": 37.6135705,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133408372",\n      "kind": "school",\n      "name": "Гимназия №1551 корпус №3 (начальная школа)",\n      "lat": 55.8460173,\n      "lon": 37.4513321,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133408558",\n      "kind": "school",\n      "name": "Гимназия №1551 корпус №2 (основное общее)",\n      "lat": 55.8455084,\n      "lon": 37.449048,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133487277",\n      "kind": "kindergarten",\n      "name": "Школа №1056 (дошкольное отделение №4)",\n      "lat": 55.8479132,\n      "lon": 37.4480763,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133487311",\n      "kind": "kindergarten",\n      "name": "Школа №1056 (дошкольное отделение №5)",\n      "lat": 55.8485331,\n      "lon": 37.4484511,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133487388",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №4 (дошкольное отделение)",\n      "lat": 55.8486631,\n      "lon": 37.4497719,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133487448",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №5 (дошкольное отделение)",\n      "lat": 55.848926,\n      "lon": 37.4519575,\n      "opening_hours": null\n    },\n    {\n      "id": "way/133738328",\n      "kind": "kindergarten",\n      "name": "Детский сад №140",\n      "lat": 55.7851111,\n      "lon": 37.7379828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/134901767",\n      "kind": "school",\n      "name": "АНО \\"Школа \\"Премьер\\"",\n      "lat": 55.6196757,\n      "lon": 37.7256132,\n      "opening_hours": null\n    },\n    {\n      "id": "way/134901780",\n      "kind": "kindergarten",\n      "name": "Детский сад №1134",\n      "lat": 55.614081,\n      "lon": 37.7265053,\n      "opening_hours": null\n    },\n    {\n      "id": "way/134901781",\n      "kind": "kindergarten",\n      "name": "Детский сад №939",\n      "lat": 55.6143531,\n      "lon": 37.727616,\n      "opening_hours": null\n    },\n    {\n      "id": "way/134901782",\n      "kind": "school",\n      "name": "Школа №939",\n      "lat": 55.6134721,\n      "lon": 37.7236603,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135398211",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 435",\n      "lat": 55.8956901,\n      "lon": 37.5327341,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135398212",\n      "kind": "school",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 668",\n      "lat": 55.8962697,\n      "lon": 37.5323568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135640537",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7467461,\n      "lon": 37.5641221,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135742368",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение школы №712",\n      "lat": 55.7266731,\n      "lon": 37.4574602,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135854614",\n      "kind": "kindergarten",\n      "name": "Детский сад №1650",\n      "lat": 55.8883133,\n      "lon": 37.6100126,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135854615",\n      "kind": "kindergarten",\n      "name": "Детский сад №1245",\n      "lat": 55.8868506,\n      "lon": 37.6066122,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135854616",\n      "kind": "school",\n      "name": "Школа №960",\n      "lat": 55.8884301,\n      "lon": 37.6074706,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135854617",\n      "kind": "kindergarten",\n      "name": "Детский сад №909",\n      "lat": 55.8870355,\n      "lon": 37.6048788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135985865",\n      "kind": "school",\n      "name": "ГБОУ Марьинская школа № 1566 корпус 6",\n      "lat": 55.6483386,\n      "lon": 37.7564702,\n      "opening_hours": null\n    },\n    {\n      "id": "way/135993453",\n      "kind": "kindergarten",\n      "name": "ГБОУ Марьинская школа № 1566 ДО 2 Теремок",\n      "lat": 55.6487485,\n      "lon": 37.7574976,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136179391",\n      "kind": "kindergarten",\n      "name": "Прогимназия №1855 (начальная школа – детский сад)",\n      "lat": 55.8942577,\n      "lon": 37.5984969,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136341087",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2323",\n      "lat": 55.64554,\n      "lon": 37.7491263,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136473981",\n      "kind": "kindergarten",\n      "name": "Детский сад №2494",\n      "lat": 55.6580937,\n      "lon": 37.7743893,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136473982",\n      "kind": "kindergarten",\n      "name": "Детский сад №2423",\n      "lat": 55.6582758,\n      "lon": 37.7725845,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136473983",\n      "kind": "kindergarten",\n      "name": "Детский сад №2427",\n      "lat": 55.6588218,\n      "lon": 37.774535,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136752933",\n      "kind": "school",\n      "name": "Школа № 1357 «На Братиславской». Здание №11",\n      "lat": 55.6663031,\n      "lon": 37.7577239,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136752938",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №12",\n      "lat": 55.664775,\n      "lon": 37.7560194,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136752941",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6602526,\n      "lon": 37.7454029,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136752947",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6615265,\n      "lon": 37.7503677,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884279",\n      "kind": "school",\n      "name": "Школа №1412",\n      "lat": 55.8910098,\n      "lon": 37.6013674,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884281",\n      "kind": "kindergarten",\n      "name": "Детский сад №1113",\n      "lat": 55.8894334,\n      "lon": 37.6014801,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884282",\n      "kind": "kindergarten",\n      "name": "Детский сад №2153",\n      "lat": 55.8930637,\n      "lon": 37.6076579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884283",\n      "kind": "kindergarten",\n      "name": "Детский сад №1014",\n      "lat": 55.8896192,\n      "lon": 37.5995616,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/136884284",\n      "kind": "kindergarten",\n      "name": "Детский сад №1115",\n      "lat": 55.8960358,\n      "lon": 37.605869,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884285",\n      "kind": "school",\n      "name": "Школа №255 (бывшая №195)",\n      "lat": 55.8950083,\n      "lon": 37.6056973,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884286",\n      "kind": "kindergarten",\n      "name": "Детский сад №1640",\n      "lat": 55.895026,\n      "lon": 37.6093112,\n      "opening_hours": null\n    },\n    {\n      "id": "way/136884287",\n      "kind": "school",\n      "name": "Школа №1412",\n      "lat": 55.8932066,\n      "lon": 37.6040258,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139537398",\n      "kind": "kindergarten",\n      "name": "Детский сад №1040",\n      "lat": 55.6204674,\n      "lon": 37.7317515,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139537400",\n      "kind": "kindergarten",\n      "name": "Центр образования № 548",\n      "lat": 55.6202134,\n      "lon": 37.7332599,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139537401",\n      "kind": "school",\n      "name": "Школа №534 корпус 2",\n      "lat": 55.6237996,\n      "lon": 37.7344389,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139537404",\n      "kind": "kindergarten",\n      "name": "Детский сад №1167",\n      "lat": 55.623089,\n      "lon": 37.7383628,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139537405",\n      "kind": "kindergarten",\n      "name": "Детский сад №1398",\n      "lat": 55.6228081,\n      "lon": 37.7336169,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139617929",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7711207,\n      "lon": 37.6765969,\n      "opening_hours": null\n    },\n    {\n      "id": "way/139853868",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №17",\n      "lat": 55.82896,\n      "lon": 37.8145966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042486",\n      "kind": "school",\n      "name": "Школа № 2116, школьное отделение № 2",\n      "lat": 55.6179107,\n      "lon": 37.7512138,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042487",\n      "kind": "school",\n      "name": "Школа № 1552, здание №1",\n      "lat": 55.6197632,\n      "lon": 37.7470344,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042488",\n      "kind": "school",\n      "name": "Школа № 2116, Классика",\n      "lat": 55.6203789,\n      "lon": 37.749712,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042489",\n      "kind": "school",\n      "name": "Школа № 1552, здание №3",\n      "lat": 55.6180298,\n      "lon": 37.7468973,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042490",\n      "kind": "kindergarten",\n      "name": "Школа № 2116, дошкольное отделение № 9",\n      "lat": 55.6162099,\n      "lon": 37.7508286,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042491",\n      "kind": "kindergarten",\n      "name": "Школа № 1552, здание №5",\n      "lat": 55.621001,\n      "lon": 37.7470633,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042492",\n      "kind": "kindergarten",\n      "name": "Школа № 1552, здание №6",\n      "lat": 55.6210318,\n      "lon": 37.7491874,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042493",\n      "kind": "school",\n      "name": "Школа № 1552, здание №4",\n      "lat": 55.6167638,\n      "lon": 37.747139,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140042494",\n      "kind": "kindergarten",\n      "name": "Школа № 2116, дошкольное отделение № 4",\n      "lat": 55.6155153,\n      "lon": 37.7505724,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140563393",\n      "kind": "school",\n      "name": "Вечерняя сменная общеобразовательная школа №17",\n      "lat": 55.8077002,\n      "lon": 37.6433932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140563396",\n      "kind": "school",\n      "name": "Школа №308",\n      "lat": 55.8072416,\n      "lon": 37.6421924,\n      "opening_hours": null\n    },\n    {\n      "id": "way/140563398",\n      "kind": "school",\n      "name": "Специальная школа-интернат №1",\n      "lat": 55.8073825,\n      "lon": 37.6441465,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141040653",\n      "kind": "school",\n      "name": "Школа № 524",\n      "lat": 55.6462674,\n      "lon": 37.584172,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141067210",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6147101,\n      "lon": 37.6139895,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643616",\n      "kind": "school",\n      "name": "Школа №318",\n      "lat": 55.8145672,\n      "lon": 37.723657,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643619",\n      "kind": "kindergarten",\n      "name": "Детский сад №534",\n      "lat": 55.8142259,\n      "lon": 37.7245145,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643622",\n      "kind": "kindergarten",\n      "name": "Детский сад №1475",\n      "lat": 55.8148942,\n      "lon": 37.7302036,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643633",\n      "kind": "kindergarten",\n      "name": "Детский сад №564",\n      "lat": 55.8156609,\n      "lon": 37.726792,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643636",\n      "kind": "school",\n      "name": "Школа №362",\n      "lat": 55.8138368,\n      "lon": 37.7286248,\n      "opening_hours": null\n    },\n    {\n      "id": "way/141643639",\n      "kind": "kindergarten",\n      "name": "Детский сад №535",\n      "lat": 55.813324,\n      "lon": 37.725322,\n      "opening_hours": null\n    },\n    {\n      "id": "way/143013486",\n      "kind": "school",\n      "name": "Школа №222 (Корпус 2)",\n      "lat": 55.8862611,\n      "lon": 37.5610346,\n      "opening_hours": null\n    },\n    {\n      "id": "way/143142432",\n      "kind": "school",\n      "name": "Школа №192",\n      "lat": 55.706648,\n      "lon": 37.574794,\n      "opening_hours": null\n    },\n    {\n      "id": "way/143422559",\n      "kind": "school",\n      "name": "Центр образования №118",\n      "lat": 55.6834934,\n      "lon": 37.5345025,\n      "opening_hours": null\n    },\n    {\n      "id": "way/143422563",\n      "kind": "kindergarten",\n      "name": "Детский сад №1225",\n      "lat": 55.6842738,\n      "lon": 37.5365974,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144131944",\n      "kind": "school",\n      "name": "Школа №847",\n      "lat": 55.882246,\n      "lon": 37.5222784,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144131946",\n      "kind": "school",\n      "name": "Школа №683",\n      "lat": 55.8895383,\n      "lon": 37.5245824,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144138168",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 1485",\n      "lat": 55.8938578,\n      "lon": 37.5273598,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144138172",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 656",\n      "lat": 55.89282,\n      "lon": 37.5279553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144138173",\n      "kind": "kindergarten",\n      "name": "Детский сад №1487",\n      "lat": 55.8889314,\n      "lon": 37.5235105,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144284830",\n      "kind": "kindergarten",\n      "name": "Детский сад №929",\n      "lat": 55.8838237,\n      "lon": 37.5250693,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144284837",\n      "kind": "kindergarten",\n      "name": "Детский сад №2246",\n      "lat": 55.8807658,\n      "lon": 37.5233492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144647442",\n      "kind": "kindergarten",\n      "name": "Гимназия №1517",\n      "lat": 55.7731176,\n      "lon": 37.471677,\n      "opening_hours": null\n    },\n    {\n      "id": "way/144884018",\n      "kind": "kindergarten",\n      "name": "Школа 1362 дошкольное отделение",\n      "lat": 55.7746688,\n      "lon": 37.733034,\n      "opening_hours": null\n    },\n    {\n      "id": "way/145111581",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1176",\n      "lat": 55.7684204,\n      "lon": 37.7313753,\n      "opening_hours": null\n    },\n    {\n      "id": "way/145111582",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2090",\n      "lat": 55.7688224,\n      "lon": 37.7346461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/145111583",\n      "kind": "kindergarten",\n      "name": "Детский сад №2566",\n      "lat": 55.769186,\n      "lon": 37.7306148,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146398902",\n      "kind": "school",\n      "name": "Эрудит",\n      "lat": 55.80417,\n      "lon": 37.4599662,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146603847",\n      "kind": "kindergarten",\n      "name": "Детский сад №907",\n      "lat": 55.6248061,\n      "lon": 37.607711,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146603937",\n      "kind": "kindergarten",\n      "name": "Детский сад №698",\n      "lat": 55.6273358,\n      "lon": 37.6001565,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146603939",\n      "kind": "kindergarten",\n      "name": "Детский сад №705",\n      "lat": 55.6256899,\n      "lon": 37.6014653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146696317",\n      "kind": "kindergarten",\n      "name": "Детский сад №212",\n      "lat": 55.7836253,\n      "lon": 37.4516603,\n      "opening_hours": null\n    },\n    {\n      "id": "way/146767851",\n      "kind": "school",\n      "name": "Школа № 2098 им. Героя Советского Союза Л.М. Доватора",\n      "lat": 55.8789895,\n      "lon": 37.5141932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/147003347",\n      "kind": "school",\n      "name": "Школа № 446",\n      "lat": 55.7838189,\n      "lon": 37.7059941,\n      "opening_hours": null\n    },\n    {\n      "id": "way/147043593",\n      "kind": "kindergarten",\n      "name": "Детский сад №151",\n      "lat": 55.6604771,\n      "lon": 37.5998884,\n      "opening_hours": null\n    },\n    {\n      "id": "way/147043594",\n      "kind": "school",\n      "name": "Школа № 516",\n      "lat": 55.6600948,\n      "lon": 37.597322,\n      "opening_hours": null\n    },\n    {\n      "id": "way/147998246",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2112",\n      "lat": 55.7531211,\n      "lon": 37.5373254,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148175692",\n      "kind": "kindergarten",\n      "name": "Детский сад №2247",\n      "lat": 55.8653629,\n      "lon": 37.5689045,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148175699",\n      "kind": "kindergarten",\n      "name": "Детский сад №1823",\n      "lat": 55.8678313,\n      "lon": 37.5668569,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148175701",\n      "kind": "school",\n      "name": "Школа № 1794",\n      "lat": 55.8727403,\n      "lon": 37.5664443,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148175913",\n      "kind": "kindergarten",\n      "name": "Детский сад №1521",\n      "lat": 55.8078002,\n      "lon": 37.7082547,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148190333",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa01704",\n      "lat": 55.6758614,\n      "lon": 37.6914285,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148698553",\n      "kind": "school",\n      "name": "Школа №272",\n      "lat": 55.8686323,\n      "lon": 37.5612752,\n      "opening_hours": null\n    },\n    {\n      "id": "way/148751530",\n      "kind": "school",\n      "name": "Школа № 1245. Корпус школьного образования №3",\n      "lat": 55.6034944,\n      "lon": 37.6016657,\n      "opening_hours": null\n    },\n    {\n      "id": "way/149573325",\n      "kind": "school",\n      "name": "Школа № 1541. Корпус школьного образования",\n      "lat": 55.6722034,\n      "lon": 37.510608,\n      "opening_hours": null\n    },\n    {\n      "id": "way/149865608",\n      "kind": "school",\n      "name": "Школа № 937",\n      "lat": 55.6231737,\n      "lon": 37.6971075,\n      "opening_hours": null\n    },\n    {\n      "id": "way/149865610",\n      "kind": "school",\n      "name": "Московский городской детский телевизионный учебный центр",\n      "lat": 55.6222491,\n      "lon": 37.6978722,\n      "opening_hours": null\n    },\n    {\n      "id": "way/149908800",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1530 \\"Школа Ломоносова\\" Дошкольные группы на Охотничьей",\n      "lat": 55.795574,\n      "lon": 37.6846887,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150185126",\n      "kind": "school",\n      "name": "ГБОУ Школа 1420, Корпус № 12",\n      "lat": 55.7001639,\n      "lon": 37.7872112,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150185142",\n      "kind": "kindergarten",\n      "name": "Детский сад №321",\n      "lat": 55.6999627,\n      "lon": 37.7889057,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150206818",\n      "kind": "kindergarten",\n      "name": "ДОУ №1407",\n      "lat": 55.6628468,\n      "lon": 37.6476168,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150470231",\n      "kind": "school",\n      "name": "Школа №1532. Корпус школьного образования №03",\n      "lat": 55.6213366,\n      "lon": 37.4846388,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150881647",\n      "kind": "school",\n      "name": "Школа №708",\n      "lat": 55.8033541,\n      "lon": 37.7675175,\n      "opening_hours": null\n    },\n    {\n      "id": "way/150964760",\n      "kind": "school",\n      "name": "Центр образования №1468",\n      "lat": 55.7427716,\n      "lon": 37.6809029,\n      "opening_hours": null\n    },\n    {\n      "id": "way/151230707",\n      "kind": "school",\n      "name": "Центр Образования №1874 (Начальная школа)",\n      "lat": 55.805346,\n      "lon": 37.4673533,\n      "opening_hours": null\n    },\n    {\n      "id": "way/151230712",\n      "kind": "kindergarten",\n      "name": "Детский сад №333",\n      "lat": 55.8047605,\n      "lon": 37.4669519,\n      "opening_hours": null\n    },\n    {\n      "id": "way/152795922",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1204",\n      "lat": 55.6837241,\n      "lon": 37.5734087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/153236806",\n      "kind": "school",\n      "name": "Школа №365",\n      "lat": 55.8173651,\n      "lon": 37.7287664,\n      "opening_hours": null\n    },\n    {\n      "id": "way/153479091",\n      "kind": "kindergarten",\n      "name": "Дом ребёнка № 9",\n      "lat": 55.6363278,\n      "lon": 37.5184343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154000034",\n      "kind": "kindergarten",\n      "name": "Детский сад №437 (компенсирующего типа)",\n      "lat": 55.8157708,\n      "lon": 37.7194102,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154000076",\n      "kind": "school",\n      "name": "Школа №442",\n      "lat": 55.8154922,\n      "lon": 37.7223619,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154000082",\n      "kind": "kindergarten",\n      "name": "Детский сад №35",\n      "lat": 55.8171879,\n      "lon": 37.721919,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154000083",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8135927,\n      "lon": 37.7325946,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154047644",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Развитие»",\n      "lat": 55.6320968,\n      "lon": 37.5993351,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154047649",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Гармония»",\n      "lat": 55.6312741,\n      "lon": 37.5967687,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154143461",\n      "kind": "kindergarten",\n      "name": "Ясли-сад \\"Остров детства\\"",\n      "lat": 55.8013716,\n      "lon": 37.5690702,\n      "opening_hours": "Mo-Su 07:00-21:00"\n    },\n    {\n      "id": "way/154331611",\n      "kind": "school",\n      "name": "Школа № 1450 Олимп",\n      "lat": 55.6586424,\n      "lon": 37.6113636,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154331612",\n      "kind": "school",\n      "name": "Школа №1450 Олимп",\n      "lat": 55.6578417,\n      "lon": 37.6123671,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154343006",\n      "kind": "kindergarten",\n      "name": "Детский сад №1040",\n      "lat": 55.8227834,\n      "lon": 37.7173988,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154357103",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7123096,\n      "lon": 37.668503,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154363995",\n      "kind": "kindergarten",\n      "name": "Детский сад №367",\n      "lat": 55.8154176,\n      "lon": 37.7006334,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154363996",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8128319,\n      "lon": 37.7013467,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154368654",\n      "kind": "school",\n      "name": "Школа №1349",\n      "lat": 55.8095204,\n      "lon": 37.7357242,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154430931",\n      "kind": "school",\n      "name": "Школа №1561, корпус 2",\n      "lat": 55.5996604,\n      "lon": 37.5423868,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154544725",\n      "kind": "kindergarten",\n      "name": "Школа 1795 \\"Лосиноостровская\\" Дошкольное отделение",\n      "lat": 55.8178352,\n      "lon": 37.694897,\n      "opening_hours": "Mo-Sa 07:00-19:00"\n    },\n    {\n      "id": "way/154980275",\n      "kind": "school",\n      "name": "",\n      "lat": 55.676499,\n      "lon": 37.5581504,\n      "opening_hours": null\n    },\n    {\n      "id": "way/154980276",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1165",\n      "lat": 55.6770954,\n      "lon": 37.558801,\n      "opening_hours": null\n    },\n    {\n      "id": "way/155340396",\n      "kind": "kindergarten",\n      "name": "СПДО № 2 школы № 1784",\n      "lat": 55.7755299,\n      "lon": 37.5565296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/155551279",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8191332,\n      "lon": 37.6999845,\n      "opening_hours": null\n    },\n    {\n      "id": "way/155653703",\n      "kind": "school",\n      "name": "Школа №1360",\n      "lat": 55.815756,\n      "lon": 37.7110323,\n      "opening_hours": null\n    },\n    {\n      "id": "way/155653757",\n      "kind": "school",\n      "name": "Школа №374",\n      "lat": 55.8163905,\n      "lon": 37.6974535,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156236091",\n      "kind": "kindergarten",\n      "name": "Детский сад №249",\n      "lat": 55.7811408,\n      "lon": 37.5127382,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156236175",\n      "kind": "school",\n      "name": "Школа №141",\n      "lat": 55.7790134,\n      "lon": 37.5144449,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156334837",\n      "kind": "school",\n      "name": "Школа №1284",\n      "lat": 55.7678791,\n      "lon": 37.6384005,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156378577",\n      "kind": "school",\n      "name": "Школа №2121 – школьный корпус №1",\n      "lat": 55.6846108,\n      "lon": 37.7539906,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156527145",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1527 Корпус Солнышко",\n      "lat": 55.6813463,\n      "lon": 37.6713146,\n      "opening_hours": null\n    },\n    {\n      "id": "way/156830473",\n      "kind": "school",\n      "name": "ГБОУ школа № 463 Корпус №2",\n      "lat": 55.6843454,\n      "lon": 37.6815914,\n      "opening_hours": "Mo-Sa 08:00-20:00"\n    },\n    {\n      "id": "way/157140955",\n      "kind": "school",\n      "name": "Школа \\"Самсон\\"",\n      "lat": 55.6838128,\n      "lon": 37.6767863,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157167712",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6491647,\n      "lon": 37.7225809,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157210046",\n      "kind": "school",\n      "name": "Лицей №1580",\n      "lat": 55.642702,\n      "lon": 37.6129445,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157566692",\n      "kind": "kindergarten",\n      "name": "Детский сад №1183",\n      "lat": 55.7443151,\n      "lon": 37.5774122,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157600516",\n      "kind": "school",\n      "name": "Центр образования №1865",\n      "lat": 55.7835789,\n      "lon": 37.5124591,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157617101",\n      "kind": "school",\n      "name": "Школа №684",\n      "lat": 55.8078239,\n      "lon": 37.5158862,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157730576",\n      "kind": "kindergarten",\n      "name": "Детский сад №2136",\n      "lat": 55.7529594,\n      "lon": 37.5048761,\n      "opening_hours": null\n    },\n    {\n      "id": "way/157730583",\n      "kind": "school",\n      "name": "Образовательный центр «Протон». Школа на Новозаводской",\n      "lat": 55.7521711,\n      "lon": 37.5044171,\n      "opening_hours": null\n    },\n    {\n      "id": "way/158029421",\n      "kind": "school",\n      "name": "Школа №1884",\n      "lat": 55.867521,\n      "lon": 37.5296699,\n      "opening_hours": null\n    },\n    {\n      "id": "way/158036921",\n      "kind": "kindergarten",\n      "name": "Пушкинская школа № 1500 Дошкольное отделение",\n      "lat": 55.7698575,\n      "lon": 37.6495632,\n      "opening_hours": null\n    },\n    {\n      "id": "way/158182125",\n      "kind": "school",\n      "name": "Класс-центр",\n      "lat": 55.8219276,\n      "lon": 37.5245478,\n      "opening_hours": null\n    },\n    {\n      "id": "way/158434309",\n      "kind": "kindergarten",\n      "name": "Детский сад №1608",\n      "lat": 55.8687143,\n      "lon": 37.5314946,\n      "opening_hours": null\n    },\n    {\n      "id": "way/158895386",\n      "kind": "kindergarten",\n      "name": "Детский сад №2017",\n      "lat": 55.6875482,\n      "lon": 37.5919705,\n      "opening_hours": null\n    },\n    {\n      "id": "way/159085693",\n      "kind": "kindergarten",\n      "name": "Школа № 1770 \\"Детство\\"",\n      "lat": 55.685896,\n      "lon": 37.6783438,\n      "opening_hours": null\n    },\n    {\n      "id": "way/159918976",\n      "kind": "kindergarten",\n      "name": "ДОУ №1330",\n      "lat": 55.6635315,\n      "lon": 37.6482227,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160150768",\n      "kind": "kindergarten",\n      "name": "ДОУ №67",\n      "lat": 55.6811795,\n      "lon": 37.6391218,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160199979",\n      "kind": "school",\n      "name": "школа №574",\n      "lat": 55.6790395,\n      "lon": 37.6543892,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160283590",\n      "kind": "kindergarten",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ДГ-10)",\n      "lat": 55.6751953,\n      "lon": 37.8078488,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160340355",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7056664,\n      "lon": 37.4723526,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160417652",\n      "kind": "school",\n      "name": "школа №1251",\n      "lat": 55.7941805,\n      "lon": 37.5068391,\n      "opening_hours": null\n    },\n    {\n      "id": "way/160674739",\n      "kind": "kindergarten",\n      "name": "Детский сад №526",\n      "lat": 55.6472563,\n      "lon": 37.651918,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161413502",\n      "kind": "school",\n      "name": "Школа № 2099. Здание 1",\n      "lat": 55.8177179,\n      "lon": 37.4930151,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161413517",\n      "kind": "kindergarten",\n      "name": "Школа № 2099. Здание 4",\n      "lat": 55.8179892,\n      "lon": 37.4911934,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161416878",\n      "kind": "kindergarten",\n      "name": "Детский сад комбинированного вида № 8 «Лебёдушка»",\n      "lat": 55.8924454,\n      "lon": 37.4747833,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161416884",\n      "kind": "kindergarten",\n      "name": "Детский сад №10 Колобок",\n      "lat": 55.8899557,\n      "lon": 37.4742221,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161416892",\n      "kind": "school",\n      "name": "Гимназия №16",\n      "lat": 55.8905784,\n      "lon": 37.4763473,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161420365",\n      "kind": "kindergarten",\n      "name": "Детский сад «Золотой ключик»",\n      "lat": 55.8886522,\n      "lon": 37.4753242,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161849900",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №13",\n      "lat": 55.5927099,\n      "lon": 37.6615374,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161863169",\n      "kind": "kindergarten",\n      "name": "Школа №508 с дошкольным отделением",\n      "lat": 55.5837918,\n      "lon": 37.6956312,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161863170",\n      "kind": "school",\n      "name": "Школа №508 с дошкольным отделением",\n      "lat": 55.5844758,\n      "lon": 37.6974443,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161863171",\n      "kind": "kindergarten",\n      "name": "Детский сад №2472",\n      "lat": 55.5839739,\n      "lon": 37.6890365,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161872281",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 14",\n      "lat": 55.8492998,\n      "lon": 37.6442441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161893388",\n      "kind": "kindergarten",\n      "name": "Детский сад №2675",\n      "lat": 55.7484788,\n      "lon": 37.7786734,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161956483",\n      "kind": "kindergarten",\n      "name": "Детский сад №1436",\n      "lat": 55.6794885,\n      "lon": 37.7572409,\n      "opening_hours": null\n    },\n    {\n      "id": "way/161956486",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6787899,\n      "lon": 37.757046,\n      "opening_hours": null\n    },\n    {\n      "id": "way/162037557",\n      "kind": "kindergarten",\n      "name": "Школа №283, дошкольное отделение 8",\n      "lat": 55.8862434,\n      "lon": 37.6641712,\n      "opening_hours": null\n    },\n    {\n      "id": "way/162037559",\n      "kind": "school",\n      "name": "Школа №283, отделение 7",\n      "lat": 55.885776,\n      "lon": 37.6676882,\n      "opening_hours": null\n    },\n    {\n      "id": "way/162811503",\n      "kind": "school",\n      "name": "Школа №1056 (начальная школа и общее образование)",\n      "lat": 55.8481568,\n      "lon": 37.4438749,\n      "opening_hours": null\n    },\n    {\n      "id": "way/164433805",\n      "kind": "kindergarten",\n      "name": "Детский сад №1477",\n      "lat": 55.584667,\n      "lon": 37.6842664,\n      "opening_hours": null\n    },\n    {\n      "id": "way/165254296",\n      "kind": "kindergarten",\n      "name": "Школа №51. Дошкольные группы",\n      "lat": 55.6417229,\n      "lon": 37.4837087,\n      "opening_hours": null\n    },\n    {\n      "id": "way/165254299",\n      "kind": "school",\n      "name": "Школа №109. Начальная школа",\n      "lat": 55.6426295,\n      "lon": 37.478506,\n      "opening_hours": null\n    },\n    {\n      "id": "way/165467022",\n      "kind": "kindergarten",\n      "name": "ЭЦДиЮТ \\"Родина\\"",\n      "lat": 55.7782153,\n      "lon": 37.8220447,\n      "opening_hours": null\n    },\n    {\n      "id": "way/165706724",\n      "kind": "kindergarten",\n      "name": "Д/C № 2618",\n      "lat": 55.6339332,\n      "lon": 37.6798809,\n      "opening_hours": null\n    },\n    {\n      "id": "way/166477542",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1530 \\"Школа Ломоносова\\" Дошкольное отделение",\n      "lat": 55.7911611,\n      "lon": 37.6816168,\n      "opening_hours": null\n    },\n    {\n      "id": "way/167637695",\n      "kind": "school",\n      "name": "Школа №1598. Здание №6",\n      "lat": 55.8280182,\n      "lon": 37.8142915,\n      "opening_hours": null\n    },\n    {\n      "id": "way/168049902",\n      "kind": "school",\n      "name": "Школа №1514",\n      "lat": 55.6657508,\n      "lon": 37.5249624,\n      "opening_hours": null\n    },\n    {\n      "id": "way/168385808",\n      "kind": "school",\n      "name": "Школа №1374",\n      "lat": 55.8688359,\n      "lon": 37.7037404,\n      "opening_hours": null\n    },\n    {\n      "id": "way/168744868",\n      "kind": "school",\n      "name": "Гимназия №45",\n      "lat": 55.6852034,\n      "lon": 37.5982635,\n      "opening_hours": null\n    },\n    {\n      "id": "way/168840466",\n      "kind": "school",\n      "name": "Школа №445",\n      "lat": 55.7935435,\n      "lon": 37.7811988,\n      "opening_hours": null\n    },\n    {\n      "id": "way/168992391",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Школьный корпус на Зарайской",\n      "lat": 55.7273187,\n      "lon": 37.7784155,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170304996",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Развитие\\"",\n      "lat": 55.6866343,\n      "lon": 37.5967276,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170333997",\n      "kind": "kindergarten",\n      "name": "Детский сад №281",\n      "lat": 55.8843784,\n      "lon": 37.5246873,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170334002",\n      "kind": "kindergarten",\n      "name": "Детский сад №2664",\n      "lat": 55.8847624,\n      "lon": 37.5242061,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170334003",\n      "kind": "school",\n      "name": "Школа №236",\n      "lat": 55.8841757,\n      "lon": 37.5228733,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170945825",\n      "kind": "kindergarten",\n      "name": "Школа № 657. Корпус дошкольного образования №1",\n      "lat": 55.6056535,\n      "lon": 37.6125019,\n      "opening_hours": null\n    },\n    {\n      "id": "way/170945868",\n      "kind": "kindergarten",\n      "name": "Школа № 657. Корпус дошкольного образования №2",\n      "lat": 55.6052753,\n      "lon": 37.6148223,\n      "opening_hours": null\n    },\n    {\n      "id": "way/171694384",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2667",\n      "lat": 55.8823821,\n      "lon": 37.5255291,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172228479",\n      "kind": "kindergarten",\n      "name": "Детский сад №497",\n      "lat": 55.8810008,\n      "lon": 37.5174284,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172278760",\n      "kind": "kindergarten",\n      "name": "Школа № 2054. Дошкольный корпус № 1",\n      "lat": 55.7723434,\n      "lon": 37.6242077,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172288880",\n      "kind": "kindergarten",\n      "name": "Детский сад для ослабленных детей",\n      "lat": 55.7882984,\n      "lon": 37.5960732,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172805530",\n      "kind": "kindergarten",\n      "name": "Детский сад №241",\n      "lat": 55.7658755,\n      "lon": 37.4833836,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172805531",\n      "kind": "kindergarten",\n      "name": "Детский сад №1414",\n      "lat": 55.7644863,\n      "lon": 37.4813463,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172805534",\n      "kind": "school",\n      "name": "Школа №696",\n      "lat": 55.7646651,\n      "lon": 37.4843043,\n      "opening_hours": null\n    },\n    {\n      "id": "way/172839742",\n      "kind": "kindergarten",\n      "name": "Школа № 1297. Дошкольное отделение",\n      "lat": 55.7823177,\n      "lon": 37.6405325,\n      "opening_hours": null\n    },\n    {\n      "id": "way/173206552",\n      "kind": "kindergarten",\n      "name": "Детский сад 2007",\n      "lat": 55.6863958,\n      "lon": 37.6010518,\n      "opening_hours": null\n    },\n    {\n      "id": "way/173567631",\n      "kind": "kindergarten",\n      "name": "Лицей №1571 дошкольное отделение Веснушки",\n      "lat": 55.8545842,\n      "lon": 37.4430321,\n      "opening_hours": null\n    },\n    {\n      "id": "way/174679496",\n      "kind": "kindergarten",\n      "name": "Школа №1352 (дошкольная группа на Щёлковском шс., 91с2)",\n      "lat": 55.8144211,\n      "lon": 37.8244129,\n      "opening_hours": null\n    },\n    {\n      "id": "way/174679502",\n      "kind": "school",\n      "name": "Школа №1352 (корпус на Новосибирской, 8А)",\n      "lat": 55.8158567,\n      "lon": 37.8227454,\n      "opening_hours": null\n    },\n    {\n      "id": "way/175916326",\n      "kind": "kindergarten",\n      "name": "Детский сад №2668",\n      "lat": 55.8691537,\n      "lon": 37.5295515,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176025435",\n      "kind": "school",\n      "name": "Пушкинская школа № 1500. Здание на Скорняжном",\n      "lat": 55.7735481,\n      "lon": 37.6425694,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176491329",\n      "kind": "kindergarten",\n      "name": "Детский сад №1064",\n      "lat": 55.7145643,\n      "lon": 37.7982918,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176751809",\n      "kind": "kindergarten",\n      "name": "Детский сад №1540",\n      "lat": 55.7254551,\n      "lon": 37.8059508,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176751812",\n      "kind": "school",\n      "name": "Школа №407",\n      "lat": 55.7239431,\n      "lon": 37.8121755,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176751813",\n      "kind": "school",\n      "name": "Школа №813",\n      "lat": 55.7200325,\n      "lon": 37.8137074,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176751989",\n      "kind": "kindergarten",\n      "name": "Детский сад №1906",\n      "lat": 55.7171706,\n      "lon": 37.8249971,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176752007",\n      "kind": "kindergarten",\n      "name": "Детский сад №62",\n      "lat": 55.7175272,\n      "lon": 37.8240275,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176752014",\n      "kind": "kindergarten",\n      "name": "Детский сад №1150",\n      "lat": 55.71956,\n      "lon": 37.8265935,\n      "opening_hours": null\n    },\n    {\n      "id": "way/176752045",\n      "kind": "kindergarten",\n      "name": "Прогимназия №1753",\n      "lat": 55.7190542,\n      "lon": 37.8282025,\n      "opening_hours": null\n    },\n    {\n      "id": "way/178441138",\n      "kind": "kindergarten",\n      "name": "Школа №1505 Дошкольные группы",\n      "lat": 55.7974773,\n      "lon": 37.7364447,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/178441142",\n      "kind": "kindergarten",\n      "name": "Детский сад №1343",\n      "lat": 55.7971331,\n      "lon": 37.730984,\n      "opening_hours": null\n    },\n    {\n      "id": "way/178441147",\n      "kind": "school",\n      "name": "Школа №1032",\n      "lat": 55.7958595,\n      "lon": 37.7309944,\n      "opening_hours": null\n    },\n    {\n      "id": "way/178441148",\n      "kind": "school",\n      "name": "Школа №1505 (начальное)",\n      "lat": 55.7969714,\n      "lon": 37.7384377,\n      "opening_hours": null\n    },\n    {\n      "id": "way/179464291",\n      "kind": "kindergarten",\n      "name": "Детский сад №1011",\n      "lat": 55.6250984,\n      "lon": 37.7243024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/179488798",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8092378,\n      "lon": 37.6330871,\n      "opening_hours": null\n    },\n    {\n      "id": "way/179615791",\n      "kind": "kindergarten",\n      "name": "Частный детский сад «Олимпик»",\n      "lat": 55.8068203,\n      "lon": 37.6896419,\n      "opening_hours": null\n    },\n    {\n      "id": "way/181158039",\n      "kind": "kindergarten",\n      "name": "Школа 949 (дошкольное отделение, здание 6)",\n      "lat": 55.6094234,\n      "lon": 37.7483112,\n      "opening_hours": null\n    },\n    {\n      "id": "way/181158040",\n      "kind": "kindergarten",\n      "name": "Школа №949 (дошкольное отделение, здание №7)",\n      "lat": 55.6088728,\n      "lon": 37.7479897,\n      "opening_hours": null\n    },\n    {\n      "id": "way/181333557",\n      "kind": "kindergarten",\n      "name": "Школа №158, дошкольное отделение №7",\n      "lat": 55.8624346,\n      "lon": 37.4760164,\n      "opening_hours": null\n    },\n    {\n      "id": "way/181999177",\n      "kind": "kindergarten",\n      "name": "Школа №949 (дошкольное отделение, здание №4)",\n      "lat": 55.6066217,\n      "lon": 37.7482978,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182027507",\n      "kind": "school",\n      "name": "Гимназия №1569 Созвездие",\n      "lat": 55.6174472,\n      "lon": 37.7392698,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182027508",\n      "kind": "school",\n      "name": "Начальная школа - детский сад №1632",\n      "lat": 55.6149475,\n      "lon": 37.7342948,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182027900",\n      "kind": "kindergarten",\n      "name": "Школа 949 (дошкольное отделение, здание №3)",\n      "lat": 55.6101142,\n      "lon": 37.7569828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182027903",\n      "kind": "school",\n      "name": "Школа № 949 (средняя школа, здание №1)",\n      "lat": 55.6088366,\n      "lon": 37.7539915,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182028220",\n      "kind": "kindergarten",\n      "name": "Школа № 2116, дошкольное отделение № 5",\n      "lat": 55.618126,\n      "lon": 37.7587734,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182028221",\n      "kind": "kindergarten",\n      "name": "Школа № 2116, дошкольное отделение № 8",\n      "lat": 55.6175227,\n      "lon": 37.7549141,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182028222",\n      "kind": "school",\n      "name": "Школа № 2116, школьное отделение № 1",\n      "lat": 55.6207859,\n      "lon": 37.754642,\n      "opening_hours": null\n    },\n    {\n      "id": "way/182028223",\n      "kind": "school",\n      "name": "Школа № 2116, школьное отделение № 3",\n      "lat": 55.6207962,\n      "lon": 37.7567888,\n      "opening_hours": null\n    },\n    {\n      "id": "way/183539979",\n      "kind": "school",\n      "name": "Школа № 1465 им. Н. Г. Кузнецова. Школьное отделение № 2",\n      "lat": 55.7478433,\n      "lon": 37.5679935,\n      "opening_hours": null\n    },\n    {\n      "id": "way/183644378",\n      "kind": "kindergarten",\n      "name": "Детский сад №733",\n      "lat": 55.7991673,\n      "lon": 37.4653919,\n      "opening_hours": null\n    },\n    {\n      "id": "way/183768721",\n      "kind": "kindergarten",\n      "name": "Детский сад №64/1 «Яблонька»",\n      "lat": 55.8887579,\n      "lon": 37.7208656,\n      "opening_hours": null\n    },\n    {\n      "id": "way/183948096",\n      "kind": "school",\n      "name": "Лицей №1580",\n      "lat": 55.6415767,\n      "lon": 37.6126431,\n      "opening_hours": null\n    },\n    {\n      "id": "way/184132673",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1305",\n      "lat": 55.7086369,\n      "lon": 37.6121409,\n      "opening_hours": null\n    },\n    {\n      "id": "way/184132674",\n      "kind": "kindergarten",\n      "name": "Детский сад ЦРР № 1662",\n      "lat": 55.708453,\n      "lon": 37.6167661,\n      "opening_hours": null\n    },\n    {\n      "id": "way/185251618",\n      "kind": "kindergarten",\n      "name": "Школа №1352 (дошкольная группа на Байкальской, 46А)",\n      "lat": 55.8151546,\n      "lon": 37.8180203,\n      "opening_hours": null\n    },\n    {\n      "id": "way/185254551",\n      "kind": "kindergarten",\n      "name": "Детский сад №2565",\n      "lat": 55.806861,\n      "lon": 37.7801568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/185254552",\n      "kind": "school",\n      "name": "Школа №2033",\n      "lat": 55.8079228,\n      "lon": 37.7794539,\n      "opening_hours": null\n    },\n    {\n      "id": "way/185426891",\n      "kind": "school",\n      "name": "Школа №378",\n      "lat": 55.7949577,\n      "lon": 37.696326,\n      "opening_hours": null\n    },\n    {\n      "id": "way/186311028",\n      "kind": "school",\n      "name": "Школа №1290 корпус №2",\n      "lat": 55.7888593,\n      "lon": 37.7795474,\n      "opening_hours": null\n    },\n    {\n      "id": "way/187243714",\n      "kind": "school",\n      "name": "Школа № 1987. Основная школа",\n      "lat": 55.6605666,\n      "lon": 37.7763621,\n      "opening_hours": null\n    },\n    {\n      "id": "way/187244540",\n      "kind": "school",\n      "name": "Центр образования №1426",\n      "lat": 55.5965294,\n      "lon": 37.6600262,\n      "opening_hours": null\n    },\n    {\n      "id": "way/187448839",\n      "kind": "kindergarten",\n      "name": "Центр образования №1404 \\"Гамма\\"",\n      "lat": 55.7982219,\n      "lon": 37.6859968,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188079728",\n      "kind": "school",\n      "name": "Школа №1275",\n      "lat": 55.7837857,\n      "lon": 37.6070414,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188330084",\n      "kind": "school",\n      "name": "Школа №1035",\n      "lat": 55.8202276,\n      "lon": 37.7539524,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188330310",\n      "kind": "kindergarten",\n      "name": "Детский сад №1332",\n      "lat": 55.8217216,\n      "lon": 37.7567904,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188434483",\n      "kind": "school",\n      "name": "Школа №373",\n      "lat": 55.8287111,\n      "lon": 37.7572137,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188434606",\n      "kind": "school",\n      "name": "Школа №1246",\n      "lat": 55.8292877,\n      "lon": 37.7605578,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188435372",\n      "kind": "kindergarten",\n      "name": "Детский сад №605",\n      "lat": 55.8244308,\n      "lon": 37.763765,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188575249",\n      "kind": "kindergarten",\n      "name": "Детский сад №989",\n      "lat": 55.8263258,\n      "lon": 37.7541493,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188575301",\n      "kind": "school",\n      "name": "Школа №835",\n      "lat": 55.8271118,\n      "lon": 37.7524487,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188576666",\n      "kind": "kindergarten",\n      "name": "Детский сад №2669",\n      "lat": 55.82346,\n      "lon": 37.7618518,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188584996",\n      "kind": "kindergarten",\n      "name": "Детский сад №357",\n      "lat": 55.8252612,\n      "lon": 37.7537113,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188584997",\n      "kind": "kindergarten",\n      "name": "Детский сад №1872",\n      "lat": 55.826902,\n      "lon": 37.757298,\n      "opening_hours": null\n    },\n    {\n      "id": "way/188593423",\n      "kind": "kindergarten",\n      "name": "Детский сад №273",\n      "lat": 55.8256595,\n      "lon": 37.7631857,\n      "opening_hours": null\n    },\n    {\n      "id": "way/189136645",\n      "kind": "school",\n      "name": "Гимназия №1572",\n      "lat": 55.7866458,\n      "lon": 37.6101903,\n      "opening_hours": null\n    },\n    {\n      "id": "way/189136646",\n      "kind": "kindergarten",\n      "name": "Детский сад №1918",\n      "lat": 55.7823876,\n      "lon": 37.60879,\n      "opening_hours": null\n    },\n    {\n      "id": "way/189725485",\n      "kind": "school",\n      "name": "СОШ № 198",\n      "lat": 55.7064943,\n      "lon": 37.5835591,\n      "opening_hours": null\n    },\n    {\n      "id": "way/190098593",\n      "kind": "school",\n      "name": "Центр образования №449",\n      "lat": 55.7872543,\n      "lon": 37.7405437,\n      "opening_hours": null\n    },\n    {\n      "id": "way/190118996",\n      "kind": "kindergarten",\n      "name": "Детский сад №573",\n      "lat": 55.7917745,\n      "lon": 37.7724748,\n      "opening_hours": null\n    },\n    {\n      "id": "way/190430436",\n      "kind": "school",\n      "name": "Гимназия №\xa09",\n      "lat": 55.8975721,\n      "lon": 37.454051,\n      "opening_hours": null\n    },\n    {\n      "id": "way/190430438",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa020 «Лукоморье»",\n      "lat": 55.8982473,\n      "lon": 37.4545036,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191278153",\n      "kind": "kindergarten",\n      "name": "Детский сад №449",\n      "lat": 55.8590909,\n      "lon": 37.4810238,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191278154",\n      "kind": "kindergarten",\n      "name": "Детский сад №503",\n      "lat": 55.8584909,\n      "lon": 37.481665,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191278155",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.861878,\n      "lon": 37.4783044,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191318881",\n      "kind": "school",\n      "name": "ДДТ \\"Исток\\"",\n      "lat": 55.6341603,\n      "lon": 37.5897241,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191905193",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6820855,\n      "lon": 37.5734275,\n      "opening_hours": null\n    },\n    {\n      "id": "way/191971338",\n      "kind": "kindergarten",\n      "name": "Детский сад №1435",\n      "lat": 55.8725379,\n      "lon": 37.6685796,\n      "opening_hours": null\n    },\n    {\n      "id": "way/192393322",\n      "kind": "kindergarten",\n      "name": "Детский сад №284",\n      "lat": 55.7776387,\n      "lon": 37.6736783,\n      "opening_hours": null\n    },\n    {\n      "id": "way/192897006",\n      "kind": "kindergarten",\n      "name": "Детский сад №1892",\n      "lat": 55.7413514,\n      "lon": 37.5483953,\n      "opening_hours": null\n    },\n    {\n      "id": "way/192917712",\n      "kind": "kindergarten",\n      "name": "Детский сад №1957",\n      "lat": 55.8300026,\n      "lon": 37.7606707,\n      "opening_hours": null\n    },\n    {\n      "id": "way/193282033",\n      "kind": "kindergarten",\n      "name": "Школа №1499 Дошкольное отделение",\n      "lat": 55.8359403,\n      "lon": 37.6394328,\n      "opening_hours": null\n    },\n    {\n      "id": "way/193443974",\n      "kind": "school",\n      "name": "Школа №1499",\n      "lat": 55.8369942,\n      "lon": 37.6427461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/193637899",\n      "kind": "kindergarten",\n      "name": "Гимназия РУТ (МИИТ)",\n      "lat": 55.8013399,\n      "lon": 37.6548101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/193747059",\n      "kind": "kindergarten",\n      "name": "Школа №158. Дошкольное отделение № 1",\n      "lat": 55.8721214,\n      "lon": 37.4660128,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194080481",\n      "kind": "kindergarten",\n      "name": "Школа №1285 структурное подразделение Золотая рыбка",\n      "lat": 55.8310319,\n      "lon": 37.4490797,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194309723",\n      "kind": "school",\n      "name": "Гимназия 625",\n      "lat": 55.6923343,\n      "lon": 37.5928813,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194309724",\n      "kind": "kindergarten",\n      "name": "Детский сад № 219",\n      "lat": 55.7054076,\n      "lon": 37.5828157,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194309725",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2486",\n      "lat": 55.6940045,\n      "lon": 37.5798872,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194898350",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8290764,\n      "lon": 37.665629,\n      "opening_hours": null\n    },\n    {\n      "id": "way/194898502",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8278859,\n      "lon": 37.6643253,\n      "opening_hours": null\n    },\n    {\n      "id": "way/195523838",\n      "kind": "school",\n      "name": "ГБОУ Школа № 2129 ШО № 1",\n      "lat": 55.700345,\n      "lon": 37.6827521,\n      "opening_hours": "07:30-21:00"\n    },\n    {\n      "id": "way/195529430",\n      "kind": "school",\n      "name": "ГБОУ Школа № 2129 ШО \\"Начальная школа\\"",\n      "lat": 55.7038274,\n      "lon": 37.6813608,\n      "opening_hours": null\n    },\n    {\n      "id": "way/195584571",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129 ДО \\"Колобок\\"",\n      "lat": 55.701554,\n      "lon": 37.6842476,\n      "opening_hours": null\n    },\n    {\n      "id": "way/195584573",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129 ДО \\"Теремок\\"",\n      "lat": 55.7013056,\n      "lon": 37.6854823,\n      "opening_hours": null\n    },\n    {\n      "id": "way/195726456",\n      "kind": "kindergarten",\n      "name": "Детский сад школы с углубленным изучением отдельных предметов № 1794\\". Здание 11",\n      "lat": 55.8628428,\n      "lon": 37.5707068,\n      "opening_hours": null\n    },\n    {\n      "id": "way/196094633",\n      "kind": "school",\n      "name": "Школа им. Маяковского. Учебный корпус «Пролетарка»",\n      "lat": 55.7294242,\n      "lon": 37.6662235,\n      "opening_hours": null\n    },\n    {\n      "id": "way/196482821",\n      "kind": "kindergarten",\n      "name": "Детский сад №1761",\n      "lat": 55.7515823,\n      "lon": 37.8278759,\n      "opening_hours": null\n    },\n    {\n      "id": "way/196482823",\n      "kind": "school",\n      "name": "Лицей № 1502 при МЭИ",\n      "lat": 55.7602998,\n      "lon": 37.8304282,\n      "opening_hours": null\n    },\n    {\n      "id": "way/196952681",\n      "kind": "school",\n      "name": "Школа-интернат №35",\n      "lat": 55.7872785,\n      "lon": 37.602729,\n      "opening_hours": null\n    },\n    {\n      "id": "way/197131575",\n      "kind": "school",\n      "name": "Школа с углубленным изучением отдельных предметов № 1794. Здание 12",\n      "lat": 55.8611353,\n      "lon": 37.5708535,\n      "opening_hours": null\n    },\n    {\n      "id": "way/197879615",\n      "kind": "kindergarten",\n      "name": "Детский сад №1763",\n      "lat": 55.7660534,\n      "lon": 37.7044463,\n      "opening_hours": null\n    },\n    {\n      "id": "way/197907188",\n      "kind": "kindergarten",\n      "name": "Школа № 2107. Здание \\"На Васнецова\\"",\n      "lat": 55.7756811,\n      "lon": 37.6260571,\n      "opening_hours": null\n    },\n    {\n      "id": "way/198366816",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6463836,\n      "lon": 37.5031408,\n      "opening_hours": null\n    },\n    {\n      "id": "way/198366833",\n      "kind": "school",\n      "name": "Школа №1995",\n      "lat": 55.6459557,\n      "lon": 37.4984602,\n      "opening_hours": null\n    },\n    {\n      "id": "way/198660320",\n      "kind": "school",\n      "name": "Школа №167",\n      "lat": 55.8605304,\n      "lon": 37.4793384,\n      "opening_hours": null\n    },\n    {\n      "id": "way/200141633",\n      "kind": "school",\n      "name": "Школа №922 (учебный корпус \\"Ивановский\\")",\n      "lat": 55.7630246,\n      "lon": 37.8270038,\n      "opening_hours": null\n    },\n    {\n      "id": "way/200657576",\n      "kind": "kindergarten",\n      "name": "Школа 1530 дошкольная группа",\n      "lat": 55.7861336,\n      "lon": 37.6974786,\n      "opening_hours": null\n    },\n    {\n      "id": "way/201014765",\n      "kind": "school",\n      "name": "Гимназия МГУПС",\n      "lat": 55.8031976,\n      "lon": 37.6452612,\n      "opening_hours": null\n    },\n    {\n      "id": "way/202018161",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8018156,\n      "lon": 37.6320398,\n      "opening_hours": null\n    },\n    {\n      "id": "way/202195837",\n      "kind": "school",\n      "name": "Школа №259",\n      "lat": 55.7954085,\n      "lon": 37.600514,\n      "opening_hours": null\n    },\n    {\n      "id": "way/202768060",\n      "kind": "kindergarten",\n      "name": "Детский сад №2716",\n      "lat": 55.6900955,\n      "lon": 37.7191721,\n      "opening_hours": null\n    },\n    {\n      "id": "way/202768061",\n      "kind": "school",\n      "name": "Физико-математический лицей № 1524",\n      "lat": 55.6910038,\n      "lon": 37.7197552,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204059721",\n      "kind": "school",\n      "name": "Школа №444",\n      "lat": 55.7906822,\n      "lon": 37.7964766,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204090333",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №7)",\n      "lat": 55.8086988,\n      "lon": 37.7667312,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204347256",\n      "kind": "kindergarten",\n      "name": "Детский сад №2386",\n      "lat": 55.8060555,\n      "lon": 37.7714204,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204356436",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №8)",\n      "lat": 55.8116714,\n      "lon": 37.7799702,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204498542",\n      "kind": "kindergarten",\n      "name": "Школа и детский сад «Колибри»",\n      "lat": 55.8063743,\n      "lon": 37.775343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/204500876",\n      "kind": "school",\n      "name": "Школа №319 (здание №5)",\n      "lat": 55.8107836,\n      "lon": 37.778981,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205589387",\n      "kind": "kindergarten",\n      "name": "Школа №51. Дошкольные группы",\n      "lat": 55.6449675,\n      "lon": 37.4834666,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205590474",\n      "kind": "school",\n      "name": "Школа №319 (здание №4)",\n      "lat": 55.8124387,\n      "lon": 37.7893134,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205680825",\n      "kind": "kindergarten",\n      "name": "Детский сад № 260",\n      "lat": 55.643116,\n      "lon": 37.6017362,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205723070",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №9)",\n      "lat": 55.8120881,\n      "lon": 37.7926665,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205723592",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №10)",\n      "lat": 55.8122372,\n      "lon": 37.7947956,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205833586",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7933307,\n      "lon": 37.57062,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205871984",\n      "kind": "kindergarten",\n      "name": "Детский нефрологический санаторий № 9",\n      "lat": 55.8219747,\n      "lon": 37.7916483,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205878083",\n      "kind": "school",\n      "name": "Школа \\"Эдвайс\\"",\n      "lat": 55.8153211,\n      "lon": 37.791076,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205880108",\n      "kind": "kindergarten",\n      "name": "Школа №319 (здание №11)",\n      "lat": 55.8138312,\n      "lon": 37.7904489,\n      "opening_hours": null\n    },\n    {\n      "id": "way/205880825",\n      "kind": "school",\n      "name": "Школа №319 (здание №3)",\n      "lat": 55.8147538,\n      "lon": 37.7926827,\n      "opening_hours": null\n    },\n    {\n      "id": "way/206079411",\n      "kind": "school",\n      "name": "Лад",\n      "lat": 55.819317,\n      "lon": 37.7954182,\n      "opening_hours": null\n    },\n    {\n      "id": "way/206226399",\n      "kind": "kindergarten",\n      "name": "Детский сад №1276",\n      "lat": 55.6845961,\n      "lon": 37.5360134,\n      "opening_hours": null\n    },\n    {\n      "id": "way/206362274",\n      "kind": "school",\n      "name": "Гимназия №\xa01590 им. Героя Советского Союза В.\xa0В.\xa0Колесника",\n      "lat": 55.8627047,\n      "lon": 37.5029639,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207155606",\n      "kind": "kindergarten",\n      "name": "Детский сад №575",\n      "lat": 55.8086448,\n      "lon": 37.7863197,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207156932",\n      "kind": "kindergarten",\n      "name": "Детский сад №2385",\n      "lat": 55.8082542,\n      "lon": 37.7833293,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207161306",\n      "kind": "kindergarten",\n      "name": "Детский сад №55",\n      "lat": 55.8055319,\n      "lon": 37.7852462,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207161871",\n      "kind": "school",\n      "name": "Школа №399",\n      "lat": 55.8059439,\n      "lon": 37.7866808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207163906",\n      "kind": "kindergarten",\n      "name": "Детский сад №2517",\n      "lat": 55.8040276,\n      "lon": 37.7865481,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207289431",\n      "kind": "school",\n      "name": "Кадетская школа №1381 ШО2",\n      "lat": 55.8740201,\n      "lon": 37.6963064,\n      "opening_hours": null\n    },\n    {\n      "id": "way/207606311",\n      "kind": "school",\n      "name": "Школа №1956",\n      "lat": 55.8028548,\n      "lon": 37.6105094,\n      "opening_hours": null\n    },\n    {\n      "id": "way/208552947",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №8",\n      "lat": 55.6086448,\n      "lon": 37.5285579,\n      "opening_hours": null\n    },\n    {\n      "id": "way/210205065",\n      "kind": "kindergarten",\n      "name": "Школа 1236 Дошкольное отделение",\n      "lat": 55.8151472,\n      "lon": 37.5965992,\n      "opening_hours": null\n    },\n    {\n      "id": "way/210508753",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1497",\n      "lat": 55.7719394,\n      "lon": 37.6083963,\n      "opening_hours": null\n    },\n    {\n      "id": "way/213490165",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 1",\n      "lat": 55.6519138,\n      "lon": 37.5573247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/213490168",\n      "kind": "school",\n      "name": "Школа № 15. Корпус школьного образования № 1",\n      "lat": 55.6526899,\n      "lon": 37.5599967,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216024137",\n      "kind": "kindergarten",\n      "name": "Детский сад №1879",\n      "lat": 55.7455538,\n      "lon": 37.7728428,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216065798",\n      "kind": "school",\n      "name": "Школа №920",\n      "lat": 55.746911,\n      "lon": 37.7780788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216252169",\n      "kind": "school",\n      "name": "Школа №660",\n      "lat": 55.8673079,\n      "lon": 37.534496,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216253188",\n      "kind": "school",\n      "name": "Школа № 656 имени А.С. Макаренко",\n      "lat": 55.8620898,\n      "lon": 37.5573871,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216266765",\n      "kind": "kindergarten",\n      "name": "Детский сад №1026",\n      "lat": 55.878866,\n      "lon": 37.5347724,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216268349",\n      "kind": "school",\n      "name": "Лицей №1594",\n      "lat": 55.8641515,\n      "lon": 37.5405072,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216268350",\n      "kind": "school",\n      "name": "Детский сад №1278",\n      "lat": 55.8651654,\n      "lon": 37.5345097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216269282",\n      "kind": "school",\n      "name": "Школа №662",\n      "lat": 55.8743239,\n      "lon": 37.5205347,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216269283",\n      "kind": "kindergarten",\n      "name": "Детский сад №2311",\n      "lat": 55.8778417,\n      "lon": 37.5132436,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216269285",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 1851",\n      "lat": 55.8950557,\n      "lon": 37.541291,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216269717",\n      "kind": "kindergarten",\n      "name": "Школа № 1631 имени Героя Советского Союза В.П. Кислякова. Учебный корпус № 1631",\n      "lat": 55.8911136,\n      "lon": 37.5396505,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216269825",\n      "kind": "school",\n      "name": "Детский сад №1776",\n      "lat": 55.8758559,\n      "lon": 37.5142108,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216647969",\n      "kind": "kindergarten",\n      "name": "Детский сад №1965",\n      "lat": 55.8829083,\n      "lon": 37.5372014,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216913382",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6639861,\n      "lon": 37.4947696,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216919488",\n      "kind": "kindergarten",\n      "name": "Детский сад №2003",\n      "lat": 55.7181944,\n      "lon": 37.7998232,\n      "opening_hours": null\n    },\n    {\n      "id": "way/216919490",\n      "kind": "school",\n      "name": "Начальная школа - детский сад №1883",\n      "lat": 55.7172537,\n      "lon": 37.8024957,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217369343",\n      "kind": "kindergarten",\n      "name": "Детский сад №2335",\n      "lat": 55.7825631,\n      "lon": 37.5578006,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217369344",\n      "kind": "school",\n      "name": "Лицей №1550",\n      "lat": 55.7824683,\n      "lon": 37.5596722,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217552197",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 2",\n      "lat": 55.7843327,\n      "lon": 37.5275135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217552202",\n      "kind": "kindergarten",\n      "name": "Детский сад №2508",\n      "lat": 55.7849796,\n      "lon": 37.5253089,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217552415",\n      "kind": "kindergarten",\n      "name": "Детский сад №2526",\n      "lat": 55.7870466,\n      "lon": 37.5219908,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217552491",\n      "kind": "kindergarten",\n      "name": "Детский сад №2527",\n      "lat": 55.7883655,\n      "lon": 37.5210882,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217563272",\n      "kind": "kindergarten",\n      "name": "Детский сад №1374",\n      "lat": 55.698004,\n      "lon": 37.7494159,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217566510",\n      "kind": "kindergarten",\n      "name": "Детский сад №2117",\n      "lat": 55.6999258,\n      "lon": 37.7564173,\n      "opening_hours": null\n    },\n    {\n      "id": "way/217573182",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1494",\n      "lat": 55.6995824,\n      "lon": 37.7609847,\n      "opening_hours": null\n    },\n    {\n      "id": "way/218506936",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №5",\n      "lat": 55.6109553,\n      "lon": 37.533306,\n      "opening_hours": null\n    },\n    {\n      "id": "way/218576191",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 1",\n      "lat": 55.8527443,\n      "lon": 37.64264,\n      "opening_hours": null\n    },\n    {\n      "id": "way/218576192",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 10",\n      "lat": 55.8539559,\n      "lon": 37.6457207,\n      "opening_hours": null\n    },\n    {\n      "id": "way/218576193",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 6",\n      "lat": 55.8533937,\n      "lon": 37.6389169,\n      "opening_hours": null\n    },\n    {\n      "id": "way/218780937",\n      "kind": "kindergarten",\n      "name": "Детский сад №2705",\n      "lat": 55.882469,\n      "lon": 37.6894582,\n      "opening_hours": null\n    },\n    {\n      "id": "way/220747230",\n      "kind": "kindergarten",\n      "name": "Школа №830 дошкольное отделение Умка",\n      "lat": 55.8309813,\n      "lon": 37.4550988,\n      "opening_hours": null\n    },\n    {\n      "id": "way/220941601",\n      "kind": "kindergarten",\n      "name": "Школа №2033 (дошкольные группы)",\n      "lat": 55.8066731,\n      "lon": 37.8082801,\n      "opening_hours": null\n    },\n    {\n      "id": "way/220952575",\n      "kind": "kindergarten",\n      "name": "Детский сад №185",\n      "lat": 55.8755934,\n      "lon": 37.5221162,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221105974",\n      "kind": "kindergarten",\n      "name": "Детский сад №438",\n      "lat": 55.659293,\n      "lon": 37.5109541,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221106410",\n      "kind": "school",\n      "name": "Школа № 1541. Корпус школьного образования",\n      "lat": 55.6671995,\n      "lon": 37.5088826,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221246676",\n      "kind": "school",\n      "name": "Школа № 1541. Корпус школьного образования",\n      "lat": 55.6726963,\n      "lon": 37.5045078,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221251555",\n      "kind": "school",\n      "name": "English International School of Moscow",\n      "lat": 55.6674115,\n      "lon": 37.5062793,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221261337",\n      "kind": "kindergarten",\n      "name": "Школа № 1541. Корпус дошкольного образования №3",\n      "lat": 55.6734106,\n      "lon": 37.5086919,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221263466",\n      "kind": "kindergarten",\n      "name": "Детский сад №629",\n      "lat": 55.6570771,\n      "lon": 37.5148594,\n      "opening_hours": null\n    },\n    {\n      "id": "way/221600360",\n      "kind": "kindergarten",\n      "name": "Школа № 1541. Корпус дошкольного образования №5",\n      "lat": 55.6683966,\n      "lon": 37.5101775,\n      "opening_hours": null\n    },\n    {\n      "id": "way/222343517",\n      "kind": "kindergarten",\n      "name": "Детский сад №375",\n      "lat": 55.664523,\n      "lon": 37.5226013,\n      "opening_hours": null\n    },\n    {\n      "id": "way/222343526",\n      "kind": "kindergarten",\n      "name": "Детский сад №406",\n      "lat": 55.6653857,\n      "lon": 37.5225782,\n      "opening_hours": null\n    },\n    {\n      "id": "way/222944753",\n      "kind": "kindergarten",\n      "name": "Детский сад №2521",\n      "lat": 55.8739076,\n      "lon": 37.5181397,\n      "opening_hours": null\n    },\n    {\n      "id": "way/225271999",\n      "kind": "school",\n      "name": "Школа № 1526. Корпус школьного образования",\n      "lat": 55.6028283,\n      "lon": 37.6202075,\n      "opening_hours": null\n    },\n    {\n      "id": "way/225272001",\n      "kind": "kindergarten",\n      "name": "Школа № 657. Корпус дошкольного образования №4",\n      "lat": 55.602798,\n      "lon": 37.6186413,\n      "opening_hours": null\n    },\n    {\n      "id": "way/226915647",\n      "kind": "kindergarten",\n      "name": "Детский сад №1312",\n      "lat": 55.8722461,\n      "lon": 37.6630477,\n      "opening_hours": null\n    },\n    {\n      "id": "way/226915648",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8733562,\n      "lon": 37.6636091,\n      "opening_hours": null\n    },\n    {\n      "id": "way/227478963",\n      "kind": "kindergarten",\n      "name": "Первый Московский Образовательный Комплекс",\n      "lat": 55.8865625,\n      "lon": 37.6499783,\n      "opening_hours": null\n    },\n    {\n      "id": "way/227488072",\n      "kind": "school",\n      "name": "Школа №274",\n      "lat": 55.8838799,\n      "lon": 37.6479545,\n      "opening_hours": null\n    },\n    {\n      "id": "way/227619070",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6912034,\n      "lon": 37.5797537,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228071832",\n      "kind": "kindergarten",\n      "name": "Школа № 950. Структурное подразделение № 12",\n      "lat": 55.8552724,\n      "lon": 37.6110146,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228137029",\n      "kind": "kindergarten",\n      "name": "Детский сад №37 ОАО \\"РЖД\\"",\n      "lat": 55.7907007,\n      "lon": 37.6273513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228407946",\n      "kind": "kindergarten",\n      "name": "Детский сад №33",\n      "lat": 55.7389998,\n      "lon": 37.6878092,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228627305",\n      "kind": "kindergarten",\n      "name": "Детский сад №1413",\n      "lat": 55.7111828,\n      "lon": 37.8110097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228627321",\n      "kind": "kindergarten",\n      "name": "Центр развития ребёнка №2402",\n      "lat": 55.7116161,\n      "lon": 37.8095175,\n      "opening_hours": null\n    },\n    {\n      "id": "way/228627322",\n      "kind": "kindergarten",\n      "name": "Центр развития ребёнка №894",\n      "lat": 55.712108,\n      "lon": 37.8080554,\n      "opening_hours": null\n    },\n    {\n      "id": "way/230771354",\n      "kind": "school",\n      "name": "ЦО №1481",\n      "lat": 55.8744474,\n      "lon": 37.5189514,\n      "opening_hours": null\n    },\n    {\n      "id": "way/230787749",\n      "kind": "school",\n      "name": "Прогимназия №1882",\n      "lat": 55.7056962,\n      "lon": 37.8123139,\n      "opening_hours": null\n    },\n    {\n      "id": "way/232169524",\n      "kind": "school",\n      "name": "Московский городской детский морской центр им. Петра Великого",\n      "lat": 55.8386875,\n      "lon": 37.4843099,\n      "opening_hours": null\n    },\n    {\n      "id": "way/234744383",\n      "kind": "kindergarten",\n      "name": "Детский сад №651",\n      "lat": 55.703062,\n      "lon": 37.6184751,\n      "opening_hours": null\n    },\n    {\n      "id": "way/234883539",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6898616,\n      "lon": 37.4691162,\n      "opening_hours": null\n    },\n    {\n      "id": "way/237560828",\n      "kind": "kindergarten",\n      "name": "Школа им. Н.М. Карамзина. Корпус дошкольного образования №8 «Елизаветинский»",\n      "lat": 55.6087131,\n      "lon": 37.5305501,\n      "opening_hours": null\n    },\n    {\n      "id": "way/238304835",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2273",\n      "lat": 55.6484251,\n      "lon": 37.6601075,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239185830",\n      "kind": "kindergarten",\n      "name": "Школа №1995. Дошкольный уровень",\n      "lat": 55.6446271,\n      "lon": 37.4754571,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239870094",\n      "kind": "school",\n      "name": "Школа № 267",\n      "lat": 55.8721182,\n      "lon": 37.6092383,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239870098",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1350",\n      "lat": 55.8729235,\n      "lon": 37.6082516,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239872122",\n      "kind": "kindergarten",\n      "name": "Детский сад комбинированного вида № 1105",\n      "lat": 55.8718275,\n      "lon": 37.6067908,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239933215",\n      "kind": "kindergarten",\n      "name": "ДОУ №300",\n      "lat": 55.7648793,\n      "lon": 37.6532127,\n      "opening_hours": null\n    },\n    {\n      "id": "way/239977947",\n      "kind": "kindergarten",\n      "name": "Детский сад №1288",\n      "lat": 55.6449616,\n      "lon": 37.7125384,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241164188",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 2 «Страна Ромашково»",\n      "lat": 55.7666389,\n      "lon": 37.5752176,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241167631",\n      "kind": "school",\n      "name": "Лицей № 1568 имени Пабло Неруды",\n      "lat": 55.8746603,\n      "lon": 37.6420919,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241354689",\n      "kind": "kindergarten",\n      "name": "ГБОУ г. Москвы Школа № 1159 (ДО)",\n      "lat": 55.8612811,\n      "lon": 37.511966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241360759",\n      "kind": "school",\n      "name": "Школа №222",\n      "lat": 55.8818245,\n      "lon": 37.5615509,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241361187",\n      "kind": "school",\n      "name": "Школа №1678",\n      "lat": 55.8917945,\n      "lon": 37.5603014,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241361651",\n      "kind": "school",\n      "name": "Школа №1678",\n      "lat": 55.881696,\n      "lon": 37.5553788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241361719",\n      "kind": "kindergarten",\n      "name": "Детский сад №1611",\n      "lat": 55.8808516,\n      "lon": 37.5533875,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241362470",\n      "kind": "school",\n      "name": "Центр образования №1419",\n      "lat": 55.8762384,\n      "lon": 37.563215,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241362598",\n      "kind": "kindergarten",\n      "name": "Школа № 1794. Дошкольное отделение",\n      "lat": 55.8737582,\n      "lon": 37.5651843,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241363249",\n      "kind": "school",\n      "name": "Школа №1296",\n      "lat": 55.8773774,\n      "lon": 37.562658,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241363401",\n      "kind": "kindergarten",\n      "name": "Детский сад №2201",\n      "lat": 55.8787325,\n      "lon": 37.561327,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241363402",\n      "kind": "kindergarten",\n      "name": "Детский сад №2104",\n      "lat": 55.8787335,\n      "lon": 37.5628056,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241364237",\n      "kind": "kindergarten",\n      "name": "Детский сад №22",\n      "lat": 55.8850932,\n      "lon": 37.5605904,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241364238",\n      "kind": "kindergarten",\n      "name": "Детский сад №1230",\n      "lat": 55.8851114,\n      "lon": 37.5616103,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241365040",\n      "kind": "kindergarten",\n      "name": "Детский сад №1241",\n      "lat": 55.8903375,\n      "lon": 37.560558,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241365041",\n      "kind": "kindergarten",\n      "name": "Детский сад №1323",\n      "lat": 55.8903394,\n      "lon": 37.5617939,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241365042",\n      "kind": "kindergarten",\n      "name": "Центр образования №1678",\n      "lat": 55.892004,\n      "lon": 37.5626638,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241366995",\n      "kind": "kindergarten",\n      "name": "Детский сад №1020",\n      "lat": 55.8876309,\n      "lon": 37.5606067,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241366996",\n      "kind": "kindergarten",\n      "name": "Детский сад № 14",\n      "lat": 55.8876311,\n      "lon": 37.5615798,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241371253",\n      "kind": "school",\n      "name": "Гимназия №1592",\n      "lat": 55.8749908,\n      "lon": 37.5548261,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241374319",\n      "kind": "kindergarten",\n      "name": "Детский сад №154",\n      "lat": 55.8659715,\n      "lon": 37.5631375,\n      "opening_hours": null\n    },\n    {\n      "id": "way/241375900",\n      "kind": "school",\n      "name": "Школа № 1213. Корпус № 2",\n      "lat": 55.8424324,\n      "lon": 37.573629,\n      "opening_hours": null\n    },\n    {\n      "id": "way/243539757",\n      "kind": "school",\n      "name": "Школа 117, дошкольное отделение",\n      "lat": 55.674477,\n      "lon": 37.5335477,\n      "opening_hours": "07:00-19:00"\n    },\n    {\n      "id": "way/244039366",\n      "kind": "school",\n      "name": "Seymour House School",\n      "lat": 55.802443,\n      "lon": 37.504716,\n      "opening_hours": null\n    },\n    {\n      "id": "way/244467130",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7366467,\n      "lon": 37.6852351,\n      "opening_hours": null\n    },\n    {\n      "id": "way/244467372",\n      "kind": "kindergarten",\n      "name": "Детский сад №2731",\n      "lat": 55.6123525,\n      "lon": 37.5762322,\n      "opening_hours": null\n    },\n    {\n      "id": "way/246898126",\n      "kind": "school",\n      "name": "Гимназия №1597",\n      "lat": 55.7092981,\n      "lon": 37.8163047,\n      "opening_hours": null\n    },\n    {\n      "id": "way/246898127",\n      "kind": "kindergarten",\n      "name": "Детский сад №2717",\n      "lat": 55.7100719,\n      "lon": 37.8129302,\n      "opening_hours": null\n    },\n    {\n      "id": "way/246898128",\n      "kind": "kindergarten",\n      "name": "Детский сад №931",\n      "lat": 55.7093926,\n      "lon": 37.8143746,\n      "opening_hours": null\n    },\n    {\n      "id": "way/247696842",\n      "kind": "kindergarten",\n      "name": "Детский сад №576",\n      "lat": 55.7993281,\n      "lon": 37.7434797,\n      "opening_hours": null\n    },\n    {\n      "id": "way/247982892",\n      "kind": "school",\n      "name": "Гимназия №1583",\n      "lat": 55.8541414,\n      "lon": 37.5081683,\n      "opening_hours": null\n    },\n    {\n      "id": "way/248420121",\n      "kind": "school",\n      "name": "Школа №1096",\n      "lat": 55.8664031,\n      "lon": 37.698543,\n      "opening_hours": null\n    },\n    {\n      "id": "way/248885411",\n      "kind": "school",\n      "name": "Лицей №1581",\n      "lat": 55.7529689,\n      "lon": 37.6629017,\n      "opening_hours": null\n    },\n    {\n      "id": "way/249169573",\n      "kind": "school",\n      "name": "ГБОУ СОШ 480 Дошкольное отделение",\n      "lat": 55.7330202,\n      "lon": 37.6803981,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/250184978",\n      "kind": "kindergarten",\n      "name": "Детский сад №1 «Солнышко»",\n      "lat": 55.8906491,\n      "lon": 37.479794,\n      "opening_hours": null\n    },\n    {\n      "id": "way/251684295",\n      "kind": "kindergarten",\n      "name": "Средняя общеобразовательная школа №224 им. П.Н. Еремеева. Дошкольное отделение",\n      "lat": 55.8325349,\n      "lon": 37.4911004,\n      "opening_hours": null\n    },\n    {\n      "id": "way/252200000",\n      "kind": "kindergarten",\n      "name": "Детский сад №1246",\n      "lat": 55.7355913,\n      "lon": 37.5812554,\n      "opening_hours": null\n    },\n    {\n      "id": "way/255945625",\n      "kind": "kindergarten",\n      "name": "Первый Московский Образовательный Комплекс",\n      "lat": 55.8856463,\n      "lon": 37.6499498,\n      "opening_hours": null\n    },\n    {\n      "id": "way/256044990",\n      "kind": "school",\n      "name": "Школа-интернат МГУ",\n      "lat": 55.6944382,\n      "lon": 37.5251815,\n      "opening_hours": null\n    },\n    {\n      "id": "way/256445730",\n      "kind": "school",\n      "name": "Гимназия №1520 им. Капцовых",\n      "lat": 55.7610236,\n      "lon": 37.6049075,\n      "opening_hours": null\n    },\n    {\n      "id": "way/258254558",\n      "kind": "kindergarten",\n      "name": "Школа № 49. Дошкольный корпус",\n      "lat": 55.6467675,\n      "lon": 37.5392193,\n      "opening_hours": null\n    },\n    {\n      "id": "way/258753558",\n      "kind": "school",\n      "name": "Школа №491",\n      "lat": 55.6489192,\n      "lon": 37.7220664,\n      "opening_hours": null\n    },\n    {\n      "id": "way/259256790",\n      "kind": "kindergarten",\n      "name": "Школа №556. Корпус дошкольного образования",\n      "lat": 55.6178179,\n      "lon": 37.5825525,\n      "opening_hours": null\n    },\n    {\n      "id": "way/259550332",\n      "kind": "school",\n      "name": "Школа «УНА»",\n      "lat": 55.8780841,\n      "lon": 37.6494028,\n      "opening_hours": null\n    },\n    {\n      "id": "way/260112101",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.796754,\n      "lon": 37.6073217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/260177247",\n      "kind": "kindergarten",\n      "name": "Школа № 629. Корпус дошкольного образования №1",\n      "lat": 55.5911265,\n      "lon": 37.5936224,\n      "opening_hours": null\n    },\n    {\n      "id": "way/260452178",\n      "kind": "school",\n      "name": "Школа №237",\n      "lat": 55.7958019,\n      "lon": 37.6120739,\n      "opening_hours": null\n    },\n    {\n      "id": "way/260600593",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8031016,\n      "lon": 37.6136497,\n      "opening_hours": null\n    },\n    {\n      "id": "way/260608752",\n      "kind": "kindergarten",\n      "name": "Школа № 17. Дошкольный корпус № 2",\n      "lat": 55.6398143,\n      "lon": 37.5302679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/261158934",\n      "kind": "school",\n      "name": "Школа №9",\n      "lat": 55.6630993,\n      "lon": 37.5237737,\n      "opening_hours": null\n    },\n    {\n      "id": "way/261158935",\n      "kind": "school",\n      "name": "Школа №9",\n      "lat": 55.6634794,\n      "lon": 37.524874,\n      "opening_hours": null\n    },\n    {\n      "id": "way/261460294",\n      "kind": "school",\n      "name": "Специализированная детско-юношеская школа олимпийского резерва №28",\n      "lat": 55.7587815,\n      "lon": 37.561584,\n      "opening_hours": null\n    },\n    {\n      "id": "way/262982881",\n      "kind": "school",\n      "name": "Школа № 113. Корпус №1",\n      "lat": 55.6374416,\n      "lon": 37.5163211,\n      "opening_hours": null\n    },\n    {\n      "id": "way/263955306",\n      "kind": "kindergarten",\n      "name": "Школа № 14. Корпус дошкольного образования №4 «Цветик-семицветик»",\n      "lat": 55.6574404,\n      "lon": 37.4701631,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264149139",\n      "kind": "kindergarten",\n      "name": "Школа № 14. Корпус дошкольного образования №3 «Теремок»",\n      "lat": 55.6600184,\n      "lon": 37.4709156,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264154345",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение «Цветочный городок»",\n      "lat": 55.6318001,\n      "lon": 37.4755183,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264154346",\n      "kind": "school",\n      "name": "ЦО ОЦ \\"Солнечный ветер\\"",\n      "lat": 55.6319114,\n      "lon": 37.4770452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264155007",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение «Сказочный городок»",\n      "lat": 55.6331354,\n      "lon": 37.4725964,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264155008",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение «Белоснежка»",\n      "lat": 55.6336279,\n      "lon": 37.4738725,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264644427",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1623",\n      "lat": 55.6897571,\n      "lon": 37.4961502,\n      "opening_hours": null\n    },\n    {\n      "id": "way/264644428",\n      "kind": "school",\n      "name": "Школа № 1119",\n      "lat": 55.6891002,\n      "lon": 37.4974201,\n      "opening_hours": null\n    },\n    {\n      "id": "way/265442692",\n      "kind": "school",\n      "name": "ГБОУ Школа №825",\n      "lat": 55.7109869,\n      "lon": 37.7574233,\n      "opening_hours": null\n    },\n    {\n      "id": "way/267390971",\n      "kind": "school",\n      "name": "Школа на проспекте Вернадского. Корпус школьного образования 1",\n      "lat": 55.6535796,\n      "lon": 37.4836595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/267625741",\n      "kind": "kindergarten",\n      "name": "Детский сад №2209",\n      "lat": 55.6491877,\n      "lon": 37.740232,\n      "opening_hours": null\n    },\n    {\n      "id": "way/270445570",\n      "kind": "kindergarten",\n      "name": "Школа №967, дошкольное отделение №5",\n      "lat": 55.8820482,\n      "lon": 37.6432623,\n      "opening_hours": null\n    },\n    {\n      "id": "way/271091464",\n      "kind": "school",\n      "name": "Школа № 814",\n      "lat": 55.7053586,\n      "lon": 37.4661088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/272980100",\n      "kind": "kindergarten",\n      "name": "Детский сад №2282",\n      "lat": 55.6664515,\n      "lon": 37.4527006,\n      "opening_hours": null\n    },\n    {\n      "id": "way/272980137",\n      "kind": "kindergarten",\n      "name": "Детский сад №2252",\n      "lat": 55.6660049,\n      "lon": 37.4542323,\n      "opening_hours": null\n    },\n    {\n      "id": "way/272980138",\n      "kind": "school",\n      "name": "Школа №1741",\n      "lat": 55.6665912,\n      "lon": 37.4554356,\n      "opening_hours": null\n    },\n    {\n      "id": "way/274246811",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2061",\n      "lat": 55.7417004,\n      "lon": 37.4795672,\n      "opening_hours": null\n    },\n    {\n      "id": "way/275801691",\n      "kind": "school",\n      "name": "Школа №484",\n      "lat": 55.7060609,\n      "lon": 37.742303,\n      "opening_hours": null\n    },\n    {\n      "id": "way/276226416",\n      "kind": "kindergarten",\n      "name": "Кембриджская международная школа",\n      "lat": 55.8556729,\n      "lon": 37.4737931,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "way/276880867",\n      "kind": "school",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №1",\n      "lat": 55.8051041,\n      "lon": 37.8280267,\n      "opening_hours": null\n    },\n    {\n      "id": "way/276975541",\n      "kind": "kindergarten",\n      "name": "Школа №439 \\"Интеллект\\" (дошкольное отделение №4)",\n      "lat": 55.7389262,\n      "lon": 37.7382263,\n      "opening_hours": null\n    },\n    {\n      "id": "way/277070499",\n      "kind": "school",\n      "name": "Средняя общеобразовательная школа «Столичный - КИТ»",\n      "lat": 55.7173804,\n      "lon": 37.6146828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/277815333",\n      "kind": "school",\n      "name": "Гимназия № 1570. Дошкольное отделение",\n      "lat": 55.7866261,\n      "lon": 37.5759701,\n      "opening_hours": null\n    },\n    {\n      "id": "way/280429324",\n      "kind": "kindergarten",\n      "name": "Детский сад №2436",\n      "lat": 55.6689648,\n      "lon": 37.7553539,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282238821",\n      "kind": "school",\n      "name": "Школа #1493",\n      "lat": 55.6536248,\n      "lon": 37.7495045,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282497054",\n      "kind": "kindergarten",\n      "name": "Детский сад №1621",\n      "lat": 55.6216372,\n      "lon": 37.7009571,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282750312",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8723561,\n      "lon": 37.5609147,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282873404",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129  ДО \\"Терем-Теремок\\"",\n      "lat": 55.7062587,\n      "lon": 37.6780758,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282875256",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129  ДО \\"Солнышко\\"",\n      "lat": 55.7055103,\n      "lon": 37.678256,\n      "opening_hours": null\n    },\n    {\n      "id": "way/282885417",\n      "kind": "kindergarten",\n      "name": "Детский сад №48",\n      "lat": 55.6542956,\n      "lon": 37.6152244,\n      "opening_hours": null\n    },\n    {\n      "id": "way/283341336",\n      "kind": "school",\n      "name": "Школа №858",\n      "lat": 55.6158049,\n      "lon": 37.6021169,\n      "opening_hours": null\n    },\n    {\n      "id": "way/284376076",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6495378,\n      "lon": 37.711225,\n      "opening_hours": null\n    },\n    {\n      "id": "way/284385908",\n      "kind": "school",\n      "name": "Школа № 1582. Корпус школьного образования",\n      "lat": 55.617823,\n      "lon": 37.6007872,\n      "opening_hours": null\n    },\n    {\n      "id": "way/284401169",\n      "kind": "school",\n      "name": "НП ЦО \\"Лицей \\"Столичный\\"",\n      "lat": 55.6168761,\n      "lon": 37.588356,\n      "opening_hours": null\n    },\n    {\n      "id": "way/284423448",\n      "kind": "school",\n      "name": "ГБОУ Школа № 2129 ШО №2",\n      "lat": 55.7063838,\n      "lon": 37.6722458,\n      "opening_hours": null\n    },\n    {\n      "id": "way/285186415",\n      "kind": "kindergarten",\n      "name": "Школа № 2103. Корпус дошкольного образования №3",\n      "lat": 55.6087959,\n      "lon": 37.5159587,\n      "opening_hours": null\n    },\n    {\n      "id": "way/285186416",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa01986",\n      "lat": 55.6085806,\n      "lon": 37.5170265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/285794541",\n      "kind": "kindergarten",\n      "name": "Школа им. Н.М. Карамзина. Корпус дошкольного образования №7 «Александровский»",\n      "lat": 55.5909684,\n      "lon": 37.5478788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/285875625",\n      "kind": "kindergarten",\n      "name": "Школа № 2103. Корпус дошкольного образования №2",\n      "lat": 55.6051207,\n      "lon": 37.5136272,\n      "opening_hours": null\n    },\n    {\n      "id": "way/285875626",\n      "kind": "kindergarten",\n      "name": "Школа № 2103. Корпус дошкольного образования №7",\n      "lat": 55.6048991,\n      "lon": 37.5146559,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286082075",\n      "kind": "kindergarten",\n      "name": "Школа №851",\n      "lat": 55.6333488,\n      "lon": 37.6059369,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286482971",\n      "kind": "kindergarten",\n      "name": "Школа № 2103. Корпус дошкольного образования №6",\n      "lat": 55.602692,\n      "lon": 37.5187104,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286482972",\n      "kind": "school",\n      "name": "Школа № 2103. Корпус школьного образования №4",\n      "lat": 55.6029143,\n      "lon": 37.5176385,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286584497",\n      "kind": "school",\n      "name": "Школа № 2103. Корпус школьного образования №3",\n      "lat": 55.6048558,\n      "lon": 37.5175329,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286584498",\n      "kind": "school",\n      "name": "Школа № 2103. Корпус школьного образования №1",\n      "lat": 55.6068056,\n      "lon": 37.5169351,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286584499",\n      "kind": "school",\n      "name": "Школа № 2103. Корпус школьного образования №2",\n      "lat": 55.6058422,\n      "lon": 37.5172227,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286584500",\n      "kind": "school",\n      "name": "Школа № 2103. Корпус школьного образования №5",\n      "lat": 55.6072267,\n      "lon": 37.5205718,\n      "opening_hours": null\n    },\n    {\n      "id": "way/286584501",\n      "kind": "kindergarten",\n      "name": "Школа № 2103. Корпус дошкольного образования №1",\n      "lat": 55.6069813,\n      "lon": 37.5216692,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287005456",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6514576,\n      "lon": 37.7361533,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287005459",\n      "kind": "kindergarten",\n      "name": "Детский сад №1881",\n      "lat": 55.6529243,\n      "lon": 37.7367682,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287198704",\n      "kind": "kindergarten",\n      "name": "СПДО № 3 школы № 1784",\n      "lat": 55.7870238,\n      "lon": 37.5708603,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287304987",\n      "kind": "school",\n      "name": "Школа №1450 Олимп",\n      "lat": 55.6539415,\n      "lon": 37.6123343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287306189",\n      "kind": "kindergarten",\n      "name": "Школа № 1103. Корпус дошкольного образования №3",\n      "lat": 55.608345,\n      "lon": 37.5505637,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287306190",\n      "kind": "kindergarten",\n      "name": "Школа № 1103. Корпус дошкольного образования №2",\n      "lat": 55.6083003,\n      "lon": 37.5493128,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287495346",\n      "kind": "kindergarten",\n      "name": "ДОУ №1177",\n      "lat": 55.6809564,\n      "lon": 37.6534372,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287553004",\n      "kind": "kindergarten",\n      "name": "Школа № 1212. Корпус дошкольного образования №5",\n      "lat": 55.6001688,\n      "lon": 37.5117202,\n      "opening_hours": null\n    },\n    {\n      "id": "way/287553005",\n      "kind": "school",\n      "name": "Православная гимназия \\"Радонеж\\"",\n      "lat": 55.5991123,\n      "lon": 37.5135717,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288601028",\n      "kind": "school",\n      "name": "Школа \\"Выбор\\"",\n      "lat": 55.65794,\n      "lon": 37.6102343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288604118",\n      "kind": "school",\n      "name": "Школа № 1212. Корпус школьного образования №2",\n      "lat": 55.5993401,\n      "lon": 37.5175476,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288604121",\n      "kind": "school",\n      "name": "Школа № 1212. Корпус школьного образования №1",\n      "lat": 55.5981508,\n      "lon": 37.5170961,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288609539",\n      "kind": "kindergarten",\n      "name": "Школа № 1862. Корпус дошкольного образования №8",\n      "lat": 55.6504413,\n      "lon": 37.617549,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288609547",\n      "kind": "school",\n      "name": "Школа № 1862. Корпус школьного образования №10",\n      "lat": 55.6461253,\n      "lon": 37.6121285,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288820295",\n      "kind": "kindergarten",\n      "name": "Школа № 1212. Корпус дошкольного образования №8",\n      "lat": 55.6055779,\n      "lon": 37.5270247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288820296",\n      "kind": "kindergarten",\n      "name": "Школа № 1212. Корпус дошкольного образования №9",\n      "lat": 55.6058261,\n      "lon": 37.5258843,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288938254",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2514",\n      "lat": 55.6560092,\n      "lon": 37.5883948,\n      "opening_hours": null\n    },\n    {\n      "id": "way/288938260",\n      "kind": "school",\n      "name": "Школа № 2042",\n      "lat": 55.6567778,\n      "lon": 37.588673,\n      "opening_hours": null\n    },\n    {\n      "id": "way/289210857",\n      "kind": "kindergarten",\n      "name": "Детский сад №1343",\n      "lat": 55.7030274,\n      "lon": 37.7468863,\n      "opening_hours": null\n    },\n    {\n      "id": "way/289860985",\n      "kind": "school",\n      "name": "Школа № 1212. Корпус школьного образования №4",\n      "lat": 55.6040732,\n      "lon": 37.5286177,\n      "opening_hours": null\n    },\n    {\n      "id": "way/290495398",\n      "kind": "school",\n      "name": "Школа № 1212. Корпус школьного образования №3",\n      "lat": 55.6015108,\n      "lon": 37.5267612,\n      "opening_hours": null\n    },\n    {\n      "id": "way/290650733",\n      "kind": "kindergarten",\n      "name": "Школа № 1212. Корпус дошкольного образования №6",\n      "lat": 55.6017092,\n      "lon": 37.523109,\n      "opening_hours": null\n    },\n    {\n      "id": "way/290650734",\n      "kind": "kindergarten",\n      "name": "Школа № 1212. Корпус дошкольного образования №7",\n      "lat": 55.6014793,\n      "lon": 37.5241883,\n      "opening_hours": null\n    },\n    {\n      "id": "way/291091227",\n      "kind": "kindergarten",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 7.",\n      "lat": 55.6527581,\n      "lon": 37.5214677,\n      "opening_hours": null\n    },\n    {\n      "id": "way/291431905",\n      "kind": "school",\n      "name": "Школа №937",\n      "lat": 55.6192688,\n      "lon": 37.7003505,\n      "opening_hours": null\n    },\n    {\n      "id": "way/291677793",\n      "kind": "kindergarten",\n      "name": "Детский сад № 391",\n      "lat": 55.7805982,\n      "lon": 37.4734718,\n      "opening_hours": null\n    },\n    {\n      "id": "way/293518110",\n      "kind": "school",\n      "name": "Школа №556. Корпус школьного образования",\n      "lat": 55.6172075,\n      "lon": 37.5853168,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294746097",\n      "kind": "school",\n      "name": "ГБОУ Школа №1476, корпус №1",\n      "lat": 55.7771975,\n      "lon": 37.8315407,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294746098",\n      "kind": "school",\n      "name": "ГБОУ Школа №1476, корпус №2",\n      "lat": 55.7760923,\n      "lon": 37.8264817,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294767909",\n      "kind": "kindergarten",\n      "name": "Детский сад 1944",\n      "lat": 55.681542,\n      "lon": 37.5203513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294767910",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2186",\n      "lat": 55.6775543,\n      "lon": 37.5266007,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294767923",\n      "kind": "school",\n      "name": "Москвич",\n      "lat": 55.6780027,\n      "lon": 37.5272994,\n      "opening_hours": null\n    },\n    {\n      "id": "way/294767929",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6755472,\n      "lon": 37.5235789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/295165515",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 9",\n      "lat": 55.7786723,\n      "lon": 37.5154246,\n      "opening_hours": null\n    },\n    {\n      "id": "way/295308773",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Самсон\\"",\n      "lat": 55.646955,\n      "lon": 37.6118135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/296404036",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8227432,\n      "lon": 37.5133024,\n      "opening_hours": null\n    },\n    {\n      "id": "way/296470165",\n      "kind": "school",\n      "name": "Школа № 1862. Корпус школьного образования №5",\n      "lat": 55.6509066,\n      "lon": 37.6144295,\n      "opening_hours": null\n    },\n    {\n      "id": "way/296766987",\n      "kind": "school",\n      "name": "Школа №726",\n      "lat": 55.8178257,\n      "lon": 37.5071475,\n      "opening_hours": null\n    },\n    {\n      "id": "way/296899004",\n      "kind": "school",\n      "name": "Центр образования №1685",\n      "lat": 55.7488656,\n      "lon": 37.6510548,\n      "opening_hours": null\n    },\n    {\n      "id": "way/297179085",\n      "kind": "kindergarten",\n      "name": "Детский сад №1162",\n      "lat": 55.7500579,\n      "lon": 37.6780667,\n      "opening_hours": null\n    },\n    {\n      "id": "way/297419999",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7057586,\n      "lon": 37.7829519,\n      "opening_hours": null\n    },\n    {\n      "id": "way/297787530",\n      "kind": "kindergarten",\n      "name": "Школа № 224. Дошкольное отделение",\n      "lat": 55.8359911,\n      "lon": 37.4867076,\n      "opening_hours": null\n    },\n    {\n      "id": "way/299334055",\n      "kind": "school",\n      "name": "Гимназия №1583",\n      "lat": 55.8444754,\n      "lon": 37.4995778,\n      "opening_hours": null\n    },\n    {\n      "id": "way/299439198",\n      "kind": "kindergarten",\n      "name": "Детский сад №2",\n      "lat": 55.8435487,\n      "lon": 37.4932647,\n      "opening_hours": null\n    },\n    {\n      "id": "way/299618941",\n      "kind": "kindergarten",\n      "name": "Школа №1282 \\"Сокольники\\" (дошкольное отделение 6)",\n      "lat": 55.7907656,\n      "lon": 37.6631069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/299959836",\n      "kind": "kindergarten",\n      "name": "Детский сад 1432",\n      "lat": 55.6804001,\n      "lon": 37.5207961,\n      "opening_hours": null\n    },\n    {\n      "id": "way/300862782",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6785969,\n      "lon": 37.5165494,\n      "opening_hours": null\n    },\n    {\n      "id": "way/301225747",\n      "kind": "school",\n      "name": "Технорама на Юго-Востоке",\n      "lat": 55.714684,\n      "lon": 37.7804105,\n      "opening_hours": null\n    },\n    {\n      "id": "way/301257076",\n      "kind": "kindergarten",\n      "name": "Детский сад №436",\n      "lat": 55.7140392,\n      "lon": 37.7743356,\n      "opening_hours": null\n    },\n    {\n      "id": "way/301261065",\n      "kind": "kindergarten",\n      "name": "Детский сад №2608",\n      "lat": 55.714085,\n      "lon": 37.7736867,\n      "opening_hours": null\n    },\n    {\n      "id": "way/301827317",\n      "kind": "kindergarten",\n      "name": "Школа № 1315. Дошкольный корпус",\n      "lat": 55.8488197,\n      "lon": 37.482704,\n      "opening_hours": null\n    },\n    {\n      "id": "way/302728535",\n      "kind": "school",\n      "name": "Кадетская школа №1783",\n      "lat": 55.8174951,\n      "lon": 37.627102,\n      "opening_hours": null\n    },\n    {\n      "id": "way/302731045",\n      "kind": "kindergarten",\n      "name": "ГБОУ Многопрофильная школа № 1220 Дошкольное отделение № 7",\n      "lat": 55.8164639,\n      "lon": 37.6280538,\n      "opening_hours": null\n    },\n    {\n      "id": "way/302731046",\n      "kind": "school",\n      "name": "Гимназия №1518",\n      "lat": 55.8166099,\n      "lon": 37.6294587,\n      "opening_hours": null\n    },\n    {\n      "id": "way/302874395",\n      "kind": "kindergarten",\n      "name": "Школа № 1220. Дошкольное отделение 7",\n      "lat": 55.8155766,\n      "lon": 37.6260994,\n      "opening_hours": null\n    },\n    {\n      "id": "way/304423695",\n      "kind": "school",\n      "name": "Школа №7",\n      "lat": 55.6833967,\n      "lon": 37.5241144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/304423701",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад №1620",\n      "lat": 55.6829149,\n      "lon": 37.5246628,\n      "opening_hours": null\n    },\n    {\n      "id": "way/304943996",\n      "kind": "school",\n      "name": "Школа №1264",\n      "lat": 55.6749372,\n      "lon": 37.5288398,\n      "opening_hours": null\n    },\n    {\n      "id": "way/305509989",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1311",\n      "lat": 55.7874154,\n      "lon": 37.4621433,\n      "opening_hours": null\n    },\n    {\n      "id": "way/305512014",\n      "kind": "kindergarten",\n      "name": "Школа № 138. Дошкольный корпус № 9",\n      "lat": 55.7873837,\n      "lon": 37.4706843,\n      "opening_hours": null\n    },\n    {\n      "id": "way/305671071",\n      "kind": "school",\n      "name": "Курчатовская школа \\"Школа-интернат №101 (для слабослышащих)\\"",\n      "lat": 55.7878736,\n      "lon": 37.4502528,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306230576",\n      "kind": "school",\n      "name": "Школа №547",\n      "lat": 55.7090957,\n      "lon": 37.619606,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306239412",\n      "kind": "kindergarten",\n      "name": "Детский сад ЦРР №208",\n      "lat": 55.710361,\n      "lon": 37.61135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306239414",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Роще\\"",\n      "lat": 55.7104506,\n      "lon": 37.6125525,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306457185",\n      "kind": "school",\n      "name": "Школа № 14. Корпус школьного образования №1",\n      "lat": 55.662527,\n      "lon": 37.4657222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306457186",\n      "kind": "school",\n      "name": "Школа № 14. Корпус школьного образования №2",\n      "lat": 55.6588522,\n      "lon": 37.4673021,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306467403",\n      "kind": "school",\n      "name": "Ломоносовская школа",\n      "lat": 55.7144431,\n      "lon": 37.4559731,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306467404",\n      "kind": "school",\n      "name": "Ломоносовская школа",\n      "lat": 55.7133511,\n      "lon": 37.4564043,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306467405",\n      "kind": "school",\n      "name": "Школа №97",\n      "lat": 55.7122262,\n      "lon": 37.4562088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/306577777",\n      "kind": "kindergarten",\n      "name": "Детский сад №407",\n      "lat": 55.6614087,\n      "lon": 37.5182497,\n      "opening_hours": null\n    },\n    {\n      "id": "way/307231038",\n      "kind": "school",\n      "name": "Классическое образование",\n      "lat": 55.7184322,\n      "lon": 37.4565557,\n      "opening_hours": null\n    },\n    {\n      "id": "way/307912086",\n      "kind": "kindergarten",\n      "name": "Детский сад №1708",\n      "lat": 55.6799912,\n      "lon": 37.5371016,\n      "opening_hours": null\n    },\n    {\n      "id": "way/307912089",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6744783,\n      "lon": 37.5335792,\n      "opening_hours": null\n    },\n    {\n      "id": "way/307912091",\n      "kind": "school",\n      "name": "English Playschool Moscow",\n      "lat": 55.6790374,\n      "lon": 37.5366821,\n      "opening_hours": null\n    },\n    {\n      "id": "way/309031590",\n      "kind": "school",\n      "name": "Школа №830 корпус №1",\n      "lat": 55.8330794,\n      "lon": 37.4596075,\n      "opening_hours": null\n    },\n    {\n      "id": "way/309031610",\n      "kind": "kindergarten",\n      "name": "Школа №830 дошкольное отделение Росток",\n      "lat": 55.8335138,\n      "lon": 37.4582196,\n      "opening_hours": null\n    },\n    {\n      "id": "way/309413338",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7661583,\n      "lon": 37.7295241,\n      "opening_hours": null\n    },\n    {\n      "id": "way/310484266",\n      "kind": "kindergarten",\n      "name": "Школа № 1989. Дошкольное отделение № 1",\n      "lat": 55.6635856,\n      "lon": 37.7780625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/311745513",\n      "kind": "school",\n      "name": "Лицей № 1547",\n      "lat": 55.6623122,\n      "lon": 37.7768224,\n      "opening_hours": null\n    },\n    {\n      "id": "way/311874351",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6526951,\n      "lon": 37.7160265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/313374880",\n      "kind": "kindergarten",\n      "name": "Детский сад №1798 \\"Марьино\\"",\n      "lat": 55.6497054,\n      "lon": 37.7328339,\n      "opening_hours": null\n    },\n    {\n      "id": "way/313678454",\n      "kind": "school",\n      "name": "Школа №1532. Корпус школьного образования №02",\n      "lat": 55.6178991,\n      "lon": 37.493707,\n      "opening_hours": null\n    },\n    {\n      "id": "way/313972931",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7435428,\n      "lon": 37.8014098,\n      "opening_hours": null\n    },\n    {\n      "id": "way/314426206",\n      "kind": "school",\n      "name": "Школа № 1103. Корпус школьного образования №6",\n      "lat": 55.6062815,\n      "lon": 37.5498466,\n      "opening_hours": null\n    },\n    {\n      "id": "way/314889731",\n      "kind": "kindergarten",\n      "name": "Школа им. Н.М. Карамзина. Корпус дошкольного образования №4 «Софийский»",\n      "lat": 55.5948356,\n      "lon": 37.5376778,\n      "opening_hours": null\n    },\n    {\n      "id": "way/314889732",\n      "kind": "kindergarten",\n      "name": "Школа им. Н.М. Карамзина. Корпус дошкольного образования №5 «Екатерининский»",\n      "lat": 55.5950594,\n      "lon": 37.5365957,\n      "opening_hours": null\n    },\n    {\n      "id": "way/315006942",\n      "kind": "school",\n      "name": "Британская Международная школа",\n      "lat": 55.5935377,\n      "lon": 37.5437821,\n      "opening_hours": null\n    },\n    {\n      "id": "way/316630443",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1661",\n      "lat": 55.6841287,\n      "lon": 37.5632973,\n      "opening_hours": null\n    },\n    {\n      "id": "way/316835898",\n      "kind": "kindergarten",\n      "name": "Детский сад №2241",\n      "lat": 55.6924597,\n      "lon": 37.6103943,\n      "opening_hours": null\n    },\n    {\n      "id": "way/317239775",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №10",\n      "lat": 55.5990282,\n      "lon": 37.6098207,\n      "opening_hours": null\n    },\n    {\n      "id": "way/317584503",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 507, Гномики (здание №10)",\n      "lat": 55.6776182,\n      "lon": 37.6703317,\n      "opening_hours": null\n    },\n    {\n      "id": "way/317792794",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №5",\n      "lat": 55.5995573,\n      "lon": 37.6192564,\n      "opening_hours": null\n    },\n    {\n      "id": "way/317792796",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.5996094,\n      "lon": 37.6170891,\n      "opening_hours": null\n    },\n    {\n      "id": "way/318098302",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №7",\n      "lat": 55.5982049,\n      "lon": 37.6149972,\n      "opening_hours": null\n    },\n    {\n      "id": "way/318098304",\n      "kind": "kindergarten",\n      "name": "Детский сад №725",\n      "lat": 55.5979473,\n      "lon": 37.617163,\n      "opening_hours": null\n    },\n    {\n      "id": "way/318866391",\n      "kind": "kindergarten",\n      "name": "Школа №654 имени А. Д. Фридмана. Дошкольное отделение №2",\n      "lat": 55.6987433,\n      "lon": 37.7455939,\n      "opening_hours": null\n    },\n    {\n      "id": "way/318866392",\n      "kind": "school",\n      "name": "Школа №355",\n      "lat": 55.6951349,\n      "lon": 37.8230844,\n      "opening_hours": null\n    },\n    {\n      "id": "way/319196588",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8104228,\n      "lon": 37.5716682,\n      "opening_hours": null\n    },\n    {\n      "id": "way/319196606",\n      "kind": "school",\n      "name": "Школа № 1454",\n      "lat": 55.8134542,\n      "lon": 37.5688665,\n      "opening_hours": null\n    },\n    {\n      "id": "way/319902984",\n      "kind": "school",\n      "name": "Школа № 843",\n      "lat": 55.6711324,\n      "lon": 37.4579789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/320747994",\n      "kind": "school",\n      "name": "Школа №672",\n      "lat": 55.7495847,\n      "lon": 37.7822088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/321358664",\n      "kind": "school",\n      "name": "Гимназия №1798 \\"Феникс\\"",\n      "lat": 55.752896,\n      "lon": 37.8139613,\n      "opening_hours": null\n    },\n    {\n      "id": "way/322896806",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6023601,\n      "lon": 37.5349225,\n      "opening_hours": null\n    },\n    {\n      "id": "way/324740801",\n      "kind": "kindergarten",\n      "name": "Школа № 1248. Здание дошкольных групп 6",\n      "lat": 55.7226459,\n      "lon": 37.4710084,\n      "opening_hours": null\n    },\n    {\n      "id": "way/325925347",\n      "kind": "kindergarten",\n      "name": "Детский сад № 489",\n      "lat": 55.6492145,\n      "lon": 37.5817353,\n      "opening_hours": null\n    },\n    {\n      "id": "way/325989403",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В. А. Молодцова, дополнительное образование",\n      "lat": 55.8761052,\n      "lon": 37.6320678,\n      "opening_hours": null\n    },\n    {\n      "id": "way/326986434",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6869049,\n      "lon": 37.5799478,\n      "opening_hours": null\n    },\n    {\n      "id": "way/326986435",\n      "kind": "school",\n      "name": "Гимназия №1534 Школьное отделение №4",\n      "lat": 55.6877374,\n      "lon": 37.5812575,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329291176",\n      "kind": "school",\n      "name": "Гимназия №1551 корпус №1 (полное общее и гимназическое)",\n      "lat": 55.8467929,\n      "lon": 37.4564324,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329291638",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №6 (дошкольное отделение)",\n      "lat": 55.8479145,\n      "lon": 37.4564773,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329353098",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №7 (дошкольное отделение)",\n      "lat": 55.8449866,\n      "lon": 37.4569229,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329535556",\n      "kind": "school",\n      "name": "Школа № 1561. Корпус №3",\n      "lat": 55.6002397,\n      "lon": 37.533737,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329630118",\n      "kind": "school",\n      "name": "Школа № 1561. Корпус №1",\n      "lat": 55.6007025,\n      "lon": 37.5374328,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329630119",\n      "kind": "kindergarten",\n      "name": "Школа № 1561. Дошкольное отделение корпус №7",\n      "lat": 55.599153,\n      "lon": 37.5352235,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329630120",\n      "kind": "kindergarten",\n      "name": "Школа № 1561. Дошкольное отделение корпус №9",\n      "lat": 55.5989161,\n      "lon": 37.5363265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/329889832",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6811381,\n      "lon": 37.5649205,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330589924",\n      "kind": "kindergarten",\n      "name": "Детский сад №1614",\n      "lat": 55.6675506,\n      "lon": 37.7457501,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330589925",\n      "kind": "kindergarten",\n      "name": "Центр образования №1877 \\"Люблино\\" - дошкольное отделение",\n      "lat": 55.6687898,\n      "lon": 37.7479076,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330589926",\n      "kind": "school",\n      "name": "Центр образования №1877 \\"Люблино\\"",\n      "lat": 55.6685781,\n      "lon": 37.7461851,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330781480",\n      "kind": "school",\n      "name": "Школа №728",\n      "lat": 55.629689,\n      "lon": 37.6075114,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330782053",\n      "kind": "kindergarten",\n      "name": "Детский сад №668",\n      "lat": 55.6294453,\n      "lon": 37.6101111,\n      "opening_hours": null\n    },\n    {\n      "id": "way/330783273",\n      "kind": "school",\n      "name": "Школа №851",\n      "lat": 55.6279349,\n      "lon": 37.6062266,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331403705",\n      "kind": "school",\n      "name": "Лицей №138, отделение начальная школа",\n      "lat": 55.782451,\n      "lon": 37.4665567,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331445723",\n      "kind": "kindergarten",\n      "name": "Детский сад №1375",\n      "lat": 55.673436,\n      "lon": 37.7693615,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331445725",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6714522,\n      "lon": 37.7709225,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331445726",\n      "kind": "kindergarten",\n      "name": "Лицей № 2010. Дошкольное отделение Радуга",\n      "lat": 55.6717546,\n      "lon": 37.7631003,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331445728",\n      "kind": "school",\n      "name": "Центр образования №2010",\n      "lat": 55.6725204,\n      "lon": 37.7641608,\n      "opening_hours": null\n    },\n    {\n      "id": "way/331445729",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6694269,\n      "lon": 37.7604846,\n      "opening_hours": null\n    },\n    {\n      "id": "way/332105703",\n      "kind": "school",\n      "name": "Гимназия №1576",\n      "lat": 55.8229834,\n      "lon": 37.5291763,\n      "opening_hours": null\n    },\n    {\n      "id": "way/332122466",\n      "kind": "school",\n      "name": "Школа № 1288, корпус № 3",\n      "lat": 55.7735963,\n      "lon": 37.542007,\n      "opening_hours": null\n    },\n    {\n      "id": "way/332446075",\n      "kind": "kindergarten",\n      "name": "Школа № 1561. Дошкольное отделение корпус №8",\n      "lat": 55.6029686,\n      "lon": 37.5393923,\n      "opening_hours": null\n    },\n    {\n      "id": "way/332446076",\n      "kind": "school",\n      "name": "Школа № 1561. Начальная школа - корпус №10",\n      "lat": 55.6031895,\n      "lon": 37.5383104,\n      "opening_hours": null\n    },\n    {\n      "id": "way/333465880",\n      "kind": "kindergarten",\n      "name": "Детский сад №2456",\n      "lat": 55.643294,\n      "lon": 37.5987303,\n      "opening_hours": null\n    },\n    {\n      "id": "way/333465883",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2456",\n      "lat": 55.6444922,\n      "lon": 37.5966251,\n      "opening_hours": null\n    },\n    {\n      "id": "way/333533055",\n      "kind": "school",\n      "name": "Школа № 2126 \\"Перово\\"",\n      "lat": 55.741556,\n      "lon": 37.7625178,\n      "opening_hours": null\n    },\n    {\n      "id": "way/334253493",\n      "kind": "school",\n      "name": "Школа №958",\n      "lat": 55.8687633,\n      "lon": 37.6091067,\n      "opening_hours": null\n    },\n    {\n      "id": "way/334458773",\n      "kind": "school",\n      "name": "ГБУ ЦССВ \\"Наш дом\\"",\n      "lat": 55.7514212,\n      "lon": 37.5036492,\n      "opening_hours": null\n    },\n    {\n      "id": "way/334658366",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6783464,\n      "lon": 37.568421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/335073907",\n      "kind": "kindergarten",\n      "name": "Школа № 2115. Корпус дошкольного образования № 5",\n      "lat": 55.6687127,\n      "lon": 37.5693168,\n      "opening_hours": null\n    },\n    {\n      "id": "way/335195464",\n      "kind": "school",\n      "name": "Школа № 15. Корпус школьного образования № 4",\n      "lat": 55.661952,\n      "lon": 37.5704672,\n      "opening_hours": null\n    },\n    {\n      "id": "way/335317974",\n      "kind": "kindergarten",\n      "name": "Детский сад №574 ФГУП ГосНИИАС",\n      "lat": 55.7958354,\n      "lon": 37.5232028,\n      "opening_hours": "Mo-Fr 07:30-19:30"\n    },\n    {\n      "id": "way/335598252",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 6",\n      "lat": 55.6640312,\n      "lon": 37.5688032,\n      "opening_hours": null\n    },\n    {\n      "id": "way/336472571",\n      "kind": "school",\n      "name": "Школа № 1329. Учебный корпус № 2",\n      "lat": 55.6735197,\n      "lon": 37.4731658,\n      "opening_hours": null\n    },\n    {\n      "id": "way/336720753",\n      "kind": "school",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ШП-2)",\n      "lat": 55.6725105,\n      "lon": 37.7701674,\n      "opening_hours": null\n    },\n    {\n      "id": "way/336883714",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 5",\n      "lat": 55.6617853,\n      "lon": 37.5678305,\n      "opening_hours": null\n    },\n    {\n      "id": "way/336890069",\n      "kind": "kindergarten",\n      "name": "Школа № 2115. Корпус дошкольного образования № 8",\n      "lat": 55.66635,\n      "lon": 37.568822,\n      "opening_hours": null\n    },\n    {\n      "id": "way/337152319",\n      "kind": "kindergarten",\n      "name": "Детский сад №2419",\n      "lat": 55.7554655,\n      "lon": 37.8170242,\n      "opening_hours": null\n    },\n    {\n      "id": "way/338052419",\n      "kind": "kindergarten",\n      "name": "Школа № 113. Корпус дошкольного образования №7",\n      "lat": 55.6454751,\n      "lon": 37.5118452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/339123293",\n      "kind": "kindergarten",\n      "name": "Школа № 2115. Корпус дошкольного образования № 4",\n      "lat": 55.6713752,\n      "lon": 37.5733048,\n      "opening_hours": null\n    },\n    {\n      "id": "way/339923948",\n      "kind": "school",\n      "name": "English International School",\n      "lat": 55.7491892,\n      "lon": 37.8224728,\n      "opening_hours": null\n    },\n    {\n      "id": "way/340910737",\n      "kind": "kindergarten",\n      "name": "Детский сад №1290",\n      "lat": 55.7481424,\n      "lon": 37.8244796,\n      "opening_hours": null\n    },\n    {\n      "id": "way/343922114",\n      "kind": "school",\n      "name": "Школа №54",\n      "lat": 55.7237568,\n      "lon": 37.5703509,\n      "opening_hours": null\n    },\n    {\n      "id": "way/343924854",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7234189,\n      "lon": 37.5590935,\n      "opening_hours": null\n    },\n    {\n      "id": "way/345118620",\n      "kind": "school",\n      "name": "Центр образования \\"Личность\\"",\n      "lat": 55.7193582,\n      "lon": 37.7892985,\n      "opening_hours": null\n    },\n    {\n      "id": "way/345628413",\n      "kind": "school",\n      "name": "Лицей №138, отделение средняя школа",\n      "lat": 55.7810707,\n      "lon": 37.4673229,\n      "opening_hours": null\n    },\n    {\n      "id": "way/349929585",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №152 СП №2",\n      "lat": 55.8090438,\n      "lon": 37.5232553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/350090313",\n      "kind": "school",\n      "name": "Лицей №1575",\n      "lat": 55.8013012,\n      "lon": 37.5469938,\n      "opening_hours": null\n    },\n    {\n      "id": "way/350097515",\n      "kind": "kindergarten",\n      "name": "Инженерно-техническая школа имени дважды Героя Советского Союза П.Р. Поповича",\n      "lat": 55.8076753,\n      "lon": 37.5367688,\n      "opening_hours": null\n    },\n    {\n      "id": "way/350100413",\n      "kind": "kindergarten",\n      "name": "ГБОУ г. Москвы \\"Инженерно-техническая школа имени дважды Героя Советского Союза П.Р. Поповича\\"",\n      "lat": 55.8096465,\n      "lon": 37.536859,\n      "opening_hours": null\n    },\n    {\n      "id": "way/350105151",\n      "kind": "school",\n      "name": "Школа №152",\n      "lat": 55.7958955,\n      "lon": 37.5452135,\n      "opening_hours": null\n    },\n    {\n      "id": "way/350320195",\n      "kind": "kindergarten",\n      "name": "Школа №1420. Корпус №11",\n      "lat": 55.6991996,\n      "lon": 37.8229847,\n      "opening_hours": null\n    },\n    {\n      "id": "way/352135092",\n      "kind": "school",\n      "name": "Школа № 15. Корпус школьного образования № 3",\n      "lat": 55.6604236,\n      "lon": 37.5693012,\n      "opening_hours": null\n    },\n    {\n      "id": "way/352807461",\n      "kind": "kindergarten",\n      "name": "Детский сад № 421",\n      "lat": 55.6606324,\n      "lon": 37.5708153,\n      "opening_hours": null\n    },\n    {\n      "id": "way/353049778",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 4",\n      "lat": 55.6620471,\n      "lon": 37.573247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/353198790",\n      "kind": "kindergarten",\n      "name": "Детский сад №797",\n      "lat": 55.7313037,\n      "lon": 37.5350636,\n      "opening_hours": null\n    },\n    {\n      "id": "way/353198792",\n      "kind": "school",\n      "name": "Школа № 12",\n      "lat": 55.7306206,\n      "lon": 37.5327473,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355262402",\n      "kind": "kindergarten",\n      "name": "Школа №875. Корпус дошкольного образования «Надежда»",\n      "lat": 55.6581095,\n      "lon": 37.4968887,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355321688",\n      "kind": "school",\n      "name": "Центр образования № 1434",\n      "lat": 55.6907361,\n      "lon": 37.5003688,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355321689",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1565",\n      "lat": 55.6905524,\n      "lon": 37.4976442,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355321690",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1544",\n      "lat": 55.6913973,\n      "lon": 37.49915,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355466540",\n      "kind": "kindergarten",\n      "name": "Детский сад №226",\n      "lat": 55.8083912,\n      "lon": 37.5256277,\n      "opening_hours": null\n    },\n    {\n      "id": "way/355662159",\n      "kind": "school",\n      "name": "Школа им. Хинксона",\n      "lat": 55.6748036,\n      "lon": 37.5765673,\n      "opening_hours": null\n    },\n    {\n      "id": "way/356023465",\n      "kind": "kindergarten",\n      "name": "Школа № 2115. Корпус дошкольного образования № 7",\n      "lat": 55.6664651,\n      "lon": 37.5722384,\n      "opening_hours": null\n    },\n    {\n      "id": "way/358194039",\n      "kind": "school",\n      "name": "Школа №1352 (корпус на Щёлковском шс., 79А)",\n      "lat": 55.8130511,\n      "lon": 37.8077561,\n      "opening_hours": null\n    },\n    {\n      "id": "way/358616273",\n      "kind": "school",\n      "name": "Гимназия №1514",\n      "lat": 55.6850391,\n      "lon": 37.5265823,\n      "opening_hours": null\n    },\n    {\n      "id": "way/358616274",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад №1851",\n      "lat": 55.6858704,\n      "lon": 37.5272578,\n      "opening_hours": null\n    },\n    {\n      "id": "way/358807892",\n      "kind": "school",\n      "name": "Романовская школа",\n      "lat": 55.7727287,\n      "lon": 37.5779719,\n      "opening_hours": null\n    },\n    {\n      "id": "way/359730497",\n      "kind": "school",\n      "name": "Школа № 49",\n      "lat": 55.6472878,\n      "lon": 37.5369015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/360180382",\n      "kind": "kindergarten",\n      "name": "Центр образования и спорта \\"Москва-98\\". Дошкольное отделение № 3",\n      "lat": 55.6838348,\n      "lon": 37.4840979,\n      "opening_hours": null\n    },\n    {\n      "id": "way/360180383",\n      "kind": "kindergarten",\n      "name": "Центр образования и спорта \\"Москва-98\\". Дошкольное отделение № 2",\n      "lat": 55.685457,\n      "lon": 37.4869063,\n      "opening_hours": null\n    },\n    {\n      "id": "way/360180385",\n      "kind": "school",\n      "name": "Центр образования и спорта \\"Москва-98\\". Школа № 3",\n      "lat": 55.6847896,\n      "lon": 37.4852296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/361018891",\n      "kind": "school",\n      "name": "Школа №494 им. А. Н. Рожкова",\n      "lat": 55.704784,\n      "lon": 37.6557062,\n      "opening_hours": null\n    },\n    {\n      "id": "way/361313197",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6658724,\n      "lon": 37.5549265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/362529929",\n      "kind": "kindergarten",\n      "name": "Set International School",\n      "lat": 55.7908035,\n      "lon": 37.7871961,\n      "opening_hours": null\n    },\n    {\n      "id": "way/362924386",\n      "kind": "school",\n      "name": "Средняя школа №723",\n      "lat": 55.7917401,\n      "lon": 37.8128954,\n      "opening_hours": null\n    },\n    {\n      "id": "way/363405576",\n      "kind": "school",\n      "name": "Центр образования № 628",\n      "lat": 55.6681172,\n      "lon": 37.6083376,\n      "opening_hours": null\n    },\n    {\n      "id": "way/363467163",\n      "kind": "school",\n      "name": "Школа №423",\n      "lat": 55.7586268,\n      "lon": 37.7975102,\n      "opening_hours": null\n    },\n    {\n      "id": "way/363470439",\n      "kind": "kindergarten",\n      "name": "Школа № 243. Дошкольные отделения",\n      "lat": 55.7632594,\n      "lon": 37.7897795,\n      "opening_hours": null\n    },\n    {\n      "id": "way/363943988",\n      "kind": "kindergarten",\n      "name": "Гимназия 1570 Дошкольное отделение",\n      "lat": 55.790683,\n      "lon": 37.5834186,\n      "opening_hours": null\n    },\n    {\n      "id": "way/364436597",\n      "kind": "kindergarten",\n      "name": "Детский сад №60",\n      "lat": 55.6677193,\n      "lon": 37.6173746,\n      "opening_hours": null\n    },\n    {\n      "id": "way/364719268",\n      "kind": "school",\n      "name": "Школа №1576",\n      "lat": 55.838709,\n      "lon": 37.5340906,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365268317",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №5",\n      "lat": 55.8533787,\n      "lon": 37.6884056,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365268318",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №4",\n      "lat": 55.8525706,\n      "lon": 37.6878097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365268319",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Гимназия №1577\\" Подразделение №3",\n      "lat": 55.8517964,\n      "lon": 37.6864865,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365355137",\n      "kind": "kindergarten",\n      "name": "Детский сад 1489",\n      "lat": 55.6795738,\n      "lon": 37.5166667,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365490472",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1431",\n      "lat": 55.6773901,\n      "lon": 37.5237762,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365533380",\n      "kind": "school",\n      "name": "Школа №1352 (корпус на Байкальской, 44)",\n      "lat": 55.8152539,\n      "lon": 37.8147714,\n      "opening_hours": null\n    },\n    {\n      "id": "way/365668976",\n      "kind": "school",\n      "name": "ДШИ им. Ф.Шуберта",\n      "lat": 55.6773707,\n      "lon": 37.4956891,\n      "opening_hours": null\n    },\n    {\n      "id": "way/366295826",\n      "kind": "school",\n      "name": "Школа №1253 (начальные классы)",\n      "lat": 55.7325984,\n      "lon": 37.5904657,\n      "opening_hours": null\n    },\n    {\n      "id": "way/366476184",\n      "kind": "school",\n      "name": "Школа № 7 филиал № 2",\n      "lat": 55.6783242,\n      "lon": 37.5222297,\n      "opening_hours": null\n    },\n    {\n      "id": "way/366476185",\n      "kind": "school",\n      "name": "Олимпия",\n      "lat": 55.6775725,\n      "lon": 37.5215053,\n      "opening_hours": null\n    },\n    {\n      "id": "way/366561360",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2466",\n      "lat": 55.6642355,\n      "lon": 37.7753399,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/366786670",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 7 «Радуга»",\n      "lat": 55.7655704,\n      "lon": 37.5736484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/367879924",\n      "kind": "school",\n      "name": "Школа №881",\n      "lat": 55.6018094,\n      "lon": 37.5839302,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368059893",\n      "kind": "kindergarten",\n      "name": "ДОУ №638",\n      "lat": 55.6797655,\n      "lon": 37.6610759,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368261204",\n      "kind": "school",\n      "name": "Школа № 1248. Школьное здание 2",\n      "lat": 55.7215291,\n      "lon": 37.4678696,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368262333",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7250567,\n      "lon": 37.4648214,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368267185",\n      "kind": "school",\n      "name": "Школа №97",\n      "lat": 55.7204483,\n      "lon": 37.460397,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368275200",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6318409,\n      "lon": 37.5116148,\n      "opening_hours": null\n    },\n    {\n      "id": "way/368275202",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6313366,\n      "lon": 37.5131849,\n      "opening_hours": null\n    },\n    {\n      "id": "way/369050718",\n      "kind": "school",\n      "name": "НОУ \\"Британская международная школа\\"",\n      "lat": 55.6721614,\n      "lon": 37.5742829,\n      "opening_hours": null\n    },\n    {\n      "id": "way/369872117",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.686673,\n      "lon": 37.5806099,\n      "opening_hours": null\n    },\n    {\n      "id": "way/370556590",\n      "kind": "kindergarten",\n      "name": "Школа №51. Дошкольные группы",\n      "lat": 55.643659,\n      "lon": 37.4873683,\n      "opening_hours": null\n    },\n    {\n      "id": "way/370556591",\n      "kind": "kindergarten",\n      "name": "Школа №51. Дошкольные группы",\n      "lat": 55.6427025,\n      "lon": 37.4854944,\n      "opening_hours": null\n    },\n    {\n      "id": "way/371851060",\n      "kind": "kindergarten",\n      "name": "Школа №1352 (дошкольная группа на Уральской, 6)",\n      "lat": 55.8125084,\n      "lon": 37.8006892,\n      "opening_hours": null\n    },\n    {\n      "id": "way/371930812",\n      "kind": "school",\n      "name": "Вальдорфская семейная инициатива «Школа под солнцем»",\n      "lat": 55.6286849,\n      "lon": 37.4534843,\n      "opening_hours": null\n    },\n    {\n      "id": "way/372219915",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Единство»",\n      "lat": 55.6375279,\n      "lon": 37.5930864,\n      "opening_hours": null\n    },\n    {\n      "id": "way/372687745",\n      "kind": "school",\n      "name": "Школа № 1228. Корпус № 2",\n      "lat": 55.7539392,\n      "lon": 37.7113082,\n      "opening_hours": null\n    },\n    {\n      "id": "way/373222414",\n      "kind": "kindergarten",\n      "name": "Детский сад №288",\n      "lat": 55.7396176,\n      "lon": 37.6828883,\n      "opening_hours": null\n    },\n    {\n      "id": "way/373614960",\n      "kind": "kindergarten",\n      "name": "Гимназия № 625 Дошкольное отделение Созвездие",\n      "lat": 55.6908704,\n      "lon": 37.5925088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/373734133",\n      "kind": "kindergarten",\n      "name": "Детский сад № 775",\n      "lat": 55.6851542,\n      "lon": 37.58927,\n      "opening_hours": null\n    },\n    {\n      "id": "way/373783441",\n      "kind": "school",\n      "name": "Школа №962",\n      "lat": 55.8596609,\n      "lon": 37.5934956,\n      "opening_hours": null\n    },\n    {\n      "id": "way/373873844",\n      "kind": "school",\n      "name": "Школа №1516 (Основной корпус)",\n      "lat": 55.8166613,\n      "lon": 37.831409,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374145748",\n      "kind": "kindergarten",\n      "name": "Школа №1078 (Модуль №4)",\n      "lat": 55.8187335,\n      "lon": 37.8060185,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374145750",\n      "kind": "school",\n      "name": "Школа №1078 (Модуль №1)",\n      "lat": 55.8186512,\n      "lon": 37.8079475,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374185846",\n      "kind": "school",\n      "name": "Школа №49. Школьный корпус",\n      "lat": 55.6436293,\n      "lon": 37.5340588,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374260412",\n      "kind": "kindergarten",\n      "name": "Детский сад № 737",\n      "lat": 55.6434527,\n      "lon": 37.5365573,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374271122",\n      "kind": "kindergarten",\n      "name": "Академическая гимназия",\n      "lat": 55.6843339,\n      "lon": 37.5943692,\n      "opening_hours": "Mo-Su 07:30-19:30"\n    },\n    {\n      "id": "way/374629533",\n      "kind": "school",\n      "name": "Школа №1076",\n      "lat": 55.8134114,\n      "lon": 37.8182652,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374939352",\n      "kind": "kindergarten",\n      "name": "Школа №109. Дошкольные группы",\n      "lat": 55.6416672,\n      "lon": 37.4762207,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374951049",\n      "kind": "kindergarten",\n      "name": "Школа №1995. Дошкольный уровень",\n      "lat": 55.6433273,\n      "lon": 37.4798488,\n      "opening_hours": null\n    },\n    {\n      "id": "way/374958831",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Замок детства\\"",\n      "lat": 55.5841738,\n      "lon": 37.7334101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/376499093",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8893178,\n      "lon": 37.6585919,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377708650",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6731937,\n      "lon": 37.4529157,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377708651",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6767323,\n      "lon": 37.4488698,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377708652",\n      "kind": "school",\n      "name": "Школа № 2025",\n      "lat": 55.6777208,\n      "lon": 37.4462442,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377708653",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6883637,\n      "lon": 37.4667261,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377708654",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6754746,\n      "lon": 37.4481651,\n      "opening_hours": null\n    },\n    {\n      "id": "way/377728857",\n      "kind": "school",\n      "name": "Школа №1173",\n      "lat": 55.6168244,\n      "lon": 37.5945999,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378038130",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6694457,\n      "lon": 37.5758566,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378124074",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7554576,\n      "lon": 37.5699743,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378186606",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 9",\n      "lat": 55.8549232,\n      "lon": 37.6410485,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378186608",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 7",\n      "lat": 55.8544926,\n      "lon": 37.6433601,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378418006",\n      "kind": "school",\n      "name": "Школа «Содружество». Учебный корпус №3",\n      "lat": 55.7635152,\n      "lon": 37.7032461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378894408",\n      "kind": "school",\n      "name": "Школа № 967, школьное отделение №2",\n      "lat": 55.8803451,\n      "lon": 37.6421581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/378894409",\n      "kind": "kindergarten",\n      "name": "Школа №967, дошкольное отделение №6",\n      "lat": 55.8788117,\n      "lon": 37.641468,\n      "opening_hours": null\n    },\n    {\n      "id": "way/379502769",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6613381,\n      "lon": 37.5940904,\n      "opening_hours": null\n    },\n    {\n      "id": "way/379502770",\n      "kind": "kindergarten",\n      "name": "Детский сад №2514",\n      "lat": 55.660408,\n      "lon": 37.5927616,\n      "opening_hours": null\n    },\n    {\n      "id": "way/382029609",\n      "kind": "kindergarten",\n      "name": "Детский сад №2021",\n      "lat": 55.7141857,\n      "lon": 37.6117815,\n      "opening_hours": null\n    },\n    {\n      "id": "way/383149167",\n      "kind": "kindergarten",\n      "name": "Детский сад №1899",\n      "lat": 55.7493118,\n      "lon": 37.8196775,\n      "opening_hours": null\n    },\n    {\n      "id": "way/383157041",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7460199,\n      "lon": 37.8206962,\n      "opening_hours": null\n    },\n    {\n      "id": "way/383319619",\n      "kind": "school",\n      "name": "Романовская школа",\n      "lat": 55.7685951,\n      "lon": 37.5695966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/385816697",\n      "kind": "school",\n      "name": "Школа № 461",\n      "lat": 55.70263,\n      "lon": 37.7692074,\n      "opening_hours": null\n    },\n    {\n      "id": "way/385965951",\n      "kind": "school",\n      "name": "Школа №368 (Здание №2)",\n      "lat": 55.8222076,\n      "lon": 37.8200132,\n      "opening_hours": null\n    },\n    {\n      "id": "way/386178150",\n      "kind": "school",\n      "name": "Школа №1411",\n      "lat": 55.8669381,\n      "lon": 37.6007503,\n      "opening_hours": null\n    },\n    {\n      "id": "way/386345936",\n      "kind": "kindergarten",\n      "name": "Школа «Содружество». Учебный корпус №5",\n      "lat": 55.758656,\n      "lon": 37.7135594,\n      "opening_hours": null\n    },\n    {\n      "id": "way/386779191",\n      "kind": "school",\n      "name": "Школа № 1514",\n      "lat": 55.6865191,\n      "lon": 37.5292895,\n      "opening_hours": null\n    },\n    {\n      "id": "way/386779744",\n      "kind": "kindergarten",\n      "name": "Детский сад №63",\n      "lat": 55.7989267,\n      "lon": 37.6201377,\n      "opening_hours": null\n    },\n    {\n      "id": "way/386906994",\n      "kind": "kindergarten",\n      "name": "Школа №760 им. А. П. Маресьева. Дошкольное отделение",\n      "lat": 55.8762201,\n      "lon": 37.7101421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/387057250",\n      "kind": "kindergarten",\n      "name": "Детский сад №1903",\n      "lat": 55.7433006,\n      "lon": 37.7622113,\n      "opening_hours": null\n    },\n    {\n      "id": "way/387463901",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1626",\n      "lat": 55.8611164,\n      "lon": 37.5940337,\n      "opening_hours": null\n    },\n    {\n      "id": "way/387463902",\n      "kind": "kindergarten",\n      "name": "Школа № 962. Дошкольное отделение",\n      "lat": 55.8616045,\n      "lon": 37.5925913,\n      "opening_hours": null\n    },\n    {\n      "id": "way/387467935",\n      "kind": "kindergarten",\n      "name": "Школа №1310 (дошкольный корпус №3)",\n      "lat": 55.743555,\n      "lon": 37.7801359,\n      "opening_hours": null\n    },\n    {\n      "id": "way/387467936",\n      "kind": "kindergarten",\n      "name": "Школа №1310 (дошкольный корпус №2)",\n      "lat": 55.7427871,\n      "lon": 37.7801718,\n      "opening_hours": null\n    },\n    {\n      "id": "way/388407978",\n      "kind": "kindergarten",\n      "name": "Школа №1352 (дошкольная группа на Чусовской, 6)",\n      "lat": 55.8132975,\n      "lon": 37.815292,\n      "opening_hours": null\n    },\n    {\n      "id": "way/389025261",\n      "kind": "kindergarten",\n      "name": "Детский сад № 682 Аленький цветочек",\n      "lat": 55.7384988,\n      "lon": 37.4696899,\n      "opening_hours": null\n    },\n    {\n      "id": "way/389190468",\n      "kind": "kindergarten",\n      "name": "Школа на проспекте Вернадского. Корпус дошкольного образования 6",\n      "lat": 55.6589259,\n      "lon": 37.4842515,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390376376",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2476",\n      "lat": 55.6811384,\n      "lon": 37.4511423,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390483182",\n      "kind": "kindergarten",\n      "name": "Школа № 629. Корпус дошкольного образования №3",\n      "lat": 55.5890295,\n      "lon": 37.5958941,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390483183",\n      "kind": "school",\n      "name": "Школа № 629. Корпус школьного образования №5",\n      "lat": 55.5876227,\n      "lon": 37.5950827,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390483184",\n      "kind": "school",\n      "name": "Школа № 629. Корпус школьного образования №4",\n      "lat": 55.591432,\n      "lon": 37.5965467,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390483185",\n      "kind": "kindergarten",\n      "name": "Школа № 629. Корпус дошкольного образования №2",\n      "lat": 55.5923739,\n      "lon": 37.5940998,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390485001",\n      "kind": "kindergarten",\n      "name": "Детский сад №1431",\n      "lat": 55.8593986,\n      "lon": 37.6705996,\n      "opening_hours": null\n    },\n    {\n      "id": "way/390485005",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8596248,\n      "lon": 37.6686069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/393266311",\n      "kind": "kindergarten",\n      "name": "Школа № 1474, дошкольное отделение",\n      "lat": 55.8735371,\n      "lon": 37.4915173,\n      "opening_hours": null\n    },\n    {\n      "id": "way/394363818",\n      "kind": "school",\n      "name": "Школа №760 им. А. П. Маресьева. Школьное отделение",\n      "lat": 55.87765,\n      "lon": 37.7149576,\n      "opening_hours": null\n    },\n    {\n      "id": "way/394589999",\n      "kind": "school",\n      "name": "Школа №760 им. А. П. Маресьева. Школьное отделение",\n      "lat": 55.8773705,\n      "lon": 37.7111649,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395166265",\n      "kind": "kindergarten",\n      "name": "Школа №1811 \\"Восточное Измайлово\\" корпус 14",\n      "lat": 55.7964263,\n      "lon": 37.8227056,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395166267",\n      "kind": "school",\n      "name": "Училище олимпийского резерва № 1",\n      "lat": 55.7981421,\n      "lon": 37.8219185,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395236550",\n      "kind": "kindergarten",\n      "name": "Школа №1811 \\"Восточное Измайлово\\" корпус 12",\n      "lat": 55.7985003,\n      "lon": 37.8201491,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395236552",\n      "kind": "kindergarten",\n      "name": "Школа №1811 \\"Восточное Измайлово\\" корпус 16",\n      "lat": 55.7956005,\n      "lon": 37.8157633,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395236553",\n      "kind": "school",\n      "name": "Школа №1811 \\"Восточное Измайлово\\" корпус 4",\n      "lat": 55.7962876,\n      "lon": 37.8157306,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395430224",\n      "kind": "school",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпуc №3 (для детей с особыми образовательными потребностями)",\n      "lat": 55.7991279,\n      "lon": 37.8259144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395430226",\n      "kind": "school",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус 2",\n      "lat": 55.8023486,\n      "lon": 37.8332336,\n      "opening_hours": null\n    },\n    {\n      "id": "way/395756677",\n      "kind": "kindergarten",\n      "name": "Детский сад №423",\n      "lat": 55.5993408,\n      "lon": 37.6558297,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396444371",\n      "kind": "kindergarten",\n      "name": "Школа самоопределения № 734 имени А. Н. Тубельского (корпус №2)",\n      "lat": 55.7997801,\n      "lon": 37.8215733,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396444372",\n      "kind": "kindergarten",\n      "name": "Школа №1811 (дошкольный корпус №10)",\n      "lat": 55.8003633,\n      "lon": 37.8176821,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396444374",\n      "kind": "school",\n      "name": "Школа №1811 (школьный корпус №6)",\n      "lat": 55.8001185,\n      "lon": 37.8155549,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396444375",\n      "kind": "school",\n      "name": "Школа самоопределения № 734 имени А. Н. Тубельского (корпус №1)",\n      "lat": 55.8011975,\n      "lon": 37.8240129,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396864913",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Сказка\\"",\n      "lat": 55.8750218,\n      "lon": 37.7107411,\n      "opening_hours": null\n    },\n    {\n      "id": "way/396923766",\n      "kind": "school",\n      "name": "Школа № 1245. Корпус школьного образования №2",\n      "lat": 55.5994146,\n      "lon": 37.5954815,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397749563",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2148",\n      "lat": 55.8698656,\n      "lon": 37.6087129,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397749564",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1554. Дошкольное отделение",\n      "lat": 55.8694,\n      "lon": 37.6056915,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397749565",\n      "kind": "kindergarten",\n      "name": "Школа № 1411. Структурное подразделение № 978",\n      "lat": 55.8685277,\n      "lon": 37.6030916,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397749567",\n      "kind": "school",\n      "name": "Детско-юношеский центр «Отрадное»",\n      "lat": 55.870072,\n      "lon": 37.6150961,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397749568",\n      "kind": "kindergarten",\n      "name": "Начальная школа -детский сад №1879",\n      "lat": 55.8709316,\n      "lon": 37.6144368,\n      "opening_hours": null\n    },\n    {\n      "id": "way/397934095",\n      "kind": "kindergarten",\n      "name": "Детский сад №2495",\n      "lat": 55.670626,\n      "lon": 37.757475,\n      "opening_hours": null\n    },\n    {\n      "id": "way/398512694",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8658295,\n      "lon": 37.4871452,\n      "opening_hours": null\n    },\n    {\n      "id": "way/398512988",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8625257,\n      "lon": 37.4883318,\n      "opening_hours": null\n    },\n    {\n      "id": "way/398981887",\n      "kind": "kindergarten",\n      "name": "Школа №368 (Здание №4)",\n      "lat": 55.8217372,\n      "lon": 37.81568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399065542",\n      "kind": "school",\n      "name": "Хорошёвская прогимназия",\n      "lat": 55.7835468,\n      "lon": 37.4657546,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399258980",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.5922383,\n      "lon": 37.6061252,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399276902",\n      "kind": "kindergarten",\n      "name": "Детский сад №2640",\n      "lat": 55.7442809,\n      "lon": 37.6822947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399584298",\n      "kind": "school",\n      "name": "Школа № 1518",\n      "lat": 55.8036181,\n      "lon": 37.6347402,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399595392",\n      "kind": "kindergarten",\n      "name": "Детский сад №957",\n      "lat": 55.6163535,\n      "lon": 37.5970222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399595394",\n      "kind": "kindergarten",\n      "name": "Детский сад № 897",\n      "lat": 55.618628,\n      "lon": 37.5886996,\n      "opening_hours": null\n    },\n    {\n      "id": "way/399595402",\n      "kind": "school",\n      "name": "Школа №556. Корпус начального образования",\n      "lat": 55.6208187,\n      "lon": 37.5858025,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400514902",\n      "kind": "school",\n      "name": "Школа № 1582. Корпус школьного образования",\n      "lat": 55.6183662,\n      "lon": 37.5971795,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400616531",\n      "kind": "school",\n      "name": "Финансово-экономическая школа",\n      "lat": 55.8242532,\n      "lon": 37.5821131,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400702131",\n      "kind": "kindergarten",\n      "name": "Школа \\"Марьина роща\\" имени В.Ф. Фролова",\n      "lat": 55.8075121,\n      "lon": 37.6137209,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400717278",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8095632,\n      "lon": 37.6299132,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400717280",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8099871,\n      "lon": 37.619417,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400717281",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8097293,\n      "lon": 37.620565,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400761385",\n      "kind": "school",\n      "name": "Школа № 1531 имени С. К. Годовикова",\n      "lat": 55.8038384,\n      "lon": 37.6327084,\n      "opening_hours": null\n    },\n    {\n      "id": "way/400804215",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.8751568,\n      "lon": 37.6295166,\n      "opening_hours": null\n    },\n    {\n      "id": "way/401282468",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №1",\n      "lat": 55.6193035,\n      "lon": 37.6005617,\n      "opening_hours": null\n    },\n    {\n      "id": "way/401282470",\n      "kind": "school",\n      "name": "Школа № 1582. Корпус школьного образования",\n      "lat": 55.6199163,\n      "lon": 37.5968862,\n      "opening_hours": null\n    },\n    {\n      "id": "way/401791217",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №2",\n      "lat": 55.6185972,\n      "lon": 37.6047427,\n      "opening_hours": null\n    },\n    {\n      "id": "way/401791223",\n      "kind": "school",\n      "name": "ДШИ \\"Тутти\\"",\n      "lat": 55.6127838,\n      "lon": 37.6005689,\n      "opening_hours": null\n    },\n    {\n      "id": "way/403258384",\n      "kind": "kindergarten",\n      "name": "Школа № 1582. Корпус дошкольного образования №3",\n      "lat": 55.6174214,\n      "lon": 37.6040874,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404189291",\n      "kind": "kindergarten",\n      "name": "Детский сад №1158",\n      "lat": 55.6278953,\n      "lon": 37.6081784,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404373621",\n      "kind": "school",\n      "name": "Начальная школа - детский сад №1820",\n      "lat": 55.5979379,\n      "lon": 37.5948011,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404373622",\n      "kind": "kindergarten",\n      "name": "Школа № 1245. Корпус дошкольного образования №6",\n      "lat": 55.5983338,\n      "lon": 37.5925095,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404379796",\n      "kind": "kindergarten",\n      "name": "Школа № 1245. Корпус дошкольного образования №4",\n      "lat": 55.6036467,\n      "lon": 37.5955711,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404896628",\n      "kind": "kindergarten",\n      "name": "Детский сад №740",\n      "lat": 55.6003145,\n      "lon": 37.5842174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404896629",\n      "kind": "school",\n      "name": "Школа №932",\n      "lat": 55.5998095,\n      "lon": 37.5856551,\n      "opening_hours": null\n    },\n    {\n      "id": "way/404896633",\n      "kind": "kindergarten",\n      "name": "Детский сад №1855",\n      "lat": 55.5984043,\n      "lon": 37.5823008,\n      "opening_hours": null\n    },\n    {\n      "id": "way/405103977",\n      "kind": "kindergarten",\n      "name": "Детский сад №364",\n      "lat": 55.7094654,\n      "lon": 37.6038531,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "way/405119704",\n      "kind": "school",\n      "name": "Школа №409",\n      "lat": 55.7460849,\n      "lon": 37.8190331,\n      "opening_hours": null\n    },\n    {\n      "id": "way/405306213",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/5",\n      "lat": 55.6210344,\n      "lon": 37.4973385,\n      "opening_hours": null\n    },\n    {\n      "id": "way/405311002",\n      "kind": "kindergarten",\n      "name": "Н.ш.-д.с. №1820",\n      "lat": 55.5987228,\n      "lon": 37.5902983,\n      "opening_hours": null\n    },\n    {\n      "id": "way/405413167",\n      "kind": "kindergarten",\n      "name": "Детский сад №2160",\n      "lat": 55.6020157,\n      "lon": 37.5869107,\n      "opening_hours": null\n    },\n    {\n      "id": "way/406178595",\n      "kind": "kindergarten",\n      "name": "Школа № 554",\n      "lat": 55.6559922,\n      "lon": 37.5790847,\n      "opening_hours": null\n    },\n    {\n      "id": "way/407197238",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6733256,\n      "lon": 37.5625679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/407197239",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6727681,\n      "lon": 37.5619856,\n      "opening_hours": null\n    },\n    {\n      "id": "way/409972622",\n      "kind": "kindergarten",\n      "name": "Детский сад № 859",\n      "lat": 55.7281841,\n      "lon": 37.6328063,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410016274",\n      "kind": "kindergarten",\n      "name": "Детский сад №1949",\n      "lat": 55.7610584,\n      "lon": 37.7149582,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410016275",\n      "kind": "kindergarten",\n      "name": "Детский сад №471",\n      "lat": 55.7654898,\n      "lon": 37.7165835,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410329463",\n      "kind": "school",\n      "name": "Школа № 554",\n      "lat": 55.6571011,\n      "lon": 37.5798305,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410648326",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa0430",\n      "lat": 55.8598473,\n      "lon": 37.5023723,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410648327",\n      "kind": "kindergarten",\n      "name": "Психоневрологический детский санаторий № 30",\n      "lat": 55.858768,\n      "lon": 37.5042696,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410673515",\n      "kind": "school",\n      "name": "Гимназия № 1590",\n      "lat": 55.859617,\n      "lon": 37.4968942,\n      "opening_hours": null\n    },\n    {\n      "id": "way/410873677",\n      "kind": "kindergarten",\n      "name": "Школа №760 им. А. П. Маресьева. Дошкольное отделение",\n      "lat": 55.8798861,\n      "lon": 37.7173487,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055126",\n      "kind": "school",\n      "name": "Вешняковская школа. Школьное отделение Р2",\n      "lat": 55.7412471,\n      "lon": 37.833085,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055128",\n      "kind": "school",\n      "name": "Вешняковская школа. Отделение В2",\n      "lat": 55.7368693,\n      "lon": 37.8194826,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055131",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение Р2",\n      "lat": 55.7402024,\n      "lon": 37.832808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055132",\n      "kind": "kindergarten",\n      "name": "Вешняковское школа. Дошкольное отделение Г12",\n      "lat": 55.7388634,\n      "lon": 37.8266828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055133",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольная группа №8",\n      "lat": 55.7415112,\n      "lon": 37.8246743,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055135",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение Р24",\n      "lat": 55.7333339,\n      "lon": 37.8343378,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411055136",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение В3",\n      "lat": 55.739526,\n      "lon": 37.8207923,\n      "opening_hours": null\n    },\n    {\n      "id": "way/411734363",\n      "kind": "school",\n      "name": "Школа № 1234",\n      "lat": 55.7513284,\n      "lon": 37.5809227,\n      "opening_hours": null\n    },\n    {\n      "id": "way/413720755",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8607172,\n      "lon": 37.5046593,\n      "opening_hours": null\n    },\n    {\n      "id": "way/413720759",\n      "kind": "kindergarten",\n      "name": "Детский сад №174",\n      "lat": 55.8574882,\n      "lon": 37.4999611,\n      "opening_hours": null\n    },\n    {\n      "id": "way/416474001",\n      "kind": "kindergarten",\n      "name": "Детский сад №2593",\n      "lat": 55.6431986,\n      "lon": 37.6518222,\n      "opening_hours": null\n    },\n    {\n      "id": "way/418714324",\n      "kind": "school",\n      "name": "",\n      "lat": 55.8904965,\n      "lon": 37.6581524,\n      "opening_hours": null\n    },\n    {\n      "id": "way/418949050",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 7",\n      "lat": 55.779392,\n      "lon": 37.5286584,\n      "opening_hours": null\n    },\n    {\n      "id": "way/418959509",\n      "kind": "kindergarten",\n      "name": "Школа № 1288, корпус № 4",\n      "lat": 55.7780677,\n      "lon": 37.5326836,\n      "opening_hours": null\n    },\n    {\n      "id": "way/422615145",\n      "kind": "kindergarten",\n      "name": "Школа № 1948 \\"Лингвист-М\\". Учебный корпус 6 и 7",\n      "lat": 55.6636299,\n      "lon": 37.5825885,\n      "opening_hours": null\n    },\n    {\n      "id": "way/423811043",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6947786,\n      "lon": 37.4585915,\n      "opening_hours": null\n    },\n    {\n      "id": "way/424314736",\n      "kind": "school",\n      "name": "Школа №1636 НИКА",\n      "lat": 55.6061841,\n      "lon": 37.729846,\n      "opening_hours": null\n    },\n    {\n      "id": "way/424822341",\n      "kind": "school",\n      "name": "Школа № 2090, структурное подразделение № 6",\n      "lat": 55.7131876,\n      "lon": 37.7845159,\n      "opening_hours": null\n    },\n    {\n      "id": "way/424822344",\n      "kind": "kindergarten",\n      "name": "Детский сад №1638",\n      "lat": 55.7111778,\n      "lon": 37.7907619,\n      "opening_hours": null\n    },\n    {\n      "id": "way/425074005",\n      "kind": "kindergarten",\n      "name": "Школа № 199. Дошкольное отделение № 7",\n      "lat": 55.6862245,\n      "lon": 37.56161,\n      "opening_hours": null\n    },\n    {\n      "id": "way/425202129",\n      "kind": "school",\n      "name": "Школа №741",\n      "lat": 55.7954249,\n      "lon": 37.80177,\n      "opening_hours": null\n    },\n    {\n      "id": "way/425304688",\n      "kind": "kindergarten",\n      "name": "Международная школа \\"Концепт\\". Детский сад",\n      "lat": 55.7942319,\n      "lon": 37.8041269,\n      "opening_hours": null\n    },\n    {\n      "id": "way/425378025",\n      "kind": "school",\n      "name": "Школа № 1948 \\"Лингвист-М\\". Центр дополнительного образования",\n      "lat": 55.6643429,\n      "lon": 37.583951,\n      "opening_hours": null\n    },\n    {\n      "id": "way/429116890",\n      "kind": "school",\n      "name": "Центр образования № 1272",\n      "lat": 55.7079002,\n      "lon": 37.663441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/429427494",\n      "kind": "kindergarten",\n      "name": "ДОУ",\n      "lat": 55.7632531,\n      "lon": 37.7434553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/430865710",\n      "kind": "kindergarten",\n      "name": "Детский сад №964",\n      "lat": 55.5883382,\n      "lon": 37.6537433,\n      "opening_hours": null\n    },\n    {\n      "id": "way/431163355",\n      "kind": "school",\n      "name": "Школа № 1288, корпус № 11",\n      "lat": 55.7761318,\n      "lon": 37.5301211,\n      "opening_hours": null\n    },\n    {\n      "id": "way/431566577",\n      "kind": "kindergarten",\n      "name": "Знайка",\n      "lat": 55.6038283,\n      "lon": 37.6608775,\n      "opening_hours": null\n    },\n    {\n      "id": "way/434375819",\n      "kind": "kindergarten",\n      "name": "Школа № 727 (Детский сад)",\n      "lat": 55.8505995,\n      "lon": 37.5160843,\n      "opening_hours": null\n    },\n    {\n      "id": "way/435300297",\n      "kind": "school",\n      "name": "Школа №2033 (средняя и старшая школа)",\n      "lat": 55.8074721,\n      "lon": 37.8104311,\n      "opening_hours": null\n    },\n    {\n      "id": "way/435937342",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7548671,\n      "lon": 37.8211338,\n      "opening_hours": null\n    },\n    {\n      "id": "way/436009599",\n      "kind": "school",\n      "name": "Школа №511",\n      "lat": 55.6459039,\n      "lon": 37.6632881,\n      "opening_hours": null\n    },\n    {\n      "id": "way/436601516",\n      "kind": "school",\n      "name": "Гимназия №1798 \\"Феникс\\"",\n      "lat": 55.7539544,\n      "lon": 37.8046638,\n      "opening_hours": null\n    },\n    {\n      "id": "way/438153842",\n      "kind": "kindergarten",\n      "name": "Школа №2107, дошкольное отделение",\n      "lat": 55.7869319,\n      "lon": 37.6243553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/439928256",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6716065,\n      "lon": 37.6124307,\n      "opening_hours": null\n    },\n    {\n      "id": "way/440610765",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2019",\n      "lat": 55.6896976,\n      "lon": 37.5909215,\n      "opening_hours": null\n    },\n    {\n      "id": "way/441522066",\n      "kind": "school",\n      "name": "Романовская школа",\n      "lat": 55.7511141,\n      "lon": 37.5240103,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443086824",\n      "kind": "school",\n      "name": "Школа №1222",\n      "lat": 55.7327777,\n      "lon": 37.7062437,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443369267",\n      "kind": "kindergarten",\n      "name": "ДМШ №35",\n      "lat": 55.7088467,\n      "lon": 37.6759961,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443375876",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129 ДО \\"Детство\\"",\n      "lat": 55.707895,\n      "lon": 37.679498,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443391981",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Жемчужинка\\"",\n      "lat": 55.7279498,\n      "lon": 37.7611351,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443470161",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа № 2129 ДО \\"Непоседы\\"",\n      "lat": 55.7063069,\n      "lon": 37.6803725,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443480698",\n      "kind": "school",\n      "name": "ГБОУ Школа № 2129 ШО для детей с ОВЗ",\n      "lat": 55.7056836,\n      "lon": 37.6722072,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443593271",\n      "kind": "school",\n      "name": "ГБОУ Школа №799, корпус 1",\n      "lat": 55.7691695,\n      "lon": 37.8253811,\n      "opening_hours": null\n    },\n    {\n      "id": "way/443593272",\n      "kind": "school",\n      "name": "ГБОУ Школа №799, корпус 2",\n      "lat": 55.7701493,\n      "lon": 37.8288644,\n      "opening_hours": null\n    },\n    {\n      "id": "way/444428136",\n      "kind": "kindergarten",\n      "name": "Детский сад №1777",\n      "lat": 55.7450525,\n      "lon": 37.6742782,\n      "opening_hours": null\n    },\n    {\n      "id": "way/444428137",\n      "kind": "kindergarten",\n      "name": "Детский сад №1939",\n      "lat": 55.7429692,\n      "lon": 37.6727728,\n      "opening_hours": null\n    },\n    {\n      "id": "way/445022081",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.8729713,\n      "lon": 37.6259949,\n      "opening_hours": null\n    },\n    {\n      "id": "way/445022088",\n      "kind": "kindergarten",\n      "name": "Школа №285 имени В.А. Молодцова, дошкольное отделение",\n      "lat": 55.8722345,\n      "lon": 37.6252449,\n      "opening_hours": null\n    },\n    {\n      "id": "way/445522765",\n      "kind": "kindergarten",\n      "name": "Лицей № 1568 имени Пабло Неруды, дошкольные отделения №2, 5, 6",\n      "lat": 55.8764058,\n      "lon": 37.6412513,\n      "opening_hours": null\n    },\n    {\n      "id": "way/445522767",\n      "kind": "kindergarten",\n      "name": "Школа №967, дошкольное отделение №5",\n      "lat": 55.8811752,\n      "lon": 37.6430426,\n      "opening_hours": null\n    },\n    {\n      "id": "way/445971629",\n      "kind": "kindergarten",\n      "name": "Школа №967, дошкольное отделение №3",\n      "lat": 55.8787132,\n      "lon": 37.6475385,\n      "opening_hours": null\n    },\n    {\n      "id": "way/447984762",\n      "kind": "kindergarten",\n      "name": "Детский сад №1818",\n      "lat": 55.8830796,\n      "lon": 37.711249,\n      "opening_hours": null\n    },\n    {\n      "id": "way/447984779",\n      "kind": "kindergarten",\n      "name": "Детский сад №278",\n      "lat": 55.8815306,\n      "lon": 37.7087141,\n      "opening_hours": null\n    },\n    {\n      "id": "way/448154309",\n      "kind": "kindergarten",\n      "name": "Детский сад №406",\n      "lat": 55.6657735,\n      "lon": 37.5194747,\n      "opening_hours": null\n    },\n    {\n      "id": "way/449497270",\n      "kind": "kindergarten",\n      "name": "Детский сад №240",\n      "lat": 55.8007876,\n      "lon": 37.8021527,\n      "opening_hours": null\n    },\n    {\n      "id": "way/449684295",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6735244,\n      "lon": 37.4678958,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450112051",\n      "kind": "kindergarten",\n      "name": "Школа №1811 (дошкольный корпус №15)",\n      "lat": 55.8018115,\n      "lon": 37.8108373,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450112055",\n      "kind": "kindergarten",\n      "name": "Детский сад №1522",\n      "lat": 55.7929112,\n      "lon": 37.8123178,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450112057",\n      "kind": "school",\n      "name": "Кладезь",\n      "lat": 55.8018366,\n      "lon": 37.800374,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450357369",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 4",\n      "lat": 55.8603655,\n      "lon": 37.6391282,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450357372",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 16",\n      "lat": 55.8593541,\n      "lon": 37.6370481,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450362997",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение школы №717",\n      "lat": 55.8185974,\n      "lon": 37.5050245,\n      "opening_hours": null\n    },\n    {\n      "id": "way/450366849",\n      "kind": "school",\n      "name": "Школа №717",\n      "lat": 55.8172835,\n      "lon": 37.5050391,\n      "opening_hours": null\n    },\n    {\n      "id": "way/452897582",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 507, Щелкунчик (здание №11)",\n      "lat": 55.673897,\n      "lon": 37.6672121,\n      "opening_hours": null\n    },\n    {\n      "id": "way/453368828",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8387752,\n      "lon": 37.5750259,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454021189",\n      "kind": "kindergarten",\n      "name": "Школа №1223, дошкольный корпус №4",\n      "lat": 55.8410143,\n      "lon": 37.5466998,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454055460",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа \\"Кузьминки\\"",\n      "lat": 55.7028199,\n      "lon": 37.7741658,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454055462",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.703316,\n      "lon": 37.770527,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454064408",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7007361,\n      "lon": 37.7702559,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454311146",\n      "kind": "kindergarten",\n      "name": "Школа № 656 им. А. С. Макаренко. Дошкольный корпус",\n      "lat": 55.869375,\n      "lon": 37.5513991,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454469587",\n      "kind": "kindergarten",\n      "name": "Школа №1290 дошкольный корпус №1",\n      "lat": 55.7929082,\n      "lon": 37.7783216,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454743223",\n      "kind": "school",\n      "name": "Школа № 1285. Школьный корпус",\n      "lat": 55.8257017,\n      "lon": 37.4496465,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454743226",\n      "kind": "kindergarten",\n      "name": "Школа №654 имени А. Д. Фридмана. Дошкольное отделение №1",\n      "lat": 55.6977138,\n      "lon": 37.7417608,\n      "opening_hours": null\n    },\n    {\n      "id": "way/454844150",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №14",\n      "lat": 55.6672437,\n      "lon": 37.7795248,\n      "opening_hours": null\n    },\n    {\n      "id": "way/455510927",\n      "kind": "kindergarten",\n      "name": "Детский сад №1187",\n      "lat": 55.7542167,\n      "lon": 37.6753953,\n      "opening_hours": null\n    },\n    {\n      "id": "way/455697968",\n      "kind": "kindergarten",\n      "name": "Детский сад №962",\n      "lat": 55.8673923,\n      "lon": 37.7001389,\n      "opening_hours": null\n    },\n    {\n      "id": "way/456669987",\n      "kind": "kindergarten",\n      "name": "Детский сад №2445",\n      "lat": 55.8805037,\n      "lon": 37.5520859,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459404196",\n      "kind": "kindergarten",\n      "name": "Детский сад №643",\n      "lat": 55.7424834,\n      "lon": 37.6874653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459574973",\n      "kind": "school",\n      "name": "Школа №1716 \\"Эврика-Огонёк\\"",\n      "lat": 55.6654085,\n      "lon": 37.774361,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459575997",\n      "kind": "kindergarten",\n      "name": "Школа № 1716 \\"Эврика-Огонек\\". Дошкольное подразделение",\n      "lat": 55.6660439,\n      "lon": 37.7724445,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459584374",\n      "kind": "kindergarten",\n      "name": "Школа № 1989. Дошкольное отделение № 1",\n      "lat": 55.6627951,\n      "lon": 37.7775865,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459586029",\n      "kind": "school",\n      "name": "Центр образования №1989",\n      "lat": 55.6636276,\n      "lon": 37.7759043,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459837877",\n      "kind": "school",\n      "name": "Школа №2011",\n      "lat": 55.6677537,\n      "lon": 37.7887291,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459838512",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6684037,\n      "lon": 37.7881625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459979623",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6718643,\n      "lon": 37.7917963,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459980354",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6723095,\n      "lon": 37.7925643,\n      "opening_hours": null\n    },\n    {\n      "id": "way/459980356",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6717319,\n      "lon": 37.7931056,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460006530",\n      "kind": "school",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ШП-1)",\n      "lat": 55.6712605,\n      "lon": 37.7690968,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460010094",\n      "kind": "school",\n      "name": "Школа №1186",\n      "lat": 55.6720073,\n      "lon": 37.7673517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460014006",\n      "kind": "kindergarten",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ДГ-3)",\n      "lat": 55.6699962,\n      "lon": 37.7677221,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460237676",\n      "kind": "kindergarten",\n      "name": "ГБОУ СОШ № 572 ДО № 8",\n      "lat": 55.6801952,\n      "lon": 37.7749467,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460240331",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6789643,\n      "lon": 37.773561,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460240720",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6791372,\n      "lon": 37.770448,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460240932",\n      "kind": "school",\n      "name": "Школа №572",\n      "lat": 55.6801343,\n      "lon": 37.7699305,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460245030",\n      "kind": "school",\n      "name": "Школа № 1547",\n      "lat": 55.663779,\n      "lon": 37.7673599,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460511550",\n      "kind": "kindergarten",\n      "name": "Детский сад №2437",\n      "lat": 55.6538686,\n      "lon": 37.7701124,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460512501",\n      "kind": "school",\n      "name": "Школа №1959 \\"Школа здоровья\\"",\n      "lat": 55.655738,\n      "lon": 37.7698676,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460514084",\n      "kind": "kindergarten",\n      "name": "Детский сад №2398",\n      "lat": 55.655292,\n      "lon": 37.7683679,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460514466",\n      "kind": "school",\n      "name": "Школа №1959 \\"Школа здоровья\\" (Гимназические классы)",\n      "lat": 55.6543652,\n      "lon": 37.7714188,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460560229",\n      "kind": "kindergarten",\n      "name": "Школа №1547. Дошкольное отделение",\n      "lat": 55.6631947,\n      "lon": 37.7661136,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460564113",\n      "kind": "kindergarten",\n      "name": "Школа №1423. Дошкольное отделение",\n      "lat": 55.6642222,\n      "lon": 37.7688131,\n      "opening_hours": null\n    },\n    {\n      "id": "way/460993900",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.5892129,\n      "lon": 37.4574601,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461030275",\n      "kind": "school",\n      "name": "Международная школа завтрашнего дня",\n      "lat": 55.6329431,\n      "lon": 37.6088891,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461109423",\n      "kind": "school",\n      "name": "Школа №1904",\n      "lat": 55.649823,\n      "lon": 37.7475564,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461110609",\n      "kind": "kindergarten",\n      "name": "Детский сад №2325",\n      "lat": 55.649498,\n      "lon": 37.7502884,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461111582",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6520484,\n      "lon": 37.7458941,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461358380",\n      "kind": "kindergarten",\n      "name": "Детский сад №2280",\n      "lat": 55.7892197,\n      "lon": 37.7872809,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461703728",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №3",\n      "lat": 55.5902005,\n      "lon": 37.6109967,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461836412",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №6",\n      "lat": 55.6511098,\n      "lon": 37.7639551,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461836978",\n      "kind": "school",\n      "name": "Школа № 1357 «На Братиславской». Здание №5",\n      "lat": 55.6505489,\n      "lon": 37.7624974,\n      "opening_hours": null\n    },\n    {\n      "id": "way/461846655",\n      "kind": "kindergarten",\n      "name": "Детский сад №2397",\n      "lat": 55.6593661,\n      "lon": 37.7449486,\n      "opening_hours": null\n    },\n    {\n      "id": "way/467769169",\n      "kind": "school",\n      "name": "Школа \\"Ника\\"",\n      "lat": 55.6566093,\n      "lon": 37.566925,\n      "opening_hours": null\n    },\n    {\n      "id": "way/467769170",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 3",\n      "lat": 55.6564087,\n      "lon": 37.5617166,\n      "opening_hours": null\n    },\n    {\n      "id": "way/468860751",\n      "kind": "school",\n      "name": "ГБОУ г. Москвы Школа № 1159 (ДО)",\n      "lat": 55.8626661,\n      "lon": 37.5159729,\n      "opening_hours": null\n    },\n    {\n      "id": "way/469126286",\n      "kind": "kindergarten",\n      "name": "Лицей № 1568 имени Пабло Неруды, дошкольные отделения №3, 4, 7",\n      "lat": 55.8731054,\n      "lon": 37.6421695,\n      "opening_hours": null\n    },\n    {\n      "id": "way/472691717",\n      "kind": "kindergarten",\n      "name": "Школа №949 (дошкольное отделение, здание №5)",\n      "lat": 55.607815,\n      "lon": 37.750898,\n      "opening_hours": null\n    },\n    {\n      "id": "way/474041399",\n      "kind": "school",\n      "name": "Школа №949 (начальная школа, здание №2)",\n      "lat": 55.6089946,\n      "lon": 37.7518158,\n      "opening_hours": null\n    },\n    {\n      "id": "way/474041402",\n      "kind": "kindergarten",\n      "name": "Детский сад №1779",\n      "lat": 55.6861192,\n      "lon": 37.7550947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/474206141",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1703",\n      "lat": 55.6220269,\n      "lon": 37.7041333,\n      "opening_hours": null\n    },\n    {\n      "id": "way/475303188",\n      "kind": "kindergarten",\n      "name": "Школа №2033 (дошкольные группы)",\n      "lat": 55.8050198,\n      "lon": 37.8086258,\n      "opening_hours": null\n    },\n    {\n      "id": "way/475662142",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №13",\n      "lat": 55.6669792,\n      "lon": 37.7649866,\n      "opening_hours": null\n    },\n    {\n      "id": "way/476155610",\n      "kind": "kindergarten",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ДГ-9)",\n      "lat": 55.6781352,\n      "lon": 37.8006441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/476412270",\n      "kind": "school",\n      "name": "Школа № 572, Отделение № 9",\n      "lat": 55.6774152,\n      "lon": 37.7710418,\n      "opening_hours": null\n    },\n    {\n      "id": "way/476903701",\n      "kind": "kindergarten",\n      "name": "Детский сад №1547",\n      "lat": 55.7555245,\n      "lon": 37.7805331,\n      "opening_hours": null\n    },\n    {\n      "id": "way/477778886",\n      "kind": "kindergarten",\n      "name": "Школа № 49. Дошкольный корпус",\n      "lat": 55.6426509,\n      "lon": 37.5309424,\n      "opening_hours": null\n    },\n    {\n      "id": "way/477787991",\n      "kind": "kindergarten",\n      "name": "Школа № 49. Дошкольный корпус",\n      "lat": 55.6443387,\n      "lon": 37.5399307,\n      "opening_hours": "Mo-Fr 08:30-20:00, Tu-Su 08:00-15:00; Sa 08:00-12:00"\n    },\n    {\n      "id": "way/477908122",\n      "kind": "school",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 5.",\n      "lat": 55.6472517,\n      "lon": 37.5237555,\n      "opening_hours": null\n    },\n    {\n      "id": "way/479152965",\n      "kind": "school",\n      "name": "Школа №37. Корпус школьного образования",\n      "lat": 55.7027906,\n      "lon": 37.5025093,\n      "opening_hours": null\n    },\n    {\n      "id": "way/480525994",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.6972358,\n      "lon": 37.4945335,\n      "opening_hours": null\n    },\n    {\n      "id": "way/480699878",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7532888,\n      "lon": 37.6599928,\n      "opening_hours": null\n    },\n    {\n      "id": "way/481075429",\n      "kind": "school",\n      "name": "Структурное подразделение № 28 школы № 2124 «Центр развития и коррекции»",\n      "lat": 55.7655501,\n      "lon": 37.6719508,\n      "opening_hours": null\n    },\n    {\n      "id": "way/481379779",\n      "kind": "kindergarten",\n      "name": "Детский сад/ясли №1837",\n      "lat": 55.8658035,\n      "lon": 37.5922666,\n      "opening_hours": null\n    },\n    {\n      "id": "way/481547265",\n      "kind": "kindergarten",\n      "name": "Школа № 1357 «На Братиславской». Здание №9",\n      "lat": 55.6581803,\n      "lon": 37.7548634,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482435437",\n      "kind": "kindergarten",\n      "name": "Школа № 1465, дошкольное отделение № 3",\n      "lat": 55.7453437,\n      "lon": 37.5617271,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482435442",\n      "kind": "kindergarten",\n      "name": "Школа №1465 имени адмирала Н. Г. Кузнецова, дошкольное подразделение №7",\n      "lat": 55.7493641,\n      "lon": 37.5688828,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482435458",\n      "kind": "school",\n      "name": "Школа № 1228 с углубленным изучением английского языка. Корпус 5",\n      "lat": 55.7675145,\n      "lon": 37.7117966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482435459",\n      "kind": "school",\n      "name": "Школа № 1228 Корпус № 2",\n      "lat": 55.750002,\n      "lon": 37.7127725,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482447167",\n      "kind": "kindergarten",\n      "name": "Школа № 1411. Структурное подразделение № 1055 детский сад \\"Ручеек счастья\\"",\n      "lat": 55.8680473,\n      "lon": 37.6002271,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482447168",\n      "kind": "school",\n      "name": "Школа № 962. Структурное подразделение № 3",\n      "lat": 55.8615952,\n      "lon": 37.5877115,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482447172",\n      "kind": "kindergarten",\n      "name": "Школа № 962, дошкольное отделение № 4",\n      "lat": 55.8580891,\n      "lon": 37.5875808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482447186",\n      "kind": "kindergarten",\n      "name": "Школа № 950, дошкольное отделение",\n      "lat": 55.8592374,\n      "lon": 37.6077987,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482447187",\n      "kind": "kindergarten",\n      "name": "Гимназия № 1554. Дошкольное отделение № 3",\n      "lat": 55.8641495,\n      "lon": 37.6189017,\n      "opening_hours": null\n    },\n    {\n      "id": "way/482740277",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6723848,\n      "lon": 37.5377625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/483201827",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.889964,\n      "lon": 37.4399616,\n      "opening_hours": null\n    },\n    {\n      "id": "way/483709852",\n      "kind": "kindergarten",\n      "name": "Детский сад № 712",\n      "lat": 55.6202141,\n      "lon": 37.66659,\n      "opening_hours": null\n    },\n    {\n      "id": "way/484519282",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.619309,\n      "lon": 37.7599569,\n      "opening_hours": null\n    },\n    {\n      "id": "way/484519295",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6192405,\n      "lon": 37.7585881,\n      "opening_hours": null\n    },\n    {\n      "id": "way/484692544",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6719262,\n      "lon": 37.5895967,\n      "opening_hours": null\n    },\n    {\n      "id": "way/484924150",\n      "kind": "kindergarten",\n      "name": "Детский сад №2068",\n      "lat": 55.6466236,\n      "lon": 37.6641687,\n      "opening_hours": null\n    },\n    {\n      "id": "way/485063924",\n      "kind": "kindergarten",\n      "name": "Российское Дошкольное Отделение \\"Филиппок\\"",\n      "lat": 55.6963003,\n      "lon": 37.5087398,\n      "opening_hours": null\n    },\n    {\n      "id": "way/485063925",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6980742,\n      "lon": 37.5086194,\n      "opening_hours": null\n    },\n    {\n      "id": "way/485063931",\n      "kind": "school",\n      "name": "Школа №1498 «ММШ-15». Корпус школьного образования",\n      "lat": 55.6974493,\n      "lon": 37.5101101,\n      "opening_hours": null\n    },\n    {\n      "id": "way/485063932",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6988892,\n      "lon": 37.5078553,\n      "opening_hours": null\n    },\n    {\n      "id": "way/485382207",\n      "kind": "school",\n      "name": "Новая гуманитарная школа",\n      "lat": 55.8023622,\n      "lon": 37.5410002,\n      "opening_hours": null\n    },\n    {\n      "id": "way/487787682",\n      "kind": "kindergarten",\n      "name": "Многопрофильный лицей № 1501. 1-е дошкольное отделение № 1921",\n      "lat": 55.7792124,\n      "lon": 37.6039133,\n      "opening_hours": null\n    },\n    {\n      "id": "way/487951922",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6602417,\n      "lon": 37.7756397,\n      "opening_hours": null\n    },\n    {\n      "id": "way/487957328",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6598351,\n      "lon": 37.7642406,\n      "opening_hours": null\n    },\n    {\n      "id": "way/488261721",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6778493,\n      "lon": 37.7673625,\n      "opening_hours": null\n    },\n    {\n      "id": "way/488268383",\n      "kind": "kindergarten",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ДГ-5)",\n      "lat": 55.6794431,\n      "lon": 37.7668384,\n      "opening_hours": null\n    },\n    {\n      "id": "way/488482961",\n      "kind": "kindergarten",\n      "name": "Школа № 2099. Здание 3",\n      "lat": 55.8167738,\n      "lon": 37.4918663,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489066386",\n      "kind": "school",\n      "name": "Центр образования №1524. Корпус 3",\n      "lat": 55.6896469,\n      "lon": 37.7220748,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489272879",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6371021,\n      "lon": 37.7549869,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489393313",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8935527,\n      "lon": 37.4475519,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489393314",\n      "kind": "school",\n      "name": "Гимназия №4",\n      "lat": 55.8933318,\n      "lon": 37.4458706,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489393337",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa045",\n      "lat": 55.8894213,\n      "lon": 37.4540406,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489393339",\n      "kind": "school",\n      "name": "Лицей №\xa011",\n      "lat": 55.89046,\n      "lon": 37.453744,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489409755",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8891392,\n      "lon": 37.4502244,\n      "opening_hours": null\n    },\n    {\n      "id": "way/489409759",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8901645,\n      "lon": 37.4469646,\n      "opening_hours": null\n    },\n    {\n      "id": "way/490122227",\n      "kind": "school",\n      "name": "Школа №393",\n      "lat": 55.6775483,\n      "lon": 37.7609936,\n      "opening_hours": null\n    },\n    {\n      "id": "way/490420670",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa050",\n      "lat": 55.8883121,\n      "lon": 37.4545854,\n      "opening_hours": null\n    },\n    {\n      "id": "way/490420686",\n      "kind": "school",\n      "name": "Лицей №7",\n      "lat": 55.8867949,\n      "lon": 37.4555735,\n      "opening_hours": null\n    },\n    {\n      "id": "way/491821617",\n      "kind": "school",\n      "name": "Школа № 654 им. А. Д. Фридмана, корпус 3",\n      "lat": 55.6956907,\n      "lon": 37.7498411,\n      "opening_hours": null\n    },\n    {\n      "id": "way/492713772",\n      "kind": "school",\n      "name": "НОУ Школа \\"Апогей\\"",\n      "lat": 55.6795447,\n      "lon": 37.753285,\n      "opening_hours": null\n    },\n    {\n      "id": "way/492974181",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6504446,\n      "lon": 37.7555789,\n      "opening_hours": null\n    },\n    {\n      "id": "way/492974505",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6507007,\n      "lon": 37.7546706,\n      "opening_hours": null\n    },\n    {\n      "id": "way/493001319",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение «Лесная сказка»",\n      "lat": 55.62595,\n      "lon": 37.4810562,\n      "opening_hours": null\n    },\n    {\n      "id": "way/494132822",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6558453,\n      "lon": 37.7471003,\n      "opening_hours": null\n    },\n    {\n      "id": "way/494136258",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6540924,\n      "lon": 37.7459609,\n      "opening_hours": null\n    },\n    {\n      "id": "way/496457582",\n      "kind": "school",\n      "name": "Младшая школа №1496",\n      "lat": 55.7219792,\n      "lon": 37.6035283,\n      "opening_hours": null\n    },\n    {\n      "id": "way/497088018",\n      "kind": "school",\n      "name": "Школа № 553",\n      "lat": 55.7242137,\n      "lon": 37.6235887,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498121462",\n      "kind": "school",\n      "name": "Школа №5",\n      "lat": 55.6602874,\n      "lon": 37.6062387,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498175708",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1403",\n      "lat": 55.6619129,\n      "lon": 37.6063568,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498299188",\n      "kind": "school",\n      "name": "Школа №1623",\n      "lat": 55.6250771,\n      "lon": 37.6140342,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498304285",\n      "kind": "kindergarten",\n      "name": "Детский сад №1834",\n      "lat": 55.6241119,\n      "lon": 37.6126637,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498314470",\n      "kind": "kindergarten",\n      "name": "Детский сад №917",\n      "lat": 55.6137791,\n      "lon": 37.5954438,\n      "opening_hours": null\n    },\n    {\n      "id": "way/498572996",\n      "kind": "kindergarten",\n      "name": "Школа № 1158. Корпус дошкольного образования «Мечта»",\n      "lat": 55.6354819,\n      "lon": 37.5905932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/499308094",\n      "kind": "school",\n      "name": "Школа №285 имени В.А. Молодцова",\n      "lat": 55.8763892,\n      "lon": 37.6290788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/500981449",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8059641,\n      "lon": 37.5470221,\n      "opening_hours": null\n    },\n    {\n      "id": "way/502645356",\n      "kind": "school",\n      "name": "Школа №\xa083",\n      "lat": 55.7598113,\n      "lon": 37.5400412,\n      "opening_hours": null\n    },\n    {\n      "id": "way/502790155",\n      "kind": "kindergarten",\n      "name": "Детский сад №229",\n      "lat": 55.7106995,\n      "lon": 37.6615929,\n      "opening_hours": null\n    },\n    {\n      "id": "way/502828348",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2091",\n      "lat": 55.7291203,\n      "lon": 37.6788028,\n      "opening_hours": null\n    },\n    {\n      "id": "way/503995297",\n      "kind": "kindergarten",\n      "name": "Школа № 460 имени дважды Героев Советского Союза А.А. Головачёва и С.Ф. Шутова (ДГ-6)",\n      "lat": 55.6808544,\n      "lon": 37.7667484,\n      "opening_hours": null\n    },\n    {\n      "id": "way/504270193",\n      "kind": "kindergarten",\n      "name": "Детский сад №381",\n      "lat": 55.8080275,\n      "lon": 37.7112367,\n      "opening_hours": null\n    },\n    {\n      "id": "way/504521532",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6810519,\n      "lon": 37.6113237,\n      "opening_hours": null\n    },\n    {\n      "id": "way/504548447",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8854302,\n      "lon": 37.4432532,\n      "opening_hours": null\n    },\n    {\n      "id": "way/505041657",\n      "kind": "kindergarten",\n      "name": "Детский сад №720",\n      "lat": 55.8802785,\n      "lon": 37.7064247,\n      "opening_hours": null\n    },\n    {\n      "id": "way/506470757",\n      "kind": "kindergarten",\n      "name": "Детский сад №2704",\n      "lat": 55.875651,\n      "lon": 37.7202259,\n      "opening_hours": null\n    },\n    {\n      "id": "way/507033427",\n      "kind": "kindergarten",\n      "name": "Лицей № 38. Дошкольное отделение № 10",\n      "lat": 55.7830639,\n      "lon": 37.4682924,\n      "opening_hours": null\n    },\n    {\n      "id": "way/511924009",\n      "kind": "kindergarten",\n      "name": "Центр образования №1998",\n      "lat": 55.636983,\n      "lon": 37.7520557,\n      "opening_hours": null\n    },\n    {\n      "id": "way/515690177",\n      "kind": "school",\n      "name": "Школа №962. Начальная школа",\n      "lat": 55.8635333,\n      "lon": 37.5959611,\n      "opening_hours": null\n    },\n    {\n      "id": "way/520900483",\n      "kind": "school",\n      "name": "Школа №1797",\n      "lat": 55.8073519,\n      "lon": 37.7122562,\n      "opening_hours": null\n    },\n    {\n      "id": "way/523390513",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1387",\n      "lat": 55.7789975,\n      "lon": 37.6110774,\n      "opening_hours": null\n    },\n    {\n      "id": "way/523731664",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7715368,\n      "lon": 37.4709838,\n      "opening_hours": null\n    },\n    {\n      "id": "way/524737858",\n      "kind": "school",\n      "name": "Школа №440 (Гимназия № 1811 «Восточное Измайлово»)",\n      "lat": 55.7896737,\n      "lon": 37.8127305,\n      "opening_hours": null\n    },\n    {\n      "id": "way/526745593",\n      "kind": "kindergarten",\n      "name": "Школа на проспекте Вернадского. Корпус дошкольного образования 5",\n      "lat": 55.6563965,\n      "lon": 37.4901662,\n      "opening_hours": null\n    },\n    {\n      "id": "way/528700091",\n      "kind": "kindergarten",\n      "name": "Школа №1256 имени Героя Советского Союза И. С. Полбина (дошкольное отделение, корпус 3)",\n      "lat": 55.6784765,\n      "lon": 37.7260469,\n      "opening_hours": null\n    },\n    {\n      "id": "way/529572336",\n      "kind": "school",\n      "name": "Школа \\"Покровский квартал\\", Отделение ОВЗ",\n      "lat": 55.7548179,\n      "lon": 37.6464144,\n      "opening_hours": null\n    },\n    {\n      "id": "way/530563722",\n      "kind": "kindergarten",\n      "name": "Детский сад №1043",\n      "lat": 55.8007834,\n      "lon": 37.4846858,\n      "opening_hours": null\n    },\n    {\n      "id": "way/531958078",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7031226,\n      "lon": 37.4904059,\n      "opening_hours": null\n    },\n    {\n      "id": "way/532935146",\n      "kind": "kindergarten",\n      "name": "Гимназия №1506, дошкольное отделение",\n      "lat": 55.8911872,\n      "lon": 37.6485366,\n      "opening_hours": null\n    },\n    {\n      "id": "way/532935469",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Умка\\"",\n      "lat": 55.8925842,\n      "lon": 37.6572345,\n      "opening_hours": null\n    },\n    {\n      "id": "way/532936533",\n      "kind": "school",\n      "name": "Гимназия №1558",\n      "lat": 55.8685922,\n      "lon": 37.6533191,\n      "opening_hours": null\n    },\n    {\n      "id": "way/532939500",\n      "kind": "school",\n      "name": "Школа Свиблово, корпус 2",\n      "lat": 55.8538772,\n      "lon": 37.6591808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/539236815",\n      "kind": "kindergarten",\n      "name": "Детский сад № 879",\n      "lat": 55.7840475,\n      "lon": 37.6386054,\n      "opening_hours": null\n    },\n    {\n      "id": "way/539723370",\n      "kind": "kindergarten",\n      "name": "Детский сад Развитие 21век",\n      "lat": 55.6761008,\n      "lon": 37.5945524,\n      "opening_hours": null\n    },\n    {\n      "id": "way/544107259",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.799077,\n      "lon": 37.5530406,\n      "opening_hours": null\n    },\n    {\n      "id": "way/547279582",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8710569,\n      "lon": 37.6923745,\n      "opening_hours": null\n    },\n    {\n      "id": "way/547624723",\n      "kind": "school",\n      "name": "Школа №750",\n      "lat": 55.8647481,\n      "lon": 37.6960994,\n      "opening_hours": null\n    },\n    {\n      "id": "way/548236327",\n      "kind": "school",\n      "name": "Школа № 1536. Корпус школьного образования № 1",\n      "lat": 55.6800806,\n      "lon": 37.5393311,\n      "opening_hours": null\n    },\n    {\n      "id": "way/548239829",\n      "kind": "kindergarten",\n      "name": "Детский сад 97",\n      "lat": 55.6781345,\n      "lon": 37.5417093,\n      "opening_hours": null\n    },\n    {\n      "id": "way/552386633",\n      "kind": "kindergarten",\n      "name": "Детский сад - Школа Свиблово, корпус 11",\n      "lat": 55.8504273,\n      "lon": 37.6412022,\n      "opening_hours": null\n    },\n    {\n      "id": "way/552591097",\n      "kind": "school",\n      "name": "Специальная (коррекционная) школа-интернат № 102",\n      "lat": 55.8506431,\n      "lon": 37.6329417,\n      "opening_hours": null\n    },\n    {\n      "id": "way/552786569",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8004074,\n      "lon": 37.5266772,\n      "opening_hours": null\n    },\n    {\n      "id": "way/553048211",\n      "kind": "kindergarten",\n      "name": "Детский сад №690",\n      "lat": 55.8613423,\n      "lon": 37.4977078,\n      "opening_hours": null\n    },\n    {\n      "id": "way/553048213",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2085",\n      "lat": 55.8592747,\n      "lon": 37.5066078,\n      "opening_hours": null\n    },\n    {\n      "id": "way/554494298",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1384 имени А.А. Леманского\\"",\n      "lat": 55.7988268,\n      "lon": 37.50608,\n      "opening_hours": null\n    },\n    {\n      "id": "way/554647030",\n      "kind": "kindergarten",\n      "name": "Детский сад №2574",\n      "lat": 55.8757158,\n      "lon": 37.4867373,\n      "opening_hours": null\n    },\n    {\n      "id": "way/554648356",\n      "kind": "kindergarten",\n      "name": "Школа № 1474. Дошкольное отделение № 8",\n      "lat": 55.8768195,\n      "lon": 37.487539,\n      "opening_hours": null\n    },\n    {\n      "id": "way/554648358",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8720704,\n      "lon": 37.4949596,\n      "opening_hours": null\n    },\n    {\n      "id": "way/554648359",\n      "kind": "kindergarten",\n      "name": "Школа №\xa01474, дошкольное отделение",\n      "lat": 55.8672474,\n      "lon": 37.4985507,\n      "opening_hours": null\n    },\n    {\n      "id": "way/555198691",\n      "kind": "school",\n      "name": "Школа № 185",\n      "lat": 55.8367731,\n      "lon": 37.5353569,\n      "opening_hours": null\n    },\n    {\n      "id": "way/555999588",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8232333,\n      "lon": 37.6625466,\n      "opening_hours": null\n    },\n    {\n      "id": "way/555999622",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8230246,\n      "lon": 37.6626039,\n      "opening_hours": null\n    },\n    {\n      "id": "way/556568449",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1646",\n      "lat": 55.8977464,\n      "lon": 37.6112121,\n      "opening_hours": null\n    },\n    {\n      "id": "way/556568491",\n      "kind": "school",\n      "name": "Школа №1413",\n      "lat": 55.8991537,\n      "lon": 37.6119921,\n      "opening_hours": null\n    },\n    {\n      "id": "way/557818356",\n      "kind": "school",\n      "name": "Школа №224",\n      "lat": 55.8324713,\n      "lon": 37.5023435,\n      "opening_hours": null\n    },\n    {\n      "id": "way/558782674",\n      "kind": "school",\n      "name": "Школа № 1474. Подразделение 2",\n      "lat": 55.87335,\n      "lon": 37.4890118,\n      "opening_hours": null\n    },\n    {\n      "id": "way/558782684",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8646341,\n      "lon": 37.4964523,\n      "opening_hours": null\n    },\n    {\n      "id": "way/558785509",\n      "kind": "school",\n      "name": "ГБОУ г. Москвы Школа № 1159",\n      "lat": 55.8626161,\n      "lon": 37.5096781,\n      "opening_hours": null\n    },\n    {\n      "id": "way/561260076",\n      "kind": "kindergarten",\n      "name": "Детский сад №1960",\n      "lat": 55.8635938,\n      "lon": 37.6942832,\n      "opening_hours": null\n    },\n    {\n      "id": "way/568303123",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 9 Дошкольное отделение Семицветик",\n      "lat": 55.6658089,\n      "lon": 37.5208623,\n      "opening_hours": null\n    },\n    {\n      "id": "way/568709881",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №799, корпус 5",\n      "lat": 55.7714964,\n      "lon": 37.8284763,\n      "opening_hours": null\n    },\n    {\n      "id": "way/568709882",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №799, корпус 3",\n      "lat": 55.7705827,\n      "lon": 37.8249913,\n      "opening_hours": null\n    },\n    {\n      "id": "way/568709883",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №799, корпус 4",\n      "lat": 55.770562,\n      "lon": 37.8260427,\n      "opening_hours": null\n    },\n    {\n      "id": "way/568709885",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №799, корпус 6",\n      "lat": 55.7715139,\n      "lon": 37.8295964,\n      "opening_hours": null\n    },\n    {\n      "id": "way/575895642",\n      "kind": "kindergarten",\n      "name": "Детский сад №1327",\n      "lat": 55.8713826,\n      "lon": 37.5887819,\n      "opening_hours": null\n    },\n    {\n      "id": "way/576198916",\n      "kind": "school",\n      "name": "Школа №727",\n      "lat": 55.8471612,\n      "lon": 37.5166713,\n      "opening_hours": null\n    },\n    {\n      "id": "way/577047930",\n      "kind": "kindergarten",\n      "name": "Детский  сад «Ладушки»",\n      "lat": 55.8520529,\n      "lon": 37.5048415,\n      "opening_hours": null\n    },\n    {\n      "id": "way/577047933",\n      "kind": "kindergarten",\n      "name": "Академия  им. М.Ю. Лермонтова (Детский сад)",\n      "lat": 55.852289,\n      "lon": 37.5029705,\n      "opening_hours": null\n    },\n    {\n      "id": "way/577139276",\n      "kind": "kindergarten",\n      "name": "Школа № 1583 имени К. А. Керимова (Дошкольный корпус)",\n      "lat": 55.8534448,\n      "lon": 37.5110635,\n      "opening_hours": null\n    },\n    {\n      "id": "way/577139278",\n      "kind": "school",\n      "name": "Детская музыкальная школа №98",\n      "lat": 55.8529296,\n      "lon": 37.5109268,\n      "opening_hours": null\n    },\n    {\n      "id": "way/579520474",\n      "kind": "kindergarten",\n      "name": "Школа № 1862. Корпус дошкольного образования №9",\n      "lat": 55.6435195,\n      "lon": 37.6172907,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948080",\n      "kind": "school",\n      "name": "Школа №922 (учебный корпус \\"Терлецкий\\")",\n      "lat": 55.7598996,\n      "lon": 37.8273607,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948085",\n      "kind": "kindergarten",\n      "name": "Школа №922 (учебный корпус \\"Измайловский\\")",\n      "lat": 55.7628859,\n      "lon": 37.824001,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948086",\n      "kind": "kindergarten",\n      "name": "Дошкольный корпус ГБОУ Школа №922",\n      "lat": 55.7639292,\n      "lon": 37.8243023,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948087",\n      "kind": "kindergarten",\n      "name": "Школа №922 (учебный корпус \\"Екатерининский\\")",\n      "lat": 55.7641655,\n      "lon": 37.8302441,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948088",\n      "kind": "kindergarten",\n      "name": "Школа №922 (учебный корпус \\"Мещерский\\")",\n      "lat": 55.7647033,\n      "lon": 37.8285922,\n      "opening_hours": null\n    },\n    {\n      "id": "way/582948089",\n      "kind": "kindergarten",\n      "name": "Школа №922 (учебный корпус \\"Воронцовский\\")",\n      "lat": 55.7649301,\n      "lon": 37.8266818,\n      "opening_hours": null\n    },\n    {\n      "id": "way/584929543",\n      "kind": "kindergarten",\n      "name": "Дошкольный корпус №3 ГБОУ Школа №1476",\n      "lat": 55.7777351,\n      "lon": 37.8338801,\n      "opening_hours": null\n    },\n    {\n      "id": "way/584929546",\n      "kind": "kindergarten",\n      "name": "Дошкольный корпус №1 ГБОУ Школа №1476",\n      "lat": 55.7742978,\n      "lon": 37.8214623,\n      "opening_hours": null\n    },\n    {\n      "id": "way/584929558",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1373, корпус №3",\n      "lat": 55.7806757,\n      "lon": 37.8334947,\n      "opening_hours": null\n    },\n    {\n      "id": "way/584929561",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1373, корпус №5",\n      "lat": 55.7794952,\n      "lon": 37.8310343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/584929562",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №1373, корпус №4",\n      "lat": 55.7801494,\n      "lon": 37.8301517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586387137",\n      "kind": "kindergarten",\n      "name": "Бывшая территория детского сада 1727",\n      "lat": 55.7776719,\n      "lon": 37.5548097,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586790386",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6753566,\n      "lon": 37.6016615,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586790387",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6774801,\n      "lon": 37.603928,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586790388",\n      "kind": "kindergarten",\n      "name": "Детский сад 1406",\n      "lat": 55.6779575,\n      "lon": 37.6103831,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586805962",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6696608,\n      "lon": 37.5947719,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586808380",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6983863,\n      "lon": 37.5574606,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586808381",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.696696,\n      "lon": 37.5571806,\n      "opening_hours": null\n    },\n    {\n      "id": "way/586808383",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6944149,\n      "lon": 37.6047447,\n      "opening_hours": null\n    },\n    {\n      "id": "way/587445099",\n      "kind": "school",\n      "name": "Московский образовательный комплекс имени Виктора Талалихина. Структурное подразделение № 5",\n      "lat": 55.7472248,\n      "lon": 37.7211595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/588597975",\n      "kind": "kindergarten",\n      "name": "ДОУ №2009",\n      "lat": 55.7527536,\n      "lon": 37.5223742,\n      "opening_hours": null\n    },\n    {\n      "id": "way/588598072",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7516926,\n      "lon": 37.5229583,\n      "opening_hours": null\n    },\n    {\n      "id": "way/590221545",\n      "kind": "kindergarten",\n      "name": "Детский сад №965",\n      "lat": 55.8622524,\n      "lon": 37.6178373,\n      "opening_hours": null\n    },\n    {\n      "id": "way/592726680",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/8",\n      "lat": 55.6244529,\n      "lon": 37.486332,\n      "opening_hours": null\n    },\n    {\n      "id": "way/593891735",\n      "kind": "kindergarten",\n      "name": "Школа № 504. Дошкольное отделение «Радуга-1»",\n      "lat": 55.6672844,\n      "lon": 37.6250138,\n      "opening_hours": null\n    },\n    {\n      "id": "way/595122415",\n      "kind": "kindergarten",\n      "name": "Детский сад №26",\n      "lat": 55.6006711,\n      "lon": 37.596141,\n      "opening_hours": null\n    },\n    {\n      "id": "way/595420149",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6452053,\n      "lon": 37.722634,\n      "opening_hours": null\n    },\n    {\n      "id": "way/596215157",\n      "kind": "kindergarten",\n      "name": "Школа № 1770. Дошкольное отделение \\"Непоседы\\"",\n      "lat": 55.6878141,\n      "lon": 37.6864357,\n      "opening_hours": null\n    },\n    {\n      "id": "way/596216287",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6866057,\n      "lon": 37.6885882,\n      "opening_hours": null\n    },\n    {\n      "id": "way/596360348",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6829692,\n      "lon": 37.7051047,\n      "opening_hours": null\n    },\n    {\n      "id": "way/596370511",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 463 корпус №1 «Жемчужинка»",\n      "lat": 55.6780005,\n      "lon": 37.6722907,\n      "opening_hours": null\n    },\n    {\n      "id": "way/596370512",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 463 корпус №2 «Жемчужинка»",\n      "lat": 55.6779398,\n      "lon": 37.674095,\n      "opening_hours": null\n    },\n    {\n      "id": "way/598588576",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7469855,\n      "lon": 37.792693,\n      "opening_hours": null\n    },\n    {\n      "id": "way/601108319",\n      "kind": "kindergarten",\n      "name": "Школа № 657. Корпус дошкольного образования №3",\n      "lat": 55.6015997,\n      "lon": 37.6236916,\n      "opening_hours": null\n    },\n    {\n      "id": "way/605601044",\n      "kind": "school",\n      "name": "Школа № 1454 «Тимирязевская»",\n      "lat": 55.8333738,\n      "lon": 37.5701,\n      "opening_hours": null\n    },\n    {\n      "id": "way/607334891",\n      "kind": "kindergarten",\n      "name": "Детский сад №923",\n      "lat": 55.6919248,\n      "lon": 37.7247782,\n      "opening_hours": null\n    },\n    {\n      "id": "way/607339934",\n      "kind": "kindergarten",\n      "name": "Детский сад №968",\n      "lat": 55.6898083,\n      "lon": 37.7241153,\n      "opening_hours": null\n    },\n    {\n      "id": "way/608127845",\n      "kind": "kindergarten",\n      "name": "Детский сад № 789",\n      "lat": 55.610534,\n      "lon": 37.5871809,\n      "opening_hours": null\n    },\n    {\n      "id": "way/608378970",\n      "kind": "school",\n      "name": "Европейская гимназия с дошкольным отделением",\n      "lat": 55.7911068,\n      "lon": 37.6682859,\n      "opening_hours": null\n    },\n    {\n      "id": "way/608667848",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Мытной\\"",\n      "lat": 55.7186688,\n      "lon": 37.6164258,\n      "opening_hours": null\n    },\n    {\n      "id": "way/609317059",\n      "kind": "school",\n      "name": "Школа № 1770. Корпус школьного образования «Фрегат»",\n      "lat": 55.6824788,\n      "lon": 37.6957807,\n      "opening_hours": null\n    },\n    {\n      "id": "way/613303406",\n      "kind": "kindergarten",\n      "name": "Детский сад №64/2 “Яблонька”",\n      "lat": 55.8935497,\n      "lon": 37.7160892,\n      "opening_hours": null\n    },\n    {\n      "id": "way/614889264",\n      "kind": "kindergarten",\n      "name": "Школа №\xa067. Отделение дошкольного образования",\n      "lat": 55.7418818,\n      "lon": 37.5207871,\n      "opening_hours": null\n    },\n    {\n      "id": "way/617465057",\n      "kind": "kindergarten",\n      "name": "Детский сад «Светлые Горы»",\n      "lat": 55.5896809,\n      "lon": 37.6028563,\n      "opening_hours": null\n    },\n    {\n      "id": "way/618533762",\n      "kind": "kindergarten",\n      "name": "Детский сад №2035",\n      "lat": 55.8685276,\n      "lon": 37.6329645,\n      "opening_hours": null\n    },\n    {\n      "id": "way/623564253",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8196852,\n      "lon": 37.5131849,\n      "opening_hours": null\n    },\n    {\n      "id": "way/624186249",\n      "kind": "school",\n      "name": "Школа №\xa0252",\n      "lat": 55.8149961,\n      "lon": 37.5944483,\n      "opening_hours": null\n    },\n    {\n      "id": "way/624783579",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.877561,\n      "lon": 37.5084158,\n      "opening_hours": null\n    },\n    {\n      "id": "way/627999917",\n      "kind": "kindergarten",\n      "name": "Школа № 2099. Здание 5",\n      "lat": 55.8175812,\n      "lon": 37.4897852,\n      "opening_hours": null\n    },\n    {\n      "id": "way/628537533",\n      "kind": "kindergarten",\n      "name": "Школа №1576, дошкольный корпус №14",\n      "lat": 55.8418732,\n      "lon": 37.5458069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/632644537",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6723207,\n      "lon": 37.5338421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/632661091",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6721082,\n      "lon": 37.5356219,\n      "opening_hours": null\n    },\n    {\n      "id": "way/632661092",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.673893,\n      "lon": 37.5310235,\n      "opening_hours": null\n    },\n    {\n      "id": "way/635814039",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8928432,\n      "lon": 37.4438674,\n      "opening_hours": null\n    },\n    {\n      "id": "way/637885652",\n      "kind": "kindergarten",\n      "name": "Детский сад №358",\n      "lat": 55.6132369,\n      "lon": 37.7108268,\n      "opening_hours": null\n    },\n    {\n      "id": "way/637885653",\n      "kind": "school",\n      "name": "Интеллект XXI",\n      "lat": 55.6133997,\n      "lon": 37.7118931,\n      "opening_hours": null\n    },\n    {\n      "id": "way/640129189",\n      "kind": "kindergarten",\n      "name": "Школа № 498. Дошкольный корпус Тихий тупик, дом 2",\n      "lat": 55.7378528,\n      "lon": 37.6651454,\n      "opening_hours": null\n    },\n    {\n      "id": "way/641355965",\n      "kind": "school",\n      "name": "Кадетская школа №1770 (Московский кадетский музыкальный корпус)",\n      "lat": 55.676287,\n      "lon": 37.695895,\n      "opening_hours": null\n    },\n    {\n      "id": "way/642317749",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8843445,\n      "lon": 37.5610928,\n      "opening_hours": null\n    },\n    {\n      "id": "way/642492630",\n      "kind": "school",\n      "name": "Школа № 138. Корпус \\"Коллаж\\"",\n      "lat": 55.786609,\n      "lon": 37.4766223,\n      "opening_hours": null\n    },\n    {\n      "id": "way/643329688",\n      "kind": "school",\n      "name": "Гимназия №1517",\n      "lat": 55.7849844,\n      "lon": 37.4528217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/644322701",\n      "kind": "kindergarten",\n      "name": "Детский сад №346",\n      "lat": 55.695234,\n      "lon": 37.7627508,\n      "opening_hours": null\n    },\n    {\n      "id": "way/644322704",\n      "kind": "school",\n      "name": "ГБОУ СОШ № 479 Дошкольное отделение",\n      "lat": 55.6960289,\n      "lon": 37.7624681,\n      "opening_hours": null\n    },\n    {\n      "id": "way/644512212",\n      "kind": "kindergarten",\n      "name": "Школа №641 им. С. Есенина с дошкольным отделением",\n      "lat": 55.6972906,\n      "lon": 37.7542788,\n      "opening_hours": null\n    },\n    {\n      "id": "way/644895848",\n      "kind": "kindergarten",\n      "name": "Школа №947, Начальная школа им. А. де Сент-Экзюпери",\n      "lat": 55.5856179,\n      "lon": 37.671176,\n      "opening_hours": null\n    },\n    {\n      "id": "way/644895854",\n      "kind": "kindergarten",\n      "name": "Школа №947, Дошкольное отделение \\"Изумрудный город\\"",\n      "lat": 55.5848112,\n      "lon": 37.6701145,\n      "opening_hours": null\n    },\n    {\n      "id": "way/645305236",\n      "kind": "kindergarten",\n      "name": "Школа №641 им. С. Есенина с дошкольным отделением",\n      "lat": 55.7037793,\n      "lon": 37.7621648,\n      "opening_hours": null\n    },\n    {\n      "id": "way/645707233",\n      "kind": "kindergarten",\n      "name": "Школа №1420 с дошкольным отделением",\n      "lat": 55.7007703,\n      "lon": 37.7891796,\n      "opening_hours": null\n    },\n    {\n      "id": "way/645707237",\n      "kind": "school",\n      "name": "Школа №1208 им. Героя Советского Союза М.С. Шумилова",\n      "lat": 55.6987748,\n      "lon": 37.7848959,\n      "opening_hours": null\n    },\n    {\n      "id": "way/647308761",\n      "kind": "kindergarten",\n      "name": "Школа № 2115. Корпус дошкольного образования № 6",\n      "lat": 55.670781,\n      "lon": 37.5638596,\n      "opening_hours": null\n    },\n    {\n      "id": "way/647423465",\n      "kind": "kindergarten",\n      "name": "Детский сад №817",\n      "lat": 55.716691,\n      "lon": 37.7795046,\n      "opening_hours": null\n    },\n    {\n      "id": "way/648098841",\n      "kind": "school",\n      "name": "Школа № 962. Структурное подразделение № 4",\n      "lat": 55.8650286,\n      "lon": 37.590761,\n      "opening_hours": null\n    },\n    {\n      "id": "way/648540752",\n      "kind": "kindergarten",\n      "name": "Детский сад №1013",\n      "lat": 55.6856803,\n      "lon": 37.7389819,\n      "opening_hours": null\n    },\n    {\n      "id": "way/650087440",\n      "kind": "school",\n      "name": "Школа №810",\n      "lat": 55.7244069,\n      "lon": 37.8054149,\n      "opening_hours": null\n    },\n    {\n      "id": "way/651507354",\n      "kind": "kindergarten",\n      "name": "Школа №1078 (Модуль №3)",\n      "lat": 55.8185884,\n      "lon": 37.8004966,\n      "opening_hours": null\n    },\n    {\n      "id": "way/656569636",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6452536,\n      "lon": 37.7252021,\n      "opening_hours": null\n    },\n    {\n      "id": "way/656685223",\n      "kind": "school",\n      "name": "ГБОУ Школа №825",\n      "lat": 55.7100413,\n      "lon": 37.7540253,\n      "opening_hours": null\n    },\n    {\n      "id": "way/656929750",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1613",\n      "lat": 55.7209396,\n      "lon": 37.7809325,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659500454",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №19",\n      "lat": 55.8241823,\n      "lon": 37.8060986,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659500458",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №8",\n      "lat": 55.8269188,\n      "lon": 37.8269476,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659500459",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №9",\n      "lat": 55.8224627,\n      "lon": 37.8315203,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659500462",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №7",\n      "lat": 55.8224695,\n      "lon": 37.8298585,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659500463",\n      "kind": "kindergarten",\n      "name": "Школа №1598. Здание №10",\n      "lat": 55.8224754,\n      "lon": 37.8283767,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659608154",\n      "kind": "kindergarten",\n      "name": "Школа № 1363, ДО Мозаика",\n      "lat": 55.7090544,\n      "lon": 37.820927,\n      "opening_hours": null\n    },\n    {\n      "id": "way/659608346",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1363, корпус № 3",\n      "lat": 55.7083047,\n      "lon": 37.8217562,\n      "opening_hours": null\n    },\n    {\n      "id": "way/661148268",\n      "kind": "kindergarten",\n      "name": "Дошкольное подразделение школы №258",\n      "lat": 55.8644397,\n      "lon": 37.6108039,\n      "opening_hours": null\n    },\n    {\n      "id": "way/661148271",\n      "kind": "school",\n      "name": "Гимназия №1554",\n      "lat": 55.8659096,\n      "lon": 37.6118085,\n      "opening_hours": null\n    },\n    {\n      "id": "way/661593268",\n      "kind": "kindergarten",\n      "name": "Школа № 1770. Корпус дошкольного образования «Бригантина»",\n      "lat": 55.6836191,\n      "lon": 37.6965265,\n      "opening_hours": null\n    },\n    {\n      "id": "way/669759734",\n      "kind": "school",\n      "name": "Школа № 1101. Здание 2/2",\n      "lat": 55.6347404,\n      "lon": 37.4871522,\n      "opening_hours": null\n    },\n    {\n      "id": "way/674812006",\n      "kind": "school",\n      "name": "Школа № 814. Корпус 5",\n      "lat": 55.7140342,\n      "lon": 37.4814636,\n      "opening_hours": null\n    },\n    {\n      "id": "way/682070168",\n      "kind": "kindergarten",\n      "name": "ГБОУ Школа №152 дошкольное отделение",\n      "lat": 55.8068595,\n      "lon": 37.5337671,\n      "opening_hours": null\n    },\n    {\n      "id": "way/683111302",\n      "kind": "school",\n      "name": "Школа №\xa02025. Здание 3",\n      "lat": 55.6743777,\n      "lon": 37.4550086,\n      "opening_hours": null\n    },\n    {\n      "id": "way/684216820",\n      "kind": "school",\n      "name": "Школа Марьина роща им. В. Ф. Орлова",\n      "lat": 55.789119,\n      "lon": 37.6158138,\n      "opening_hours": null\n    },\n    {\n      "id": "way/684938116",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.6980046,\n      "lon": 37.4892972,\n      "opening_hours": null\n    },\n    {\n      "id": "way/686660334",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №2 \\"Улыбка\\"",\n      "lat": 55.8082904,\n      "lon": 37.8276224,\n      "opening_hours": null\n    },\n    {\n      "id": "way/686669814",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №3 \\"Умка\\"",\n      "lat": 55.8064227,\n      "lon": 37.8277862,\n      "opening_hours": null\n    },\n    {\n      "id": "way/686938091",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №1 \\"Теремок\\"",\n      "lat": 55.8103014,\n      "lon": 37.8274037,\n      "opening_hours": null\n    },\n    {\n      "id": "way/690243295",\n      "kind": "school",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №4",\n      "lat": 55.8083844,\n      "lon": 37.8216907,\n      "opening_hours": null\n    },\n    {\n      "id": "way/690812150",\n      "kind": "school",\n      "name": "Школа № 1494. Классы начального, основного и среднего общего образования",\n      "lat": 55.8450925,\n      "lon": 37.5816088,\n      "opening_hours": null\n    },\n    {\n      "id": "way/691065085",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1353",\n      "lat": 55.8444875,\n      "lon": 37.5862854,\n      "opening_hours": null\n    },\n    {\n      "id": "way/693839893",\n      "kind": "kindergarten",\n      "name": "Школа № 1245. Корпус дошкольного образования №5",\n      "lat": 55.6009517,\n      "lon": 37.5937766,\n      "opening_hours": null\n    },\n    {\n      "id": "way/694729851",\n      "kind": "kindergarten",\n      "name": "Школа № 37. Дошкольное отделение",\n      "lat": 55.7045586,\n      "lon": 37.5083282,\n      "opening_hours": null\n    },\n    {\n      "id": "way/704215334",\n      "kind": "school",\n      "name": "Школа № 830. Корпус № 2",\n      "lat": 55.8269759,\n      "lon": 37.4546993,\n      "opening_hours": null\n    },\n    {\n      "id": "way/704357047",\n      "kind": "school",\n      "name": "Московский кадетский корпус полиции",\n      "lat": 55.8441505,\n      "lon": 37.5445622,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708821993",\n      "kind": "kindergarten",\n      "name": "Детский сад №1907 (ортопедический)",\n      "lat": 55.7224953,\n      "lon": 37.8177979,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708821994",\n      "kind": "kindergarten",\n      "name": "Детский сад №1307",\n      "lat": 55.7226758,\n      "lon": 37.8154802,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708821995",\n      "kind": "kindergarten",\n      "name": "Детский сад №1287",\n      "lat": 55.7190083,\n      "lon": 37.8168477,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708833031",\n      "kind": "kindergarten",\n      "name": "Дом ребенка №15",\n      "lat": 55.7361265,\n      "lon": 37.8335538,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708833032",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольное отделение К6",\n      "lat": 55.7429381,\n      "lon": 37.8235109,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708833033",\n      "kind": "school",\n      "name": "Вешняковская школа. Школьное отделение К8",\n      "lat": 55.7429746,\n      "lon": 37.825451,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708833035",\n      "kind": "school",\n      "name": "Вешняковская школа. Школьное отделение К10",\n      "lat": 55.7427065,\n      "lon": 37.8280743,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708929428",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8984304,\n      "lon": 37.6212814,\n      "opening_hours": null\n    },\n    {\n      "id": "way/708929429",\n      "kind": "kindergarten",\n      "name": "Детский сад №820",\n      "lat": 55.8921888,\n      "lon": 37.5961004,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709206068",\n      "kind": "school",\n      "name": "Школа №1502 \\"Энергия\\"",\n      "lat": 55.7919408,\n      "lon": 37.7132723,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709227297",\n      "kind": "kindergarten",\n      "name": "Детский сад №1195",\n      "lat": 55.8558715,\n      "lon": 37.6175702,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709227298",\n      "kind": "kindergarten",\n      "name": "Детский сад №1195",\n      "lat": 55.8565466,\n      "lon": 37.6184917,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709227304",\n      "kind": "school",\n      "name": "Школа №959",\n      "lat": 55.8606277,\n      "lon": 37.6087716,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709231665",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1128",\n      "lat": 55.8635375,\n      "lon": 37.6166178,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709231668",\n      "kind": "school",\n      "name": "Гимназия №1554",\n      "lat": 55.8648387,\n      "lon": 37.6124939,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709231669",\n      "kind": "school",\n      "name": "Школа №258",\n      "lat": 55.8655541,\n      "lon": 37.6097456,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709236497",\n      "kind": "school",\n      "name": "Пенаты",\n      "lat": 55.8542638,\n      "lon": 37.5880798,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709265694",\n      "kind": "kindergarten",\n      "name": "Детский сад №1282",\n      "lat": 55.8692612,\n      "lon": 37.6414764,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709265695",\n      "kind": "school",\n      "name": "Разум-Л",\n      "lat": 55.8694434,\n      "lon": 37.6431078,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709265696",\n      "kind": "school",\n      "name": "Центр образования №1482",\n      "lat": 55.8672369,\n      "lon": 37.6357424,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709276384",\n      "kind": "kindergarten",\n      "name": "Детский сад №665",\n      "lat": 55.8918395,\n      "lon": 37.6571627,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709276385",\n      "kind": "school",\n      "name": "Детский дом № 5",\n      "lat": 55.89074,\n      "lon": 37.6502373,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709548475",\n      "kind": "school",\n      "name": "Гимназия №1506",\n      "lat": 55.889765,\n      "lon": 37.6504661,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709548479",\n      "kind": "school",\n      "name": "ЧУ СОШ \\"Надежда\\"",\n      "lat": 55.8842774,\n      "lon": 37.609371,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709548480",\n      "kind": "kindergarten",\n      "name": "Детский сад Надежда",\n      "lat": 55.8846808,\n      "lon": 37.610653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709548485",\n      "kind": "kindergarten",\n      "name": "Детский сад №932",\n      "lat": 55.8919058,\n      "lon": 37.6021049,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709568857",\n      "kind": "kindergarten",\n      "name": "Социальный приют",\n      "lat": 55.6269318,\n      "lon": 37.7252742,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709568859",\n      "kind": "school",\n      "name": "Центр образования № 548",\n      "lat": 55.6237667,\n      "lon": 37.7015439,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580265",\n      "kind": "kindergarten",\n      "name": "Школа № 667. Корпус дошкольного образования №7",\n      "lat": 55.5856782,\n      "lon": 37.6448971,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580266",\n      "kind": "school",\n      "name": "Школа №933",\n      "lat": 55.5837199,\n      "lon": 37.6493815,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580269",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.592495,\n      "lon": 37.6445906,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580271",\n      "kind": "kindergarten",\n      "name": "Детский сад №2471",\n      "lat": 55.5927515,\n      "lon": 37.6419358,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580272",\n      "kind": "school",\n      "name": "Школа №2001",\n      "lat": 55.5908387,\n      "lon": 37.6401232,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709580273",\n      "kind": "kindergarten",\n      "name": "Детский сад №972",\n      "lat": 55.5897339,\n      "lon": 37.6430669,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709588441",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.5913847,\n      "lon": 37.6051539,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709647798",\n      "kind": "school",\n      "name": "Гимназия №1554",\n      "lat": 55.8645463,\n      "lon": 37.6234977,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709647799",\n      "kind": "kindergarten",\n      "name": "Детский сад №1524",\n      "lat": 55.8627249,\n      "lon": 37.5875968,\n      "opening_hours": null\n    },\n    {\n      "id": "way/709647800",\n      "kind": "kindergarten",\n      "name": "Детский сад №1933",\n      "lat": 55.7939269,\n      "lon": 37.7229808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/710644450",\n      "kind": "kindergarten",\n      "name": "Школа № 627 имени генерала Д. Д. Лелюшенко. Учебный корпус № 7",\n      "lat": 55.7272136,\n      "lon": 37.6210533,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279699",\n      "kind": "kindergarten",\n      "name": "Школа №947, Дошкольное отделение \\"Цветочный город\\"",\n      "lat": 55.5850066,\n      "lon": 37.6760841,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279700",\n      "kind": "school",\n      "name": "Школа №947, Начальная школа им Л. да Винчи",\n      "lat": 55.5879099,\n      "lon": 37.6673035,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279701",\n      "kind": "kindergarten",\n      "name": "Школа №947, Дошкольное отделение \\"Лукоморье\\"",\n      "lat": 55.5875779,\n      "lon": 37.6713781,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279703",\n      "kind": "school",\n      "name": "Школа № 902 \\"Диалог\\", здание №1",\n      "lat": 55.5963153,\n      "lon": 37.6654733,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279704",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №6",\n      "lat": 55.5964405,\n      "lon": 37.6676727,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279705",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №7",\n      "lat": 55.5965485,\n      "lon": 37.6690315,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279706",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №4",\n      "lat": 55.5991447,\n      "lon": 37.6669189,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279707",\n      "kind": "kindergarten",\n      "name": "Школа № 902 \\"Диалог\\", здание №5",\n      "lat": 55.5992447,\n      "lon": 37.6684337,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279708",\n      "kind": "kindergarten",\n      "name": "Детский сад №960",\n      "lat": 55.597429,\n      "lon": 37.6574516,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279709",\n      "kind": "kindergarten",\n      "name": "Детский сад №1972",\n      "lat": 55.5983525,\n      "lon": 37.6569542,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711279710",\n      "kind": "school",\n      "name": "Школа №508",\n      "lat": 55.6005267,\n      "lon": 37.6554268,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287578",\n      "kind": "kindergarten",\n      "name": "Детский сад №2232",\n      "lat": 55.5838878,\n      "lon": 37.6513664,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287579",\n      "kind": "kindergarten",\n      "name": "Детский сад №667",\n      "lat": 55.5853673,\n      "lon": 37.6511133,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287580",\n      "kind": "kindergarten",\n      "name": "Детский сад №834",\n      "lat": 55.5858187,\n      "lon": 37.6532842,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287581",\n      "kind": "kindergarten",\n      "name": "Детский сад №901",\n      "lat": 55.5904482,\n      "lon": 37.649388,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287582",\n      "kind": "kindergarten",\n      "name": "Детский сад №486",\n      "lat": 55.5931886,\n      "lon": 37.647639,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287583",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.5999399,\n      "lon": 37.6149476,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287584",\n      "kind": "kindergarten",\n      "name": "Школа № 924. Корпус дошкольного образования №4",\n      "lat": 55.600395,\n      "lon": 37.6129039,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711287585",\n      "kind": "school",\n      "name": "Школа № 924. Корпус школьного образования",\n      "lat": 55.6005052,\n      "lon": 37.6109571,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711301314",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7248448,\n      "lon": 37.801857,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304801",\n      "kind": "kindergarten",\n      "name": "Школа № 1245. Корпус дошкольного образования №7",\n      "lat": 55.6049098,\n      "lon": 37.6032411,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304802",\n      "kind": "kindergarten",\n      "name": "Школа № 1245. Корпус дошкольного образования №8",\n      "lat": 55.6078665,\n      "lon": 37.604734,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304803",\n      "kind": "kindergarten",\n      "name": "Детский сад №1318",\n      "lat": 55.6063973,\n      "lon": 37.5849161,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304804",\n      "kind": "kindergarten",\n      "name": "Детский сад №990",\n      "lat": 55.6094234,\n      "lon": 37.5811713,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304805",\n      "kind": "school",\n      "name": "Школа №652",\n      "lat": 55.6110709,\n      "lon": 37.5814015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711304806",\n      "kind": "kindergarten",\n      "name": "Школа №1179. Корпус дошкольного образования №2",\n      "lat": 55.6391561,\n      "lon": 37.6128891,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306802",\n      "kind": "school",\n      "name": "Школа №878 отделение №3",\n      "lat": 55.6169734,\n      "lon": 37.7087544,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306803",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1502",\n      "lat": 55.6223831,\n      "lon": 37.708636,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306804",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1513",\n      "lat": 55.6246774,\n      "lon": 37.7047102,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306805",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1637",\n      "lat": 55.626826,\n      "lon": 37.715952,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306806",\n      "kind": "kindergarten",\n      "name": "Детский сад №905",\n      "lat": 55.6238256,\n      "lon": 37.72416,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711306807",\n      "kind": "school",\n      "name": "Школа №939",\n      "lat": 55.6223236,\n      "lon": 37.723614,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711388921",\n      "kind": "kindergarten",\n      "name": "Детский сад №1709",\n      "lat": 55.6106533,\n      "lon": 37.7265286,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711388922",\n      "kind": "school",\n      "name": "Гимназия №1636 НИКА",\n      "lat": 55.6110785,\n      "lon": 37.7288174,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711588628",\n      "kind": "kindergarten",\n      "name": "Детский сад №413",\n      "lat": 55.8077059,\n      "lon": 37.7116342,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711588631",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8967409,\n      "lon": 37.6288236,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711588632",\n      "kind": "kindergarten",\n      "name": "Детский сад №531",\n      "lat": 55.8964179,\n      "lon": 37.6302669,\n      "opening_hours": null\n    },\n    {\n      "id": "way/711588633",\n      "kind": "kindergarten",\n      "name": "Школа Глория, корпус № 8",\n      "lat": 55.8953672,\n      "lon": 37.6278932,\n      "opening_hours": null\n    },\n    {\n      "id": "way/713630942",\n      "kind": "kindergarten",\n      "name": "Школа №2094. Дошкольная образовательная площадка №3",\n      "lat": 55.5906259,\n      "lon": 37.453851,\n      "opening_hours": null\n    },\n    {\n      "id": "way/722407298",\n      "kind": "school",\n      "name": "ОК \\"Юго-Запад\\"",\n      "lat": 55.68364,\n      "lon": 37.5932506,\n      "opening_hours": null\n    },\n    {\n      "id": "way/722633303",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 1272 Академия детства",\n      "lat": 55.701769,\n      "lon": 37.6613565,\n      "opening_hours": null\n    },\n    {\n      "id": "way/723104569",\n      "kind": "school",\n      "name": "Школа № 548 «Царицыно». Корпус школьного образования «ЗилАрт»",\n      "lat": 55.7010224,\n      "lon": 37.6387796,\n      "opening_hours": null\n    },\n    {\n      "id": "way/734561897",\n      "kind": "kindergarten",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г. П. Кравченко. Здание № 4",\n      "lat": 55.7113463,\n      "lon": 37.5979744,\n      "opening_hours": null\n    },\n    {\n      "id": "way/744280212",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение школы №224",\n      "lat": 55.8316249,\n      "lon": 37.5025618,\n      "opening_hours": null\n    },\n    {\n      "id": "way/744569171",\n      "kind": "kindergarten",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 8.",\n      "lat": 55.6470461,\n      "lon": 37.5209714,\n      "opening_hours": null\n    },\n    {\n      "id": "way/746494190",\n      "kind": "school",\n      "name": "Школа №978",\n      "lat": 55.6767103,\n      "lon": 37.6314956,\n      "opening_hours": null\n    },\n    {\n      "id": "way/756873354",\n      "kind": "school",\n      "name": "Школа № 15. Корпус школьного образования № 2",\n      "lat": 55.6552954,\n      "lon": 37.5624595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/756877742",\n      "kind": "kindergarten",\n      "name": "Школа № 15. Корпус дошкольного образования № 2",\n      "lat": 55.6517768,\n      "lon": 37.5627347,\n      "opening_hours": null\n    },\n    {\n      "id": "way/760941887",\n      "kind": "school",\n      "name": "Школа №1601 им. Героя Советского Союза Е. К. Лютикова",\n      "lat": 55.7981571,\n      "lon": 37.5595217,\n      "opening_hours": null\n    },\n    {\n      "id": "way/773586915",\n      "kind": "school",\n      "name": "ГБОУ города Москвы «Школа № 1601 имени Героя Советского Союза Е. К. Лютикова»",\n      "lat": 55.7918953,\n      "lon": 37.567021,\n      "opening_hours": null\n    },\n    {\n      "id": "way/777144344",\n      "kind": "kindergarten",\n      "name": "Школа № 1222 имени Маршала Советского Союза И. Х. Баграмяна. Дошкольное  отделение № 2",\n      "lat": 55.7322736,\n      "lon": 37.7219154,\n      "opening_hours": "Mo-Su 08:00-19:00"\n    },\n    {\n      "id": "way/784611808",\n      "kind": "school",\n      "name": "Школа на Яузе",\n      "lat": 55.7748598,\n      "lon": 37.7085442,\n      "opening_hours": null\n    },\n    {\n      "id": "way/785862762",\n      "kind": "school",\n      "name": "Школа №\xa0568",\n      "lat": 55.6756404,\n      "lon": 37.5959945,\n      "opening_hours": null\n    },\n    {\n      "id": "way/805810950",\n      "kind": "kindergarten",\n      "name": "Детский сад №1108",\n      "lat": 55.7208497,\n      "lon": 37.5770152,\n      "opening_hours": null\n    },\n    {\n      "id": "way/805810951",\n      "kind": "kindergarten",\n      "name": "Детский сад №1093",\n      "lat": 55.720705,\n      "lon": 37.575624,\n      "opening_hours": null\n    },\n    {\n      "id": "way/806227829",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7187018,\n      "lon": 37.5786616,\n      "opening_hours": null\n    },\n    {\n      "id": "way/806227830",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7176747,\n      "lon": 37.5772695,\n      "opening_hours": null\n    },\n    {\n      "id": "way/813559738",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Европейская гимназия\\"",\n      "lat": 55.7865816,\n      "lon": 37.6840973,\n      "opening_hours": null\n    },\n    {\n      "id": "way/814267642",\n      "kind": "school",\n      "name": "СААШ \\"Марина\\"",\n      "lat": 55.6820311,\n      "lon": 37.5415069,\n      "opening_hours": null\n    },\n    {\n      "id": "way/814995890",\n      "kind": "kindergarten",\n      "name": "Детский сад №1708",\n      "lat": 55.6795924,\n      "lon": 37.535807,\n      "opening_hours": null\n    },\n    {\n      "id": "way/820333794",\n      "kind": "school",\n      "name": "Школа № 1507. Корпус школьного образования",\n      "lat": 55.6329838,\n      "lon": 37.509036,\n      "opening_hours": null\n    },\n    {\n      "id": "way/824290373",\n      "kind": "kindergarten",\n      "name": "Школа № 1321 \\"Ковчег\\". Дошкольное отделение здание №3",\n      "lat": 55.750761,\n      "lon": 37.7186581,\n      "opening_hours": null\n    },\n    {\n      "id": "way/845403776",\n      "kind": "kindergarten",\n      "name": "Школа № 1103. Корпус дошкольного образования №5",\n      "lat": 55.6061184,\n      "lon": 37.5404105,\n      "opening_hours": null\n    },\n    {\n      "id": "way/851409274",\n      "kind": "school",\n      "name": "Прогимназия №1774",\n      "lat": 55.8197589,\n      "lon": 37.5111443,\n      "opening_hours": null\n    },\n    {\n      "id": "way/853678162",\n      "kind": "kindergarten",\n      "name": "Детский сад №1446",\n      "lat": 55.8275213,\n      "lon": 37.575913,\n      "opening_hours": null\n    },\n    {\n      "id": "way/860122160",\n      "kind": "kindergarten",\n      "name": "Детский сад №1746",\n      "lat": 55.6822542,\n      "lon": 37.7417905,\n      "opening_hours": null\n    },\n    {\n      "id": "way/867853165",\n      "kind": "kindergarten",\n      "name": "Детский сад №2142",\n      "lat": 55.8235066,\n      "lon": 37.6492761,\n      "opening_hours": null\n    },\n    {\n      "id": "way/870571581",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №7 \\"Солнышко\\"",\n      "lat": 55.810496,\n      "lon": 37.8218343,\n      "opening_hours": null\n    },\n    {\n      "id": "way/874223859",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 10 (дошкольное отделение)",\n      "lat": 55.8060129,\n      "lon": 37.814694,\n      "opening_hours": null\n    },\n    {\n      "id": "way/874415684",\n      "kind": "kindergarten",\n      "name": "Школа № 2107. Здание школы \\"На Напрудном\\"",\n      "lat": 55.7886075,\n      "lon": 37.63108,\n      "opening_hours": null\n    },\n    {\n      "id": "way/874475981",\n      "kind": "school",\n      "name": "Школа №2200 корпус 8",\n      "lat": 55.8039545,\n      "lon": 37.8153901,\n      "opening_hours": null\n    },\n    {\n      "id": "way/874983433",\n      "kind": "school",\n      "name": "Школа №2033 (начальная школа)",\n      "lat": 55.8059649,\n      "lon": 37.8104058,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878383228",\n      "kind": "kindergarten",\n      "name": "Школа №1811 \\"Восточное Измайлово\\" корпус 11",\n      "lat": 55.7977501,\n      "lon": 37.8199268,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878384754",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8003224,\n      "lon": 37.813406,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878548821",\n      "kind": "kindergarten",\n      "name": "Школа №1748 \\"Вертикаль\\". Корпус №8 \\"Золотая рыбка\\"",\n      "lat": 55.8095872,\n      "lon": 37.82048,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878690744",\n      "kind": "kindergarten",\n      "name": "Школа №2200 корпус 11 (дошкольное отделение)",\n      "lat": 55.8060318,\n      "lon": 37.815974,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878979417",\n      "kind": "school",\n      "name": "Школа №1290",\n      "lat": 55.7939863,\n      "lon": 37.7931653,\n      "opening_hours": null\n    },\n    {\n      "id": "way/878979512",\n      "kind": "school",\n      "name": "Измайловская гимназия №1508",\n      "lat": 55.7939085,\n      "lon": 37.7962294,\n      "opening_hours": null\n    },\n    {\n      "id": "way/879568393",\n      "kind": "school",\n      "name": "Школа № 1222 имени Маршала Советского Союза И. Х. Баграмяна. Школьное отделение № 2",\n      "lat": 55.7322892,\n      "lon": 37.7200179,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "way/879949379",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Школьный корпус на Яснополянской",\n      "lat": 55.7231819,\n      "lon": 37.7903059,\n      "opening_hours": null\n    },\n    {\n      "id": "way/880734466",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Радость\\"",\n      "lat": 55.7306236,\n      "lon": 37.766762,\n      "opening_hours": null\n    },\n    {\n      "id": "way/880734467",\n      "kind": "kindergarten",\n      "name": "Образовательная школа \\"Академическая гимназия\\"",\n      "lat": 55.7301615,\n      "lon": 37.7684411,\n      "opening_hours": null\n    },\n    {\n      "id": "way/880735669",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Школьный корпус на Маёвок",\n      "lat": 55.7285241,\n      "lon": 37.7696011,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881052775",\n      "kind": "kindergarten",\n      "name": "Детский сад №872",\n      "lat": 55.6878981,\n      "lon": 37.7212016,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881124918",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Школьный корпус на Шатурской",\n      "lat": 55.7300836,\n      "lon": 37.765602,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881223214",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Фокус\\"",\n      "lat": 55.7222851,\n      "lon": 37.7853544,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881223348",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Семицветик\\"",\n      "lat": 55.721046,\n      "lon": 37.7843286,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881231366",\n      "kind": "school",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Школьный корпус на Михайлова",\n      "lat": 55.725454,\n      "lon": 37.7682396,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881232227",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Звёздочка\\"",\n      "lat": 55.7248986,\n      "lon": 37.7699148,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881232954",\n      "kind": "kindergarten",\n      "name": "Школа № 777 имени Героя Советского Союза Е. В. Михайлова. Дошкольный корпус \\"Лукоморье\\"",\n      "lat": 55.7274065,\n      "lon": 37.760757,\n      "opening_hours": null\n    },\n    {\n      "id": "way/881325381",\n      "kind": "kindergarten",\n      "name": "Детский сад №2027",\n      "lat": 55.7514368,\n      "lon": 37.8289433,\n      "opening_hours": null\n    },\n    {\n      "id": "way/908462514",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6518802,\n      "lon": 37.7114786,\n      "opening_hours": null\n    },\n    {\n      "id": "way/929277115",\n      "kind": "school",\n      "name": "ГБОУ СОШ № 2105 \\"Школа на Яузе\\" корпус №1",\n      "lat": 55.7789746,\n      "lon": 37.7062422,\n      "opening_hours": null\n    },\n    {\n      "id": "way/937806650",\n      "kind": "kindergarten",\n      "name": "Школа № 1362. Корпус дошкольного образования",\n      "lat": 55.7754397,\n      "lon": 37.736683,\n      "opening_hours": "Mo-Fr 07:00-20:00"\n    },\n    {\n      "id": "way/943940531",\n      "kind": "school",\n      "name": "Школа №1498 «ММШ-23». Корпус школьного образования",\n      "lat": 55.6946266,\n      "lon": 37.506344,\n      "opening_hours": null\n    },\n    {\n      "id": "way/945400139",\n      "kind": "kindergarten",\n      "name": "Лицей №1571 дошкольное отделение Лучики",\n      "lat": 55.8546815,\n      "lon": 37.4409175,\n      "opening_hours": null\n    },\n    {\n      "id": "way/946129394",\n      "kind": "kindergarten",\n      "name": "Школа №1056 (дошкольное отделение №2)",\n      "lat": 55.8470934,\n      "lon": 37.4410136,\n      "opening_hours": null\n    },\n    {\n      "id": "way/952069499",\n      "kind": "kindergarten",\n      "name": "Школа № 1517. Дошкольное отделение",\n      "lat": 55.7799074,\n      "lon": 37.4588802,\n      "opening_hours": null\n    },\n    {\n      "id": "way/952784277",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6218133,\n      "lon": 37.6534386,\n      "opening_hours": null\n    },\n    {\n      "id": "way/952784421",\n      "kind": "kindergarten",\n      "name": "Д/C № 1168",\n      "lat": 55.6221106,\n      "lon": 37.6549616,\n      "opening_hours": null\n    },\n    {\n      "id": "way/958193582",\n      "kind": "kindergarten",\n      "name": "Школа № 1272. Корпус дошкольного образования",\n      "lat": 55.7013421,\n      "lon": 37.6362218,\n      "opening_hours": null\n    },\n    {\n      "id": "way/964951015",\n      "kind": "kindergarten",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 10.",\n      "lat": 55.6503467,\n      "lon": 37.5277066,\n      "opening_hours": null\n    },\n    {\n      "id": "way/964951021",\n      "kind": "school",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 5.",\n      "lat": 55.6502546,\n      "lon": 37.5252847,\n      "opening_hours": null\n    },\n    {\n      "id": "way/964951027",\n      "kind": "kindergarten",\n      "name": "Школа № 170 имени А. П. Чехова. Корпус 9.",\n      "lat": 55.6510375,\n      "lon": 37.5241165,\n      "opening_hours": null\n    },\n    {\n      "id": "way/965961813",\n      "kind": "kindergarten",\n      "name": "Школа № 1101. Дошкольное отделение «Фруктовая радуга»",\n      "lat": 55.6266,\n      "lon": 37.4800534,\n      "opening_hours": null\n    },\n    {\n      "id": "way/965961825",\n      "kind": "school",\n      "name": "Школа № 1101. Здание 2/3",\n      "lat": 55.6286959,\n      "lon": 37.4816317,\n      "opening_hours": null\n    },\n    {\n      "id": "way/965961888",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6336695,\n      "lon": 37.4767149,\n      "opening_hours": null\n    },\n    {\n      "id": "way/966197845",\n      "kind": "school",\n      "name": "Школа № 1694 «Ясенево». Здание 1",\n      "lat": 55.6130815,\n      "lon": 37.5450963,\n      "opening_hours": null\n    },\n    {\n      "id": "way/966227910",\n      "kind": "kindergarten",\n      "name": "Школа № 1561. Дошкольное отделение корпус №6",\n      "lat": 55.5971756,\n      "lon": 37.5430883,\n      "opening_hours": null\n    },\n    {\n      "id": "way/966227926",\n      "kind": "school",\n      "name": "Школа №1561, начальная школа, корпус 4",\n      "lat": 55.6019397,\n      "lon": 37.5438795,\n      "opening_hours": null\n    },\n    {\n      "id": "way/967847042",\n      "kind": "kindergarten",\n      "name": "Школа № 113. Корпус дошкольного образования №4",\n      "lat": 55.6383727,\n      "lon": 37.5136462,\n      "opening_hours": null\n    },\n    {\n      "id": "way/967917167",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6232121,\n      "lon": 37.5037784,\n      "opening_hours": null\n    },\n    {\n      "id": "way/967917171",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6237342,\n      "lon": 37.5022421,\n      "opening_hours": null\n    },\n    {\n      "id": "way/967917190",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6273145,\n      "lon": 37.5084937,\n      "opening_hours": null\n    },\n    {\n      "id": "way/967917191",\n      "kind": "kindergarten",\n      "name": "Школа № 1507. Корпус дошкольного образования",\n      "lat": 55.6278657,\n      "lon": 37.5069219,\n      "opening_hours": null\n    },\n    {\n      "id": "way/972027575",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/2",\n      "lat": 55.6150753,\n      "lon": 37.4935368,\n      "opening_hours": null\n    },\n    {\n      "id": "way/972027578",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/3",\n      "lat": 55.6179576,\n      "lon": 37.4985487,\n      "opening_hours": null\n    },\n    {\n      "id": "way/972027586",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/4",\n      "lat": 55.6176937,\n      "lon": 37.4964275,\n      "opening_hours": null\n    },\n    {\n      "id": "way/972027597",\n      "kind": "kindergarten",\n      "name": "Школа №1532. Корпус дошкольного образования №05/6",\n      "lat": 55.6198426,\n      "lon": 37.4898063,\n      "opening_hours": null\n    },\n    {\n      "id": "way/972047403",\n      "kind": "kindergarten",\n      "name": "Школа №2094. Дошкольная образовательная площадка №2",\n      "lat": 55.6125285,\n      "lon": 37.4724735,\n      "opening_hours": null\n    },\n    {\n      "id": "way/975646976",\n      "kind": "school",\n      "name": "Школа №90 «Многопрофильный образовательный комплекс» имени Героя Советского Союза Е. Г. Ларикова",\n      "lat": 55.8220524,\n      "lon": 37.5041657,\n      "opening_hours": null\n    },\n    {\n      "id": "way/976037691",\n      "kind": "kindergarten",\n      "name": "Школа им. Н.М. Карамзина. Корпус дошкольного образования №6 «Андреевский»",\n      "lat": 55.59211,\n      "lon": 37.5376585,\n      "opening_hours": null\n    },\n    {\n      "id": "way/976678942",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 224 дошкольное отделение",\n      "lat": 55.8348462,\n      "lon": 37.5067296,\n      "opening_hours": null\n    },\n    {\n      "id": "way/977876539",\n      "kind": "school",\n      "name": "Школа № 630 имени дважды Героя Советского Союза Г.П. Кравченко. Здание \\"На Загородке\\"",\n      "lat": 55.6933174,\n      "lon": 37.6036611,\n      "opening_hours": null\n    },\n    {\n      "id": "way/980596512",\n      "kind": "kindergarten",\n      "name": "Школа № 1206. Корпус дошкольного образования №11 «Наш дом»",\n      "lat": 55.6094806,\n      "lon": 37.5399204,\n      "opening_hours": null\n    },\n    {\n      "id": "way/985558075",\n      "kind": "kindergarten",\n      "name": "Школа №875. Корпус дошкольного образования «Мир детства»",\n      "lat": 55.6655656,\n      "lon": 37.4913243,\n      "opening_hours": null\n    },\n    {\n      "id": "way/985558079",\n      "kind": "kindergarten",\n      "name": "Школа №875. Корпус дошкольного образования «Содружество»",\n      "lat": 55.6651349,\n      "lon": 37.4973504,\n      "opening_hours": null\n    },\n    {\n      "id": "way/985565307",\n      "kind": "kindergarten",\n      "name": "Школа №875. Корпус дошкольного образования «Радуга»",\n      "lat": 55.6601085,\n      "lon": 37.4931627,\n      "opening_hours": null\n    },\n    {\n      "id": "way/985565313",\n      "kind": "kindergarten",\n      "name": "Школа №875. Корпус дошкольного образования «Сказка»",\n      "lat": 55.6614491,\n      "lon": 37.4905988,\n      "opening_hours": null\n    },\n    {\n      "id": "way/986753456",\n      "kind": "kindergarten",\n      "name": "Детский сад №299",\n      "lat": 55.7754649,\n      "lon": 37.6929529,\n      "opening_hours": null\n    },\n    {\n      "id": "way/987757025",\n      "kind": "school",\n      "name": "Школа на проспекте Вернадского. Корпус школьного образования 3",\n      "lat": 55.6538607,\n      "lon": 37.4856074,\n      "opening_hours": null\n    },\n    {\n      "id": "way/987757026",\n      "kind": "school",\n      "name": "Школа на проспекте Вернадского. Корпус школьного образования 2",\n      "lat": 55.6526647,\n      "lon": 37.4851661,\n      "opening_hours": null\n    },\n    {\n      "id": "way/991278392",\n      "kind": "school",\n      "name": "Школа №\xa01247",\n      "lat": 55.7710673,\n      "lon": 37.6879321,\n      "opening_hours": null\n    },\n    {\n      "id": "way/993985457",\n      "kind": "school",\n      "name": "Лицей №6",\n      "lat": 55.8914902,\n      "lon": 37.4719645,\n      "opening_hours": "Mo-Sa 08:00-18:00"\n    },\n    {\n      "id": "way/995333442",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Островок\\"",\n      "lat": 55.8012865,\n      "lon": 37.8154756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/995337699",\n      "kind": "kindergarten",\n      "name": "Детский сад №1651",\n      "lat": 55.815968,\n      "lon": 37.8023462,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1007452119",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6485861,\n      "lon": 37.7321824,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1022286575",\n      "kind": "school",\n      "name": "Школа № 2115. Корпус школьного образования № 2",\n      "lat": 55.6695321,\n      "lon": 37.5714823,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1024554009",\n      "kind": "kindergarten",\n      "name": "Школа № 324 «Жар-птица». Корпус дошкольного образования «Солнышко»",\n      "lat": 55.6785907,\n      "lon": 37.4975735,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1024555865",\n      "kind": "kindergarten",\n      "name": "Школа № 324 «Жар-птица». Корпус дошкольного образования «Садко»",\n      "lat": 55.6754806,\n      "lon": 37.4968372,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1029616464",\n      "kind": "school",\n      "name": "Школа № 2055. Учебный корпус",\n      "lat": 55.7627449,\n      "lon": 37.5467382,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1033315598",\n      "kind": "school",\n      "name": "Центр образования № 1699",\n      "lat": 55.7350632,\n      "lon": 37.7457263,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1039436694",\n      "kind": "kindergarten",\n      "name": "Вешняковская школа. Дошкольная группа №9",\n      "lat": 55.7420548,\n      "lon": 37.8243971,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1052414608",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7619091,\n      "lon": 37.5152847,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1056570456",\n      "kind": "kindergarten",\n      "name": "Детский сад №1865",\n      "lat": 55.7753786,\n      "lon": 37.689577,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1056570460",\n      "kind": "kindergarten",\n      "name": "Детский сад №1976",\n      "lat": 55.7732997,\n      "lon": 37.6895622,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1068412729",\n      "kind": "kindergarten",\n      "name": "Школа № 1454 «Тимирязевская», дошкольное отделение «Дубки»",\n      "lat": 55.8233506,\n      "lon": 37.5615985,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1069628566",\n      "kind": "kindergarten",\n      "name": "Детский сад №2543",\n      "lat": 55.6938567,\n      "lon": 37.7207985,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1076569790",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8713653,\n      "lon": 37.4887808,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1078433003",\n      "kind": "kindergarten",\n      "name": "Школа № 830. Дошкольное отделение \\"Крепыш\\"",\n      "lat": 55.8247815,\n      "lon": 37.4532705,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1079522044",\n      "kind": "kindergarten",\n      "name": "Школа № 2121 имени Маршала Советского Союза С.К. Куркоткина. Дошкольный корпус № 4 \\"Кораблик\\"",\n      "lat": 55.6744005,\n      "lon": 37.7508414,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1086788732",\n      "kind": "kindergarten",\n      "name": "Школа № 1222 имени Маршала Советского Союза И. Х. Баграмяна. Дошкольное отделение № 3",\n      "lat": 55.7334197,\n      "lon": 37.7201807,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1090983005",\n      "kind": "school",\n      "name": "Второй корпус школы № 1384",\n      "lat": 55.8025114,\n      "lon": 37.5158186,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1093229206",\n      "kind": "school",\n      "name": "Школа №641",\n      "lat": 55.7042448,\n      "lon": 37.7572998,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1094253413",\n      "kind": "kindergarten",\n      "name": "Лицей №1571 дошкольное отделение Конфетти",\n      "lat": 55.8532562,\n      "lon": 37.4470517,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562107",\n      "kind": "school",\n      "name": "Лицей №1524",\n      "lat": 55.6740384,\n      "lon": 37.7570109,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562108",\n      "kind": "kindergarten",\n      "name": "Детский сад №616",\n      "lat": 55.675768,\n      "lon": 37.7554346,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562109",\n      "kind": "kindergarten",\n      "name": "Детский сад №1821",\n      "lat": 55.6749303,\n      "lon": 37.7470481,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562110",\n      "kind": "school",\n      "name": "НОУ Школа \\"Карьера\\"",\n      "lat": 55.6776009,\n      "lon": 37.7429604,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562111",\n      "kind": "school",\n      "name": "Школа №1148",\n      "lat": 55.6783193,\n      "lon": 37.7424465,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562112",\n      "kind": "school",\n      "name": "НОУ Школа \\"Интеллект\\"",\n      "lat": 55.6801717,\n      "lon": 37.7481319,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095562113",\n      "kind": "kindergarten",\n      "name": "Детский сад №2129",\n      "lat": 55.6829347,\n      "lon": 37.7470999,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095563380",\n      "kind": "kindergarten",\n      "name": "Детский сад №499",\n      "lat": 55.7016697,\n      "lon": 37.7439589,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095563381",\n      "kind": "school",\n      "name": "Школа №654 имени А. Д. Фридмана. Учебный корпус №2",\n      "lat": 55.7010935,\n      "lon": 37.7447165,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095563382",\n      "kind": "school",\n      "name": "Школа №687",\n      "lat": 55.701199,\n      "lon": 37.7409905,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095563383",\n      "kind": "kindergarten",\n      "name": "Детский сад №884",\n      "lat": 55.7028332,\n      "lon": 37.7395109,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1095563384",\n      "kind": "kindergarten",\n      "name": "Детский сад №88",\n      "lat": 55.7023772,\n      "lon": 37.7401763,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096324017",\n      "kind": "kindergarten",\n      "name": "Школа №654 имени А. Д. Фридмана. Дошкольное отделение №4",\n      "lat": 55.6952483,\n      "lon": 37.7413111,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096324021",\n      "kind": "kindergarten",\n      "name": "Школа №654 имени А. Д. Фридмана. Дошкольное отделение №5",\n      "lat": 55.6958816,\n      "lon": 37.7415304,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096324034",\n      "kind": "school",\n      "name": "Школа №654 имени А. Д. Фридмана. Учебный корпус №1",\n      "lat": 55.6978464,\n      "lon": 37.7399458,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096326152",\n      "kind": "school",\n      "name": "Спортшкола № 64",\n      "lat": 55.7009693,\n      "lon": 37.7810595,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096326153",\n      "kind": "kindergarten",\n      "name": "Детский сад № 149",\n      "lat": 55.7003701,\n      "lon": 37.772826,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1096326154",\n      "kind": "school",\n      "name": "Школа № 333",\n      "lat": 55.7037588,\n      "lon": 37.7602107,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097355893",\n      "kind": "school",\n      "name": "Школа №338",\n      "lat": 55.6752795,\n      "lon": 37.7415239,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097358480",\n      "kind": "school",\n      "name": "Школа №482",\n      "lat": 55.7018149,\n      "lon": 37.7363211,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097447006",\n      "kind": "school",\n      "name": "Школа № 1469",\n      "lat": 55.7021938,\n      "lon": 37.754003,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097447008",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7019521,\n      "lon": 37.7555023,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097448183",\n      "kind": "kindergarten",\n      "name": "Детский сад №183",\n      "lat": 55.7052188,\n      "lon": 37.7516711,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097682346",\n      "kind": "school",\n      "name": "Центр образования №2121-начальная школа",\n      "lat": 55.6798684,\n      "lon": 37.7519334,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097682347",\n      "kind": "school",\n      "name": "Центр образования №2121",\n      "lat": 55.6786958,\n      "lon": 37.7520085,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1097701062",\n      "kind": "school",\n      "name": "Школа №480 им. В.В. Талалихина",\n      "lat": 55.7354192,\n      "lon": 37.6846312,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1118849806",\n      "kind": "kindergarten",\n      "name": "Школа №1362 дошкольный корпус",\n      "lat": 55.7918528,\n      "lon": 37.7343115,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1120376459",\n      "kind": "kindergarten",\n      "name": "Детский сад №\xa0832",\n      "lat": 55.827139,\n      "lon": 37.5834897,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1120376460",\n      "kind": "school",\n      "name": "Школа №\xa0968",\n      "lat": 55.826211,\n      "lon": 37.5851886,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1120517398",\n      "kind": "kindergarten",\n      "name": "Детский сад №425",\n      "lat": 55.8164913,\n      "lon": 37.5854418,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1122007526",\n      "kind": "school",\n      "name": "Школа № 625. Начальная и средняя школа",\n      "lat": 55.6894775,\n      "lon": 37.5888729,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1134331221",\n      "kind": "kindergarten",\n      "name": "Детский сад №5",\n      "lat": 55.8201318,\n      "lon": 37.5889146,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1152214665",\n      "kind": "school",\n      "name": "Школа № 1586. Учебный корпус № 1",\n      "lat": 55.7092526,\n      "lon": 37.5134592,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1154845325",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7432396,\n      "lon": 37.802756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1156606030",\n      "kind": "kindergarten",\n      "name": "Школа № 1236 имени С.В. Милашенкова. Учебный корпус №121",\n      "lat": 55.8219526,\n      "lon": 37.5816334,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1157204034",\n      "kind": "kindergarten",\n      "name": "ДОУ №64",\n      "lat": 55.6784299,\n      "lon": 37.6279123,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1163425107",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8827442,\n      "lon": 37.6377306,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1168924918",\n      "kind": "kindergarten",\n      "name": "Детский сад №154 МЖД МПС РФ",\n      "lat": 55.7401194,\n      "lon": 37.5483008,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1176375533",\n      "kind": "school",\n      "name": "Школа № 1508. Корпус Софьи Ковалевской",\n      "lat": 55.8018027,\n      "lon": 37.79074,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1178180614",\n      "kind": "kindergarten",\n      "name": "Детский сад №461",\n      "lat": 55.802211,\n      "lon": 37.7962243,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1180159484",\n      "kind": "school",\n      "name": "начальная школа «Интеграция XXI век»",\n      "lat": 55.8009481,\n      "lon": 37.4898015,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1182571822",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6391524,\n      "lon": 37.798654,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1188713166",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7150868,\n      "lon": 37.754811,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1198830522",\n      "kind": "school",\n      "name": "Бауманская инженерная школа № 1580",\n      "lat": 55.7502931,\n      "lon": 37.7018673,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1198908691",\n      "kind": "kindergarten",\n      "name": "Ясли-сад №2099",\n      "lat": 55.6792155,\n      "lon": 37.7188036,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1198908692",\n      "kind": "school",\n      "name": "Школа №\xa01041",\n      "lat": 55.677969,\n      "lon": 37.72018,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1198909608",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6800556,\n      "lon": 37.7199422,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1199316488",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6647467,\n      "lon": 37.7318461,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1214119130",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад № 1620",\n      "lat": 55.6822661,\n      "lon": 37.5334921,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1216010923",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8256143,\n      "lon": 37.5248514,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1226935123",\n      "kind": "kindergarten",\n      "name": "Протон",\n      "lat": 55.7394294,\n      "lon": 37.5020085,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1230105741",\n      "kind": "school",\n      "name": "Новое образование",\n      "lat": 55.7337701,\n      "lon": 37.7163769,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1235854567",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6862176,\n      "lon": 37.5393207,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1242441869",\n      "kind": "kindergarten",\n      "name": "Детский сад №2149",\n      "lat": 55.8229795,\n      "lon": 37.5767771,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1243054430",\n      "kind": "kindergarten",\n      "name": "Детский сад №1832",\n      "lat": 55.8209024,\n      "lon": 37.5769887,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1246044684",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7474048,\n      "lon": 37.5516422,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1249918852",\n      "kind": "school",\n      "name": "Путь зерна",\n      "lat": 55.6435668,\n      "lon": 37.5189347,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1255000953",\n      "kind": "school",\n      "name": "Школа №1543. Корпус школьного образования",\n      "lat": 55.6556246,\n      "lon": 37.486239,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1257008884",\n      "kind": "school",\n      "name": "Школа № 1213",\n      "lat": 55.8397827,\n      "lon": 37.5761873,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1259771713",\n      "kind": "school",\n      "name": "Школа № 1324",\n      "lat": 55.7586314,\n      "lon": 37.8237309,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1259810311",\n      "kind": "kindergarten",\n      "name": "Школа № 185, дошкольное отделение",\n      "lat": 55.8391121,\n      "lon": 37.5419756,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1262490343",\n      "kind": "school",\n      "name": "Школа №439 \\"Интеллект\\" (корпус №1)",\n      "lat": 55.7311773,\n      "lon": 37.7453032,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1263868352",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7318894,\n      "lon": 37.7586297,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1267863905",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7552717,\n      "lon": 37.5148694,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1268336549",\n      "kind": "school",\n      "name": "Кадетская школа №1785",\n      "lat": 55.7358086,\n      "lon": 37.6917636,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1274403627",\n      "kind": "school",\n      "name": "Новая школа",\n      "lat": 55.7049909,\n      "lon": 37.4899723,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1276348994",\n      "kind": "school",\n      "name": "Школа №\xa01579",\n      "lat": 55.6503976,\n      "lon": 37.632768,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1278312025",\n      "kind": "kindergarten",\n      "name": "Детский сад №1174",\n      "lat": 55.7586329,\n      "lon": 37.7842092,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1285516011",\n      "kind": "school",\n      "name": "Школа № 463 им. Д. Н. Медведева",\n      "lat": 55.7219225,\n      "lon": 37.6617855,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1289119543",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6516216,\n      "lon": 37.6311792,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1302584894",\n      "kind": "kindergarten",\n      "name": "Школа № 1363, ДО Незабудка",\n      "lat": 55.69349,\n      "lon": 37.821866,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1303268971",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6747159,\n      "lon": 37.8094696,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1305195718",\n      "kind": "school",\n      "name": "Школа №1637",\n      "lat": 55.7594673,\n      "lon": 37.7837123,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1312881247",\n      "kind": "school",\n      "name": "Школа № 2090",\n      "lat": 55.7093043,\n      "lon": 37.7924488,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1313956567",\n      "kind": "school",\n      "name": "Школа № 463 имени Героя Советского Союза Д.Н. Медведева",\n      "lat": 55.6900128,\n      "lon": 37.6682002,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1329403077",\n      "kind": "kindergarten",\n      "name": "Детский сад №1770, Школа \\"Наши Пенаты\\"",\n      "lat": 55.6492061,\n      "lon": 37.5275594,\n      "opening_hours": null\n    },\n    {\n      "id": "way/1334478416",\n      "kind": "kindergarten",\n      "name": "Школа №1576. Дошкольное отделение №10",\n      "lat": 55.8315226,\n      "lon": 37.5286559,\n      "opening_hours": null\n    },\n    {\n      "id": "node/416876296",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6839716,\n      "lon": 37.5734071,\n      "opening_hours": null\n    },\n    {\n      "id": "node/517247653",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8852577,\n      "lon": 37.7469699,\n      "opening_hours": null\n    },\n    {\n      "id": "node/643737879",\n      "kind": "kindergarten",\n      "name": "Школа №439 \\"Интеллект\\" (дошкольное отделение №3)",\n      "lat": 55.735387,\n      "lon": 37.7470929,\n      "opening_hours": null\n    },\n    {\n      "id": "node/745608832",\n      "kind": "kindergarten",\n      "name": "Детский сад №1071",\n      "lat": 55.7931586,\n      "lon": 37.5080654,\n      "opening_hours": null\n    },\n    {\n      "id": "node/949916341",\n      "kind": "kindergarten",\n      "name": "Детский сад №2143",\n      "lat": 55.8059612,\n      "lon": 37.6377796,\n      "opening_hours": null\n    },\n    {\n      "id": "node/954912765",\n      "kind": "school",\n      "name": "Школа №1",\n      "lat": 55.6890655,\n      "lon": 37.5378735,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1013879417",\n      "kind": "school",\n      "name": "Московская международная гимназия",\n      "lat": 55.755926,\n      "lon": 37.8012033,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1072766433",\n      "kind": "kindergarten",\n      "name": "Центр Родник",\n      "lat": 55.707117,\n      "lon": 37.6588989,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1072770629",\n      "kind": "kindergarten",\n      "name": "Жемчужина",\n      "lat": 55.7070518,\n      "lon": 37.6585598,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1106954251",\n      "kind": "kindergarten",\n      "name": "Детский сад №1309",\n      "lat": 55.8133975,\n      "lon": 37.639607,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1141776015",\n      "kind": "kindergarten",\n      "name": "Школа 1574 Дошкольное отделение",\n      "lat": 55.7733226,\n      "lon": 37.6004213,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1142158796",\n      "kind": "school",\n      "name": "Школа №1362",\n      "lat": 55.7898108,\n      "lon": 37.7410624,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1145370005",\n      "kind": "school",\n      "name": "Средняя школа №72",\n      "lat": 55.7475855,\n      "lon": 37.4985787,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1147951174",\n      "kind": "kindergarten",\n      "name": "Детский сад №252",\n      "lat": 55.6934079,\n      "lon": 37.585788,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1192496379",\n      "kind": "school",\n      "name": "Школа надомного обучения №381",\n      "lat": 55.8047269,\n      "lon": 37.7928053,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1252450767",\n      "kind": "kindergarten",\n      "name": "Детский сад №1321",\n      "lat": 55.7222138,\n      "lon": 37.6662232,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1274329843",\n      "kind": "kindergarten",\n      "name": "Школа № 67. Отделение дошкольного образования № 4",\n      "lat": 55.7372044,\n      "lon": 37.5328738,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1277580872",\n      "kind": "kindergarten",\n      "name": "Детский сад №1123",\n      "lat": 55.6904012,\n      "lon": 37.5451365,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1288895160",\n      "kind": "kindergarten",\n      "name": "Детский сад №3",\n      "lat": 55.793622,\n      "lon": 37.4818594,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1333238486",\n      "kind": "kindergarten",\n      "name": "Детский сад компенсирующего вида №1093",\n      "lat": 55.7207776,\n      "lon": 37.5750012,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1333238570",\n      "kind": "kindergarten",\n      "name": "Детский сад №1359",\n      "lat": 55.7185682,\n      "lon": 37.5796113,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1355396778",\n      "kind": "kindergarten",\n      "name": "Почемучка",\n      "lat": 55.6446753,\n      "lon": 37.5198495,\n      "opening_hours": "Mo-Su 10:00-18:00"\n    },\n    {\n      "id": "node/1373666635",\n      "kind": "school",\n      "name": "Школа № 7, филиал № 1",\n      "lat": 55.6791957,\n      "lon": 37.5253409,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669698",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1136",\n      "lat": 55.6925084,\n      "lon": 37.5397693,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669716",\n      "kind": "kindergarten",\n      "name": "Детский сад №2042",\n      "lat": 55.6940211,\n      "lon": 37.554457,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669718",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2558",\n      "lat": 55.6862785,\n      "lon": 37.5379804,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669722",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2558",\n      "lat": 55.6872354,\n      "lon": 37.5396966,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669725",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2558",\n      "lat": 55.6899587,\n      "lon": 37.5391276,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669729",\n      "kind": "kindergarten",\n      "name": "Детский сад №4",\n      "lat": 55.6998837,\n      "lon": 37.5549115,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669735",\n      "kind": "school",\n      "name": "Н.ш. - д.с. № 1708",\n      "lat": 55.6853591,\n      "lon": 37.5386444,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373669736",\n      "kind": "kindergarten",\n      "name": "Начальная школа, д/с №1851",\n      "lat": 55.6914932,\n      "lon": 37.5361552,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373694626",\n      "kind": "kindergarten",\n      "name": "Начальная школа-детский сад №1620",\n      "lat": 55.6812095,\n      "lon": 37.5279342,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373694627",\n      "kind": "school",\n      "name": "Начальная школа-детский сад №1620",\n      "lat": 55.6814973,\n      "lon": 37.5295702,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373694628",\n      "kind": "school",\n      "name": "Начальная школа-детский сад №1620",\n      "lat": 55.6824634,\n      "lon": 37.5250336,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373694629",\n      "kind": "school",\n      "name": "Школа №7 (филиал №2)",\n      "lat": 55.6829241,\n      "lon": 37.5270539,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373694630",\n      "kind": "school",\n      "name": "Школа №119",\n      "lat": 55.6798473,\n      "lon": 37.5213993,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373755234",\n      "kind": "school",\n      "name": "СААШ \\"Марина\\"",\n      "lat": 55.6763945,\n      "lon": 37.5207685,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373756112",\n      "kind": "kindergarten",\n      "name": "Детский сад №375",\n      "lat": 55.668598,\n      "lon": 37.5233243,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373756113",\n      "kind": "school",\n      "name": "Школа 1181 (для пациентов больницы)",\n      "lat": 55.6510102,\n      "lon": 37.493042,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373768269",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1041",\n      "lat": 55.6854919,\n      "lon": 37.5637465,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373768284",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2171",\n      "lat": 55.6798608,\n      "lon": 37.5641675,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373768288",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2171",\n      "lat": 55.6810221,\n      "lon": 37.565399,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373776071",\n      "kind": "school",\n      "name": "Лицей №1525 \\"Воробьевы горы\\"",\n      "lat": 55.7029962,\n      "lon": 37.5544024,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373778976",\n      "kind": "school",\n      "name": "Школа № 1318 (киношкола)",\n      "lat": 55.6870452,\n      "lon": 37.5298416,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373825239",\n      "kind": "school",\n      "name": "Центр дополнительного образования детей \\"Диалог наук\\"",\n      "lat": 55.6950395,\n      "lon": 37.5518003,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1373831939",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1462",\n      "lat": 55.6665921,\n      "lon": 37.5851997,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374506143",\n      "kind": "school",\n      "name": "Школа \\"Промо-М\\"",\n      "lat": 55.643379,\n      "lon": 37.47992,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374543558",\n      "kind": "kindergarten",\n      "name": "ДОУ 2415",\n      "lat": 55.6521405,\n      "lon": 37.5192693,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374741490",\n      "kind": "school",\n      "name": "ДООЦ \\"Юго-Западный\\"",\n      "lat": 55.6085997,\n      "lon": 37.5287946,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374741494",\n      "kind": "school",\n      "name": "Лингвистический центр \\"Маленький принц\\"",\n      "lat": 55.6018739,\n      "lon": 37.5452392,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374741534",\n      "kind": "school",\n      "name": "Школа \\"Ясенево\\"",\n      "lat": 55.5950379,\n      "lon": 37.5381405,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1374876546",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2480",\n      "lat": 55.6514906,\n      "lon": 37.5973955,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1376740671",\n      "kind": "kindergarten",\n      "name": "Детский сад №2694",\n      "lat": 55.6769338,\n      "lon": 37.547569,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1376761322",\n      "kind": "kindergarten",\n      "name": "Школа 1948 \\"Лингвист-М\\"",\n      "lat": 55.6641266,\n      "lon": 37.5549242,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1426996766",\n      "kind": "school",\n      "name": "Православная школа во имя иконы божией Матери \\"Скоропослушница\\"",\n      "lat": 55.5899136,\n      "lon": 37.6517472,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427761505",\n      "kind": "kindergarten",\n      "name": "ДОУ №87",\n      "lat": 55.6696127,\n      "lon": 37.6289439,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427761506",\n      "kind": "school",\n      "name": "Государственное бюджетное образовательное учреждение города Москвы центр лечебной педагогики и дифференцированного обучения \\"Возрождение\\"",\n      "lat": 55.6632193,\n      "lon": 37.6485723,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427798458",\n      "kind": "kindergarten",\n      "name": "ГБОУ школа № 507, здание № 14 Семицветик",\n      "lat": 55.6616667,\n      "lon": 37.6322709,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427798459",\n      "kind": "school",\n      "name": "ННОУ \\"Эрудит\\"",\n      "lat": 55.6793447,\n      "lon": 37.6545815,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427798462",\n      "kind": "school",\n      "name": "Негосударственное образовательное учреждение русская гуманитарная школа",\n      "lat": 55.6655864,\n      "lon": 37.6431838,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427803906",\n      "kind": "kindergarten",\n      "name": "Детский сад №67",\n      "lat": 55.7999182,\n      "lon": 37.5265136,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427853539",\n      "kind": "kindergarten",\n      "name": "Средняя общеобразовательная школа № 1073",\n      "lat": 55.6622784,\n      "lon": 37.6261054,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427855779",\n      "kind": "kindergarten",\n      "name": "Детский сад № 131",\n      "lat": 55.7079922,\n      "lon": 37.661336,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427855782",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1672",\n      "lat": 55.7035467,\n      "lon": 37.6639557,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427855785",\n      "kind": "kindergarten",\n      "name": "Школа № 630 им. дважды Героя Советского Союза Г.П. Кравченко. Дошкольное отделение",\n      "lat": 55.7112246,\n      "lon": 37.6111345,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427855794",\n      "kind": "kindergarten",\n      "name": "Детский сад № 307",\n      "lat": 55.7192549,\n      "lon": 37.6484758,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427855795",\n      "kind": "kindergarten",\n      "name": "Детский сад №852",\n      "lat": 55.7195446,\n      "lon": 37.6607312,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427956880",\n      "kind": "kindergarten",\n      "name": "Детский сад №2172",\n      "lat": 55.7221258,\n      "lon": 37.5768038,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427982444",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1257",\n      "lat": 55.7634798,\n      "lon": 37.5567572,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427982445",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 9 «Анютины глазки»",\n      "lat": 55.7502211,\n      "lon": 37.5235882,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427982447",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7580315,\n      "lon": 37.5528012,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1427996071",\n      "kind": "kindergarten",\n      "name": "ДО №6 Улыбка",\n      "lat": 55.7725055,\n      "lon": 37.5832127,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1428076955",\n      "kind": "kindergarten",\n      "name": "Детский сад №281",\n      "lat": 55.7235473,\n      "lon": 37.6068753,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429235509",\n      "kind": "school",\n      "name": "ЦПМСС \\"Взаимодействие\\"",\n      "lat": 55.6603447,\n      "lon": 37.6252917,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429376733",\n      "kind": "kindergarten",\n      "name": "Детский сад №37",\n      "lat": 55.7758727,\n      "lon": 37.6468774,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429379251",\n      "kind": "school",\n      "name": "ЦДТ \\"Логос\\"",\n      "lat": 55.7098599,\n      "lon": 37.6427218,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429406139",\n      "kind": "kindergarten",\n      "name": "Детский сад №65",\n      "lat": 55.7132417,\n      "lon": 37.6679007,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429495670",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7034523,\n      "lon": 37.6735585,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1429499809",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7033024,\n      "lon": 37.6691982,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430419667",\n      "kind": "kindergarten",\n      "name": "ДОУ №729",\n      "lat": 55.7540076,\n      "lon": 37.5844648,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430428669",\n      "kind": "kindergarten",\n      "name": "НОУ \\"Атлант\\"",\n      "lat": 55.624655,\n      "lon": 37.6074245,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430457015",\n      "kind": "kindergarten",\n      "name": "Детский сад №2500",\n      "lat": 55.6936898,\n      "lon": 37.7376052,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430522302",\n      "kind": "school",\n      "name": "АНО СОШ \\"Учебный центр Перспектива\\"",\n      "lat": 55.6374278,\n      "lon": 37.6143473,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430546453",\n      "kind": "school",\n      "name": "Школа №435",\n      "lat": 55.7746135,\n      "lon": 37.7048526,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430612438",\n      "kind": "kindergarten",\n      "name": "Детский сад №1609",\n      "lat": 55.7236994,\n      "lon": 37.7697169,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1430804068",\n      "kind": "school",\n      "name": "Школа №1113",\n      "lat": 55.7691104,\n      "lon": 37.6030203,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1432138391",\n      "kind": "kindergarten",\n      "name": "Детский сад №1192",\n      "lat": 55.780008,\n      "lon": 37.615793,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1433444316",\n      "kind": "kindergarten",\n      "name": "Романовская школа, дошкольное отделение № 4 «Солнышко»",\n      "lat": 55.7650358,\n      "lon": 37.5621154,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1433744179",\n      "kind": "kindergarten",\n      "name": "Детский сад №1548",\n      "lat": 55.7214707,\n      "lon": 37.578384,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1434207695",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1786",\n      "lat": 55.7573124,\n      "lon": 37.5420359,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1434221468",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.767322,\n      "lon": 37.5560413,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1434221469",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7678856,\n      "lon": 37.5558112,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1434238335",\n      "kind": "kindergarten",\n      "name": "ЦРР детский сад № 1842",\n      "lat": 55.7706674,\n      "lon": 37.586669,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1435421467",\n      "kind": "kindergarten",\n      "name": "Школа №1621 \\"Древо жизни\\". Дошкольное отделение",\n      "lat": 55.7665753,\n      "lon": 37.6536785,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1435673590",\n      "kind": "kindergarten",\n      "name": "Детский сад № 26",\n      "lat": 55.7085092,\n      "lon": 37.6595251,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1436508002",\n      "kind": "kindergarten",\n      "name": "Центр развития ребёнка №2364",\n      "lat": 55.7671373,\n      "lon": 37.5829561,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1436607111",\n      "kind": "kindergarten",\n      "name": "ДОУ №2237",\n      "lat": 55.6793049,\n      "lon": 37.646904,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1437387424",\n      "kind": "kindergarten",\n      "name": "ДОУ №6",\n      "lat": 55.7714842,\n      "lon": 37.667638,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1437658381",\n      "kind": "kindergarten",\n      "name": "Московская международная школа. Подразделение \\"Планета Марс\\"",\n      "lat": 55.7644746,\n      "lon": 37.7862673,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1437658382",\n      "kind": "kindergarten",\n      "name": "Московская международная школа. Подразделение \\"Планета Венера\\"",\n      "lat": 55.7638021,\n      "lon": 37.7828847,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1443197494",\n      "kind": "kindergarten",\n      "name": "Школа № 1231 им. В. Д. Поленова. Дошкольный корпус",\n      "lat": 55.7278139,\n      "lon": 37.5606804,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1443307436",\n      "kind": "school",\n      "name": "Школа №654 имени А. Д. Фридмана. Учебный корпус №4",\n      "lat": 55.694833,\n      "lon": 37.7362385,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1444803462",\n      "kind": "school",\n      "name": "Центр психолого-педагогической реабилитации и коррекции \\"Восточный\\"",\n      "lat": 55.7820621,\n      "lon": 37.7276916,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1444819629",\n      "kind": "kindergarten",\n      "name": "Детский сад №118",\n      "lat": 55.7801928,\n      "lon": 37.730691,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1456162724",\n      "kind": "kindergarten",\n      "name": "Детский сад №227 (встр)",\n      "lat": 55.7837038,\n      "lon": 37.7331763,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1456659414",\n      "kind": "kindergarten",\n      "name": "Детский сад №1089",\n      "lat": 55.7268272,\n      "lon": 37.5619022,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1456659419",\n      "kind": "kindergarten",\n      "name": "Детский сад компенсирующего вида №1476",\n      "lat": 55.7239375,\n      "lon": 37.5823418,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1456659423",\n      "kind": "kindergarten",\n      "name": "Детский сад №676",\n      "lat": 55.72473,\n      "lon": 37.5664425,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1460061975",\n      "kind": "kindergarten",\n      "name": "Московская международная школа. Подразделение \\"Планета Меркурий\\"",\n      "lat": 55.7589128,\n      "lon": 37.7886511,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1472892444",\n      "kind": "kindergarten",\n      "name": "Детский сад №1107",\n      "lat": 55.7822439,\n      "lon": 37.7422118,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1504923799",\n      "kind": "kindergarten",\n      "name": "Детский сад №960",\n      "lat": 55.794034,\n      "lon": 37.7905245,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1504923815",\n      "kind": "kindergarten",\n      "name": "Детский сад №2251",\n      "lat": 55.7936427,\n      "lon": 37.7982305,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1560232958",\n      "kind": "school",\n      "name": "Центр образования № 118",\n      "lat": 55.6869036,\n      "lon": 37.5295957,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1561070468",\n      "kind": "school",\n      "name": "Школа №213",\n      "lat": 55.8167885,\n      "lon": 37.5198901,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1566443974",\n      "kind": "kindergarten",\n      "name": "Детский сад №1500",\n      "lat": 55.7061792,\n      "lon": 37.5725787,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1588932818",\n      "kind": "school",\n      "name": "Центр образования \\"Измайлово\\" № 1811",\n      "lat": 55.7939167,\n      "lon": 37.8147527,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1599296100",\n      "kind": "school",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa01",\n      "lat": 55.7077593,\n      "lon": 37.7712348,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1602410893",\n      "kind": "kindergarten",\n      "name": "Детский сад №1905 (встр)",\n      "lat": 55.7619833,\n      "lon": 37.7347428,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1603692724",\n      "kind": "kindergarten",\n      "name": "Вальдорфовская школа",\n      "lat": 55.6411221,\n      "lon": 37.5223934,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1618108202",\n      "kind": "school",\n      "name": "Прогимназия № 130",\n      "lat": 55.7796573,\n      "lon": 37.4960053,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1635684783",\n      "kind": "school",\n      "name": "Спортивно-технический центр \\"Икар\\"",\n      "lat": 55.7585593,\n      "lon": 37.7115837,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1635684784",\n      "kind": "school",\n      "name": "Спортивно-технический центр \\"Икар\\"",\n      "lat": 55.7718921,\n      "lon": 37.7111201,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1719646868",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2416",\n      "lat": 55.7821094,\n      "lon": 37.736209,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1727533933",\n      "kind": "school",\n      "name": "Знаменская гимназия",\n      "lat": 55.8575416,\n      "lon": 37.4997164,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1735446417",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2296",\n      "lat": 55.7920113,\n      "lon": 37.5674501,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1792256966",\n      "kind": "school",\n      "name": "Детская музыкально-хоровая школа \\"Радость\\"",\n      "lat": 55.8395702,\n      "lon": 37.5294679,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1841073860",\n      "kind": "kindergarten",\n      "name": "Школа №439 \\"Интеллект\\" (дошкольное отделение №1)",\n      "lat": 55.7345153,\n      "lon": 37.745368,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1858733747",\n      "kind": "kindergarten",\n      "name": "Детский сад №1945",\n      "lat": 55.7540246,\n      "lon": 37.7102476,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1867012052",\n      "kind": "school",\n      "name": "Центр еврейского образования \\"Бейт швидлер\\"",\n      "lat": 55.7897637,\n      "lon": 37.6089304,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1945951836",\n      "kind": "kindergarten",\n      "name": "Школа № 113. Корпус дошкольного образования №6",\n      "lat": 55.6425327,\n      "lon": 37.5030157,\n      "opening_hours": null\n    },\n    {\n      "id": "node/1947167782",\n      "kind": "school",\n      "name": "Школа №423",\n      "lat": 55.7571365,\n      "lon": 37.8020241,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2055643773",\n      "kind": "kindergarten",\n      "name": "Школа №218. Дошкольное отделение",\n      "lat": 55.8127657,\n      "lon": 37.5732614,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2107834140",\n      "kind": "kindergarten",\n      "name": "Детский сад № 1284",\n      "lat": 55.7247929,\n      "lon": 37.4408259,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2107834141",\n      "kind": "kindergarten",\n      "name": "Детский сад № 943",\n      "lat": 55.7237007,\n      "lon": 37.4403719,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2146273545",\n      "kind": "school",\n      "name": "Школа №77",\n      "lat": 55.7668171,\n      "lon": 37.7275087,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2239554842",\n      "kind": "kindergarten",\n      "name": "ЛУЧиК",\n      "lat": 55.7427543,\n      "lon": 37.6009831,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2298658284",\n      "kind": "kindergarten",\n      "name": "Школа 1601 дошкольное отделение",\n      "lat": 55.8004005,\n      "lon": 37.5683713,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2298842448",\n      "kind": "kindergarten",\n      "name": "Детский сад №2509",\n      "lat": 55.6595529,\n      "lon": 37.5179687,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2352319437",\n      "kind": "school",\n      "name": "Детский досуговый центр «Аист»",\n      "lat": 55.7153262,\n      "lon": 37.779866,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2367806239",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7341456,\n      "lon": 37.7051295,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2525214326",\n      "kind": "kindergarten",\n      "name": "Детский развивающий центр",\n      "lat": 55.7085875,\n      "lon": 37.8097785,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2558642751",\n      "kind": "school",\n      "name": "Воскресная школа церкви Евангельских христиан-баптистов Воскресение",\n      "lat": 55.6427918,\n      "lon": 37.6760494,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2661065760",\n      "kind": "school",\n      "name": "Молодежный центр «Звездный»",\n      "lat": 55.8960402,\n      "lon": 37.7094401,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2661667230",\n      "kind": "kindergarten",\n      "name": "Детский сад №1422",\n      "lat": 55.7616574,\n      "lon": 37.7364917,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2828438665",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8975524,\n      "lon": 37.4796576,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2834722596",\n      "kind": "kindergarten",\n      "name": "Детский сад № 232",\n      "lat": 55.8151625,\n      "lon": 37.5670383,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2942619202",\n      "kind": "kindergarten",\n      "name": "Лицей № 1586, дошкольное отделение 2",\n      "lat": 55.7151446,\n      "lon": 37.5173189,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2942619203",\n      "kind": "kindergarten",\n      "name": "Лицей № 1586, дошкольное отделение 3",\n      "lat": 55.7130673,\n      "lon": 37.5169617,\n      "opening_hours": null\n    },\n    {\n      "id": "node/2942619204",\n      "kind": "kindergarten",\n      "name": "Лицей №1586, дошкольное отделение",\n      "lat": 55.7127864,\n      "lon": 37.5206021,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3005725110",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8182793,\n      "lon": 37.5054235,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3008237259",\n      "kind": "kindergarten",\n      "name": "Школа с углубленным изучением английского языка № 1270, дошкольное отделение № 2",\n      "lat": 55.749667,\n      "lon": 37.659845,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3008237260",\n      "kind": "school",\n      "name": "Школа № 2104",\n      "lat": 55.7495507,\n      "lon": 37.6585411,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "node/3041391839",\n      "kind": "school",\n      "name": "Филипповская Школа",\n      "lat": 55.7293782,\n      "lon": 37.5708461,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3139685557",\n      "kind": "school",\n      "name": "Школа №\xa01208 имени Героя Советского Союза М.С. Шумилова. Корпус №\xa04",\n      "lat": 55.7071909,\n      "lon": 37.7710406,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3363053807",\n      "kind": "kindergarten",\n      "name": "Гимназия №1551 корпус №3 (дошкольное отделение)",\n      "lat": 55.8457996,\n      "lon": 37.4503084,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3478996648",\n      "kind": "school",\n      "name": "Детская музыкально-хоровая школа имени И.И.Радченко",\n      "lat": 55.7357379,\n      "lon": 37.6598187,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3558549610",\n      "kind": "kindergarten",\n      "name": "Лукоморье",\n      "lat": 55.8044871,\n      "lon": 37.5385491,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3558834930",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7992699,\n      "lon": 37.5535226,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3632408193",\n      "kind": "kindergarten",\n      "name": "P\'titCREF",\n      "lat": 55.750418,\n      "lon": 37.597092,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3741570155",\n      "kind": "school",\n      "name": "",\n      "lat": 55.686451,\n      "lon": 37.5382072,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3741570172",\n      "kind": "school",\n      "name": "",\n      "lat": 55.6871486,\n      "lon": 37.5394425,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3794027238",\n      "kind": "school",\n      "name": "Школа Английского",\n      "lat": 55.8871929,\n      "lon": 37.6509059,\n      "opening_hours": "Mo-Fr 10:00-21:00; Sa-Su 10:00-18:00"\n    },\n    {\n      "id": "node/3894595446",\n      "kind": "kindergarten",\n      "name": "Ч.У.Д.О. Детский сад №\xa01",\n      "lat": 55.8171088,\n      "lon": 37.5074547,\n      "opening_hours": null\n    },\n    {\n      "id": "node/3955821430",\n      "kind": "kindergarten",\n      "name": "Детский центр \\"Вешняки\\"",\n      "lat": 55.7402605,\n      "lon": 37.8243783,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4107853391",\n      "kind": "kindergarten",\n      "name": "Sun School",\n      "lat": 55.8008773,\n      "lon": 37.6337751,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4119301621",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7703871,\n      "lon": 37.5868692,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4243926389",\n      "kind": "kindergarten",\n      "name": "Дошкольное отделение ГБОУ Школы №444",\n      "lat": 55.7896286,\n      "lon": 37.7966572,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4257318691",\n      "kind": "kindergarten",\n      "name": "Талантино",\n      "lat": 55.7729464,\n      "lon": 37.4645667,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4334671589",\n      "kind": "kindergarten",\n      "name": "Мамин Садик Seasons",\n      "lat": 55.7696732,\n      "lon": 37.6078508,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4351546398",\n      "kind": "kindergarten",\n      "name": "Даниил",\n      "lat": 55.7186774,\n      "lon": 37.6234095,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4381937594",\n      "kind": "school",\n      "name": "О\'кей инглиш",\n      "lat": 55.7131516,\n      "lon": 37.7554185,\n      "opening_hours": "Mo-Fr 10:00-20:00; Sa 10:00-16:00"\n    },\n    {\n      "id": "node/4393094689",\n      "kind": "kindergarten",\n      "name": "Семейный центр \\"Веста\\"",\n      "lat": 55.800881,\n      "lon": 37.7199409,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4429110930",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7313423,\n      "lon": 37.7581814,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4518521699",\n      "kind": "school",\n      "name": "Мини-школа",\n      "lat": 55.6942016,\n      "lon": 37.7877321,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4518521717",\n      "kind": "kindergarten",\n      "name": "Голицынский",\n      "lat": 55.6894547,\n      "lon": 37.7908161,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4520379714",\n      "kind": "kindergarten",\n      "name": "Лукошко",\n      "lat": 55.8205812,\n      "lon": 37.5822579,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4575871390",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8366297,\n      "lon": 37.6602361,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4588995214",\n      "kind": "kindergarten",\n      "name": "Детский сад №2566",\n      "lat": 55.7633007,\n      "lon": 37.7366041,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4642004020",\n      "kind": "school",\n      "name": "Ховринская гимназия \\"Лампада\\"",\n      "lat": 55.8649026,\n      "lon": 37.5089872,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4683882102",\n      "kind": "kindergarten",\n      "name": "Домик Клуб",\n      "lat": 55.8650457,\n      "lon": 37.5039408,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4732895821",\n      "kind": "kindergarten",\n      "name": "ЛучиК",\n      "lat": 55.7988505,\n      "lon": 37.5009718,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4742086648",\n      "kind": "kindergarten",\n      "name": "Детский развивающий центр Катюша",\n      "lat": 55.7042701,\n      "lon": 37.6686927,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4750851597",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7652441,\n      "lon": 37.5637221,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4750851599",\n      "kind": "kindergarten",\n      "name": "Школа № 1241, дошкольное отделение",\n      "lat": 55.7698779,\n      "lon": 37.5858852,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4750904980",\n      "kind": "kindergarten",\n      "name": "Радуга",\n      "lat": 55.7201055,\n      "lon": 37.5745181,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4750904981",\n      "kind": "school",\n      "name": "Школа Виктория-2000",\n      "lat": 55.7169438,\n      "lon": 37.5770098,\n      "opening_hours": "Mo-Fr 08:00-18:30"\n    },\n    {\n      "id": "node/4750904982",\n      "kind": "kindergarten",\n      "name": "Школа № 192. Дошкольное отделение",\n      "lat": 55.702035,\n      "lon": 37.5751536,\n      "opening_hours": "Mo-Fr 07:00-19:00"\n    },\n    {\n      "id": "node/4750904983",\n      "kind": "school",\n      "name": "Школа № 19",\n      "lat": 55.7025627,\n      "lon": 37.5761214,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4750941909",\n      "kind": "kindergarten",\n      "name": "Центр образования и спорта \\"Москва-98\\". Дошкольное отделение № 1",\n      "lat": 55.6851927,\n      "lon": 37.49421,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4751092803",\n      "kind": "kindergarten",\n      "name": "Звездочет",\n      "lat": 55.6521046,\n      "lon": 37.7645701,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4752529949",\n      "kind": "school",\n      "name": "Школа № 1331 \\"Молодая Иверия\\"",\n      "lat": 55.7478298,\n      "lon": 37.5942907,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4752529957",\n      "kind": "kindergarten",\n      "name": "Школа № 1465, дошкольное отделение № 2",\n      "lat": 55.7489559,\n      "lon": 37.5713427,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4752529964",\n      "kind": "kindergarten",\n      "name": "СОШ № 1232. Дошкольное отделение № 1",\n      "lat": 55.751215,\n      "lon": 37.5618039,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4752922517",\n      "kind": "kindergarten",\n      "name": "Детский сад №58",\n      "lat": 55.8973585,\n      "lon": 37.4778193,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4760975781",\n      "kind": "school",\n      "name": "Школа № 887. Отделение дополнительного образования детей",\n      "lat": 55.7351452,\n      "lon": 37.4406248,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4760975806",\n      "kind": "kindergarten",\n      "name": "Центр раннего развития «Тиша & Маруся»",\n      "lat": 55.6959326,\n      "lon": 37.5011659,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4792431751",\n      "kind": "kindergarten",\n      "name": "Лицей № 429 «Соколиная гора»",\n      "lat": 55.7678304,\n      "lon": 37.7325179,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4802013987",\n      "kind": "kindergarten",\n      "name": "Божья коровка",\n      "lat": 55.7770639,\n      "lon": 37.6010791,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4857880422",\n      "kind": "school",\n      "name": "Утро",\n      "lat": 55.7253324,\n      "lon": 37.5256742,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "node/4924865509",\n      "kind": "kindergarten",\n      "name": "Вгорошек",\n      "lat": 55.6458458,\n      "lon": 37.616462,\n      "opening_hours": null\n    },\n    {\n      "id": "node/4959152543",\n      "kind": "school",\n      "name": "Коррекционная школа-интернат VIII вида №29",\n      "lat": 55.8015246,\n      "lon": 37.7016433,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5045257699",\n      "kind": "school",\n      "name": "Школа баристы и барменов",\n      "lat": 55.8052147,\n      "lon": 37.528516,\n      "opening_hours": "Mo-Su 11:00-23:00"\n    },\n    {\n      "id": "node/5054611710",\n      "kind": "school",\n      "name": "Школа \\"Виктория\\"",\n      "lat": 55.7573563,\n      "lon": 37.5723477,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5054611711",\n      "kind": "school",\n      "name": "Свято-Димитриевская школа",\n      "lat": 55.7232298,\n      "lon": 37.6002213,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5092714240",\n      "kind": "school",\n      "name": "Дополнительное образование школы № 601",\n      "lat": 55.8034934,\n      "lon": 37.5409607,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5113395065",\n      "kind": "school",\n      "name": "Школа №370 им. генерала П. И. Батова",\n      "lat": 55.810851,\n      "lon": 37.7292703,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5161282123",\n      "kind": "kindergarten",\n      "name": "Парк детского периода",\n      "lat": 55.7023809,\n      "lon": 37.5042457,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5180531053",\n      "kind": "kindergarten",\n      "name": "Земляне",\n      "lat": 55.8045196,\n      "lon": 37.5445216,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5180531089",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8061398,\n      "lon": 37.5466437,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5199351213",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Глобус\\"",\n      "lat": 55.7898956,\n      "lon": 37.7710953,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5223728204",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7184332,\n      "lon": 37.6243684,\n      "opening_hours": "Mo-Su 08:00-19:00"\n    },\n    {\n      "id": "node/5241859752",\n      "kind": "kindergarten",\n      "name": "Детский центр \\"Умка\\"",\n      "lat": 55.6464796,\n      "lon": 37.7585041,\n      "opening_hours": "Mo-Su 07:00-20:00"\n    },\n    {\n      "id": "node/5241878658",\n      "kind": "kindergarten",\n      "name": "Детский центр \\"Антей\\"",\n      "lat": 55.6462993,\n      "lon": 37.7566806,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5251043166",\n      "kind": "kindergarten",\n      "name": "Маленький гений",\n      "lat": 55.652822,\n      "lon": 37.7559923,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5251072777",\n      "kind": "kindergarten",\n      "name": "Детский развивающий центр \\"Радуга\\"",\n      "lat": 55.6528416,\n      "lon": 37.7547201,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5260578062",\n      "kind": "kindergarten",\n      "name": "Детский сад №988",\n      "lat": 55.8015282,\n      "lon": 37.789439,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5273631003",\n      "kind": "kindergarten",\n      "name": "Детский сад №1221",\n      "lat": 55.7965665,\n      "lon": 37.8146034,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5274311218",\n      "kind": "kindergarten",\n      "name": "Детский сад № 457",\n      "lat": 55.8045805,\n      "lon": 37.5503065,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5292569524",\n      "kind": "kindergarten",\n      "name": "Школа № 429 - детский сад",\n      "lat": 55.7612682,\n      "lon": 37.7333849,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5293959915",\n      "kind": "kindergarten",\n      "name": "ДОУ №2233",\n      "lat": 55.6784207,\n      "lon": 37.6585405,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5300129067",\n      "kind": "kindergarten",\n      "name": "Детский сад №1155",\n      "lat": 55.8003112,\n      "lon": 37.7893176,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719149",\n      "kind": "kindergarten",\n      "name": "Детский сад №2455",\n      "lat": 55.7959106,\n      "lon": 37.7835327,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719152",\n      "kind": "kindergarten",\n      "name": "Детский сад №1127 (встр)",\n      "lat": 55.7977305,\n      "lon": 37.7867031,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719153",\n      "kind": "kindergarten",\n      "name": "Детский сад №1084",\n      "lat": 55.8016157,\n      "lon": 37.7962437,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719154",\n      "kind": "kindergarten",\n      "name": "Детский сад №461",\n      "lat": 55.802389,\n      "lon": 37.7975526,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719155",\n      "kind": "kindergarten",\n      "name": "Детский сад №461",\n      "lat": 55.8023453,\n      "lon": 37.7948651,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719158",\n      "kind": "kindergarten",\n      "name": "Школа «Содружество». Учебный корпус №1-1",\n      "lat": 55.7615986,\n      "lon": 37.7011981,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719159",\n      "kind": "kindergarten",\n      "name": "Детский сад №1798",\n      "lat": 55.7631259,\n      "lon": 37.7117421,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5305719162",\n      "kind": "kindergarten",\n      "name": "ДОО Розовый Слоненок",\n      "lat": 55.6690273,\n      "lon": 37.7535564,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5332088797",\n      "kind": "kindergarten",\n      "name": "Школа № 1529 имени А. С. Грибоедова. Здание № 1",\n      "lat": 55.7464549,\n      "lon": 37.5927941,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5334698649",\n      "kind": "kindergarten",\n      "name": "Детский сад №367",\n      "lat": 55.8168407,\n      "lon": 37.7037691,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5336819124",\n      "kind": "kindergarten",\n      "name": "Центр дополнительного образования \\"Наши дети\\"",\n      "lat": 55.5854041,\n      "lon": 37.7336197,\n      "opening_hours": "Mo-Su 09:00-21:00"\n    },\n    {\n      "id": "node/5336819321",\n      "kind": "school",\n      "name": "Школа искусств \\"Перспектива\\"",\n      "lat": 55.5854819,\n      "lon": 37.7335077,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5338565872",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1251 имени генерала Шарля де Голля\\"",\n      "lat": 55.7967049,\n      "lon": 37.5051915,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5338693858",\n      "kind": "kindergarten",\n      "name": "ГБОУ \\"Школа № 1251 имени генерала Шарля де Голля\\"",\n      "lat": 55.7984227,\n      "lon": 37.5053771,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5392473021",\n      "kind": "school",\n      "name": "Перспектива",\n      "lat": 55.6966698,\n      "lon": 37.5075131,\n      "opening_hours": "Su-Sa 09:00-21:00"\n    },\n    {\n      "id": "node/5418157015",\n      "kind": "kindergarten",\n      "name": "Самый лучший",\n      "lat": 55.6254039,\n      "lon": 37.6646795,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "node/5465097276",\n      "kind": "kindergarten",\n      "name": "Школа № 1530 \\"Школа Ломоносова\\". Дошкольная группа",\n      "lat": 55.7913991,\n      "lon": 37.6842642,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5477281121",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7276692,\n      "lon": 37.5339605,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5601185228",\n      "kind": "school",\n      "name": "Международный инновационный образовательный центр",\n      "lat": 55.691929,\n      "lon": 37.6243869,\n      "opening_hours": "Mo-Fr 10:00-20:00"\n    },\n    {\n      "id": "node/5634481148",\n      "kind": "school",\n      "name": "НОУ «Первая Школа»",\n      "lat": 55.8829735,\n      "lon": 37.4915995,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5660523622",\n      "kind": "school",\n      "name": "Православный центр непрерывного образования во имя прп. Серафима Саровского",\n      "lat": 55.5903957,\n      "lon": 37.7598343,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5800842228",\n      "kind": "school",\n      "name": "Центр немецкого языка DE Lingua",\n      "lat": 55.7063408,\n      "lon": 37.5965042,\n      "opening_hours": null\n    },\n    {\n      "id": "node/5919855496",\n      "kind": "kindergarten",\n      "name": "ONE! International School",\n      "lat": 55.7681478,\n      "lon": 37.5907513,\n      "opening_hours": "Mo-Fr 08:30-18:30"\n    },\n    {\n      "id": "node/6015545790",\n      "kind": "kindergarten",\n      "name": "Умка",\n      "lat": 55.5894948,\n      "lon": 37.7621307,\n      "opening_hours": "Mo-Fr 09:00-20:00"\n    },\n    {\n      "id": "node/6040072881",\n      "kind": "kindergarten",\n      "name": "Большая Черепаха",\n      "lat": 55.6844441,\n      "lon": 37.5342768,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6050114811",\n      "kind": "school",\n      "name": "Big Ben",\n      "lat": 55.6649753,\n      "lon": 37.7545153,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6052716488",\n      "kind": "school",\n      "name": "АНО СДЦ \\"Барс\\"",\n      "lat": 55.6977391,\n      "lon": 37.7738466,\n      "opening_hours": "Mo-Fr 11:00-21:00; Sa 12:00-18:00"\n    },\n    {\n      "id": "node/6067290559",\n      "kind": "school",\n      "name": "Барс",\n      "lat": 55.7029264,\n      "lon": 37.7502666,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6100272958",\n      "kind": "school",\n      "name": "Образовательный центр \\"Юниум\\"",\n      "lat": 55.74963,\n      "lon": 37.8232839,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6299092036",\n      "kind": "school",\n      "name": "Лицей НИУ ВШЭ",\n      "lat": 55.7523702,\n      "lon": 37.6374355,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6341309524",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Смышлёнок\\"",\n      "lat": 55.6230445,\n      "lon": 37.7051784,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6413499947",\n      "kind": "school",\n      "name": "Итальянская школа им. Итало Кальвино",\n      "lat": 55.682994,\n      "lon": 37.5341919,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6413499950",\n      "kind": "school",\n      "name": "Московская японская школа",\n      "lat": 55.6830015,\n      "lon": 37.5337172,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6415850521",\n      "kind": "kindergarten",\n      "name": "Sun School",\n      "lat": 55.8408588,\n      "lon": 37.4942243,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6435567397",\n      "kind": "kindergarten",\n      "name": "Центр детского развития Сундучок",\n      "lat": 55.7654354,\n      "lon": 37.7028599,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6452268986",\n      "kind": "kindergarten",\n      "name": "Школа № 1531 имени С. К. Годовикова. Дошкольное отделение",\n      "lat": 55.8012686,\n      "lon": 37.6330007,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6625005385",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6984981,\n      "lon": 37.6361301,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6632780514",\n      "kind": "kindergarten",\n      "name": "Английский детский сад",\n      "lat": 55.8856659,\n      "lon": 37.6780571,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6659929215",\n      "kind": "kindergarten",\n      "name": "Happy Elephant",\n      "lat": 55.7993168,\n      "lon": 37.5207014,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6665428945",\n      "kind": "school",\n      "name": "Центр спорта и образования \\"Локомотив\\"",\n      "lat": 55.8061697,\n      "lon": 37.7423295,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6841199202",\n      "kind": "kindergarten",\n      "name": "Послушайка",\n      "lat": 55.670609,\n      "lon": 37.6117617,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6866820508",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.7961228,\n      "lon": 37.8017832,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6868031109",\n      "kind": "kindergarten",\n      "name": "iQ-Сад",\n      "lat": 55.6671613,\n      "lon": 37.7885163,\n      "opening_hours": null\n    },\n    {\n      "id": "node/6988316009",\n      "kind": "school",\n      "name": "Первый экстернат",\n      "lat": 55.7429862,\n      "lon": 37.5374481,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7018057539",\n      "kind": "school",\n      "name": "Русская классическая гимназия",\n      "lat": 55.7711404,\n      "lon": 37.6166484,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7047225544",\n      "kind": "kindergarten",\n      "name": "Школа № 1231 дошкольное отделение",\n      "lat": 55.7428661,\n      "lon": 37.5803554,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7057170497",\n      "kind": "kindergarten",\n      "name": "Китенок",\n      "lat": 55.6290768,\n      "lon": 37.7411759,\n      "opening_hours": "Mo-Su 09:00-21:00"\n    },\n    {\n      "id": "node/7118093215",\n      "kind": "school",\n      "name": "Общеобразовательная православная школа искусств",\n      "lat": 55.781397,\n      "lon": 37.5767317,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7219021062",\n      "kind": "kindergarten",\n      "name": "Smile Fish",\n      "lat": 55.7875385,\n      "lon": 37.6333371,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7223451181",\n      "kind": "kindergarten",\n      "name": "Совёнок",\n      "lat": 55.6049805,\n      "lon": 37.7245161,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7223585386",\n      "kind": "kindergarten",\n      "name": "Детский сад Kidsberry",\n      "lat": 55.7548657,\n      "lon": 37.5977827,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "node/7223585387",\n      "kind": "kindergarten",\n      "name": "Детский сад Chekins",\n      "lat": 55.7493615,\n      "lon": 37.5654449,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "node/7223585388",\n      "kind": "kindergarten",\n      "name": "Детский сад Kidsberry",\n      "lat": 55.7496902,\n      "lon": 37.5768652,\n      "opening_hours": "Mo-Fr 08:30-19:00"\n    },\n    {\n      "id": "node/7257951974",\n      "kind": "kindergarten",\n      "name": "KidStarter",\n      "lat": 55.7880531,\n      "lon": 37.6710359,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7296709837",\n      "kind": "kindergarten",\n      "name": "Островок",\n      "lat": 55.6919141,\n      "lon": 37.5582648,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7315431992",\n      "kind": "kindergarten",\n      "name": "Мой садик",\n      "lat": 55.6038138,\n      "lon": 37.7253462,\n      "opening_hours": "Mo-Su 08:00-20:00"\n    },\n    {\n      "id": "node/7366466241",\n      "kind": "kindergarten",\n      "name": "Чудесная лужайка",\n      "lat": 55.828081,\n      "lon": 37.6646787,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7388381455",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Тигрёнок\\"",\n      "lat": 55.7912608,\n      "lon": 37.5010622,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7388383892",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Василёк\\"",\n      "lat": 55.7928782,\n      "lon": 37.48464,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7388383895",\n      "kind": "kindergarten",\n      "name": "Курчатовская школа корпус \\"Незабудка\\"",\n      "lat": 55.7968576,\n      "lon": 37.4838136,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7403153966",\n      "kind": "kindergarten",\n      "name": "Непоседы",\n      "lat": 55.7441279,\n      "lon": 37.8031669,\n      "opening_hours": "Mo-Fr 07:00-20:00; Sa-Su 08:00-19:30"\n    },\n    {\n      "id": "node/7403181580",\n      "kind": "kindergarten",\n      "name": "Карапузики",\n      "lat": 55.7438525,\n      "lon": 37.8045053,\n      "opening_hours": "Mo-Fr 07:00-20:00; Sa-Su 08:00-19:30"\n    },\n    {\n      "id": "node/7632284524",\n      "kind": "kindergarten",\n      "name": "Алые паруса",\n      "lat": 55.8723226,\n      "lon": 37.4856616,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7714534120",\n      "kind": "kindergarten",\n      "name": "Kids World International Preschool",\n      "lat": 55.7676724,\n      "lon": 37.6605467,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7732273582",\n      "kind": "kindergarten",\n      "name": "Bambini-Club",\n      "lat": 55.7982687,\n      "lon": 37.5686797,\n      "opening_hours": "Mo-Fr 07:30-20:00"\n    },\n    {\n      "id": "node/7832365986",\n      "kind": "school",\n      "name": "Bambini-Club",\n      "lat": 55.6697134,\n      "lon": 37.4738556,\n      "opening_hours": "Mo-Fr 07:30-19:00;Sa-Su 09:00-18:00"\n    },\n    {\n      "id": "node/7876939796",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.730467,\n      "lon": 37.5801543,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7934421513",\n      "kind": "kindergarten",\n      "name": "Гений",\n      "lat": 55.8178886,\n      "lon": 37.8217582,\n      "opening_hours": "Mo,We 11:00-12:15,16:00-20:00; Tu,Th,Fr 16:00-20:00; Su 11:00-17:00"\n    },\n    {\n      "id": "node/7986070109",\n      "kind": "kindergarten",\n      "name": "People&kids",\n      "lat": 55.7329437,\n      "lon": 37.6177575,\n      "opening_hours": null\n    },\n    {\n      "id": "node/7992280518",\n      "kind": "kindergarten",\n      "name": "Мозаика",\n      "lat": 55.7316491,\n      "lon": 37.5856852,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8033690833",\n      "kind": "school",\n      "name": "Лицей \\"Москвич\\"",\n      "lat": 55.6778188,\n      "lon": 37.5178345,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8082242566",\n      "kind": "school",\n      "name": "Ирида",\n      "lat": 55.7301696,\n      "lon": 37.8118483,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8329707709",\n      "kind": "kindergarten",\n      "name": "Alertum Kids",\n      "lat": 55.8120963,\n      "lon": 37.6584774,\n      "opening_hours": "Mo-Su 08:00-20:00"\n    },\n    {\n      "id": "node/8369406538",\n      "kind": "kindergarten",\n      "name": "Школа 152 дошкольное отделение",\n      "lat": 55.7981345,\n      "lon": 37.5453217,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8429777503",\n      "kind": "kindergarten",\n      "name": "Детский сад \\"Сфера\\"",\n      "lat": 55.8927808,\n      "lon": 37.6611203,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8445715562",\n      "kind": "kindergarten",\n      "name": "Кембриджуниор",\n      "lat": 55.7831029,\n      "lon": 37.6604274,\n      "opening_hours": "Mo-Su 08:00-20:00"\n    },\n    {\n      "id": "node/8606067848",\n      "kind": "kindergarten",\n      "name": "Умный малыш",\n      "lat": 55.7427861,\n      "lon": 37.6608029,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8628476525",\n      "kind": "kindergarten",\n      "name": "Начальная школа — детский сад «Радуга»",\n      "lat": 55.7178006,\n      "lon": 37.5766106,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8628476532",\n      "kind": "kindergarten",\n      "name": "Начальная школа — детский сад «Радуга»",\n      "lat": 55.7188998,\n      "lon": 37.5781032,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8635443934",\n      "kind": "kindergarten",\n      "name": "Бемби-клуб",\n      "lat": 55.8145182,\n      "lon": 37.5621486,\n      "opening_hours": "Mo-Fr 08:00-19:00"\n    },\n    {\n      "id": "node/8715324604",\n      "kind": "school",\n      "name": "Школа №1505 \\"Преображенская\\"",\n      "lat": 55.8039734,\n      "lon": 37.7354515,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8715324619",\n      "kind": "school",\n      "name": "Школа №1505 \\"Преображенская\\"",\n      "lat": 55.8008166,\n      "lon": 37.737962,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8752557700",\n      "kind": "kindergarten",\n      "name": "Avokado kids",\n      "lat": 55.780932,\n      "lon": 37.44999,\n      "opening_hours": "Mo-Fr 07:45-20:15"\n    },\n    {\n      "id": "node/8839210217",\n      "kind": "school",\n      "name": "Русский Гарвард",\n      "lat": 55.6359375,\n      "lon": 37.617555,\n      "opening_hours": "Mo-Su 09:00-20:00"\n    },\n    {\n      "id": "node/8840934476",\n      "kind": "school",\n      "name": "Православная школа им. Преподобного Сергия Радонежского",\n      "lat": 55.8537147,\n      "lon": 37.6316461,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8920785773",\n      "kind": "kindergarten",\n      "name": "Школа «Содружество». Учебный корпус №7",\n      "lat": 55.7538675,\n      "lon": 37.7138382,\n      "opening_hours": null\n    },\n    {\n      "id": "node/8933385375",\n      "kind": "kindergarten",\n      "name": "Школа № 1248. Центр «Ватутино»",\n      "lat": 55.7209908,\n      "lon": 37.4565554,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9008304817",\n      "kind": "school",\n      "name": "Полиглот",\n      "lat": 55.6474353,\n      "lon": 37.7496871,\n      "opening_hours": "Mo-Fr 09:00-21:00; Sa 09:00-18:00; Su 09:00-15:00"\n    },\n    {\n      "id": "node/9024732566",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8542844,\n      "lon": 37.5879219,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9288072222",\n      "kind": "kindergarten",\n      "name": "Детвариум",\n      "lat": 55.8548335,\n      "lon": 37.4484108,\n      "opening_hours": "Mo-Su 10:00-22:00"\n    },\n    {\n      "id": "node/9506713321",\n      "kind": "kindergarten",\n      "name": "Хорошие дети",\n      "lat": 55.879124,\n      "lon": 37.6933034,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9580996533",\n      "kind": "kindergarten",\n      "name": "Совёнок",\n      "lat": 55.6053532,\n      "lon": 37.7224629,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9687922817",\n      "kind": "kindergarten",\n      "name": "Sun School",\n      "lat": 55.7843624,\n      "lon": 37.4484654,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9725167200",\n      "kind": "kindergarten",\n      "name": "Magic Castle",\n      "lat": 55.7719793,\n      "lon": 37.6118258,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9806916428",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8231654,\n      "lon": 37.5685573,\n      "opening_hours": null\n    },\n    {\n      "id": "node/9880328119",\n      "kind": "kindergarten",\n      "name": "Чудо чердачок",\n      "lat": 55.7533666,\n      "lon": 37.7057881,\n      "opening_hours": "Mo-Su 10:00-22:00"\n    },\n    {\n      "id": "node/9886687621",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.8740544,\n      "lon": 37.5990769,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10044484701",\n      "kind": "kindergarten",\n      "name": "Обнимая солнце",\n      "lat": 55.8009863,\n      "lon": 37.7089167,\n      "opening_hours": "Mo-Sa 09:00-20:00"\n    },\n    {\n      "id": "node/10064274923",\n      "kind": "kindergarten",\n      "name": "Смарт скул",\n      "lat": 55.7436924,\n      "lon": 37.5105308,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10649421167",\n      "kind": "kindergarten",\n      "name": "Дарвиль",\n      "lat": 55.8810075,\n      "lon": 37.5260088,\n      "opening_hours": "Mo-Fr 08:00-20:00"\n    },\n    {\n      "id": "node/10709028671",\n      "kind": "school",\n      "name": "Подсолнух",\n      "lat": 55.8184515,\n      "lon": 37.5900516,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10741456003",\n      "kind": "kindergarten",\n      "name": "Многогранник Тропарёво",\n      "lat": 55.6443406,\n      "lon": 37.4907795,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10741504207",\n      "kind": "kindergarten",\n      "name": "Многогранник Раменки",\n      "lat": 55.7047156,\n      "lon": 37.4996105,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10795032875",\n      "kind": "kindergarten",\n      "name": "1450 Олимп",\n      "lat": 55.6517294,\n      "lon": 37.6166643,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10810258626",\n      "kind": "kindergarten",\n      "name": "Горница-узорница",\n      "lat": 55.7099574,\n      "lon": 37.6090493,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10821765498",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.6641116,\n      "lon": 37.5157208,\n      "opening_hours": "Mo-Fr 08:00-20:00; Sa 10:00-16:00"\n    },\n    {\n      "id": "node/10821842294",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.6888531,\n      "lon": 37.5809841,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10821944261",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.825085,\n      "lon": 37.5974592,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10821994773",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.8533145,\n      "lon": 37.4974823,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10823789312",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.6577986,\n      "lon": 37.5877138,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10823804270",\n      "kind": "kindergarten",\n      "name": "Prokids",\n      "lat": 55.8704288,\n      "lon": 37.5933234,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10936228694",\n      "kind": "kindergarten",\n      "name": "Детский сад № 2243",\n      "lat": 55.7166404,\n      "lon": 37.5956299,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10977220398",\n      "kind": "kindergarten",\n      "name": "Sky Garden",\n      "lat": 55.853089,\n      "lon": 37.6234719,\n      "opening_hours": null\n    },\n    {\n      "id": "node/10994921689",\n      "kind": "kindergarten",\n      "name": "Sky Garden",\n      "lat": 55.8517153,\n      "lon": 37.6242535,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11043096771",\n      "kind": "kindergarten",\n      "name": "Космо Кидс Отрадное",\n      "lat": 55.8510603,\n      "lon": 37.6176533,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11085165913",\n      "kind": "kindergarten",\n      "name": "Домик счастья",\n      "lat": 55.7530238,\n      "lon": 37.7084085,\n      "opening_hours": "Mo-Fr 09:00-20:00"\n    },\n    {\n      "id": "node/11085236054",\n      "kind": "kindergarten",\n      "name": "Ласточка",\n      "lat": 55.7510438,\n      "lon": 37.7093043,\n      "opening_hours": "Mo-Fr 10:00-19:30; Sa 10:00-18:00"\n    },\n    {\n      "id": "node/11108426366",\n      "kind": "kindergarten",\n      "name": "Киндерли",\n      "lat": 55.8107715,\n      "lon": 37.5573152,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11361866977",\n      "kind": "kindergarten",\n      "name": "Moscow International School",\n      "lat": 55.7476556,\n      "lon": 37.5188023,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11361932184",\n      "kind": "kindergarten",\n      "name": "Academkids",\n      "lat": 55.7506234,\n      "lon": 37.5162488,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11505302813",\n      "kind": "kindergarten",\n      "name": "Школа №1251 имени генерала Шарля де Голля, корпус №9",\n      "lat": 55.7938781,\n      "lon": 37.512356,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11505302818",\n      "kind": "kindergarten",\n      "name": "Школа №1251 имени генерала Шарля де Голля, корпус №1",\n      "lat": 55.7927389,\n      "lon": 37.5123118,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11584098151",\n      "kind": "kindergarten",\n      "name": "Школа № 56 имени Легасова. Дошкольное отделение",\n      "lat": 55.7480058,\n      "lon": 37.5514124,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11598702097",\n      "kind": "kindergarten",\n      "name": "Школа №\xa067. Отделение дошкольного образования",\n      "lat": 55.7393167,\n      "lon": 37.515432,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11618355902",\n      "kind": "kindergarten",\n      "name": "Discovery",\n      "lat": 55.7361332,\n      "lon": 37.5010461,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11618734695",\n      "kind": "kindergarten",\n      "name": "",\n      "lat": 55.6434032,\n      "lon": 37.5196015,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11634229389",\n      "kind": "kindergarten",\n      "name": "Могу - Детская Архитектурная Студия",\n      "lat": 55.7550089,\n      "lon": 37.6477373,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11760867026",\n      "kind": "school",\n      "name": "Академическая Гимназия",\n      "lat": 55.755475,\n      "lon": 37.5085821,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11775013424",\n      "kind": "kindergarten",\n      "name": "Старт",\n      "lat": 55.7567506,\n      "lon": 37.5105635,\n      "opening_hours": null\n    },\n    {\n      "id": "node/11780110793",\n      "kind": "kindergarten",\n      "name": "Хэппи Нэппи",\n      "lat": 55.795978,\n      "lon": 37.7070808,\n      "opening_hours": null\n    },\n    {\n      "id": "node/12043951605",\n      "kind": "kindergarten",\n      "name": "Детская Академия Развития",\n      "lat": 55.6505559,\n      "lon": 37.5715486,\n      "opening_hours": null\n    },\n    {\n      "id": "node/12215008301",\n      "kind": "school",\n      "name": "Школа №1636, здание 17",\n      "lat": 55.6017334,\n      "lon": 37.731282,\n      "opening_hours": null\n    },\n    {\n      "id": "node/12241062907",\n      "kind": "kindergarten",\n      "name": "Космо Кидс",\n      "lat": 55.7254195,\n      "lon": 37.7429032,\n      "opening_hours": null\n    },\n    {\n      "id": "node/12241615196",\n      "kind": "kindergarten",\n      "name": "Discovery",\n      "lat": 55.7405802,\n      "lon": 37.6012324,\n      "opening_hours": null\n    },\n    {\n      "id": "node/12346514390",\n      "kind": "school",\n      "name": "",\n      "lat": 55.7634623,\n      "lon": 37.5112229,\n      "opening_hours": null\n    }\n  ],\n  "limitations": [\n    "Counts are approximate mapped sites, not enrollment or daily attendance.",\n    "Distance uses OSM point/area center to GTFS stop, not walking distance.",\n    "OSM missing sites and mislabeled objects remain possible.",\n    "POIs use Jan 2025 snapshot; GTFS is retrospective June 2026 geometry.",\n    "Opening-hours tags are too sparse to assign verified timetables to every institution."\n  ]\n}\n', 'ml/forecast/build_education_registry.py': '"""Build a small attributed historical OSM exposure registry, without passenger imputation."""\nimport json,math,hashlib\nfrom pathlib import Path\nimport numpy as np\nROOT=Path(__file__).resolve().parents[2]\nRADIUS=500.\ndef distances(lat,lon,coords):\n    a=np.radians(coords);p=math.radians(lat);dl=a[:,0]-p;dn=a[:,1]-math.radians(lon)\n    return 6371000*2*np.arcsin(np.sqrt(np.clip(np.sin(dl/2)**2+np.cos(p)*np.cos(a[:,0])*np.sin(dn/2)**2,0,1)))\ndef main():\n    path=ROOT/\'artifacts/education/osm_school_kindergarten_20250101.json\';raw=json.loads(path.read_text());feed=json.loads((ROOT/\'artifacts/metro_audit/gtfs_routes.json\').read_text())\n    points=[];dropped=[]\n    # Prefer areas to nodes where both map the same object; conservative geometric dedup.\n    for e in sorted(raw[\'elements\'],key=lambda x:({\'relation\':0,\'way\':1,\'node\':2}[x[\'type\']],x[\'id\'])):\n        tags=e[\'tags\'];pos=e.get(\'center\',e);lat,lon=pos[\'lat\'],pos[\'lon\'];kind=tags[\'amenity\'];name=tags.get(\'name\',\'\').casefold().strip()\n        same=[p for p in points if p[\'kind\']==kind]\n        duplicate=False\n        if same:\n            ds=distances(lat,lon,np.array([[p[\'lat\'],p[\'lon\']] for p in same]))\n            duplicate=any(d<=20 or (name and name==p[\'name\'].casefold().strip() and d<=60) for p,d in zip(same,ds))\n        if duplicate:dropped.append(f"{e[\'type\']}/{e[\'id\']}");continue\n        points.append({\'id\':f"{e[\'type\']}/{e[\'id\']}",\'kind\':kind,\'name\':tags.get(\'name\',\'\'),\'lat\':lat,\'lon\':lon,\'opening_hours\':tags.get(\'opening_hours\')})\n    routes={}\n    for route,item in feed[\'routes\'].items():\n        stops=np.array([[float(s[\'stop_lat\']),float(s[\'stop_lon\'])] for s in item[\'stops\'].values()]);near=[]\n        for p in points:\n            if min(distances(p[\'lat\'],p[\'lon\'],stops))<=RADIUS:near.append(p)\n        routes[route]={\'school\':sum(p[\'kind\']==\'school\' for p in near),\'kindergarten\':sum(p[\'kind\']==\'kindergarten\' for p in near),\'poi_ids\':[p[\'id\'] for p in near],\'known_opening_hours\':sum(bool(p[\'opening_hours\']) for p in near)}\n    out={\'source\':\'https://www.openstreetmap.org/copyright\',\'license\':\'ODbL; © OpenStreetMap contributors\',\'api\':\'https://overpass-api.de/api/interpreter\',\'osm_snapshot\':\'2025-01-01T00:00:00Z\',\'osm_sha256\':hashlib.file_digest(path.open(\'rb\'),\'sha256\').hexdigest(),\'query\':(ROOT/\'artifacts/education/overpass_query.txt\').read_text(),\'gtfs_sha256\':feed[\'archive_sha256\'],\'radius_m\':RADIUS,\'raw_objects\':len(raw[\'elements\']),\'retained_objects\':len(points),\'deduplicated_ids\':dropped,\'routes\':routes,\'poi\':points,\'limitations\':[\'Counts are approximate mapped sites, not enrollment or daily attendance.\',\'Distance uses OSM point/area center to GTFS stop, not walking distance.\',\'OSM missing sites and mislabeled objects remain possible.\',\'POIs use Jan 2025 snapshot; GTFS is retrospective June 2026 geometry.\',\'Opening-hours tags are too sparse to assign verified timetables to every institution.\']}\n    (ROOT/\'ml/forecast/education_registry.json\').write_text(json.dumps(out,ensure_ascii=False,indent=2)+\'\\n\')\n    print(\'Raw\',len(raw[\'elements\']),\'retained\',len(points),\'routes\',routes)\nif __name__==\'__main__\':main()\n', 'ml/forecast/test_education.py': "import unittest\nfrom datetime import date\nimport numpy as np\nfrom ml.forecast.education_features import calendar_features,exposure_features\nfrom ml.forecast.education_competition import matrix,active,choose\nfrom ml.forecast.build_education_registry import distances\n\nclass EducationTests(unittest.TestCase):\n    def test_daycare_not_closed_for_school_summer(self):\n        c=calendar_features(('17',date(2025,7,8),8))\n        self.assertEqual(c[:3],[0,0,1]);self.assertEqual(c[7],1)\n\n    def test_calendar_scenarios_remain_separate(self):\n        c=calendar_features(('11',date(2025,10,6),8))\n        self.assertEqual(c[:3],[1,0,1])\n        c=calendar_features(('11',date(2025,10,27),8))\n        self.assertEqual(c[:3],[0,1,1])\n\n    def test_holidays_and_unknown_year(self):\n        self.assertEqual(calendar_features(('11',date(2025,11,4),8)),[0]*9)\n        with self.assertRaises(ValueError):calendar_features(('11',date(2026,1,1),8))\n\n    def test_unknown_route_no_fake_pois_and_service_routes_protected(self):\n        self.assertEqual(exposure_features(('5',date(2025,9,1),8)),[0]*11)\n        for r in ('5','7','50'):self.assertFalse(active((r,date(2025,9,1),8)))\n\n    def test_matched_arms_preserve_control_columns(self):\n        k=('17',date(2025,10,6),8);b={k:200};w={(k[1],8):[1.]*10}\n        a=matrix([k],b,w,'control');c=matrix([k],b,w,'calendar');e=matrix([k],b,w,'education')\n        np.testing.assert_equal(a,c[:,:a.shape[1]]);np.testing.assert_equal(c,e[:,:c.shape[1]])\n\n    def test_distance_units(self):\n        d=distances(55,37,np.array([[55,37],[55.01,37]]))\n        self.assertAlmostEqual(d[0],0);self.assertTrue(1100<d[1]<1120)\n\n    def test_poi_not_credited_for_calendar_gain(self):\n        ks=[('17',date(2025,m,6),8) for m in (9,10)];truth=dict.fromkeys(ks,100)\n        p={'incumbent':dict.fromkeys(ks,80)}\n        for w in (.25,.5,1.):\n            p[f'control_{w}']=dict.fromkeys(ks,80)\n            p[f'calendar_{w}']=dict.fromkeys(ks,90)\n            p[f'education_{w}']=dict.fromkeys(ks,90)\n        choices,_=choose(ks,truth,p,ks,p)\n        self.assertTrue(choices['17'].startswith('calendar_'))\n        for w in (.25,.5,1.):p[f'education_{w}']=dict.fromkeys(ks,95)\n        choices,_=choose(ks,truth,p,ks,p)\n        self.assertTrue(choices['17'].startswith('education_'))\n\nclass EducationProfileTests(unittest.TestCase):\n    def test_day_totals_and_protected_routes(self):\n        from ml.forecast.education_profiles import EducationProfiles\n        from ml.forecast.core import grid\n        history={k:(100 if k[2]==8 else 10) for k in grid(['17'],date(2025,1,1),date(2025,4,30))}\n        model=EducationProfiles().fit(history,date(2025,4,30))\n        keys=grid(['17','7','50','5'],date(2025,5,5),date(2025,5,5));base={k:(0 if k[0]=='5' or k[2]<5 else 30) for k in keys}\n        for name,p in model.variants(keys,base).items():\n            self.assertEqual(sum(p.values()),sum(base.values()))\n            self.assertTrue(all(p[k]==base[k] for k in keys if k[0]!='17'))\n            self.assertTrue(all(p[k]==0 for k in keys if base[k]==0))\n\n    def test_profile_rejects_future_history_and_partial_days(self):\n        from ml.forecast.education_profiles import EducationProfiles\n        with self.assertRaises(ValueError):EducationProfiles().fit({('17',date(2025,2,1),8):100},date(2025,1,31))\n        model=EducationProfiles().fit({('17',date(2025,1,1),8):100},date(2025,1,31))\n        k=('17',date(2025,2,1),8)\n        with self.assertRaises(ValueError):model.variants([k],{k:100})\n", 'ml/forecast/education_profiles.py': '"""Second preregistered experiment: calendar-conditioned hour shares; daily totals fixed."""\nfrom collections import defaultdict\nfrom datetime import date\nimport json,pickle,argparse\nfrom pathlib import Path\nimport numpy as np\nfrom .education_features import closed,QUARTER,MODULAR\nfrom .education_competition import TARGETS\nfrom .hourly_experts import allocate,HourlyExperts,HourlyExpertEnsemble,combine\nfrom .core import days,effective_weekday,grid,load_labels,sha256\nfrom .run import FIRST,FINAL_CUTOFF\nfrom . import network_competition as nc\nfrom .weather_competition import weather_rows\nfrom ml.submit.adapter import read_sample,write_submission\n\nWEIGHTS=(.25,.5)\n\ndef summer(d):return date(2025,5,27)<=d<=date(2025,8,31)\ndef condition(d,mode):\n    if mode==\'control\':return 0\n    return int(closed(d,QUARTER if mode==\'quarter\' else MODULAR))\n\nclass EducationProfiles:\n    def fit(self,history,cutoff):\n        if any(k[1]>cutoff or k[1]<FIRST for k in history):raise ValueError(\'History crosses cutoff\')\n        self.cutoff=cutoff;groups=defaultdict(list)\n        for r in TARGETS:\n            for d in days(FIRST,cutoff):\n                y=np.array([history.get((r,d,h),0) for h in range(24)],dtype=float)\n                if y.sum()<=0:continue\n                for mode in (\'control\',\'quarter\',\'modular\'):\n                    groups[r,effective_weekday(d),summer(d),mode,condition(d,mode)].append(y/y.sum())\n        self.shapes={}\n        for k,rows in groups.items():\n            if len(rows)<4:continue\n            a=np.median(rows,axis=0);b=np.median(rows[-8:],axis=0)\n            if a.sum()>0 and b.sum()>0:self.shapes[k]=.5*a/a.sum()+.5*b/b.sum()\n        return self\n    def shape(self,r,d,mode):\n        return self.shapes.get((r,effective_weekday(d),summer(d),mode,condition(d,mode)),self.shapes.get((r,effective_weekday(d),summer(d),\'control\',0)))\n    def variants(self,keys,base):\n        if any(k[1]<=self.cutoff for k in keys):raise ValueError(\'Predict after cutoff\')\n        result={\'incumbent\':dict(base)}\n        for mode in (\'control\',\'quarter\',\'modular\',\'mixed\'):\n            for w in WEIGHTS:result[f\'{mode}_{w}\']=dict(base)\n        grouped=defaultdict(list)\n        for k in keys:\n            if k[0] in TARGETS:grouped[k[0],k[1]].append(k)\n        for (r,d),ks in grouped.items():\n            if len(ks)!=24 or {k[2] for k in ks}!=set(range(24)):raise ValueError(\'Require full unique daily hours\')\n            ks=sorted(ks,key=lambda k:k[2]);v=np.array([base[k] for k in ks],float);total=int(v.sum())\n            if total<=0:continue\n            old=v/total\n            for mode in (\'control\',\'quarter\',\'modular\',\'mixed\'):\n                if mode==\'mixed\':\n                    a=self.shape(r,d,\'quarter\');b=self.shape(r,d,\'modular\');shape=(a+b)/2 if a is not None and b is not None else None\n                else:shape=self.shape(r,d,mode)\n                if shape is None:continue\n                shape=np.clip(shape,.5*old,1.5*old)\n                if shape.sum()<=0:continue\n                shape=shape/shape.sum()\n                for w in WEIGHTS:\n                    values=allocate(total,(1-w)*old+w*shape)\n                    result[f\'{mode}_{w}\'].update(zip(ks,values))\n        return result\n\ndef choose(keys,truth,preds,oks,op):\n    choices={};audit={}\n    for r in TARGETS:\n        parts=[[k for k in keys if k[0]==r and k[1].month==m] for m in (9,10)]\n        errors={n:[sum(abs(p[k]-truth.get(k,0)) for k in ks) for ks in parts] for n,p in preds.items()}\n        oe={n:sum(abs(p[k]-truth.get(k,0)) for k in oks if k[0]==r) for n,p in op.items()}\n        accepted=[]\n        for mode in (\'quarter\',\'modular\',\'mixed\'):\n            for w in WEIGHTS:\n                n=f\'{mode}_{w}\';c=f\'control_{w}\'\n                if all(a<=.995*b for a,b in zip(errors[n],errors[\'incumbent\'])) and all(a<=.995*b for a,b in zip(errors[n],errors[c])) and oe[n]<=1.005*oe[\'incumbent\'] and oe[n]<=1.005*oe[c]:accepted.append(n)\n        choices[r]=min(accepted,key=lambda n:sum(errors[n])) if accepted else \'incumbent\'\n        audit[r]={\'errors_sep_oct\':errors,\'errors_october_refit\':oe,\'accepted\':accepted,\'selected\':choices[r]}\n    return choices,audit\n\ndef join(keys,ps,choice):return {k:ps[choice.get(k[0],\'incumbent\')][k] for k in keys}\nclass Ensemble:\n    def __init__(self,baseline,profiles,choices):self.baseline,self.profiles,self.choices=baseline,profiles,choices\n    def predict(self,keys):\n        keys=list(keys);return join(keys,self.profiles.variants(keys,self.baseline.predict(keys)),self.choices)\n\ndef main(argv=None):\n    ap=argparse.ArgumentParser()\n    for name in (\'train\',\'test\',\'sample\',\'weather\',\'baseline-dir\',\'out\'):ap.add_argument(\'--\'+name,type=Path,required=True)\n    ap.add_argument(\'--sample-from-reference\',action=\'store_true\');args=ap.parse_args(argv)\n    nc.verify_competition_inputs({n:getattr(args,n) for n in (\'train\',\'test\',\'sample\')},args.sample_from_reference)\n    previous=json.loads((args.baseline_dir/\'hourly_experts_report.json\').read_text());values=load_labels([args.train,args.test]);ref=previous[\'reference\'];route_choices=previous[\'selection\'][\'by_route\'];hourly=previous[\'hourly_experts\'][\'choices\']\n    weather=weather_rows(json.loads(args.weather.read_text()));bank=nc.HistoryBank(values,ref[\'choices\']);held={};folds={};args.out.mkdir(parents=True,exist_ok=True)\n    for label,cutoff,lo,hi in [(\'sep_oct\',date(2025,8,31),date(2025,9,1),date(2025,10,31)),(\'october_refit\',date(2025,9,30),date(2025,10,1),date(2025,10,31))]:\n        print(\'Calendar hour profiles\',label,flush=True)\n        history={k:v for k,v in values.items() if k[1]<=cutoff};keys=grid(bank.routes,lo,hi);bases=bank.predict_bases(cutoff,keys);models,_=nc.fit_models(bank,cutoff,weather)\n        base=nc.assemble(keys,nc.predict_models(keys,bases,models,weather),route_choices);hp=HourlyExperts().fit(history,FIRST,cutoff);base=combine(keys,hp.variants(keys,base),hourly)\n        ps=EducationProfiles().fit(history,cutoff).variants(keys,base);held[label]=(keys,ps);folds[label]={\'metrics\':{n:nc.metrics(keys,values,p) for n,p in ps.items()}}\n    choices,audit=choose(held[\'sep_oct\'][0],values,held[\'sep_oct\'][1],*held[\'october_refit\'])\n    for label,(ks,ps) in held.items():folds[label][\'selected\']=nc.metrics(ks,values,join(ks,ps,choices))\n    _,sample=read_sample(args.sample);reference,regimes=bank.fitted(FINAL_CUTOFF);models,_=nc.fit_models(bank,FINAL_CUTOFF,weather)\n    baseline=nc.HolidayServiceEnsemble(nc.NetworkWeatherEnsemble(reference,regimes,models,weather,route_choices),reference,regimes)\n    baseline=HourlyExpertEnsemble(baseline,HourlyExperts().fit(values,FIRST,FINAL_CUTOFF),hourly)\n    before=baseline.predict(sample);write_submission(args.sample,before,args.out/\'submission_best_088078.csv\')\n    from .education_competition import BEST_HASH\n    assert sha256(args.out/\'submission_best_088078.csv\')==BEST_HASH\n    ensemble=Ensemble(baseline,EducationProfiles().fit(values,FINAL_CUTOFF),choices);after=ensemble.predict(sample);artifact=pickle.dumps(ensemble,protocol=5);assert pickle.loads(artifact).predict(sample)==after\n    write_submission(args.sample,after,args.out/\'submission_education_profiles.csv\');(args.out/\'model.local.pkl\').write_bytes(artifact)\n    report={\'experiment\':\'Second exploratory experiment: calendar-conditioned daily shares; not a proven POI effect.\',\'source_mode\':\'competition\',\'selected\':choices,\'selection\':audit,\'folds\':folds,\'primary_submission\':\'submission_education_profiles.csv\',\'changed_rows\':{r:sum(before[k]!=after[k] for k in sample if k[0]==r) for r in sorted({k[0] for k in sample},key=int)},\'daily_totals_preserved\':True,\'outputs_sha256\':{n:sha256(args.out/n) for n in (\'submission_education_profiles.csv\',\'submission_best_088078.csv\')},\'platform_score\':None}\n    (args.out/\'education_profiles_report.json\').write_text(json.dumps(report,ensure_ascii=False,indent=2)+\'\\n\');print(\'Selected:\',choices,flush=True)\nif __name__==\'__main__\':\n    from ml.forecast.education_profiles import main as run_main\n    run_main()\n'}
SOURCE_HASHES = {'ml/forecast/core.py': '819cfdc88d5b6b96bf80fb0e019eeef0678628b617d2dd077409d26265369b5f', 'ml/forecast/models.py': 'd02f6120641005057c411d120894617027ff8b6a10a0afba37a8e5612b5e2009', 'ml/forecast/route_specific.py': '3f9a0f92eb4bdccf0a4ad777efc06dcf24cf09ac03718c4eb678c553855f4a4c', 'ml/forecast/test_route_specific.py': 'a4e356ccbfafade57b8bd33821f63895a866c13f98ba80832bead1caeba7b5f5', 'ml/forecast/run.py': 'ce421c6fc8d77406a3817c0a15253f3464d360cc7c69bb78e8bd131edef46072', 'ml/forecast/prepare_inputs.py': '9fdb609444a3e443961504e76bda2ccb1d02dadb3ad88bb0420f70fe01711dd5', 'ml/forecast/test_pipeline.py': '267a014e6b267f4c792b7a90c8679c98ee94e3e36e070baa706a89f5fa8f399b', 'ml/forecast/download_yandex.py': '300dec623de4bf983a7960e23bbd85a128c120d40e25059d1a000d898c2ff37d', 'ml/forecast/test_download_yandex.py': '9ca4e25061931b50b2b1730e4085f44add1c9d9a0071cd1290779e5b939666d1', 'ml/submit/adapter.py': '9767bfd9573ea7637669e37bac42e75e45205f29d533d24d1d397f646047d97d', 'ml/forecast/weather_competition.py': 'cd21d93a65ae8f8ee66f54005ece5063e26e15cff64cd64c4e6dcf45e1d4dc11', 'ml/forecast/reference_route_model.json': 'd346c229308017f192e614a6dde8a1f88026c014325e664fe8ed611325baf894', 'ml/forecast/test_weather_competition.py': '69a0829537f335cae8d4f00e218fff3af3030069fbba319a94343791693db09d', 'ml/forecast/weather_source.json': '0a6f625573c314cfde5991f1da6c505dd5c74841141e7c94c07a9594bcbee3aa', 'ml/forecast/network_regimes.py': '4cdf72b14761f337a68d191ee8aaae816bc14cdc089170bb20199e5003e32116', 'ml/forecast/network_competition.py': '40e8902a8c0be0d2bb1818ff2049389f485c19d8a27b33053fc4bcfc83f50ebb', 'ml/forecast/test_network_competition.py': 'bd4852e10fab2e3b896d2f75b38e751f60b19bf0046888e81f8ef60019858ee4', 'ml/forecast/network_sources.json': 'e3443edfa338b44d93430f39741fb6f7b924fca00475b8642cba43d10b44ca6a', 'ml/forecast/research_audit.json': 'f947e15e786cc979ac9acf57c11f153f3c7fdffc8828aaa3a620dbd34a4821cb', 'ml/forecast/hourly_experts.py': '7661cd5ea210164959b0f995b9c64b1cd3f5fe0231881b87692f44ea35d28e11', 'ml/forecast/test_hourly_experts.py': '249664d4c5b914b0fee7776adfa15c7ef7390664c5e7aad2cbb2f97d83d0d4ea', 'ml/forecast/education_competition.py': 'e9cd5b4ecf8c9270820bbcd2bf3fa4eabc252fa5d8adcc1e8e8fa0189fb9a3e0', 'ml/forecast/education_features.py': '2a492e272b98db0b76b71ada0b53ae3a1e885780da52b42f7731be7661616530', 'ml/forecast/education_registry.json': '525a2c8d61ffef9c938906ad055a9ae776e79f8e86e0b5683ee0ee947d6f0644', 'ml/forecast/build_education_registry.py': 'bfd33fbcb355091853b0263c225dabaf59efcaaf52bc8fe022b13ca636c90b5e', 'ml/forecast/test_education.py': '8eea644a2ecaec5d57d0df5107a2fbb9297367b72e1d2dfac9c455e37e1edf2e', 'ml/forecast/education_profiles.py': '704a13512547a5aed38d3b022adfdb3819b63bbb72ac7226020c38df5aabc243'}
for name, source in SOURCES.items():
    if hashlib.sha256(source.encode()).hexdigest() != SOURCE_HASHES[name]:
        raise RuntimeError('Изменился встроенный исходник: ' + name)
    target = RUNTIME / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
print('Развёрнуты и проверены', len(SOURCES), 'модулей DS‑1.')

def run_module(module, *arguments):
    if not globals().get('MODEL_READY', False):
        raise RuntimeError('Сначала успешно выполните ячейку настройки среды.')
    command = [str(TRAIN_PYTHON), '-u', '-m', module, *map(str, arguments)]
    with subprocess.Popen(command, cwd=RUNTIME, env=MODEL_PROCESS_ENV,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    if result:
        raise RuntimeError(f'{module} завершился с ошибкой {result}. См. вывод выше.')

if RUN_TESTS:
    run_module('unittest', 'discover', '-s', 'ml/forecast', '-p', 'test_*.py', '-v')


## 3. Скачать архив с Яндекс Диска и найти входы
Официальный публичный API не требует токена. Скачивание идёт потоком в `/tmp`,
с прогрессом, повторными попытками, проверкой размера и хеша. При повторном запуске
проверенный архив используется снова; частичная загрузка докачивается, если сервер
поддерживает Range. В память целый архив не загружается.

Распаковываются только небольшие метки и шаблон; сырые транзакции остаются в ZIP.
При сетевой ошибке ячейка остановится с объяснением, а не начнёт обучение без данных.
Официальный способ получения ссылки: https://yandex.cloud/en/docs/datasphere/operations/data/connect-to-ya-disk


In [ ]:
SEARCH_ROOT = INPUT_ROOT
if DATA_SOURCE == 'yandex':
    import importlib.util
    spec = importlib.util.spec_from_file_location('ds1_yandex', RUNTIME / 'ml/forecast/download_yandex.py')
    downloader = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(downloader)
    downloaded_archive = downloader.download(YANDEX_URL, DOWNLOAD_CACHE)
    SEARCH_ROOT = downloaded_archive.parent
elif DATA_SOURCE != 'input':
    raise ValueError("DATA_SOURCE должен быть 'yandex' или 'input'")
if not SEARCH_ROOT.is_dir():
    raise FileNotFoundError('Нет входной папки. Для ручного режима добавьте набор через Add Input.')
required = {'train': 'labels_day_train.csv', 'test': 'labels_day_test.csv',
            'sample': 'test_submission.csv'}
files = {}
for role, name in required.items():
    if INPUT_FILES[role]:
        path = Path(INPUT_FILES[role])
        if not path.is_file():
            raise FileNotFoundError(path)
        files[role] = path
    else:
        matches = sorted(SEARCH_ROOT.rglob(name))
        if len(matches) > 1:
            raise RuntimeError(f'Несколько {name}: {matches}. Укажите INPUT_FILES[{role!r}].')
        if matches:
            files[role] = matches[0]

missing = set(required) - set(files)
archive_candidates = {role: [] for role in missing}
if missing:
    for archive in sorted(SEARCH_ROOT.rglob('*.zip')):
        with zipfile.ZipFile(archive) as bundle:
            for info in bundle.infolist():
                for role in missing:
                    if Path(info.filename).name == required[role]:
                        archive_candidates[role].append((archive, info.filename, info.file_size))
    for role in sorted(missing):
        matches = archive_candidates[role]
        if len(matches) != 1:
            raise RuntimeError(f'{required[role]}: найдено вариантов в ZIP: {len(matches)}. '
                               'Добавьте этот файл отдельно и укажите путь в INPUT_FILES.')
        archive, member, size = matches[0]
        if size > 20_000_000:
            raise ValueError('Неожиданно большой файл меток/шаблона: ' + member)
        with zipfile.ZipFile(archive) as bundle:
            path = EXTRACTED / required[role]
            path.write_bytes(bundle.read(member))
            files[role] = path

for role, path in files.items():
    print(role, '→', path, '|', f'{path.stat().st_size:,}', 'байт')
# Описание внутри исходного ZIP тоже можно прочитать без извлечения raw CSV.
for archive in sorted(SEARCH_ROOT.rglob('*.zip')):
    with zipfile.ZipFile(archive) as bundle:
        descriptions_in_zip = [i for i in bundle.infolist()
                               if Path(i.filename).name.lower() == 'readme.md' and i.file_size < 200_000]
        if len(descriptions_in_zip) == 1:
            description_path = EXTRACTED / 'README.md'
            description_path.write_bytes(bundle.read(descriptions_in_zip[0]))
            print('Описание из архива:\n', description_path.read_text(encoding='utf-8-sig'))
descriptions = sorted(SEARCH_ROOT.rglob('README.md'))
if len(descriptions) == 1 and descriptions[0].stat().st_size < 200_000:
    print('\nОписание набора:\n', descriptions[0].read_text(encoding='utf-8-sig'))
else:
    print('Описание не показано автоматически; сверьте README организаторов отдельно.')


## Обучение и раздельная проверка

Вначале воспроизводится модель 0,88078. Затем сравниваются одинаковые модели:
контроль; контроль + образовательный календарь; календарь + школьные и дошкольные POI.
Обучение использует только пассажирскую историю до соответствующей отсечки.
Никаких выдуманных учеников, посадок по остановкам или будущих фактических лагов.


In [ ]:
import base64, zlib
WEATHER = EXTRACTED / 'moscow_2025_era5.json'
WEATHER.write_bytes(zlib.decompress(base64.b64decode('...')))
assert hashlib.sha256(WEATHER.read_bytes()).hexdigest() == 'e6330c286fde16bdd97ed815376b7d221a383e3ad8a041a51b3df70bef80f9e0'
run_module('ml.forecast.network_competition', '--train', files['train'], '--test', files['test'],
           '--sample', files['sample'], '--weather', WEATHER,
           '--reference', RUNTIME / 'ml/forecast/reference_route_model.json', '--out', OUT, '--hourly-experts')

EDUCATION_OUT = WORK/'education_results'
run_module('ml.forecast.education_competition',
           '--train', files['train'], '--test', files['test'], '--sample', files['sample'],
           '--weather', WEATHER, '--baseline-dir', OUT, '--out', EDUCATION_OUT)

PROFILE_OUT = WORK/'education_profiles_results'
run_module('ml.forecast.education_profiles', '--train', files['train'], '--test', files['test'], '--sample', files['sample'], '--weather', WEATHER, '--baseline-dir', OUT, '--out', PROFILE_OUT)


## Итог

Основной файл — **submission_education.csv**. `calendar_...` означает, что
принят только календарь. `education_...` означает, что дополнительно прошла
проверку география школ/детсадов. `incumbent` — прежний прогноз.


In [ ]:
from IPython.display import display, Markdown, FileLink
import shutil
geo = json.loads((EDUCATION_OUT/'education_report.json').read_text())
profiles = json.loads((PROFILE_OUT/'education_profiles_report.json').read_text())
geo_error = geo['folds']['sep_oct']['selected']['all']['absolute_error_sum']
profile_error = profiles['folds']['sep_oct']['selected']['all']['absolute_error_sum']
if profile_error < geo_error:
    selected, selected_dir, selected_file, label = profiles, PROFILE_OUT, 'submission_education_profiles.csv', 'calendar_profiles'
else:
    selected, selected_dir, selected_file, label = geo, EDUCATION_OUT, 'submission_education.csv', 'calendar_and_poi'
print('Выбранный опыт:', label)
print('Варианты по маршрутам:', selected['selected'])
print('Изменено строк:', selected['changed_rows'])
print('Локальный score до:', selected['folds']['sep_oct']['metrics']['incumbent']['all']['wape_score'])
print('Локальный score после:', selected['folds']['sep_oct']['selected']['all']['wape_score'])
if not sum(selected['changed_rows'].values()):
    display(Markdown('**Улучшение не принято. CSV совпадает с лучшим 0,88078 — повторно отправлять не нужно.**'))
else:
    display(Markdown('**Отправлять submission_education.csv. Платформенный score нового файла пока неизвестен.**'))
report = {'primary_submission': 'submission_education.csv', 'selected_experiment': label,
          'experiments': {'calendar_and_poi': geo, 'calendar_profiles': profiles},
          'warning': 'Two exploratory experiments on reused validation. Calendar profile gain is not evidence of a POI effect.'}
(WORK/'education_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
shutil.copy2(selected_dir/selected_file, WORK/'submission_education.csv')
shutil.copy2(EDUCATION_OUT/'submission_best_088078.csv', WORK/'submission_best_088078.csv')
lines = ['| Маршрут | Школьные объекты | Дошкольные объекты |', '|---|---:|---:|']
for route, item in sorted(geo['registry']['routes'].items(), key=lambda kv: int(kv[0])):
    lines.append(f"| {route} | {item['school']} | {item['kindergarten']} |")
display(Markdown('\n'.join(lines)))
display(Markdown('Источник географии: © OpenStreetMap contributors, ODbL — https://www.openstreetmap.org/copyright'))
for name in ('submission_education.csv', 'education_report.json', 'submission_best_088078.csv'):
    target = WORK/name
    display(FileLink(str(target.relative_to(Path.cwd())) if target.is_relative_to(Path.cwd()) else str(target)))
